# Machine-Learning-Based Land Suitability Analysis — Enugu State

## Reproducible publication workflow

This notebook is a cleaned execution path derived from the original 59-cell production notebook. Superseded recovery attempts and cells that failed because of temporary Google Drive or Earth Engine state were removed. Their final corrected replacements are retained.

### Before running

1. Run in Google Colab.
2. Set the environment variable `EE_PROJECT` to a Google Earth Engine Cloud Project available to your account.
3. Mount Google Drive when prompted.
4. Keep the project directory at `/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability`, or update `PROJECT_ROOT` consistently.
5. Earth Engine exports are asynchronous. Confirm each export is complete before running its local-validation stage.

> The retained code cells passed static Python syntax checks. A full end-to-end execution still requires the external datasets, Earth Engine authentication and Google Drive files documented in the workflow.


In [ ]:
# =====================================================================================
# PROJECT 7
# Machine-Learning-Based Land Suitability Analysis
# for Sustainable Urban Development — Enugu, Nigeria
#
# STAGE 1A — COLAB ENVIRONMENT + PROJECT WORKSPACE
# =====================================================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# -----------------------------------------------------------------------------
# INSTALL REQUIRED PACKAGES
# -----------------------------------------------------------------------------
!pip -q install \
    geopandas \
    rasterio \
    rioxarray \
    osmnx \
    geemap \
    earthengine-api \
    pyogrio \
    shapely \
    scikit-learn \
    xgboost \
    matplotlib \
    pandas \
    numpy \
    requests

# -----------------------------------------------------------------------------
# IMPORT LIBRARIES
# -----------------------------------------------------------------------------
import os
import sys
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import shapely
import sklearn

# -----------------------------------------------------------------------------
# CREATE PROJECT DIRECTORY
# -----------------------------------------------------------------------------
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/"
    "Project_7_Enugu_ML_Land_Suitability"
)

folders = [
    "00_Project_Admin",
    "01_Boundary",
    "02_Terrain",
    "03_Remote_Sensing",
    "04_Land_Cover",
    "05_Roads_Accessibility",
    "06_Environmental_Constraints",
    "07_Population_Urban_Pressure",
    "08_Predictor_Stack",
    "09_Training_Data",
    "10_ML_Models",
    "11_Validation",
    "12_Suitability_Outputs",
    "13_Planning_Priority",
    "14_Maps",
    "15_Charts",
    "16_Tables",
    "17_Final_Report",
    "18_Infographic",
    "19_Portfolio",
    "20_GitHub",
    "21_Final_Deliverables",
]

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

for folder in folders:
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# CREATE INITIAL DATA-PROVENANCE REGISTER
# -----------------------------------------------------------------------------
provenance_file = PROJECT_ROOT / "00_Project_Admin" / "Data_Provenance_Register.csv"

if not provenance_file.exists():
    provenance = pd.DataFrame(
        columns=[
            "Dataset",
            "Provider",
            "Source_URL_or_ID",
            "Acquisition_Date",
            "Data_Date_or_Period",
            "Spatial_Resolution",
            "CRS",
            "Licence",
            "Project_Use",
            "Validation_Status",
            "Notes",
        ]
    )
    provenance.to_csv(provenance_file, index=False)

# -----------------------------------------------------------------------------
# ENVIRONMENT VALIDATION
# -----------------------------------------------------------------------------
print("=" * 92)
print("PROJECT 7 — STAGE 1A ENVIRONMENT VALIDATION")
print("=" * 92)

print(f"\nProject root:\n{PROJECT_ROOT}")

print("\nSOFTWARE ENVIRONMENT")
print("-" * 92)
print("Python:       ", sys.version.split()[0])
print("Platform:     ", platform.platform())
print("GeoPandas:    ", gpd.__version__)
print("Rasterio:     ", rasterio.__version__)
print("Shapely:      ", shapely.__version__)
print("Scikit-learn: ", sklearn.__version__)

print("\nPROJECT FOLDERS")
print("-" * 92)

existing = 0

for folder in folders:
    path = PROJECT_ROOT / folder
    status = "✓" if path.exists() else "✗"

    if path.exists():
        existing += 1

    print(f"{status} {folder}")

print("\nVALIDATION")
print("-" * 92)
print(f"Folders created: {existing}/{len(folders)}")
print(
    "Data provenance register:",
    "✓ CREATED" if provenance_file.exists() else "✗ NOT CREATED"
)

if existing == len(folders) and provenance_file.exists():
    print("\n✓ STAGE 1A PASSED")
    print("Project environment is ready.")
else:
    print("\n✗ STAGE 1A FAILED")
    print("Do not continue until the errors above are resolved.")

print("=" * 92)

In [ ]:
# PUBLICATION CONFIGURATION
import os
from pathlib import Path

EE_PROJECT = os.environ.get("EE_PROJECT", "").strip()
if not EE_PROJECT:
    raise ValueError(
        "Set the EE_PROJECT environment variable to your Google Earth Engine Cloud Project before continuing."
    )

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

print("Earth Engine project:", EE_PROJECT)
print("Project root:", PROJECT_ROOT)


In [ ]:
# =====================================================================================
# PROJECT 7 — STAGE 1B
# DOWNLOAD + VALIDATE ENUGU STATE ADMINISTRATIVE BOUNDARY
#
# Source: geoBoundaries gbOpen — Nigeria ADM1
# =====================================================================================

import os
import json
import requests
import pandas as pd
import geopandas as gpd
from pathlib import Path
from datetime import datetime

# -----------------------------------------------------------------------------
# PROJECT PATHS
# -----------------------------------------------------------------------------
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

BOUNDARY_DIR = PROJECT_ROOT / "01_Boundary"
ADMIN_DIR = PROJECT_ROOT / "00_Project_Admin"

BOUNDARY_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# OFFICIAL GEOBOUNDARIES API
# -----------------------------------------------------------------------------
API_URL = "https://www.geoboundaries.org/api/current/gbOpen/NGA/ADM1/"

print("=" * 100)
print("STAGE 1B — ENUGU STATE BOUNDARY ACQUISITION")
print("=" * 100)

print("\nRequesting geoBoundaries metadata...")

response = requests.get(API_URL, timeout=60)
response.raise_for_status()

metadata = response.json()

print("✓ geoBoundaries API connection successful")

# -----------------------------------------------------------------------------
# DISPLAY IMPORTANT SOURCE METADATA
# -----------------------------------------------------------------------------
print("\nSOURCE METADATA")
print("-" * 100)

metadata_fields = [
    "boundaryID",
    "boundaryName",
    "boundaryISO",
    "boundaryYearRepresented",
    "boundaryType",
    "boundarySource",
    "boundaryLicense",
    "gjDownloadURL"
]

for field in metadata_fields:
    print(f"{field}: {metadata.get(field, 'Not supplied')}")

geojson_url = metadata.get("gjDownloadURL")

if not geojson_url:
    raise RuntimeError(
        "geoBoundaries API did not return a GeoJSON download URL."
    )

# -----------------------------------------------------------------------------
# DOWNLOAD NIGERIA ADM1
# -----------------------------------------------------------------------------
print("\nDOWNLOADING NIGERIA ADM1")
print("-" * 100)

nigeria_adm1 = gpd.read_file(geojson_url)

print(f"✓ Features downloaded: {len(nigeria_adm1)}")
print(f"✓ CRS: {nigeria_adm1.crs}")

print("\nAvailable fields:")
print(list(nigeria_adm1.columns))

# -----------------------------------------------------------------------------
# IDENTIFY ADMINISTRATIVE NAME FIELD
# -----------------------------------------------------------------------------
possible_name_fields = [
    "shapeName",
    "NAME_1",
    "name",
    "Name",
    "ADM1_NAME"
]

name_field = None

for col in possible_name_fields:
    if col in nigeria_adm1.columns:
        name_field = col
        break

if name_field is None:
    raise RuntimeError(
        "Could not automatically identify the ADM1 name field."
    )

print(f"\nAdministrative-name field detected: {name_field}")

print("\nNigeria ADM1 names:")
print(sorted(nigeria_adm1[name_field].astype(str).tolist()))

# -----------------------------------------------------------------------------
# EXTRACT ENUGU STATE
# -----------------------------------------------------------------------------
enugu = nigeria_adm1[
    nigeria_adm1[name_field]
    .astype(str)
    .str.strip()
    .str.casefold()
    == "enugu"
].copy()

print("\nENUGU SELECTION")
print("-" * 100)

print(f"Features selected: {len(enugu)}")

if len(enugu) != 1:
    raise RuntimeError(
        f"Expected exactly 1 Enugu ADM1 feature, found {len(enugu)}."
    )

# -----------------------------------------------------------------------------
# GEOMETRY VALIDATION
# -----------------------------------------------------------------------------
print("\nGEOMETRY VALIDATION")
print("-" * 100)

print("Original CRS:", enugu.crs)
print("Geometry type:", enugu.geometry.iloc[0].geom_type)
print("Geometry valid:", bool(enugu.geometry.is_valid.all()))
print("Geometry empty:", bool(enugu.geometry.is_empty.any()))

# Repair only if necessary
if not enugu.geometry.is_valid.all():
    enugu["geometry"] = enugu.geometry.make_valid()
    print("✓ Invalid geometry repaired")

if enugu.crs is None:
    raise RuntimeError("Boundary has no defined CRS.")

# -----------------------------------------------------------------------------
# STANDARDISE TO WGS84
# -----------------------------------------------------------------------------
enugu_wgs84 = enugu.to_crs("EPSG:4326")

# -----------------------------------------------------------------------------
# PROJECT TO UTM ZONE 32N
#
# Enugu State lies approximately around longitude 7–8°E.
# UTM Zone 32N is appropriate for metric area/distance operations.
# -----------------------------------------------------------------------------
enugu_utm = enugu_wgs84.to_crs("EPSG:32632")

# -----------------------------------------------------------------------------
# CALCULATE AREA
# -----------------------------------------------------------------------------
area_km2 = float(enugu_utm.geometry.area.sum() / 1_000_000)

centroid_utm = enugu_utm.geometry.union_all().centroid
centroid_wgs84 = (
    gpd.GeoSeries([centroid_utm], crs="EPSG:32632")
    .to_crs("EPSG:4326")
    .iloc[0]
)

minx, miny, maxx, maxy = enugu_wgs84.total_bounds

print("\nSPATIAL CHECK")
print("-" * 100)

print(f"Area: {area_km2:,.2f} km²")
print(
    f"Centroid: {centroid_wgs84.y:.5f}°N, "
    f"{centroid_wgs84.x:.5f}°E"
)

print(
    f"Bounding box: "
    f"{minx:.5f}, {miny:.5f}, "
    f"{maxx:.5f}, {maxy:.5f}"
)

print("Projected CRS: EPSG:32632 — WGS 84 / UTM zone 32N")

# -----------------------------------------------------------------------------
# SAVE FILES
# -----------------------------------------------------------------------------
wgs84_file = BOUNDARY_DIR / "Enugu_State_Boundary_WGS84.gpkg"
utm_file = BOUNDARY_DIR / "Enugu_State_Boundary_UTM32N.gpkg"
geojson_file = BOUNDARY_DIR / "Enugu_State_Boundary.geojson"

enugu_wgs84.to_file(
    wgs84_file,
    layer="Enugu_State",
    driver="GPKG"
)

enugu_utm.to_file(
    utm_file,
    layer="Enugu_State",
    driver="GPKG"
)

enugu_wgs84.to_file(
    geojson_file,
    driver="GeoJSON"
)

# -----------------------------------------------------------------------------
# SAVE SOURCE METADATA
# -----------------------------------------------------------------------------
metadata_file = ADMIN_DIR / "geoBoundaries_Enugu_ADM1_Metadata.json"

with open(metadata_file, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

# -----------------------------------------------------------------------------
# UPDATE DATA PROVENANCE REGISTER
# -----------------------------------------------------------------------------
provenance_file = ADMIN_DIR / "Data_Provenance_Register.csv"

prov = pd.read_csv(provenance_file)

record = {
    "Dataset": "Enugu State Administrative Boundary",
    "Provider": "geoBoundaries",
    "Source_URL_or_ID": API_URL,
    "Acquisition_Date": datetime.utcnow().strftime("%Y-%m-%d"),
    "Data_Date_or_Period": metadata.get(
        "boundaryYearRepresented", ""
    ),
    "Spatial_Resolution": "Vector administrative boundary",
    "CRS": "EPSG:4326 / EPSG:32632",
    "Licence": metadata.get("boundaryLicense", ""),
    "Project_Use": "Study-area boundary and raster clipping",
    "Validation_Status": "Validated",
    "Notes": (
        f"geoBoundaries gbOpen ADM1; "
        f"area calculated in UTM32N = {area_km2:.2f} km²"
    ),
}

# Prevent duplicate record if cell is rerun
prov = prov[
    prov["Dataset"].astype(str)
    != "Enugu State Administrative Boundary"
]

prov = pd.concat(
    [prov, pd.DataFrame([record])],
    ignore_index=True
)

prov.to_csv(provenance_file, index=False)

# -----------------------------------------------------------------------------
# FINAL VALIDATION
# -----------------------------------------------------------------------------
print("\nOUTPUT FILES")
print("-" * 100)

for f in [wgs84_file, utm_file, geojson_file, metadata_file]:
    print(
        "✓" if f.exists() else "✗",
        f.name
    )

checks = {
    "Exactly one Enugu feature": len(enugu) == 1,
    "Geometry valid": enugu_wgs84.geometry.is_valid.all(),
    "Geometry non-empty": not enugu_wgs84.geometry.is_empty.any(),
    "WGS84 CRS correct": enugu_wgs84.crs.to_epsg() == 4326,
    "UTM CRS correct": enugu_utm.crs.to_epsg() == 32632,
    "Positive area": area_km2 > 0,
}

print("\nVALIDATION SUMMARY")
print("-" * 100)

for check, status in checks.items():
    print(f"{'✓' if status else '✗'} {check}")

if all(checks.values()):
    print("\n✓ STAGE 1B PASSED")
    print("Enugu State boundary successfully acquired and validated.")
else:
    print("\n✗ STAGE 1B FAILED")
    print("Do not continue until the failed checks are resolved.")

print("=" * 100)

In [ ]:
# =====================================================================================
# PROJECT 7 — STAGE 1C
# DOWNLOAD + VALIDATE ENUGU STATE LGA BOUNDARIES
#
# Source: geoBoundaries gbOpen — Nigeria ADM2
# =====================================================================================

import json
import requests
import pandas as pd
import geopandas as gpd
from pathlib import Path
from datetime import datetime, timezone

# -----------------------------------------------------------------------------
# PROJECT PATHS
# -----------------------------------------------------------------------------
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

BOUNDARY_DIR = PROJECT_ROOT / "01_Boundary"
ADMIN_DIR = PROJECT_ROOT / "00_Project_Admin"

STATE_FILE = BOUNDARY_DIR / "Enugu_State_Boundary_WGS84.gpkg"

# -----------------------------------------------------------------------------
# GEOBOUNDARIES ADM2 API
# -----------------------------------------------------------------------------
API_URL = "https://www.geoboundaries.org/api/current/gbOpen/NGA/ADM2/"

print("=" * 105)
print("STAGE 1C — ENUGU STATE LGA BOUNDARY ACQUISITION + VALIDATION")
print("=" * 105)

# -----------------------------------------------------------------------------
# READ OUR ALREADY-VALIDATED ENUGU STATE BOUNDARY
# -----------------------------------------------------------------------------
if not STATE_FILE.exists():
    raise FileNotFoundError(
        "Validated Enugu State boundary from Stage 1B was not found."
    )

enugu_state = gpd.read_file(STATE_FILE).to_crs("EPSG:4326")

print("\n✓ Validated Enugu State boundary loaded")

# -----------------------------------------------------------------------------
# REQUEST ADM2 METADATA
# -----------------------------------------------------------------------------
print("\nRequesting geoBoundaries ADM2 metadata...")

response = requests.get(API_URL, timeout=60)
response.raise_for_status()

metadata = response.json()

print("✓ geoBoundaries ADM2 API connection successful")

print("\nSOURCE METADATA")
print("-" * 105)

for field in [
    "boundaryID",
    "boundaryName",
    "boundaryISO",
    "boundaryYearRepresented",
    "boundaryType",
    "boundarySource",
    "boundaryLicense",
    "gjDownloadURL"
]:
    print(f"{field}: {metadata.get(field, 'Not supplied')}")

geojson_url = metadata.get("gjDownloadURL")

if not geojson_url:
    raise RuntimeError(
        "ADM2 metadata did not contain a GeoJSON download URL."
    )

# -----------------------------------------------------------------------------
# DOWNLOAD ALL NIGERIAN ADM2 FEATURES
# -----------------------------------------------------------------------------
print("\nDOWNLOADING NIGERIA ADM2")
print("-" * 105)

nigeria_adm2 = gpd.read_file(geojson_url).to_crs("EPSG:4326")

print(f"✓ Total ADM2 features downloaded: {len(nigeria_adm2)}")
print(f"✓ CRS: {nigeria_adm2.crs}")
print("Available fields:", list(nigeria_adm2.columns))

# -----------------------------------------------------------------------------
# DETECT NAME FIELD
# -----------------------------------------------------------------------------
candidate_fields = [
    "shapeName",
    "NAME_2",
    "name",
    "Name",
    "ADM2_NAME"
]

name_field = next(
    (f for f in candidate_fields if f in nigeria_adm2.columns),
    None
)

if name_field is None:
    raise RuntimeError(
        "Could not identify ADM2 administrative-name field."
    )

print(f"\nAdministrative-name field detected: {name_field}")

# -----------------------------------------------------------------------------
# SPATIALLY IDENTIFY LGAs BELONGING TO ENUGU STATE
#
# Use representative points instead of simple intersection because neighbouring
# LGAs may share state boundary lines.
# -----------------------------------------------------------------------------
adm2_proj = nigeria_adm2.to_crs("EPSG:32632")
state_proj = enugu_state.to_crs("EPSG:32632")

representative_points = adm2_proj.geometry.representative_point()

inside_enugu = representative_points.within(
    state_proj.geometry.union_all()
)

enugu_lgas = nigeria_adm2.loc[inside_enugu.values].copy()

print("\nENUGU LGA EXTRACTION")
print("-" * 105)

print(f"Features selected: {len(enugu_lgas)}")

print("\nSelected LGA names:")
for name in sorted(enugu_lgas[name_field].astype(str)):
    print(" -", name)

# -----------------------------------------------------------------------------
# BASIC GEOMETRY VALIDATION
# -----------------------------------------------------------------------------
print("\nGEOMETRY VALIDATION")
print("-" * 105)

print("All geometries valid:",
      bool(enugu_lgas.geometry.is_valid.all()))

print("Any empty geometries:",
      bool(enugu_lgas.geometry.is_empty.any()))

print("Geometry types:",
      sorted(enugu_lgas.geometry.geom_type.unique().tolist()))

if not enugu_lgas.geometry.is_valid.all():
    enugu_lgas["geometry"] = enugu_lgas.geometry.make_valid()
    print("✓ Invalid geometries repaired")

# -----------------------------------------------------------------------------
# STANDARDISE NAME FIELD
# -----------------------------------------------------------------------------
enugu_lgas = enugu_lgas.rename(
    columns={name_field: "LGA_Name"}
)

enugu_lgas["LGA_Name"] = (
    enugu_lgas["LGA_Name"]
    .astype(str)
    .str.strip()
)

# -----------------------------------------------------------------------------
# PROJECT TO UTM 32N
# -----------------------------------------------------------------------------
enugu_lgas_utm = enugu_lgas.to_crs("EPSG:32632")

enugu_lgas_utm["Area_km2"] = (
    enugu_lgas_utm.geometry.area / 1_000_000
)

total_lga_area = enugu_lgas_utm["Area_km2"].sum()

state_utm = enugu_state.to_crs("EPSG:32632")
state_area = state_utm.geometry.area.sum() / 1_000_000

area_difference = abs(total_lga_area - state_area)
area_difference_pct = (
    area_difference / state_area
) * 100

# -----------------------------------------------------------------------------
# CHECK FOR OVERLAPPING LGAs
# -----------------------------------------------------------------------------
overlap_count = 0

for i in range(len(enugu_lgas_utm)):
    geom_i = enugu_lgas_utm.geometry.iloc[i]

    for j in range(i + 1, len(enugu_lgas_utm)):
        geom_j = enugu_lgas_utm.geometry.iloc[j]

        intersection = geom_i.intersection(geom_j)

        # Count only real polygon overlap — not shared boundaries
        if (
            not intersection.is_empty
            and intersection.area > 1
        ):
            overlap_count += 1

# -----------------------------------------------------------------------------
# DISPLAY AREA STATISTICS
# -----------------------------------------------------------------------------
print("\nLGA AREA STATISTICS")
print("-" * 105)

area_table = (
    enugu_lgas_utm[
        ["LGA_Name", "Area_km2"]
    ]
    .sort_values("LGA_Name")
    .reset_index(drop=True)
)

print(
    area_table.to_string(
        index=False,
        formatters={
            "Area_km2": lambda x: f"{x:,.2f}"
        }
    )
)

print("\nAREA CONSISTENCY CHECK")
print("-" * 105)

print(f"State boundary area: {state_area:,.2f} km²")
print(f"Sum of LGA areas:    {total_lga_area:,.2f} km²")
print(f"Absolute difference: {area_difference:,.2f} km²")
print(f"Difference:          {area_difference_pct:.4f}%")
print(f"Polygon overlaps detected: {overlap_count}")

# -----------------------------------------------------------------------------
# SAVE OUTPUTS
# -----------------------------------------------------------------------------
wgs84_file = BOUNDARY_DIR / "Enugu_LGAs_WGS84.gpkg"
utm_file = BOUNDARY_DIR / "Enugu_LGAs_UTM32N.gpkg"
csv_file = BOUNDARY_DIR / "Enugu_LGA_Area_Validation.csv"
metadata_file = ADMIN_DIR / "geoBoundaries_Enugu_ADM2_Metadata.json"

enugu_lgas.to_file(
    wgs84_file,
    layer="Enugu_LGAs",
    driver="GPKG"
)

enugu_lgas_utm.to_file(
    utm_file,
    layer="Enugu_LGAs",
    driver="GPKG"
)

area_table.to_csv(
    csv_file,
    index=False
)

with open(metadata_file, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

# -----------------------------------------------------------------------------
# UPDATE PROVENANCE REGISTER
# -----------------------------------------------------------------------------
provenance_file = ADMIN_DIR / "Data_Provenance_Register.csv"

prov = pd.read_csv(provenance_file)

record = {
    "Dataset": "Enugu State LGA Administrative Boundaries",
    "Provider": "geoBoundaries",
    "Source_URL_or_ID": API_URL,
    "Acquisition_Date": datetime.now(
        timezone.utc
    ).strftime("%Y-%m-%d"),
    "Data_Date_or_Period": metadata.get(
        "boundaryYearRepresented", ""
    ),
    "Spatial_Resolution": "Vector ADM2 administrative boundaries",
    "CRS": "EPSG:4326 / EPSG:32632",
    "Licence": metadata.get(
        "boundaryLicense", ""
    ),
    "Project_Use": (
        "LGA-level suitability statistics and planning summaries"
    ),
    "Validation_Status": "Validated",
    "Notes": (
        f"{len(enugu_lgas)} LGAs extracted spatially; "
        f"combined area = {total_lga_area:.2f} km²; "
        f"difference from ADM1 = {area_difference_pct:.4f}%"
    ),
}

prov = prov[
    prov["Dataset"].astype(str)
    != "Enugu State LGA Administrative Boundaries"
]

prov = pd.concat(
    [prov, pd.DataFrame([record])],
    ignore_index=True
)

prov.to_csv(
    provenance_file,
    index=False
)

# -----------------------------------------------------------------------------
# FINAL VALIDATION
# -----------------------------------------------------------------------------
checks = {
    "ADM2 features selected": len(enugu_lgas) > 0,
    "Expected 17 Enugu LGAs": len(enugu_lgas) == 17,
    "All LGA names unique":
        enugu_lgas["LGA_Name"].nunique() == len(enugu_lgas),

    "All geometries valid":
        enugu_lgas.geometry.is_valid.all(),

    "No empty geometries":
        not enugu_lgas.geometry.is_empty.any(),

    "Correct WGS84 CRS":
        enugu_lgas.crs.to_epsg() == 4326,

    "Correct projected CRS":
        enugu_lgas_utm.crs.to_epsg() == 32632,

    "No polygon overlaps":
        overlap_count == 0,

    "ADM1/ADM2 area difference < 2%":
        area_difference_pct < 2.0,
}

print("\nVALIDATION SUMMARY")
print("-" * 105)

for check, status in checks.items():
    print(
        f"{'✓' if status else '✗'} {check}"
    )

print("\nOUTPUT FILES")
print("-" * 105)

for path in [
    wgs84_file,
    utm_file,
    csv_file,
    metadata_file
]:
    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{path.name}"
    )

if all(checks.values()):
    print("\n✓ STAGE 1C PASSED")
    print(
        "Enugu State LGA framework successfully "
        "acquired and validated."
    )
else:
    print("\n⚠ STAGE 1C REQUIRES REVIEW")
    print(
        "Send the complete output before proceeding."
    )

print("=" * 105)

In [ ]:
import os
# =====================================================================================
# PROJECT 7 — EARTH ENGINE INITIALIZATION
# =====================================================================================

import ee

EE_PROJECT = os.environ.get("EE_PROJECT", "").strip()
if not EE_PROJECT:
    raise ValueError("Set EE_PROJECT before running this stage.")

print("=" * 90)
print("EARTH ENGINE AUTHENTICATION + INITIALIZATION")
print("=" * 90)

# Authenticate
ee.Authenticate(auth_mode="colab")

# Initialize using the verified project visible in Earth Engine
ee.Initialize(project=EE_PROJECT)

# Server-side test
test = ee.Number(10).multiply(5).getInfo()

print("\nVALIDATION")
print("-" * 90)
print("Cloud Project:", EE_PROJECT)
print("Earth Engine test result:", test)

if test == 50:
    print("\n✓ EARTH ENGINE INITIALIZATION PASSED")
    print("Earth Engine is ready for Project 7.")
else:
    print("\n✗ EARTH ENGINE INITIALIZATION FAILED")

print("=" * 90)

In [ ]:
import os
# =====================================================================================
# PROJECT 7 — STAGE 2A FIX
# COPERNICUS DEM GLO-30 2024_1 + CORRECT SLOPE + DRIVE BATCH EXPORT
# =====================================================================================

import ee
import geemap
import geopandas as gpd
from pathlib import Path

# -----------------------------------------------------------------------------
# SETTINGS
# -----------------------------------------------------------------------------
EE_PROJECT = os.environ.get("EE_PROJECT", "").strip()
if not EE_PROJECT:
    raise ValueError("Set EE_PROJECT before running this stage.")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

BOUNDARY_FILE = (
    PROJECT_ROOT /
    "01_Boundary" /
    "Enugu_State_Boundary_WGS84.gpkg"
)

# Earth Engine exports will appear in this Google Drive folder
DRIVE_FOLDER = "Project_7_Enugu_Terrain"

DEM_ID = "COPERNICUS/DEM/GLO30_2024_1"

# -----------------------------------------------------------------------------
# INITIALIZE
# -----------------------------------------------------------------------------
print("=" * 105)
print("STAGE 2A FIX — COPERNICUS DEM 2024_1 + BATCH EXPORT")
print("=" * 105)

ee.Initialize(project=EE_PROJECT)

print("\n✓ Earth Engine initialized")
print("Cloud Project:", EE_PROJECT)

# -----------------------------------------------------------------------------
# STUDY AREA
# -----------------------------------------------------------------------------
enugu = gpd.read_file(BOUNDARY_FILE).to_crs("EPSG:4326")

if len(enugu) != 1:
    raise RuntimeError(
        f"Expected 1 Enugu State feature, found {len(enugu)}"
    )

enugu_fc = geemap.geopandas_to_ee(enugu)
enugu_geom = enugu_fc.geometry()

print("✓ Enugu State boundary loaded")

# -----------------------------------------------------------------------------
# CURRENT COPERNICUS DEM
# -----------------------------------------------------------------------------
collection = (
    ee.ImageCollection(DEM_ID)
    .filterBounds(enugu_geom)
)

tile_count = collection.size().getInfo()

if tile_count == 0:
    raise RuntimeError("No DEM tiles found over Enugu State.")

# IMPORTANT:
# Copernicus GLO30 is an ImageCollection.
# Preserve a native source projection before terrain analysis.
native_projection = (
    collection.first()
    .select("DEM")
    .projection()
)

dem = (
    collection
    .select("DEM")
    .mosaic()
    .setDefaultProjection(native_projection)
    .clip(enugu_geom)
    .rename("Elevation_m")
)

# -----------------------------------------------------------------------------
# SLOPE — CALCULATED AFTER SETTING DEM PROJECTION
# -----------------------------------------------------------------------------
slope = (
    ee.Terrain
    .slope(dem)
    .rename("Slope_deg")
    .clip(enugu_geom)
)

print("\nDEM SOURCE")
print("-" * 105)
print("Dataset: Copernicus DEM GLO-30 2024_1")
print("Earth Engine ID:", DEM_ID)
print("Tiles intersecting Enugu:", tile_count)
print("Native projection:", native_projection.getInfo())

# -----------------------------------------------------------------------------
# SERVER-SIDE STATISTICS
# -----------------------------------------------------------------------------
reducer = (
    ee.Reducer.minMax()
    .combine(
        ee.Reducer.mean(),
        sharedInputs=True
    )
    .combine(
        ee.Reducer.stdDev(),
        sharedInputs=True
    )
)

dem_stats = dem.reduceRegion(
    reducer=reducer,
    geometry=enugu_geom,
    scale=30,
    maxPixels=1e9,
    tileScale=4
).getInfo()

slope_stats = slope.reduceRegion(
    reducer=reducer,
    geometry=enugu_geom,
    scale=30,
    maxPixels=1e9,
    tileScale=4
).getInfo()

print("\nEARTH ENGINE STATISTICS")
print("-" * 105)

print("\nElevation:")
for key, value in dem_stats.items():
    print(f"  {key}: {value}")

print("\nSlope:")
for key, value in slope_stats.items():
    print(f"  {key}: {value}")

# -----------------------------------------------------------------------------
# CHECK STATISTICS BEFORE EXPORT
# -----------------------------------------------------------------------------
dem_values = [
    v for v in dem_stats.values()
    if v is not None
]

slope_values = [
    v for v in slope_stats.values()
    if v is not None
]

if len(dem_values) != 4:
    raise RuntimeError(
        "Elevation statistics are incomplete."
    )

if len(slope_values) != 4:
    raise RuntimeError(
        "Slope statistics are incomplete. Do not export."
    )

# -----------------------------------------------------------------------------
# CREATE EARTH ENGINE BATCH EXPORTS TO GOOGLE DRIVE
# -----------------------------------------------------------------------------
print("\nSTARTING GOOGLE DRIVE EXPORT TASKS")
print("-" * 105)

dem_task = ee.batch.Export.image.toDrive(
    image=dem,
    description="Project7_Enugu_Elevation_30m",
    folder=DRIVE_FOLDER,
    fileNamePrefix="Enugu_Elevation_30m",
    region=enugu_geom,
    scale=30,
    crs="EPSG:32632",
    maxPixels=1e10,
    fileFormat="GeoTIFF",
    formatOptions={
        "cloudOptimized": True
    }
)

slope_task = ee.batch.Export.image.toDrive(
    image=slope,
    description="Project7_Enugu_Slope_30m",
    folder=DRIVE_FOLDER,
    fileNamePrefix="Enugu_Slope_Degrees_30m",
    region=enugu_geom,
    scale=30,
    crs="EPSG:32632",
    maxPixels=1e10,
    fileFormat="GeoTIFF",
    formatOptions={
        "cloudOptimized": True
    }
)

dem_task.start()
slope_task.start()

# -----------------------------------------------------------------------------
# TASK STATUS
# -----------------------------------------------------------------------------
dem_status = dem_task.status()
slope_status = slope_task.status()

print("Elevation task:")
print("  ID:", dem_status.get("id"))
print("  State:", dem_status.get("state"))

print("\nSlope task:")
print("  ID:", slope_status.get("id"))
print("  State:", slope_status.get("state"))

print("\n" + "-" * 105)

if (
    dem_status.get("state") in ["READY", "RUNNING"]
    and
    slope_status.get("state") in ["READY", "RUNNING"]
):
    print("✓ STAGE 2A EXPORT TASKS STARTED SUCCESSFULLY")
    print()
    print("Google Drive destination:")
    print(f"MyDrive/{DRIVE_FOLDER}")
    print()
    print("Do NOT rerun this cell.")
else:
    print("⚠ One or more export tasks did not start correctly.")

print("=" * 105)

In [ ]:
# =====================================================================================
# PROJECT 7 — STAGE 2A FINAL
# CONSOLIDATE ELEVATION + SLOPE AND PERFORM FINAL RASTER VALIDATION
# =====================================================================================

import shutil
import rasterio
import numpy as np
import pandas as pd
from pathlib import Path

# -----------------------------------------------------------------------------
# PROJECT PATHS
# -----------------------------------------------------------------------------
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

TERRAIN_DIR = PROJECT_ROOT / "02_Terrain"
TERRAIN_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_ELEVATION = Path(
    "/content/drive/MyDrive/"
    "Project_7_Enugu_Terrain/"
    "Enugu_Elevation_30m.tif"
)

SOURCE_SLOPE = Path(
    "/content/drive/MyDrive/"
    "Project_7_Enugu_Terrain (1)/"
    "Enugu_Slope_Degrees_30m.tif"
)

TARGET_ELEVATION = TERRAIN_DIR / "Enugu_Elevation_30m.tif"
TARGET_SLOPE = TERRAIN_DIR / "Enugu_Slope_Degrees_30m.tif"

VALIDATION_FILE = TERRAIN_DIR / "Terrain_Validation_Summary.csv"

print("=" * 105)
print("STAGE 2A — FINAL TERRAIN CONSOLIDATION + VALIDATION")
print("=" * 105)

# -----------------------------------------------------------------------------
# VERIFY SOURCE FILES
# -----------------------------------------------------------------------------
print("\nSOURCE FILE CHECK")
print("-" * 105)

for path in [SOURCE_ELEVATION, SOURCE_SLOPE]:
    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{path}"
    )

if not SOURCE_ELEVATION.exists():
    raise FileNotFoundError(
        f"Elevation source file not found: {SOURCE_ELEVATION}"
    )

if not SOURCE_SLOPE.exists():
    raise FileNotFoundError(
        f"Slope source file not found: {SOURCE_SLOPE}"
    )

# -----------------------------------------------------------------------------
# COPY TO PERMANENT PROJECT DIRECTORY
# -----------------------------------------------------------------------------
shutil.copy2(
    SOURCE_ELEVATION,
    TARGET_ELEVATION
)

shutil.copy2(
    SOURCE_SLOPE,
    TARGET_SLOPE
)

print("\nFILES CONSOLIDATED")
print("-" * 105)

print("✓", TARGET_ELEVATION)
print("✓", TARGET_SLOPE)

# -----------------------------------------------------------------------------
# INSPECT RASTERS
# -----------------------------------------------------------------------------
def inspect_raster(path):

    with rasterio.open(path) as src:

        arr = src.read(1, masked=True)

        valid = arr.compressed()

        if valid.size == 0:
            raise RuntimeError(
                f"{path.name} contains no valid pixels."
            )

        return {
            "Filename": path.name,
            "CRS": str(src.crs),
            "EPSG": src.crs.to_epsg()
                if src.crs else None,
            "Width": src.width,
            "Height": src.height,
            "Resolution_X_m": abs(src.transform.a),
            "Resolution_Y_m": abs(src.transform.e),
            "Nodata": src.nodata,
            "Valid_Pixels": int(valid.size),
            "Min": float(valid.min()),
            "Max": float(valid.max()),
            "Mean": float(valid.mean()),
            "StdDev": float(valid.std()),
            "Bounds_Left": float(src.bounds.left),
            "Bounds_Bottom": float(src.bounds.bottom),
            "Bounds_Right": float(src.bounds.right),
            "Bounds_Top": float(src.bounds.top),
        }

elev = inspect_raster(TARGET_ELEVATION)
slope = inspect_raster(TARGET_SLOPE)

# -----------------------------------------------------------------------------
# DISPLAY RASTER DETAILS
# -----------------------------------------------------------------------------
print("\nLOCAL RASTER STATISTICS")
print("-" * 105)

for result in [elev, slope]:

    print(f"\n{result['Filename']}")

    print("CRS:", result["CRS"])
    print("EPSG:", result["EPSG"])

    print(
        f"Dimensions: "
        f"{result['Width']} × {result['Height']}"
    )

    print(
        f"Resolution: "
        f"{result['Resolution_X_m']:.3f} × "
        f"{result['Resolution_Y_m']:.3f} m"
    )

    print(
        f"Valid pixels: "
        f"{result['Valid_Pixels']:,}"
    )

    print(f"Minimum: {result['Min']:.3f}")
    print(f"Maximum: {result['Max']:.3f}")
    print(f"Mean: {result['Mean']:.3f}")
    print(f"Std Dev: {result['StdDev']:.3f}")

# -----------------------------------------------------------------------------
# ALIGNMENT CHECKS
# -----------------------------------------------------------------------------
same_crs = (
    elev["EPSG"] == slope["EPSG"] == 32632
)

same_dimensions = (
    elev["Width"] == slope["Width"]
    and
    elev["Height"] == slope["Height"]
)

same_resolution = (
    abs(
        elev["Resolution_X_m"]
        - slope["Resolution_X_m"]
    ) < 0.001
    and
    abs(
        elev["Resolution_Y_m"]
        - slope["Resolution_Y_m"]
    ) < 0.001
)

bounds_elev = (
    elev["Bounds_Left"],
    elev["Bounds_Bottom"],
    elev["Bounds_Right"],
    elev["Bounds_Top"]
)

bounds_slope = (
    slope["Bounds_Left"],
    slope["Bounds_Bottom"],
    slope["Bounds_Right"],
    slope["Bounds_Top"]
)

same_bounds = all(
    abs(a - b) < 0.01
    for a, b in zip(
        bounds_elev,
        bounds_slope
    )
)

same_valid_pixels = (
    elev["Valid_Pixels"]
    == slope["Valid_Pixels"]
)

# -----------------------------------------------------------------------------
# RANGE VALIDATION
# -----------------------------------------------------------------------------
elevation_range_ok = (
    0 <= elev["Min"]
    < elev["Max"]
    < 3000
)

slope_range_ok = (
    0 <= slope["Min"]
    <= slope["Max"]
    <= 90
)

# -----------------------------------------------------------------------------
# COMPARE AGAINST PREVIOUS EARTH ENGINE STATISTICS
# -----------------------------------------------------------------------------
EE_ELEV_MEAN = 199.62979117869662
EE_SLOPE_MEAN = 4.3328247250836185

elev_mean_diff = abs(
    elev["Mean"] - EE_ELEV_MEAN
)

slope_mean_diff = abs(
    slope["Mean"] - EE_SLOPE_MEAN
)

# -----------------------------------------------------------------------------
# SAVE VALIDATION SUMMARY
# -----------------------------------------------------------------------------
validation_df = pd.DataFrame([
    {
        "Layer": "Elevation",
        **elev,
        "EE_Mean": EE_ELEV_MEAN,
        "Local_EE_Mean_Difference":
            elev_mean_diff
    },
    {
        "Layer": "Slope",
        **slope,
        "EE_Mean": EE_SLOPE_MEAN,
        "Local_EE_Mean_Difference":
            slope_mean_diff
    }
])

validation_df.to_csv(
    VALIDATION_FILE,
    index=False
)

# -----------------------------------------------------------------------------
# FINAL VALIDATION
# -----------------------------------------------------------------------------
checks = {
    "Elevation source located":
        SOURCE_ELEVATION.exists(),

    "Slope source located":
        SOURCE_SLOPE.exists(),

    "Elevation stored in project":
        TARGET_ELEVATION.exists(),

    "Slope stored in project":
        TARGET_SLOPE.exists(),

    "Both rasters EPSG:32632":
        same_crs,

    "Raster dimensions identical":
        same_dimensions,

    "Raster resolution identical":
        same_resolution,

    "Raster bounds identical":
        same_bounds,

    "Valid-pixel counts identical":
        same_valid_pixels,

    "Elevation range plausible":
        elevation_range_ok,

    "Slope range plausible":
        slope_range_ok,

    "Elevation local/EE mean difference < 5 m":
        elev_mean_diff < 5,

    "Slope local/EE mean difference < 2°":
        slope_mean_diff < 2,
}

print("\nVALIDATION SUMMARY")
print("-" * 105)

for check, status in checks.items():
    print(
        f"{'✓' if status else '✗'} {check}"
    )

print("\nEARTH ENGINE VS LOCAL EXPORT")
print("-" * 105)

print(
    f"Elevation EE mean:    {EE_ELEV_MEAN:.4f} m"
)

print(
    f"Elevation local mean: {elev['Mean']:.4f} m"
)

print(
    f"Difference:           {elev_mean_diff:.4f} m"
)

print()

print(
    f"Slope EE mean:        {EE_SLOPE_MEAN:.4f}°"
)

print(
    f"Slope local mean:     {slope['Mean']:.4f}°"
)

print(
    f"Difference:           {slope_mean_diff:.4f}°"
)

print("\nOUTPUT FILES")
print("-" * 105)

for path in [
    TARGET_ELEVATION,
    TARGET_SLOPE,
    VALIDATION_FILE
]:
    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{path.name}"
    )

if all(checks.values()):

    print("\n✓ STAGE 2A FULLY PASSED")

    print(
        "Elevation and slope predictors are "
        "permanently stored, aligned and validated."
    )

else:

    print("\n⚠ STAGE 2A REQUIRES REVIEW")

print("=" * 105)

In [ ]:
# =====================================================================================
# PROJECT 7 — STAGE 2A CORRECTED FINAL VALIDATION
# HANDLE EARTH ENGINE NaN PIXELS CORRECTLY
# =====================================================================================

import rasterio
import numpy as np
import pandas as pd
from pathlib import Path

# -----------------------------------------------------------------------------
# PATHS
# -----------------------------------------------------------------------------
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

TERRAIN_DIR = PROJECT_ROOT / "02_Terrain"

ELEVATION_FILE = TERRAIN_DIR / "Enugu_Elevation_30m.tif"
SLOPE_FILE = TERRAIN_DIR / "Enugu_Slope_Degrees_30m.tif"

VALIDATION_FILE = TERRAIN_DIR / "Terrain_Validation_Summary.csv"

STATE_AREA_KM2 = 7625.21

# Previous Earth Engine statistics
EE_ELEV_MIN = 20.16160774230957
EE_ELEV_MAX = 594.212890625
EE_ELEV_MEAN = 199.62979117869662

EE_SLOPE_MIN = 0.0
EE_SLOPE_MAX = 49.01313400268555
EE_SLOPE_MEAN = 4.3328247250836185

print("=" * 105)
print("STAGE 2A — CORRECTED TERRAIN RASTER VALIDATION")
print("=" * 105)

# -----------------------------------------------------------------------------
# CORRECT RASTER INSPECTION
# -----------------------------------------------------------------------------
def inspect_raster_correctly(path):

    if not path.exists():
        raise FileNotFoundError(path)

    with rasterio.open(path) as src:

        data = src.read(1)

        # -------------------------------------------------------------
        # Correct validity mask:
        # 1. Pixel must be finite
        # 2. If explicit nodata exists, exclude it as well
        # -------------------------------------------------------------
        valid_mask = np.isfinite(data)

        if src.nodata is not None and np.isfinite(src.nodata):
            valid_mask &= data != src.nodata

        valid = data[valid_mask]

        if valid.size == 0:
            raise RuntimeError(
                f"{path.name} contains no finite valid pixels."
            )

        total_pixels = data.size
        invalid_pixels = total_pixels - valid.size

        pixel_area_m2 = (
            abs(src.transform.a)
            *
            abs(src.transform.e)
        )

        raster_valid_area_km2 = (
            valid.size * pixel_area_m2
            / 1_000_000
        )

        return {
            "Filename": path.name,
            "CRS": str(src.crs),
            "EPSG": src.crs.to_epsg(),
            "Width": src.width,
            "Height": src.height,
            "Resolution_X_m": abs(src.transform.a),
            "Resolution_Y_m": abs(src.transform.e),
            "Declared_Nodata": src.nodata,
            "Total_Pixels": int(total_pixels),
            "Valid_Finite_Pixels": int(valid.size),
            "Invalid_or_NaN_Pixels": int(invalid_pixels),
            "Valid_Percent": float(
                valid.size / total_pixels * 100
            ),
            "Min": float(np.min(valid)),
            "Max": float(np.max(valid)),
            "Mean": float(np.mean(valid)),
            "StdDev": float(np.std(valid)),
            "Valid_Area_km2": float(
                raster_valid_area_km2
            ),
            "Bounds": tuple(src.bounds)
        }

# -----------------------------------------------------------------------------
# INSPECT BOTH RASTERS
# -----------------------------------------------------------------------------
elev = inspect_raster_correctly(ELEVATION_FILE)
slope = inspect_raster_correctly(SLOPE_FILE)

# -----------------------------------------------------------------------------
# DISPLAY RESULTS
# -----------------------------------------------------------------------------
print("\nCORRECTED LOCAL RASTER STATISTICS")
print("-" * 105)

for r in [elev, slope]:

    print(f"\n{r['Filename']}")
    print("CRS:", r["CRS"])
    print("EPSG:", r["EPSG"])

    print(
        f"Dimensions: "
        f"{r['Width']} × {r['Height']}"
    )

    print(
        f"Resolution: "
        f"{r['Resolution_X_m']:.3f} × "
        f"{r['Resolution_Y_m']:.3f} m"
    )

    print(
        f"Declared NoData: "
        f"{r['Declared_Nodata']}"
    )

    print(
        f"Total pixels: "
        f"{r['Total_Pixels']:,}"
    )

    print(
        f"Finite valid pixels: "
        f"{r['Valid_Finite_Pixels']:,}"
    )

    print(
        f"NaN/invalid pixels: "
        f"{r['Invalid_or_NaN_Pixels']:,}"
    )

    print(
        f"Valid coverage: "
        f"{r['Valid_Percent']:.2f}%"
    )

    print(
        f"Valid-pixel area: "
        f"{r['Valid_Area_km2']:,.2f} km²"
    )

    print(f"Minimum: {r['Min']:.3f}")
    print(f"Maximum: {r['Max']:.3f}")
    print(f"Mean: {r['Mean']:.3f}")
    print(f"Std Dev: {r['StdDev']:.3f}")

# -----------------------------------------------------------------------------
# ALIGNMENT CHECK
# -----------------------------------------------------------------------------
same_crs = (
    elev["EPSG"] == slope["EPSG"] == 32632
)

same_dimensions = (
    elev["Width"] == slope["Width"]
    and
    elev["Height"] == slope["Height"]
)

same_resolution = (
    np.isclose(
        elev["Resolution_X_m"],
        slope["Resolution_X_m"]
    )
    and
    np.isclose(
        elev["Resolution_Y_m"],
        slope["Resolution_Y_m"]
    )
)

same_bounds = np.allclose(
    elev["Bounds"],
    slope["Bounds"],
    atol=0.01
)

# Small boundary/rasterization differences are expected.
elev_area_diff_pct = (
    abs(
        elev["Valid_Area_km2"]
        - STATE_AREA_KM2
    )
    / STATE_AREA_KM2
    * 100
)

slope_area_diff_pct = (
    abs(
        slope["Valid_Area_km2"]
        - STATE_AREA_KM2
    )
    / STATE_AREA_KM2
    * 100
)

# -----------------------------------------------------------------------------
# EARTH ENGINE VS LOCAL STATISTICS
# -----------------------------------------------------------------------------
elev_mean_diff = abs(
    elev["Mean"] - EE_ELEV_MEAN
)

slope_mean_diff = abs(
    slope["Mean"] - EE_SLOPE_MEAN
)

elev_min_diff = abs(
    elev["Min"] - EE_ELEV_MIN
)

elev_max_diff = abs(
    elev["Max"] - EE_ELEV_MAX
)

slope_min_diff = abs(
    slope["Min"] - EE_SLOPE_MIN
)

slope_max_diff = abs(
    slope["Max"] - EE_SLOPE_MAX
)

# -----------------------------------------------------------------------------
# VALIDATION CHECKS
# -----------------------------------------------------------------------------
checks = {
    "Elevation file exists":
        ELEVATION_FILE.exists(),

    "Slope file exists":
        SLOPE_FILE.exists(),

    "Both rasters EPSG:32632":
        same_crs,

    "Raster dimensions identical":
        same_dimensions,

    "Raster resolutions identical":
        same_resolution,

    "Raster bounds identical":
        same_bounds,

    "Elevation contains finite pixels":
        elev["Valid_Finite_Pixels"] > 0,

    "Slope contains finite pixels":
        slope["Valid_Finite_Pixels"] > 0,

    "Elevation range plausible":
        (
            0 <= elev["Min"]
            < elev["Max"]
            < 3000
        ),

    "Slope range plausible":
        (
            0 <= slope["Min"]
            <= slope["Max"]
            <= 90
        ),

    "Elevation raster area within 2% of state area":
        elev_area_diff_pct < 2,

    "Slope raster area within 2% of state area":
        slope_area_diff_pct < 2,

    "Elevation mean agrees with EE":
        elev_mean_diff < 5,

    "Slope mean agrees with EE":
        slope_mean_diff < 2,
}

# -----------------------------------------------------------------------------
# SAVE CORRECTED VALIDATION TABLE
# -----------------------------------------------------------------------------
validation_df = pd.DataFrame([
    {
        "Layer": "Elevation",
        **elev,
        "State_Area_km2": STATE_AREA_KM2,
        "Area_Difference_pct":
            elev_area_diff_pct,
        "EE_Min": EE_ELEV_MIN,
        "EE_Max": EE_ELEV_MAX,
        "EE_Mean": EE_ELEV_MEAN,
        "Local_EE_Mean_Difference":
            elev_mean_diff,
    },
    {
        "Layer": "Slope",
        **slope,
        "State_Area_km2": STATE_AREA_KM2,
        "Area_Difference_pct":
            slope_area_diff_pct,
        "EE_Min": EE_SLOPE_MIN,
        "EE_Max": EE_SLOPE_MAX,
        "EE_Mean": EE_SLOPE_MEAN,
        "Local_EE_Mean_Difference":
            slope_mean_diff,
    }
])

validation_df.to_csv(
    VALIDATION_FILE,
    index=False
)

# -----------------------------------------------------------------------------
# REPORT
# -----------------------------------------------------------------------------
print("\nEARTH ENGINE vs LOCAL EXPORT")
print("-" * 105)

print(
    f"Elevation mean: "
    f"EE={EE_ELEV_MEAN:.4f} m | "
    f"Local={elev['Mean']:.4f} m | "
    f"Difference={elev_mean_diff:.4f} m"
)

print(
    f"Slope mean:     "
    f"EE={EE_SLOPE_MEAN:.4f}° | "
    f"Local={slope['Mean']:.4f}° | "
    f"Difference={slope_mean_diff:.4f}°"
)

print("\nAREA CONSISTENCY")
print("-" * 105)

print(
    f"Vector Enugu area: "
    f"{STATE_AREA_KM2:,.2f} km²"
)

print(
    f"Elevation valid-pixel area: "
    f"{elev['Valid_Area_km2']:,.2f} km² "
    f"({elev_area_diff_pct:.3f}% difference)"
)

print(
    f"Slope valid-pixel area: "
    f"{slope['Valid_Area_km2']:,.2f} km² "
    f"({slope_area_diff_pct:.3f}% difference)"
)

print("\nVALIDATION SUMMARY")
print("-" * 105)

for label, status in checks.items():
    print(
        f"{'✓' if status else '✗'} "
        f"{label}"
    )

print("\nOUTPUT")
print("-" * 105)
print("✓", VALIDATION_FILE.name)

if all(checks.values()):

    print("\n✓ STAGE 2A FULLY PASSED")

    print(
        "Elevation and slope predictors are correctly "
        "exported, aligned, clipped and validated."
    )

else:

    print("\n⚠ STAGE 2A REQUIRES REVIEW")

print("=" * 105)

In [ ]:
# =====================================================================================
# PROJECT 7 — STAGE 2B
# TERRAIN / SLOPE DISTRIBUTION DIAGNOSTICS
#
# Purpose:
# Quantify the real distribution of slope conditions across Enugu State.
#
# NOTE:
# These are descriptive terrain classes — NOT final suitability scores.
# The continuous slope raster will later be used directly by the ML model.
# =====================================================================================

import rasterio
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# -----------------------------------------------------------------------------
# PATHS
# -----------------------------------------------------------------------------
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

TERRAIN_DIR = PROJECT_ROOT / "02_Terrain"
TABLE_DIR = PROJECT_ROOT / "16_Tables"
CHART_DIR = PROJECT_ROOT / "15_Charts"

SLOPE_FILE = TERRAIN_DIR / "Enugu_Slope_Degrees_30m.tif"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
CHART_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_CSV = TABLE_DIR / "Enugu_Slope_Distribution.csv"
OUTPUT_CHART = CHART_DIR / "Enugu_Slope_Distribution.png"

print("=" * 105)
print("STAGE 2B — ENUGU TERRAIN / SLOPE DISTRIBUTION")
print("=" * 105)

# -----------------------------------------------------------------------------
# LOAD SLOPE RASTER
# -----------------------------------------------------------------------------
if not SLOPE_FILE.exists():
    raise FileNotFoundError(
        f"Slope raster not found: {SLOPE_FILE}"
    )

with rasterio.open(SLOPE_FILE) as src:

    slope = src.read(1)

    transform = src.transform
    crs = src.crs

    valid_mask = np.isfinite(slope)

    if src.nodata is not None and np.isfinite(src.nodata):
        valid_mask &= slope != src.nodata

    valid = slope[valid_mask].astype("float64")

    pixel_area_km2 = (
        abs(transform.a) *
        abs(transform.e)
    ) / 1_000_000

print("\nRASTER INPUT")
print("-" * 105)

print("CRS:", crs)
print(
    f"Resolution: "
    f"{abs(transform.a):.2f} × "
    f"{abs(transform.e):.2f} m"
)
print(f"Finite slope pixels: {valid.size:,}")
print(f"Pixel area: {pixel_area_km2:.6f} km²")

# -----------------------------------------------------------------------------
# DESCRIPTIVE SLOPE CLASSES
#
# These bins are used only to describe terrain distribution.
# They are NOT ML training labels or final suitability classes.
# -----------------------------------------------------------------------------
classes = [
    (0, 5, "0–5°"),
    (5, 10, "5–10°"),
    (10, 15, "10–15°"),
    (15, 25, "15–25°"),
    (25, np.inf, ">25°"),
]

records = []

for lower, upper, label in classes:

    if np.isinf(upper):
        mask = valid >= lower
    else:
        mask = (
            (valid >= lower) &
            (valid < upper)
        )

    count = int(np.count_nonzero(mask))

    area_km2 = count * pixel_area_km2

    pct = (
        count / valid.size * 100
        if valid.size > 0
        else np.nan
    )

    records.append({
        "Slope_Class": label,
        "Pixel_Count": count,
        "Area_km2": area_km2,
        "Percent_of_Valid_Area": pct
    })

slope_table = pd.DataFrame(records)

# -----------------------------------------------------------------------------
# OVERALL DESCRIPTIVE STATISTICS
# -----------------------------------------------------------------------------
percentiles = np.percentile(
    valid,
    [5, 25, 50, 75, 90, 95]
)

mean_slope = float(np.mean(valid))
median_slope = float(np.median(valid))
std_slope = float(np.std(valid))

total_area = (
    slope_table["Area_km2"].sum()
)

print("\nSLOPE DISTRIBUTION")
print("-" * 105)

print(
    slope_table.to_string(
        index=False,
        formatters={
            "Area_km2":
                lambda x: f"{x:,.2f}",
            "Percent_of_Valid_Area":
                lambda x: f"{x:.2f}%"
        }
    )
)

print("\nDESCRIPTIVE STATISTICS")
print("-" * 105)

print(f"Mean slope:       {mean_slope:.3f}°")
print(f"Median slope:     {median_slope:.3f}°")
print(f"Std deviation:    {std_slope:.3f}°")

print()
print(f"5th percentile:   {percentiles[0]:.3f}°")
print(f"25th percentile:  {percentiles[1]:.3f}°")
print(f"50th percentile:  {percentiles[2]:.3f}°")
print(f"75th percentile:  {percentiles[3]:.3f}°")
print(f"90th percentile:  {percentiles[4]:.3f}°")
print(f"95th percentile:  {percentiles[5]:.3f}°")

# -----------------------------------------------------------------------------
# ADD SUMMARY STATISTICS TO CSV
# -----------------------------------------------------------------------------
summary_rows = pd.DataFrame([
    {
        "Statistic": "Mean slope (degrees)",
        "Value": mean_slope
    },
    {
        "Statistic": "Median slope (degrees)",
        "Value": median_slope
    },
    {
        "Statistic": "5th percentile",
        "Value": percentiles[0]
    },
    {
        "Statistic": "25th percentile",
        "Value": percentiles[1]
    },
    {
        "Statistic": "75th percentile",
        "Value": percentiles[3]
    },
    {
        "Statistic": "90th percentile",
        "Value": percentiles[4]
    },
    {
        "Statistic": "95th percentile",
        "Value": percentiles[5]
    },
])

slope_table.to_csv(
    OUTPUT_CSV,
    index=False
)

summary_csv = (
    TABLE_DIR /
    "Enugu_Slope_Descriptive_Statistics.csv"
)

summary_rows.to_csv(
    summary_csv,
    index=False
)

# -----------------------------------------------------------------------------
# CREATE CHART
# -----------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.bar(
    slope_table["Slope_Class"],
    slope_table["Percent_of_Valid_Area"]
)

ax.set_title(
    "Slope Distribution Across Enugu State",
    fontsize=14,
    fontweight="bold"
)

ax.set_xlabel("Slope Class")
ax.set_ylabel("Share of Valid Terrain (%)")

ax.grid(
    axis="y",
    linestyle="--",
    alpha=0.35
)

for bar, pct in zip(
    bars,
    slope_table["Percent_of_Valid_Area"]
):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.5,
        f"{pct:.1f}%",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.tight_layout()

plt.savefig(
    OUTPUT_CHART,
    dpi=300,
    bbox_inches="tight"
)

plt.close()

# -----------------------------------------------------------------------------
# VALIDATION
# -----------------------------------------------------------------------------
percent_sum = (
    slope_table[
        "Percent_of_Valid_Area"
    ].sum()
)

pixel_sum = (
    slope_table[
        "Pixel_Count"
    ].sum()
)

classified_area = (
    slope_table[
        "Area_km2"
    ].sum()
)

expected_area = (
    valid.size *
    pixel_area_km2
)

checks = {
    "All finite pixels classified":
        pixel_sum == valid.size,

    "Slope percentages sum to ~100%":
        abs(percent_sum - 100) < 0.001,

    "Classified area matches raster valid area":
        abs(
            classified_area -
            expected_area
        ) < 0.01,

    "Mean slope is finite":
        np.isfinite(mean_slope),

    "Median slope is finite":
        np.isfinite(median_slope),

    "CSV created":
        OUTPUT_CSV.exists(),

    "Statistics CSV created":
        summary_csv.exists(),

    "Chart created":
        OUTPUT_CHART.exists(),
}

print("\nVALIDATION SUMMARY")
print("-" * 105)

for label, status in checks.items():
    print(
        f"{'✓' if status else '✗'} "
        f"{label}"
    )

print("\nTOTALS")
print("-" * 105)

print(
    f"Pixels classified: "
    f"{pixel_sum:,} / {valid.size:,}"
)

print(
    f"Area classified: "
    f"{classified_area:,.2f} km²"
)

print(
    f"Percentage total: "
    f"{percent_sum:.6f}%"
)

print("\nOUTPUT FILES")
print("-" * 105)

for f in [
    OUTPUT_CSV,
    summary_csv,
    OUTPUT_CHART
]:
    print(
        f"{'✓' if f.exists() else '✗'} "
        f"{f.name}"
    )

if all(checks.values()):

    print("\n✓ STAGE 2B PASSED")
    print(
        "Enugu terrain distribution has been "
        "quantified and validated."
    )

else:

    print("\n⚠ STAGE 2B REQUIRES REVIEW")

print("=" * 105)

In [ ]:
import os
# =====================================================================================
# PROJECT 7 — STAGE 3A
# DYNAMIC WORLD 2025 LAND-COVER ACQUISITION + SERVER-SIDE VALIDATION
#
# This stage:
# 1. Loads real Dynamic World observations for 2025.
# 2. Builds an annual modal land-cover composite.
# 3. Calculates class areas.
# 4. Starts a Drive batch export.
#
# No arbitrary suitability scores are assigned.
# =====================================================================================

import ee
import geemap
import geopandas as gpd
from pathlib import Path
from datetime import datetime, timezone

# -----------------------------------------------------------------------------
# SETTINGS
# -----------------------------------------------------------------------------
EE_PROJECT = os.environ.get("EE_PROJECT", "").strip()
if not EE_PROJECT:
    raise ValueError("Set EE_PROJECT before running this stage.")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

BOUNDARY_FILE = (
    PROJECT_ROOT /
    "01_Boundary" /
    "Enugu_State_Boundary_WGS84.gpkg"
)

DW_ID = "GOOGLE/DYNAMICWORLD/V1"

START_DATE = "2025-01-01"
END_DATE = "2026-01-01"

# Use a unique folder to avoid the duplicate-folder problem encountered earlier.
DRIVE_FOLDER = "Project_7_Enugu_DynamicWorld_2025"

print("=" * 108)
print("STAGE 3A — DYNAMIC WORLD 2025 LAND-COVER ACQUISITION")
print("=" * 108)

# -----------------------------------------------------------------------------
# INITIALIZE EARTH ENGINE
# -----------------------------------------------------------------------------
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate(auth_mode="colab")
    ee.Initialize(project=EE_PROJECT)

print("\n✓ Earth Engine initialized")
print("Cloud Project:", EE_PROJECT)

# -----------------------------------------------------------------------------
# LOAD VALIDATED STUDY BOUNDARY
# -----------------------------------------------------------------------------
enugu = gpd.read_file(
    BOUNDARY_FILE
).to_crs("EPSG:4326")

if len(enugu) != 1:
    raise RuntimeError(
        f"Expected one Enugu State feature, found {len(enugu)}."
    )

enugu_geom = (
    geemap
    .geopandas_to_ee(enugu)
    .geometry()
)

print("✓ Validated Enugu boundary loaded")

# -----------------------------------------------------------------------------
# LOAD DYNAMIC WORLD
# -----------------------------------------------------------------------------
dw = (
    ee.ImageCollection(DW_ID)
    .filterBounds(enugu_geom)
    .filterDate(START_DATE, END_DATE)
)

image_count = dw.size().getInfo()

print("\nDATASET")
print("-" * 108)

print("Dataset: Dynamic World V1")
print("Earth Engine ID:", DW_ID)
print("Period:", START_DATE, "to", END_DATE)
print("Images intersecting Enugu:", image_count)

if image_count == 0:
    raise RuntimeError(
        "No Dynamic World observations found for Enugu in 2025."
    )

# -----------------------------------------------------------------------------
# INSPECT TEMPORAL COVERAGE
# -----------------------------------------------------------------------------
first_date = ee.Date(
    dw.aggregate_min("system:time_start")
).format("YYYY-MM-dd").getInfo()

last_date = ee.Date(
    dw.aggregate_max("system:time_start")
).format("YYYY-MM-dd").getInfo()

print("First observation:", first_date)
print("Last observation:", last_date)

# -----------------------------------------------------------------------------
# BUILD ANNUAL MODE COMPOSITE
#
# Dynamic World labels:
# 0 Water
# 1 Trees
# 2 Grass
# 3 Flooded vegetation
# 4 Crops
# 5 Shrub & scrub
# 6 Built
# 7 Bare
# 8 Snow & ice
# -----------------------------------------------------------------------------
label_collection = dw.select("label")

lulc_2025 = (
    label_collection
    .mode()
    .clip(enugu_geom)
    .rename("LULC_2025")
    .toByte()
)

# Number of valid Dynamic World observations contributing at each pixel
observation_count = (
    label_collection
    .count()
    .clip(enugu_geom)
    .rename("DW_Obs_Count")
)

# -----------------------------------------------------------------------------
# CLASS DEFINITIONS
# -----------------------------------------------------------------------------
class_names = {
    0: "Water",
    1: "Trees",
    2: "Grass",
    3: "Flooded vegetation",
    4: "Crops",
    5: "Shrub and scrub",
    6: "Built",
    7: "Bare",
    8: "Snow and ice"
}

# -----------------------------------------------------------------------------
# SERVER-SIDE CLASS HISTOGRAM
# -----------------------------------------------------------------------------
hist = lulc_2025.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=enugu_geom,
    scale=10,
    maxPixels=2e9,
    tileScale=4
).getInfo()

histogram = hist.get("LULC_2025", {})

print("\n2025 MODAL LAND-COVER HISTOGRAM")
print("-" * 108)

total_hist_pixels = sum(
    float(v) for v in histogram.values()
)

class_records = []

for class_id in range(9):

    count = float(
        histogram.get(
            str(class_id),
            histogram.get(class_id, 0)
        )
    )

    # Dynamic World nominal pixel size = 10 m
    nominal_area_km2 = (
        count * 100 / 1_000_000
    )

    percentage = (
        count / total_hist_pixels * 100
        if total_hist_pixels > 0
        else 0
    )

    class_records.append({
        "Class_ID": class_id,
        "Class_Name": class_names[class_id],
        "Pixel_Count_Nominal": count,
        "Nominal_Area_km2": nominal_area_km2,
        "Percent": percentage
    })

    print(
        f"{class_id} | "
        f"{class_names[class_id]:<20} | "
        f"{count:>12,.0f} nominal pixels | "
        f"{percentage:>7.3f}%"
    )

# -----------------------------------------------------------------------------
# MORE ACCURATE CLASS AREA USING EE PIXELAREA
# -----------------------------------------------------------------------------
area_image = (
    ee.Image.pixelArea()
    .divide(1_000_000)
    .rename("Area_km2")
    .addBands(lulc_2025)
)

area_stats = area_image.reduceRegion(
    reducer=ee.Reducer.sum().group(
        groupField=1,
        groupName="class"
    ),
    geometry=enugu_geom,
    scale=10,
    maxPixels=2e9,
    tileScale=4
).getInfo()

groups = area_stats.get("groups", [])

area_lookup = {
    int(item["class"]): float(item["sum"])
    for item in groups
}

print("\nPIXEL-AREA-BASED LAND-COVER AREA")
print("-" * 108)

total_dw_area = sum(area_lookup.values())

for class_id in range(9):

    area = area_lookup.get(class_id, 0.0)

    pct = (
        area / total_dw_area * 100
        if total_dw_area > 0
        else 0
    )

    print(
        f"{class_id} | "
        f"{class_names[class_id]:<20} | "
        f"{area:>10,.2f} km² | "
        f"{pct:>7.3f}%"
    )

print(
    f"\nTotal classified Dynamic World area: "
    f"{total_dw_area:,.2f} km²"
)

# -----------------------------------------------------------------------------
# OBSERVATION COUNT STATISTICS
# -----------------------------------------------------------------------------
obs_stats = observation_count.reduceRegion(
    reducer=(
        ee.Reducer.minMax()
        .combine(
            ee.Reducer.mean(),
            sharedInputs=True
        )
    ),
    geometry=enugu_geom,
    scale=10,
    maxPixels=2e9,
    tileScale=4
).getInfo()

print("\nOBSERVATION COVERAGE")
print("-" * 108)

for key, value in obs_stats.items():
    print(f"{key}: {value}")

# -----------------------------------------------------------------------------
# START BATCH EXPORT
# -----------------------------------------------------------------------------
print("\nSTARTING GOOGLE DRIVE EXPORT")
print("-" * 108)

export_image = (
    lulc_2025
    .addBands(
        observation_count.toUint16()
    )
)

task = ee.batch.Export.image.toDrive(
    image=export_image,
    description="Project7_Enugu_DynamicWorld_2025",
    folder=DRIVE_FOLDER,
    fileNamePrefix="Enugu_DynamicWorld_2025",
    region=enugu_geom,
    scale=10,
    crs="EPSG:32632",
    maxPixels=2e9,
    fileFormat="GeoTIFF",
    formatOptions={
        "cloudOptimized": True
    }
)

task.start()

task_status = task.status()

print("Task ID:", task_status.get("id"))
print("Task state:", task_status.get("state"))
print(
    "Drive destination:",
    f"MyDrive/{DRIVE_FOLDER}"
)

# -----------------------------------------------------------------------------
# VALIDATION
# -----------------------------------------------------------------------------
state_area_km2 = 7625.21

area_difference_pct = (
    abs(total_dw_area - state_area_km2)
    / state_area_km2
    * 100
)

present_classes = sum(
    area_lookup.get(i, 0) > 0
    for i in range(9)
)

checks = {
    "Dynamic World images found":
        image_count > 0,

    "Temporal coverage begins in 2025":
        first_date.startswith("2025"),

    "Temporal coverage ends in 2025":
        last_date.startswith("2025"),

    "Modal composite has classified pixels":
        total_hist_pixels > 0,

    "Pixel-area calculation > 0":
        total_dw_area > 0,

    "DW area within 2% of validated state area":
        area_difference_pct < 2,

    "Multiple land-cover classes present":
        present_classes >= 4,

    "Observation statistics available":
        all(
            value is not None
            for value in obs_stats.values()
        ),

    "Export task started":
        task_status.get("state")
        in ["READY", "RUNNING"]
}

print("\nVALIDATION SUMMARY")
print("-" * 108)

for label, status in checks.items():
    print(
        f"{'✓' if status else '✗'} "
        f"{label}"
    )

print("\nAREA CONSISTENCY")
print("-" * 108)

print(
    f"Validated vector area: "
    f"{state_area_km2:,.2f} km²"
)

print(
    f"Dynamic World classified area: "
    f"{total_dw_area:,.2f} km²"
)

print(
    f"Difference: "
    f"{area_difference_pct:.4f}%"
)

print("\nIMPORTANT — SAVE THESE VALUES")
print("-" * 108)

print("DW_TASK_ID =", task_status.get("id"))
print("DW_DRIVE_FOLDER =", DRIVE_FOLDER)

if all(checks.values()):

    print("\n✓ STAGE 3A SERVER-SIDE VALIDATION PASSED")
    print(
        "The 2025 Dynamic World composite is valid "
        "and its export has started."
    )

else:

    print("\n⚠ STAGE 3A REQUIRES REVIEW")

print("=" * 108)

In [ ]:
import os
# =====================================================================================
# PROJECT 7 — STAGE 3B FIX
# RE-EXPORT DYNAMIC WORLD WITH COMPATIBLE BAND DATA TYPES
#
# Fix:
# Cast BOTH bands to UInt16 before exporting.
# =====================================================================================

import ee
import geemap
import geopandas as gpd
from pathlib import Path

# -----------------------------------------------------------------------------
# SETTINGS
# -----------------------------------------------------------------------------
EE_PROJECT = os.environ.get("EE_PROJECT", "").strip()
if not EE_PROJECT:
    raise ValueError("Set EE_PROJECT before running this stage.")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

BOUNDARY_FILE = (
    PROJECT_ROOT /
    "01_Boundary" /
    "Enugu_State_Boundary_WGS84.gpkg"
)

DW_ID = "GOOGLE/DYNAMICWORLD/V1"

START_DATE = "2025-01-01"
END_DATE = "2026-01-01"

# Use a fresh folder name so this corrected export is unambiguous.
DRIVE_FOLDER = "Project_7_Enugu_DynamicWorld_2025_Fixed"

print("=" * 110)
print("STAGE 3B FIX — DYNAMIC WORLD COMPATIBLE-TYPE EXPORT")
print("=" * 110)

# -----------------------------------------------------------------------------
# INITIALIZE
# -----------------------------------------------------------------------------
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate(auth_mode="colab")
    ee.Initialize(project=EE_PROJECT)

print("\n✓ Earth Engine initialized")

# -----------------------------------------------------------------------------
# LOAD VALIDATED ENUGU BOUNDARY
# -----------------------------------------------------------------------------
enugu = gpd.read_file(
    BOUNDARY_FILE
).to_crs("EPSG:4326")

enugu_geom = (
    geemap
    .geopandas_to_ee(enugu)
    .geometry()
)

print("✓ Enugu boundary loaded")

# -----------------------------------------------------------------------------
# REBUILD SAME VERIFIED 2025 COMPOSITE
# -----------------------------------------------------------------------------
dw = (
    ee.ImageCollection(DW_ID)
    .filterBounds(enugu_geom)
    .filterDate(START_DATE, END_DATE)
)

image_count = dw.size().getInfo()

if image_count != 94:
    print(
        f"⚠ Image count is now {image_count}; "
        "previous validated count was 94."
    )
else:
    print(
        "✓ Dynamic World image count matches "
        "previous validation: 94"
    )

label_collection = dw.select("label")

# Cast LULC explicitly to UInt16
lulc_2025 = (
    label_collection
    .mode()
    .clip(enugu_geom)
    .rename("LULC_2025")
    .toUint16()
)

# Observation count also UInt16
observation_count = (
    label_collection
    .count()
    .clip(enugu_geom)
    .rename("DW_Obs_Count")
    .toUint16()
)

# -----------------------------------------------------------------------------
# VERIFY BAND TYPES BEFORE EXPORT
# -----------------------------------------------------------------------------
export_image = (
    lulc_2025
    .addBands(observation_count)
)

band_types = export_image.bandTypes().getInfo()

print("\nBAND TYPE CHECK")
print("-" * 110)

for band, info in band_types.items():
    print(
        f"{band}: "
        f"{info.get('precision')} / "
        f"{info.get('min')} to {info.get('max')}"
    )

# -----------------------------------------------------------------------------
# START CORRECTED EXPORT
# -----------------------------------------------------------------------------
print("\nSTARTING CORRECTED EXPORT")
print("-" * 110)

task = ee.batch.Export.image.toDrive(
    image=export_image,
    description="Project7_Enugu_DynamicWorld_2025_Fixed",
    folder=DRIVE_FOLDER,
    fileNamePrefix="Enugu_DynamicWorld_2025",
    region=enugu_geom,
    scale=10,
    crs="EPSG:32632",
    maxPixels=2e9,
    fileFormat="GeoTIFF",
    formatOptions={
        "cloudOptimized": True
    }
)

task.start()

status = task.status()

print("Task ID:", status.get("id"))
print("Task state:", status.get("state"))
print(
    "Drive destination:",
    f"MyDrive/{DRIVE_FOLDER}"
)

# -----------------------------------------------------------------------------
# FINAL CHECK
# -----------------------------------------------------------------------------
checks = {
    "Dynamic World images present":
        image_count > 0,

    "LULC band cast to UInt16":
        "LULC_2025" in band_types,

    "Observation band cast to UInt16":
        "DW_Obs_Count" in band_types,

    "Corrected export started":
        status.get("state")
        in ["READY", "RUNNING"]
}

print("\nVALIDATION SUMMARY")
print("-" * 110)

for label, passed in checks.items():
    print(
        f"{'✓' if passed else '✗'} "
        f"{label}"
    )

print("\nIMPORTANT — SAVE THESE VALUES")
print("-" * 110)

print(
    "DW_FIXED_TASK_ID =",
    status.get("id")
)

print(
    "DW_FIXED_DRIVE_FOLDER =",
    DRIVE_FOLDER
)

if all(checks.values()):

    print("\n✓ STAGE 3B FIX PASSED")
    print(
        "Corrected Dynamic World export "
        "has started successfully."
    )

else:

    print("\n⚠ STAGE 3B FIX REQUIRES REVIEW")

print("=" * 110)

In [ ]:
import os
# =====================================================================================
# PROJECT 7 — STAGE 3C
# WAIT FOR CORRECTED DYNAMIC WORLD EXPORT + LOCAL VALIDATION
#
# Existing task only — DOES NOT create another export.
# =====================================================================================

import ee
import time
import shutil
import rasterio
import numpy as np
import pandas as pd
from pathlib import Path

# -----------------------------------------------------------------------------
# SETTINGS
# -----------------------------------------------------------------------------
EE_PROJECT = os.environ.get("EE_PROJECT", "").strip()
if not EE_PROJECT:
    raise ValueError("Set EE_PROJECT before running this stage.")

DW_TASK_ID = "3ANT7NHGAQ4VNJXBD5NBFIFO"

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

MYDRIVE = Path("/content/drive/MyDrive")

LANDCOVER_DIR = PROJECT_ROOT / "04_Land_Cover"
TABLE_DIR = PROJECT_ROOT / "16_Tables"

LANDCOVER_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

TARGET_FILE = (
    LANDCOVER_DIR /
    "Enugu_DynamicWorld_2025.tif"
)

CLASS_TABLE_FILE = (
    TABLE_DIR /
    "Enugu_DynamicWorld_2025_Class_Area.csv"
)

VALIDATION_FILE = (
    LANDCOVER_DIR /
    "DynamicWorld_2025_Validation.csv"
)

STATE_AREA_KM2 = 7625.21

CLASS_NAMES = {
    0: "Water",
    1: "Trees",
    2: "Grass",
    3: "Flooded vegetation",
    4: "Crops",
    5: "Shrub and scrub",
    6: "Built",
    7: "Bare",
    8: "Snow and ice"
}

print("=" * 110)
print("STAGE 3C — DYNAMIC WORLD EXPORT COMPLETION + LOCAL VALIDATION")
print("=" * 110)

# -----------------------------------------------------------------------------
# INITIALIZE EARTH ENGINE
# -----------------------------------------------------------------------------
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate(auth_mode="colab")
    ee.Initialize(project=EE_PROJECT)

print("\n✓ Earth Engine initialized")

# -----------------------------------------------------------------------------
# WAIT FOR EXPORT
# -----------------------------------------------------------------------------
print("\nEARTH ENGINE EXPORT STATUS")
print("-" * 110)

for i in range(61):

    status = ee.data.getTaskStatus(
        DW_TASK_ID
    )[0]

    state = status.get(
        "state",
        "UNKNOWN"
    )

    print(
        f"Check {i + 1:02d}: {state}"
    )

    if state == "COMPLETED":
        print("\n✓ Export completed")
        break

    if state in [
        "FAILED",
        "CANCELLED"
    ]:
        raise RuntimeError(
            status.get(
                "error_message",
                f"Task state = {state}"
            )
        )

    if i == 60:
        raise TimeoutError(
            "Export still running after 20 minutes. "
            "Do not rerun the export cell."
        )

    time.sleep(20)

# -----------------------------------------------------------------------------
# LOCATE EXPORT
# -----------------------------------------------------------------------------
print("\nLOCATING EXPORTED FILE")
print("-" * 110)

matches = []

for pattern in [
    "Enugu_DynamicWorld_2025*.tif",
    "Enugu_DynamicWorld_2025*.tiff"
]:
    matches.extend(
        MYDRIVE.rglob(pattern)
    )

# Remove permanent project copy if cell is rerun
matches = [
    p for p in matches
    if p.resolve() != TARGET_FILE.resolve()
]

matches = list({
    str(p): p
    for p in matches
}.values())

print(
    "Candidate files found:",
    len(matches)
)

for path in matches:
    print(
        f" - {path} "
        f"({path.stat().st_size / 1024**2:.2f} MB)"
    )

if not matches:
    raise RuntimeError(
        "Task completed but export file is not yet visible in Drive."
    )

source_file = max(
    matches,
    key=lambda p: p.stat().st_mtime
)

print("\nSelected file:")
print(source_file)

# -----------------------------------------------------------------------------
# COPY TO PROJECT
# -----------------------------------------------------------------------------
shutil.copy2(
    source_file,
    TARGET_FILE
)

print("\n✓ Copied to:")
print(TARGET_FILE)

# -----------------------------------------------------------------------------
# READ RASTER
# -----------------------------------------------------------------------------
with rasterio.open(TARGET_FILE) as src:

    print("\nRASTER METADATA")
    print("-" * 110)

    print("CRS:", src.crs)
    print("EPSG:", src.crs.to_epsg())
    print(
        f"Dimensions: "
        f"{src.width} × {src.height}"
    )
    print(
        f"Resolution: "
        f"{abs(src.transform.a):.3f} × "
        f"{abs(src.transform.e):.3f} m"
    )
    print("Band count:", src.count)
    print("Data types:", src.dtypes)
    print("NoData:", src.nodata)

    if src.count != 2:
        raise RuntimeError(
            f"Expected 2 bands, found {src.count}"
        )

    lulc = src.read(1)
    obs = src.read(2)

    epsg = src.crs.to_epsg()

    res_x = abs(src.transform.a)
    res_y = abs(src.transform.e)

    pixel_area_km2 = (
        res_x *
        res_y /
        1_000_000
    )

# -----------------------------------------------------------------------------
# VALID PIXELS
# -----------------------------------------------------------------------------
valid_mask = (
    np.isfinite(lulc)
    &
    (lulc >= 0)
    &
    (lulc <= 8)
    &
    np.isfinite(obs)
    &
    (obs >= 1)
)

valid_lulc = lulc[
    valid_mask
].astype(np.uint16)

valid_obs = obs[
    valid_mask
].astype(np.uint16)

print("\nVALID PIXEL SUMMARY")
print("-" * 110)

print(
    f"Valid pixels: "
    f"{valid_lulc.size:,}"
)

print(
    "Unique classes:",
    sorted(
        np.unique(
            valid_lulc
        ).tolist()
    )
)

print(
    f"Observation range: "
    f"{valid_obs.min()}–{valid_obs.max()}"
)

# -----------------------------------------------------------------------------
# CLASS AREA TABLE
# -----------------------------------------------------------------------------
records = []

for class_id in range(9):

    count = int(
        np.count_nonzero(
            valid_lulc == class_id
        )
    )

    area = (
        count *
        pixel_area_km2
    )

    pct = (
        count /
        valid_lulc.size *
        100
    )

    records.append({
        "Class_ID": class_id,
        "Class_Name": CLASS_NAMES[class_id],
        "Pixel_Count": count,
        "Area_km2": area,
        "Percent": pct
    })

class_df = pd.DataFrame(
    records
)

print("\nLOCAL LAND-COVER DISTRIBUTION")
print("-" * 110)

print(
    class_df.to_string(
        index=False,
        formatters={
            "Area_km2":
                lambda x: f"{x:,.2f}",
            "Percent":
                lambda x: f"{x:.3f}%"
        }
    )
)

local_area = float(
    class_df["Area_km2"].sum()
)

area_diff_pct = (
    abs(
        local_area -
        STATE_AREA_KM2
    )
    /
    STATE_AREA_KM2 *
    100
)

# -----------------------------------------------------------------------------
# OBSERVATION QUALITY
# -----------------------------------------------------------------------------
obs_min = int(
    valid_obs.min()
)

obs_max = int(
    valid_obs.max()
)

obs_mean = float(
    valid_obs.mean()
)

obs_median = float(
    np.median(valid_obs)
)

one_obs_pct = (
    np.count_nonzero(
        valid_obs == 1
    )
    /
    valid_obs.size *
    100
)

print("\nOBSERVATION QUALITY")
print("-" * 110)

print(
    f"Minimum: {obs_min}"
)

print(
    f"Maximum: {obs_max}"
)

print(
    f"Mean: {obs_mean:.3f}"
)

print(
    f"Median: {obs_median:.3f}"
)

print(
    f"Pixels with only 1 observation: "
    f"{one_obs_pct:.2f}%"
)

# -----------------------------------------------------------------------------
# ANOMALOUS CLASS 8
# -----------------------------------------------------------------------------
snow_area = float(
    class_df.loc[
        class_df["Class_ID"] == 8,
        "Area_km2"
    ].iloc[0]
)

snow_pct = float(
    class_df.loc[
        class_df["Class_ID"] == 8,
        "Percent"
    ].iloc[0]
)

print("\nANOMALY CHECK")
print("-" * 110)

print(
    f"Class 8 ('Snow and ice'): "
    f"{snow_area:.4f} km² "
    f"({snow_pct:.5f}%)"
)

if snow_area > 0:
    print(
        "⚠ Treat as classification artifact/noise, "
        "not literal snow cover."
    )

# -----------------------------------------------------------------------------
# SAVE TABLES
# -----------------------------------------------------------------------------
class_df.to_csv(
    CLASS_TABLE_FILE,
    index=False
)

validation_df = pd.DataFrame([{
    "Task_ID": DW_TASK_ID,
    "EPSG": epsg,
    "Resolution_m": res_x,
    "Valid_Pixels": int(
        valid_lulc.size
    ),
    "Class_Count": int(
        len(
            np.unique(
                valid_lulc
            )
        )
    ),
    "Local_Area_km2": local_area,
    "State_Area_km2": STATE_AREA_KM2,
    "Area_Difference_pct": area_diff_pct,
    "Obs_Min": obs_min,
    "Obs_Max": obs_max,
    "Obs_Mean": obs_mean,
    "Obs_Median": obs_median,
    "One_Obs_pct": one_obs_pct,
    "Class8_Artifact_Area_km2": snow_area
}])

validation_df.to_csv(
    VALIDATION_FILE,
    index=False
)

# -----------------------------------------------------------------------------
# FINAL CHECKS
# -----------------------------------------------------------------------------
checks = {
    "Export completed":
        state == "COMPLETED",

    "Permanent raster exists":
        TARGET_FILE.exists(),

    "CRS is EPSG:32632":
        epsg == 32632,

    "Resolution is 10 m":
        (
            np.isclose(res_x, 10)
            and
            np.isclose(res_y, 10)
        ),

    "Two raster bands present":
        lulc is not None
        and obs is not None,

    "Land-cover IDs restricted to 0–8":
        (
            valid_lulc.min() >= 0
            and
            valid_lulc.max() <= 8
        ),

    "Multiple classes present":
        len(
            np.unique(
                valid_lulc
            )
        ) >= 4,

    "Area within 2% of state":
        area_diff_pct < 2,

    "Observation counts >= 1":
        obs_min >= 1,

    "Maximum observation count = 36":
        obs_max == 36,

    "Mean observation count close to EE result":
        abs(
            obs_mean -
            6.806841117758183
        ) < 0.2,

    "Class-area table saved":
        CLASS_TABLE_FILE.exists(),

    "Validation file saved":
        VALIDATION_FILE.exists()
}

print("\nAREA CONSISTENCY")
print("-" * 110)

print(
    f"Validated state area: "
    f"{STATE_AREA_KM2:,.2f} km²"
)

print(
    f"Local classified area: "
    f"{local_area:,.2f} km²"
)

print(
    f"Difference: "
    f"{area_diff_pct:.4f}%"
)

print("\nVALIDATION SUMMARY")
print("-" * 110)

for label, passed in checks.items():
    print(
        f"{'✓' if passed else '✗'} "
        f"{label}"
    )

print("\nOUTPUT FILES")
print("-" * 110)

for path in [
    TARGET_FILE,
    CLASS_TABLE_FILE,
    VALIDATION_FILE
]:
    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{path.name}"
    )

if all(checks.values()):
    print("\n✓ STAGE 3C FULLY PASSED")
    print(
        "Dynamic World 2025 is permanently stored "
        "and locally validated."
    )
else:
    print("\n⚠ STAGE 3C REQUIRES REVIEW")

print("=" * 110)

In [ ]:
import os
# =====================================================================================
# PROJECT 7 — STAGE 4A
# SENTINEL-2 2025 SPECTRAL PREDICTORS
#
# Predictors:
#   NDVI  = (B8 - B4) / (B8 + B4)
#   NDBI  = (B11 - B8) / (B11 + B8)
#   MNDWI = (B3 - B11) / (B3 + B11)
#
# Source:
# COPERNICUS/S2_SR_HARMONIZED
#
# Processing:
# - 2025 observations
# - scene cloud filter <= 20%
# - SCL cloud/shadow/snow masking
# - annual median composite
# - export at 20 m because B11 is natively 20 m
# =====================================================================================

import ee
import geemap
import geopandas as gpd
from pathlib import Path

# -----------------------------------------------------------------------------
# SETTINGS
# -----------------------------------------------------------------------------
EE_PROJECT = os.environ.get("EE_PROJECT", "").strip()
if not EE_PROJECT:
    raise ValueError("Set EE_PROJECT before running this stage.")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

BOUNDARY_FILE = (
    PROJECT_ROOT /
    "01_Boundary" /
    "Enugu_State_Boundary_WGS84.gpkg"
)

S2_ID = "COPERNICUS/S2_SR_HARMONIZED"

START_DATE = "2025-01-01"
END_DATE = "2026-01-01"

MAX_SCENE_CLOUD = 20

DRIVE_FOLDER = (
    "Project_7_Enugu_Sentinel2_Indices_2025"
)

print("=" * 110)
print("STAGE 4A — SENTINEL-2 2025 SPECTRAL PREDICTORS")
print("=" * 110)

# -----------------------------------------------------------------------------
# INITIALIZE EARTH ENGINE
# -----------------------------------------------------------------------------
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate(auth_mode="colab")
    ee.Initialize(project=EE_PROJECT)

print("\n✓ Earth Engine initialized")
print("Cloud Project:", EE_PROJECT)

# -----------------------------------------------------------------------------
# LOAD VALIDATED ENUGU BOUNDARY
# -----------------------------------------------------------------------------
enugu = gpd.read_file(
    BOUNDARY_FILE
).to_crs("EPSG:4326")

if len(enugu) != 1:
    raise RuntimeError(
        f"Expected exactly one Enugu feature, found {len(enugu)}."
    )

enugu_geom = (
    geemap
    .geopandas_to_ee(enugu)
    .geometry()
)

print("✓ Validated Enugu boundary loaded")

# -----------------------------------------------------------------------------
# CLOUD / SHADOW MASK USING SENTINEL-2 SCL
#
# SCL classes excluded:
# 0  No data
# 1  Saturated/defective
# 3  Cloud shadow
# 8  Medium-probability cloud
# 9  High-probability cloud
# 10 Cirrus
# 11 Snow/ice
# -----------------------------------------------------------------------------
def mask_s2_scl(image):

    scl = image.select("SCL")

    clear = (
        scl.neq(0)
        .And(scl.neq(1))
        .And(scl.neq(3))
        .And(scl.neq(8))
        .And(scl.neq(9))
        .And(scl.neq(10))
        .And(scl.neq(11))
    )

    # Convert reflectance scale factor 10000 -> 0–1
    spectral = (
        image
        .select(
            ["B3", "B4", "B8", "B11"]
        )
        .multiply(0.0001)
        .updateMask(clear)
    )

    return (
        spectral
        .copyProperties(
            image,
            ["system:time_start"]
        )
    )

# -----------------------------------------------------------------------------
# SENTINEL-2 COLLECTION
# -----------------------------------------------------------------------------
raw_s2 = (
    ee.ImageCollection(S2_ID)
    .filterBounds(enugu_geom)
    .filterDate(
        START_DATE,
        END_DATE
    )
)

raw_count = raw_s2.size().getInfo()

filtered_s2 = (
    raw_s2
    .filter(
        ee.Filter.lte(
            "CLOUDY_PIXEL_PERCENTAGE",
            MAX_SCENE_CLOUD
        )
    )
)

filtered_count = (
    filtered_s2
    .size()
    .getInfo()
)

print("\nSENTINEL-2 COLLECTION")
print("-" * 110)

print("Dataset:", S2_ID)
print("Period:", START_DATE, "to", END_DATE)
print(
    "All intersecting granules:",
    raw_count
)
print(
    f"Granules ≤{MAX_SCENE_CLOUD}% cloud:",
    filtered_count
)

if filtered_count == 0:
    raise RuntimeError(
        "No Sentinel-2 scenes passed the cloud filter."
    )

first_date = ee.Date(
    filtered_s2.aggregate_min(
        "system:time_start"
    )
).format(
    "YYYY-MM-dd"
).getInfo()

last_date = ee.Date(
    filtered_s2.aggregate_max(
        "system:time_start"
    )
).format(
    "YYYY-MM-dd"
).getInfo()

print("First retained observation:", first_date)
print("Last retained observation:", last_date)

# -----------------------------------------------------------------------------
# APPLY PIXEL-LEVEL CLOUD/SHADOW MASK
# -----------------------------------------------------------------------------
clean_s2 = (
    filtered_s2
    .map(mask_s2_scl)
)

# Annual median reflectance composite
composite = (
    clean_s2
    .median()
    .clip(enugu_geom)
)

# -----------------------------------------------------------------------------
# DERIVE INDICES
# -----------------------------------------------------------------------------
ndvi = (
    composite
    .normalizedDifference(
        ["B8", "B4"]
    )
    .rename("NDVI_2025")
)

ndbi = (
    composite
    .normalizedDifference(
        ["B11", "B8"]
    )
    .rename("NDBI_2025")
)

mndwi = (
    composite
    .normalizedDifference(
        ["B3", "B11"]
    )
    .rename("MNDWI_2025")
)

indices = (
    ndvi
    .addBands(ndbi)
    .addBands(mndwi)
    .toFloat()
    .clip(enugu_geom)
)

# -----------------------------------------------------------------------------
# VALID-OBSERVATION COUNT
#
# Count clear B8 observations contributing to the 2025 composite.
# -----------------------------------------------------------------------------
obs_count = (
    clean_s2
    .select("B8")
    .count()
    .rename("S2_Clear_Obs_Count")
    .clip(enugu_geom)
)

# -----------------------------------------------------------------------------
# SERVER-SIDE INDEX STATISTICS
# -----------------------------------------------------------------------------
reducer = (
    ee.Reducer.minMax()
    .combine(
        reducer2=ee.Reducer.mean(),
        sharedInputs=True
    )
    .combine(
        reducer2=ee.Reducer.stdDev(),
        sharedInputs=True
    )
)

index_stats = indices.reduceRegion(
    reducer=reducer,
    geometry=enugu_geom,
    scale=20,
    maxPixels=1e9,
    tileScale=4
).getInfo()

obs_stats = obs_count.reduceRegion(
    reducer=(
        ee.Reducer.minMax()
        .combine(
            reducer2=ee.Reducer.mean(),
            sharedInputs=True
        )
    ),
    geometry=enugu_geom,
    scale=20,
    maxPixels=1e9,
    tileScale=4
).getInfo()

print("\nSPECTRAL INDEX STATISTICS")
print("-" * 110)

for key in sorted(
    index_stats.keys()
):
    print(
        f"{key}: "
        f"{index_stats[key]}"
    )

print("\nCLEAR-OBSERVATION STATISTICS")
print("-" * 110)

for key in sorted(
    obs_stats.keys()
):
    print(
        f"{key}: "
        f"{obs_stats[key]}"
    )

# -----------------------------------------------------------------------------
# BASIC SCIENTIFIC RANGE CHECK
# -----------------------------------------------------------------------------
required_stats = [
    "NDVI_2025_min",
    "NDVI_2025_max",
    "NDVI_2025_mean",
    "NDBI_2025_min",
    "NDBI_2025_max",
    "NDBI_2025_mean",
    "MNDWI_2025_min",
    "MNDWI_2025_max",
    "MNDWI_2025_mean",
]

missing_stats = [
    key for key in required_stats
    if index_stats.get(key) is None
]

if missing_stats:
    raise RuntimeError(
        "Missing index statistics: "
        + ", ".join(missing_stats)
    )

# Normalized differences theoretically range [-1, 1].
range_checks = {
    "NDVI": (
        -1 <= index_stats["NDVI_2025_min"]
        <= index_stats["NDVI_2025_max"]
        <= 1
    ),
    "NDBI": (
        -1 <= index_stats["NDBI_2025_min"]
        <= index_stats["NDBI_2025_max"]
        <= 1
    ),
    "MNDWI": (
        -1 <= index_stats["MNDWI_2025_min"]
        <= index_stats["MNDWI_2025_max"]
        <= 1
    )
}

# -----------------------------------------------------------------------------
# PREPARE COMPATIBLE EXPORT
#
# Index bands are Float32.
# Observation count is also cast to Float32 so all bands have compatible type.
# -----------------------------------------------------------------------------
export_image = (
    indices
    .addBands(
        obs_count.toFloat()
    )
)

band_types = (
    export_image
    .bandTypes()
    .getInfo()
)

print("\nEXPORT BAND TYPE CHECK")
print("-" * 110)

for band, info in band_types.items():
    print(
        f"{band}: "
        f"{info.get('precision')}"
    )

# -----------------------------------------------------------------------------
# START BATCH EXPORT
# -----------------------------------------------------------------------------
print("\nSTARTING GOOGLE DRIVE EXPORT")
print("-" * 110)

task = ee.batch.Export.image.toDrive(
    image=export_image,
    description=(
        "Project7_Enugu_"
        "Sentinel2_Indices_2025"
    ),
    folder=DRIVE_FOLDER,
    fileNamePrefix=(
        "Enugu_Sentinel2_"
        "Indices_2025"
    ),
    region=enugu_geom,
    scale=20,
    crs="EPSG:32632",
    maxPixels=1e9,
    fileFormat="GeoTIFF",
    formatOptions={
        "cloudOptimized": True
    }
)

task.start()

task_status = task.status()

print(
    "Task ID:",
    task_status.get("id")
)

print(
    "Task state:",
    task_status.get("state")
)

print(
    "Drive folder:",
    f"MyDrive/{DRIVE_FOLDER}"
)

# -----------------------------------------------------------------------------
# VALIDATION
# -----------------------------------------------------------------------------
checks = {
    "Sentinel-2 scenes found":
        raw_count > 0,

    "Cloud-filtered scenes retained":
        filtered_count > 0,

    "2025 temporal coverage retained":
        (
            first_date.startswith("2025")
            and
            last_date.startswith("2025")
        ),

    "NDVI within theoretical range":
        range_checks["NDVI"],

    "NDBI within theoretical range":
        range_checks["NDBI"],

    "MNDWI within theoretical range":
        range_checks["MNDWI"],

    "Clear-observation stats available":
        all(
            v is not None
            for v in obs_stats.values()
        ),

    "Four export bands present":
        len(band_types) == 4,

    "Export task started":
        task_status.get("state")
        in ["READY", "RUNNING"]
}

print("\nVALIDATION SUMMARY")
print("-" * 110)

for label, passed in checks.items():
    print(
        f"{'✓' if passed else '✗'} "
        f"{label}"
    )

print("\nIMPORTANT — SAVE THESE VALUES")
print("-" * 110)

print(
    "S2_INDEX_TASK_ID =",
    task_status.get("id")
)

print(
    "S2_INDEX_DRIVE_FOLDER =",
    DRIVE_FOLDER
)

if all(checks.values()):

    print(
        "\n✓ STAGE 4A SERVER-SIDE VALIDATION PASSED"
    )

    print(
        "Sentinel-2 spectral predictors are "
        "scientifically valid and export has started."
    )

else:

    print(
        "\n⚠ STAGE 4A REQUIRES REVIEW"
    )

print("=" * 110)

In [ ]:
import os
# =====================================================================================
# PROJECT 7 — STAGE 4B
# SENTINEL-2 INDICES EXPORT COMPLETION + LOCAL VALIDATION
#
# Existing Earth Engine task only — DOES NOT create another export.
# =====================================================================================

import ee
import time
import shutil
import rasterio
import numpy as np
import pandas as pd
from pathlib import Path

# -----------------------------------------------------------------------------
# SETTINGS
# -----------------------------------------------------------------------------
EE_PROJECT = os.environ.get("EE_PROJECT", "").strip()
if not EE_PROJECT:
    raise ValueError("Set EE_PROJECT before running this stage.")

S2_TASK_ID = "BUNBP5XXKESJ367LRCE4ARJB"

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

MYDRIVE = Path("/content/drive/MyDrive")

RS_DIR = PROJECT_ROOT / "03_Remote_Sensing"
TABLE_DIR = PROJECT_ROOT / "16_Tables"

RS_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

TARGET_FILE = (
    RS_DIR /
    "Enugu_Sentinel2_Indices_2025.tif"
)

VALIDATION_FILE = (
    RS_DIR /
    "Sentinel2_Indices_2025_Validation.csv"
)

STATS_FILE = (
    TABLE_DIR /
    "Enugu_Sentinel2_Index_Statistics.csv"
)

STATE_AREA_KM2 = 7625.21

# Server-side reference means from Stage 4A
EE_REFERENCE = {
    "NDVI_2025": {
        "min": -0.29946112632751465,
        "max": 0.8490167260169983,
        "mean": 0.43019815153357327
    },
    "NDBI_2025": {
        "min": -0.4450867176055908,
        "max": 0.8170576095581055,
        "mean": 0.009738171691328202
    },
    "MNDWI_2025": {
        "min": -0.907649040222168,
        "max": 0.5472924113273621,
        "mean": -0.49186337945012487
    }
}

EE_OBS_MEAN = 7.1867888349220275
EE_OBS_MAX = 18

print("=" * 112)
print("STAGE 4B — SENTINEL-2 EXPORT COMPLETION + LOCAL VALIDATION")
print("=" * 112)

# -----------------------------------------------------------------------------
# INITIALIZE EARTH ENGINE
# -----------------------------------------------------------------------------
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate(auth_mode="colab")
    ee.Initialize(project=EE_PROJECT)

print("\n✓ Earth Engine initialized")

# -----------------------------------------------------------------------------
# WAIT FOR ORIGINAL EXPORT TASK
# -----------------------------------------------------------------------------
print("\nEARTH ENGINE EXPORT STATUS")
print("-" * 112)

task_state = None

for i in range(61):

    status = ee.data.getTaskStatus(
        S2_TASK_ID
    )[0]

    task_state = status.get(
        "state",
        "UNKNOWN"
    )

    print(
        f"Check {i + 1:02d}: "
        f"{task_state}"
    )

    if task_state == "COMPLETED":
        print("\n✓ Sentinel-2 export completed")
        break

    if task_state in [
        "FAILED",
        "CANCELLED"
    ]:
        raise RuntimeError(
            "Sentinel-2 export failed:\n"
            + str(
                status.get(
                    "error_message",
                    "No error message supplied"
                )
            )
        )

    if i == 60:
        raise TimeoutError(
            "Export is still running after 20 minutes. "
            "Do NOT rerun Stage 4A."
        )

    time.sleep(20)

# -----------------------------------------------------------------------------
# LOCATE EXPORTED FILE ANYWHERE IN MYDRIVE
# -----------------------------------------------------------------------------
print("\nLOCATING EXPORTED GEOTIFF")
print("-" * 112)

matches = []

for pattern in [
    "Enugu_Sentinel2_Indices_2025*.tif",
    "Enugu_Sentinel2_Indices_2025*.tiff"
]:
    matches.extend(
        MYDRIVE.rglob(pattern)
    )

# Exclude permanent project copy if this cell is rerun
matches = [
    p for p in matches
    if p.resolve() != TARGET_FILE.resolve()
]

# Deduplicate
matches = list({
    str(p): p
    for p in matches
}.values())

print(
    "Candidate files found:",
    len(matches)
)

for path in matches:
    print(
        f" - {path} "
        f"({path.stat().st_size / 1024**2:.2f} MB)"
    )

if len(matches) == 0:
    raise RuntimeError(
        "Earth Engine reports COMPLETED but the "
        "Sentinel-2 GeoTIFF is not visible in mounted Drive."
    )

# Choose most recently modified candidate
SOURCE_FILE = max(
    matches,
    key=lambda p: p.stat().st_mtime
)

print("\nSelected export:")
print(SOURCE_FILE)

# -----------------------------------------------------------------------------
# COPY INTO PERMANENT PROJECT DIRECTORY
# -----------------------------------------------------------------------------
shutil.copy2(
    SOURCE_FILE,
    TARGET_FILE
)

print("\n✓ Copied to permanent project folder:")
print(TARGET_FILE)

# -----------------------------------------------------------------------------
# READ RASTER
# -----------------------------------------------------------------------------
with rasterio.open(TARGET_FILE) as src:

    print("\nRASTER METADATA")
    print("-" * 112)

    print("CRS:", src.crs)
    print("EPSG:", src.crs.to_epsg())

    print(
        f"Dimensions: "
        f"{src.width} × {src.height}"
    )

    print(
        f"Resolution: "
        f"{abs(src.transform.a):.3f} × "
        f"{abs(src.transform.e):.3f} m"
    )

    print("Band count:", src.count)
    print("Data types:", src.dtypes)
    print("NoData:", src.nodata)

    epsg = src.crs.to_epsg()
    res_x = abs(src.transform.a)
    res_y = abs(src.transform.e)

    if src.count != 4:
        raise RuntimeError(
            f"Expected 4 bands, found {src.count}."
        )

    # Export order from Stage 4A:
    # 1 NDVI
    # 2 NDBI
    # 3 MNDWI
    # 4 clear observation count
    ndvi = src.read(1)
    ndbi = src.read(2)
    mndwi = src.read(3)
    obs = src.read(4)

    pixel_area_km2 = (
        res_x *
        res_y /
        1_000_000
    )

# -----------------------------------------------------------------------------
# COMMON VALID MASK
# -----------------------------------------------------------------------------
valid_mask = (
    np.isfinite(ndvi)
    &
    np.isfinite(ndbi)
    &
    np.isfinite(mndwi)
    &
    np.isfinite(obs)
    &
    (obs >= 1)
)

ndvi_v = ndvi[valid_mask].astype("float64")
ndbi_v = ndbi[valid_mask].astype("float64")
mndwi_v = mndwi[valid_mask].astype("float64")
obs_v = obs[valid_mask].astype("float64")

if ndvi_v.size == 0:
    raise RuntimeError(
        "No valid common Sentinel-2 pixels were found."
    )

# -----------------------------------------------------------------------------
# STATISTICS FUNCTION
# -----------------------------------------------------------------------------
def summarize(values, name):

    return {
        "Predictor": name,
        "Valid_Pixels": int(values.size),
        "Min": float(np.min(values)),
        "Max": float(np.max(values)),
        "Mean": float(np.mean(values)),
        "StdDev": float(np.std(values)),
        "P05": float(np.percentile(values, 5)),
        "P25": float(np.percentile(values, 25)),
        "Median": float(np.percentile(values, 50)),
        "P75": float(np.percentile(values, 75)),
        "P95": float(np.percentile(values, 95))
    }

stats = [
    summarize(ndvi_v, "NDVI_2025"),
    summarize(ndbi_v, "NDBI_2025"),
    summarize(mndwi_v, "MNDWI_2025")
]

stats_df = pd.DataFrame(stats)

# -----------------------------------------------------------------------------
# DISPLAY LOCAL INDEX STATISTICS
# -----------------------------------------------------------------------------
print("\nLOCAL SPECTRAL INDEX STATISTICS")
print("-" * 112)

for row in stats:

    print(f"\n{row['Predictor']}")
    print(
        f"  Valid pixels: "
        f"{row['Valid_Pixels']:,}"
    )
    print(
        f"  Min:    {row['Min']:.6f}"
    )
    print(
        f"  Max:    {row['Max']:.6f}"
    )
    print(
        f"  Mean:   {row['Mean']:.6f}"
    )
    print(
        f"  StdDev: {row['StdDev']:.6f}"
    )
    print(
        f"  Median: {row['Median']:.6f}"
    )

# -----------------------------------------------------------------------------
# OBSERVATION QUALITY
# -----------------------------------------------------------------------------
obs_min = float(np.min(obs_v))
obs_max = float(np.max(obs_v))
obs_mean = float(np.mean(obs_v))
obs_median = float(np.median(obs_v))

one_obs_count = int(
    np.count_nonzero(
        obs_v == 1
    )
)

one_obs_pct = (
    one_obs_count /
    obs_v.size *
    100
)

print("\nCLEAR-OBSERVATION QUALITY")
print("-" * 112)

print(
    f"Minimum observations: "
    f"{obs_min:.0f}"
)

print(
    f"Maximum observations: "
    f"{obs_max:.0f}"
)

print(
    f"Mean observations: "
    f"{obs_mean:.3f}"
)

print(
    f"Median observations: "
    f"{obs_median:.3f}"
)

print(
    f"Pixels with only 1 clear observation: "
    f"{one_obs_count:,} "
    f"({one_obs_pct:.2f}%)"
)

# -----------------------------------------------------------------------------
# VALID AREA
# -----------------------------------------------------------------------------
valid_area_km2 = (
    ndvi_v.size *
    pixel_area_km2
)

area_difference_pct = (
    abs(
        valid_area_km2 -
        STATE_AREA_KM2
    )
    /
    STATE_AREA_KM2 *
    100
)

print("\nAREA CONSISTENCY")
print("-" * 112)

print(
    f"Validated state area: "
    f"{STATE_AREA_KM2:,.2f} km²"
)

print(
    f"Common valid Sentinel-2 area: "
    f"{valid_area_km2:,.2f} km²"
)

print(
    f"Difference: "
    f"{area_difference_pct:.4f}%"
)

# -----------------------------------------------------------------------------
# EARTH ENGINE VS LOCAL MEANS
# -----------------------------------------------------------------------------
local_means = {
    "NDVI_2025":
        float(np.mean(ndvi_v)),
    "NDBI_2025":
        float(np.mean(ndbi_v)),
    "MNDWI_2025":
        float(np.mean(mndwi_v))
}

print("\nEARTH ENGINE vs LOCAL MEANS")
print("-" * 112)

mean_differences = {}

for name in [
    "NDVI_2025",
    "NDBI_2025",
    "MNDWI_2025"
]:

    ee_mean = (
        EE_REFERENCE[name]["mean"]
    )

    local_mean = (
        local_means[name]
    )

    diff = abs(
        local_mean -
        ee_mean
    )

    mean_differences[name] = diff

    print(
        f"{name:<12} | "
        f"EE={ee_mean:.6f} | "
        f"Local={local_mean:.6f} | "
        f"Difference={diff:.6f}"
    )

print(
    f"\nObservation mean | "
    f"EE={EE_OBS_MEAN:.6f} | "
    f"Local={obs_mean:.6f} | "
    f"Difference="
    f"{abs(obs_mean - EE_OBS_MEAN):.6f}"
)

# -----------------------------------------------------------------------------
# SAVE STATISTICS
# -----------------------------------------------------------------------------
stats_df.to_csv(
    STATS_FILE,
    index=False
)

validation_df = pd.DataFrame([
    {
        "Dataset":
            "Sentinel-2 2025 Spectral Predictors",
        "Task_ID":
            S2_TASK_ID,
        "EPSG":
            epsg,
        "Resolution_m":
            res_x,
        "Valid_Common_Pixels":
            int(ndvi_v.size),
        "Valid_Area_km2":
            valid_area_km2,
        "State_Area_km2":
            STATE_AREA_KM2,
        "Area_Difference_pct":
            area_difference_pct,
        "NDVI_Min":
            float(ndvi_v.min()),
        "NDVI_Max":
            float(ndvi_v.max()),
        "NDVI_Mean":
            float(ndvi_v.mean()),
        "NDBI_Min":
            float(ndbi_v.min()),
        "NDBI_Max":
            float(ndbi_v.max()),
        "NDBI_Mean":
            float(ndbi_v.mean()),
        "MNDWI_Min":
            float(mndwi_v.min()),
        "MNDWI_Max":
            float(mndwi_v.max()),
        "MNDWI_Mean":
            float(mndwi_v.mean()),
        "Obs_Min":
            obs_min,
        "Obs_Max":
            obs_max,
        "Obs_Mean":
            obs_mean,
        "Obs_Median":
            obs_median,
        "One_Clear_Observation_pct":
            one_obs_pct
    }
])

validation_df.to_csv(
    VALIDATION_FILE,
    index=False
)

# -----------------------------------------------------------------------------
# FINAL CHECKS
# -----------------------------------------------------------------------------
checks = {
    "Earth Engine export completed":
        task_state == "COMPLETED",

    "Permanent GeoTIFF exists":
        TARGET_FILE.exists(),

    "CRS is EPSG:32632":
        epsg == 32632,

    "Resolution is 20 m":
        (
            np.isclose(res_x, 20)
            and
            np.isclose(res_y, 20)
        ),

    "Four bands present":
        (
            ndvi is not None
            and
            ndbi is not None
            and
            mndwi is not None
            and
            obs is not None
        ),

    "NDVI range valid":
        (
            ndvi_v.min() >= -1
            and
            ndvi_v.max() <= 1
        ),

    "NDBI range valid":
        (
            ndbi_v.min() >= -1
            and
            ndbi_v.max() <= 1
        ),

    "MNDWI range valid":
        (
            mndwi_v.min() >= -1
            and
            mndwi_v.max() <= 1
        ),

    "Valid Sentinel-2 area within 2% of state":
        area_difference_pct < 2,

    "Observation minimum >= 1":
        obs_min >= 1,

    "Observation maximum matches EE":
        obs_max == EE_OBS_MAX,

    "Observation mean close to EE":
        abs(
            obs_mean -
            EE_OBS_MEAN
        ) < 0.2,

    "NDVI local mean agrees with EE":
        mean_differences[
            "NDVI_2025"
        ] < 0.02,

    "NDBI local mean agrees with EE":
        mean_differences[
            "NDBI_2025"
        ] < 0.02,

    "MNDWI local mean agrees with EE":
        mean_differences[
            "MNDWI_2025"
        ] < 0.02,

    "Statistics table saved":
        STATS_FILE.exists(),

    "Validation table saved":
        VALIDATION_FILE.exists()
}

print("\nVALIDATION SUMMARY")
print("-" * 112)

for label, passed in checks.items():
    print(
        f"{'✓' if passed else '✗'} "
        f"{label}"
    )

print("\nOUTPUT FILES")
print("-" * 112)

for path in [
    TARGET_FILE,
    STATS_FILE,
    VALIDATION_FILE
]:
    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{path.name}"
    )

if all(checks.values()):

    print("\n✓ STAGE 4B FULLY PASSED")

    print(
        "NDVI, NDBI and MNDWI predictors are "
        "permanently stored and locally validated."
    )

else:

    print("\n⚠ STAGE 4B REQUIRES REVIEW")

print("=" * 112)

In [ ]:
# =====================================================================================
# PROJECT 7 — STAGE 5A
# OPENSTREETMAP ROAD NETWORK ACQUISITION + VALIDATION
#
# Source:
# OpenStreetMap via OSMnx / Overpass API
#
# Purpose:
# Build a verified road-accessibility dataset for later
# distance-to-road predictor generation.
# =====================================================================================

import os
import time
import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timezone

# -----------------------------------------------------------------------------
# PROJECT PATHS
# -----------------------------------------------------------------------------
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

BOUNDARY_DIR = PROJECT_ROOT / "01_Boundary"
ROAD_DIR = PROJECT_ROOT / "05_Roads_Accessibility"
ADMIN_DIR = PROJECT_ROOT / "00_Project_Admin"
TABLE_DIR = PROJECT_ROOT / "16_Tables"

ROAD_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

BOUNDARY_FILE = (
    BOUNDARY_DIR /
    "Enugu_State_Boundary_WGS84.gpkg"
)

ROAD_GPKG = (
    ROAD_DIR /
    "Enugu_OSM_Drive_Roads_2026.gpkg"
)

ROAD_SUMMARY_CSV = (
    TABLE_DIR /
    "Enugu_OSM_Road_Network_Summary.csv"
)

ROAD_CLASS_CSV = (
    TABLE_DIR /
    "Enugu_OSM_Road_Length_by_Class.csv"
)

# -----------------------------------------------------------------------------
# OSMNX SETTINGS
# -----------------------------------------------------------------------------
print("=" * 112)
print("STAGE 5A — OPENSTREETMAP ROAD NETWORK ACQUISITION")
print("=" * 112)

print("\nOSMnx version:", ox.__version__)

# Increase Overpass timeout because Enugu State is a relatively large polygon.
ox.settings.requests_timeout = 300

# Use cache so successful downloads do not need to be repeated.
ox.settings.use_cache = True

# Log requests to make acquisition reproducible.
ox.settings.log_console = False

# -----------------------------------------------------------------------------
# LOAD VALIDATED ENUGU STATE BOUNDARY
# -----------------------------------------------------------------------------
enugu = gpd.read_file(
    BOUNDARY_FILE
).to_crs("EPSG:4326")

if len(enugu) != 1:
    raise RuntimeError(
        f"Expected exactly one Enugu State feature, found {len(enugu)}."
    )

study_polygon = enugu.geometry.iloc[0]

print("\n✓ Validated Enugu State boundary loaded")
print("CRS:", enugu.crs)

# -----------------------------------------------------------------------------
# DOWNLOAD DRIVEABLE ROAD NETWORK
#
# retain_all=True prevents OSMnx from discarding valid disconnected road
# components in rural areas.
#
# truncate_by_edge=True preserves roads whose edge intersects the state boundary.
# -----------------------------------------------------------------------------
print("\nDOWNLOADING OPENSTREETMAP DRIVE NETWORK")
print("-" * 112)

print(
    "This is a large state-wide Overpass query and may take several minutes."
)

start_time = time.time()

G = ox.graph.graph_from_polygon(
    study_polygon,
    network_type="drive",
    simplify=True,
    retain_all=True,
    truncate_by_edge=True
)

elapsed_minutes = (
    time.time() - start_time
) / 60

print(
    f"✓ OSM road network downloaded in "
    f"{elapsed_minutes:.2f} minutes"
)

# -----------------------------------------------------------------------------
# BASIC GRAPH VALIDATION
# -----------------------------------------------------------------------------
node_count = len(G.nodes)
edge_count = len(G.edges)

print("\nRAW NETWORK")
print("-" * 112)

print(f"Graph nodes: {node_count:,}")
print(f"Graph directed edges: {edge_count:,}")

if node_count == 0 or edge_count == 0:
    raise RuntimeError(
        "The downloaded OpenStreetMap road network is empty."
    )

# -----------------------------------------------------------------------------
# CONVERT GRAPH TO GEODATAFRAMES
# -----------------------------------------------------------------------------
nodes, edges = ox.graph_to_gdfs(
    G,
    nodes=True,
    edges=True
)

print("Node CRS:", nodes.crs)
print("Edge CRS:", edges.crs)

# -----------------------------------------------------------------------------
# PROJECT TO ENUGU PROJECT CRS
# -----------------------------------------------------------------------------
nodes_utm = nodes.to_crs("EPSG:32632")
edges_utm = edges.to_crs("EPSG:32632")

enugu_utm = enugu.to_crs("EPSG:32632")

print("\n✓ Road network projected to EPSG:32632")

# -----------------------------------------------------------------------------
# CLIP EDGE GEOMETRIES STRICTLY TO ENUGU STATE
#
# This removes any edge portions retained just outside the polygon.
# -----------------------------------------------------------------------------
state_geom_utm = enugu_utm.geometry.union_all()

roads = edges_utm.copy()

roads["geometry"] = roads.geometry.intersection(
    state_geom_utm
)

roads = roads[
    ~roads.geometry.is_empty
].copy()

# Keep only line geometries
roads = roads[
    roads.geometry.geom_type.isin(
        [
            "LineString",
            "MultiLineString"
        ]
    )
].copy()

roads = roads.reset_index()

# -----------------------------------------------------------------------------
# CALCULATE ROAD LENGTH
# -----------------------------------------------------------------------------
roads["Length_m"] = roads.geometry.length
roads["Length_km"] = roads["Length_m"] / 1000

# Remove zero-length geometries
roads = roads[
    roads["Length_m"] > 0
].copy()

total_directed_length_km = float(
    roads["Length_km"].sum()
)

# -----------------------------------------------------------------------------
# IMPORTANT:
# OSMnx drive graph edges are directed. Two-way roads normally appear twice,
# once in each direction.
#
# For LAND-SUITABILITY distance-to-road analysis we require physical road
# geometries rather than directed routing arcs.
#
# We therefore construct a geometry-based undirected road layer.
# -----------------------------------------------------------------------------

def canonical_geometry_key(geom):
    """
    Produce an orientation-independent coordinate key so the same physical
    LineString represented in opposite directions is counted only once.
    """
    if geom.geom_type == "LineString":
        coords = tuple(
            (round(x, 3), round(y, 3))
            for x, y in geom.coords
        )

        reverse = tuple(reversed(coords))

        return min(coords, reverse)

    # MultiLineStrings are retained using WKB as fallback.
    return geom.wkb_hex


roads["geometry_key"] = roads.geometry.apply(
    canonical_geometry_key
)

physical_roads = (
    roads
    .drop_duplicates(
        subset=["geometry_key"]
    )
    .copy()
)

physical_roads.drop(
    columns=["geometry_key"],
    inplace=True
)

physical_roads["Length_m"] = (
    physical_roads.geometry.length
)

physical_roads["Length_km"] = (
    physical_roads["Length_m"] /
    1000
)

physical_road_length_km = float(
    physical_roads["Length_km"].sum()
)

# -----------------------------------------------------------------------------
# STANDARDISE HIGHWAY CLASS
#
# OSM highway values may be strings OR Python lists.
# -----------------------------------------------------------------------------
def normalize_highway(value):

    if isinstance(
        value,
        (list, tuple, set, np.ndarray)
    ):
        return str(value[0]) if len(value) else "unknown"

    if pd.isna(value):
        return "unknown"

    return str(value)

if "highway" in physical_roads.columns:

    physical_roads["Road_Class"] = (
        physical_roads["highway"]
        .apply(normalize_highway)
    )

else:

    physical_roads["Road_Class"] = "unknown"

# -----------------------------------------------------------------------------
# ROAD CLASS STATISTICS
# -----------------------------------------------------------------------------
road_class_summary = (
    physical_roads
    .groupby(
        "Road_Class",
        dropna=False
    )
    .agg(
        Segment_Count=("Length_km", "size"),
        Total_Length_km=("Length_km", "sum")
    )
    .reset_index()
    .sort_values(
        "Total_Length_km",
        ascending=False
    )
)

road_class_summary[
    "Percent_of_Road_Length"
] = (
    road_class_summary[
        "Total_Length_km"
    ]
    /
    physical_road_length_km
    *
    100
)

# -----------------------------------------------------------------------------
# NETWORK / GEOMETRY VALIDATION
# -----------------------------------------------------------------------------
invalid_count = int(
    (~physical_roads.geometry.is_valid).sum()
)

empty_count = int(
    physical_roads.geometry.is_empty.sum()
)

duplicate_geom_count = int(
    physical_roads.geometry
    .apply(lambda g: g.wkb_hex)
    .duplicated()
    .sum()
)

physical_segments = len(
    physical_roads
)

# Approximate road density against validated state area.
STATE_AREA_KM2 = 7625.21

road_density = (
    physical_road_length_km /
    STATE_AREA_KM2
)

# Bounding check:
# every retained road should intersect the Enugu polygon.
within_or_intersects = physical_roads.geometry.apply(
    lambda geom:
    geom.within(state_geom_utm)
    or
    geom.intersects(
        state_geom_utm.boundary
    )
)

outside_count = int(
    (~within_or_intersects).sum()
)

# -----------------------------------------------------------------------------
# DISPLAY SUMMARY
# -----------------------------------------------------------------------------
print("\nPHYSICAL ROAD NETWORK SUMMARY")
print("-" * 112)

print(
    f"Directed graph edges after clipping: "
    f"{len(roads):,}"
)

print(
    f"Physical road segments: "
    f"{physical_segments:,}"
)

print(
    f"Directed-edge length: "
    f"{total_directed_length_km:,.2f} km"
)

print(
    f"Deduplicated physical road length: "
    f"{physical_road_length_km:,.2f} km"
)

print(
    f"Road density: "
    f"{road_density:.3f} km/km²"
)

print(
    f"Invalid geometries: "
    f"{invalid_count}"
)

print(
    f"Empty geometries: "
    f"{empty_count}"
)

print(
    f"Exact duplicate geometries remaining: "
    f"{duplicate_geom_count}"
)

print(
    f"Roads outside state after clipping: "
    f"{outside_count}"
)

# -----------------------------------------------------------------------------
# DISPLAY ROAD CLASSES
# -----------------------------------------------------------------------------
print("\nROAD LENGTH BY OSM HIGHWAY CLASS")
print("-" * 112)

print(
    road_class_summary.to_string(
        index=False,
        formatters={
            "Total_Length_km":
                lambda x: f"{x:,.2f}",
            "Percent_of_Road_Length":
                lambda x: f"{x:.2f}%"
        }
    )
)

# -----------------------------------------------------------------------------
# SAVE ROAD DATA
# -----------------------------------------------------------------------------
# Keep useful OSM fields if present
candidate_fields = [
    "osmid",
    "name",
    "ref",
    "highway",
    "Road_Class",
    "oneway",
    "lanes",
    "maxspeed",
    "bridge",
    "tunnel",
    "Length_m",
    "Length_km",
    "geometry"
]

save_fields = [
    field
    for field in candidate_fields
    if field in physical_roads.columns
]

physical_roads[
    save_fields
].to_file(
    ROAD_GPKG,
    layer="Enugu_OSM_Drive_Roads",
    driver="GPKG"
)

# -----------------------------------------------------------------------------
# SAVE STATISTICS
# -----------------------------------------------------------------------------
road_class_summary.to_csv(
    ROAD_CLASS_CSV,
    index=False
)

summary_df = pd.DataFrame([
    {
        "Dataset":
            "OpenStreetMap driveable roads",
        "Acquisition_Date_UTC":
            datetime.now(
                timezone.utc
            ).strftime("%Y-%m-%d"),
        "OSMnx_Version":
            ox.__version__,
        "Network_Type":
            "drive",
        "Raw_Graph_Nodes":
            node_count,
        "Raw_Directed_Edges":
            edge_count,
        "Clipped_Directed_Edges":
            len(roads),
        "Physical_Road_Segments":
            physical_segments,
        "Physical_Road_Length_km":
            physical_road_length_km,
        "State_Area_km2":
            STATE_AREA_KM2,
        "Road_Density_km_per_km2":
            road_density,
        "Invalid_Geometries":
            invalid_count,
        "Empty_Geometries":
            empty_count,
        "Outside_State_Count":
            outside_count
    }
])

summary_df.to_csv(
    ROAD_SUMMARY_CSV,
    index=False
)

# -----------------------------------------------------------------------------
# UPDATE PROVENANCE REGISTER
# -----------------------------------------------------------------------------
provenance_file = (
    ADMIN_DIR /
    "Data_Provenance_Register.csv"
)

prov = pd.read_csv(
    provenance_file
)

record = {
    "Dataset":
        "Enugu OpenStreetMap Drive Road Network",

    "Provider":
        "OpenStreetMap contributors",

    "Source_URL_or_ID":
        "OpenStreetMap via OSMnx / Overpass API",

    "Acquisition_Date":
        datetime.now(
            timezone.utc
        ).strftime("%Y-%m-%d"),

    "Data_Date_or_Period":
        "Current OSM database at acquisition",

    "Spatial_Resolution":
        "Vector road centreline network",

    "CRS":
        "EPSG:4326 source; EPSG:32632 project output",

    "Licence":
        "Open Data Commons Open Database License (ODbL)",

    "Project_Use":
        "Road accessibility and distance-to-road predictor",

    "Validation_Status":
        "Validated",

    "Notes":
        (
            f"{physical_segments:,} physical road segments; "
            f"{physical_road_length_km:.2f} km total road length"
        )
}

prov = prov[
    prov["Dataset"].astype(str)
    != "Enugu OpenStreetMap Drive Road Network"
]

prov = pd.concat(
    [
        prov,
        pd.DataFrame([record])
    ],
    ignore_index=True
)

prov.to_csv(
    provenance_file,
    index=False
)

# -----------------------------------------------------------------------------
# FINAL VALIDATION
# -----------------------------------------------------------------------------
checks = {
    "OSM graph contains nodes":
        node_count > 0,

    "OSM graph contains edges":
        edge_count > 0,

    "Physical roads extracted":
        physical_segments > 0,

    "Road length positive":
        physical_road_length_km > 0,

    "CRS is EPSG:32632":
        physical_roads.crs.to_epsg() == 32632,

    "No invalid geometries":
        invalid_count == 0,

    "No empty geometries":
        empty_count == 0,

    "No roads outside Enugu after clipping":
        outside_count == 0,

    "Multiple road classes present":
        physical_roads[
            "Road_Class"
        ].nunique() >= 3,

    "Road GeoPackage saved":
        ROAD_GPKG.exists(),

    "Road summary CSV saved":
        ROAD_SUMMARY_CSV.exists(),

    "Road-class CSV saved":
        ROAD_CLASS_CSV.exists()
}

print("\nVALIDATION SUMMARY")
print("-" * 112)

for label, passed in checks.items():
    print(
        f"{'✓' if passed else '✗'} "
        f"{label}"
    )

print("\nOUTPUT FILES")
print("-" * 112)

for path in [
    ROAD_GPKG,
    ROAD_SUMMARY_CSV,
    ROAD_CLASS_CSV
]:
    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{path.name}"
    )

if all(checks.values()):

    print("\n✓ STAGE 5A PASSED")

    print(
        "OpenStreetMap road network has been "
        "acquired, cleaned, projected and validated."
    )

else:

    print("\n⚠ STAGE 5A REQUIRES REVIEW")

print("=" * 112)

In [ ]:
# =====================================================================================
# PROJECT 7 — STAGE 5A FIX
# INSTALL + VALIDATE OSMNX
# =====================================================================================

!pip -q install osmnx==2.1.1

import osmnx as ox
import geopandas as gpd
import shapely
import pandas as pd

print("=" * 90)
print("OSMNX INSTALLATION VALIDATION")
print("=" * 90)

print("OSMnx version:     ", ox.__version__)
print("GeoPandas version: ", gpd.__version__)
print("Shapely version:   ", shapely.__version__)
print("Pandas version:    ", pd.__version__)

# Basic functional test
assert hasattr(ox, "graph")
assert hasattr(ox.graph, "graph_from_polygon")

print("\n✓ OSMNX INSTALLATION PASSED")
print("Road-network acquisition can proceed.")
print("=" * 90)

In [ ]:
# =====================================================================================
# PROJECT 7 — STAGE 5B
# DISTANCE-TO-ROAD ACCESSIBILITY PREDICTOR
#
# Input:
#   Validated OSM physical road network
#
# Reference grid:
#   Enugu_Elevation_30m.tif
#
# Output:
#   Continuous Euclidean distance to nearest driveable road (metres)
# =====================================================================================

import geopandas as gpd
import rasterio
from rasterio.features import rasterize
from scipy.ndimage import distance_transform_edt
import numpy as np
import pandas as pd
from pathlib import Path

# -----------------------------------------------------------------------------
# PATHS
# -----------------------------------------------------------------------------
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

ROAD_FILE = (
    PROJECT_ROOT /
    "05_Roads_Accessibility" /
    "Enugu_OSM_Drive_Roads_2026.gpkg"
)

REFERENCE_FILE = (
    PROJECT_ROOT /
    "02_Terrain" /
    "Enugu_Elevation_30m.tif"
)

OUTPUT_FILE = (
    PROJECT_ROOT /
    "05_Roads_Accessibility" /
    "Enugu_Distance_to_Drive_Road_30m.tif"
)

STATS_FILE = (
    PROJECT_ROOT /
    "16_Tables" /
    "Enugu_Distance_to_Road_Statistics.csv"
)

VALIDATION_FILE = (
    PROJECT_ROOT /
    "05_Roads_Accessibility" /
    "Distance_to_Road_Validation.csv"
)

print("=" * 112)
print("STAGE 5B — DISTANCE-TO-ROAD ACCESSIBILITY PREDICTOR")
print("=" * 112)

# -----------------------------------------------------------------------------
# LOAD REFERENCE GRID
# -----------------------------------------------------------------------------
with rasterio.open(REFERENCE_FILE) as ref:

    profile = ref.profile.copy()

    transform = ref.transform
    crs = ref.crs

    width = ref.width
    height = ref.height

    elevation = ref.read(1)

    # Terrain exports use NaN outside Enugu.
    study_mask = np.isfinite(elevation)

    res_x = abs(transform.a)
    res_y = abs(transform.e)

    bounds = ref.bounds

print("\nREFERENCE GRID")
print("-" * 112)

print("CRS:", crs)
print("EPSG:", crs.to_epsg())

print(
    f"Dimensions: "
    f"{width:,} × {height:,}"
)

print(
    f"Resolution: "
    f"{res_x:.3f} × {res_y:.3f} m"
)

print(
    f"Study-area pixels: "
    f"{study_mask.sum():,}"
)

# -----------------------------------------------------------------------------
# LOAD ROADS
# -----------------------------------------------------------------------------
roads = gpd.read_file(
    ROAD_FILE
)

print("\nROAD INPUT")
print("-" * 112)

print(
    f"Road features loaded: "
    f"{len(roads):,}"
)

print("Input CRS:", roads.crs)

if roads.crs != crs:
    roads = roads.to_crs(crs)

print("Working CRS:", roads.crs)

if len(roads) == 0:
    raise RuntimeError(
        "Road layer is empty."
    )

# -----------------------------------------------------------------------------
# RASTERIZE ROADS
#
# all_touched=True is appropriate for centreline accessibility:
# every grid cell touched by a road is considered a road cell.
# -----------------------------------------------------------------------------
print("\nRASTERIZING ROAD NETWORK")
print("-" * 112)

road_raster = rasterize(
    (
        (geom, 1)
        for geom in roads.geometry
        if geom is not None
        and not geom.is_empty
    ),
    out_shape=(
        height,
        width
    ),
    transform=transform,
    fill=0,
    dtype="uint8",
    all_touched=True
)

road_cells = int(
    np.count_nonzero(
        road_raster == 1
    )
)

print(
    f"Road cells: "
    f"{road_cells:,}"
)

if road_cells == 0:
    raise RuntimeError(
        "No roads were rasterized onto the reference grid."
    )

# -----------------------------------------------------------------------------
# EUCLIDEAN DISTANCE TRANSFORM
#
# distance_transform_edt calculates distance from each non-road cell
# to the nearest zero cell.
#
# Therefore:
# road raster 1 = road
# invert to:
# 0 = road
# 1 = non-road
# -----------------------------------------------------------------------------
print("\nCALCULATING EUCLIDEAN DISTANCE")
print("-" * 112)

nonroad = (
    road_raster == 0
)

distance_m = distance_transform_edt(
    nonroad,
    sampling=(
        res_y,
        res_x
    )
).astype("float32")

# Outside study boundary -> NaN
distance_m[
    ~study_mask
] = np.nan

valid = distance_m[
    study_mask
]

print("✓ Distance transform completed")

# -----------------------------------------------------------------------------
# STATISTICS
# -----------------------------------------------------------------------------
stats = {
    "Valid_Pixels":
        int(valid.size),

    "Minimum_m":
        float(np.min(valid)),

    "Maximum_m":
        float(np.max(valid)),

    "Mean_m":
        float(np.mean(valid)),

    "StdDev_m":
        float(np.std(valid)),

    "P05_m":
        float(np.percentile(valid, 5)),

    "P25_m":
        float(np.percentile(valid, 25)),

    "Median_m":
        float(np.percentile(valid, 50)),

    "P75_m":
        float(np.percentile(valid, 75)),

    "P90_m":
        float(np.percentile(valid, 90)),

    "P95_m":
        float(np.percentile(valid, 95))
}

print("\nDISTANCE-TO-ROAD STATISTICS")
print("-" * 112)

print(
    f"Minimum:   "
    f"{stats['Minimum_m']:,.2f} m"
)

print(
    f"Maximum:   "
    f"{stats['Maximum_m']:,.2f} m"
)

print(
    f"Mean:      "
    f"{stats['Mean_m']:,.2f} m"
)

print(
    f"Median:    "
    f"{stats['Median_m']:,.2f} m"
)

print(
    f"Std Dev:   "
    f"{stats['StdDev_m']:,.2f} m"
)

print(
    f"90th pct:  "
    f"{stats['P90_m']:,.2f} m"
)

print(
    f"95th pct:  "
    f"{stats['P95_m']:,.2f} m"
)

# -----------------------------------------------------------------------------
# ACCESSIBILITY DESCRIPTIVE BANDS
#
# These are descriptive only.
# They are NOT ML suitability scores.
# -----------------------------------------------------------------------------
thresholds = [
    100,
    250,
    500,
    1000,
    2000,
    5000
]

threshold_records = []

print("\nPROXIMITY DISTRIBUTION")
print("-" * 112)

for threshold in thresholds:

    count = int(
        np.count_nonzero(
            valid <= threshold
        )
    )

    pct = (
        count /
        valid.size *
        100
    )

    threshold_records.append({
        "Threshold_m": threshold,
        "Pixel_Count": count,
        "Percent_of_Study_Area": pct
    })

    print(
        f"≤ {threshold:>5,} m : "
        f"{count:>10,} pixels | "
        f"{pct:6.2f}%"
    )

# -----------------------------------------------------------------------------
# ROAD-CELL ZERO-DISTANCE CHECK
# -----------------------------------------------------------------------------
road_inside = (
    (road_raster == 1)
    &
    study_mask
)

road_distance_values = (
    distance_m[
        road_inside
    ]
)

zero_road_pct = (
    np.count_nonzero(
        road_distance_values == 0
    )
    /
    road_distance_values.size
    *
    100
)

print("\nROAD-CELL VALIDATION")
print("-" * 112)

print(
    f"Road cells inside study area: "
    f"{road_distance_values.size:,}"
)

print(
    f"Road cells with distance = 0: "
    f"{zero_road_pct:.4f}%"
)

# -----------------------------------------------------------------------------
# SAVE RASTER
# -----------------------------------------------------------------------------
profile.update(
    dtype="float32",
    count=1,
    nodata=np.nan,
    compress="deflate",
    predictor=3
)

with rasterio.open(
    OUTPUT_FILE,
    "w",
    **profile
) as dst:

    dst.write(
        distance_m,
        1
    )

    dst.set_band_description(
        1,
        "Distance_to_Drive_Road_m"
    )

print("\n✓ Distance raster saved")

# -----------------------------------------------------------------------------
# SAVE STATISTICS
# -----------------------------------------------------------------------------
stats_df = pd.DataFrame([
    stats
])

stats_df.to_csv(
    STATS_FILE,
    index=False
)

threshold_df = pd.DataFrame(
    threshold_records
)

# Add threshold information to same statistics CSV in a separate companion file
threshold_file = (
    PROJECT_ROOT /
    "16_Tables" /
    "Enugu_Road_Proximity_Distribution.csv"
)

threshold_df.to_csv(
    threshold_file,
    index=False
)

# -----------------------------------------------------------------------------
# REOPEN OUTPUT FOR INDEPENDENT VALIDATION
# -----------------------------------------------------------------------------
with rasterio.open(
    OUTPUT_FILE
) as src:

    saved = src.read(1)

    saved_crs = src.crs
    saved_epsg = src.crs.to_epsg()

    saved_width = src.width
    saved_height = src.height

    saved_res_x = abs(
        src.transform.a
    )

    saved_res_y = abs(
        src.transform.e
    )

    saved_bounds = src.bounds

saved_valid = (
    saved[
        np.isfinite(saved)
    ]
)

# -----------------------------------------------------------------------------
# ALIGNMENT CHECKS
# -----------------------------------------------------------------------------
same_bounds = np.allclose(
    [
        bounds.left,
        bounds.bottom,
        bounds.right,
        bounds.top
    ],
    [
        saved_bounds.left,
        saved_bounds.bottom,
        saved_bounds.right,
        saved_bounds.top
    ],
    atol=0.001
)

same_dimensions = (
    saved_width == width
    and
    saved_height == height
)

same_resolution = (
    np.isclose(
        saved_res_x,
        res_x
    )
    and
    np.isclose(
        saved_res_y,
        res_y
    )
)

# -----------------------------------------------------------------------------
# VALIDATION TABLE
# -----------------------------------------------------------------------------
validation_df = pd.DataFrame([
    {
        "Dataset":
            "Enugu distance to driveable OSM road",

        "CRS":
            str(saved_crs),

        "EPSG":
            saved_epsg,

        "Width":
            saved_width,

        "Height":
            saved_height,

        "Resolution_X_m":
            saved_res_x,

        "Resolution_Y_m":
            saved_res_y,

        "Valid_Pixels":
            int(saved_valid.size),

        "Road_Cells":
            road_cells,

        "Minimum_Distance_m":
            float(saved_valid.min()),

        "Maximum_Distance_m":
            float(saved_valid.max()),

        "Mean_Distance_m":
            float(saved_valid.mean()),

        "Median_Distance_m":
            float(np.median(saved_valid)),

        "Road_Cells_Zero_Distance_pct":
            zero_road_pct,

        "Aligned_with_Terrain":
            (
                same_bounds
                and
                same_dimensions
                and
                same_resolution
            )
    }
])

validation_df.to_csv(
    VALIDATION_FILE,
    index=False
)

# -----------------------------------------------------------------------------
# FINAL VALIDATION
# -----------------------------------------------------------------------------
checks = {
    "Road input exists":
        ROAD_FILE.exists(),

    "Reference terrain exists":
        REFERENCE_FILE.exists(),

    "Road cells rasterized":
        road_cells > 0,

    "Distance raster contains valid pixels":
        saved_valid.size > 0,

    "Minimum distance is zero":
        np.isclose(
            saved_valid.min(),
            0
        ),

    "Maximum distance positive":
        saved_valid.max() > 0,

    "All road cells have zero distance":
        np.isclose(
            zero_road_pct,
            100
        ),

    "Output CRS is EPSG:32632":
        saved_epsg == 32632,

    "Dimensions match terrain":
        same_dimensions,

    "Resolution matches terrain":
        same_resolution,

    "Bounds match terrain":
        same_bounds,

    "Output raster saved":
        OUTPUT_FILE.exists(),

    "Statistics CSV saved":
        STATS_FILE.exists(),

    "Proximity table saved":
        threshold_file.exists(),

    "Validation CSV saved":
        VALIDATION_FILE.exists()
}

print("\nVALIDATION SUMMARY")
print("-" * 112)

for label, passed in checks.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{label}"
    )

print("\nOUTPUT FILES")
print("-" * 112)

for path in [
    OUTPUT_FILE,
    STATS_FILE,
    threshold_file,
    VALIDATION_FILE
]:

    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{path.name}"
    )

if all(checks.values()):

    print(
        "\n✓ STAGE 5B FULLY PASSED"
    )

    print(
        "Distance-to-road accessibility predictor "
        "is generated, aligned and validated."
    )

else:

    print(
        "\n⚠ STAGE 5B REQUIRES REVIEW"
    )

print("=" * 112)

In [ ]:
import os
# =====================================================================================
# PROJECT 7 — STAGE 6A
# SURFACE-WATER / DRAINAGE PROXIMITY PREDICTOR
#
# Source:
# JRC Global Surface Water
#
# Purpose:
# Create a validated distance-to-observed-surface-water raster.
# =====================================================================================

import ee
import geemap
import geopandas as gpd
from pathlib import Path

# -----------------------------------------------------------------------------
# SETTINGS
# -----------------------------------------------------------------------------
EE_PROJECT = os.environ.get("EE_PROJECT", "").strip()
if not EE_PROJECT:
    raise ValueError("Set EE_PROJECT before running this stage.")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

BOUNDARY_FILE = (
    PROJECT_ROOT /
    "01_Boundary" /
    "Enugu_State_Boundary_WGS84.gpkg"
)

DRIVE_FOLDER = "Project_7_Enugu_SurfaceWater"

print("=" * 112)
print("STAGE 6A — JRC SURFACE WATER ACQUISITION")
print("=" * 112)

# -----------------------------------------------------------------------------
# INITIALIZE EARTH ENGINE
# -----------------------------------------------------------------------------
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate(auth_mode="colab")
    ee.Initialize(project=EE_PROJECT)

print("\n✓ Earth Engine initialized")

# -----------------------------------------------------------------------------
# LOAD VALIDATED ENUGU BOUNDARY
# -----------------------------------------------------------------------------
enugu = gpd.read_file(
    BOUNDARY_FILE
).to_crs("EPSG:4326")

if len(enugu) != 1:
    raise RuntimeError(
        f"Expected one Enugu boundary feature, found {len(enugu)}"
    )

enugu_geom = (
    geemap
    .geopandas_to_ee(enugu)
    .geometry()
)

print("✓ Validated Enugu boundary loaded")

# -----------------------------------------------------------------------------
# JRC GLOBAL SURFACE WATER
# -----------------------------------------------------------------------------
JRC_ID = "JRC/GSW1_4/GlobalSurfaceWater"

jrc = ee.Image(JRC_ID)

print("\nDATASET")
print("-" * 112)

print("Dataset: JRC Global Surface Water")
print("Earth Engine ID:", JRC_ID)

# -----------------------------------------------------------------------------
# OCCURRENCE BAND
#
# occurrence = frequency of observed water presence (0-100%)
#
# We define recurring/perennial surface water conservatively as occurrence >= 50%.
# This is NOT a suitability score; it is a hydrological feature extraction threshold.
# -----------------------------------------------------------------------------
occurrence = (
    jrc
    .select("occurrence")
    .clip(enugu_geom)
    .rename("Water_Occurrence")
)

water_mask = (
    occurrence
    .gte(50)
    .selfMask()
    .rename("Recurring_Water")
    .toByte()
)

# -----------------------------------------------------------------------------
# SERVER-SIDE STATISTICS
# -----------------------------------------------------------------------------
occ_stats = occurrence.reduceRegion(
    reducer=(
        ee.Reducer.minMax()
        .combine(
            ee.Reducer.mean(),
            sharedInputs=True
        )
    ),
    geometry=enugu_geom,
    scale=30,
    maxPixels=1e9,
    tileScale=4
).getInfo()

water_area = (
    ee.Image.pixelArea()
    .updateMask(water_mask)
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=enugu_geom,
        scale=30,
        maxPixels=1e9,
        tileScale=4
    )
    .get("area")
    .getInfo()
)

water_area_km2 = (
    float(water_area) / 1_000_000
    if water_area is not None
    else 0
)

print("\nWATER OCCURRENCE STATISTICS")
print("-" * 112)

for key, value in occ_stats.items():
    print(f"{key}: {value}")

print(
    f"\nRecurring water area (occurrence >= 50%): "
    f"{water_area_km2:,.2f} km²"
)

# -----------------------------------------------------------------------------
# PREPARE EXPORT
#
# Band 1 = occurrence percentage
# Band 2 = recurring water mask
#
# Cast both to UInt16 for compatible export.
# -----------------------------------------------------------------------------
export_image = (
    occurrence
    .toUint16()
    .addBands(
        water_mask
        .unmask(0)
        .toUint16()
    )
)

band_types = export_image.bandTypes().getInfo()

print("\nBAND TYPE CHECK")
print("-" * 112)

for band, info in band_types.items():
    print(
        f"{band}: {info.get('precision')}"
    )

# -----------------------------------------------------------------------------
# EXPORT
# -----------------------------------------------------------------------------
print("\nSTARTING GOOGLE DRIVE EXPORT")
print("-" * 112)

task = ee.batch.Export.image.toDrive(
    image=export_image,
    description="Project7_Enugu_SurfaceWater",
    folder=DRIVE_FOLDER,
    fileNamePrefix="Enugu_JRC_SurfaceWater",
    region=enugu_geom,
    scale=30,
    crs="EPSG:32632",
    maxPixels=1e9,
    fileFormat="GeoTIFF",
    formatOptions={
        "cloudOptimized": True
    }
)

task.start()

status = task.status()

print(
    "Task ID:",
    status.get("id")
)

print(
    "Task state:",
    status.get("state")
)

print(
    "Drive folder:",
    f"MyDrive/{DRIVE_FOLDER}"
)

# -----------------------------------------------------------------------------
# VALIDATION
# -----------------------------------------------------------------------------
checks = {
    "Occurrence stats available":
        all(
            v is not None
            for v in occ_stats.values()
        ),

    "Occurrence minimum valid":
        (
            occ_stats["Water_Occurrence_min"] >= 0
        ),

    "Occurrence maximum valid":
        (
            occ_stats["Water_Occurrence_max"] <= 100
        ),

    "Recurring water area positive":
        water_area_km2 > 0,

    "Two compatible export bands":
        len(band_types) == 2,

    "Export started":
        status.get("state")
        in ["READY", "RUNNING"]
}

print("\nVALIDATION SUMMARY")
print("-" * 112)

for label, passed in checks.items():
    print(
        f"{'✓' if passed else '✗'} "
        f"{label}"
    )

print("\nIMPORTANT — SAVE THESE VALUES")
print("-" * 112)

print(
    "WATER_TASK_ID =",
    status.get("id")
)

print(
    "WATER_DRIVE_FOLDER =",
    DRIVE_FOLDER
)

if all(checks.values()):

    print("\n✓ STAGE 6A PASSED")

    print(
        "Surface-water layer is valid "
        "and export has started."
    )

else:

    print("\n⚠ STAGE 6A REQUIRES REVIEW")

print("=" * 112)

In [ ]:
import os
# =====================================================================================
# PROJECT 7 — STAGE 6B
# COMPLETE JRC EXPORT + CREATE DISTANCE-TO-SURFACE-WATER PREDICTOR
#
# Existing Earth Engine task only — DOES NOT create another export.
# =====================================================================================

import ee
import time
import shutil
import rasterio
from rasterio.warp import reproject, Resampling
from scipy.ndimage import distance_transform_edt
import numpy as np
import pandas as pd
from pathlib import Path

# -----------------------------------------------------------------------------
# SETTINGS
# -----------------------------------------------------------------------------
EE_PROJECT = os.environ.get("EE_PROJECT", "").strip()
if not EE_PROJECT:
    raise ValueError("Set EE_PROJECT before running this stage.")

WATER_TASK_ID = "NEN5DUZ5EK5ZZOOV42TYKNFV"

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

MYDRIVE = Path("/content/drive/MyDrive")

ENV_DIR = PROJECT_ROOT / "06_Environmental_Constraints"
TABLE_DIR = PROJECT_ROOT / "16_Tables"

ENV_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

REFERENCE_FILE = (
    PROJECT_ROOT /
    "02_Terrain" /
    "Enugu_Elevation_30m.tif"
)

TARGET_JRC = (
    ENV_DIR /
    "Enugu_JRC_SurfaceWater_30m.tif"
)

DISTANCE_FILE = (
    ENV_DIR /
    "Enugu_Distance_to_Recurring_SurfaceWater_30m.tif"
)

STATS_FILE = (
    TABLE_DIR /
    "Enugu_SurfaceWater_Distance_Statistics.csv"
)

VALIDATION_FILE = (
    ENV_DIR /
    "SurfaceWater_Distance_Validation.csv"
)

print("=" * 112)
print("STAGE 6B — SURFACE-WATER EXPORT + DISTANCE PREDICTOR")
print("=" * 112)

# -----------------------------------------------------------------------------
# INITIALIZE EARTH ENGINE
# -----------------------------------------------------------------------------
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate(auth_mode="colab")
    ee.Initialize(project=EE_PROJECT)

print("\n✓ Earth Engine initialized")

# -----------------------------------------------------------------------------
# WAIT FOR EXISTING EXPORT TASK
# -----------------------------------------------------------------------------
print("\nEARTH ENGINE EXPORT STATUS")
print("-" * 112)

task_state = None

for i in range(61):

    status = ee.data.getTaskStatus(
        WATER_TASK_ID
    )[0]

    task_state = status.get(
        "state",
        "UNKNOWN"
    )

    print(
        f"Check {i + 1:02d}: "
        f"{task_state}"
    )

    if task_state == "COMPLETED":
        print("\n✓ Surface-water export completed")
        break

    if task_state in [
        "FAILED",
        "CANCELLED"
    ]:
        raise RuntimeError(
            status.get(
                "error_message",
                f"Task state = {task_state}"
            )
        )

    if i == 60:
        raise TimeoutError(
            "Export still running after 20 minutes. "
            "Do NOT rerun Stage 6A."
        )

    time.sleep(20)

# -----------------------------------------------------------------------------
# LOCATE EXPORTED FILE
# -----------------------------------------------------------------------------
print("\nLOCATING EXPORTED GEOTIFF")
print("-" * 112)

matches = []

for pattern in [
    "Enugu_JRC_SurfaceWater*.tif",
    "Enugu_JRC_SurfaceWater*.tiff"
]:
    matches.extend(
        MYDRIVE.rglob(pattern)
    )

matches = [
    p for p in matches
    if p.resolve() != TARGET_JRC.resolve()
]

matches = list({
    str(p): p
    for p in matches
}.values())

print(
    "Candidate files found:",
    len(matches)
)

for path in matches:
    print(
        f" - {path} "
        f"({path.stat().st_size / 1024**2:.2f} MB)"
    )

if not matches:
    raise RuntimeError(
        "Task completed but the exported JRC file "
        "is not visible in mounted Drive."
    )

SOURCE_FILE = max(
    matches,
    key=lambda p: p.stat().st_mtime
)

print("\nSelected export:")
print(SOURCE_FILE)

# -----------------------------------------------------------------------------
# COPY TO PROJECT
# -----------------------------------------------------------------------------
shutil.copy2(
    SOURCE_FILE,
    TARGET_JRC
)

print("\n✓ Copied to:")
print(TARGET_JRC)

# -----------------------------------------------------------------------------
# LOAD REFERENCE GRID
# -----------------------------------------------------------------------------
with rasterio.open(
    REFERENCE_FILE
) as ref:

    reference = ref.read(1)

    ref_profile = ref.profile.copy()

    ref_transform = ref.transform
    ref_crs = ref.crs

    ref_width = ref.width
    ref_height = ref.height

    study_mask = np.isfinite(
        reference
    )

    res_x = abs(
        ref_transform.a
    )

    res_y = abs(
        ref_transform.e
    )

print("\nREFERENCE GRID")
print("-" * 112)

print("CRS:", ref_crs)
print(
    f"Dimensions: "
    f"{ref_width} × {ref_height}"
)
print(
    f"Resolution: "
    f"{res_x:.3f} × {res_y:.3f} m"
)

# -----------------------------------------------------------------------------
# READ JRC EXPORT
# -----------------------------------------------------------------------------
with rasterio.open(
    TARGET_JRC
) as src:

    print("\nJRC RASTER METADATA")
    print("-" * 112)

    print("CRS:", src.crs)
    print("EPSG:", src.crs.to_epsg())

    print(
        f"Dimensions: "
        f"{src.width} × {src.height}"
    )

    print(
        f"Resolution: "
        f"{abs(src.transform.a):.3f} × "
        f"{abs(src.transform.e):.3f} m"
    )

    print("Band count:", src.count)
    print("Data types:", src.dtypes)

    if src.count != 2:
        raise RuntimeError(
            f"Expected 2 JRC bands, found {src.count}"
        )

    # Stage 6A export order:
    # 1 Water_Occurrence
    # 2 Recurring_Water
    occurrence_src = src.read(1)
    water_src = src.read(2)

    src_transform = src.transform
    src_crs = src.crs

# -----------------------------------------------------------------------------
# RESAMPLE WATER MASK TO EXACT TERRAIN GRID
#
# Nearest-neighbour is mandatory for the binary mask.
# -----------------------------------------------------------------------------
water_aligned = np.zeros(
    (
        ref_height,
        ref_width
    ),
    dtype=np.uint8
)

reproject(
    source=water_src,
    destination=water_aligned,

    src_transform=src_transform,
    src_crs=src_crs,

    dst_transform=ref_transform,
    dst_crs=ref_crs,

    src_nodata=None,
    dst_nodata=0,

    resampling=Resampling.nearest
)

# Binary enforcement
water_aligned = (
    water_aligned > 0
).astype(
    np.uint8
)

# Only consider inside-study water cells
water_inside = (
    (water_aligned == 1)
    &
    study_mask
)

water_cells = int(
    water_inside.sum()
)

water_area_km2 = (
    water_cells *
    res_x *
    res_y /
    1_000_000
)

print("\nALIGNED RECURRING-WATER MASK")
print("-" * 112)

print(
    f"Water cells: "
    f"{water_cells:,}"
)

print(
    f"30 m water-cell area: "
    f"{water_area_km2:.4f} km²"
)

if water_cells == 0:
    raise RuntimeError(
        "No recurring-water cells remained after grid alignment."
    )

# -----------------------------------------------------------------------------
# DISTANCE TRANSFORM
#
# Water cells = 0 for EDT.
# Non-water cells = 1.
# -----------------------------------------------------------------------------
print("\nCALCULATING DISTANCE TO RECURRING SURFACE WATER")
print("-" * 112)

nonwater = (
    water_aligned == 0
)

distance = (
    distance_transform_edt(
        nonwater,
        sampling=(
            res_y,
            res_x
        )
    )
    .astype("float32")
)

distance[
    ~study_mask
] = np.nan

valid = distance[
    study_mask
]

print("✓ Distance calculation completed")

# -----------------------------------------------------------------------------
# STATISTICS
# -----------------------------------------------------------------------------
stats = {
    "Valid_Pixels":
        int(valid.size),

    "Minimum_m":
        float(np.min(valid)),

    "Maximum_m":
        float(np.max(valid)),

    "Mean_m":
        float(np.mean(valid)),

    "StdDev_m":
        float(np.std(valid)),

    "Median_m":
        float(
            np.median(valid)
        ),

    "P25_m":
        float(
            np.percentile(
                valid,
                25
            )
        ),

    "P75_m":
        float(
            np.percentile(
                valid,
                75
            )
        ),

    "P90_m":
        float(
            np.percentile(
                valid,
                90
            )
        ),

    "P95_m":
        float(
            np.percentile(
                valid,
                95
            )
        )
}

print("\nDISTANCE STATISTICS")
print("-" * 112)

print(
    f"Minimum:  "
    f"{stats['Minimum_m']:,.2f} m"
)

print(
    f"Maximum:  "
    f"{stats['Maximum_m']:,.2f} m"
)

print(
    f"Mean:     "
    f"{stats['Mean_m']:,.2f} m"
)

print(
    f"Median:   "
    f"{stats['Median_m']:,.2f} m"
)

print(
    f"90th pct: "
    f"{stats['P90_m']:,.2f} m"
)

print(
    f"95th pct: "
    f"{stats['P95_m']:,.2f} m"
)

# -----------------------------------------------------------------------------
# DESCRIPTIVE PROXIMITY DISTRIBUTION
# -----------------------------------------------------------------------------
thresholds = [
    100,
    250,
    500,
    1000,
    2000,
    5000,
    10000
]

records = []

print("\nSURFACE-WATER PROXIMITY")
print("-" * 112)

for threshold in thresholds:

    count = int(
        np.count_nonzero(
            valid <= threshold
        )
    )

    pct = (
        count /
        valid.size *
        100
    )

    records.append({
        "Threshold_m": threshold,
        "Pixel_Count": count,
        "Percent_of_Study_Area": pct
    })

    print(
        f"≤ {threshold:>6,} m : "
        f"{count:>10,} pixels | "
        f"{pct:6.2f}%"
    )

# -----------------------------------------------------------------------------
# ZERO-DISTANCE CHECK
# -----------------------------------------------------------------------------
water_distance = (
    distance[
        water_inside
    ]
)

zero_water_pct = (
    np.count_nonzero(
        water_distance == 0
    )
    /
    water_distance.size *
    100
)

print("\nWATER-CELL VALIDATION")
print("-" * 112)

print(
    f"Water cells with distance = 0: "
    f"{zero_water_pct:.4f}%"
)

# -----------------------------------------------------------------------------
# SAVE DISTANCE RASTER
# -----------------------------------------------------------------------------
out_profile = ref_profile.copy()

out_profile.update(
    dtype="float32",
    count=1,
    nodata=np.nan,
    compress="deflate",
    predictor=3
)

with rasterio.open(
    DISTANCE_FILE,
    "w",
    **out_profile
) as dst:

    dst.write(
        distance,
        1
    )

    dst.set_band_description(
        1,
        "Distance_to_Recurring_SurfaceWater_m"
    )

print("\n✓ Distance raster saved")

# -----------------------------------------------------------------------------
# SAVE TABLES
# -----------------------------------------------------------------------------
stats_df = pd.DataFrame([
    stats
])

stats_df[
    "Water_Cells"
] = water_cells

stats_df[
    "Water_Area_km2"
] = water_area_km2

stats_df.to_csv(
    STATS_FILE,
    index=False
)

proximity_file = (
    TABLE_DIR /
    "Enugu_SurfaceWater_Proximity_Distribution.csv"
)

pd.DataFrame(
    records
).to_csv(
    proximity_file,
    index=False
)

# -----------------------------------------------------------------------------
# INDEPENDENT OUTPUT VALIDATION
# -----------------------------------------------------------------------------
with rasterio.open(
    DISTANCE_FILE
) as src:

    saved = src.read(1)

    saved_epsg = src.crs.to_epsg()

    saved_width = src.width
    saved_height = src.height

    saved_res_x = abs(
        src.transform.a
    )

    saved_res_y = abs(
        src.transform.e
    )

saved_valid = (
    saved[
        np.isfinite(saved)
    ]
)

same_dimensions = (
    saved_width == ref_width
    and
    saved_height == ref_height
)

same_resolution = (
    np.isclose(
        saved_res_x,
        res_x
    )
    and
    np.isclose(
        saved_res_y,
        res_y
    )
)

# -----------------------------------------------------------------------------
# VALIDATION TABLE
# -----------------------------------------------------------------------------
validation_df = pd.DataFrame([
    {
        "Dataset":
            "Distance to recurring JRC surface water",

        "Task_ID":
            WATER_TASK_ID,

        "EPSG":
            saved_epsg,

        "Resolution_m":
            saved_res_x,

        "Width":
            saved_width,

        "Height":
            saved_height,

        "Water_Cells":
            water_cells,

        "Water_Area_km2":
            water_area_km2,

        "Valid_Distance_Pixels":
            int(
                saved_valid.size
            ),

        "Minimum_Distance_m":
            float(
                saved_valid.min()
            ),

        "Maximum_Distance_m":
            float(
                saved_valid.max()
            ),

        "Mean_Distance_m":
            float(
                saved_valid.mean()
            ),

        "Water_Cells_Zero_Distance_pct":
            zero_water_pct
    }
])

validation_df.to_csv(
    VALIDATION_FILE,
    index=False
)

# -----------------------------------------------------------------------------
# FINAL CHECKS
# -----------------------------------------------------------------------------
checks = {
    "Earth Engine export completed":
        task_state == "COMPLETED",

    "JRC raster stored":
        TARGET_JRC.exists(),

    "Recurring-water cells present":
        water_cells > 0,

    "Distance raster has valid pixels":
        saved_valid.size > 0,

    "Minimum distance = 0":
        np.isclose(
            saved_valid.min(),
            0
        ),

    "Maximum distance positive":
        saved_valid.max() > 0,

    "All water cells have zero distance":
        np.isclose(
            zero_water_pct,
            100
        ),

    "Output CRS EPSG:32632":
        saved_epsg == 32632,

    "Dimensions align with terrain":
        same_dimensions,

    "Resolution aligns with terrain":
        same_resolution,

    "Distance raster saved":
        DISTANCE_FILE.exists(),

    "Statistics saved":
        STATS_FILE.exists(),

    "Proximity table saved":
        proximity_file.exists(),

    "Validation file saved":
        VALIDATION_FILE.exists()
}

print("\nVALIDATION SUMMARY")
print("-" * 112)

for label, passed in checks.items():
    print(
        f"{'✓' if passed else '✗'} "
        f"{label}"
    )

print("\nOUTPUT FILES")
print("-" * 112)

for path in [
    TARGET_JRC,
    DISTANCE_FILE,
    STATS_FILE,
    proximity_file,
    VALIDATION_FILE
]:

    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{path.name}"
    )

if all(checks.values()):

    print("\n✓ STAGE 6B FULLY PASSED")

    print(
        "Surface-water proximity predictor is "
        "generated, aligned and validated."
    )

else:

    print("\n⚠ STAGE 6B REQUIRES REVIEW")

print("=" * 112)

In [ ]:
import os
# =====================================================================================
# PROJECT 7 — STAGE 6C
# MERIT HYDRO DRAINAGE NETWORK EXTRACTION
#
# Source:
# MERIT Hydro via Google Earth Engine
#
# Purpose:
# Create a physically meaningful drainage-network layer
# for later distance-to-drainage modelling.
# =====================================================================================

import ee
import geemap
import geopandas as gpd
from pathlib import Path

# -----------------------------------------------------------------------------
# SETTINGS
# -----------------------------------------------------------------------------
EE_PROJECT = os.environ.get("EE_PROJECT", "").strip()
if not EE_PROJECT:
    raise ValueError("Set EE_PROJECT before running this stage.")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

BOUNDARY_FILE = (
    PROJECT_ROOT /
    "01_Boundary" /
    "Enugu_State_Boundary_WGS84.gpkg"
)

DRIVE_FOLDER = "Project_7_Enugu_MERIT_Drainage"

MERIT_ID = "MERIT/Hydro/v1_0_1"

# Flow-accumulation threshold in upstream pixels.
# We will validate its resulting density before accepting it.
FLOW_ACC_THRESHOLD = 1000

print("=" * 112)
print("STAGE 6C — MERIT HYDRO DRAINAGE NETWORK EXTRACTION")
print("=" * 112)

# -----------------------------------------------------------------------------
# INITIALIZE
# -----------------------------------------------------------------------------
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate(auth_mode="colab")
    ee.Initialize(project=EE_PROJECT)

print("\n✓ Earth Engine initialized")

# -----------------------------------------------------------------------------
# LOAD ENUGU BOUNDARY
# -----------------------------------------------------------------------------
enugu = gpd.read_file(
    BOUNDARY_FILE
).to_crs("EPSG:4326")

if len(enugu) != 1:
    raise RuntimeError(
        f"Expected 1 Enugu feature, found {len(enugu)}"
    )

enugu_geom = (
    geemap
    .geopandas_to_ee(enugu)
    .geometry()
)

print("✓ Validated Enugu boundary loaded")

# -----------------------------------------------------------------------------
# LOAD MERIT HYDRO
# -----------------------------------------------------------------------------
merit = ee.Image(MERIT_ID)

print("\nDATASET")
print("-" * 112)

print("Dataset: MERIT Hydro")
print("Earth Engine ID:", MERIT_ID)

print("\nAvailable bands:")
print(merit.bandNames().getInfo())

# -----------------------------------------------------------------------------
# FLOW ACCUMULATION
#
# 'upa' = upstream drainage area in km² in MERIT Hydro.
# We use an upstream-area threshold, which is more interpretable
# than a raw pixel-count threshold.
# -----------------------------------------------------------------------------
upa = (
    merit
    .select("upa")
    .clip(enugu_geom)
    .rename("Upstream_Area_km2")
)

# For a regional planning analysis, use drainage channels with
# upstream contributing area >= 10 km².
UPSTREAM_AREA_THRESHOLD_KM2 = 10

drainage = (
    upa
    .gte(UPSTREAM_AREA_THRESHOLD_KM2)
    .selfMask()
    .rename("Drainage_Network")
    .toByte()
)

# -----------------------------------------------------------------------------
# SERVER-SIDE STATISTICS
# -----------------------------------------------------------------------------
upa_stats = upa.reduceRegion(
    reducer=(
        ee.Reducer.minMax()
        .combine(
            reducer2=ee.Reducer.mean(),
            sharedInputs=True
        )
    ),
    geometry=enugu_geom,
    scale=90,
    maxPixels=1e9,
    tileScale=4
).getInfo()

print("\nUPSTREAM AREA STATISTICS")
print("-" * 112)

for key, value in upa_stats.items():
    print(f"{key}: {value}")

# -----------------------------------------------------------------------------
# DRAINAGE PIXEL COUNT / AREA
# -----------------------------------------------------------------------------
drainage_count = drainage.reduceRegion(
    reducer=ee.Reducer.count(),
    geometry=enugu_geom,
    scale=90,
    maxPixels=1e9,
    tileScale=4
).getInfo()

drainage_pixels = drainage_count.get(
    "Drainage_Network",
    0
)

print("\nDRAINAGE EXTRACTION")
print("-" * 112)

print(
    f"Upstream-area threshold: "
    f"{UPSTREAM_AREA_THRESHOLD_KM2} km²"
)

print(
    f"Drainage pixels identified: "
    f"{drainage_pixels:,}"
)

# -----------------------------------------------------------------------------
# APPROXIMATE DRAINAGE LENGTH
#
# 90 m MERIT grid.
# Pixel count × 90 m is only a rough diagnostic, not final network length.
# -----------------------------------------------------------------------------
approx_length_km = (
    float(drainage_pixels) *
    90 /
    1000
)

STATE_AREA_KM2 = 7625.21

approx_drainage_density = (
    approx_length_km /
    STATE_AREA_KM2
)

print(
    f"Approximate raster-network length: "
    f"{approx_length_km:,.2f} km"
)

print(
    f"Approximate drainage density: "
    f"{approx_drainage_density:.3f} km/km²"
)

# -----------------------------------------------------------------------------
# PREPARE EXPORT
#
# Band 1 = upstream area
# Band 2 = binary drainage network
#
# Cast both to Float32 for compatible export.
# -----------------------------------------------------------------------------
export_image = (
    upa
    .toFloat()
    .addBands(
        drainage
        .unmask(0)
        .toFloat()
    )
)

band_types = export_image.bandTypes().getInfo()

print("\nBAND TYPE CHECK")
print("-" * 112)

for band, info in band_types.items():
    print(
        f"{band}: "
        f"{info.get('precision')}"
    )

# -----------------------------------------------------------------------------
# START EXPORT
# -----------------------------------------------------------------------------
print("\nSTARTING GOOGLE DRIVE EXPORT")
print("-" * 112)

task = ee.batch.Export.image.toDrive(
    image=export_image,
    description="Project7_Enugu_MERIT_Drainage",
    folder=DRIVE_FOLDER,
    fileNamePrefix="Enugu_MERIT_Drainage",
    region=enugu_geom,
    scale=90,
    crs="EPSG:32632",
    maxPixels=1e9,
    fileFormat="GeoTIFF",
    formatOptions={
        "cloudOptimized": True
    }
)

task.start()

status = task.status()

print(
    "Task ID:",
    status.get("id")
)

print(
    "Task state:",
    status.get("state")
)

print(
    "Drive folder:",
    f"MyDrive/{DRIVE_FOLDER}"
)

# -----------------------------------------------------------------------------
# VALIDATION
# -----------------------------------------------------------------------------
checks = {
    "Upstream-area statistics available":
        all(
            value is not None
            for value in upa_stats.values()
        ),

    "Drainage pixels identified":
        drainage_pixels > 0,

    "Approximate drainage length positive":
        approx_length_km > 0,

    "Drainage density positive":
        approx_drainage_density > 0,

    "Two export bands present":
        len(band_types) == 2,

    "Export task started":
        status.get("state")
        in ["READY", "RUNNING"]
}

print("\nVALIDATION SUMMARY")
print("-" * 112)

for label, passed in checks.items():
    print(
        f"{'✓' if passed else '✗'} "
        f"{label}"
    )

print("\nIMPORTANT — SAVE THESE VALUES")
print("-" * 112)

print(
    "MERIT_TASK_ID =",
    status.get("id")
)

print(
    "MERIT_DRIVE_FOLDER =",
    DRIVE_FOLDER
)

print(
    "UPSTREAM_AREA_THRESHOLD_KM2 =",
    UPSTREAM_AREA_THRESHOLD_KM2
)

if all(checks.values()):

    print("\n✓ STAGE 6C PASSED")

    print(
        "MERIT Hydro drainage network has been "
        "extracted and export has started."
    )

else:

    print("\n⚠ STAGE 6C REQUIRES REVIEW")

print("=" * 112)

In [ ]:
import os
# =====================================================================================
# PROJECT 7 — STAGE 6D
# MERIT DRAINAGE EXPORT COMPLETION + DISTANCE-TO-DRAINAGE PREDICTOR
#
# Existing Earth Engine task only — DOES NOT create another export.
# =====================================================================================

import ee
import time
import shutil
import rasterio
from rasterio.warp import reproject, Resampling
from scipy.ndimage import distance_transform_edt
import numpy as np
import pandas as pd
from pathlib import Path

# -----------------------------------------------------------------------------
# SETTINGS
# -----------------------------------------------------------------------------
EE_PROJECT = os.environ.get("EE_PROJECT", "").strip()
if not EE_PROJECT:
    raise ValueError("Set EE_PROJECT before running this stage.")
MERIT_TASK_ID = "NWYU5CPYLCQNI2SXEL3MHDHN"

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

MYDRIVE = Path("/content/drive/MyDrive")

ENV_DIR = PROJECT_ROOT / "06_Environmental_Constraints"
TABLE_DIR = PROJECT_ROOT / "16_Tables"

REFERENCE_FILE = (
    PROJECT_ROOT /
    "02_Terrain" /
    "Enugu_Elevation_30m.tif"
)

TARGET_MERIT = (
    ENV_DIR /
    "Enugu_MERIT_Drainage_90m.tif"
)

DISTANCE_FILE = (
    ENV_DIR /
    "Enugu_Distance_to_Drainage_30m.tif"
)

STATS_FILE = (
    TABLE_DIR /
    "Enugu_Distance_to_Drainage_Statistics.csv"
)

PROXIMITY_FILE = (
    TABLE_DIR /
    "Enugu_Drainage_Proximity_Distribution.csv"
)

VALIDATION_FILE = (
    ENV_DIR /
    "Distance_to_Drainage_Validation.csv"
)

print("=" * 112)
print("STAGE 6D — MERIT EXPORT + DISTANCE-TO-DRAINAGE PREDICTOR")
print("=" * 112)

# -----------------------------------------------------------------------------
# INITIALIZE EE
# -----------------------------------------------------------------------------
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate(auth_mode="colab")
    ee.Initialize(project=EE_PROJECT)

print("\n✓ Earth Engine initialized")

# -----------------------------------------------------------------------------
# WAIT FOR EXPORT
# -----------------------------------------------------------------------------
print("\nEARTH ENGINE EXPORT STATUS")
print("-" * 112)

task_state = None

for i in range(61):

    status = ee.data.getTaskStatus(
        MERIT_TASK_ID
    )[0]

    task_state = status.get(
        "state",
        "UNKNOWN"
    )

    print(
        f"Check {i + 1:02d}: {task_state}"
    )

    if task_state == "COMPLETED":
        print("\n✓ MERIT drainage export completed")
        break

    if task_state in ["FAILED", "CANCELLED"]:
        raise RuntimeError(
            status.get(
                "error_message",
                f"Task state = {task_state}"
            )
        )

    if i == 60:
        raise TimeoutError(
            "Export still running after 20 minutes. "
            "Do NOT rerun Stage 6C."
        )

    time.sleep(20)

# -----------------------------------------------------------------------------
# LOCATE EXPORTED FILE
# -----------------------------------------------------------------------------
print("\nLOCATING EXPORTED GEOTIFF")
print("-" * 112)

matches = []

for pattern in [
    "Enugu_MERIT_Drainage*.tif",
    "Enugu_MERIT_Drainage*.tiff"
]:
    matches.extend(
        MYDRIVE.rglob(pattern)
    )

matches = [
    p for p in matches
    if p.resolve() != TARGET_MERIT.resolve()
]

matches = list({
    str(p): p
    for p in matches
}.values())

print(
    "Candidate files found:",
    len(matches)
)

for path in matches:
    print(
        f" - {path} "
        f"({path.stat().st_size / 1024**2:.2f} MB)"
    )

if not matches:
    raise RuntimeError(
        "MERIT export completed but file is not visible in Drive."
    )

SOURCE_FILE = max(
    matches,
    key=lambda p: p.stat().st_mtime
)

print("\nSelected export:")
print(SOURCE_FILE)

# -----------------------------------------------------------------------------
# COPY TO PROJECT
# -----------------------------------------------------------------------------
shutil.copy2(
    SOURCE_FILE,
    TARGET_MERIT
)

print("\n✓ Copied to:")
print(TARGET_MERIT)

# -----------------------------------------------------------------------------
# LOAD REFERENCE 30 m GRID
# -----------------------------------------------------------------------------
with rasterio.open(REFERENCE_FILE) as ref:

    elevation = ref.read(1)

    ref_profile = ref.profile.copy()
    ref_transform = ref.transform
    ref_crs = ref.crs

    width = ref.width
    height = ref.height

    res_x = abs(ref_transform.a)
    res_y = abs(ref_transform.e)

    study_mask = np.isfinite(
        elevation
    )

    ref_bounds = ref.bounds

# -----------------------------------------------------------------------------
# READ MERIT EXPORT
# -----------------------------------------------------------------------------
with rasterio.open(TARGET_MERIT) as src:

    print("\nMERIT RASTER METADATA")
    print("-" * 112)

    print("CRS:", src.crs)
    print("EPSG:", src.crs.to_epsg())

    print(
        f"Dimensions: "
        f"{src.width} × {src.height}"
    )

    print(
        f"Resolution: "
        f"{abs(src.transform.a):.3f} × "
        f"{abs(src.transform.e):.3f} m"
    )

    print("Band count:", src.count)
    print("Data types:", src.dtypes)

    if src.count != 2:
        raise RuntimeError(
            f"Expected 2 MERIT bands, found {src.count}."
        )

    # Stage 6C export order:
    # 1 Upstream_Area_km2
    # 2 Drainage_Network
    upa_src = src.read(1)
    drainage_src = src.read(2)

    src_transform = src.transform
    src_crs = src.crs

# -----------------------------------------------------------------------------
# ALIGN BINARY DRAINAGE NETWORK TO 30 m REFERENCE GRID
# -----------------------------------------------------------------------------
drainage_aligned = np.zeros(
    (height, width),
    dtype=np.uint8
)

reproject(
    source=drainage_src,
    destination=drainage_aligned,

    src_transform=src_transform,
    src_crs=src_crs,

    dst_transform=ref_transform,
    dst_crs=ref_crs,

    src_nodata=None,
    dst_nodata=0,

    resampling=Resampling.nearest
)

drainage_aligned = (
    drainage_aligned > 0
).astype(np.uint8)

drainage_inside = (
    (drainage_aligned == 1)
    &
    study_mask
)

drainage_cells = int(
    drainage_inside.sum()
)

drainage_area_km2 = (
    drainage_cells *
    res_x *
    res_y /
    1_000_000
)

print("\nALIGNED DRAINAGE NETWORK")
print("-" * 112)

print(
    f"Drainage cells at 30 m: "
    f"{drainage_cells:,}"
)

print(
    f"Raster drainage-cell area: "
    f"{drainage_area_km2:,.2f} km²"
)

if drainage_cells == 0:
    raise RuntimeError(
        "No drainage cells survived grid alignment."
    )

# -----------------------------------------------------------------------------
# DISTANCE TRANSFORM
# -----------------------------------------------------------------------------
print("\nCALCULATING DISTANCE TO DRAINAGE")
print("-" * 112)

non_drainage = (
    drainage_aligned == 0
)

distance = distance_transform_edt(
    non_drainage,
    sampling=(
        res_y,
        res_x
    )
).astype("float32")

distance[
    ~study_mask
] = np.nan

valid = distance[
    study_mask
]

print("✓ Distance calculation completed")

# -----------------------------------------------------------------------------
# STATISTICS
# -----------------------------------------------------------------------------
stats = {
    "Valid_Pixels":
        int(valid.size),

    "Minimum_m":
        float(valid.min()),

    "Maximum_m":
        float(valid.max()),

    "Mean_m":
        float(valid.mean()),

    "StdDev_m":
        float(valid.std()),

    "P25_m":
        float(np.percentile(valid, 25)),

    "Median_m":
        float(np.median(valid)),

    "P75_m":
        float(np.percentile(valid, 75)),

    "P90_m":
        float(np.percentile(valid, 90)),

    "P95_m":
        float(np.percentile(valid, 95))
}

print("\nDISTANCE-TO-DRAINAGE STATISTICS")
print("-" * 112)

print(f"Minimum:  {stats['Minimum_m']:,.2f} m")
print(f"Maximum:  {stats['Maximum_m']:,.2f} m")
print(f"Mean:     {stats['Mean_m']:,.2f} m")
print(f"Median:   {stats['Median_m']:,.2f} m")
print(f"90th pct: {stats['P90_m']:,.2f} m")
print(f"95th pct: {stats['P95_m']:,.2f} m")

# -----------------------------------------------------------------------------
# DESCRIPTIVE PROXIMITY DISTRIBUTION
# -----------------------------------------------------------------------------
thresholds = [
    100,
    250,
    500,
    1000,
    2000,
    5000
]

records = []

print("\nDRAINAGE PROXIMITY")
print("-" * 112)

for threshold in thresholds:

    count = int(
        np.count_nonzero(
            valid <= threshold
        )
    )

    pct = (
        count /
        valid.size *
        100
    )

    records.append({
        "Threshold_m": threshold,
        "Pixel_Count": count,
        "Percent_of_Study_Area": pct
    })

    print(
        f"≤ {threshold:>5,} m : "
        f"{count:>10,} pixels | "
        f"{pct:6.2f}%"
    )

# -----------------------------------------------------------------------------
# ZERO-DISTANCE CHECK
# -----------------------------------------------------------------------------
drainage_distances = (
    distance[
        drainage_inside
    ]
)

zero_pct = (
    np.count_nonzero(
        drainage_distances == 0
    )
    /
    drainage_distances.size *
    100
)

print("\nDRAINAGE-CELL VALIDATION")
print("-" * 112)

print(
    f"Drainage cells with distance = 0: "
    f"{zero_pct:.4f}%"
)

# -----------------------------------------------------------------------------
# SAVE DISTANCE RASTER
# -----------------------------------------------------------------------------
out_profile = ref_profile.copy()

out_profile.update(
    dtype="float32",
    count=1,
    nodata=np.nan,
    compress="deflate",
    predictor=3
)

with rasterio.open(
    DISTANCE_FILE,
    "w",
    **out_profile
) as dst:

    dst.write(
        distance,
        1
    )

    dst.set_band_description(
        1,
        "Distance_to_MERIT_Drainage_m"
    )

print("\n✓ Distance raster saved")

# -----------------------------------------------------------------------------
# SAVE TABLES
# -----------------------------------------------------------------------------
stats_df = pd.DataFrame([
    {
        **stats,
        "Drainage_Cells_30m":
            drainage_cells,
        "Drainage_Cell_Area_km2":
            drainage_area_km2
    }
])

stats_df.to_csv(
    STATS_FILE,
    index=False
)

pd.DataFrame(
    records
).to_csv(
    PROXIMITY_FILE,
    index=False
)

# -----------------------------------------------------------------------------
# REOPEN OUTPUT FOR INDEPENDENT VALIDATION
# -----------------------------------------------------------------------------
with rasterio.open(
    DISTANCE_FILE
) as src:

    saved = src.read(1)

    saved_epsg = (
        src.crs.to_epsg()
    )

    saved_width = src.width
    saved_height = src.height

    saved_res_x = abs(
        src.transform.a
    )

    saved_res_y = abs(
        src.transform.e
    )

    saved_bounds = src.bounds

saved_valid = saved[
    np.isfinite(saved)
]

same_dimensions = (
    saved_width == width
    and
    saved_height == height
)

same_resolution = (
    np.isclose(
        saved_res_x,
        res_x
    )
    and
    np.isclose(
        saved_res_y,
        res_y
    )
)

same_bounds = np.allclose(
    [
        saved_bounds.left,
        saved_bounds.bottom,
        saved_bounds.right,
        saved_bounds.top
    ],
    [
        ref_bounds.left,
        ref_bounds.bottom,
        ref_bounds.right,
        ref_bounds.top
    ],
    atol=0.001
)

# -----------------------------------------------------------------------------
# VALIDATION FILE
# -----------------------------------------------------------------------------
validation_df = pd.DataFrame([
    {
        "Dataset":
            "Distance to MERIT drainage network",

        "Task_ID":
            MERIT_TASK_ID,

        "EPSG":
            saved_epsg,

        "Resolution_m":
            saved_res_x,

        "Width":
            saved_width,

        "Height":
            saved_height,

        "Drainage_Cells_30m":
            drainage_cells,

        "Valid_Distance_Pixels":
            int(saved_valid.size),

        "Minimum_Distance_m":
            float(saved_valid.min()),

        "Maximum_Distance_m":
            float(saved_valid.max()),

        "Mean_Distance_m":
            float(saved_valid.mean()),

        "Median_Distance_m":
            float(np.median(saved_valid)),

        "Drainage_Cells_Zero_Distance_pct":
            zero_pct,

        "Aligned_with_Terrain":
            (
                same_dimensions
                and
                same_resolution
                and
                same_bounds
            )
    }
])

validation_df.to_csv(
    VALIDATION_FILE,
    index=False
)

# -----------------------------------------------------------------------------
# FINAL VALIDATION
# -----------------------------------------------------------------------------
checks = {
    "Earth Engine export completed":
        task_state == "COMPLETED",

    "MERIT raster stored":
        TARGET_MERIT.exists(),

    "Drainage cells present":
        drainage_cells > 0,

    "Distance raster contains valid pixels":
        saved_valid.size > 0,

    "Minimum distance = 0":
        np.isclose(
            saved_valid.min(),
            0
        ),

    "Maximum distance positive":
        saved_valid.max() > 0,

    "All drainage cells have zero distance":
        np.isclose(
            zero_pct,
            100
        ),

    "Output CRS is EPSG:32632":
        saved_epsg == 32632,

    "Dimensions match terrain":
        same_dimensions,

    "Resolution matches terrain":
        same_resolution,

    "Bounds match terrain":
        same_bounds,

    "Distance raster saved":
        DISTANCE_FILE.exists(),

    "Statistics saved":
        STATS_FILE.exists(),

    "Proximity table saved":
        PROXIMITY_FILE.exists(),

    "Validation file saved":
        VALIDATION_FILE.exists()
}

print("\nVALIDATION SUMMARY")
print("-" * 112)

for label, passed in checks.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{label}"
    )

print("\nOUTPUT FILES")
print("-" * 112)

for path in [
    TARGET_MERIT,
    DISTANCE_FILE,
    STATS_FILE,
    PROXIMITY_FILE,
    VALIDATION_FILE
]:

    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{path.name}"
    )

if all(checks.values()):

    print("\n✓ STAGE 6D FULLY PASSED")

    print(
        "Distance-to-drainage predictor is "
        "generated, aligned and validated."
    )

else:

    print("\n⚠ STAGE 6D REQUIRES REVIEW")

print("=" * 112)

In [ ]:
import os
# =====================================================================================
# PROJECT 7 — STAGE 7A
# WORLDPOP 2020 POPULATION PRESSURE ACQUISITION
#
# Source:
# WorldPop GP 100 m population
#
# Purpose:
# Add a real socio-spatial population-pressure predictor.
#
# Important:
# 2020 is used explicitly as a baseline year.
# It is NOT presented as 2025 population.
# =====================================================================================

import ee
import geemap
import geopandas as gpd
from pathlib import Path

# -----------------------------------------------------------------------------
# SETTINGS
# -----------------------------------------------------------------------------
EE_PROJECT = os.environ.get("EE_PROJECT", "").strip()
if not EE_PROJECT:
    raise ValueError("Set EE_PROJECT before running this stage.")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

BOUNDARY_FILE = (
    PROJECT_ROOT /
    "01_Boundary" /
    "Enugu_State_Boundary_WGS84.gpkg"
)

WORLDPOP_ID = "WorldPop/GP/100m/pop"
TARGET_YEAR = 2020

DRIVE_FOLDER = "Project_7_Enugu_WorldPop_2020"

print("=" * 112)
print("STAGE 7A — WORLDPOP 2020 POPULATION ACQUISITION")
print("=" * 112)

# -----------------------------------------------------------------------------
# INITIALIZE EARTH ENGINE
# -----------------------------------------------------------------------------
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate(auth_mode="colab")
    ee.Initialize(project=EE_PROJECT)

print("\n✓ Earth Engine initialized")
print("Cloud Project:", EE_PROJECT)

# -----------------------------------------------------------------------------
# LOAD VALIDATED ENUGU BOUNDARY
# -----------------------------------------------------------------------------
enugu = gpd.read_file(
    BOUNDARY_FILE
).to_crs("EPSG:4326")

if len(enugu) != 1:
    raise RuntimeError(
        f"Expected one Enugu State feature, found {len(enugu)}."
    )

enugu_geom = (
    geemap
    .geopandas_to_ee(enugu)
    .geometry()
)

print("✓ Validated Enugu boundary loaded")

# -----------------------------------------------------------------------------
# LOAD WORLDPOP 2020 NIGERIA IMAGE
# -----------------------------------------------------------------------------
collection = (
    ee.ImageCollection(WORLDPOP_ID)
    .filter(
        ee.Filter.eq(
            "country",
            "NGA"
        )
    )
    .filter(
        ee.Filter.eq(
            "year",
            TARGET_YEAR
        )
    )
)

image_count = collection.size().getInfo()

print("\nDATASET")
print("-" * 112)

print("Dataset: WorldPop GP 100m Population")
print("Earth Engine ID:", WORLDPOP_ID)
print("Country filter: NGA")
print("Target year:", TARGET_YEAR)
print("Matching images:", image_count)

if image_count != 1:
    raise RuntimeError(
        f"Expected exactly one Nigeria WorldPop image for {TARGET_YEAR}, "
        f"found {image_count}."
    )

population = (
    ee.Image(
        collection.first()
    )
    .select("population")
    .clip(enugu_geom)
    .rename("Population_2020")
)

# -----------------------------------------------------------------------------
# SERVER-SIDE POPULATION STATISTICS
# -----------------------------------------------------------------------------
population_stats = population.reduceRegion(
    reducer=(
        ee.Reducer.minMax()
        .combine(
            reducer2=ee.Reducer.mean(),
            sharedInputs=True
        )
        .combine(
            reducer2=ee.Reducer.stdDev(),
            sharedInputs=True
        )
    ),
    geometry=enugu_geom,
    scale=100,
    maxPixels=1e9,
    tileScale=4
).getInfo()

# Total estimated population
population_total = population.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=enugu_geom,
    scale=100,
    maxPixels=1e9,
    tileScale=4
).getInfo()

total_pop = population_total.get(
    "Population_2020"
)

if total_pop is None:
    raise RuntimeError(
        "WorldPop total population calculation returned no value."
    )

total_pop = float(total_pop)

print("\nWORLDPOP STATISTICS")
print("-" * 112)

for key, value in population_stats.items():
    print(f"{key}: {value}")

print(
    f"\nEstimated Enugu population represented "
    f"by clipped WorldPop 2020 raster: "
    f"{total_pop:,.0f}"
)

# -----------------------------------------------------------------------------
# POPULATION DENSITY
#
# WorldPop stores estimated people per raster cell.
# We also derive people/km² for a more interpretable continuous predictor.
#
# Earth Engine pixelArea is used so density is not based on an assumed
# constant pixel size.
# -----------------------------------------------------------------------------
pixel_area_km2 = (
    ee.Image.pixelArea()
    .divide(1_000_000)
)

population_density = (
    population
    .divide(pixel_area_km2)
    .rename("Population_Density_2020")
)

density_stats = population_density.reduceRegion(
    reducer=(
        ee.Reducer.minMax()
        .combine(
            reducer2=ee.Reducer.mean(),
            sharedInputs=True
        )
        .combine(
            reducer2=ee.Reducer.stdDev(),
            sharedInputs=True
        )
    ),
    geometry=enugu_geom,
    scale=100,
    maxPixels=1e9,
    tileScale=4
).getInfo()

print("\nPOPULATION-DENSITY STATISTICS")
print("-" * 112)

for key, value in density_stats.items():
    print(f"{key}: {value}")

# -----------------------------------------------------------------------------
# DESCRIPTIVE POPULATION CONCENTRATION
# -----------------------------------------------------------------------------
density_thresholds = [
    100,
    250,
    500,
    1000,
    2500,
    5000
]

print("\nPOPULATION-DENSITY COVERAGE")
print("-" * 112)

density_records = []

study_area_m2 = (
    ee.Image.pixelArea()
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=enugu_geom,
        scale=100,
        maxPixels=1e9,
        tileScale=4
    )
    .get("area")
    .getInfo()
)

study_area_km2 = float(
    study_area_m2
) / 1_000_000

for threshold in density_thresholds:

    mask = population_density.gte(
        threshold
    )

    area = (
        ee.Image.pixelArea()
        .updateMask(mask)
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=enugu_geom,
            scale=100,
            maxPixels=1e9,
            tileScale=4
        )
        .get("area")
        .getInfo()
    )

    area_km2 = (
        float(area) / 1_000_000
        if area is not None
        else 0
    )

    pct = (
        area_km2 /
        study_area_km2 *
        100
    )

    density_records.append(
        (
            threshold,
            area_km2,
            pct
        )
    )

    print(
        f"≥ {threshold:>5,} persons/km² : "
        f"{area_km2:>9,.2f} km² | "
        f"{pct:6.2f}%"
    )

# -----------------------------------------------------------------------------
# PREPARE COMPATIBLE EXPORT
#
# Both bands cast to Float32.
# -----------------------------------------------------------------------------
export_image = (
    population
    .toFloat()
    .addBands(
        population_density
        .toFloat()
    )
)

band_types = (
    export_image
    .bandTypes()
    .getInfo()
)

print("\nEXPORT BAND TYPE CHECK")
print("-" * 112)

for band, info in band_types.items():
    print(
        f"{band}: "
        f"{info.get('precision')}"
    )

# -----------------------------------------------------------------------------
# EXPORT
# -----------------------------------------------------------------------------
print("\nSTARTING GOOGLE DRIVE EXPORT")
print("-" * 112)

task = ee.batch.Export.image.toDrive(
    image=export_image,
    description="Project7_Enugu_WorldPop_2020",
    folder=DRIVE_FOLDER,
    fileNamePrefix="Enugu_WorldPop_2020",
    region=enugu_geom,
    scale=100,
    crs="EPSG:32632",
    maxPixels=1e9,
    fileFormat="GeoTIFF",
    formatOptions={
        "cloudOptimized": True
    }
)

task.start()

status = task.status()

print(
    "Task ID:",
    status.get("id")
)

print(
    "Task state:",
    status.get("state")
)

print(
    "Drive folder:",
    f"MyDrive/{DRIVE_FOLDER}"
)

# -----------------------------------------------------------------------------
# VALIDATION
# -----------------------------------------------------------------------------
population_values_ok = (
    population_stats[
        "Population_2020_min"
    ] >= 0
    and
    population_stats[
        "Population_2020_max"
    ] > 0
)

density_values_ok = (
    density_stats[
        "Population_Density_2020_min"
    ] >= 0
    and
    density_stats[
        "Population_Density_2020_max"
    ] > 0
)

checks = {
    "Exactly one Nigeria 2020 image found":
        image_count == 1,

    "Population statistics available":
        all(
            value is not None
            for value in population_stats.values()
        ),

    "Population values non-negative":
        population_values_ok,

    "Population total positive":
        total_pop > 0,

    "Density statistics available":
        all(
            value is not None
            for value in density_stats.values()
        ),

    "Density values non-negative":
        density_values_ok,

    "Study-area calculation positive":
        study_area_km2 > 0,

    "Two compatible export bands":
        len(band_types) == 2,

    "Export task started":
        status.get("state")
        in ["READY", "RUNNING"]
}

print("\nVALIDATION SUMMARY")
print("-" * 112)

for label, passed in checks.items():
    print(
        f"{'✓' if passed else '✗'} "
        f"{label}"
    )

print("\nIMPORTANT — SAVE THESE VALUES")
print("-" * 112)

print(
    "WORLDPOP_TASK_ID =",
    status.get("id")
)

print(
    "WORLDPOP_DRIVE_FOLDER =",
    DRIVE_FOLDER
)

print(
    "WORLDPOP_2020_TOTAL =",
    f"{total_pop:.3f}"
)

if all(checks.values()):

    print("\n✓ STAGE 7A PASSED")

    print(
        "WorldPop 2020 population-pressure data "
        "is valid and export has started."
    )

else:

    print("\n⚠ STAGE 7A REQUIRES REVIEW")

print("=" * 112)

In [ ]:
import os
# =====================================================================================
# PROJECT 7 — STAGE 7B
# WORLDPOP EXPORT COMPLETION + 30 m GRID ALIGNMENT
#
# Existing Earth Engine task only — DOES NOT create another export.
#
# Outputs:
# 1. Original WorldPop 100 m raster
# 2. Population-density predictor aligned to terrain 30 m grid
# 3. Validation tables
# =====================================================================================

import ee
import time
import shutil
import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np
import pandas as pd
from pathlib import Path

# -----------------------------------------------------------------------------
# SETTINGS
# -----------------------------------------------------------------------------
EE_PROJECT = os.environ.get("EE_PROJECT", "").strip()
if not EE_PROJECT:
    raise ValueError("Set EE_PROJECT before running this stage.")

WORLDPOP_TASK_ID = "AEWJYHKNZAKINK7ICX4ZAYPQ"

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

MYDRIVE = Path("/content/drive/MyDrive")

POP_DIR = PROJECT_ROOT / "07_Population_Urban_Pressure"
TABLE_DIR = PROJECT_ROOT / "16_Tables"

POP_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

REFERENCE_FILE = (
    PROJECT_ROOT /
    "02_Terrain" /
    "Enugu_Elevation_30m.tif"
)

TARGET_WORLDPOP = (
    POP_DIR /
    "Enugu_WorldPop_2020_100m.tif"
)

DENSITY_30M_FILE = (
    POP_DIR /
    "Enugu_Population_Density_2020_30m.tif"
)

VALIDATION_FILE = (
    POP_DIR /
    "WorldPop_2020_Validation.csv"
)

STATS_FILE = (
    TABLE_DIR /
    "Enugu_Population_Density_2020_Statistics.csv"
)

EE_TOTAL_POP = 4083971.941
EE_DENSITY_MEAN = 535.6354579449735

print("=" * 112)
print("STAGE 7B — WORLDPOP EXPORT + 30 m ALIGNMENT")
print("=" * 112)

# -----------------------------------------------------------------------------
# INITIALIZE EARTH ENGINE
# -----------------------------------------------------------------------------
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate(auth_mode="colab")
    ee.Initialize(project=EE_PROJECT)

print("\n✓ Earth Engine initialized")

# -----------------------------------------------------------------------------
# WAIT FOR EXISTING EXPORT
# -----------------------------------------------------------------------------
print("\nEARTH ENGINE EXPORT STATUS")
print("-" * 112)

task_state = None

for i in range(61):

    status = ee.data.getTaskStatus(
        WORLDPOP_TASK_ID
    )[0]

    task_state = status.get(
        "state",
        "UNKNOWN"
    )

    print(
        f"Check {i + 1:02d}: "
        f"{task_state}"
    )

    if task_state == "COMPLETED":
        print("\n✓ WorldPop export completed")
        break

    if task_state in [
        "FAILED",
        "CANCELLED"
    ]:
        raise RuntimeError(
            status.get(
                "error_message",
                f"Task state = {task_state}"
            )
        )

    if i == 60:
        raise TimeoutError(
            "WorldPop export still running after 20 minutes. "
            "Do NOT rerun Stage 7A."
        )

    time.sleep(20)

# -----------------------------------------------------------------------------
# LOCATE EXPORT
# -----------------------------------------------------------------------------
print("\nLOCATING EXPORTED GEOTIFF")
print("-" * 112)

matches = []

for pattern in [
    "Enugu_WorldPop_2020*.tif",
    "Enugu_WorldPop_2020*.tiff"
]:
    matches.extend(
        MYDRIVE.rglob(pattern)
    )

matches = [
    p for p in matches
    if p.resolve() != TARGET_WORLDPOP.resolve()
]

matches = list({
    str(p): p
    for p in matches
}.values())

print(
    "Candidate files found:",
    len(matches)
)

for path in matches:
    print(
        f" - {path} "
        f"({path.stat().st_size / 1024**2:.2f} MB)"
    )

if not matches:
    raise RuntimeError(
        "Earth Engine reports COMPLETED, but the WorldPop "
        "GeoTIFF is not visible in mounted Google Drive."
    )

SOURCE_FILE = max(
    matches,
    key=lambda p: p.stat().st_mtime
)

print("\nSelected export:")
print(SOURCE_FILE)

# -----------------------------------------------------------------------------
# COPY ORIGINAL 100 m RASTER
# -----------------------------------------------------------------------------
shutil.copy2(
    SOURCE_FILE,
    TARGET_WORLDPOP
)

print("\n✓ WorldPop source raster copied to:")
print(TARGET_WORLDPOP)

# -----------------------------------------------------------------------------
# LOAD REFERENCE 30 m GRID
# -----------------------------------------------------------------------------
with rasterio.open(REFERENCE_FILE) as ref:

    elevation = ref.read(1)

    ref_profile = ref.profile.copy()
    ref_transform = ref.transform
    ref_crs = ref.crs

    ref_width = ref.width
    ref_height = ref.height
    ref_bounds = ref.bounds

    study_mask = np.isfinite(
        elevation
    )

    ref_res_x = abs(
        ref_transform.a
    )

    ref_res_y = abs(
        ref_transform.e
    )

print("\nREFERENCE GRID")
print("-" * 112)

print("CRS:", ref_crs)
print(
    f"Dimensions: "
    f"{ref_width} × {ref_height}"
)
print(
    f"Resolution: "
    f"{ref_res_x:.3f} × "
    f"{ref_res_y:.3f} m"
)

# -----------------------------------------------------------------------------
# READ WORLDPOP EXPORT
# -----------------------------------------------------------------------------
with rasterio.open(TARGET_WORLDPOP) as src:

    print("\nWORLDPOP RASTER METADATA")
    print("-" * 112)

    print("CRS:", src.crs)
    print("EPSG:", src.crs.to_epsg())

    print(
        f"Dimensions: "
        f"{src.width} × {src.height}"
    )

    print(
        f"Resolution: "
        f"{abs(src.transform.a):.3f} × "
        f"{abs(src.transform.e):.3f} m"
    )

    print("Band count:", src.count)
    print("Data types:", src.dtypes)

    if src.count != 2:
        raise RuntimeError(
            f"Expected 2 WorldPop bands, found {src.count}."
        )

    population_100m = src.read(1)
    density_100m = src.read(2)

    src_transform = src.transform
    src_crs = src.crs

# -----------------------------------------------------------------------------
# LOCAL 100 m VALIDATION
# -----------------------------------------------------------------------------
valid_pop = population_100m[
    np.isfinite(population_100m)
]

valid_density = density_100m[
    np.isfinite(density_100m)
]

if valid_pop.size == 0 or valid_density.size == 0:
    raise RuntimeError(
        "WorldPop export contains no finite data."
    )

local_total_pop = float(
    np.sum(valid_pop)
)

local_density_mean = float(
    np.mean(valid_density)
)

print("\nLOCAL WORLDPOP STATISTICS")
print("-" * 112)

print(
    f"Population raster total: "
    f"{local_total_pop:,.0f}"
)

print(
    f"Population density mean: "
    f"{local_density_mean:,.3f} persons/km²"
)

print(
    f"Population density minimum: "
    f"{valid_density.min():,.3f}"
)

print(
    f"Population density maximum: "
    f"{valid_density.max():,.3f}"
)

# -----------------------------------------------------------------------------
# ALIGN DENSITY TO 30 m GRID
#
# Bilinear interpolation is appropriate for a continuous density surface.
#
# IMPORTANT:
# We are NOT resampling population counts as counts.
# Only the continuous density predictor is aligned to 30 m.
# -----------------------------------------------------------------------------
density_30m = np.full(
    (
        ref_height,
        ref_width
    ),
    np.nan,
    dtype="float32"
)

reproject(
    source=density_100m,
    destination=density_30m,

    src_transform=src_transform,
    src_crs=src_crs,

    dst_transform=ref_transform,
    dst_crs=ref_crs,

    src_nodata=np.nan,
    dst_nodata=np.nan,

    resampling=Resampling.bilinear
)

# Strictly mask outside validated Enugu terrain footprint.
density_30m[
    ~study_mask
] = np.nan

valid_30m = density_30m[
    np.isfinite(density_30m)
]

if valid_30m.size == 0:
    raise RuntimeError(
        "No valid population-density pixels after 30 m alignment."
    )

print("\n30 m ALIGNED DENSITY")
print("-" * 112)

print(
    f"Valid pixels: "
    f"{valid_30m.size:,}"
)

print(
    f"Minimum: "
    f"{valid_30m.min():,.3f} persons/km²"
)

print(
    f"Maximum: "
    f"{valid_30m.max():,.3f} persons/km²"
)

print(
    f"Mean: "
    f"{valid_30m.mean():,.3f} persons/km²"
)

print(
    f"Median: "
    f"{np.median(valid_30m):,.3f} persons/km²"
)

# -----------------------------------------------------------------------------
# SAVE 30 m DENSITY RASTER
# -----------------------------------------------------------------------------
out_profile = ref_profile.copy()

out_profile.update(
    dtype="float32",
    count=1,
    nodata=np.nan,
    compress="deflate",
    predictor=3
)

with rasterio.open(
    DENSITY_30M_FILE,
    "w",
    **out_profile
) as dst:

    dst.write(
        density_30m,
        1
    )

    dst.set_band_description(
        1,
        "Population_Density_2020_persons_per_km2"
    )

print("\n✓ 30 m population-density raster saved")

# -----------------------------------------------------------------------------
# DESCRIPTIVE DISTRIBUTION ON 30 m GRID
# -----------------------------------------------------------------------------
thresholds = [
    100,
    250,
    500,
    1000,
    2500,
    5000
]

records = []

print("\n30 m POPULATION-DENSITY DISTRIBUTION")
print("-" * 112)

for threshold in thresholds:

    count = int(
        np.count_nonzero(
            valid_30m >= threshold
        )
    )

    pct = (
        count /
        valid_30m.size *
        100
    )

    records.append({
        "Threshold_persons_per_km2": threshold,
        "Pixel_Count": count,
        "Percent_of_Valid_Area": pct
    })

    print(
        f"≥ {threshold:>5,} persons/km² : "
        f"{count:>10,} pixels | "
        f"{pct:6.2f}%"
    )

# -----------------------------------------------------------------------------
# REOPEN OUTPUT FOR ALIGNMENT VALIDATION
# -----------------------------------------------------------------------------
with rasterio.open(
    DENSITY_30M_FILE
) as src:

    saved = src.read(1)

    saved_epsg = src.crs.to_epsg()
    saved_width = src.width
    saved_height = src.height

    saved_res_x = abs(
        src.transform.a
    )

    saved_res_y = abs(
        src.transform.e
    )

    saved_bounds = src.bounds

saved_valid = saved[
    np.isfinite(saved)
]

same_dimensions = (
    saved_width == ref_width
    and
    saved_height == ref_height
)

same_resolution = (
    np.isclose(
        saved_res_x,
        ref_res_x
    )
    and
    np.isclose(
        saved_res_y,
        ref_res_y
    )
)

same_bounds = np.allclose(
    [
        saved_bounds.left,
        saved_bounds.bottom,
        saved_bounds.right,
        saved_bounds.top
    ],
    [
        ref_bounds.left,
        ref_bounds.bottom,
        ref_bounds.right,
        ref_bounds.top
    ],
    atol=0.001
)

# -----------------------------------------------------------------------------
# CONSISTENCY CHECKS
# -----------------------------------------------------------------------------
pop_total_difference_pct = (
    abs(
        local_total_pop -
        EE_TOTAL_POP
    )
    /
    EE_TOTAL_POP *
    100
)

density_mean_difference = abs(
    local_density_mean -
    EE_DENSITY_MEAN
)

# -----------------------------------------------------------------------------
# SAVE TABLES
# -----------------------------------------------------------------------------
stats_df = pd.DataFrame([
    {
        "Dataset":
            "WorldPop 2020 population density",
        "WorldPop_100m_Total_Population":
            local_total_pop,
        "EE_Total_Population":
            EE_TOTAL_POP,
        "Population_Total_Difference_pct":
            pop_total_difference_pct,
        "WorldPop_100m_Density_Mean":
            local_density_mean,
        "EE_Density_Mean":
            EE_DENSITY_MEAN,
        "Density_Mean_Difference":
            density_mean_difference,
        "Aligned_30m_Valid_Pixels":
            int(valid_30m.size),
        "Aligned_30m_Min":
            float(valid_30m.min()),
        "Aligned_30m_Max":
            float(valid_30m.max()),
        "Aligned_30m_Mean":
            float(valid_30m.mean()),
        "Aligned_30m_Median":
            float(np.median(valid_30m))
    }
])

stats_df.to_csv(
    STATS_FILE,
    index=False
)

distribution_file = (
    TABLE_DIR /
    "Enugu_Population_Density_2020_Distribution.csv"
)

pd.DataFrame(
    records
).to_csv(
    distribution_file,
    index=False
)

validation_df = pd.DataFrame([
    {
        "Task_ID":
            WORLDPOP_TASK_ID,
        "EPSG":
            saved_epsg,
        "Resolution_m":
            saved_res_x,
        "Width":
            saved_width,
        "Height":
            saved_height,
        "Valid_Pixels":
            int(saved_valid.size),
        "Population_Total_Difference_pct":
            pop_total_difference_pct,
        "Density_Mean_Difference":
            density_mean_difference,
        "Aligned_with_Terrain":
            (
                same_dimensions
                and
                same_resolution
                and
                same_bounds
            )
    }
])

validation_df.to_csv(
    VALIDATION_FILE,
    index=False
)

# -----------------------------------------------------------------------------
# FINAL VALIDATION
# -----------------------------------------------------------------------------
checks = {
    "Earth Engine export completed":
        task_state == "COMPLETED",

    "Original WorldPop raster stored":
        TARGET_WORLDPOP.exists(),

    "Local population total positive":
        local_total_pop > 0,

    "Local population total agrees with EE":
        pop_total_difference_pct < 2,

    "Local density mean agrees with EE":
        density_mean_difference < 20,

    "30 m density contains valid pixels":
        saved_valid.size > 0,

    "30 m density non-negative":
        saved_valid.min() >= 0,

    "Output CRS is EPSG:32632":
        saved_epsg == 32632,

    "Dimensions match terrain":
        same_dimensions,

    "Resolution matches terrain":
        same_resolution,

    "Bounds match terrain":
        same_bounds,

    "30 m density raster saved":
        DENSITY_30M_FILE.exists(),

    "Statistics table saved":
        STATS_FILE.exists(),

    "Validation table saved":
        VALIDATION_FILE.exists()
}

print("\nCONSISTENCY CHECK")
print("-" * 112)

print(
    f"EE population total: "
    f"{EE_TOTAL_POP:,.0f}"
)

print(
    f"Local population total: "
    f"{local_total_pop:,.0f}"
)

print(
    f"Population-total difference: "
    f"{pop_total_difference_pct:.4f}%"
)

print(
    f"EE density mean: "
    f"{EE_DENSITY_MEAN:.3f}"
)

print(
    f"Local 100 m density mean: "
    f"{local_density_mean:.3f}"
)

print(
    f"Density-mean difference: "
    f"{density_mean_difference:.4f}"
)

print("\nVALIDATION SUMMARY")
print("-" * 112)

for label, passed in checks.items():
    print(
        f"{'✓' if passed else '✗'} "
        f"{label}"
    )

print("\nOUTPUT FILES")
print("-" * 112)

for path in [
    TARGET_WORLDPOP,
    DENSITY_30M_FILE,
    STATS_FILE,
    distribution_file,
    VALIDATION_FILE
]:

    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{path.name}"
    )

if all(checks.values()):

    print("\n✓ STAGE 7B FULLY PASSED")

    print(
        "WorldPop population-pressure predictor "
        "is stored, aligned and validated."
    )

else:

    print("\n⚠ STAGE 7B REQUIRES REVIEW")

print("=" * 112)

In [ ]:
# =====================================================================================
# PROJECT 7 — STAGE 7C
# DISTANCE TO EXISTING BUILT-UP AREAS
#
# Source:
# Validated Dynamic World 2025 land-cover raster
#
# Purpose:
# Create a continuous urban-proximity predictor representing
# distance to the existing 2025 built-up footprint.
#
# Dynamic World class:
#   6 = Built
#
# No arbitrary suitability weights are applied.
# =====================================================================================

import rasterio
from rasterio.warp import reproject, Resampling
from scipy.ndimage import distance_transform_edt
import numpy as np
import pandas as pd
from pathlib import Path

# -----------------------------------------------------------------------------
# PATHS
# -----------------------------------------------------------------------------
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

DW_FILE = (
    PROJECT_ROOT /
    "04_Land_Cover" /
    "Enugu_DynamicWorld_2025.tif"
)

REFERENCE_FILE = (
    PROJECT_ROOT /
    "02_Terrain" /
    "Enugu_Elevation_30m.tif"
)

OUTPUT_DIR = (
    PROJECT_ROOT /
    "07_Population_Urban_Pressure"
)

TABLE_DIR = (
    PROJECT_ROOT /
    "16_Tables"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

BUILT_MASK_FILE = (
    OUTPUT_DIR /
    "Enugu_BuiltUp_2025_30m.tif"
)

DISTANCE_FILE = (
    OUTPUT_DIR /
    "Enugu_Distance_to_BuiltUp_2025_30m.tif"
)

STATS_FILE = (
    TABLE_DIR /
    "Enugu_Distance_to_BuiltUp_2025_Statistics.csv"
)

PROXIMITY_FILE = (
    TABLE_DIR /
    "Enugu_BuiltUp_Proximity_Distribution.csv"
)

VALIDATION_FILE = (
    OUTPUT_DIR /
    "Distance_to_BuiltUp_2025_Validation.csv"
)

print("=" * 114)
print("STAGE 7C — DISTANCE TO EXISTING BUILT-UP AREAS")
print("=" * 114)

# -----------------------------------------------------------------------------
# LOAD 30 m REFERENCE GRID
# -----------------------------------------------------------------------------
with rasterio.open(REFERENCE_FILE) as ref:

    elevation = ref.read(1)

    ref_profile = ref.profile.copy()
    ref_transform = ref.transform
    ref_crs = ref.crs

    ref_width = ref.width
    ref_height = ref.height

    ref_res_x = abs(ref_transform.a)
    ref_res_y = abs(ref_transform.e)

    ref_bounds = ref.bounds

    study_mask = np.isfinite(elevation)

print("\nREFERENCE GRID")
print("-" * 114)

print("CRS:", ref_crs)
print("EPSG:", ref_crs.to_epsg())

print(
    f"Dimensions: "
    f"{ref_width} × {ref_height}"
)

print(
    f"Resolution: "
    f"{ref_res_x:.3f} × "
    f"{ref_res_y:.3f} m"
)

# -----------------------------------------------------------------------------
# LOAD DYNAMIC WORLD 2025
# -----------------------------------------------------------------------------
if not DW_FILE.exists():
    raise FileNotFoundError(
        f"Dynamic World raster not found: {DW_FILE}"
    )

with rasterio.open(DW_FILE) as src:

    print("\nDYNAMIC WORLD INPUT")
    print("-" * 114)

    print("CRS:", src.crs)
    print("EPSG:", src.crs.to_epsg())

    print(
        f"Dimensions: "
        f"{src.width} × {src.height}"
    )

    print(
        f"Resolution: "
        f"{abs(src.transform.a):.3f} × "
        f"{abs(src.transform.e):.3f} m"
    )

    print("Band count:", src.count)

    if src.count < 1:
        raise RuntimeError(
            "Dynamic World raster contains no bands."
        )

    lulc_10m = src.read(1)

    dw_transform = src.transform
    dw_crs = src.crs

# -----------------------------------------------------------------------------
# EXTRACT BUILT-UP CLASS
#
# Dynamic World:
# class 6 = Built
# -----------------------------------------------------------------------------
built_10m = (
    lulc_10m == 6
).astype(np.uint8)

built_10m_count = int(
    np.count_nonzero(
        built_10m == 1
    )
)

print("\nBUILT-UP EXTRACTION")
print("-" * 114)

print(
    f"Built-up pixels at 10 m: "
    f"{built_10m_count:,}"
)

if built_10m_count == 0:
    raise RuntimeError(
        "No Dynamic World built-up pixels were found."
    )

# -----------------------------------------------------------------------------
# ALIGN BUILT-UP MASK TO 30 m REFERENCE GRID
#
# Nearest neighbour is appropriate for categorical class data.
# -----------------------------------------------------------------------------
built_30m = np.zeros(
    (
        ref_height,
        ref_width
    ),
    dtype=np.uint8
)

reproject(
    source=built_10m,
    destination=built_30m,

    src_transform=dw_transform,
    src_crs=dw_crs,

    dst_transform=ref_transform,
    dst_crs=ref_crs,

    src_nodata=0,
    dst_nodata=0,

    resampling=Resampling.nearest
)

# Restrict to validated study area
built_inside = (
    (built_30m == 1)
    &
    study_mask
)

built_cells = int(
    built_inside.sum()
)

built_area_km2 = (
    built_cells *
    ref_res_x *
    ref_res_y /
    1_000_000
)

study_area_km2 = (
    study_mask.sum() *
    ref_res_x *
    ref_res_y /
    1_000_000
)

built_pct = (
    built_area_km2 /
    study_area_km2 *
    100
)

print("\n30 m BUILT-UP FOOTPRINT")
print("-" * 114)

print(
    f"Built-up cells: "
    f"{built_cells:,}"
)

print(
    f"Built-up area: "
    f"{built_area_km2:,.2f} km²"
)

print(
    f"Built-up share of valid terrain: "
    f"{built_pct:.2f}%"
)

if built_cells == 0:
    raise RuntimeError(
        "No built-up cells remained after 30 m alignment."
    )

# -----------------------------------------------------------------------------
# DISTANCE TO BUILT-UP
#
# Built-up cells must equal zero for distance_transform_edt.
# -----------------------------------------------------------------------------
print("\nCALCULATING DISTANCE TO EXISTING BUILT-UP")
print("-" * 114)

non_built = (
    built_30m == 0
)

distance = (
    distance_transform_edt(
        non_built,
        sampling=(
            ref_res_y,
            ref_res_x
        )
    )
    .astype("float32")
)

distance[
    ~study_mask
] = np.nan

valid = distance[
    study_mask
]

print("✓ Distance calculation completed")

# -----------------------------------------------------------------------------
# STATISTICS
# -----------------------------------------------------------------------------
stats = {
    "Valid_Pixels":
        int(valid.size),

    "Minimum_m":
        float(valid.min()),

    "Maximum_m":
        float(valid.max()),

    "Mean_m":
        float(valid.mean()),

    "StdDev_m":
        float(valid.std()),

    "P25_m":
        float(np.percentile(valid, 25)),

    "Median_m":
        float(np.median(valid)),

    "P75_m":
        float(np.percentile(valid, 75)),

    "P90_m":
        float(np.percentile(valid, 90)),

    "P95_m":
        float(np.percentile(valid, 95))
}

print("\nDISTANCE-TO-BUILT-UP STATISTICS")
print("-" * 114)

print(
    f"Minimum:  "
    f"{stats['Minimum_m']:,.2f} m"
)

print(
    f"Maximum:  "
    f"{stats['Maximum_m']:,.2f} m"
)

print(
    f"Mean:     "
    f"{stats['Mean_m']:,.2f} m"
)

print(
    f"Median:   "
    f"{stats['Median_m']:,.2f} m"
)

print(
    f"90th pct: "
    f"{stats['P90_m']:,.2f} m"
)

print(
    f"95th pct: "
    f"{stats['P95_m']:,.2f} m"
)

# -----------------------------------------------------------------------------
# DESCRIPTIVE PROXIMITY BANDS
#
# These are diagnostic summaries only, not suitability classes.
# -----------------------------------------------------------------------------
thresholds = [
    100,
    250,
    500,
    1000,
    2000,
    5000
]

records = []

print("\nBUILT-UP PROXIMITY")
print("-" * 114)

for threshold in thresholds:

    count = int(
        np.count_nonzero(
            valid <= threshold
        )
    )

    pct = (
        count /
        valid.size *
        100
    )

    records.append({
        "Threshold_m": threshold,
        "Pixel_Count": count,
        "Percent_of_Study_Area": pct
    })

    print(
        f"≤ {threshold:>5,} m : "
        f"{count:>10,} pixels | "
        f"{pct:6.2f}%"
    )

# -----------------------------------------------------------------------------
# ZERO-DISTANCE CHECK
# -----------------------------------------------------------------------------
built_distances = (
    distance[
        built_inside
    ]
)

zero_built_pct = (
    np.count_nonzero(
        built_distances == 0
    )
    /
    built_distances.size *
    100
)

print("\nBUILT-UP CELL VALIDATION")
print("-" * 114)

print(
    f"Built-up cells with distance = 0: "
    f"{zero_built_pct:.4f}%"
)

# -----------------------------------------------------------------------------
# SAVE BUILT-UP MASK
# -----------------------------------------------------------------------------
mask_profile = ref_profile.copy()

mask_profile.update(
    dtype="uint8",
    count=1,
    nodata=0,
    compress="deflate"
)

with rasterio.open(
    BUILT_MASK_FILE,
    "w",
    **mask_profile
) as dst:

    out_mask = built_30m.copy()

    out_mask[
        ~study_mask
    ] = 0

    dst.write(
        out_mask,
        1
    )

    dst.set_band_description(
        1,
        "DynamicWorld_BuiltUp_2025"
    )

# -----------------------------------------------------------------------------
# SAVE DISTANCE RASTER
# -----------------------------------------------------------------------------
distance_profile = ref_profile.copy()

distance_profile.update(
    dtype="float32",
    count=1,
    nodata=np.nan,
    compress="deflate",
    predictor=3
)

with rasterio.open(
    DISTANCE_FILE,
    "w",
    **distance_profile
) as dst:

    dst.write(
        distance,
        1
    )

    dst.set_band_description(
        1,
        "Distance_to_BuiltUp_2025_m"
    )

print("\n✓ Built-up mask and distance raster saved")

# -----------------------------------------------------------------------------
# SAVE TABLES
# -----------------------------------------------------------------------------
stats_df = pd.DataFrame([
    {
        **stats,
        "BuiltUp_Cells_30m":
            built_cells,
        "BuiltUp_Area_km2":
            built_area_km2,
        "BuiltUp_Percent":
            built_pct
    }
])

stats_df.to_csv(
    STATS_FILE,
    index=False
)

pd.DataFrame(
    records
).to_csv(
    PROXIMITY_FILE,
    index=False
)

# -----------------------------------------------------------------------------
# INDEPENDENT OUTPUT VALIDATION
# -----------------------------------------------------------------------------
with rasterio.open(
    DISTANCE_FILE
) as src:

    saved = src.read(1)

    saved_epsg = (
        src.crs.to_epsg()
    )

    saved_width = src.width
    saved_height = src.height

    saved_res_x = abs(
        src.transform.a
    )

    saved_res_y = abs(
        src.transform.e
    )

    saved_bounds = src.bounds

saved_valid = saved[
    np.isfinite(saved)
]

same_dimensions = (
    saved_width == ref_width
    and
    saved_height == ref_height
)

same_resolution = (
    np.isclose(
        saved_res_x,
        ref_res_x
    )
    and
    np.isclose(
        saved_res_y,
        ref_res_y
    )
)

same_bounds = np.allclose(
    [
        saved_bounds.left,
        saved_bounds.bottom,
        saved_bounds.right,
        saved_bounds.top
    ],
    [
        ref_bounds.left,
        ref_bounds.bottom,
        ref_bounds.right,
        ref_bounds.top
    ],
    atol=0.001
)

validation_df = pd.DataFrame([
    {
        "Dataset":
            "Distance to Dynamic World 2025 built-up footprint",

        "EPSG":
            saved_epsg,

        "Resolution_m":
            saved_res_x,

        "Width":
            saved_width,

        "Height":
            saved_height,

        "BuiltUp_Cells":
            built_cells,

        "BuiltUp_Area_km2":
            built_area_km2,

        "BuiltUp_Percent":
            built_pct,

        "Valid_Distance_Pixels":
            int(saved_valid.size),

        "Minimum_Distance_m":
            float(saved_valid.min()),

        "Maximum_Distance_m":
            float(saved_valid.max()),

        "Mean_Distance_m":
            float(saved_valid.mean()),

        "Median_Distance_m":
            float(np.median(saved_valid)),

        "BuiltUp_Zero_Distance_pct":
            zero_built_pct,

        "Aligned_with_Terrain":
            (
                same_dimensions
                and
                same_resolution
                and
                same_bounds
            )
    }
])

validation_df.to_csv(
    VALIDATION_FILE,
    index=False
)

# -----------------------------------------------------------------------------
# FINAL VALIDATION
# -----------------------------------------------------------------------------
checks = {
    "Dynamic World input exists":
        DW_FILE.exists(),

    "Built-up cells extracted":
        built_cells > 0,

    "Built-up area positive":
        built_area_km2 > 0,

    "Distance raster has valid pixels":
        saved_valid.size > 0,

    "Minimum distance = 0":
        np.isclose(
            saved_valid.min(),
            0
        ),

    "Maximum distance positive":
        saved_valid.max() > 0,

    "All built-up cells have zero distance":
        np.isclose(
            zero_built_pct,
            100
        ),

    "Output CRS is EPSG:32632":
        saved_epsg == 32632,

    "Dimensions match terrain":
        same_dimensions,

    "Resolution matches terrain":
        same_resolution,

    "Bounds match terrain":
        same_bounds,

    "Built-up mask saved":
        BUILT_MASK_FILE.exists(),

    "Distance raster saved":
        DISTANCE_FILE.exists(),

    "Statistics saved":
        STATS_FILE.exists(),

    "Proximity table saved":
        PROXIMITY_FILE.exists(),

    "Validation file saved":
        VALIDATION_FILE.exists()
}

print("\nVALIDATION SUMMARY")
print("-" * 114)

for label, passed in checks.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{label}"
    )

print("\nOUTPUT FILES")
print("-" * 114)

for path in [
    BUILT_MASK_FILE,
    DISTANCE_FILE,
    STATS_FILE,
    PROXIMITY_FILE,
    VALIDATION_FILE
]:

    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{path.name}"
    )

if all(checks.values()):

    print("\n✓ STAGE 7C FULLY PASSED")

    print(
        "Existing urban-footprint proximity predictor "
        "is generated, aligned and validated."
    )

else:

    print("\n⚠ STAGE 7C REQUIRES REVIEW")

print("=" * 114)

In [ ]:
# =====================================================================================
# PROJECT 7 — STAGE 8A
# HARMONIZE ALL CORE PREDICTORS TO 30 m + BUILD CANDIDATE ML STACK
#
# Reference grid:
#   Enugu_Elevation_30m.tif
#
# Candidate predictors:
#   1 Elevation_m
#   2 Slope_deg
#   3 NDVI_2025
#   4 NDBI_2025
#   5 MNDWI_2025
#   6 Distance_to_Road_m
#   7 Distance_to_Drainage_m
#   8 Population_Density_2020
#   9 Distance_to_BuiltUp_2025_m
#
# NOTE:
# This is a CANDIDATE predictor stack.
# Leakage/redundancy screening occurs before model training.
# =====================================================================================

import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np
import pandas as pd
from pathlib import Path

# -----------------------------------------------------------------------------
# PATHS
# -----------------------------------------------------------------------------
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

STACK_DIR = PROJECT_ROOT / "08_Predictor_Stack"
TABLE_DIR = PROJECT_ROOT / "16_Tables"

STACK_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

REFERENCE_FILE = (
    PROJECT_ROOT /
    "02_Terrain" /
    "Enugu_Elevation_30m.tif"
)

SLOPE_FILE = (
    PROJECT_ROOT /
    "02_Terrain" /
    "Enugu_Slope_Degrees_30m.tif"
)

S2_FILE = (
    PROJECT_ROOT /
    "03_Remote_Sensing" /
    "Enugu_Sentinel2_Indices_2025.tif"
)

ROAD_FILE = (
    PROJECT_ROOT /
    "05_Roads_Accessibility" /
    "Enugu_Distance_to_Drive_Road_30m.tif"
)

DRAINAGE_FILE = (
    PROJECT_ROOT /
    "06_Environmental_Constraints" /
    "Enugu_Distance_to_Drainage_30m.tif"
)

POP_FILE = (
    PROJECT_ROOT /
    "07_Population_Urban_Pressure" /
    "Enugu_Population_Density_2020_30m.tif"
)

BUILT_DIST_FILE = (
    PROJECT_ROOT /
    "07_Population_Urban_Pressure" /
    "Enugu_Distance_to_BuiltUp_2025_30m.tif"
)

STACK_FILE = (
    STACK_DIR /
    "Enugu_ML_Candidate_Predictor_Stack_30m.tif"
)

SUMMARY_FILE = (
    TABLE_DIR /
    "Enugu_ML_Candidate_Predictor_Summary.csv"
)

VALIDATION_FILE = (
    STACK_DIR /
    "Candidate_Predictor_Stack_Validation.csv"
)

print("=" * 116)
print("STAGE 8A — HARMONIZED 30 m ML CANDIDATE PREDICTOR STACK")
print("=" * 116)

# -----------------------------------------------------------------------------
# VERIFY INPUT FILES
# -----------------------------------------------------------------------------
inputs = {
    "Elevation": REFERENCE_FILE,
    "Slope": SLOPE_FILE,
    "Sentinel2": S2_FILE,
    "RoadDistance": ROAD_FILE,
    "DrainageDistance": DRAINAGE_FILE,
    "PopulationDensity": POP_FILE,
    "BuiltUpDistance": BUILT_DIST_FILE
}

print("\nINPUT FILE CHECK")
print("-" * 116)

for name, path in inputs.items():
    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{name:<20} {path.name}"
    )

missing = [
    str(path)
    for path in inputs.values()
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "Missing required predictor files:\n"
        + "\n".join(missing)
    )

# -----------------------------------------------------------------------------
# LOAD REFERENCE GRID
# -----------------------------------------------------------------------------
with rasterio.open(REFERENCE_FILE) as ref:

    elevation = ref.read(1).astype("float32")

    ref_profile = ref.profile.copy()
    ref_transform = ref.transform
    ref_crs = ref.crs

    width = ref.width
    height = ref.height

    res_x = abs(ref_transform.a)
    res_y = abs(ref_transform.e)

    bounds = ref.bounds

    study_mask = np.isfinite(elevation)

print("\nREFERENCE GRID")
print("-" * 116)

print("CRS:", ref_crs)
print("EPSG:", ref_crs.to_epsg())
print(f"Dimensions: {width} × {height}")
print(f"Resolution: {res_x:.3f} × {res_y:.3f} m")
print(f"Study-area pixels: {study_mask.sum():,}")

# -----------------------------------------------------------------------------
# HELPER — READ ALREADY ALIGNED SINGLE-BAND RASTER
# -----------------------------------------------------------------------------
def read_aligned(path, label):

    with rasterio.open(path) as src:

        arr = src.read(1).astype("float32")

        checks = {
            "epsg": src.crs.to_epsg(),
            "width": src.width,
            "height": src.height,
            "res_x": abs(src.transform.a),
            "res_y": abs(src.transform.e),
            "bounds": tuple(src.bounds)
        }

    if checks["epsg"] != 32632:
        raise RuntimeError(
            f"{label}: unexpected CRS {checks['epsg']}"
        )

    if (
        checks["width"] != width
        or
        checks["height"] != height
    ):
        raise RuntimeError(
            f"{label}: dimensions do not match reference."
        )

    if not (
        np.isclose(checks["res_x"], res_x)
        and
        np.isclose(checks["res_y"], res_y)
    ):
        raise RuntimeError(
            f"{label}: resolution does not match reference."
        )

    if not np.allclose(
        checks["bounds"],
        tuple(bounds),
        atol=0.001
    ):
        raise RuntimeError(
            f"{label}: bounds do not match reference."
        )

    arr[~study_mask] = np.nan

    return arr

# -----------------------------------------------------------------------------
# LOAD ALREADY-ALIGNED 30 m PREDICTORS
# -----------------------------------------------------------------------------
slope = read_aligned(
    SLOPE_FILE,
    "Slope"
)

road_distance = read_aligned(
    ROAD_FILE,
    "Distance to road"
)

drainage_distance = read_aligned(
    DRAINAGE_FILE,
    "Distance to drainage"
)

population_density = read_aligned(
    POP_FILE,
    "Population density"
)

built_distance = read_aligned(
    BUILT_DIST_FILE,
    "Distance to built-up"
)

print("\n✓ Existing 30 m predictors passed alignment checks")

# -----------------------------------------------------------------------------
# SENTINEL-2: ALIGN 20 m CONTINUOUS INDICES TO 30 m GRID
#
# Bilinear resampling is appropriate for continuous spectral indices.
# Observation-count band is not included in the ML predictor stack.
# -----------------------------------------------------------------------------
print("\nHARMONIZING SENTINEL-2 INDICES: 20 m → 30 m")
print("-" * 116)

with rasterio.open(S2_FILE) as src:

    if src.count != 4:
        raise RuntimeError(
            f"Expected 4 Sentinel-2 bands, found {src.count}"
        )

    s2_transform = src.transform
    s2_crs = src.crs

    source_indices = {
        "NDVI_2025": src.read(1),
        "NDBI_2025": src.read(2),
        "MNDWI_2025": src.read(3)
    }

aligned_indices = {}

for name, source in source_indices.items():

    destination = np.full(
        (height, width),
        np.nan,
        dtype="float32"
    )

    reproject(
        source=source,
        destination=destination,

        src_transform=s2_transform,
        src_crs=s2_crs,

        dst_transform=ref_transform,
        dst_crs=ref_crs,

        src_nodata=np.nan,
        dst_nodata=np.nan,

        resampling=Resampling.bilinear
    )

    destination[
        ~study_mask
    ] = np.nan

    aligned_indices[name] = destination

    finite = destination[
        np.isfinite(destination)
    ]

    print(
        f"✓ {name:<12} | "
        f"valid={finite.size:,} | "
        f"min={finite.min():.4f} | "
        f"max={finite.max():.4f} | "
        f"mean={finite.mean():.4f}"
    )

ndvi = aligned_indices["NDVI_2025"]
ndbi = aligned_indices["NDBI_2025"]
mndwi = aligned_indices["MNDWI_2025"]

# -----------------------------------------------------------------------------
# BUILD COMMON VALID MASK
# -----------------------------------------------------------------------------
predictors = {
    "Elevation_m": elevation,
    "Slope_deg": slope,
    "NDVI_2025": ndvi,
    "NDBI_2025": ndbi,
    "MNDWI_2025": mndwi,
    "Distance_to_Road_m": road_distance,
    "Distance_to_Drainage_m": drainage_distance,
    "Population_Density_2020": population_density,
    "Distance_to_BuiltUp_2025_m": built_distance
}

common_valid = study_mask.copy()

for arr in predictors.values():
    common_valid &= np.isfinite(arr)

common_count = int(
    common_valid.sum()
)

study_count = int(
    study_mask.sum()
)

common_pct = (
    common_count /
    study_count *
    100
)

pixel_area_km2 = (
    res_x *
    res_y /
    1_000_000
)

common_area_km2 = (
    common_count *
    pixel_area_km2
)

print("\nCOMMON PREDICTOR COVERAGE")
print("-" * 116)

print(
    f"Study-area pixels: "
    f"{study_count:,}"
)

print(
    f"Pixels valid across all 9 predictors: "
    f"{common_count:,}"
)

print(
    f"Common valid coverage: "
    f"{common_pct:.4f}%"
)

print(
    f"Common valid area: "
    f"{common_area_km2:,.2f} km²"
)

# -----------------------------------------------------------------------------
# PREDICTOR STATISTICS
# -----------------------------------------------------------------------------
records = []

print("\nPREDICTOR STATISTICS")
print("-" * 116)

for i, (name, arr) in enumerate(
    predictors.items(),
    start=1
):

    values = arr[
        common_valid
    ].astype("float64")

    record = {
        "Band": i,
        "Predictor": name,
        "Valid_Pixels": int(values.size),
        "Min": float(values.min()),
        "Max": float(values.max()),
        "Mean": float(values.mean()),
        "StdDev": float(values.std()),
        "P05": float(np.percentile(values, 5)),
        "Median": float(np.median(values)),
        "P95": float(np.percentile(values, 95))
    }

    records.append(record)

    print(
        f"{i:02d}. {name:<30} "
        f"min={record['Min']:>10.3f} | "
        f"max={record['Max']:>12.3f} | "
        f"mean={record['Mean']:>10.3f}"
    )

summary_df = pd.DataFrame(
    records
)

# -----------------------------------------------------------------------------
# SIMPLE DATA-QUALITY CHECKS
# -----------------------------------------------------------------------------
range_checks = {
    "Elevation plausible":
        (
            summary_df.loc[
                summary_df["Predictor"] == "Elevation_m",
                "Min"
            ].iloc[0] >= 0
        ),

    "Slope in 0–90 degrees":
        (
            summary_df.loc[
                summary_df["Predictor"] == "Slope_deg",
                "Min"
            ].iloc[0] >= 0
            and
            summary_df.loc[
                summary_df["Predictor"] == "Slope_deg",
                "Max"
            ].iloc[0] <= 90
        ),

    "NDVI in -1–1":
        (
            summary_df.loc[
                summary_df["Predictor"] == "NDVI_2025",
                "Min"
            ].iloc[0] >= -1
            and
            summary_df.loc[
                summary_df["Predictor"] == "NDVI_2025",
                "Max"
            ].iloc[0] <= 1
        ),

    "NDBI in -1–1":
        (
            summary_df.loc[
                summary_df["Predictor"] == "NDBI_2025",
                "Min"
            ].iloc[0] >= -1
            and
            summary_df.loc[
                summary_df["Predictor"] == "NDBI_2025",
                "Max"
            ].iloc[0] <= 1
        ),

    "MNDWI in -1–1":
        (
            summary_df.loc[
                summary_df["Predictor"] == "MNDWI_2025",
                "Min"
            ].iloc[0] >= -1
            and
            summary_df.loc[
                summary_df["Predictor"] == "MNDWI_2025",
                "Max"
            ].iloc[0] <= 1
        ),

    "Road distance non-negative":
        road_distance[
            common_valid
        ].min() >= 0,

    "Drainage distance non-negative":
        drainage_distance[
            common_valid
        ].min() >= 0,

    "Population density non-negative":
        population_density[
            common_valid
        ].min() >= 0,

    "Built-up distance non-negative":
        built_distance[
            common_valid
        ].min() >= 0
}

# -----------------------------------------------------------------------------
# WRITE MULTIBAND PREDICTOR STACK
#
# Pixels without complete predictor information are written as NaN.
# -----------------------------------------------------------------------------
print("\nWRITING 9-BAND PREDICTOR STACK")
print("-" * 116)

stack_profile = ref_profile.copy()

stack_profile.update(
    dtype="float32",
    count=len(predictors),
    nodata=np.nan,
    compress="deflate",
    predictor=3,
    BIGTIFF="IF_SAFER"
)

with rasterio.open(
    STACK_FILE,
    "w",
    **stack_profile
) as dst:

    for band_index, (name, arr) in enumerate(
        predictors.items(),
        start=1
    ):

        output = arr.astype(
            "float32"
        ).copy()

        # Enforce one common modelling footprint.
        output[
            ~common_valid
        ] = np.nan

        dst.write(
            output,
            band_index
        )

        dst.set_band_description(
            band_index,
            name
        )

print("✓ Predictor stack written")

# -----------------------------------------------------------------------------
# REOPEN STACK FOR INDEPENDENT QA
# -----------------------------------------------------------------------------
with rasterio.open(
    STACK_FILE
) as src:

    saved_epsg = src.crs.to_epsg()
    saved_width = src.width
    saved_height = src.height

    saved_res_x = abs(
        src.transform.a
    )

    saved_res_y = abs(
        src.transform.e
    )

    saved_bounds = src.bounds
    saved_count = src.count

    descriptions = list(
        src.descriptions
    )

    saved_common = np.ones(
        (
            src.height,
            src.width
        ),
        dtype=bool
    )

    saved_band_stats = []

    for band in range(
        1,
        src.count + 1
    ):

        arr = src.read(band)

        finite = np.isfinite(arr)

        saved_common &= finite

        values = arr[
            finite
        ]

        saved_band_stats.append(
            {
                "Band": band,
                "Description":
                    src.descriptions[
                        band - 1
                    ],
                "Finite_Pixels":
                    int(values.size)
            }
        )

saved_common_count = int(
    saved_common.sum()
)

# -----------------------------------------------------------------------------
# ALIGNMENT CHECKS
# -----------------------------------------------------------------------------
same_dimensions = (
    saved_width == width
    and
    saved_height == height
)

same_resolution = (
    np.isclose(
        saved_res_x,
        res_x
    )
    and
    np.isclose(
        saved_res_y,
        res_y
    )
)

same_bounds = np.allclose(
    [
        saved_bounds.left,
        saved_bounds.bottom,
        saved_bounds.right,
        saved_bounds.top
    ],
    [
        bounds.left,
        bounds.bottom,
        bounds.right,
        bounds.top
    ],
    atol=0.001
)

expected_descriptions = list(
    predictors.keys()
)

# -----------------------------------------------------------------------------
# SAVE TABLES
# -----------------------------------------------------------------------------
summary_df.to_csv(
    SUMMARY_FILE,
    index=False
)

validation_df = pd.DataFrame([
    {
        "Predictor_Count":
            len(predictors),

        "EPSG":
            saved_epsg,

        "Resolution_m":
            saved_res_x,

        "Width":
            saved_width,

        "Height":
            saved_height,

        "Study_Area_Pixels":
            study_count,

        "Common_Valid_Pixels":
            common_count,

        "Common_Valid_Percent":
            common_pct,

        "Common_Valid_Area_km2":
            common_area_km2,

        "Saved_Common_Valid_Pixels":
            saved_common_count,

        "Band_Descriptions_Correct":
            descriptions == expected_descriptions,

        "Aligned_with_Reference":
            (
                same_dimensions
                and
                same_resolution
                and
                same_bounds
            )
    }
])

validation_df.to_csv(
    VALIDATION_FILE,
    index=False
)

# -----------------------------------------------------------------------------
# FINAL VALIDATION
# -----------------------------------------------------------------------------
checks = {
    "Nine predictors included":
        len(predictors) == 9,

    "Common coverage > 99%":
        common_pct > 99,

    "All predictor ranges valid":
        all(
            range_checks.values()
        ),

    "Stack file exists":
        STACK_FILE.exists(),

    "Stack has 9 bands":
        saved_count == 9,

    "CRS is EPSG:32632":
        saved_epsg == 32632,

    "Dimensions match reference":
        same_dimensions,

    "Resolution matches reference":
        same_resolution,

    "Bounds match reference":
        same_bounds,

    "Band descriptions correct":
        descriptions == expected_descriptions,

    "Saved common-valid count matches":
        saved_common_count == common_count,

    "Summary CSV saved":
        SUMMARY_FILE.exists(),

    "Validation CSV saved":
        VALIDATION_FILE.exists()
}

print("\nRANGE CHECKS")
print("-" * 116)

for label, passed in range_checks.items():
    print(
        f"{'✓' if passed else '✗'} "
        f"{label}"
    )

print("\nSTACK BAND ORDER")
print("-" * 116)

for i, name in enumerate(
    descriptions,
    start=1
):
    print(
        f"{i}. {name}"
    )

print("\nVALIDATION SUMMARY")
print("-" * 116)

for label, passed in checks.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{label}"
    )

print("\nOUTPUT FILES")
print("-" * 116)

for path in [
    STACK_FILE,
    SUMMARY_FILE,
    VALIDATION_FILE
]:

    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{path.name}"
    )

if all(checks.values()):

    print("\n✓ STAGE 8A FULLY PASSED")

    print(
        "Nine candidate predictors have been "
        "harmonized to one validated 30 m modelling grid."
    )

    print(
        "Next step: predictor redundancy/leakage diagnostics "
        "before constructing ML training labels."
    )

else:

    print("\n⚠ STAGE 8A REQUIRES REVIEW")

print("=" * 116)

In [ ]:
import os
# ==========================================================================================
# PROJECT 7 — STAGE 9A FIX
# COMPLETE 2020 BASELINE VALIDATION + START EXPORT
#
# Fix:
# The validated 2025 class-area table is stored in 16_Tables, not 04_Land_Cover.
#
# This cell DOES NOT repeat the full Stage 9A calculations.
# ==========================================================================================

import ee
import geopandas as gpd
import pandas as pd
import json
from pathlib import Path
from datetime import datetime, timezone

# ------------------------------------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------------------------------------
EE_PROJECT = os.environ.get("EE_PROJECT", "").strip()
if not EE_PROJECT:
    raise ValueError("Set EE_PROJECT before running this stage.")

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

BOUNDARY_FILE = (
    PROJECT_ROOT /
    "01_Boundary" /
    "Enugu_State_Boundary_WGS84.gpkg"
)

TABLE_DIR = PROJECT_ROOT / "16_Tables"
TRAINING_DIR = PROJECT_ROOT / "09_Training_Data"

AREA_2025_FILE = (
    TABLE_DIR /
    "Enugu_DynamicWorld_2025_Class_Area.csv"
)

AREA_2020_FILE = (
    TABLE_DIR /
    "Enugu_DynamicWorld_2020_Server_Class_Area.csv"
)

DESIGN_FILE = (
    TRAINING_DIR /
    "ML_Experiment_Design_2020_2025.json"
)

DRIVE_FOLDER = "Project_7_Enugu_DynamicWorld_2020"

DATASET_ID = "GOOGLE/DYNAMICWORLD/V1"

START_DATE = "2020-01-01"
END_DATE = "2021-01-01"

CLASS_NAMES = {
    0: "Water",
    1: "Trees",
    2: "Grass",
    3: "Flooded vegetation",
    4: "Crops",
    5: "Shrub and scrub",
    6: "Built",
    7: "Bare",
    8: "Snow and ice"
}

print("=" * 118)
print("STAGE 9A FIX — COMPLETE BASELINE VALIDATION + EXPORT")
print("=" * 118)

# ------------------------------------------------------------------------------------------
# INITIALIZE EARTH ENGINE
# ------------------------------------------------------------------------------------------
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate(auth_mode="colab")
    ee.Initialize(project=EE_PROJECT)

print("\n✓ Earth Engine initialized")

# ------------------------------------------------------------------------------------------
# LOAD STUDY AREA
# ------------------------------------------------------------------------------------------
boundary = (
    gpd.read_file(BOUNDARY_FILE)
    .to_crs("EPSG:4326")
)

if len(boundary) != 1:
    raise RuntimeError(
        f"Expected one Enugu State feature, found {len(boundary)}."
    )

geom = boundary.geometry.iloc[0]

ee_boundary = ee.Geometry(
    geom.__geo_interface__
)

vector_area_km2 = float(
    boundary
    .to_crs("EPSG:32632")
    .geometry
    .area
    .sum()
    / 1_000_000
)

print("✓ Validated Enugu boundary loaded")

# ------------------------------------------------------------------------------------------
# REBUILD VERIFIED 2020 COMPOSITE
# ------------------------------------------------------------------------------------------
dw = (
    ee.ImageCollection(DATASET_ID)
    .filterBounds(ee_boundary)
    .filterDate(
        START_DATE,
        END_DATE
    )
)

image_count = int(
    dw.size().getInfo()
)

lulc_2020 = (
    dw.select("label")
    .mode()
    .rename("LULC_2020")
    .clip(ee_boundary)
)

obs_2020 = (
    dw.select("label")
    .count()
    .rename("DW_Obs_Count_2020")
    .clip(ee_boundary)
)

# ------------------------------------------------------------------------------------------
# RE-CALCULATE 2020 CLASS AREA
# ------------------------------------------------------------------------------------------
area_image = (
    ee.Image.pixelArea()
    .divide(1_000_000)
    .rename("Area_km2")
)

records = []

for class_id in range(9):

    area = (
        area_image
        .updateMask(
            lulc_2020.eq(class_id)
        )
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=ee_boundary,
            scale=10,
            maxPixels=1e10,
            tileScale=4
        )
        .get("Area_km2")
        .getInfo()
    )

    area = (
        float(area)
        if area is not None
        else 0.0
    )

    records.append({
        "Class_ID": class_id,
        "Class_Name": CLASS_NAMES[class_id],
        "Area_km2": area
    })

area_2020_df = pd.DataFrame(records)

total_2020_area = float(
    area_2020_df["Area_km2"].sum()
)

area_2020_df["Percent"] = (
    area_2020_df["Area_km2"]
    / total_2020_area
    * 100
)

built_2020_area = float(
    area_2020_df.loc[
        area_2020_df["Class_ID"] == 6,
        "Area_km2"
    ].iloc[0]
)

area_2020_df.to_csv(
    AREA_2020_FILE,
    index=False
)

# ------------------------------------------------------------------------------------------
# LOAD VALIDATED 2025 TABLE FROM CORRECT LOCATION
# ------------------------------------------------------------------------------------------
print("\n2025 REFERENCE TABLE")
print("-" * 118)

print("Expected path:")
print(AREA_2025_FILE)

if not AREA_2025_FILE.exists():
    raise FileNotFoundError(
        "Validated 2025 Dynamic World class table still not found:\n"
        f"{AREA_2025_FILE}"
    )

area_2025_df = pd.read_csv(
    AREA_2025_FILE
)

print("✓ 2025 validated class-area table found")

built_2025_rows = area_2025_df[
    area_2025_df["Class_ID"] == 6
]

if len(built_2025_rows) != 1:
    raise RuntimeError(
        "Could not uniquely identify class 6 in the 2025 table."
    )

built_2025_area = float(
    built_2025_rows.iloc[0][
        "Area_km2"
    ]
)

# ------------------------------------------------------------------------------------------
# AREA COMPARISON
# ------------------------------------------------------------------------------------------
built_difference = (
    built_2025_area -
    built_2020_area
)

relative_difference = (
    built_difference /
    built_2020_area *
    100
)

print("\nBUILT-UP AREA COMPARISON")
print("-" * 118)

print(
    f"2020 built-up area: "
    f"{built_2020_area:,.2f} km²"
)

print(
    f"2025 built-up area: "
    f"{built_2025_area:,.2f} km²"
)

print(
    f"Net class-area difference: "
    f"{built_difference:+,.2f} km²"
)

print(
    f"Relative difference: "
    f"{relative_difference:+.2f}%"
)

if built_difference < 0:

    print(
        "\n⚠ IMPORTANT QA FLAG:"
    )

    print(
        "The 2020 modal Dynamic World composite contains "
        "more class-6 built-up area than the 2025 composite."
    )

    print(
        "We will NOT interpret this as confirmed urban decline."
    )

    print(
        "Pixel-level transition stability and confidence filtering "
        "must be assessed before constructing ML labels."
    )

# ------------------------------------------------------------------------------------------
# OBSERVATION STATISTICS
# ------------------------------------------------------------------------------------------
obs_stats = (
    obs_2020.reduceRegion(
        reducer=(
            ee.Reducer.minMax()
            .combine(
                reducer2=ee.Reducer.mean(),
                sharedInputs=True
            )
        ),
        geometry=ee_boundary,
        scale=10,
        maxPixels=1e10,
        tileScale=4
    )
    .getInfo()
)

obs_min = float(
    obs_stats["DW_Obs_Count_2020_min"]
)

obs_max = float(
    obs_stats["DW_Obs_Count_2020_max"]
)

obs_mean = float(
    obs_stats["DW_Obs_Count_2020_mean"]
)

# ------------------------------------------------------------------------------------------
# AREA CONSISTENCY
# ------------------------------------------------------------------------------------------
area_difference_pct = (
    abs(
        total_2020_area -
        vector_area_km2
    )
    /
    vector_area_km2 *
    100
)

print("\n2020 VALIDATION")
print("-" * 118)

print(
    f"Images intersecting Enugu: "
    f"{image_count}"
)

print(
    f"Classified area: "
    f"{total_2020_area:,.2f} km²"
)

print(
    f"Validated vector area: "
    f"{vector_area_km2:,.2f} km²"
)

print(
    f"Area difference: "
    f"{area_difference_pct:.4f}%"
)

print(
    f"Observation minimum: "
    f"{obs_min:.0f}"
)

print(
    f"Observation maximum: "
    f"{obs_max:.0f}"
)

print(
    f"Observation mean: "
    f"{obs_mean:.3f}"
)

# ------------------------------------------------------------------------------------------
# MODEL-DESIGN REGISTER
# ------------------------------------------------------------------------------------------
design_record = {
    "Experiment":
        "Observed urban expansion modelling",

    "Provisional_Baseline_Year":
        2020,

    "Outcome_Year":
        2025,

    "Candidate_Positive_Class":
        "Non-built in 2020 and built in 2025",

    "Candidate_Negative_Class":
        "Non-built in 2020 and non-built in 2025",

    "Important_QA_Status":
        (
            "2020 built-up class area exceeds 2025 built-up area; "
            "pixel-level stability/confidence screening required "
            "before labels are accepted."
        ),

    "2020_Built_Area_km2":
        built_2020_area,

    "2025_Built_Area_km2":
        built_2025_area,

    "Net_Class_Area_Difference_km2":
        built_difference,

    "Created_UTC":
        datetime.now(
            timezone.utc
        ).isoformat()
}

with open(
    DESIGN_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        design_record,
        f,
        indent=4
    )

# ------------------------------------------------------------------------------------------
# PREPARE COMPATIBLE EXPORT
# ------------------------------------------------------------------------------------------
export_image = (
    lulc_2020
    .toUint16()
    .addBands(
        obs_2020.toUint16()
    )
)

band_types = (
    export_image
    .bandTypes()
    .getInfo()
)

print("\nEXPORT BAND TYPE CHECK")
print("-" * 118)

for band, info in band_types.items():

    print(
        f"{band}: "
        f"{info.get('precision')} / "
        f"{info.get('min')} to "
        f"{info.get('max')}"
    )

# ------------------------------------------------------------------------------------------
# PRE-EXPORT CHECKS
# ------------------------------------------------------------------------------------------
checks = {
    "Dynamic World 2020 images found":
        image_count > 0,

    "2020 class table saved":
        AREA_2020_FILE.exists(),

    "2025 reference table found":
        AREA_2025_FILE.exists(),

    "Classified area within 2% of state":
        area_difference_pct < 2,

    "2020 built-up area positive":
        built_2020_area > 0,

    "2025 built-up area positive":
        built_2025_area > 0,

    "Observation minimum >= 1":
        obs_min >= 1,

    "Observation maximum > 1":
        obs_max > 1,

    "Observation mean positive":
        obs_mean > 0,

    "Two compatible export bands":
        len(band_types) == 2,

    "Experiment design saved":
        DESIGN_FILE.exists()
}

print("\nPRE-EXPORT VALIDATION")
print("-" * 118)

for label, passed in checks.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{label}"
    )

if not all(checks.values()):

    raise RuntimeError(
        "Pre-export validation failed. "
        "No export was started."
    )

# ------------------------------------------------------------------------------------------
# START EXPORT
# ------------------------------------------------------------------------------------------
print("\nSTARTING GOOGLE DRIVE EXPORT")
print("-" * 118)

task = ee.batch.Export.image.toDrive(
    image=export_image,
    description="Project7_Enugu_DynamicWorld_2020",
    folder=DRIVE_FOLDER,
    fileNamePrefix="Enugu_DynamicWorld_2020",
    region=ee_boundary,
    scale=10,
    crs="EPSG:32632",
    maxPixels=1e13,
    fileFormat="GeoTIFF",
    formatOptions={
        "cloudOptimized": True
    }
)

task.start()

status = task.status()

task_id = status.get("id")
task_state = status.get("state")

print(
    f"Task ID: {task_id}"
)

print(
    f"Task state: {task_state}"
)

print(
    f"Drive folder: "
    f"MyDrive/{DRIVE_FOLDER}"
)

# ------------------------------------------------------------------------------------------
# FINAL
# ------------------------------------------------------------------------------------------
print("\nVALIDATION SUMMARY")
print("-" * 118)

for label, passed in checks.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{label}"
    )

export_started = (
    task_state in [
        "READY",
        "RUNNING"
    ]
)

print(
    f"{'✓' if export_started else '✗'} "
    f"2020 export task started"
)

print("\nIMPORTANT — SAVE THESE VALUES")
print("-" * 118)

print(
    f"DW_2020_TASK_ID = {task_id}"
)

print(
    f"DW_2020_DRIVE_FOLDER = {DRIVE_FOLDER}"
)

print(
    f"DW_2020_BUILT_AREA_KM2 = "
    f"{built_2020_area:.6f}"
)

print(
    f"DW_2025_BUILT_AREA_KM2 = "
    f"{built_2025_area:.6f}"
)

if all(checks.values()) and export_started:

    print(
        "\n✓ STAGE 9A FIX PASSED"
    )

    print(
        "The 2020 baseline composite is validated "
        "and export has started."
    )

    print(
        "The apparent 2020>2025 built-up area discrepancy "
        "has been formally flagged for transition QA."
    )

else:

    print(
        "\n⚠ STAGE 9A FIX REQUIRES REVIEW"
    )

print("=" * 118)

In [ ]:
# ==========================================================================================
# PROJECT 7 — STAGE 9B
# DYNAMIC WORLD 2020 LOCAL VALIDATION + 2020–2025 BUILT-UP TRANSITION QA
#
# This stage:
# 1. Validates the located 2020 export.
# 2. Copies it into the permanent project folder.
# 3. Aligns the 2020 and 2025 land-cover rasters to the 30 m reference grid.
# 4. Calculates stable non-built, stable built, apparent gain and apparent loss.
# 5. Tests sensitivity to observation-count thresholds.
#
# No ML labels are accepted in this stage.
# ==========================================================================================

import shutil
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
from rasterio.warp import reproject, Resampling

# ------------------------------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------------------------------
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

SOURCE_2020_FILE = Path(
    "/content/drive/MyDrive/"
    "Project_7_Enugu_DynamicWorld_2020/"
    "Enugu_DynamicWorld_2020.tif"
)

LANDCOVER_DIR = PROJECT_ROOT / "04_Land_Cover"
TRAINING_DIR = PROJECT_ROOT / "09_Training_Data"
TABLE_DIR = PROJECT_ROOT / "16_Tables"

LANDCOVER_DIR.mkdir(parents=True, exist_ok=True)
TRAINING_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

TARGET_2020_FILE = (
    LANDCOVER_DIR /
    "Enugu_DynamicWorld_2020.tif"
)

DW_2025_FILE = (
    LANDCOVER_DIR /
    "Enugu_DynamicWorld_2025.tif"
)

REFERENCE_FILE = (
    PROJECT_ROOT /
    "02_Terrain" /
    "Enugu_Elevation_30m.tif"
)

TRANSITION_FILE = (
    TRAINING_DIR /
    "Enugu_BuiltUp_Transition_QA_2020_2025_30m.tif"
)

TRANSITION_TABLE_FILE = (
    TABLE_DIR /
    "Enugu_BuiltUp_Transition_QA_2020_2025.csv"
)

OBS_FILTER_TABLE_FILE = (
    TABLE_DIR /
    "Enugu_BuiltUp_Transition_Observation_Filter_QA.csv"
)

VALIDATION_FILE = (
    TRAINING_DIR /
    "DynamicWorld_2020_2025_Transition_QA_Validation.csv"
)

CLASS_NAMES = {
    0: "Water",
    1: "Trees",
    2: "Grass",
    3: "Flooded vegetation",
    4: "Crops",
    5: "Shrub and scrub",
    6: "Built",
    7: "Bare",
    8: "Snow and ice"
}

print("=" * 120)
print("STAGE 9B — DYNAMIC WORLD 2020 VALIDATION + 2020–2025 TRANSITION QA")
print("=" * 120)

# ------------------------------------------------------------------------------------------
# CHECK REQUIRED FILES
# ------------------------------------------------------------------------------------------
required_files = {
    "Located 2020 export": SOURCE_2020_FILE,
    "Validated 2025 raster": DW_2025_FILE,
    "30 m reference raster": REFERENCE_FILE
}

print("\nREQUIRED FILE CHECK")
print("-" * 120)

for label, path in required_files.items():

    exists = path.exists()

    print(
        f"{'✓' if exists else '✗'} "
        f"{label}: {path}"
    )

if not all(path.exists() for path in required_files.values()):

    raise FileNotFoundError(
        "One or more required input files are missing."
    )

# ------------------------------------------------------------------------------------------
# VALIDATE 2020 EXPORT BEFORE COPYING
# ------------------------------------------------------------------------------------------
print("\nVALIDATING LOCATED 2020 EXPORT")
print("-" * 120)

with rasterio.open(SOURCE_2020_FILE) as src:

    source_2020_epsg = (
        src.crs.to_epsg()
        if src.crs is not None
        else None
    )

    source_2020_count = src.count
    source_2020_width = src.width
    source_2020_height = src.height
    source_2020_dtypes = src.dtypes
    source_2020_res = (
        abs(src.transform.a),
        abs(src.transform.e)
    )

    descriptions_2020 = src.descriptions

print("Path:", SOURCE_2020_FILE)
print(
    f"Size: {SOURCE_2020_FILE.stat().st_size / 1024**2:.2f} MB"
)
print("EPSG:", source_2020_epsg)
print("Bands:", source_2020_count)
print(
    f"Dimensions: {source_2020_width} × {source_2020_height}"
)
print(
    f"Resolution: {source_2020_res[0]:.3f} × "
    f"{source_2020_res[1]:.3f} m"
)
print("Data types:", source_2020_dtypes)
print("Band descriptions:", descriptions_2020)

source_checks = {
    "2020 export uses EPSG:32632":
        source_2020_epsg == 32632,

    "2020 export contains two bands":
        source_2020_count == 2,

    "2020 export dimensions are plausible":
        source_2020_width > 1000
        and source_2020_height > 1000,

    "2020 export file is non-empty":
        SOURCE_2020_FILE.stat().st_size > 1_000_000
}

for label, passed in source_checks.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{label}"
    )

if not all(source_checks.values()):

    raise RuntimeError(
        "The located 2020 export failed structural validation."
    )

# ------------------------------------------------------------------------------------------
# COPY TO PERMANENT PROJECT FOLDER
# ------------------------------------------------------------------------------------------
print("\nPERMANENT PROJECT COPY")
print("-" * 120)

if SOURCE_2020_FILE.resolve() != TARGET_2020_FILE.resolve():

    shutil.copy2(
        SOURCE_2020_FILE,
        TARGET_2020_FILE
    )

    print("✓ Export copied to:")
    print(TARGET_2020_FILE)

else:

    print("✓ Export is already in the permanent location")

# ------------------------------------------------------------------------------------------
# READ REFERENCE GRID
# ------------------------------------------------------------------------------------------
with rasterio.open(REFERENCE_FILE) as ref:

    elevation = ref.read(1)

    ref_profile = ref.profile.copy()
    ref_transform = ref.transform
    ref_crs = ref.crs

    ref_width = ref.width
    ref_height = ref.height
    ref_bounds = ref.bounds

    ref_res_x = abs(ref.transform.a)
    ref_res_y = abs(ref.transform.e)

study_mask = np.isfinite(elevation)

pixel_area_km2 = (
    ref_res_x *
    ref_res_y /
    1_000_000
)

print("\nREFERENCE GRID")
print("-" * 120)

print("CRS:", ref_crs)
print(
    f"Dimensions: {ref_width} × {ref_height}"
)
print(
    f"Resolution: {ref_res_x:.3f} × {ref_res_y:.3f} m"
)
print(
    f"Valid study-area cells: {study_mask.sum():,}"
)

# ------------------------------------------------------------------------------------------
# READ 2020 EXPORT
#
# Export order:
# Band 1 = LULC_2020
# Band 2 = DW_Obs_Count_2020
# ------------------------------------------------------------------------------------------
with rasterio.open(TARGET_2020_FILE) as src:

    lulc_2020_10m = src.read(1)
    obs_2020_10m = src.read(2)

    transform_2020 = src.transform
    crs_2020 = src.crs

    width_2020 = src.width
    height_2020 = src.height

valid_2020 = (
    np.isfinite(lulc_2020_10m)
    &
    np.isfinite(obs_2020_10m)
    &
    (lulc_2020_10m >= 0)
    &
    (lulc_2020_10m <= 8)
    &
    (obs_2020_10m >= 1)
)

labels_2020 = lulc_2020_10m[
    valid_2020
].astype(np.uint8)

obs_values_2020 = obs_2020_10m[
    valid_2020
].astype(np.float32)

pixel_area_2020_km2 = (
    abs(transform_2020.a) *
    abs(transform_2020.e) /
    1_000_000
)

print("\nLOCAL 2020 LAND-COVER DISTRIBUTION")
print("-" * 120)

local_records = []

for class_id in range(9):

    pixel_count = int(
        np.count_nonzero(
            labels_2020 == class_id
        )
    )

    area_km2 = (
        pixel_count *
        pixel_area_2020_km2
    )

    percentage = (
        pixel_count /
        labels_2020.size *
        100
        if labels_2020.size > 0
        else 0
    )

    local_records.append({
        "Class_ID": class_id,
        "Class_Name": CLASS_NAMES[class_id],
        "Pixel_Count": pixel_count,
        "Area_km2": area_km2,
        "Percent": percentage
    })

local_2020_df = pd.DataFrame(
    local_records
)

print(
    local_2020_df.to_string(
        index=False,
        formatters={
            "Area_km2": lambda x: f"{x:,.2f}",
            "Percent": lambda x: f"{x:.3f}%"
        }
    )
)

local_built_2020_area = float(
    local_2020_df.loc[
        local_2020_df["Class_ID"] == 6,
        "Area_km2"
    ].iloc[0]
)

print("\nLOCAL 2020 OBSERVATION COVERAGE")
print("-" * 120)

print(
    f"Valid pixels: {labels_2020.size:,}"
)
print(
    f"Minimum observations: {obs_values_2020.min():.0f}"
)
print(
    f"Maximum observations: {obs_values_2020.max():.0f}"
)
print(
    f"Mean observations: {obs_values_2020.mean():.3f}"
)
print(
    f"Median observations: {np.median(obs_values_2020):.3f}"
)

# ------------------------------------------------------------------------------------------
# READ 2025 DYNAMIC WORLD RASTER
# ------------------------------------------------------------------------------------------
with rasterio.open(DW_2025_FILE) as src:

    if src.count != 2:

        raise RuntimeError(
            f"Expected two bands in 2025 raster, found {src.count}."
        )

    lulc_2025_10m = src.read(1)
    obs_2025_10m = src.read(2)

    transform_2025 = src.transform
    crs_2025 = src.crs

print("\n✓ Validated 2025 Dynamic World raster loaded")

# ------------------------------------------------------------------------------------------
# ALIGN LAND COVER AND OBSERVATION COUNTS TO 30 m
# ------------------------------------------------------------------------------------------
print("\nALIGNING BOTH YEARS TO THE 30 m REFERENCE GRID")
print("-" * 120)

lulc_2020_30m = np.full(
    (ref_height, ref_width),
    255,
    dtype=np.uint8
)

lulc_2025_30m = np.full(
    (ref_height, ref_width),
    255,
    dtype=np.uint8
)

obs_2020_30m = np.full(
    (ref_height, ref_width),
    np.nan,
    dtype=np.float32
)

obs_2025_30m = np.full(
    (ref_height, ref_width),
    np.nan,
    dtype=np.float32
)

# Categorical labels — nearest-neighbour resampling.
reproject(
    source=lulc_2020_10m,
    destination=lulc_2020_30m,
    src_transform=transform_2020,
    src_crs=crs_2020,
    dst_transform=ref_transform,
    dst_crs=ref_crs,
    src_nodata=None,
    dst_nodata=255,
    resampling=Resampling.nearest
)

reproject(
    source=lulc_2025_10m,
    destination=lulc_2025_30m,
    src_transform=transform_2025,
    src_crs=crs_2025,
    dst_transform=ref_transform,
    dst_crs=ref_crs,
    src_nodata=None,
    dst_nodata=255,
    resampling=Resampling.nearest
)

# Observation counts — average contributing 10 m cells.
reproject(
    source=obs_2020_10m.astype(np.float32),
    destination=obs_2020_30m,
    src_transform=transform_2020,
    src_crs=crs_2020,
    dst_transform=ref_transform,
    dst_crs=ref_crs,
    src_nodata=None,
    dst_nodata=np.nan,
    resampling=Resampling.average
)

reproject(
    source=obs_2025_10m.astype(np.float32),
    destination=obs_2025_30m,
    src_transform=transform_2025,
    src_crs=crs_2025,
    dst_transform=ref_transform,
    dst_crs=ref_crs,
    src_nodata=None,
    dst_nodata=np.nan,
    resampling=Resampling.average
)

# Release large 10 m arrays after reprojection.
del lulc_2020_10m
del obs_2020_10m
del lulc_2025_10m
del obs_2025_10m
del labels_2020
del obs_values_2020

gc.collect()

common_valid = (
    study_mask
    &
    (lulc_2020_30m <= 8)
    &
    (lulc_2025_30m <= 8)
    &
    np.isfinite(obs_2020_30m)
    &
    np.isfinite(obs_2025_30m)
    &
    (obs_2020_30m >= 1)
    &
    (obs_2025_30m >= 1)
)

common_count = int(
    common_valid.sum()
)

common_area_km2 = (
    common_count *
    pixel_area_km2
)

print(
    f"Common valid pixels: {common_count:,}"
)
print(
    f"Common valid area: {common_area_km2:,.2f} km²"
)

if common_count == 0:

    raise RuntimeError(
        "No common valid pixels were produced after alignment."
    )

# ------------------------------------------------------------------------------------------
# CREATE BUILT-UP TRANSITION CLASSES
#
# 0 = Stable non-built
# 1 = Stable built
# 2 = Apparent built-up gain
# 3 = Apparent built-up loss
# 255 = NoData
# ------------------------------------------------------------------------------------------
built_2020 = (
    lulc_2020_30m == 6
)

built_2025 = (
    lulc_2025_30m == 6
)

transition = np.full(
    (ref_height, ref_width),
    255,
    dtype=np.uint8
)

transition[
    common_valid
    &
    (~built_2020)
    &
    (~built_2025)
] = 0

transition[
    common_valid
    &
    built_2020
    &
    built_2025
] = 1

transition[
    common_valid
    &
    (~built_2020)
    &
    built_2025
] = 2

transition[
    common_valid
    &
    built_2020
    &
    (~built_2025)
] = 3

transition_names = {
    0: "Stable non-built",
    1: "Stable built",
    2: "Apparent built-up gain",
    3: "Apparent built-up loss"
}

print("\n30 m BUILT-UP TRANSITION RESULTS")
print("-" * 120)

transition_records = []

for transition_id in range(4):

    count = int(
        np.count_nonzero(
            transition == transition_id
        )
    )

    area = (
        count *
        pixel_area_km2
    )

    percent = (
        count /
        common_count *
        100
    )

    transition_records.append({
        "Transition_ID": transition_id,
        "Transition": transition_names[transition_id],
        "Pixel_Count": count,
        "Area_km2": area,
        "Percent_of_Common_Area": percent
    })

transition_df = pd.DataFrame(
    transition_records
)

print(
    transition_df.to_string(
        index=False,
        formatters={
            "Area_km2": lambda x: f"{x:,.2f}",
            "Percent_of_Common_Area":
                lambda x: f"{x:.3f}%"
        }
    )
)

stable_built_area = float(
    transition_df.loc[
        transition_df["Transition_ID"] == 1,
        "Area_km2"
    ].iloc[0]
)

gain_area = float(
    transition_df.loc[
        transition_df["Transition_ID"] == 2,
        "Area_km2"
    ].iloc[0]
)

loss_area = float(
    transition_df.loc[
        transition_df["Transition_ID"] == 3,
        "Area_km2"
    ].iloc[0]
)

built_2020_area_30m = (
    stable_built_area +
    loss_area
)

built_2025_area_30m = (
    stable_built_area +
    gain_area
)

persistence_rate = (
    stable_built_area /
    built_2020_area_30m *
    100
    if built_2020_area_30m > 0
    else np.nan
)

gain_share_2025 = (
    gain_area /
    built_2025_area_30m *
    100
    if built_2025_area_30m > 0
    else np.nan
)

loss_to_gain_ratio = (
    loss_area /
    gain_area
    if gain_area > 0
    else np.inf
)

net_change_area = (
    gain_area -
    loss_area
)

print("\nBUILT-UP STABILITY METRICS")
print("-" * 120)

print(
    f"2020 built-up area at 30 m: "
    f"{built_2020_area_30m:,.2f} km²"
)

print(
    f"2025 built-up area at 30 m: "
    f"{built_2025_area_30m:,.2f} km²"
)

print(
    f"Stable built-up: "
    f"{stable_built_area:,.2f} km²"
)

print(
    f"Apparent built-up gain: "
    f"{gain_area:,.2f} km²"
)

print(
    f"Apparent built-up loss: "
    f"{loss_area:,.2f} km²"
)

print(
    f"Net built-up change: "
    f"{net_change_area:+,.2f} km²"
)

print(
    f"2020 built-up persistence rate: "
    f"{persistence_rate:.2f}%"
)

print(
    f"Share of 2025 built-up classified as gain: "
    f"{gain_share_2025:.2f}%"
)

print(
    f"Apparent loss-to-gain ratio: "
    f"{loss_to_gain_ratio:.3f}"
)

# ------------------------------------------------------------------------------------------
# OBSERVATION-COUNT SENSITIVITY
# ------------------------------------------------------------------------------------------
print("\nOBSERVATION-COUNT FILTER SENSITIVITY")
print("-" * 120)

filter_records = []

for threshold in [1, 3, 5, 7, 10]:

    quality_mask = (
        common_valid
        &
        (obs_2020_30m >= threshold)
        &
        (obs_2025_30m >= threshold)
    )

    valid_count = int(
        quality_mask.sum()
    )

    if valid_count == 0:
        continue

    stable_count = int(
        np.count_nonzero(
            quality_mask
            &
            built_2020
            &
            built_2025
        )
    )

    gain_count = int(
        np.count_nonzero(
            quality_mask
            &
            (~built_2020)
            &
            built_2025
        )
    )

    loss_count = int(
        np.count_nonzero(
            quality_mask
            &
            built_2020
            &
            (~built_2025)
        )
    )

    baseline_built_count = (
        stable_count +
        loss_count
    )

    outcome_built_count = (
        stable_count +
        gain_count
    )

    persistence = (
        stable_count /
        baseline_built_count *
        100
        if baseline_built_count > 0
        else np.nan
    )

    loss_gain_ratio = (
        loss_count /
        gain_count
        if gain_count > 0
        else np.inf
    )

    filter_records.append({
        "Minimum_Observations_Both_Years": threshold,
        "Valid_Pixels": valid_count,
        "Valid_Area_km2":
            valid_count * pixel_area_km2,
        "Stable_Built_Area_km2":
            stable_count * pixel_area_km2,
        "Apparent_Gain_Area_km2":
            gain_count * pixel_area_km2,
        "Apparent_Loss_Area_km2":
            loss_count * pixel_area_km2,
        "Net_Change_Area_km2":
            (gain_count - loss_count) * pixel_area_km2,
        "Built_Persistence_Percent":
            persistence,
        "Loss_to_Gain_Ratio":
            loss_gain_ratio
    })

filter_df = pd.DataFrame(
    filter_records
)

print(
    filter_df.to_string(
        index=False,
        formatters={
            "Valid_Area_km2":
                lambda x: f"{x:,.2f}",
            "Stable_Built_Area_km2":
                lambda x: f"{x:,.2f}",
            "Apparent_Gain_Area_km2":
                lambda x: f"{x:,.2f}",
            "Apparent_Loss_Area_km2":
                lambda x: f"{x:,.2f}",
            "Net_Change_Area_km2":
                lambda x: f"{x:+,.2f}",
            "Built_Persistence_Percent":
                lambda x: f"{x:.2f}%",
            "Loss_to_Gain_Ratio":
                lambda x: f"{x:.3f}"
        }
    )
)

# ------------------------------------------------------------------------------------------
# SAVE TRANSITION RASTER
# ------------------------------------------------------------------------------------------
transition_profile = ref_profile.copy()

transition_profile.update(
    dtype="uint8",
    count=1,
    nodata=255,
    compress="deflate"
)

with rasterio.open(
    TRANSITION_FILE,
    "w",
    **transition_profile
) as dst:

    dst.write(
        transition,
        1
    )

    dst.set_band_description(
        1,
        "BuiltUp_Transition_QA_2020_2025"
    )

# ------------------------------------------------------------------------------------------
# SAVE TABLES
# ------------------------------------------------------------------------------------------
transition_df.to_csv(
    TRANSITION_TABLE_FILE,
    index=False
)

filter_df.to_csv(
    OBS_FILTER_TABLE_FILE,
    index=False
)

# ------------------------------------------------------------------------------------------
# REOPEN AND VALIDATE SAVED TRANSITION RASTER
# ------------------------------------------------------------------------------------------
with rasterio.open(TRANSITION_FILE) as src:

    saved_transition = src.read(1)

    saved_epsg = (
        src.crs.to_epsg()
        if src.crs is not None
        else None
    )

    saved_width = src.width
    saved_height = src.height

    saved_res_x = abs(src.transform.a)
    saved_res_y = abs(src.transform.e)

    saved_bounds = src.bounds

saved_valid = (
    saved_transition != 255
)

saved_ids = sorted(
    np.unique(
        saved_transition[saved_valid]
    ).tolist()
)

same_dimensions = (
    saved_width == ref_width
    and
    saved_height == ref_height
)

same_resolution = (
    np.isclose(saved_res_x, ref_res_x)
    and
    np.isclose(saved_res_y, ref_res_y)
)

same_bounds = np.allclose(
    [
        saved_bounds.left,
        saved_bounds.bottom,
        saved_bounds.right,
        saved_bounds.top
    ],
    [
        ref_bounds.left,
        ref_bounds.bottom,
        ref_bounds.right,
        ref_bounds.top
    ],
    atol=0.001
)

# ------------------------------------------------------------------------------------------
# SAVE VALIDATION REGISTER
# ------------------------------------------------------------------------------------------
validation_df = pd.DataFrame([{
    "Source_2020_File":
        str(SOURCE_2020_FILE),

    "Permanent_2020_File":
        str(TARGET_2020_FILE),

    "Local_2020_Built_Area_10m_km2":
        local_built_2020_area,

    "Common_Transition_Pixels":
        common_count,

    "Common_Transition_Area_km2":
        common_area_km2,

    "Built_2020_Area_30m_km2":
        built_2020_area_30m,

    "Built_2025_Area_30m_km2":
        built_2025_area_30m,

    "Stable_Built_Area_km2":
        stable_built_area,

    "Apparent_Gain_Area_km2":
        gain_area,

    "Apparent_Loss_Area_km2":
        loss_area,

    "Net_Change_Area_km2":
        net_change_area,

    "Built_Persistence_Percent":
        persistence_rate,

    "Gain_Share_of_2025_Built_Percent":
        gain_share_2025,

    "Loss_to_Gain_Ratio":
        loss_to_gain_ratio,

    "Transition_IDs":
        str(saved_ids),

    "Aligned_With_Reference":
        (
            same_dimensions
            and same_resolution
            and same_bounds
        ),

    "ML_Labels_Accepted":
        False
}])

validation_df.to_csv(
    VALIDATION_FILE,
    index=False
)

# ------------------------------------------------------------------------------------------
# FINAL VALIDATION
# ------------------------------------------------------------------------------------------
checks = {
    "Located export passed structural validation":
        all(source_checks.values()),

    "Permanent 2020 raster saved":
        TARGET_2020_FILE.exists(),

    "Local 2020 built-up area positive":
        local_built_2020_area > 0,

    "Common transition pixels present":
        common_count > 0,

    "All four transition classes represented":
        saved_ids == [0, 1, 2, 3],

    "Transition raster uses EPSG:32632":
        saved_epsg == 32632,

    "Transition dimensions match reference":
        same_dimensions,

    "Transition resolution matches reference":
        same_resolution,

    "Transition bounds match reference":
        same_bounds,

    "Transition table saved":
        TRANSITION_TABLE_FILE.exists(),

    "Observation-filter table saved":
        OBS_FILTER_TABLE_FILE.exists(),

    "Transition raster saved":
        TRANSITION_FILE.exists(),

    "Validation register saved":
        VALIDATION_FILE.exists(),

    "ML labels remain unaccepted":
        validation_df[
            "ML_Labels_Accepted"
        ].iloc[0] == False
}

print("\nVALIDATION SUMMARY")
print("-" * 120)

for label, passed in checks.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{label}"
    )

print("\nOUTPUT FILES")
print("-" * 120)

for path in [
    TARGET_2020_FILE,
    TRANSITION_FILE,
    TRANSITION_TABLE_FILE,
    OBS_FILTER_TABLE_FILE,
    VALIDATION_FILE
]:

    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{path.name}"
    )

if all(checks.values()):

    print("\n✓ STAGE 9B FULLY PASSED")

    print(
        "The 2020 export and pixel-level 2020–2025 "
        "built-up transitions are validated."
    )

    print(
        "ML labels remain intentionally unaccepted "
        "until transition reliability is assessed."
    )

else:

    print("\n⚠ STAGE 9B REQUIRES REVIEW")

print("=" * 120)

In [ ]:
# ==========================================================================================
# PROJECT 7 — STAGE 9C
# SPATIAL RELIABILITY DIAGNOSTICS FOR 2020–2025 BUILT-UP TRANSITIONS
#
# This stage does NOT create final ML labels.
#
# It evaluates:
# 1. Connected-component sizes of apparent built-up gains and losses.
# 2. Distance of apparent gains to stable built-up areas.
# 3. Local 2025 built-up neighbourhood support.
# 4. Sensitivity to minimum patch size and spatial-support thresholds.
# 5. The amount of apparent gain that could qualify as a reliable expansion candidate.
# ==========================================================================================

from pathlib import Path
import gc

import numpy as np
import pandas as pd
import rasterio

from rasterio.warp import reproject, Resampling
from scipy import ndimage

# ------------------------------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------------------------------
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

LANDCOVER_DIR = PROJECT_ROOT / "04_Land_Cover"
TRAINING_DIR = PROJECT_ROOT / "09_Training_Data"
TABLE_DIR = PROJECT_ROOT / "16_Tables"

DW_2020_FILE = (
    LANDCOVER_DIR /
    "Enugu_DynamicWorld_2020.tif"
)

DW_2025_FILE = (
    LANDCOVER_DIR /
    "Enugu_DynamicWorld_2025.tif"
)

REFERENCE_FILE = (
    PROJECT_ROOT /
    "02_Terrain" /
    "Enugu_Elevation_30m.tif"
)

TRANSITION_FILE = (
    TRAINING_DIR /
    "Enugu_BuiltUp_Transition_QA_2020_2025_30m.tif"
)

PATCH_SUMMARY_FILE = (
    TABLE_DIR /
    "Enugu_Transition_Patch_Spatial_QA.csv"
)

GAIN_SUPPORT_FILE = (
    TABLE_DIR /
    "Enugu_Apparent_Gain_Spatial_Support_QA.csv"
)

THRESHOLD_SENSITIVITY_FILE = (
    TABLE_DIR /
    "Enugu_Expansion_Candidate_Threshold_Sensitivity.csv"
)

DIAGNOSTIC_RASTER_FILE = (
    TRAINING_DIR /
    "Enugu_Apparent_Gain_Spatial_Reliability_30m.tif"
)

VALIDATION_FILE = (
    TRAINING_DIR /
    "Stage_9C_Spatial_Reliability_Validation.csv"
)

print("=" * 120)
print("STAGE 9C — SPATIAL RELIABILITY DIAGNOSTICS FOR BUILT-UP TRANSITIONS")
print("=" * 120)

# ------------------------------------------------------------------------------------------
# REQUIRED FILE CHECK
# ------------------------------------------------------------------------------------------
required_files = {
    "Dynamic World 2020": DW_2020_FILE,
    "Dynamic World 2025": DW_2025_FILE,
    "30 m reference raster": REFERENCE_FILE,
    "Stage 9B transition raster": TRANSITION_FILE
}

print("\nREQUIRED FILE CHECK")
print("-" * 120)

for label, path in required_files.items():

    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{label}: {path}"
    )

if not all(path.exists() for path in required_files.values()):

    raise FileNotFoundError(
        "One or more required Stage 9C inputs are missing."
    )

# ------------------------------------------------------------------------------------------
# READ REFERENCE GRID
# ------------------------------------------------------------------------------------------
with rasterio.open(REFERENCE_FILE) as ref:

    elevation = ref.read(1)

    ref_profile = ref.profile.copy()
    ref_transform = ref.transform
    ref_crs = ref.crs
    ref_width = ref.width
    ref_height = ref.height
    ref_bounds = ref.bounds

    ref_res_x = abs(ref.transform.a)
    ref_res_y = abs(ref.transform.e)

study_mask = np.isfinite(elevation)

pixel_area_km2 = (
    ref_res_x *
    ref_res_y /
    1_000_000
)

pixel_area_ha = (
    ref_res_x *
    ref_res_y /
    10_000
)

print("\nREFERENCE GRID")
print("-" * 120)

print("CRS:", ref_crs)
print(
    f"Dimensions: {ref_width} × {ref_height}"
)
print(
    f"Resolution: {ref_res_x:.1f} × {ref_res_y:.1f} m"
)
print(
    f"Pixel area: {pixel_area_ha:.3f} ha"
)

# ------------------------------------------------------------------------------------------
# READ TRANSITION RASTER
#
# 0 = Stable non-built
# 1 = Stable built
# 2 = Apparent gain
# 3 = Apparent loss
# 255 = NoData
# ------------------------------------------------------------------------------------------
with rasterio.open(TRANSITION_FILE) as src:

    transition = src.read(1)

    transition_epsg = (
        src.crs.to_epsg()
        if src.crs is not None
        else None
    )

    transition_width = src.width
    transition_height = src.height

stable_nonbuilt = (
    transition == 0
)

stable_built = (
    transition == 1
)

apparent_gain = (
    transition == 2
)

apparent_loss = (
    transition == 3
)

valid_transition = (
    transition != 255
)

print("\nTRANSITION INPUT")
print("-" * 120)

print(
    f"Stable non-built pixels: {stable_nonbuilt.sum():,}"
)
print(
    f"Stable built pixels: {stable_built.sum():,}"
)
print(
    f"Apparent gain pixels: {apparent_gain.sum():,}"
)
print(
    f"Apparent loss pixels: {apparent_loss.sum():,}"
)

# ------------------------------------------------------------------------------------------
# ALIGN OBSERVATION COUNTS AND 2025 LAND COVER TO 30 m
# ------------------------------------------------------------------------------------------
def read_dynamic_world(path):

    with rasterio.open(path) as src:

        lulc = src.read(1)
        observations = src.read(2).astype(np.float32)

        transform = src.transform
        crs = src.crs

    return lulc, observations, transform, crs


lulc_2020_10m, obs_2020_10m, transform_2020, crs_2020 = (
    read_dynamic_world(DW_2020_FILE)
)

lulc_2025_10m, obs_2025_10m, transform_2025, crs_2025 = (
    read_dynamic_world(DW_2025_FILE)
)

lulc_2025_30m = np.full(
    (ref_height, ref_width),
    255,
    dtype=np.uint8
)

obs_2020_30m = np.full(
    (ref_height, ref_width),
    np.nan,
    dtype=np.float32
)

obs_2025_30m = np.full(
    (ref_height, ref_width),
    np.nan,
    dtype=np.float32
)

reproject(
    source=lulc_2025_10m,
    destination=lulc_2025_30m,
    src_transform=transform_2025,
    src_crs=crs_2025,
    dst_transform=ref_transform,
    dst_crs=ref_crs,
    src_nodata=None,
    dst_nodata=255,
    resampling=Resampling.nearest
)

reproject(
    source=obs_2020_10m,
    destination=obs_2020_30m,
    src_transform=transform_2020,
    src_crs=crs_2020,
    dst_transform=ref_transform,
    dst_crs=ref_crs,
    src_nodata=None,
    dst_nodata=np.nan,
    resampling=Resampling.average
)

reproject(
    source=obs_2025_10m,
    destination=obs_2025_30m,
    src_transform=transform_2025,
    src_crs=crs_2025,
    dst_transform=ref_transform,
    dst_crs=ref_crs,
    src_nodata=None,
    dst_nodata=np.nan,
    resampling=Resampling.average
)

del lulc_2020_10m
del obs_2020_10m
del lulc_2025_10m
del obs_2025_10m

gc.collect()

# ------------------------------------------------------------------------------------------
# QUALITY MASK
#
# Minimum of five observations in both years.
# This retained approximately 5,088.54 km² in Stage 9B.
# ------------------------------------------------------------------------------------------
quality_mask_5 = (
    valid_transition
    &
    study_mask
    &
    np.isfinite(obs_2020_30m)
    &
    np.isfinite(obs_2025_30m)
    &
    (obs_2020_30m >= 5)
    &
    (obs_2025_30m >= 5)
)

quality_gain = (
    apparent_gain
    &
    quality_mask_5
)

quality_loss = (
    apparent_loss
    &
    quality_mask_5
)

quality_stable_built = (
    stable_built
    &
    quality_mask_5
)

print("\nOBSERVATION-QUALITY FILTER")
print("-" * 120)

print(
    f"Valid area at ≥5 observations in both years: "
    f"{quality_mask_5.sum() * pixel_area_km2:,.2f} km²"
)

print(
    f"Apparent gain retained: "
    f"{quality_gain.sum() * pixel_area_km2:,.2f} km²"
)

print(
    f"Apparent loss retained: "
    f"{quality_loss.sum() * pixel_area_km2:,.2f} km²"
)

# ------------------------------------------------------------------------------------------
# CONNECTED-COMPONENT ANALYSIS
#
# Eight-neighbour connectivity is used because diagonally touching urban pixels
# can belong to the same continuous settlement patch.
# ------------------------------------------------------------------------------------------
eight_neighbour_structure = np.ones(
    (3, 3),
    dtype=np.uint8
)

gain_labels, gain_component_count = ndimage.label(
    quality_gain,
    structure=eight_neighbour_structure
)

loss_labels, loss_component_count = ndimage.label(
    quality_loss,
    structure=eight_neighbour_structure
)

gain_sizes = np.bincount(
    gain_labels.ravel()
)

loss_sizes = np.bincount(
    loss_labels.ravel()
)

gain_sizes[0] = 0
loss_sizes[0] = 0

gain_patch_size_pixels = gain_sizes[
    gain_labels
]

loss_patch_size_pixels = loss_sizes[
    loss_labels
]

print("\nCONNECTED-COMPONENT RESULTS")
print("-" * 120)

print(
    f"Apparent-gain patches: {gain_component_count:,}"
)

print(
    f"Apparent-loss patches: {loss_component_count:,}"
)

# ------------------------------------------------------------------------------------------
# PATCH-SIZE SUMMARY
# ------------------------------------------------------------------------------------------
patch_bins = [
    (1, 1, "1 pixel"),
    (2, 2, "2 pixels"),
    (3, 4, "3–4 pixels"),
    (5, 9, "5–9 pixels"),
    (10, 24, "10–24 pixels"),
    (25, 99, "25–99 pixels"),
    (100, np.inf, "≥100 pixels")
]

patch_records = []

for transition_name, component_sizes in [
    ("Apparent gain", gain_sizes[1:]),
    ("Apparent loss", loss_sizes[1:])
]:

    total_pixels = int(
        component_sizes.sum()
    )

    for minimum, maximum, label in patch_bins:

        selected_components = (
            (component_sizes >= minimum)
            &
            (component_sizes <= maximum)
        )

        component_count = int(
            selected_components.sum()
        )

        pixels_in_bin = int(
            component_sizes[
                selected_components
            ].sum()
        )

        area_km2 = (
            pixels_in_bin *
            pixel_area_km2
        )

        share_percent = (
            pixels_in_bin /
            total_pixels *
            100
            if total_pixels > 0
            else 0
        )

        patch_records.append({
            "Transition": transition_name,
            "Patch_Size_Class": label,
            "Minimum_Pixels": minimum,
            "Maximum_Pixels":
                maximum if np.isfinite(maximum) else None,
            "Component_Count": component_count,
            "Pixel_Count": pixels_in_bin,
            "Area_km2": area_km2,
            "Share_of_Transition_Area_Percent":
                share_percent
        })

patch_df = pd.DataFrame(
    patch_records
)

print("\nPATCH-SIZE DISTRIBUTION")
print("-" * 120)

print(
    patch_df.to_string(
        index=False,
        formatters={
            "Area_km2":
                lambda x: f"{x:,.2f}",
            "Share_of_Transition_Area_Percent":
                lambda x: f"{x:.2f}%"
        }
    )
)

# ------------------------------------------------------------------------------------------
# DISTANCE TO STABLE BUILT-UP
# ------------------------------------------------------------------------------------------
distance_to_stable_built_pixels = (
    ndimage.distance_transform_edt(
        ~quality_stable_built
    )
)

distance_to_stable_built_m = (
    distance_to_stable_built_pixels *
    ref_res_x
)

gain_distances = distance_to_stable_built_m[
    quality_gain
]

# ------------------------------------------------------------------------------------------
# LOCAL 2025 BUILT-UP SUPPORT
#
# Calculates the percentage of cells classified as built-up within:
# - 3 × 3 neighbourhood: approximately 90 × 90 m
# - 5 × 5 neighbourhood: approximately 150 × 150 m
# ------------------------------------------------------------------------------------------
built_2025_30m = (
    lulc_2025_30m == 6
).astype(np.float32)

support_3x3 = ndimage.uniform_filter(
    built_2025_30m,
    size=3,
    mode="constant",
    cval=0
)

support_5x5 = ndimage.uniform_filter(
    built_2025_30m,
    size=5,
    mode="constant",
    cval=0
)

gain_support_3x3 = support_3x3[
    quality_gain
]

gain_support_5x5 = support_5x5[
    quality_gain
]

gain_patch_pixels = gain_patch_size_pixels[
    quality_gain
]

# ------------------------------------------------------------------------------------------
# APPARENT-GAIN DISTRIBUTION STATISTICS
# ------------------------------------------------------------------------------------------
gain_statistics = {
    "Quality_Gain_Pixels":
        int(quality_gain.sum()),

    "Quality_Gain_Area_km2":
        float(quality_gain.sum() * pixel_area_km2),

    "Gain_Component_Count":
        int(gain_component_count),

    "Median_Patch_Size_Pixels":
        float(np.median(gain_patch_pixels)),

    "Median_Patch_Size_ha":
        float(
            np.median(gain_patch_pixels) *
            pixel_area_ha
        ),

    "Median_Distance_to_Stable_Built_m":
        float(np.median(gain_distances)),

    "P75_Distance_to_Stable_Built_m":
        float(np.percentile(gain_distances, 75)),

    "P90_Distance_to_Stable_Built_m":
        float(np.percentile(gain_distances, 90)),

    "Median_3x3_Built_Support":
        float(np.median(gain_support_3x3)),

    "Median_5x5_Built_Support":
        float(np.median(gain_support_5x5))
}

gain_support_df = pd.DataFrame([
    gain_statistics
])

print("\nAPPARENT-GAIN SPATIAL SUPPORT")
print("-" * 120)

for key, value in gain_statistics.items():

    if isinstance(value, float):
        print(
            f"{key}: {value:,.3f}"
        )
    else:
        print(
            f"{key}: {value:,}"
        )

# ------------------------------------------------------------------------------------------
# THRESHOLD SENSITIVITY
#
# No threshold is accepted yet.
# This table shows how much apparent gain survives combinations of:
# - minimum patch size;
# - maximum distance from stable built-up;
# - minimum 3 × 3 built-up support.
# ------------------------------------------------------------------------------------------
threshold_records = []

minimum_patch_options = [
    1,
    3,
    5,
    10
]

maximum_distance_options = [
    30,
    60,
    90,
    150,
    300
]

minimum_support_options = [
    0.33,
    0.50,
    0.67
]

quality_gain_count = int(
    quality_gain.sum()
)

for minimum_patch in minimum_patch_options:

    for maximum_distance in maximum_distance_options:

        for minimum_support in minimum_support_options:

            candidate_mask = (
                quality_gain
                &
                (gain_patch_size_pixels >= minimum_patch)
                &
                (
                    distance_to_stable_built_m
                    <= maximum_distance
                )
                &
                (
                    support_3x3
                    >= minimum_support
                )
            )

            candidate_pixels = int(
                candidate_mask.sum()
            )

            candidate_area_km2 = (
                candidate_pixels *
                pixel_area_km2
            )

            retained_percent = (
                candidate_pixels /
                quality_gain_count *
                100
                if quality_gain_count > 0
                else 0
            )

            candidate_components = np.unique(
                gain_labels[
                    candidate_mask
                ]
            )

            candidate_components = candidate_components[
                candidate_components > 0
            ]

            threshold_records.append({
                "Minimum_Patch_Pixels":
                    minimum_patch,

                "Minimum_Patch_Area_ha":
                    minimum_patch * pixel_area_ha,

                "Maximum_Distance_to_Stable_Built_m":
                    maximum_distance,

                "Minimum_3x3_Built_Support":
                    minimum_support,

                "Candidate_Component_Count":
                    int(candidate_components.size),

                "Candidate_Pixels":
                    candidate_pixels,

                "Candidate_Area_km2":
                    candidate_area_km2,

                "Percent_of_Quality_Gain_Retained":
                    retained_percent
            })

threshold_df = pd.DataFrame(
    threshold_records
)

print("\nSELECTED THRESHOLD COMBINATIONS")
print("-" * 120)

selected_display = threshold_df[
    (
        threshold_df[
            "Minimum_Patch_Pixels"
        ].isin([3, 5, 10])
    )
    &
    (
        threshold_df[
            "Maximum_Distance_to_Stable_Built_m"
        ].isin([60, 90, 150])
    )
    &
    (
        threshold_df[
            "Minimum_3x3_Built_Support"
        ].isin([0.50, 0.67])
    )
].copy()

print(
    selected_display.to_string(
        index=False,
        formatters={
            "Minimum_Patch_Area_ha":
                lambda x: f"{x:.2f}",
            "Minimum_3x3_Built_Support":
                lambda x: f"{x:.2f}",
            "Candidate_Area_km2":
                lambda x: f"{x:,.2f}",
            "Percent_of_Quality_Gain_Retained":
                lambda x: f"{x:.2f}%"
        }
    )
)

# ------------------------------------------------------------------------------------------
# CREATE DIAGNOSTIC RASTER
#
# 0 = Other valid transition area
# 1 = Quality-filtered apparent gain
# 2 = Gain patch with at least 3 pixels
# 3 = Gain patch ≥3 pixels and within 90 m of stable built
# 4 = Spatially supported candidate:
#     patch ≥3 pixels
#     distance ≤90 m
#     3 × 3 built support ≥50%
# 255 = NoData
#
# This remains a diagnostic output, not a final ML target.
# ------------------------------------------------------------------------------------------
diagnostic = np.full(
    (ref_height, ref_width),
    255,
    dtype=np.uint8
)

diagnostic[
    valid_transition
] = 0

diagnostic[
    quality_gain
] = 1

diagnostic[
    quality_gain
    &
    (gain_patch_size_pixels >= 3)
] = 2

diagnostic[
    quality_gain
    &
    (gain_patch_size_pixels >= 3)
    &
    (distance_to_stable_built_m <= 90)
] = 3

provisional_supported_candidate = (
    quality_gain
    &
    (gain_patch_size_pixels >= 3)
    &
    (distance_to_stable_built_m <= 90)
    &
    (support_3x3 >= 0.50)
)

diagnostic[
    provisional_supported_candidate
] = 4

provisional_candidate_pixels = int(
    provisional_supported_candidate.sum()
)

provisional_candidate_area = (
    provisional_candidate_pixels *
    pixel_area_km2
)

provisional_retained_percent = (
    provisional_candidate_pixels /
    quality_gain_count *
    100
    if quality_gain_count > 0
    else 0
)

print("\nPROVISIONAL DIAGNOSTIC COMBINATION")
print("-" * 120)

print(
    "Minimum observations in both years: 5"
)
print(
    "Minimum patch size: 3 pixels "
    f"({3 * pixel_area_ha:.2f} ha)"
)
print(
    "Maximum distance to stable built-up: 90 m"
)
print(
    "Minimum 3 × 3 built-up support: 50%"
)

print(
    f"Candidate area: "
    f"{provisional_candidate_area:,.2f} km²"
)

print(
    f"Quality-filtered apparent gain retained: "
    f"{provisional_retained_percent:.2f}%"
)

# ------------------------------------------------------------------------------------------
# SAVE OUTPUTS
# ------------------------------------------------------------------------------------------
patch_df.to_csv(
    PATCH_SUMMARY_FILE,
    index=False
)

gain_support_df.to_csv(
    GAIN_SUPPORT_FILE,
    index=False
)

threshold_df.to_csv(
    THRESHOLD_SENSITIVITY_FILE,
    index=False
)

diagnostic_profile = ref_profile.copy()

diagnostic_profile.update(
    dtype="uint8",
    count=1,
    nodata=255,
    compress="deflate"
)

with rasterio.open(
    DIAGNOSTIC_RASTER_FILE,
    "w",
    **diagnostic_profile
) as dst:

    dst.write(
        diagnostic,
        1
    )

    dst.set_band_description(
        1,
        "Apparent_Gain_Spatial_Reliability"
    )

# ------------------------------------------------------------------------------------------
# REOPEN DIAGNOSTIC RASTER
# ------------------------------------------------------------------------------------------
with rasterio.open(
    DIAGNOSTIC_RASTER_FILE
) as src:

    saved_diagnostic = src.read(1)

    saved_epsg = (
        src.crs.to_epsg()
        if src.crs is not None
        else None
    )

    saved_width = src.width
    saved_height = src.height

    saved_res_x = abs(src.transform.a)
    saved_res_y = abs(src.transform.e)

    saved_bounds = src.bounds

saved_valid = (
    saved_diagnostic != 255
)

saved_classes = sorted(
    np.unique(
        saved_diagnostic[
            saved_valid
        ]
    ).tolist()
)

same_dimensions = (
    saved_width == ref_width
    and
    saved_height == ref_height
)

same_resolution = (
    np.isclose(saved_res_x, ref_res_x)
    and
    np.isclose(saved_res_y, ref_res_y)
)

same_bounds = np.allclose(
    [
        saved_bounds.left,
        saved_bounds.bottom,
        saved_bounds.right,
        saved_bounds.top
    ],
    [
        ref_bounds.left,
        ref_bounds.bottom,
        ref_bounds.right,
        ref_bounds.top
    ],
    atol=0.001
)

validation_df = pd.DataFrame([{
    "Quality_Observation_Threshold":
        5,

    "Quality_Gain_Area_km2":
        quality_gain_count * pixel_area_km2,

    "Quality_Loss_Area_km2":
        quality_loss.sum() * pixel_area_km2,

    "Gain_Component_Count":
        gain_component_count,

    "Loss_Component_Count":
        loss_component_count,

    "Provisional_Minimum_Patch_Pixels":
        3,

    "Provisional_Maximum_Distance_m":
        90,

    "Provisional_Minimum_3x3_Support":
        0.50,

    "Provisional_Candidate_Area_km2":
        provisional_candidate_area,

    "Provisional_Gain_Retained_Percent":
        provisional_retained_percent,

    "Diagnostic_Classes":
        str(saved_classes),

    "Aligned_With_Reference":
        (
            same_dimensions
            and same_resolution
            and same_bounds
        ),

    "Final_ML_Labels_Accepted":
        False
}])

validation_df.to_csv(
    VALIDATION_FILE,
    index=False
)

# ------------------------------------------------------------------------------------------
# FINAL VALIDATION
# ------------------------------------------------------------------------------------------
checks = {
    "Transition input uses EPSG:32632":
        transition_epsg == 32632,

    "Transition input matches reference dimensions":
        (
            transition_width == ref_width
            and transition_height == ref_height
        ),

    "Observation-quality gain pixels present":
        quality_gain_count > 0,

    "Gain connected components created":
        gain_component_count > 0,

    "Loss connected components created":
        loss_component_count > 0,

    "Gain-distance values calculated":
        gain_distances.size == quality_gain_count,

    "Neighbourhood-support values calculated":
        gain_support_3x3.size == quality_gain_count,

    "Patch summary saved":
        PATCH_SUMMARY_FILE.exists(),

    "Gain-support summary saved":
        GAIN_SUPPORT_FILE.exists(),

    "Threshold-sensitivity table saved":
        THRESHOLD_SENSITIVITY_FILE.exists(),

    "Diagnostic raster saved":
        DIAGNOSTIC_RASTER_FILE.exists(),

    "Diagnostic raster uses EPSG:32632":
        saved_epsg == 32632,

    "Diagnostic dimensions match reference":
        same_dimensions,

    "Diagnostic resolution matches reference":
        same_resolution,

    "Diagnostic bounds match reference":
        same_bounds,

    "Validation register saved":
        VALIDATION_FILE.exists(),

    "Final ML labels remain unaccepted":
        validation_df[
            "Final_ML_Labels_Accepted"
        ].iloc[0] == False
}

print("\nVALIDATION SUMMARY")
print("-" * 120)

for label, passed in checks.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{label}"
    )

print("\nOUTPUT FILES")
print("-" * 120)

for path in [
    PATCH_SUMMARY_FILE,
    GAIN_SUPPORT_FILE,
    THRESHOLD_SENSITIVITY_FILE,
    DIAGNOSTIC_RASTER_FILE,
    VALIDATION_FILE
]:

    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{path.name}"
    )

if all(checks.values()):

    print("\n✓ STAGE 9C FULLY PASSED")

    print(
        "The spatial structure and reliability of apparent "
        "built-up expansion have been quantified."
    )

    print(
        "Final ML training labels remain intentionally "
        "unaccepted pending threshold assessment."
    )

else:

    print("\n⚠ STAGE 9C REQUIRES REVIEW")

print("=" * 120)

In [ ]:
# ==========================================================================================
# PROJECT 7 — STAGE 9D
# FINAL URBAN-EXPANSION LABEL ELIGIBILITY DEFINITION
#
# Target period: 2020–2025
#
# POSITIVE LABEL:
#   Reliable non-built → built transition satisfying:
#   - minimum 5 observations in both years;
#   - transition patch ≥5 connected 30 m pixels (≥0.45 ha);
#   - distance to stable built-up ≤90 m;
#   - 2025 built-up support in 3×3 neighbourhood ≥67%.
#
# ELIGIBLE NEGATIVE:
#   Stable non-built land satisfying:
#   - minimum 5 observations in both years;
#   - not water in either year;
#   - slope ≤15°;
#   - located 90–1,500 m from 2020 built-up;
#   - at least 90 m from an accepted positive label.
#
# Raster classes:
#   0   = Unlabelled/excluded valid area
#   1   = Accepted urban-expansion positive
#   2   = Eligible stable non-built negative
#   255 = NoData
#
# This stage defines eligible labels only.
# Balanced spatial sampling will be performed in the next stage.
# ==========================================================================================

from pathlib import Path
import json
import gc

import numpy as np
import pandas as pd
import rasterio

from rasterio.warp import reproject, Resampling
from scipy import ndimage

# ------------------------------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------------------------------
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

LANDCOVER_DIR = PROJECT_ROOT / "04_Land_Cover"
TERRAIN_DIR = PROJECT_ROOT / "02_Terrain"
TRAINING_DIR = PROJECT_ROOT / "09_Training_Data"
TABLE_DIR = PROJECT_ROOT / "16_Tables"
ADMIN_DIR = PROJECT_ROOT / "00_Project_Admin"

for folder in [
    TRAINING_DIR,
    TABLE_DIR,
    ADMIN_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

DW_2020_FILE = (
    LANDCOVER_DIR /
    "Enugu_DynamicWorld_2020.tif"
)

DW_2025_FILE = (
    LANDCOVER_DIR /
    "Enugu_DynamicWorld_2025.tif"
)

REFERENCE_FILE = (
    TERRAIN_DIR /
    "Enugu_Elevation_30m.tif"
)

SLOPE_FILE = (
    TERRAIN_DIR /
    "Enugu_Slope_30m.tif"
)

TRANSITION_FILE = (
    TRAINING_DIR /
    "Enugu_BuiltUp_Transition_QA_2020_2025_30m.tif"
)

LABEL_RASTER_FILE = (
    TRAINING_DIR /
    "Enugu_ML_Label_Eligibility_2020_2025_30m.tif"
)

POSITIVE_COMPONENT_FILE = (
    TABLE_DIR /
    "Enugu_Accepted_Expansion_Positive_Components.csv"
)

LABEL_SUMMARY_FILE = (
    TABLE_DIR /
    "Enugu_ML_Label_Eligibility_Summary.csv"
)

VALIDATION_FILE = (
    TRAINING_DIR /
    "Stage_9D_Label_Eligibility_Validation.csv"
)

DESIGN_FILE = (
    ADMIN_DIR /
    "ML_Label_Design_2020_2025.json"
)

print("=" * 120)
print("STAGE 9D — FINAL URBAN-EXPANSION LABEL ELIGIBILITY DEFINITION")
print("=" * 120)

# ------------------------------------------------------------------------------------------
# FIND SLOPE FILE IF THE STANDARD NAME DIFFERS
# ------------------------------------------------------------------------------------------
if not SLOPE_FILE.exists():

    slope_candidates = sorted(
        TERRAIN_DIR.glob("*Slope*30m*.tif")
    )

    if not slope_candidates:
        slope_candidates = sorted(
            TERRAIN_DIR.glob("*slope*.tif")
        )

    if slope_candidates:
        SLOPE_FILE = slope_candidates[0]

# ------------------------------------------------------------------------------------------
# REQUIRED INPUT CHECK
# ------------------------------------------------------------------------------------------
required_files = {
    "Dynamic World 2020": DW_2020_FILE,
    "Dynamic World 2025": DW_2025_FILE,
    "Elevation reference": REFERENCE_FILE,
    "Slope raster": SLOPE_FILE,
    "Stage 9B transition raster": TRANSITION_FILE
}

print("\nREQUIRED FILE CHECK")
print("-" * 120)

for label, path in required_files.items():

    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{label}: {path}"
    )

if not all(
    path.exists()
    for path in required_files.values()
):
    raise FileNotFoundError(
        "One or more required Stage 9D inputs are missing."
    )

# ------------------------------------------------------------------------------------------
# READ REFERENCE GRID
# ------------------------------------------------------------------------------------------
with rasterio.open(REFERENCE_FILE) as ref:

    elevation = ref.read(1).astype(np.float32)

    ref_profile = ref.profile.copy()
    ref_transform = ref.transform
    ref_crs = ref.crs
    ref_width = ref.width
    ref_height = ref.height
    ref_bounds = ref.bounds

    ref_res_x = abs(ref.transform.a)
    ref_res_y = abs(ref.transform.e)

study_mask = np.isfinite(elevation)

pixel_area_km2 = (
    ref_res_x *
    ref_res_y /
    1_000_000
)

pixel_area_ha = (
    ref_res_x *
    ref_res_y /
    10_000
)

print("\nREFERENCE GRID")
print("-" * 120)

print("CRS:", ref_crs)
print(
    f"Dimensions: {ref_width} × {ref_height}"
)
print(
    f"Resolution: {ref_res_x:.1f} × {ref_res_y:.1f} m"
)
print(
    f"Pixel area: {pixel_area_ha:.3f} ha"
)

# ------------------------------------------------------------------------------------------
# READ AND ALIGN SLOPE
# ------------------------------------------------------------------------------------------
with rasterio.open(SLOPE_FILE) as src:

    if (
        src.width == ref_width
        and
        src.height == ref_height
        and
        src.crs == ref_crs
        and
        np.allclose(
            tuple(src.transform),
            tuple(ref_transform)
        )
    ):

        slope = src.read(1).astype(np.float32)

    else:

        slope = np.full(
            (ref_height, ref_width),
            np.nan,
            dtype=np.float32
        )

        reproject(
            source=src.read(1).astype(np.float32),
            destination=slope,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=ref_transform,
            dst_crs=ref_crs,
            src_nodata=src.nodata,
            dst_nodata=np.nan,
            resampling=Resampling.bilinear
        )

valid_slope = np.isfinite(slope)

print("\n✓ Slope raster loaded and aligned")

# ------------------------------------------------------------------------------------------
# READ TRANSITION RASTER
# ------------------------------------------------------------------------------------------
with rasterio.open(TRANSITION_FILE) as src:

    transition = src.read(1)

    transition_epsg = (
        src.crs.to_epsg()
        if src.crs is not None
        else None
    )

    transition_width = src.width
    transition_height = src.height

stable_nonbuilt = transition == 0
stable_built = transition == 1
apparent_gain = transition == 2
valid_transition = transition != 255

# ------------------------------------------------------------------------------------------
# READ AND ALIGN DYNAMIC WORLD RASTERS
# ------------------------------------------------------------------------------------------
def read_dynamic_world(path):

    with rasterio.open(path) as src:

        lulc = src.read(1)
        observations = src.read(2).astype(np.float32)
        transform = src.transform
        crs = src.crs

    return lulc, observations, transform, crs


(
    lulc_2020_10m,
    obs_2020_10m,
    transform_2020,
    crs_2020
) = read_dynamic_world(DW_2020_FILE)

(
    lulc_2025_10m,
    obs_2025_10m,
    transform_2025,
    crs_2025
) = read_dynamic_world(DW_2025_FILE)

lulc_2020_30m = np.full(
    (ref_height, ref_width),
    255,
    dtype=np.uint8
)

lulc_2025_30m = np.full(
    (ref_height, ref_width),
    255,
    dtype=np.uint8
)

obs_2020_30m = np.full(
    (ref_height, ref_width),
    np.nan,
    dtype=np.float32
)

obs_2025_30m = np.full(
    (ref_height, ref_width),
    np.nan,
    dtype=np.float32
)

for source, destination, transform, crs in [
    (
        lulc_2020_10m,
        lulc_2020_30m,
        transform_2020,
        crs_2020
    ),
    (
        lulc_2025_10m,
        lulc_2025_30m,
        transform_2025,
        crs_2025
    )
]:

    reproject(
        source=source,
        destination=destination,
        src_transform=transform,
        src_crs=crs,
        dst_transform=ref_transform,
        dst_crs=ref_crs,
        src_nodata=None,
        dst_nodata=255,
        resampling=Resampling.nearest
    )

for source, destination, transform, crs in [
    (
        obs_2020_10m,
        obs_2020_30m,
        transform_2020,
        crs_2020
    ),
    (
        obs_2025_10m,
        obs_2025_30m,
        transform_2025,
        crs_2025
    )
]:

    reproject(
        source=source,
        destination=destination,
        src_transform=transform,
        src_crs=crs,
        dst_transform=ref_transform,
        dst_crs=ref_crs,
        src_nodata=None,
        dst_nodata=np.nan,
        resampling=Resampling.average
    )

del lulc_2020_10m
del lulc_2025_10m
del obs_2020_10m
del obs_2025_10m

gc.collect()

# ------------------------------------------------------------------------------------------
# COMMON OBSERVATION-QUALITY MASK
# ------------------------------------------------------------------------------------------
quality_mask = (
    study_mask
    &
    valid_transition
    &
    valid_slope
    &
    np.isfinite(obs_2020_30m)
    &
    np.isfinite(obs_2025_30m)
    &
    (obs_2020_30m >= 5)
    &
    (obs_2025_30m >= 5)
)

quality_gain = (
    apparent_gain
    &
    quality_mask
)

print("\nOBSERVATION-QUALITY FILTER")
print("-" * 120)

print(
    f"Valid quality area: "
    f"{quality_mask.sum() * pixel_area_km2:,.2f} km²"
)

print(
    f"Quality-filtered apparent gain: "
    f"{quality_gain.sum() * pixel_area_km2:,.2f} km²"
)

# ------------------------------------------------------------------------------------------
# POSITIVE-LABEL SPATIAL RELIABILITY
# ------------------------------------------------------------------------------------------
eight_neighbour = np.ones(
    (3, 3),
    dtype=np.uint8
)

gain_components, gain_component_count = ndimage.label(
    quality_gain,
    structure=eight_neighbour
)

gain_component_sizes = np.bincount(
    gain_components.ravel()
)

gain_component_sizes[0] = 0

gain_patch_pixels = gain_component_sizes[
    gain_components
]

quality_stable_built = (
    stable_built
    &
    quality_mask
)

distance_to_stable_built_m = (
    ndimage.distance_transform_edt(
        ~quality_stable_built
    )
    *
    ref_res_x
)

built_2025_binary = (
    lulc_2025_30m == 6
).astype(np.float32)

support_3x3 = ndimage.uniform_filter(
    built_2025_binary,
    size=3,
    mode="constant",
    cval=0
)

accepted_positive = (
    quality_gain
    &
    (gain_patch_pixels >= 5)
    &
    (distance_to_stable_built_m <= 90)
    &
    (support_3x3 >= 0.67)
)

positive_count = int(
    accepted_positive.sum()
)

positive_area_km2 = (
    positive_count *
    pixel_area_km2
)

positive_labels, positive_component_count = ndimage.label(
    accepted_positive,
    structure=eight_neighbour
)

positive_component_sizes = np.bincount(
    positive_labels.ravel()
)

positive_component_sizes[0] = 0

print("\nACCEPTED POSITIVE EXPANSION LABELS")
print("-" * 120)

print("Minimum observations in both years: 5")
print(
    "Minimum original gain-patch size: "
    f"5 pixels ({5 * pixel_area_ha:.2f} ha)"
)
print("Maximum distance to stable built-up: 90 m")
print("Minimum 3×3 built-up support: 67%")

print(
    f"Accepted positive pixels: {positive_count:,}"
)
print(
    f"Accepted positive area: {positive_area_km2:,.2f} km²"
)
print(
    f"Accepted positive components: "
    f"{positive_component_count:,}"
)

quality_gain_count = int(
    quality_gain.sum()
)

positive_retained_percent = (
    positive_count /
    quality_gain_count *
    100
    if quality_gain_count > 0
    else 0
)

print(
    f"Quality gain retained: "
    f"{positive_retained_percent:.2f}%"
)

# ------------------------------------------------------------------------------------------
# POSITIVE COMPONENT TABLE
# ------------------------------------------------------------------------------------------
positive_component_records = []

for component_id in range(
    1,
    positive_component_count + 1
):

    component_pixels = int(
        positive_component_sizes[
            component_id
        ]
    )

    if component_pixels == 0:
        continue

    component_mask = (
        positive_labels == component_id
    )

    rows, cols = np.where(
        component_mask
    )

    mean_distance = float(
        np.mean(
            distance_to_stable_built_m[
                component_mask
            ]
        )
    )

    mean_support = float(
        np.mean(
            support_3x3[
                component_mask
            ]
        )
    )

    positive_component_records.append({
        "Component_ID": component_id,
        "Pixel_Count": component_pixels,
        "Area_ha":
            component_pixels * pixel_area_ha,
        "Area_km2":
            component_pixels * pixel_area_km2,
        "Minimum_Row": int(rows.min()),
        "Maximum_Row": int(rows.max()),
        "Minimum_Column": int(cols.min()),
        "Maximum_Column": int(cols.max()),
        "Mean_Distance_to_Stable_Built_m":
            mean_distance,
        "Mean_3x3_Built_Support":
            mean_support
    })

positive_component_df = pd.DataFrame(
    positive_component_records
)

positive_component_df.to_csv(
    POSITIVE_COMPONENT_FILE,
    index=False
)

# ------------------------------------------------------------------------------------------
# ELIGIBLE NEGATIVE LABELS
#
# Use stable non-built areas near the 2020 urban fringe.
# This prevents the model from learning an overly simple distinction
# between urban-edge positives and extremely remote rural negatives.
# ------------------------------------------------------------------------------------------
built_2020 = (
    lulc_2020_30m == 6
)

distance_to_2020_built_m = (
    ndimage.distance_transform_edt(
        ~built_2020
    )
    *
    ref_res_x
)

distance_to_positive_m = (
    ndimage.distance_transform_edt(
        ~accepted_positive
    )
    *
    ref_res_x
)

water_2020 = (
    lulc_2020_30m == 0
)

water_2025 = (
    lulc_2025_30m == 0
)

eligible_negative = (
    stable_nonbuilt
    &
    quality_mask
    &
    (~water_2020)
    &
    (~water_2025)
    &
    (slope <= 15)
    &
    (distance_to_2020_built_m >= 90)
    &
    (distance_to_2020_built_m <= 1500)
    &
    (distance_to_positive_m >= 90)
)

negative_count = int(
    eligible_negative.sum()
)

negative_area_km2 = (
    negative_count *
    pixel_area_km2
)

negative_to_positive_ratio = (
    negative_count /
    positive_count
    if positive_count > 0
    else np.nan
)

print("\nELIGIBLE STABLE NON-BUILT NEGATIVES")
print("-" * 120)

print("Stable non-built in both years: required")
print("Minimum observations in both years: 5")
print("Water in either year: excluded")
print("Slope greater than 15°: excluded")
print("Distance from 2020 built-up: 90–1,500 m")
print("Minimum distance from accepted positives: 90 m")

print(
    f"Eligible negative pixels: {negative_count:,}"
)
print(
    f"Eligible negative area: {negative_area_km2:,.2f} km²"
)
print(
    f"Eligible negative-to-positive ratio: "
    f"{negative_to_positive_ratio:,.2f}:1"
)

# ------------------------------------------------------------------------------------------
# CREATE LABEL ELIGIBILITY RASTER
# ------------------------------------------------------------------------------------------
label_raster = np.full(
    (ref_height, ref_width),
    255,
    dtype=np.uint8
)

label_raster[
    study_mask
] = 0

label_raster[
    accepted_positive
] = 1

label_raster[
    eligible_negative
] = 2

label_profile = ref_profile.copy()

label_profile.update(
    dtype="uint8",
    count=1,
    nodata=255,
    compress="deflate"
)

with rasterio.open(
    LABEL_RASTER_FILE,
    "w",
    **label_profile
) as dst:

    dst.write(
        label_raster,
        1
    )

    dst.set_band_description(
        1,
        "ML_Label_Eligibility_2020_2025"
    )

    dst.update_tags(
        class_0="Unlabelled_or_excluded",
        class_1="Accepted_urban_expansion_positive",
        class_2="Eligible_stable_nonbuilt_negative",
        nodata_255="Outside_study_area"
    )

# ------------------------------------------------------------------------------------------
# LABEL SUMMARY TABLE
# ------------------------------------------------------------------------------------------
summary_records = [
    {
        "Label_ID": 0,
        "Label_Name": "Unlabelled or excluded",
        "Pixel_Count":
            int(np.count_nonzero(label_raster == 0)),
        "Area_km2":
            np.count_nonzero(label_raster == 0)
            * pixel_area_km2
    },
    {
        "Label_ID": 1,
        "Label_Name":
            "Accepted urban-expansion positive",
        "Pixel_Count": positive_count,
        "Area_km2": positive_area_km2
    },
    {
        "Label_ID": 2,
        "Label_Name":
            "Eligible stable non-built negative",
        "Pixel_Count": negative_count,
        "Area_km2": negative_area_km2
    }
]

label_summary_df = pd.DataFrame(
    summary_records
)

label_summary_df.to_csv(
    LABEL_SUMMARY_FILE,
    index=False
)

print("\nLABEL ELIGIBILITY SUMMARY")
print("-" * 120)

print(
    label_summary_df.to_string(
        index=False,
        formatters={
            "Area_km2":
                lambda value: f"{value:,.2f}"
        }
    )
)

# ------------------------------------------------------------------------------------------
# SAVE LABEL DESIGN REGISTER
# ------------------------------------------------------------------------------------------
label_design = {
    "project":
        "Machine-Learning-Based Land Suitability Analysis "
        "for Sustainable Urban Development — Enugu",

    "target":
        "Observed urban expansion from 2020 to 2025",

    "positive_definition": {
        "transition":
            "Dynamic World non-built 2020 to built 2025",
        "minimum_observations_each_year": 5,
        "minimum_gain_patch_pixels": 5,
        "minimum_gain_patch_area_hectares":
            5 * pixel_area_ha,
        "maximum_distance_to_stable_built_m": 90,
        "minimum_3x3_built_support": 0.67
    },

    "negative_eligibility_definition": {
        "transition":
            "Stable non-built 2020 to 2025",
        "minimum_observations_each_year": 5,
        "water_excluded": True,
        "maximum_slope_degrees": 15,
        "minimum_distance_to_2020_built_m": 90,
        "maximum_distance_to_2020_built_m": 1500,
        "minimum_distance_to_positive_m": 90
    },

    "predictor_leakage_rules": {
        "exclude_2025_NDVI": True,
        "exclude_2025_NDBI": True,
        "exclude_2025_MNDWI": True,
        "exclude_distance_to_built_2025": True,
        "baseline_or_static_predictors_only": True
    },

    "positive_pixel_count": positive_count,
    "positive_area_km2": positive_area_km2,
    "eligible_negative_pixel_count": negative_count,
    "eligible_negative_area_km2": negative_area_km2,

    "sampling_status":
        "Pending spatially balanced sampling",

    "label_definition_status":
        "Accepted"
}

with open(
    DESIGN_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        label_design,
        file,
        indent=4
    )

# ------------------------------------------------------------------------------------------
# REOPEN SAVED LABEL RASTER
# ------------------------------------------------------------------------------------------
with rasterio.open(
    LABEL_RASTER_FILE
) as src:

    saved_labels = src.read(1)

    saved_epsg = (
        src.crs.to_epsg()
        if src.crs is not None
        else None
    )

    saved_width = src.width
    saved_height = src.height

    saved_res_x = abs(src.transform.a)
    saved_res_y = abs(src.transform.e)

    saved_bounds = src.bounds

saved_valid = (
    saved_labels != 255
)

saved_classes = sorted(
    np.unique(
        saved_labels[saved_valid]
    ).tolist()
)

same_dimensions = (
    saved_width == ref_width
    and
    saved_height == ref_height
)

same_resolution = (
    np.isclose(saved_res_x, ref_res_x)
    and
    np.isclose(saved_res_y, ref_res_y)
)

same_bounds = np.allclose(
    [
        saved_bounds.left,
        saved_bounds.bottom,
        saved_bounds.right,
        saved_bounds.top
    ],
    [
        ref_bounds.left,
        ref_bounds.bottom,
        ref_bounds.right,
        ref_bounds.top
    ],
    atol=0.001
)

# ------------------------------------------------------------------------------------------
# VALIDATION REGISTER
# ------------------------------------------------------------------------------------------
validation_df = pd.DataFrame([{
    "Positive_Minimum_Observations":
        5,

    "Positive_Minimum_Patch_Pixels":
        5,

    "Positive_Maximum_Distance_m":
        90,

    "Positive_Minimum_3x3_Support":
        0.67,

    "Accepted_Positive_Pixels":
        positive_count,

    "Accepted_Positive_Area_km2":
        positive_area_km2,

    "Accepted_Positive_Components":
        positive_component_count,

    "Quality_Gain_Retained_Percent":
        positive_retained_percent,

    "Eligible_Negative_Pixels":
        negative_count,

    "Eligible_Negative_Area_km2":
        negative_area_km2,

    "Negative_to_Positive_Ratio":
        negative_to_positive_ratio,

    "Saved_Label_Classes":
        str(saved_classes),

    "Aligned_With_Reference":
        (
            same_dimensions
            and
            same_resolution
            and
            same_bounds
        ),

    "Label_Definition_Accepted":
        True,

    "Balanced_Sampling_Completed":
        False
}])

validation_df.to_csv(
    VALIDATION_FILE,
    index=False
)

# ------------------------------------------------------------------------------------------
# FINAL VALIDATION
# ------------------------------------------------------------------------------------------
checks = {
    "Transition raster uses EPSG:32632":
        transition_epsg == 32632,

    "Transition dimensions match reference":
        (
            transition_width == ref_width
            and
            transition_height == ref_height
        ),

    "Accepted positive labels present":
        positive_count > 0,

    "At least 10,000 positive pixels available":
        positive_count >= 10_000,

    "Multiple positive components available":
        positive_component_count >= 100,

    "Eligible negative labels present":
        negative_count > 0,

    "Negative pool exceeds positive pool":
        negative_count > positive_count,

    "Negative-to-positive ratio at least 2:1":
        negative_to_positive_ratio >= 2,

    "Positive component table saved":
        POSITIVE_COMPONENT_FILE.exists(),

    "Label summary table saved":
        LABEL_SUMMARY_FILE.exists(),

    "Label raster saved":
        LABEL_RASTER_FILE.exists(),

    "Label design register saved":
        DESIGN_FILE.exists(),

    "Validation register saved":
        VALIDATION_FILE.exists(),

    "Saved label classes are 0, 1 and 2":
        saved_classes == [0, 1, 2],

    "Saved label raster uses EPSG:32632":
        saved_epsg == 32632,

    "Saved dimensions match reference":
        same_dimensions,

    "Saved resolution matches reference":
        same_resolution,

    "Saved bounds match reference":
        same_bounds,

    "Label definition accepted":
        validation_df[
            "Label_Definition_Accepted"
        ].iloc[0] == True,

    "Balanced sampling remains pending":
        validation_df[
            "Balanced_Sampling_Completed"
        ].iloc[0] == False
}

print("\nVALIDATION SUMMARY")
print("-" * 120)

for label, passed in checks.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{label}"
    )

print("\nOUTPUT FILES")
print("-" * 120)

for path in [
    LABEL_RASTER_FILE,
    POSITIVE_COMPONENT_FILE,
    LABEL_SUMMARY_FILE,
    DESIGN_FILE,
    VALIDATION_FILE
]:

    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{path.name}"
    )

if all(checks.values()):

    print("\n✓ STAGE 9D FULLY PASSED")

    print(
        "The final 2020–2025 urban-expansion label "
        "eligibility definition has been accepted."
    )

    print(
        "The next stage will perform spatially balanced "
        "positive and negative sampling."
    )

else:

    print("\n⚠ STAGE 9D REQUIRES REVIEW")

print("=" * 120)

In [ ]:
# ==========================================================================================
# PROJECT 7 — STAGE 9E
# SPATIALLY BALANCED ML SAMPLING + BLOCK-BASED TRAIN/VALIDATION/TEST SPLIT
#
# Sampling design:
# - Target positive samples: 12,000
# - Target negative samples: 12,000
# - Class ratio: 1:1
# - Spatial block size: 5 km
# - Dataset split: 70% training, 15% validation, 15% testing
# - Entire spatial blocks are assigned to only one split.
#
# Positive sampling:
# - Component-aware sampling;
# - all positive components receive an opportunity to contribute;
# - large patches cannot dominate the sample.
#
# Negative sampling:
# - preferentially selected from spatial blocks containing sampled positives;
# - remaining requirement filled from the wider eligible-negative pool.
#
# This stage creates sample points and split assignments only.
# Predictor values will be extracted in the next stage.
# ==========================================================================================

from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

from scipy import ndimage
from shapely.geometry import Point

# ------------------------------------------------------------------------------------------
# PARAMETERS
# ------------------------------------------------------------------------------------------
RANDOM_SEED = 42

TARGET_POSITIVE_SAMPLES = 12_000
TARGET_NEGATIVE_SAMPLES = 12_000

BLOCK_SIZE_M = 5_000

TRAIN_FRACTION = 0.70
VALIDATION_FRACTION = 0.15
TEST_FRACTION = 0.15

rng = np.random.default_rng(
    RANDOM_SEED
)

# ------------------------------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------------------------------
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

TRAINING_DIR = (
    PROJECT_ROOT /
    "09_Training_Data"
)

TABLE_DIR = (
    PROJECT_ROOT /
    "16_Tables"
)

ADMIN_DIR = (
    PROJECT_ROOT /
    "00_Project_Admin"
)

REFERENCE_FILE = (
    PROJECT_ROOT /
    "02_Terrain" /
    "Enugu_Elevation_30m.tif"
)

LABEL_RASTER_FILE = (
    TRAINING_DIR /
    "Enugu_ML_Label_Eligibility_2020_2025_30m.tif"
)

SAMPLE_GPKG_FILE = (
    TRAINING_DIR /
    "Enugu_ML_Spatial_Samples_2020_2025.gpkg"
)

SAMPLE_CSV_FILE = (
    TRAINING_DIR /
    "Enugu_ML_Spatial_Samples_2020_2025.csv"
)

SAMPLE_SUMMARY_FILE = (
    TABLE_DIR /
    "Enugu_ML_Spatial_Sampling_Summary.csv"
)

BLOCK_SUMMARY_FILE = (
    TABLE_DIR /
    "Enugu_ML_Spatial_Block_Split_Summary.csv"
)

COMPONENT_SUMMARY_FILE = (
    TABLE_DIR /
    "Enugu_Positive_Component_Sampling_Summary.csv"
)

VALIDATION_FILE = (
    TRAINING_DIR /
    "Stage_9E_Spatial_Sampling_Validation.csv"
)

DESIGN_FILE = (
    ADMIN_DIR /
    "ML_Spatial_Sampling_Design.json"
)

for folder in [
    TRAINING_DIR,
    TABLE_DIR,
    ADMIN_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

print("=" * 120)
print("STAGE 9E — SPATIALLY BALANCED SAMPLING + BLOCK-BASED DATASET SPLIT")
print("=" * 120)

# ------------------------------------------------------------------------------------------
# REQUIRED FILE CHECK
# ------------------------------------------------------------------------------------------
required_files = {
    "ML label eligibility raster":
        LABEL_RASTER_FILE,

    "30 m reference raster":
        REFERENCE_FILE
}

print("\nREQUIRED FILE CHECK")
print("-" * 120)

for label, path in required_files.items():

    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{label}: {path}"
    )

if not all(
    path.exists()
    for path in required_files.values()
):

    raise FileNotFoundError(
        "One or more required Stage 9E input files are missing."
    )

# ------------------------------------------------------------------------------------------
# READ REFERENCE GRID AND LABEL RASTER
# ------------------------------------------------------------------------------------------
with rasterio.open(
    REFERENCE_FILE
) as ref:

    elevation = ref.read(1)

    ref_transform = ref.transform
    ref_crs = ref.crs
    ref_width = ref.width
    ref_height = ref.height
    ref_bounds = ref.bounds

    ref_res_x = abs(
        ref.transform.a
    )

    ref_res_y = abs(
        ref.transform.e
    )

study_mask = np.isfinite(
    elevation
)

with rasterio.open(
    LABEL_RASTER_FILE
) as src:

    label_raster = src.read(1)

    label_crs = src.crs
    label_transform = src.transform
    label_width = src.width
    label_height = src.height
    label_bounds = src.bounds

    label_epsg = (
        src.crs.to_epsg()
        if src.crs is not None
        else None
    )

same_dimensions = (
    label_width == ref_width
    and
    label_height == ref_height
)

same_transform = np.allclose(
    tuple(label_transform),
    tuple(ref_transform)
)

same_bounds = np.allclose(
    [
        label_bounds.left,
        label_bounds.bottom,
        label_bounds.right,
        label_bounds.top
    ],
    [
        ref_bounds.left,
        ref_bounds.bottom,
        ref_bounds.right,
        ref_bounds.top
    ],
    atol=0.001
)

if not (
    same_dimensions
    and
    same_transform
    and
    same_bounds
    and
    label_crs == ref_crs
):

    raise RuntimeError(
        "The label raster is not aligned with the reference grid."
    )

positive_mask = (
    label_raster == 1
)

negative_mask = (
    label_raster == 2
)

positive_pool_count = int(
    positive_mask.sum()
)

negative_pool_count = int(
    negative_mask.sum()
)

print("\nLABEL POOLS")
print("-" * 120)

print(
    f"Available positive pixels: "
    f"{positive_pool_count:,}"
)

print(
    f"Available negative pixels: "
    f"{negative_pool_count:,}"
)

if positive_pool_count < TARGET_POSITIVE_SAMPLES:

    raise RuntimeError(
        "The positive pool is smaller than the requested sample size."
    )

if negative_pool_count < TARGET_NEGATIVE_SAMPLES:

    raise RuntimeError(
        "The negative pool is smaller than the requested sample size."
    )

# ------------------------------------------------------------------------------------------
# CREATE 5 km SPATIAL BLOCK IDS
# ------------------------------------------------------------------------------------------
rows_grid, cols_grid = np.indices(
    (
        ref_height,
        ref_width
    ),
    dtype=np.int32
)

x_centres = (
    ref_transform.c
    +
    (
        cols_grid.astype(np.float64)
        +
        0.5
    )
    *
    ref_transform.a
)

y_centres = (
    ref_transform.f
    +
    (
        rows_grid.astype(np.float64)
        +
        0.5
    )
    *
    ref_transform.e
)

block_col = np.floor(
    (
        x_centres -
        ref_bounds.left
    )
    /
    BLOCK_SIZE_M
).astype(np.int32)

block_row = np.floor(
    (
        ref_bounds.top -
        y_centres
    )
    /
    BLOCK_SIZE_M
).astype(np.int32)

maximum_block_col = int(
    block_col.max()
) + 1

block_id_raster = (
    block_row.astype(np.int64)
    *
    maximum_block_col
    +
    block_col.astype(np.int64)
)

print("\nSPATIAL BLOCK GRID")
print("-" * 120)

print(
    f"Block size: "
    f"{BLOCK_SIZE_M / 1000:.1f} km × "
    f"{BLOCK_SIZE_M / 1000:.1f} km"
)

print(
    f"Study-area blocks: "
    f"{np.unique(block_id_raster[study_mask]).size:,}"
)

# ------------------------------------------------------------------------------------------
# LABEL POSITIVE CONNECTED COMPONENTS
# ------------------------------------------------------------------------------------------
eight_neighbour = np.ones(
    (3, 3),
    dtype=np.uint8
)

positive_components, component_count = ndimage.label(
    positive_mask,
    structure=eight_neighbour
)

component_sizes = np.bincount(
    positive_components.ravel()
)

component_sizes[0] = 0

print("\nPOSITIVE COMPONENT STRUCTURE")
print("-" * 120)

print(
    f"Positive connected components: "
    f"{component_count:,}"
)

print(
    f"Median component size: "
    f"{np.median(component_sizes[1:]):.1f} pixels"
)

print(
    f"Maximum component size: "
    f"{component_sizes[1:].max():,} pixels"
)

# ------------------------------------------------------------------------------------------
# COMPONENT-AWARE POSITIVE SAMPLING
# ------------------------------------------------------------------------------------------
positive_rows, positive_cols = np.where(
    positive_mask
)

positive_component_ids = positive_components[
    positive_rows,
    positive_cols
]

component_to_indices = {}

for array_index, component_id in enumerate(
    positive_component_ids
):

    component_to_indices.setdefault(
        int(component_id),
        []
    ).append(
        array_index
    )

for component_id in component_to_indices:

    component_to_indices[
        component_id
    ] = np.asarray(
        component_to_indices[
            component_id
        ],
        dtype=np.int64
    )

# Start with one sample from every component where possible.
selected_positive_indices = []

component_sample_counts = {
    component_id: 0
    for component_id in component_to_indices
}

component_ids_random = np.array(
    list(
        component_to_indices.keys()
    ),
    dtype=np.int64
)

rng.shuffle(
    component_ids_random
)

for component_id in component_ids_random:

    available_indices = component_to_indices[
        int(component_id)
    ]

    chosen_index = rng.choice(
        available_indices
    )

    selected_positive_indices.append(
        int(chosen_index)
    )

    component_sample_counts[
        int(component_id)
    ] += 1

    if (
        len(selected_positive_indices)
        >= TARGET_POSITIVE_SAMPLES
    ):
        break

# Fill the remaining requirement while restricting domination by large patches.
current_component_cap = 2

selected_positive_set = set(
    selected_positive_indices
)

while (
    len(selected_positive_indices)
    <
    TARGET_POSITIVE_SAMPLES
):

    candidates_added = 0

    shuffled_components = np.array(
        list(
            component_to_indices.keys()
        ),
        dtype=np.int64
    )

    rng.shuffle(
        shuffled_components
    )

    for component_id in shuffled_components:

        component_id = int(
            component_id
        )

        already_selected = (
            component_sample_counts[
                component_id
            ]
        )

        if already_selected >= current_component_cap:
            continue

        available_indices = component_to_indices[
            component_id
        ]

        unselected = np.array(
            [
                index
                for index in available_indices
                if int(index)
                not in selected_positive_set
            ],
            dtype=np.int64
        )

        if unselected.size == 0:
            continue

        remaining_component_capacity = (
            current_component_cap -
            already_selected
        )

        remaining_global_requirement = (
            TARGET_POSITIVE_SAMPLES -
            len(selected_positive_indices)
        )

        number_to_select = min(
            remaining_component_capacity,
            unselected.size,
            remaining_global_requirement
        )

        chosen = rng.choice(
            unselected,
            size=number_to_select,
            replace=False
        )

        for index in np.atleast_1d(
            chosen
        ):

            selected_positive_indices.append(
                int(index)
            )

            selected_positive_set.add(
                int(index)
            )

            component_sample_counts[
                component_id
            ] += 1

            candidates_added += 1

        if (
            len(selected_positive_indices)
            >= TARGET_POSITIVE_SAMPLES
        ):
            break

    if candidates_added == 0:

        current_component_cap += 1

    elif (
        len(selected_positive_indices)
        <
        TARGET_POSITIVE_SAMPLES
    ):

        current_component_cap += 1

    if current_component_cap > 100:

        raise RuntimeError(
            "Positive component-aware sampling could not reach "
            "the requested sample size."
        )

selected_positive_indices = np.asarray(
    selected_positive_indices[
        :TARGET_POSITIVE_SAMPLES
    ],
    dtype=np.int64
)

sampled_positive_rows = positive_rows[
    selected_positive_indices
]

sampled_positive_cols = positive_cols[
    selected_positive_indices
]

sampled_positive_components = positive_component_ids[
    selected_positive_indices
]

sampled_positive_blocks = block_id_raster[
    sampled_positive_rows,
    sampled_positive_cols
]

print("\nPOSITIVE SAMPLING")
print("-" * 120)

print(
    f"Requested positive samples: "
    f"{TARGET_POSITIVE_SAMPLES:,}"
)

print(
    f"Selected positive samples: "
    f"{sampled_positive_rows.size:,}"
)

print(
    f"Positive components represented: "
    f"{np.unique(sampled_positive_components).size:,}"
)

print(
    f"Positive spatial blocks represented: "
    f"{np.unique(sampled_positive_blocks).size:,}"
)

print(
    f"Final component sample cap used: "
    f"{current_component_cap}"
)

# ------------------------------------------------------------------------------------------
# NEGATIVE SAMPLING
#
# First sample from blocks containing positives, approximately matching
# the positive sample count in each block. Then fill any remaining requirement
# from the broader negative pool.
# ------------------------------------------------------------------------------------------
negative_rows, negative_cols = np.where(
    negative_mask
)

negative_blocks = block_id_raster[
    negative_rows,
    negative_cols
]

negative_block_to_indices = {}

for index, block_id in enumerate(
    negative_blocks
):

    negative_block_to_indices.setdefault(
        int(block_id),
        []
    ).append(
        index
    )

for block_id in negative_block_to_indices:

    negative_block_to_indices[
        block_id
    ] = np.asarray(
        negative_block_to_indices[
            block_id
        ],
        dtype=np.int64
    )

positive_block_counts = pd.Series(
    sampled_positive_blocks
).value_counts()

selected_negative_indices = []
selected_negative_set = set()

# Match positive counts inside corresponding blocks where possible.
for block_id, required_count in positive_block_counts.items():

    block_id = int(
        block_id
    )

    required_count = int(
        required_count
    )

    available = negative_block_to_indices.get(
        block_id,
        np.array(
            [],
            dtype=np.int64
        )
    )

    if available.size == 0:
        continue

    number_to_select = min(
        required_count,
        available.size
    )

    chosen = rng.choice(
        available,
        size=number_to_select,
        replace=False
    )

    for index in np.atleast_1d(
        chosen
    ):

        selected_negative_indices.append(
            int(index)
        )

        selected_negative_set.add(
            int(index)
        )

# Fill remaining negatives globally.
remaining_negative_requirement = (
    TARGET_NEGATIVE_SAMPLES -
    len(selected_negative_indices)
)

if remaining_negative_requirement > 0:

    all_negative_indices = np.arange(
        negative_rows.size,
        dtype=np.int64
    )

    remaining_candidates = np.array(
        [
            index
            for index in all_negative_indices
            if int(index)
            not in selected_negative_set
        ],
        dtype=np.int64
    )

    if (
        remaining_candidates.size
        <
        remaining_negative_requirement
    ):

        raise RuntimeError(
            "Insufficient remaining negatives to complete sampling."
        )

    additional_negative_indices = rng.choice(
        remaining_candidates,
        size=remaining_negative_requirement,
        replace=False
    )

    selected_negative_indices.extend(
        additional_negative_indices.tolist()
    )

selected_negative_indices = np.asarray(
    selected_negative_indices[
        :TARGET_NEGATIVE_SAMPLES
    ],
    dtype=np.int64
)

sampled_negative_rows = negative_rows[
    selected_negative_indices
]

sampled_negative_cols = negative_cols[
    selected_negative_indices
]

sampled_negative_blocks = block_id_raster[
    sampled_negative_rows,
    sampled_negative_cols
]

print("\nNEGATIVE SAMPLING")
print("-" * 120)

print(
    f"Requested negative samples: "
    f"{TARGET_NEGATIVE_SAMPLES:,}"
)

print(
    f"Selected negative samples: "
    f"{sampled_negative_rows.size:,}"
)

print(
    f"Negative spatial blocks represented: "
    f"{np.unique(sampled_negative_blocks).size:,}"
)

positive_blocks_set = set(
    sampled_positive_blocks.tolist()
)

negative_in_positive_blocks = int(
    np.count_nonzero(
        np.isin(
            sampled_negative_blocks,
            list(
                positive_blocks_set
            )
        )
    )
)

print(
    f"Negatives selected from positive-containing blocks: "
    f"{negative_in_positive_blocks:,} "
    f"({negative_in_positive_blocks / TARGET_NEGATIVE_SAMPLES * 100:.2f}%)"
)

# ------------------------------------------------------------------------------------------
# COMBINE SAMPLE ARRAYS
# ------------------------------------------------------------------------------------------
sample_rows = np.concatenate(
    [
        sampled_positive_rows,
        sampled_negative_rows
    ]
)

sample_cols = np.concatenate(
    [
        sampled_positive_cols,
        sampled_negative_cols
    ]
)

sample_classes = np.concatenate(
    [
        np.ones(
            TARGET_POSITIVE_SAMPLES,
            dtype=np.uint8
        ),
        np.zeros(
            TARGET_NEGATIVE_SAMPLES,
            dtype=np.uint8
        )
    ]
)

sample_component_ids = np.concatenate(
    [
        sampled_positive_components.astype(
            np.int32
        ),
        np.full(
            TARGET_NEGATIVE_SAMPLES,
            -1,
            dtype=np.int32
        )
    ]
)

sample_block_ids = block_id_raster[
    sample_rows,
    sample_cols
].astype(np.int64)

sample_block_rows = block_row[
    sample_rows,
    sample_cols
].astype(np.int32)

sample_block_cols = block_col[
    sample_rows,
    sample_cols
].astype(np.int32)

sample_x = (
    ref_transform.c
    +
    (
        sample_cols.astype(np.float64)
        +
        0.5
    )
    *
    ref_transform.a
)

sample_y = (
    ref_transform.f
    +
    (
        sample_rows.astype(np.float64)
        +
        0.5
    )
    *
    ref_transform.e
)

# ------------------------------------------------------------------------------------------
# BLOCK-LEVEL TRAIN / VALIDATION / TEST ASSIGNMENT
#
# Greedy assignment is based on total class counts per block.
# Entire blocks are kept together.
# ------------------------------------------------------------------------------------------
block_dataframe = pd.DataFrame({
    "Block_ID": sample_block_ids,
    "Class": sample_classes
})

block_counts = (
    block_dataframe
    .groupby(
        [
            "Block_ID",
            "Class"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reset_index()
)

if 0 not in block_counts.columns:
    block_counts[0] = 0

if 1 not in block_counts.columns:
    block_counts[1] = 0

block_counts = block_counts.rename(
    columns={
        0: "Negative_Count",
        1: "Positive_Count"
    }
)

block_counts[
    "Total_Count"
] = (
    block_counts[
        "Positive_Count"
    ]
    +
    block_counts[
        "Negative_Count"
    ]
)

# Prioritise the largest and most class-diverse blocks.
block_counts[
    "Class_Balance_Score"
] = np.minimum(
    block_counts[
        "Positive_Count"
    ],
    block_counts[
        "Negative_Count"
    ]
)

block_counts = block_counts.sort_values(
    by=[
        "Total_Count",
        "Class_Balance_Score"
    ],
    ascending=False
).reset_index(
    drop=True
)

split_names = [
    "Train",
    "Validation",
    "Test"
]

split_fractions = {
    "Train": TRAIN_FRACTION,
    "Validation": VALIDATION_FRACTION,
    "Test": TEST_FRACTION
}

target_counts = {}

for split_name, fraction in split_fractions.items():

    target_counts[
        split_name
    ] = {
        "Positive_Count":
            TARGET_POSITIVE_SAMPLES * fraction,

        "Negative_Count":
            TARGET_NEGATIVE_SAMPLES * fraction,

        "Total_Count":
            (
                TARGET_POSITIVE_SAMPLES
                +
                TARGET_NEGATIVE_SAMPLES
            )
            *
            fraction
    }

current_counts = {
    split_name: {
        "Positive_Count": 0,
        "Negative_Count": 0,
        "Total_Count": 0
    }
    for split_name in split_names
}

block_assignment = {}

for _, block_record in block_counts.iterrows():

    block_id = int(
        block_record[
            "Block_ID"
        ]
    )

    block_positive = int(
        block_record[
            "Positive_Count"
        ]
    )

    block_negative = int(
        block_record[
            "Negative_Count"
        ]
    )

    block_total = int(
        block_record[
            "Total_Count"
        ]
    )

    candidate_scores = {}

    for split_name in split_names:

        hypothetical_positive = (
            current_counts[
                split_name
            ][
                "Positive_Count"
            ]
            +
            block_positive
        )

        hypothetical_negative = (
            current_counts[
                split_name
            ][
                "Negative_Count"
            ]
            +
            block_negative
        )

        hypothetical_total = (
            current_counts[
                split_name
            ][
                "Total_Count"
            ]
            +
            block_total
        )

        positive_error = abs(
            hypothetical_positive
            -
            target_counts[
                split_name
            ][
                "Positive_Count"
            ]
        ) / max(
            target_counts[
                split_name
            ][
                "Positive_Count"
            ],
            1
        )

        negative_error = abs(
            hypothetical_negative
            -
            target_counts[
                split_name
            ][
                "Negative_Count"
            ]
        ) / max(
            target_counts[
                split_name
            ][
                "Negative_Count"
            ],
            1
        )

        total_error = abs(
            hypothetical_total
            -
            target_counts[
                split_name
            ][
                "Total_Count"
            ]
        ) / max(
            target_counts[
                split_name
            ][
                "Total_Count"
            ],
            1
        )

        current_fill_ratio = (
            current_counts[
                split_name
            ][
                "Total_Count"
            ]
            /
            max(
                target_counts[
                    split_name
                ][
                    "Total_Count"
                ],
                1
            )
        )

        candidate_scores[
            split_name
        ] = (
            positive_error
            +
            negative_error
            +
            total_error
            +
            0.20 * current_fill_ratio
        )

    selected_split = min(
        candidate_scores,
        key=candidate_scores.get
    )

    block_assignment[
        block_id
    ] = selected_split

    current_counts[
        selected_split
    ][
        "Positive_Count"
    ] += block_positive

    current_counts[
        selected_split
    ][
        "Negative_Count"
    ] += block_negative

    current_counts[
        selected_split
    ][
        "Total_Count"
    ] += block_total

sample_splits = np.array(
    [
        block_assignment[
            int(block_id)
        ]
        for block_id in sample_block_ids
    ],
    dtype=object
)

# ------------------------------------------------------------------------------------------
# BUILD SAMPLE DATAFRAME
# ------------------------------------------------------------------------------------------
sample_ids = np.arange(
    1,
    sample_rows.size + 1,
    dtype=np.int32
)

sample_df = pd.DataFrame({
    "Sample_ID":
        sample_ids,

    "Target":
        sample_classes,

    "Target_Name":
        np.where(
            sample_classes == 1,
            "Urban expansion",
            "Stable non-built"
        ),

    "Dataset_Split":
        sample_splits,

    "Raster_Row":
        sample_rows.astype(
            np.int32
        ),

    "Raster_Column":
        sample_cols.astype(
            np.int32
        ),

    "X":
        sample_x,

    "Y":
        sample_y,

    "Block_ID":
        sample_block_ids,

    "Block_Row":
        sample_block_rows,

    "Block_Column":
        sample_block_cols,

    "Positive_Component_ID":
        sample_component_ids
})

# Shuffle record order without changing assignments.
shuffle_order = rng.permutation(
    sample_df.shape[0]
)

sample_df = sample_df.iloc[
    shuffle_order
].reset_index(
    drop=True
)

sample_df[
    "Sample_ID"
] = np.arange(
    1,
    sample_df.shape[0] + 1,
    dtype=np.int32
)

# ------------------------------------------------------------------------------------------
# CREATE GEODATAFRAME
# ------------------------------------------------------------------------------------------
sample_geometry = [
    Point(
        x,
        y
    )
    for x, y in zip(
        sample_df["X"],
        sample_df["Y"]
    )
]

sample_gdf = gpd.GeoDataFrame(
    sample_df.copy(),
    geometry=sample_geometry,
    crs=ref_crs
)

# ------------------------------------------------------------------------------------------
# SAVE SAMPLE DATA
# ------------------------------------------------------------------------------------------
if SAMPLE_GPKG_FILE.exists():

    SAMPLE_GPKG_FILE.unlink()

sample_gdf.to_file(
    SAMPLE_GPKG_FILE,
    layer="ml_samples",
    driver="GPKG"
)

sample_df.to_csv(
    SAMPLE_CSV_FILE,
    index=False
)

# ------------------------------------------------------------------------------------------
# COMPONENT SAMPLING SUMMARY
# ------------------------------------------------------------------------------------------
component_summary_df = (
    pd.DataFrame({
        "Component_ID":
            sampled_positive_components
    })
    .groupby(
        "Component_ID"
    )
    .size()
    .reset_index(
        name="Selected_Sample_Count"
    )
)

component_summary_df[
    "Original_Component_Pixel_Count"
] = component_summary_df[
    "Component_ID"
].map(
    lambda component_id:
        int(
            component_sizes[
                int(component_id)
            ]
        )
)

component_summary_df[
    "Original_Component_Area_ha"
] = (
    component_summary_df[
        "Original_Component_Pixel_Count"
    ]
    *
    ref_res_x
    *
    ref_res_y
    /
    10_000
)

component_summary_df.to_csv(
    COMPONENT_SUMMARY_FILE,
    index=False
)

# ------------------------------------------------------------------------------------------
# DATASET-SPLIT SUMMARY
# ------------------------------------------------------------------------------------------
sample_summary_df = (
    sample_df
    .groupby(
        [
            "Dataset_Split",
            "Target",
            "Target_Name"
        ]
    )
    .size()
    .reset_index(
        name="Sample_Count"
    )
)

sample_summary_df[
    "Percent_of_Class"
] = (
    sample_summary_df[
        "Sample_Count"
    ]
    /
    sample_summary_df.groupby(
        "Target"
    )[
        "Sample_Count"
    ].transform(
        "sum"
    )
    *
    100
)

sample_summary_df.to_csv(
    SAMPLE_SUMMARY_FILE,
    index=False
)

block_split_summary_df = (
    sample_df
    .groupby(
        [
            "Dataset_Split",
            "Block_ID"
        ]
    )
    .agg(
        Total_Samples=(
            "Sample_ID",
            "count"
        ),
        Positive_Samples=(
            "Target",
            "sum"
        )
    )
    .reset_index()
)

block_split_summary_df[
    "Negative_Samples"
] = (
    block_split_summary_df[
        "Total_Samples"
    ]
    -
    block_split_summary_df[
        "Positive_Samples"
    ]
)

block_split_summary_df.to_csv(
    BLOCK_SUMMARY_FILE,
    index=False
)

print("\nDATASET SPLIT SUMMARY")
print("-" * 120)

display_summary = (
    sample_df
    .groupby(
        [
            "Dataset_Split",
            "Target_Name"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reindex(
        [
            "Train",
            "Validation",
            "Test"
        ]
    )
)

display_summary[
    "Total"
] = display_summary.sum(
    axis=1
)

print(
    display_summary.to_string()
)

split_block_counts = (
    sample_df
    .groupby(
        "Dataset_Split"
    )[
        "Block_ID"
    ]
    .nunique()
    .reindex(
        [
            "Train",
            "Validation",
            "Test"
        ]
    )
)

print("\nSPATIAL BLOCK COUNTS")
print("-" * 120)

for split_name, block_count_value in split_block_counts.items():

    print(
        f"{split_name}: "
        f"{int(block_count_value):,} blocks"
    )

# ------------------------------------------------------------------------------------------
# CHECK BLOCK OVERLAP
# ------------------------------------------------------------------------------------------
train_blocks = set(
    sample_df.loc[
        sample_df[
            "Dataset_Split"
        ] == "Train",
        "Block_ID"
    ]
)

validation_blocks = set(
    sample_df.loc[
        sample_df[
            "Dataset_Split"
        ] == "Validation",
        "Block_ID"
    ]
)

test_blocks = set(
    sample_df.loc[
        sample_df[
            "Dataset_Split"
        ] == "Test",
        "Block_ID"
    ]
)

train_validation_overlap = (
    train_blocks
    &
    validation_blocks
)

train_test_overlap = (
    train_blocks
    &
    test_blocks
)

validation_test_overlap = (
    validation_blocks
    &
    test_blocks
)

# ------------------------------------------------------------------------------------------
# REOPEN SAVED GEOPACKAGE
# ------------------------------------------------------------------------------------------
saved_gdf = gpd.read_file(
    SAMPLE_GPKG_FILE,
    layer="ml_samples"
)

saved_sample_count = (
    saved_gdf.shape[0]
)

saved_epsg = (
    saved_gdf.crs.to_epsg()
    if saved_gdf.crs is not None
    else None
)

saved_target_values = sorted(
    saved_gdf[
        "Target"
    ].unique().tolist()
)

saved_split_values = sorted(
    saved_gdf[
        "Dataset_Split"
    ].unique().tolist()
)

saved_positive_count = int(
    (
        saved_gdf[
            "Target"
        ] == 1
    ).sum()
)

saved_negative_count = int(
    (
        saved_gdf[
            "Target"
        ] == 0
    ).sum()
)

# ------------------------------------------------------------------------------------------
# SPLIT PROPORTIONS
# ------------------------------------------------------------------------------------------
split_totals = (
    sample_df[
        "Dataset_Split"
    ]
    .value_counts()
)

total_samples = int(
    sample_df.shape[0]
)

actual_train_fraction = (
    split_totals.get(
        "Train",
        0
    )
    /
    total_samples
)

actual_validation_fraction = (
    split_totals.get(
        "Validation",
        0
    )
    /
    total_samples
)

actual_test_fraction = (
    split_totals.get(
        "Test",
        0
    )
    /
    total_samples
)

# Block-level allocation can prevent exact percentages.
split_fraction_tolerance = 0.08

# ------------------------------------------------------------------------------------------
# SAVE SAMPLING DESIGN
# ------------------------------------------------------------------------------------------
sampling_design = {
    "random_seed":
        RANDOM_SEED,

    "positive_sample_target":
        TARGET_POSITIVE_SAMPLES,

    "negative_sample_target":
        TARGET_NEGATIVE_SAMPLES,

    "class_ratio":
        "1:1",

    "spatial_block_size_metres":
        BLOCK_SIZE_M,

    "requested_split": {
        "training":
            TRAIN_FRACTION,

        "validation":
            VALIDATION_FRACTION,

        "testing":
            TEST_FRACTION
    },

    "actual_split": {
        "training":
            actual_train_fraction,

        "validation":
            actual_validation_fraction,

        "testing":
            actual_test_fraction
    },

    "positive_sampling":
        "Connected-component-aware sampling with capped "
        "representation from large patches",

    "negative_sampling":
        "Preferential matching to positive-containing blocks, "
        "followed by global eligible-pool sampling",

    "split_method":
        "Entire 5 km spatial blocks assigned exclusively to "
        "training, validation or testing",

    "predictor_extraction_status":
        "Pending",

    "model_training_status":
        "Pending"
}

with open(
    DESIGN_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        sampling_design,
        file,
        indent=4
    )

# ------------------------------------------------------------------------------------------
# VALIDATION REGISTER
# ------------------------------------------------------------------------------------------
validation_df = pd.DataFrame([{
    "Random_Seed":
        RANDOM_SEED,

    "Block_Size_m":
        BLOCK_SIZE_M,

    "Positive_Pool_Pixels":
        positive_pool_count,

    "Negative_Pool_Pixels":
        negative_pool_count,

    "Selected_Positive_Samples":
        saved_positive_count,

    "Selected_Negative_Samples":
        saved_negative_count,

    "Total_Samples":
        saved_sample_count,

    "Positive_Components_Represented":
        int(
            np.unique(
                sampled_positive_components
            ).size
        ),

    "Positive_Blocks_Represented":
        int(
            np.unique(
                sampled_positive_blocks
            ).size
        ),

    "Negative_Blocks_Represented":
        int(
            np.unique(
                sampled_negative_blocks
            ).size
        ),

    "Train_Fraction":
        actual_train_fraction,

    "Validation_Fraction":
        actual_validation_fraction,

    "Test_Fraction":
        actual_test_fraction,

    "Train_Validation_Block_Overlap":
        len(
            train_validation_overlap
        ),

    "Train_Test_Block_Overlap":
        len(
            train_test_overlap
        ),

    "Validation_Test_Block_Overlap":
        len(
            validation_test_overlap
        ),

    "Predictor_Extraction_Completed":
        False,

    "Model_Training_Completed":
        False
}])

validation_df.to_csv(
    VALIDATION_FILE,
    index=False
)

# ------------------------------------------------------------------------------------------
# FINAL VALIDATION
# ------------------------------------------------------------------------------------------
each_split_classes = (
    sample_df
    .groupby(
        "Dataset_Split"
    )[
        "Target"
    ]
    .nunique()
)

checks = {
    "Label raster uses EPSG:32632":
        label_epsg == 32632,

    "Label raster matches reference dimensions":
        same_dimensions,

    "Label raster matches reference transform":
        same_transform,

    "Positive sample target reached":
        saved_positive_count
        ==
        TARGET_POSITIVE_SAMPLES,

    "Negative sample target reached":
        saved_negative_count
        ==
        TARGET_NEGATIVE_SAMPLES,

    "Classes are balanced":
        saved_positive_count
        ==
        saved_negative_count,

    "Total sample count is 24,000":
        saved_sample_count
        ==
        (
            TARGET_POSITIVE_SAMPLES
            +
            TARGET_NEGATIVE_SAMPLES
        ),

    "Multiple positive components represented":
        np.unique(
            sampled_positive_components
        ).size >= 1_000,

    "Multiple positive spatial blocks represented":
        np.unique(
            sampled_positive_blocks
        ).size >= 20,

    "Training split contains both classes":
        each_split_classes.get(
            "Train",
            0
        ) == 2,

    "Validation split contains both classes":
        each_split_classes.get(
            "Validation",
            0
        ) == 2,

    "Test split contains both classes":
        each_split_classes.get(
            "Test",
            0
        ) == 2,

    "Training fraction is acceptable":
        abs(
            actual_train_fraction
            -
            TRAIN_FRACTION
        )
        <=
        split_fraction_tolerance,

    "Validation fraction is acceptable":
        abs(
            actual_validation_fraction
            -
            VALIDATION_FRACTION
        )
        <=
        split_fraction_tolerance,

    "Test fraction is acceptable":
        abs(
            actual_test_fraction
            -
            TEST_FRACTION
        )
        <=
        split_fraction_tolerance,

    "No train-validation block overlap":
        len(
            train_validation_overlap
        ) == 0,

    "No train-test block overlap":
        len(
            train_test_overlap
        ) == 0,

    "No validation-test block overlap":
        len(
            validation_test_overlap
        ) == 0,

    "Saved sample CRS is EPSG:32632":
        saved_epsg == 32632,

    "Saved target classes are 0 and 1":
        saved_target_values == [0, 1],

    "Saved dataset splits are complete":
        saved_split_values
        ==
        [
            "Test",
            "Train",
            "Validation"
        ],

    "GeoPackage saved":
        SAMPLE_GPKG_FILE.exists(),

    "CSV sample table saved":
        SAMPLE_CSV_FILE.exists(),

    "Sampling summary saved":
        SAMPLE_SUMMARY_FILE.exists(),

    "Block summary saved":
        BLOCK_SUMMARY_FILE.exists(),

    "Component summary saved":
        COMPONENT_SUMMARY_FILE.exists(),

    "Sampling design saved":
        DESIGN_FILE.exists(),

    "Validation register saved":
        VALIDATION_FILE.exists(),

    "Predictor extraction remains pending":
        validation_df[
            "Predictor_Extraction_Completed"
        ].iloc[0] == False,

    "Model training remains pending":
        validation_df[
            "Model_Training_Completed"
        ].iloc[0] == False
}

print("\nVALIDATION SUMMARY")
print("-" * 120)

for label, passed in checks.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{label}"
    )

print("\nOUTPUT FILES")
print("-" * 120)

for path in [
    SAMPLE_GPKG_FILE,
    SAMPLE_CSV_FILE,
    SAMPLE_SUMMARY_FILE,
    BLOCK_SUMMARY_FILE,
    COMPONENT_SUMMARY_FILE,
    DESIGN_FILE,
    VALIDATION_FILE
]:

    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{path.name}"
    )

if all(
    checks.values()
):

    print("\n✓ STAGE 9E FULLY PASSED")

    print(
        "A balanced and spatially independent ML sample "
        "dataset has been created."
    )

    print(
        "The next stage will extract leakage-safe predictor "
        "values for the 24,000 sample points."
    )

else:

    print("\n⚠ STAGE 9E REQUIRES REVIEW")

print("=" * 120)

In [ ]:
# ==========================================================================================
# PROJECT 7 — STAGE 9E FIX
# REBALANCE EXISTING 5 KM BLOCKS INTO 70/15/15 DATASET SPLITS
#
# This fix:
# - preserves all 24,000 existing sample points;
# - preserves the 1:1 class balance;
# - preserves complete block independence;
# - does not resample or alter sample coordinates;
# - replaces only the Dataset_Split assignments.
#
# Allocation method:
# - sort blocks from largest to smallest;
# - assign each whole block to the split with the greatest remaining need;
# - run multiple randomised block-order trials;
# - retain the allocation closest to 70% / 15% / 15%.
# ==========================================================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd
import geopandas as gpd

# ------------------------------------------------------------------------------------------
# PARAMETERS
# ------------------------------------------------------------------------------------------
RANDOM_SEED = 42
NUMBER_OF_TRIALS = 10_000

TARGET_FRACTIONS = {
    "Train": 0.70,
    "Validation": 0.15,
    "Test": 0.15
}

FRACTION_TOLERANCE = 0.03

rng = np.random.default_rng(RANDOM_SEED)

# ------------------------------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------------------------------
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

TRAINING_DIR = PROJECT_ROOT / "09_Training_Data"
TABLE_DIR = PROJECT_ROOT / "16_Tables"
ADMIN_DIR = PROJECT_ROOT / "00_Project_Admin"

SAMPLE_GPKG_FILE = (
    TRAINING_DIR /
    "Enugu_ML_Spatial_Samples_2020_2025.gpkg"
)

SAMPLE_CSV_FILE = (
    TRAINING_DIR /
    "Enugu_ML_Spatial_Samples_2020_2025.csv"
)

SAMPLE_SUMMARY_FILE = (
    TABLE_DIR /
    "Enugu_ML_Spatial_Sampling_Summary.csv"
)

BLOCK_SUMMARY_FILE = (
    TABLE_DIR /
    "Enugu_ML_Spatial_Block_Split_Summary.csv"
)

VALIDATION_FILE = (
    TRAINING_DIR /
    "Stage_9E_Spatial_Sampling_Validation.csv"
)

DESIGN_FILE = (
    ADMIN_DIR /
    "ML_Spatial_Sampling_Design.json"
)

print("=" * 120)
print("STAGE 9E FIX — REBALANCE EXISTING SPATIAL BLOCK SPLITS")
print("=" * 120)

# ------------------------------------------------------------------------------------------
# REQUIRED FILE CHECK
# ------------------------------------------------------------------------------------------
required_files = {
    "Sample GeoPackage": SAMPLE_GPKG_FILE,
    "Sample CSV": SAMPLE_CSV_FILE
}

print("\nREQUIRED FILE CHECK")
print("-" * 120)

for label, path in required_files.items():

    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{label}: {path}"
    )

if not all(path.exists() for path in required_files.values()):

    raise FileNotFoundError(
        "The Stage 9E sample files could not be found."
    )

# ------------------------------------------------------------------------------------------
# LOAD EXISTING SAMPLES
# ------------------------------------------------------------------------------------------
sample_gdf = gpd.read_file(
    SAMPLE_GPKG_FILE,
    layer="ml_samples"
)

required_columns = {
    "Sample_ID",
    "Target",
    "Target_Name",
    "Dataset_Split",
    "Block_ID"
}

missing_columns = (
    required_columns -
    set(sample_gdf.columns)
)

if missing_columns:

    raise RuntimeError(
        f"Required sample columns are missing: {missing_columns}"
    )

sample_gdf["Block_ID"] = (
    sample_gdf["Block_ID"]
    .astype(np.int64)
)

sample_gdf["Target"] = (
    sample_gdf["Target"]
    .astype(np.uint8)
)

total_samples = int(
    sample_gdf.shape[0]
)

positive_samples = int(
    (sample_gdf["Target"] == 1).sum()
)

negative_samples = int(
    (sample_gdf["Target"] == 0).sum()
)

print("\nEXISTING SAMPLE DATASET")
print("-" * 120)

print(f"Total samples: {total_samples:,}")
print(f"Positive samples: {positive_samples:,}")
print(f"Negative samples: {negative_samples:,}")
print(
    f"Unique spatial blocks: "
    f"{sample_gdf['Block_ID'].nunique():,}"
)

if total_samples != 24_000:

    raise RuntimeError(
        f"Expected 24,000 samples, found {total_samples:,}."
    )

if positive_samples != 12_000 or negative_samples != 12_000:

    raise RuntimeError(
        "The existing sample dataset is not balanced 1:1."
    )

# ------------------------------------------------------------------------------------------
# SUMMARISE EACH SPATIAL BLOCK
# ------------------------------------------------------------------------------------------
block_counts = (
    sample_gdf
    .groupby("Block_ID")
    .agg(
        Total_Count=("Sample_ID", "count"),
        Positive_Count=("Target", "sum")
    )
    .reset_index()
)

block_counts["Negative_Count"] = (
    block_counts["Total_Count"]
    -
    block_counts["Positive_Count"]
)

block_ids = (
    block_counts["Block_ID"]
    .to_numpy(dtype=np.int64)
)

block_totals = (
    block_counts["Total_Count"]
    .to_numpy(dtype=np.int64)
)

block_positives = (
    block_counts["Positive_Count"]
    .to_numpy(dtype=np.int64)
)

block_negatives = (
    block_counts["Negative_Count"]
    .to_numpy(dtype=np.int64)
)

split_names = [
    "Train",
    "Validation",
    "Test"
]

target_total = {
    split: total_samples * fraction
    for split, fraction in TARGET_FRACTIONS.items()
}

target_positive = {
    split: positive_samples * fraction
    for split, fraction in TARGET_FRACTIONS.items()
}

target_negative = {
    split: negative_samples * fraction
    for split, fraction in TARGET_FRACTIONS.items()
}

print("\nTARGET SAMPLE COUNTS")
print("-" * 120)

for split in split_names:

    print(
        f"{split}: "
        f"{target_total[split]:,.0f} total, "
        f"{target_positive[split]:,.0f} positive, "
        f"{target_negative[split]:,.0f} negative"
    )

# ------------------------------------------------------------------------------------------
# ALLOCATION OBJECTIVE
# ------------------------------------------------------------------------------------------
def calculate_score(assignments):
    """
    Lower score is better.

    Measures deviation from target totals and class counts.
    """

    score = 0.0

    for split in split_names:

        mask = assignments == split

        actual_total = block_totals[mask].sum()
        actual_positive = block_positives[mask].sum()
        actual_negative = block_negatives[mask].sum()

        total_error = (
            (actual_total - target_total[split])
            /
            max(target_total[split], 1)
        ) ** 2

        positive_error = (
            (actual_positive - target_positive[split])
            /
            max(target_positive[split], 1)
        ) ** 2

        negative_error = (
            (actual_negative - target_negative[split])
            /
            max(target_negative[split], 1)
        ) ** 2

        score += (
            total_error
            +
            positive_error
            +
            negative_error
        )

    return float(score)


def create_greedy_assignment(order):
    """
    Assign each entire block to the split with the greatest
    remaining proportional requirement.
    """

    assignments = np.empty(
        block_ids.size,
        dtype=object
    )

    running = {
        split: {
            "Total": 0,
            "Positive": 0,
            "Negative": 0
        }
        for split in split_names
    }

    for block_index in order:

        candidate_scores = {}

        for split in split_names:

            new_total = (
                running[split]["Total"]
                +
                block_totals[block_index]
            )

            new_positive = (
                running[split]["Positive"]
                +
                block_positives[block_index]
            )

            new_negative = (
                running[split]["Negative"]
                +
                block_negatives[block_index]
            )

            total_deviation = abs(
                new_total
                -
                target_total[split]
            ) / max(
                target_total[split],
                1
            )

            positive_deviation = abs(
                new_positive
                -
                target_positive[split]
            ) / max(
                target_positive[split],
                1
            )

            negative_deviation = abs(
                new_negative
                -
                target_negative[split]
            ) / max(
                target_negative[split],
                1
            )

            current_fill = (
                running[split]["Total"]
                /
                max(target_total[split], 1)
            )

            overfill_penalty = max(
                0,
                (
                    new_total
                    -
                    target_total[split]
                )
                /
                max(target_total[split], 1)
            )

            candidate_scores[split] = (
                total_deviation
                +
                positive_deviation
                +
                negative_deviation
                +
                0.05 * current_fill
                +
                4.0 * overfill_penalty
            )

        selected_split = min(
            candidate_scores,
            key=candidate_scores.get
        )

        assignments[block_index] = selected_split

        running[selected_split]["Total"] += (
            block_totals[block_index]
        )

        running[selected_split]["Positive"] += (
            block_positives[block_index]
        )

        running[selected_split]["Negative"] += (
            block_negatives[block_index]
        )

    return assignments


# ------------------------------------------------------------------------------------------
# MULTI-TRIAL SEARCH
# ------------------------------------------------------------------------------------------
print("\nSEARCHING FOR IMPROVED BLOCK ALLOCATION")
print("-" * 120)

largest_first_order = np.argsort(
    -block_totals
)

best_assignment = create_greedy_assignment(
    largest_first_order
)

best_score = calculate_score(
    best_assignment
)

for trial in range(NUMBER_OF_TRIALS):

    # Retain broad largest-first behaviour while slightly randomising
    # blocks with comparable sample sizes.
    random_noise = rng.uniform(
        0,
        max(block_totals.max() * 0.15, 1),
        size=block_totals.size
    )

    trial_order = np.argsort(
        -(
            block_totals
            +
            random_noise
        )
    )

    trial_assignment = create_greedy_assignment(
        trial_order
    )

    trial_score = calculate_score(
        trial_assignment
    )

    if trial_score < best_score:

        best_score = trial_score
        best_assignment = trial_assignment.copy()

print(f"Trials completed: {NUMBER_OF_TRIALS:,}")
print(f"Best allocation score: {best_score:.8f}")

# ------------------------------------------------------------------------------------------
# MAP BLOCK IDS TO NEW SPLITS
# ------------------------------------------------------------------------------------------
block_assignment = {
    int(block_id): split
    for block_id, split in zip(
        block_ids,
        best_assignment
    )
}

sample_gdf["Dataset_Split"] = (
    sample_gdf["Block_ID"]
    .map(block_assignment)
)

if sample_gdf["Dataset_Split"].isna().any():

    raise RuntimeError(
        "One or more spatial blocks were not assigned."
    )

# ------------------------------------------------------------------------------------------
# NEW SPLIT SUMMARY
# ------------------------------------------------------------------------------------------
split_summary = (
    sample_gdf
    .groupby(
        [
            "Dataset_Split",
            "Target_Name"
        ]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(split_names)
)

split_summary["Total"] = (
    split_summary.sum(axis=1)
)

print("\nREBALANCED DATASET SPLIT")
print("-" * 120)

print(
    split_summary.to_string()
)

actual_fractions = {
    split:
        int(split_summary.loc[split, "Total"])
        /
        total_samples
    for split in split_names
}

print("\nACTUAL SPLIT FRACTIONS")
print("-" * 120)

for split in split_names:

    print(
        f"{split}: "
        f"{actual_fractions[split] * 100:.2f}% "
        f"(target {TARGET_FRACTIONS[split] * 100:.0f}%)"
    )

# ------------------------------------------------------------------------------------------
# BLOCK COUNTS AND OVERLAP
# ------------------------------------------------------------------------------------------
split_blocks = {
    split: set(
        sample_gdf.loc[
            sample_gdf["Dataset_Split"] == split,
            "Block_ID"
        ].astype(np.int64)
    )
    for split in split_names
}

train_validation_overlap = (
    split_blocks["Train"]
    &
    split_blocks["Validation"]
)

train_test_overlap = (
    split_blocks["Train"]
    &
    split_blocks["Test"]
)

validation_test_overlap = (
    split_blocks["Validation"]
    &
    split_blocks["Test"]
)

print("\nSPATIAL BLOCK COUNTS")
print("-" * 120)

for split in split_names:

    print(
        f"{split}: "
        f"{len(split_blocks[split]):,} blocks"
    )

print("\nBLOCK OVERLAP")
print("-" * 120)

print(
    "Train–Validation overlap:",
    len(train_validation_overlap)
)

print(
    "Train–Test overlap:",
    len(train_test_overlap)
)

print(
    "Validation–Test overlap:",
    len(validation_test_overlap)
)

# ------------------------------------------------------------------------------------------
# VALIDATE CLASS PRESENCE
# ------------------------------------------------------------------------------------------
class_counts_by_split = (
    sample_gdf
    .groupby("Dataset_Split")["Target"]
    .nunique()
)

# ------------------------------------------------------------------------------------------
# SAVE UPDATED SAMPLE FILES
# ------------------------------------------------------------------------------------------
sample_df = pd.DataFrame(
    sample_gdf.drop(
        columns="geometry"
    )
)

sample_df.to_csv(
    SAMPLE_CSV_FILE,
    index=False
)

if SAMPLE_GPKG_FILE.exists():

    SAMPLE_GPKG_FILE.unlink()

sample_gdf.to_file(
    SAMPLE_GPKG_FILE,
    layer="ml_samples",
    driver="GPKG"
)

# ------------------------------------------------------------------------------------------
# SAVE UPDATED SAMPLE SUMMARY
# ------------------------------------------------------------------------------------------
sample_summary_df = (
    sample_df
    .groupby(
        [
            "Dataset_Split",
            "Target",
            "Target_Name"
        ]
    )
    .size()
    .reset_index(
        name="Sample_Count"
    )
)

sample_summary_df["Percent_of_Class"] = (
    sample_summary_df["Sample_Count"]
    /
    sample_summary_df.groupby(
        "Target"
    )["Sample_Count"].transform("sum")
    *
    100
)

sample_summary_df.to_csv(
    SAMPLE_SUMMARY_FILE,
    index=False
)

# ------------------------------------------------------------------------------------------
# SAVE UPDATED BLOCK SUMMARY
# ------------------------------------------------------------------------------------------
block_split_summary_df = (
    sample_df
    .groupby(
        [
            "Dataset_Split",
            "Block_ID"
        ]
    )
    .agg(
        Total_Samples=("Sample_ID", "count"),
        Positive_Samples=("Target", "sum")
    )
    .reset_index()
)

block_split_summary_df["Negative_Samples"] = (
    block_split_summary_df["Total_Samples"]
    -
    block_split_summary_df["Positive_Samples"]
)

block_split_summary_df.to_csv(
    BLOCK_SUMMARY_FILE,
    index=False
)

# ------------------------------------------------------------------------------------------
# UPDATE SAMPLING DESIGN FILE
# ------------------------------------------------------------------------------------------
sampling_design = {
    "random_seed":
        RANDOM_SEED,

    "positive_sample_target":
        12_000,

    "negative_sample_target":
        12_000,

    "class_ratio":
        "1:1",

    "spatial_block_size_metres":
        5_000,

    "requested_split": {
        "training": 0.70,
        "validation": 0.15,
        "testing": 0.15
    },

    "actual_split": {
        "training":
            actual_fractions["Train"],

        "validation":
            actual_fractions["Validation"],

        "testing":
            actual_fractions["Test"]
    },

    "split_correction":
        "Existing samples retained; whole 5 km blocks "
        "reassigned using multi-trial constrained greedy allocation",

    "allocation_trials":
        NUMBER_OF_TRIALS,

    "allocation_score":
        best_score,

    "block_overlap":
        {
            "train_validation":
                len(train_validation_overlap),

            "train_test":
                len(train_test_overlap),

            "validation_test":
                len(validation_test_overlap)
        },

    "predictor_extraction_status":
        "Pending",

    "model_training_status":
        "Pending"
}

with open(
    DESIGN_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        sampling_design,
        file,
        indent=4
    )

# ------------------------------------------------------------------------------------------
# REOPEN SAVED GEOPACKAGE
# ------------------------------------------------------------------------------------------
saved_gdf = gpd.read_file(
    SAMPLE_GPKG_FILE,
    layer="ml_samples"
)

saved_count = int(
    saved_gdf.shape[0]
)

saved_positive = int(
    (saved_gdf["Target"] == 1).sum()
)

saved_negative = int(
    (saved_gdf["Target"] == 0).sum()
)

saved_epsg = (
    saved_gdf.crs.to_epsg()
    if saved_gdf.crs is not None
    else None
)

saved_splits = sorted(
    saved_gdf["Dataset_Split"]
    .unique()
    .tolist()
)

# ------------------------------------------------------------------------------------------
# VALIDATION REGISTER
# ------------------------------------------------------------------------------------------
validation_df = pd.DataFrame([{
    "Random_Seed":
        RANDOM_SEED,

    "Allocation_Trials":
        NUMBER_OF_TRIALS,

    "Allocation_Score":
        best_score,

    "Selected_Positive_Samples":
        saved_positive,

    "Selected_Negative_Samples":
        saved_negative,

    "Total_Samples":
        saved_count,

    "Train_Samples":
        int(
            split_summary.loc[
                "Train",
                "Total"
            ]
        ),

    "Validation_Samples":
        int(
            split_summary.loc[
                "Validation",
                "Total"
            ]
        ),

    "Test_Samples":
        int(
            split_summary.loc[
                "Test",
                "Total"
            ]
        ),

    "Train_Fraction":
        actual_fractions["Train"],

    "Validation_Fraction":
        actual_fractions["Validation"],

    "Test_Fraction":
        actual_fractions["Test"],

    "Train_Blocks":
        len(split_blocks["Train"]),

    "Validation_Blocks":
        len(split_blocks["Validation"]),

    "Test_Blocks":
        len(split_blocks["Test"]),

    "Train_Validation_Block_Overlap":
        len(train_validation_overlap),

    "Train_Test_Block_Overlap":
        len(train_test_overlap),

    "Validation_Test_Block_Overlap":
        len(validation_test_overlap),

    "Predictor_Extraction_Completed":
        False,

    "Model_Training_Completed":
        False
}])

validation_df.to_csv(
    VALIDATION_FILE,
    index=False
)

# ------------------------------------------------------------------------------------------
# FINAL VALIDATION
# ------------------------------------------------------------------------------------------
checks = {
    "All 24,000 samples preserved":
        saved_count == 24_000,

    "All 12,000 positive samples preserved":
        saved_positive == 12_000,

    "All 12,000 negative samples preserved":
        saved_negative == 12_000,

    "Training split contains both classes":
        class_counts_by_split.get(
            "Train",
            0
        ) == 2,

    "Validation split contains both classes":
        class_counts_by_split.get(
            "Validation",
            0
        ) == 2,

    "Test split contains both classes":
        class_counts_by_split.get(
            "Test",
            0
        ) == 2,

    "Training fraction within ±3 percentage points":
        abs(
            actual_fractions["Train"]
            -
            TARGET_FRACTIONS["Train"]
        ) <= FRACTION_TOLERANCE,

    "Validation fraction within ±3 percentage points":
        abs(
            actual_fractions["Validation"]
            -
            TARGET_FRACTIONS["Validation"]
        ) <= FRACTION_TOLERANCE,

    "Test fraction within ±3 percentage points":
        abs(
            actual_fractions["Test"]
            -
            TARGET_FRACTIONS["Test"]
        ) <= FRACTION_TOLERANCE,

    "No train-validation block overlap":
        len(train_validation_overlap) == 0,

    "No train-test block overlap":
        len(train_test_overlap) == 0,

    "No validation-test block overlap":
        len(validation_test_overlap) == 0,

    "Saved GeoPackage CRS is EPSG:32632":
        saved_epsg == 32632,

    "Saved dataset splits are complete":
        saved_splits
        ==
        [
            "Test",
            "Train",
            "Validation"
        ],

    "Updated sample GeoPackage saved":
        SAMPLE_GPKG_FILE.exists(),

    "Updated sample CSV saved":
        SAMPLE_CSV_FILE.exists(),

    "Updated sampling summary saved":
        SAMPLE_SUMMARY_FILE.exists(),

    "Updated block summary saved":
        BLOCK_SUMMARY_FILE.exists(),

    "Updated design register saved":
        DESIGN_FILE.exists(),

    "Updated validation register saved":
        VALIDATION_FILE.exists(),

    "Predictor extraction remains pending":
        validation_df[
            "Predictor_Extraction_Completed"
        ].iloc[0] == False,

    "Model training remains pending":
        validation_df[
            "Model_Training_Completed"
        ].iloc[0] == False
}

print("\nVALIDATION SUMMARY")
print("-" * 120)

for label, passed in checks.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{label}"
    )

print("\nUPDATED OUTPUT FILES")
print("-" * 120)

for path in [
    SAMPLE_GPKG_FILE,
    SAMPLE_CSV_FILE,
    SAMPLE_SUMMARY_FILE,
    BLOCK_SUMMARY_FILE,
    DESIGN_FILE,
    VALIDATION_FILE
]:

    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{path.name}"
    )

if all(checks.values()):

    print("\n✓ STAGE 9E FIX FULLY PASSED")

    print(
        "The existing 24,000 samples have been successfully "
        "reallocated into approximately 70% training, "
        "15% validation and 15% testing datasets."
    )

    print(
        "All 5 km spatial blocks remain exclusive to one split."
    )

else:

    print("\n⚠ STAGE 9E FIX REQUIRES REVIEW")

print("=" * 120)

In [ ]:
# ==========================================================================================
# PROJECT 7 — STAGE 9E FIX 2
# OPTIMISED 5 KM BLOCK ALLOCATION USING MIXED-INTEGER LINEAR PROGRAMMING
#
# This cell:
# - preserves all 24,000 existing samples;
# - preserves their coordinates, classes and positive-component representation;
# - assigns each entire 5 km block to exactly one dataset split;
# - directly minimises deviation from 70% / 15% / 15%;
# - prevents spatial-block overlap;
# - updates the existing sample and validation files.
#
# No sampling is repeated.
# ==========================================================================================

from pathlib import Path
import json

import numpy as np
import pandas as pd
import geopandas as gpd

from scipy.optimize import (
    milp,
    LinearConstraint,
    Bounds
)

# ------------------------------------------------------------------------------------------
# PARAMETERS
# ------------------------------------------------------------------------------------------
TARGET_FRACTIONS = {
    "Train": 0.70,
    "Validation": 0.15,
    "Test": 0.15
}

SPLIT_NAMES = [
    "Train",
    "Validation",
    "Test"
]

FRACTION_TOLERANCE = 0.02

# ------------------------------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------------------------------
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

TRAINING_DIR = (
    PROJECT_ROOT /
    "09_Training_Data"
)

TABLE_DIR = (
    PROJECT_ROOT /
    "16_Tables"
)

ADMIN_DIR = (
    PROJECT_ROOT /
    "00_Project_Admin"
)

SAMPLE_GPKG_FILE = (
    TRAINING_DIR /
    "Enugu_ML_Spatial_Samples_2020_2025.gpkg"
)

SAMPLE_CSV_FILE = (
    TRAINING_DIR /
    "Enugu_ML_Spatial_Samples_2020_2025.csv"
)

SAMPLE_SUMMARY_FILE = (
    TABLE_DIR /
    "Enugu_ML_Spatial_Sampling_Summary.csv"
)

BLOCK_SUMMARY_FILE = (
    TABLE_DIR /
    "Enugu_ML_Spatial_Block_Split_Summary.csv"
)

VALIDATION_FILE = (
    TRAINING_DIR /
    "Stage_9E_Spatial_Sampling_Validation.csv"
)

DESIGN_FILE = (
    ADMIN_DIR /
    "ML_Spatial_Sampling_Design.json"
)

print("=" * 120)
print("STAGE 9E FIX 2 — OPTIMISED SPATIAL BLOCK ALLOCATION")
print("=" * 120)

# ------------------------------------------------------------------------------------------
# REQUIRED FILE CHECK
# ------------------------------------------------------------------------------------------
required_files = {
    "Sample GeoPackage": SAMPLE_GPKG_FILE,
    "Sample CSV": SAMPLE_CSV_FILE
}

print("\nREQUIRED FILE CHECK")
print("-" * 120)

for label, path in required_files.items():

    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{label}: {path}"
    )

if not all(
    path.exists()
    for path in required_files.values()
):

    raise FileNotFoundError(
        "The existing Stage 9E sample files are missing."
    )

# ------------------------------------------------------------------------------------------
# LOAD EXISTING SAMPLE DATA
# ------------------------------------------------------------------------------------------
sample_gdf = gpd.read_file(
    SAMPLE_GPKG_FILE,
    layer="ml_samples"
)

required_columns = {
    "Sample_ID",
    "Target",
    "Target_Name",
    "Dataset_Split",
    "Block_ID"
}

missing_columns = (
    required_columns -
    set(sample_gdf.columns)
)

if missing_columns:

    raise RuntimeError(
        f"Missing required columns: {sorted(missing_columns)}"
    )

sample_gdf["Block_ID"] = (
    sample_gdf["Block_ID"]
    .astype(np.int64)
)

sample_gdf["Target"] = (
    sample_gdf["Target"]
    .astype(np.uint8)
)

total_samples = int(
    sample_gdf.shape[0]
)

positive_samples = int(
    np.count_nonzero(
        sample_gdf["Target"] == 1
    )
)

negative_samples = int(
    np.count_nonzero(
        sample_gdf["Target"] == 0
    )
)

unique_blocks = int(
    sample_gdf["Block_ID"].nunique()
)

print("\nEXISTING SAMPLE DATASET")
print("-" * 120)

print(
    f"Total samples: {total_samples:,}"
)

print(
    f"Positive samples: {positive_samples:,}"
)

print(
    f"Negative samples: {negative_samples:,}"
)

print(
    f"Unique spatial blocks: {unique_blocks:,}"
)

if total_samples != 24_000:

    raise RuntimeError(
        f"Expected 24,000 samples, found {total_samples:,}."
    )

if (
    positive_samples != 12_000
    or
    negative_samples != 12_000
):

    raise RuntimeError(
        "The existing sample dataset is not balanced."
    )

# ------------------------------------------------------------------------------------------
# SUMMARISE SAMPLES BY BLOCK
# ------------------------------------------------------------------------------------------
block_df = (
    sample_gdf
    .groupby("Block_ID")
    .agg(
        Total_Count=("Sample_ID", "count"),
        Positive_Count=("Target", "sum")
    )
    .reset_index()
)

block_df["Negative_Count"] = (
    block_df["Total_Count"]
    -
    block_df["Positive_Count"]
)

block_ids = (
    block_df["Block_ID"]
    .to_numpy(dtype=np.int64)
)

block_total = (
    block_df["Total_Count"]
    .to_numpy(dtype=np.float64)
)

block_positive = (
    block_df["Positive_Count"]
    .to_numpy(dtype=np.float64)
)

block_negative = (
    block_df["Negative_Count"]
    .to_numpy(dtype=np.float64)
)

number_of_blocks = int(
    block_df.shape[0]
)

number_of_splits = len(
    SPLIT_NAMES
)

target_total = np.array(
    [
        total_samples *
        TARGET_FRACTIONS[split]
        for split in SPLIT_NAMES
    ],
    dtype=np.float64
)

target_positive = np.array(
    [
        positive_samples *
        TARGET_FRACTIONS[split]
        for split in SPLIT_NAMES
    ],
    dtype=np.float64
)

target_negative = np.array(
    [
        negative_samples *
        TARGET_FRACTIONS[split]
        for split in SPLIT_NAMES
    ],
    dtype=np.float64
)

print("\nTARGET SAMPLE COUNTS")
print("-" * 120)

for index, split in enumerate(SPLIT_NAMES):

    print(
        f"{split}: "
        f"{target_total[index]:,.0f} total, "
        f"{target_positive[index]:,.0f} positive, "
        f"{target_negative[index]:,.0f} negative"
    )

# ------------------------------------------------------------------------------------------
# MILP VARIABLE DESIGN
#
# Binary variables:
# x[b,s] = 1 when block b is assigned to split s.
#
# Continuous deviation variables:
# For each split and metric:
# - positive deviation
# - negative deviation
#
# Metrics:
# - total samples
# - positive samples
# - negative samples
# ------------------------------------------------------------------------------------------
number_of_assignment_variables = (
    number_of_blocks *
    number_of_splits
)

number_of_metrics = 3

number_of_deviation_variables = (
    number_of_splits *
    number_of_metrics *
    2
)

number_of_variables = (
    number_of_assignment_variables
    +
    number_of_deviation_variables
)

def assignment_index(block_index, split_index):

    return (
        block_index *
        number_of_splits
        +
        split_index
    )


def deviation_index(
    split_index,
    metric_index,
    direction_index
):
    """
    direction_index:
    0 = positive deviation
    1 = negative deviation
    """

    offset = (
        split_index *
        number_of_metrics *
        2
        +
        metric_index *
        2
        +
        direction_index
    )

    return (
        number_of_assignment_variables
        +
        offset
    )

# ------------------------------------------------------------------------------------------
# OBJECTIVE FUNCTION
#
# Assignment variables have zero direct cost.
# Deviations are normalised by their target sizes.
# ------------------------------------------------------------------------------------------
objective = np.zeros(
    number_of_variables,
    dtype=np.float64
)

for split_index in range(number_of_splits):

    metric_targets = [
        target_total[split_index],
        target_positive[split_index],
        target_negative[split_index]
    ]

    for metric_index, metric_target in enumerate(
        metric_targets
    ):

        normalised_cost = (
            1.0 /
            max(metric_target, 1)
        )

        objective[
            deviation_index(
                split_index,
                metric_index,
                0
            )
        ] = normalised_cost

        objective[
            deviation_index(
                split_index,
                metric_index,
                1
            )
        ] = normalised_cost

# ------------------------------------------------------------------------------------------
# CONSTRAINT 1
# EACH BLOCK MUST BE ASSIGNED TO EXACTLY ONE SPLIT
# ------------------------------------------------------------------------------------------
block_assignment_matrix = np.zeros(
    (
        number_of_blocks,
        number_of_variables
    ),
    dtype=np.float64
)

for block_index in range(number_of_blocks):

    for split_index in range(number_of_splits):

        block_assignment_matrix[
            block_index,
            assignment_index(
                block_index,
                split_index
            )
        ] = 1

block_assignment_constraint = LinearConstraint(
    block_assignment_matrix,
    lb=np.ones(number_of_blocks),
    ub=np.ones(number_of_blocks)
)

# ------------------------------------------------------------------------------------------
# CONSTRAINT 2
# SPLIT TOTALS EQUAL TARGET PLUS OR MINUS DEVIATION
#
# Actual - positive_deviation + negative_deviation = target
# ------------------------------------------------------------------------------------------
metric_constraint_rows = []
metric_constraint_targets = []

for split_index in range(number_of_splits):

    metric_values = [
        block_total,
        block_positive,
        block_negative
    ]

    metric_targets = [
        target_total[split_index],
        target_positive[split_index],
        target_negative[split_index]
    ]

    for metric_index in range(number_of_metrics):

        row = np.zeros(
            number_of_variables,
            dtype=np.float64
        )

        for block_index in range(number_of_blocks):

            row[
                assignment_index(
                    block_index,
                    split_index
                )
            ] = metric_values[
                metric_index
            ][
                block_index
            ]

        row[
            deviation_index(
                split_index,
                metric_index,
                0
            )
        ] = -1

        row[
            deviation_index(
                split_index,
                metric_index,
                1
            )
        ] = 1

        metric_constraint_rows.append(
            row
        )

        metric_constraint_targets.append(
            metric_targets[
                metric_index
            ]
        )

metric_constraint_matrix = np.vstack(
    metric_constraint_rows
)

metric_constraint_targets = np.asarray(
    metric_constraint_targets,
    dtype=np.float64
)

metric_constraint = LinearConstraint(
    metric_constraint_matrix,
    lb=metric_constraint_targets,
    ub=metric_constraint_targets
)

# ------------------------------------------------------------------------------------------
# VARIABLE TYPES AND BOUNDS
# ------------------------------------------------------------------------------------------
integrality = np.zeros(
    number_of_variables,
    dtype=np.uint8
)

integrality[
    :number_of_assignment_variables
] = 1

lower_bounds = np.zeros(
    number_of_variables,
    dtype=np.float64
)

upper_bounds = np.full(
    number_of_variables,
    np.inf,
    dtype=np.float64
)

upper_bounds[
    :number_of_assignment_variables
] = 1

bounds = Bounds(
    lower_bounds,
    upper_bounds
)

# ------------------------------------------------------------------------------------------
# RUN MILP OPTIMISATION
# ------------------------------------------------------------------------------------------
print("\nRUNNING MIXED-INTEGER OPTIMISATION")
print("-" * 120)

result = milp(
    c=objective,
    integrality=integrality,
    bounds=bounds,
    constraints=[
        block_assignment_constraint,
        metric_constraint
    ],
    options={
        "time_limit": 120,
        "mip_rel_gap": 0.0,
        "presolve": True
    }
)

print(
    f"Solver status code: {result.status}"
)

print(
    f"Solver message: {result.message}"
)

if result.x is None:

    raise RuntimeError(
        "The MILP solver did not return a feasible solution."
    )

print(
    f"Objective value: {result.fun:.10f}"
)

# ------------------------------------------------------------------------------------------
# EXTRACT BLOCK ASSIGNMENTS
# ------------------------------------------------------------------------------------------
assignment_values = (
    result.x[
        :number_of_assignment_variables
    ]
    .reshape(
        number_of_blocks,
        number_of_splits
    )
)

assigned_split_indices = np.argmax(
    assignment_values,
    axis=1
)

assigned_split_names = np.array(
    [
        SPLIT_NAMES[index]
        for index in assigned_split_indices
    ],
    dtype=object
)

block_assignment = {
    int(block_id): split_name
    for block_id, split_name in zip(
        block_ids,
        assigned_split_names
    )
}

sample_gdf["Dataset_Split"] = (
    sample_gdf["Block_ID"]
    .map(block_assignment)
)

if sample_gdf["Dataset_Split"].isna().any():

    raise RuntimeError(
        "One or more blocks received no dataset assignment."
    )

# ------------------------------------------------------------------------------------------
# CALCULATE OPTIMISED SPLIT SUMMARY
# ------------------------------------------------------------------------------------------
split_summary = (
    sample_gdf
    .groupby(
        [
            "Dataset_Split",
            "Target_Name"
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reindex(
        SPLIT_NAMES,
        fill_value=0
    )
)

for required_class in [
    "Stable non-built",
    "Urban expansion"
]:

    if required_class not in split_summary.columns:

        split_summary[
            required_class
        ] = 0

split_summary["Total"] = (
    split_summary[
        "Stable non-built"
    ]
    +
    split_summary[
        "Urban expansion"
    ]
)

print("\nOPTIMISED DATASET SPLIT")
print("-" * 120)

print(
    split_summary.to_string()
)

actual_fractions = {
    split:
        int(
            split_summary.loc[
                split,
                "Total"
            ]
        )
        /
        total_samples
    for split in SPLIT_NAMES
}

print("\nACTUAL SPLIT FRACTIONS")
print("-" * 120)

for split in SPLIT_NAMES:

    print(
        f"{split}: "
        f"{actual_fractions[split] * 100:.3f}% "
        f"(target "
        f"{TARGET_FRACTIONS[split] * 100:.0f}%)"
    )

# ------------------------------------------------------------------------------------------
# BLOCK COUNTS AND OVERLAP
# ------------------------------------------------------------------------------------------
split_blocks = {
    split: set(
        sample_gdf.loc[
            sample_gdf[
                "Dataset_Split"
            ] == split,
            "Block_ID"
        ].astype(np.int64)
    )
    for split in SPLIT_NAMES
}

train_validation_overlap = (
    split_blocks["Train"]
    &
    split_blocks["Validation"]
)

train_test_overlap = (
    split_blocks["Train"]
    &
    split_blocks["Test"]
)

validation_test_overlap = (
    split_blocks["Validation"]
    &
    split_blocks["Test"]
)

print("\nSPATIAL BLOCK COUNTS")
print("-" * 120)

for split in SPLIT_NAMES:

    print(
        f"{split}: "
        f"{len(split_blocks[split]):,} blocks"
    )

print("\nBLOCK OVERLAP")
print("-" * 120)

print(
    "Train–Validation overlap:",
    len(train_validation_overlap)
)

print(
    "Train–Test overlap:",
    len(train_test_overlap)
)

print(
    "Validation–Test overlap:",
    len(validation_test_overlap)
)

# ------------------------------------------------------------------------------------------
# SAVE UPDATED SAMPLE FILES
# ------------------------------------------------------------------------------------------
sample_df = pd.DataFrame(
    sample_gdf.drop(
        columns="geometry"
    )
)

sample_df.to_csv(
    SAMPLE_CSV_FILE,
    index=False
)

if SAMPLE_GPKG_FILE.exists():

    SAMPLE_GPKG_FILE.unlink()

sample_gdf.to_file(
    SAMPLE_GPKG_FILE,
    layer="ml_samples",
    driver="GPKG"
)

# ------------------------------------------------------------------------------------------
# SAVE UPDATED SAMPLE SUMMARY
# ------------------------------------------------------------------------------------------
sample_summary_df = (
    sample_df
    .groupby(
        [
            "Dataset_Split",
            "Target",
            "Target_Name"
        ]
    )
    .size()
    .reset_index(
        name="Sample_Count"
    )
)

sample_summary_df["Percent_of_Class"] = (
    sample_summary_df["Sample_Count"]
    /
    sample_summary_df.groupby(
        "Target"
    )["Sample_Count"].transform(
        "sum"
    )
    *
    100
)

sample_summary_df.to_csv(
    SAMPLE_SUMMARY_FILE,
    index=False
)

# ------------------------------------------------------------------------------------------
# SAVE UPDATED BLOCK SUMMARY
# ------------------------------------------------------------------------------------------
block_split_summary_df = (
    sample_df
    .groupby(
        [
            "Dataset_Split",
            "Block_ID"
        ]
    )
    .agg(
        Total_Samples=(
            "Sample_ID",
            "count"
        ),
        Positive_Samples=(
            "Target",
            "sum"
        )
    )
    .reset_index()
)

block_split_summary_df[
    "Negative_Samples"
] = (
    block_split_summary_df[
        "Total_Samples"
    ]
    -
    block_split_summary_df[
        "Positive_Samples"
    ]
)

block_split_summary_df.to_csv(
    BLOCK_SUMMARY_FILE,
    index=False
)

# ------------------------------------------------------------------------------------------
# SAVE UPDATED DESIGN REGISTER
# ------------------------------------------------------------------------------------------
sampling_design = {
    "positive_samples":
        positive_samples,

    "negative_samples":
        negative_samples,

    "total_samples":
        total_samples,

    "class_ratio":
        "1:1",

    "spatial_block_size_metres":
        5_000,

    "requested_split": {
        "training":
            TARGET_FRACTIONS["Train"],

        "validation":
            TARGET_FRACTIONS["Validation"],

        "testing":
            TARGET_FRACTIONS["Test"]
    },

    "actual_split": {
        "training":
            actual_fractions["Train"],

        "validation":
            actual_fractions["Validation"],

        "testing":
            actual_fractions["Test"]
    },

    "allocation_method":
        "Mixed-integer linear programming",

    "allocation_objective":
        "Minimise normalised absolute deviations from "
        "total and class-specific split targets",

    "solver_status":
        int(result.status),

    "solver_message":
        str(result.message),

    "objective_value":
        float(result.fun),

    "block_overlap": {
        "train_validation":
            len(train_validation_overlap),

        "train_test":
            len(train_test_overlap),

        "validation_test":
            len(validation_test_overlap)
    },

    "predictor_extraction_status":
        "Pending",

    "model_training_status":
        "Pending"
}

with open(
    DESIGN_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        sampling_design,
        file,
        indent=4
    )

# ------------------------------------------------------------------------------------------
# REOPEN SAVED SAMPLE GEOPACKAGE
# ------------------------------------------------------------------------------------------
saved_gdf = gpd.read_file(
    SAMPLE_GPKG_FILE,
    layer="ml_samples"
)

saved_count = int(
    saved_gdf.shape[0]
)

saved_positive = int(
    np.count_nonzero(
        saved_gdf["Target"] == 1
    )
)

saved_negative = int(
    np.count_nonzero(
        saved_gdf["Target"] == 0
    )
)

saved_epsg = (
    saved_gdf.crs.to_epsg()
    if saved_gdf.crs is not None
    else None
)

saved_splits = sorted(
    saved_gdf[
        "Dataset_Split"
    ]
    .unique()
    .tolist()
)

saved_class_counts = (
    saved_gdf
    .groupby(
        "Dataset_Split"
    )[
        "Target"
    ]
    .nunique()
)

# ------------------------------------------------------------------------------------------
# VALIDATION REGISTER
# ------------------------------------------------------------------------------------------
validation_df = pd.DataFrame([{
    "Allocation_Method":
        "Mixed-integer linear programming",

    "Solver_Status":
        int(result.status),

    "Solver_Objective":
        float(result.fun),

    "Selected_Positive_Samples":
        saved_positive,

    "Selected_Negative_Samples":
        saved_negative,

    "Total_Samples":
        saved_count,

    "Train_Samples":
        int(
            split_summary.loc[
                "Train",
                "Total"
            ]
        ),

    "Validation_Samples":
        int(
            split_summary.loc[
                "Validation",
                "Total"
            ]
        ),

    "Test_Samples":
        int(
            split_summary.loc[
                "Test",
                "Total"
            ]
        ),

    "Train_Fraction":
        actual_fractions["Train"],

    "Validation_Fraction":
        actual_fractions["Validation"],

    "Test_Fraction":
        actual_fractions["Test"],

    "Train_Blocks":
        len(
            split_blocks["Train"]
        ),

    "Validation_Blocks":
        len(
            split_blocks["Validation"]
        ),

    "Test_Blocks":
        len(
            split_blocks["Test"]
        ),

    "Train_Validation_Block_Overlap":
        len(train_validation_overlap),

    "Train_Test_Block_Overlap":
        len(train_test_overlap),

    "Validation_Test_Block_Overlap":
        len(validation_test_overlap),

    "Predictor_Extraction_Completed":
        False,

    "Model_Training_Completed":
        False
}])

validation_df.to_csv(
    VALIDATION_FILE,
    index=False
)

# ------------------------------------------------------------------------------------------
# FINAL VALIDATION
# ------------------------------------------------------------------------------------------
checks = {
    "MILP returned a feasible solution":
        result.x is not None,

    "All 24,000 samples preserved":
        saved_count == 24_000,

    "All 12,000 positive samples preserved":
        saved_positive == 12_000,

    "All 12,000 negative samples preserved":
        saved_negative == 12_000,

    "Training split contains both classes":
        saved_class_counts.get(
            "Train",
            0
        ) == 2,

    "Validation split contains both classes":
        saved_class_counts.get(
            "Validation",
            0
        ) == 2,

    "Test split contains both classes":
        saved_class_counts.get(
            "Test",
            0
        ) == 2,

    "Training fraction within ±2 percentage points":
        abs(
            actual_fractions["Train"]
            -
            TARGET_FRACTIONS["Train"]
        )
        <=
        FRACTION_TOLERANCE,

    "Validation fraction within ±2 percentage points":
        abs(
            actual_fractions["Validation"]
            -
            TARGET_FRACTIONS["Validation"]
        )
        <=
        FRACTION_TOLERANCE,

    "Test fraction within ±2 percentage points":
        abs(
            actual_fractions["Test"]
            -
            TARGET_FRACTIONS["Test"]
        )
        <=
        FRACTION_TOLERANCE,

    "No train-validation block overlap":
        len(
            train_validation_overlap
        ) == 0,

    "No train-test block overlap":
        len(
            train_test_overlap
        ) == 0,

    "No validation-test block overlap":
        len(
            validation_test_overlap
        ) == 0,

    "Saved sample CRS is EPSG:32632":
        saved_epsg == 32632,

    "Saved dataset splits are complete":
        saved_splits
        ==
        [
            "Test",
            "Train",
            "Validation"
        ],

    "Updated sample GeoPackage saved":
        SAMPLE_GPKG_FILE.exists(),

    "Updated sample CSV saved":
        SAMPLE_CSV_FILE.exists(),

    "Updated sample summary saved":
        SAMPLE_SUMMARY_FILE.exists(),

    "Updated block summary saved":
        BLOCK_SUMMARY_FILE.exists(),

    "Updated design register saved":
        DESIGN_FILE.exists(),

    "Updated validation register saved":
        VALIDATION_FILE.exists(),

    "Predictor extraction remains pending":
        validation_df[
            "Predictor_Extraction_Completed"
        ].iloc[0] == False,

    "Model training remains pending":
        validation_df[
            "Model_Training_Completed"
        ].iloc[0] == False
}

print("\nVALIDATION SUMMARY")
print("-" * 120)

for label, passed in checks.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{label}"
    )

print("\nUPDATED OUTPUT FILES")
print("-" * 120)

for path in [
    SAMPLE_GPKG_FILE,
    SAMPLE_CSV_FILE,
    SAMPLE_SUMMARY_FILE,
    BLOCK_SUMMARY_FILE,
    DESIGN_FILE,
    VALIDATION_FILE
]:

    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{path.name}"
    )

if all(checks.values()):

    print("\n✓ STAGE 9E FIX 2 FULLY PASSED")

    print(
        "The 24,000 balanced samples have been assigned "
        "to optimised and spatially independent training, "
        "validation and testing datasets."
    )

    print(
        "The next stage will extract leakage-safe predictor "
        "values for all sample points."
    )

else:

    print("\n⚠ STAGE 9E FIX 2 REQUIRES REVIEW")

print("=" * 120)

In [ ]:
# ==========================================================================================
# PROJECT 7 — STAGE 9F
# LEAKAGE-SAFE PREDICTOR EXTRACTION FOR THE 24,000 SPATIAL ML SAMPLES
#
# Predictors:
# - Elevation
# - Slope
# - Distance to roads
# - Distance to surface water
# - Distance to drainage
# - Population density
# - Distance to 2020 built-up
# - 2020 baseline land-cover class
#
# Explicitly excluded:
# - 2025 NDVI
# - 2025 NDBI
# - 2025 MNDWI
# - 2025 built-up distance
# - 2025 land-cover class
# - transition or label rasters
#
# Outputs:
# - predictor-enriched GeoPackage
# - predictor-enriched CSV
# - predictor metadata table
# - missing-value diagnostic table
# - split/class predictor summary
# - validation register
# ==========================================================================================

from pathlib import Path
import json
import gc
import re

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio

from rasterio.warp import reproject, Resampling
from scipy import ndimage

# ------------------------------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------------------------------
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

TRAINING_DIR = PROJECT_ROOT / "09_Training_Data"
TABLE_DIR = PROJECT_ROOT / "16_Tables"
ADMIN_DIR = PROJECT_ROOT / "00_Project_Admin"
TERRAIN_DIR = PROJECT_ROOT / "02_Terrain"
LANDCOVER_DIR = PROJECT_ROOT / "04_Land_Cover"

REFERENCE_FILE = (
    TERRAIN_DIR /
    "Enugu_Elevation_30m.tif"
)

DW_2020_FILE = (
    LANDCOVER_DIR /
    "Enugu_DynamicWorld_2020.tif"
)

SAMPLE_GPKG_FILE = (
    TRAINING_DIR /
    "Enugu_ML_Spatial_Samples_2020_2025.gpkg"
)

SAMPLE_CSV_FILE = (
    TRAINING_DIR /
    "Enugu_ML_Spatial_Samples_2020_2025.csv"
)

OUTPUT_GPKG_FILE = (
    TRAINING_DIR /
    "Enugu_ML_Samples_With_Predictors_2020_2025.gpkg"
)

OUTPUT_CSV_FILE = (
    TRAINING_DIR /
    "Enugu_ML_Samples_With_Predictors_2020_2025.csv"
)

PREDICTOR_METADATA_FILE = (
    TABLE_DIR /
    "Enugu_ML_Predictor_Metadata.csv"
)

MISSING_VALUES_FILE = (
    TABLE_DIR /
    "Enugu_ML_Predictor_Missing_Value_QA.csv"
)

PREDICTOR_SUMMARY_FILE = (
    TABLE_DIR /
    "Enugu_ML_Predictor_Split_Class_Summary.csv"
)

VALIDATION_FILE = (
    TRAINING_DIR /
    "Stage_9F_Predictor_Extraction_Validation.csv"
)

DESIGN_FILE = (
    ADMIN_DIR /
    "ML_Leakage_Safe_Predictor_Design.json"
)

for folder in [
    TRAINING_DIR,
    TABLE_DIR,
    ADMIN_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

print("=" * 120)
print("STAGE 9F — LEAKAGE-SAFE PREDICTOR EXTRACTION")
print("=" * 120)

# ------------------------------------------------------------------------------------------
# REQUIRED FILE CHECK
# ------------------------------------------------------------------------------------------
required_files = {
    "Reference elevation raster":
        REFERENCE_FILE,

    "Dynamic World 2020 raster":
        DW_2020_FILE,

    "Spatial ML sample GeoPackage":
        SAMPLE_GPKG_FILE,

    "Spatial ML sample CSV":
        SAMPLE_CSV_FILE
}

print("\nREQUIRED FILE CHECK")
print("-" * 120)

for label, path in required_files.items():

    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{label}: {path}"
    )

if not all(
    path.exists()
    for path in required_files.values()
):
    raise FileNotFoundError(
        "One or more required Stage 9F inputs are missing."
    )

# ------------------------------------------------------------------------------------------
# DISCOVER STATIC OR BASELINE-SAFE RASTERS
# ------------------------------------------------------------------------------------------
all_tifs = sorted(
    PROJECT_ROOT.rglob("*.tif")
)

excluded_terms = [
    "2025_ndvi",
    "ndvi_2025",
    "2025_ndbi",
    "ndbi_2025",
    "2025_mndwi",
    "mndwi_2025",
    "transition",
    "label_eligibility",
    "spatial_reliability",
    "suitability",
    "prediction",
    "predicted",
    "probability"
]

def normalise_name(path):

    return re.sub(
        r"[^a-z0-9]+",
        "_",
        path.stem.lower()
    ).strip("_")


def is_excluded(path):

    name = normalise_name(path)

    return any(
        term in name
        for term in excluded_terms
    )


safe_candidate_tifs = [
    path
    for path in all_tifs
    if not is_excluded(path)
]

# ------------------------------------------------------------------------------------------
# FILE-SELECTION FUNCTION
# ------------------------------------------------------------------------------------------
def find_best_raster(
    include_groups,
    preferred_terms=None,
    reject_terms=None
):
    """
    include_groups:
        Each inner group represents alternative words.
        At least one word from every group must occur.

    Example:
        [["distance", "dist"], ["road", "roads"]]
    """

    preferred_terms = (
        preferred_terms
        if preferred_terms is not None
        else []
    )

    reject_terms = (
        reject_terms
        if reject_terms is not None
        else []
    )

    matches = []

    for path in safe_candidate_tifs:

        name = normalise_name(path)

        if any(
            term in name
            for term in reject_terms
        ):
            continue

        group_match = all(
            any(
                alternative in name
                for alternative in group
            )
            for group in include_groups
        )

        if not group_match:
            continue

        score = 0

        score += sum(
            5
            for term in preferred_terms
            if term in name
        )

        if "30m" in name or "30_m" in name:
            score += 4

        if "aligned" in name:
            score += 3

        if "harmon" in name:
            score += 2

        if "predictor" in str(path.parent).lower():
            score += 2

        matches.append(
            (
                score,
                len(str(path)),
                path
            )
        )

    if not matches:
        return None

    matches.sort(
        key=lambda item: (
            -item[0],
            item[1],
            str(item[2])
        )
    )

    return matches[0][2]

# ------------------------------------------------------------------------------------------
# IDENTIFY PREDICTOR FILES
# ------------------------------------------------------------------------------------------
slope_file = find_best_raster(
    include_groups=[
        ["slope"]
    ],
    preferred_terms=[
        "degrees",
        "30m"
    ]
)

road_distance_file = find_best_raster(
    include_groups=[
        ["distance", "dist"],
        ["road", "roads"]
    ],
    preferred_terms=[
        "30m",
        "euclidean"
    ]
)

water_distance_file = find_best_raster(
    include_groups=[
        ["distance", "dist"],
        ["water"]
    ],
    preferred_terms=[
        "surface",
        "30m"
    ],
    reject_terms=[
        "drain",
        "river"
    ]
)

drainage_distance_file = find_best_raster(
    include_groups=[
        ["distance", "dist"],
        ["drain", "drainage", "river", "stream"]
    ],
    preferred_terms=[
        "merit",
        "30m"
    ]
)

population_file = find_best_raster(
    include_groups=[
        ["population", "worldpop"]
    ],
    preferred_terms=[
        "density",
        "30m"
    ],
    reject_terms=[
        "distance"
    ]
)

discovered_predictors = {
    "Elevation_m":
        REFERENCE_FILE,

    "Slope_deg":
        slope_file,

    "Distance_to_Road_m":
        road_distance_file,

    "Distance_to_Surface_Water_m":
        water_distance_file,

    "Distance_to_Drainage_m":
        drainage_distance_file,

    "Population_Density":
        population_file
}

print("\nDISCOVERED LEAKAGE-SAFE RASTERS")
print("-" * 120)

for predictor, path in discovered_predictors.items():

    print(
        f"{'✓' if path is not None and path.exists() else '✗'} "
        f"{predictor}: {path}"
    )

missing_predictor_files = [
    predictor
    for predictor, path in discovered_predictors.items()
    if path is None or not path.exists()
]

if missing_predictor_files:

    print("\nAVAILABLE CANDIDATE RASTERS")
    print("-" * 120)

    for path in safe_candidate_tifs:
        print(path)

    raise FileNotFoundError(
        "The following required static predictor rasters "
        f"could not be identified: {missing_predictor_files}"
    )

# ------------------------------------------------------------------------------------------
# LOAD REFERENCE GRID
# ------------------------------------------------------------------------------------------
with rasterio.open(
    REFERENCE_FILE
) as ref:

    elevation = ref.read(1).astype(
        np.float32
    )

    ref_profile = ref.profile.copy()
    ref_transform = ref.transform
    ref_crs = ref.crs
    ref_width = ref.width
    ref_height = ref.height
    ref_bounds = ref.bounds

    ref_res_x = abs(
        ref.transform.a
    )

    ref_res_y = abs(
        ref.transform.e
    )

    ref_nodata = ref.nodata

if ref_nodata is not None:

    elevation[
        elevation == ref_nodata
    ] = np.nan

study_mask = np.isfinite(
    elevation
)

print("\nREFERENCE GRID")
print("-" * 120)

print("CRS:", ref_crs)

print(
    f"Dimensions: "
    f"{ref_width} × {ref_height}"
)

print(
    f"Resolution: "
    f"{ref_res_x:.1f} × {ref_res_y:.1f} m"
)

# ------------------------------------------------------------------------------------------
# LOAD SAMPLE POINTS
# ------------------------------------------------------------------------------------------
sample_gdf = gpd.read_file(
    SAMPLE_GPKG_FILE,
    layer="ml_samples"
)

if sample_gdf.crs != ref_crs:

    sample_gdf = sample_gdf.to_crs(
        ref_crs
    )

required_sample_columns = {
    "Sample_ID",
    "Target",
    "Target_Name",
    "Dataset_Split",
    "Raster_Row",
    "Raster_Column",
    "Block_ID"
}

missing_sample_columns = (
    required_sample_columns -
    set(sample_gdf.columns)
)

if missing_sample_columns:

    raise RuntimeError(
        "The following sample fields are missing: "
        f"{sorted(missing_sample_columns)}"
    )

sample_gdf["Raster_Row"] = (
    sample_gdf["Raster_Row"]
    .astype(np.int32)
)

sample_gdf["Raster_Column"] = (
    sample_gdf["Raster_Column"]
    .astype(np.int32)
)

sample_rows = sample_gdf[
    "Raster_Row"
].to_numpy()

sample_cols = sample_gdf[
    "Raster_Column"
].to_numpy()

sample_count = int(
    sample_gdf.shape[0]
)

print("\nSAMPLE DATASET")
print("-" * 120)

print(
    f"Samples loaded: {sample_count:,}"
)

print(
    "Dataset splits:",
    sample_gdf[
        "Dataset_Split"
    ].value_counts().to_dict()
)

print(
    "Target classes:",
    sample_gdf[
        "Target"
    ].value_counts().to_dict()
)

if sample_count != 24_000:

    raise RuntimeError(
        "Expected 24,000 samples."
    )

# ------------------------------------------------------------------------------------------
# GENERIC RASTER ALIGNMENT FUNCTION
# ------------------------------------------------------------------------------------------
def load_aligned_continuous_raster(path):

    with rasterio.open(path) as src:

        source_array = src.read(1).astype(
            np.float32
        )

        source_nodata = src.nodata

        if source_nodata is not None:

            source_array[
                source_array == source_nodata
            ] = np.nan

        already_aligned = (
            src.width == ref_width
            and
            src.height == ref_height
            and
            src.crs == ref_crs
            and
            np.allclose(
                tuple(src.transform),
                tuple(ref_transform)
            )
        )

        if already_aligned:

            aligned = source_array

        else:

            aligned = np.full(
                (
                    ref_height,
                    ref_width
                ),
                np.nan,
                dtype=np.float32
            )

            reproject(
                source=source_array,
                destination=aligned,
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=ref_transform,
                dst_crs=ref_crs,
                src_nodata=np.nan,
                dst_nodata=np.nan,
                resampling=Resampling.bilinear
            )

    aligned[
        ~study_mask
    ] = np.nan

    return aligned


def extract_at_samples(array):

    return array[
        sample_rows,
        sample_cols
    ].astype(np.float32)

# ------------------------------------------------------------------------------------------
# EXTRACT STATIC CONTINUOUS PREDICTORS
# ------------------------------------------------------------------------------------------
predictor_metadata_records = []

continuous_arrays = {}

for predictor_name, predictor_path in discovered_predictors.items():

    print(
        f"\nLoading {predictor_name}..."
    )

    if predictor_name == "Elevation_m":

        aligned_array = elevation.copy()

    else:

        aligned_array = load_aligned_continuous_raster(
            predictor_path
        )

    continuous_arrays[
        predictor_name
    ] = aligned_array

    extracted_values = extract_at_samples(
        aligned_array
    )

    sample_gdf[
        predictor_name
    ] = extracted_values

    valid_values = extracted_values[
        np.isfinite(extracted_values)
    ]

    predictor_metadata_records.append({
        "Predictor":
            predictor_name,

        "Predictor_Type":
            "Continuous",

        "Temporal_Status":
            (
                "Baseline/static"
                if predictor_name != "Population_Density"
                else "Reference population surface"
            ),

        "Source_File":
            str(predictor_path),

        "Resampling_Method":
            (
                "Native reference grid"
                if predictor_name == "Elevation_m"
                else "Bilinear where alignment required"
            ),

        "Sample_Valid_Count":
            int(valid_values.size),

        "Sample_Minimum":
            (
                float(np.min(valid_values))
                if valid_values.size > 0
                else np.nan
            ),

        "Sample_Maximum":
            (
                float(np.max(valid_values))
                if valid_values.size > 0
                else np.nan
            ),

        "Sample_Mean":
            (
                float(np.mean(valid_values))
                if valid_values.size > 0
                else np.nan
            )
    })

# ------------------------------------------------------------------------------------------
# ALIGN DYNAMIC WORLD 2020
# ------------------------------------------------------------------------------------------
with rasterio.open(
    DW_2020_FILE
) as src:

    lulc_2020_10m = src.read(1)

    obs_2020_10m = src.read(2).astype(
        np.float32
    )

    dw_transform = src.transform
    dw_crs = src.crs

lulc_2020_30m = np.full(
    (
        ref_height,
        ref_width
    ),
    255,
    dtype=np.uint8
)

obs_2020_30m = np.full(
    (
        ref_height,
        ref_width
    ),
    np.nan,
    dtype=np.float32
)

reproject(
    source=lulc_2020_10m,
    destination=lulc_2020_30m,
    src_transform=dw_transform,
    src_crs=dw_crs,
    dst_transform=ref_transform,
    dst_crs=ref_crs,
    src_nodata=None,
    dst_nodata=255,
    resampling=Resampling.nearest
)

reproject(
    source=obs_2020_10m,
    destination=obs_2020_30m,
    src_transform=dw_transform,
    src_crs=dw_crs,
    dst_transform=ref_transform,
    dst_crs=ref_crs,
    src_nodata=None,
    dst_nodata=np.nan,
    resampling=Resampling.average
)

del lulc_2020_10m
del obs_2020_10m

gc.collect()

# ------------------------------------------------------------------------------------------
# CREATE DISTANCE TO 2020 BUILT-UP
# ------------------------------------------------------------------------------------------
built_2020 = (
    lulc_2020_30m == 6
)

distance_to_built_2020_m = (
    ndimage.distance_transform_edt(
        ~built_2020
    )
    *
    ref_res_x
).astype(np.float32)

distance_to_built_2020_m[
    ~study_mask
] = np.nan

sample_gdf[
    "Distance_to_Built_2020_m"
] = extract_at_samples(
    distance_to_built_2020_m
)

sample_gdf[
    "Baseline_LULC_2020"
] = (
    lulc_2020_30m[
        sample_rows,
        sample_cols
    ]
    .astype(np.int16)
)

sample_gdf[
    "DW_Observations_2020"
] = (
    obs_2020_30m[
        sample_rows,
        sample_cols
    ]
    .astype(np.float32)
)

distance_values = sample_gdf[
    "Distance_to_Built_2020_m"
].to_numpy()

predictor_metadata_records.append({
    "Predictor":
        "Distance_to_Built_2020_m",

    "Predictor_Type":
        "Continuous",

    "Temporal_Status":
        "2020 baseline",

    "Source_File":
        str(DW_2020_FILE),

    "Resampling_Method":
        "Nearest land-cover alignment followed by Euclidean distance",

    "Sample_Valid_Count":
        int(
            np.isfinite(
                distance_values
            ).sum()
        ),

    "Sample_Minimum":
        float(
            np.nanmin(
                distance_values
            )
        ),

    "Sample_Maximum":
        float(
            np.nanmax(
                distance_values
            )
        ),

    "Sample_Mean":
        float(
            np.nanmean(
                distance_values
            )
        )
})

predictor_metadata_records.append({
    "Predictor":
        "Baseline_LULC_2020",

    "Predictor_Type":
        "Categorical",

    "Temporal_Status":
        "2020 baseline",

    "Source_File":
        str(DW_2020_FILE),

    "Resampling_Method":
        "Nearest neighbour",

    "Sample_Valid_Count":
        int(
            np.count_nonzero(
                sample_gdf[
                    "Baseline_LULC_2020"
                ].to_numpy() != 255
            )
        ),

    "Sample_Minimum":
        int(
            sample_gdf[
                "Baseline_LULC_2020"
            ].min()
        ),

    "Sample_Maximum":
        int(
            sample_gdf[
                "Baseline_LULC_2020"
            ].max()
        ),

    "Sample_Mean":
        float(
            sample_gdf[
                "Baseline_LULC_2020"
            ].mean()
        )
})

# ------------------------------------------------------------------------------------------
# DEFINE FINAL PREDICTOR SET
# ------------------------------------------------------------------------------------------
continuous_predictors = [
    "Elevation_m",
    "Slope_deg",
    "Distance_to_Road_m",
    "Distance_to_Surface_Water_m",
    "Distance_to_Drainage_m",
    "Population_Density",
    "Distance_to_Built_2020_m"
]

categorical_predictors = [
    "Baseline_LULC_2020"
]

final_predictors = (
    continuous_predictors
    +
    categorical_predictors
)

explicitly_excluded_predictors = [
    "NDVI_2025",
    "NDBI_2025",
    "MNDWI_2025",
    "Distance_to_Built_2025",
    "Dynamic_World_2025",
    "BuiltUp_Transition_2020_2025",
    "ML_Label_Eligibility"
]

# ------------------------------------------------------------------------------------------
# CHECK FOR INVALID VALUES
# ------------------------------------------------------------------------------------------
missing_records = []

for predictor in final_predictors:

    values = sample_gdf[
        predictor
    ].to_numpy()

    if predictor in categorical_predictors:

        missing_mask = (
            pd.isna(values)
            |
            (values == 255)
        )

    else:

        missing_mask = ~np.isfinite(
            values.astype(np.float64)
        )

    missing_count = int(
        missing_mask.sum()
    )

    missing_records.append({
        "Predictor":
            predictor,

        "Missing_Count":
            missing_count,

        "Missing_Percent":
            missing_count /
            sample_count *
            100
    })

missing_df = pd.DataFrame(
    missing_records
)

print("\nPREDICTOR MISSING-VALUE QA")
print("-" * 120)

print(
    missing_df.to_string(
        index=False,
        formatters={
            "Missing_Percent":
                lambda value: f"{value:.4f}%"
        }
    )
)

# ------------------------------------------------------------------------------------------
# COMPLETE-CASE STATUS
# ------------------------------------------------------------------------------------------
continuous_complete = np.ones(
    sample_count,
    dtype=bool
)

for predictor in continuous_predictors:

    continuous_complete &= np.isfinite(
        sample_gdf[
            predictor
        ].to_numpy(
            dtype=np.float64
        )
    )

categorical_complete = (
    sample_gdf[
        "Baseline_LULC_2020"
    ].notna().to_numpy()
    &
    (
        sample_gdf[
            "Baseline_LULC_2020"
        ].to_numpy() != 255
    )
)

complete_case_mask = (
    continuous_complete
    &
    categorical_complete
)

sample_gdf[
    "Predictor_Complete"
] = complete_case_mask

complete_case_count = int(
    complete_case_mask.sum()
)

incomplete_case_count = int(
    (~complete_case_mask).sum()
)

print("\nCOMPLETE-CASE RESULTS")
print("-" * 120)

print(
    f"Complete predictor records: "
    f"{complete_case_count:,}"
)

print(
    f"Incomplete predictor records: "
    f"{incomplete_case_count:,}"
)

print(
    f"Complete-record rate: "
    f"{complete_case_count / sample_count * 100:.3f}%"
)

# ------------------------------------------------------------------------------------------
# SPLIT AND CLASS QA
# ------------------------------------------------------------------------------------------
split_class_count_df = (
    sample_gdf
    .groupby(
        [
            "Dataset_Split",
            "Target"
        ]
    )
    .agg(
        Total_Records=(
            "Sample_ID",
            "count"
        ),

        Complete_Records=(
            "Predictor_Complete",
            "sum"
        )
    )
    .reset_index()
)

split_class_count_df[
    "Incomplete_Records"
] = (
    split_class_count_df[
        "Total_Records"
    ]
    -
    split_class_count_df[
        "Complete_Records"
    ]
)

print("\nCOMPLETE RECORDS BY SPLIT AND CLASS")
print("-" * 120)

print(
    split_class_count_df.to_string(
        index=False
    )
)

# ------------------------------------------------------------------------------------------
# PREDICTOR SUMMARY BY SPLIT AND TARGET
# ------------------------------------------------------------------------------------------
summary_records = []

for split_name in [
    "Train",
    "Validation",
    "Test"
]:

    for target_value in [
        0,
        1
    ]:

        group_mask = (
            (
                sample_gdf[
                    "Dataset_Split"
                ] == split_name
            )
            &
            (
                sample_gdf[
                    "Target"
                ] == target_value
            )
            &
            sample_gdf[
                "Predictor_Complete"
            ]
        )

        group = sample_gdf.loc[
            group_mask
        ]

        for predictor in continuous_predictors:

            values = group[
                predictor
            ].to_numpy(
                dtype=np.float64
            )

            summary_records.append({
                "Dataset_Split":
                    split_name,

                "Target":
                    target_value,

                "Target_Name":
                    (
                        "Urban expansion"
                        if target_value == 1
                        else "Stable non-built"
                    ),

                "Predictor":
                    predictor,

                "Record_Count":
                    int(values.size),

                "Mean":
                    (
                        float(np.mean(values))
                        if values.size > 0
                        else np.nan
                    ),

                "Standard_Deviation":
                    (
                        float(np.std(values))
                        if values.size > 0
                        else np.nan
                    ),

                "Minimum":
                    (
                        float(np.min(values))
                        if values.size > 0
                        else np.nan
                    ),

                "Median":
                    (
                        float(np.median(values))
                        if values.size > 0
                        else np.nan
                    ),

                "Maximum":
                    (
                        float(np.max(values))
                        if values.size > 0
                        else np.nan
                    )
            })

predictor_summary_df = pd.DataFrame(
    summary_records
)

# ------------------------------------------------------------------------------------------
# SAVE OUTPUTS
# ------------------------------------------------------------------------------------------
predictor_metadata_df = pd.DataFrame(
    predictor_metadata_records
)

predictor_metadata_df.to_csv(
    PREDICTOR_METADATA_FILE,
    index=False
)

missing_df.to_csv(
    MISSING_VALUES_FILE,
    index=False
)

predictor_summary_df.to_csv(
    PREDICTOR_SUMMARY_FILE,
    index=False
)

output_df = pd.DataFrame(
    sample_gdf.drop(
        columns="geometry"
    )
)

output_df.to_csv(
    OUTPUT_CSV_FILE,
    index=False
)

if OUTPUT_GPKG_FILE.exists():

    OUTPUT_GPKG_FILE.unlink()

sample_gdf.to_file(
    OUTPUT_GPKG_FILE,
    layer="ml_samples_with_predictors",
    driver="GPKG"
)

# ------------------------------------------------------------------------------------------
# SAVE PREDICTOR DESIGN REGISTER
# ------------------------------------------------------------------------------------------
predictor_design = {
    "prediction_target":
        "Observed urban expansion from 2020 to 2025",

    "sample_count":
        sample_count,

    "continuous_predictors":
        continuous_predictors,

    "categorical_predictors":
        categorical_predictors,

    "explicitly_excluded_predictors":
        explicitly_excluded_predictors,

    "leakage_control":
        "Only baseline-2020 or effectively static predictors retained",

    "population_predictor_note":
        "Population density retained as an urban-demand reference surface; "
        "its source year must be reported in the methodology",

    "categorical_encoding_status":
        "Pending one-hot encoding inside modelling pipeline",

    "missing_value_strategy":
        "No imputation performed at extraction stage",

    "complete_case_count":
        complete_case_count,

    "incomplete_case_count":
        incomplete_case_count,

    "model_training_status":
        "Pending"
}

with open(
    DESIGN_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        predictor_design,
        file,
        indent=4
    )

# ------------------------------------------------------------------------------------------
# REOPEN OUTPUT GEOPACKAGE
# ------------------------------------------------------------------------------------------
saved_gdf = gpd.read_file(
    OUTPUT_GPKG_FILE,
    layer="ml_samples_with_predictors"
)

saved_count = int(
    saved_gdf.shape[0]
)

saved_epsg = (
    saved_gdf.crs.to_epsg()
    if saved_gdf.crs is not None
    else None
)

saved_columns = set(
    saved_gdf.columns
)

saved_split_counts = (
    saved_gdf[
        "Dataset_Split"
    ]
    .value_counts()
    .to_dict()
)

saved_target_counts = (
    saved_gdf[
        "Target"
    ]
    .value_counts()
    .to_dict()
)

# ------------------------------------------------------------------------------------------
# VALIDATION REGISTER
# ------------------------------------------------------------------------------------------
all_predictors_present = all(
    predictor in saved_columns
    for predictor in final_predictors
)

no_explicit_leakage_columns = all(
    predictor not in saved_columns
    for predictor in explicitly_excluded_predictors
)

maximum_missing_percent = float(
    missing_df[
        "Missing_Percent"
    ].max()
)

validation_df = pd.DataFrame([{
    "Total_Samples":
        saved_count,

    "Continuous_Predictor_Count":
        len(continuous_predictors),

    "Categorical_Predictor_Count":
        len(categorical_predictors),

    "Total_Predictor_Count":
        len(final_predictors),

    "Complete_Case_Count":
        complete_case_count,

    "Incomplete_Case_Count":
        incomplete_case_count,

    "Complete_Case_Percent":
        complete_case_count /
        sample_count *
        100,

    "Maximum_Predictor_Missing_Percent":
        maximum_missing_percent,

    "Train_Samples":
        int(
            saved_split_counts.get(
                "Train",
                0
            )
        ),

    "Validation_Samples":
        int(
            saved_split_counts.get(
                "Validation",
                0
            )
        ),

    "Test_Samples":
        int(
            saved_split_counts.get(
                "Test",
                0
            )
        ),

    "Negative_Samples":
        int(
            saved_target_counts.get(
                0,
                0
            )
        ),

    "Positive_Samples":
        int(
            saved_target_counts.get(
                1,
                0
            )
        ),

    "Output_EPSG":
        saved_epsg,

    "All_Predictors_Present":
        all_predictors_present,

    "Explicit_Leakage_Columns_Absent":
        no_explicit_leakage_columns,

    "Predictor_Extraction_Completed":
        True,

    "Model_Training_Completed":
        False
}])

validation_df.to_csv(
    VALIDATION_FILE,
    index=False
)

# ------------------------------------------------------------------------------------------
# FINAL VALIDATION
# ------------------------------------------------------------------------------------------
checks = {
    "All 24,000 samples preserved":
        saved_count == 24_000,

    "Training sample count preserved":
        saved_split_counts.get(
            "Train",
            0
        ) == 16_800,

    "Validation sample count preserved":
        saved_split_counts.get(
            "Validation",
            0
        ) == 3_600,

    "Test sample count preserved":
        saved_split_counts.get(
            "Test",
            0
        ) == 3_600,

    "All 12,000 negative samples preserved":
        saved_target_counts.get(
            0,
            0
        ) == 12_000,

    "All 12,000 positive samples preserved":
        saved_target_counts.get(
            1,
            0
        ) == 12_000,

    "All final predictor fields present":
        all_predictors_present,

    "Explicit leakage predictor fields absent":
        no_explicit_leakage_columns,

    "At least 99% complete predictor records":
        (
            complete_case_count /
            sample_count
        ) >= 0.99,

    "Maximum predictor missingness below 1%":
        maximum_missing_percent <= 1.0,

    "Saved output CRS is EPSG:32632":
        saved_epsg == 32632,

    "Predictor GeoPackage saved":
        OUTPUT_GPKG_FILE.exists(),

    "Predictor CSV saved":
        OUTPUT_CSV_FILE.exists(),

    "Predictor metadata saved":
        PREDICTOR_METADATA_FILE.exists(),

    "Missing-value QA saved":
        MISSING_VALUES_FILE.exists(),

    "Predictor split/class summary saved":
        PREDICTOR_SUMMARY_FILE.exists(),

    "Predictor design register saved":
        DESIGN_FILE.exists(),

    "Validation register saved":
        VALIDATION_FILE.exists(),

    "Predictor extraction marked complete":
        validation_df[
            "Predictor_Extraction_Completed"
        ].iloc[0] == True,

    "Model training remains pending":
        validation_df[
            "Model_Training_Completed"
        ].iloc[0] == False
}

print("\nVALIDATION SUMMARY")
print("-" * 120)

for label, passed in checks.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{label}"
    )

print("\nOUTPUT FILES")
print("-" * 120)

for path in [
    OUTPUT_GPKG_FILE,
    OUTPUT_CSV_FILE,
    PREDICTOR_METADATA_FILE,
    MISSING_VALUES_FILE,
    PREDICTOR_SUMMARY_FILE,
    DESIGN_FILE,
    VALIDATION_FILE
]:

    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{path.name}"
    )

if all(
    checks.values()
):

    print("\n✓ STAGE 9F FULLY PASSED")

    print(
        "Leakage-safe predictor values have been extracted "
        "for all spatial ML samples."
    )

    print(
        "The next stage will prepare the modelling matrix, "
        "encode baseline land cover and train candidate models."
    )

else:

    print("\n⚠ STAGE 9F REQUIRES REVIEW")

print("=" * 120)

In [ ]:
# ==========================================================================================
# PROJECT 7 — STAGE 10A
# CANDIDATE MODEL TRAINING + VALIDATION-BASED MODEL SELECTION
#
# Modelling rules:
# - Train split used for model fitting only.
# - Validation split used for candidate-model comparison only.
# - Test split remains completely untouched.
# - Baseline land cover is one-hot encoded.
# - Continuous variables are median-imputed and standardised.
# - Candidate models:
#     1. Logistic Regression
#     2. Random Forest
#     3. Extra Trees
#     4. Histogram Gradient Boosting
#
# Primary model-selection metric:
# - Validation ROC-AUC
#
# Tie-breakers:
# - Validation PR-AUC
# - Validation balanced accuracy
#
# The selected model will be tested only in the next stage.
# ==========================================================================================

from pathlib import Path
import json
import warnings
import platform
import sys

import numpy as np
import pandas as pd

import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    confusion_matrix,
    brier_score_loss,
    log_loss
)

import joblib

warnings.filterwarnings(
    "ignore",
    category=FutureWarning
)

# ------------------------------------------------------------------------------------------
# PARAMETERS
# ------------------------------------------------------------------------------------------
RANDOM_SEED = 42
CLASSIFICATION_THRESHOLD = 0.50

# Used only to identify effectively tied validation ROC-AUC values.
ROC_AUC_TIE_TOLERANCE = 0.001

# ------------------------------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------------------------------
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

TRAINING_DIR = (
    PROJECT_ROOT /
    "09_Training_Data"
)

MODEL_DIR = (
    PROJECT_ROOT /
    "11_ML_Models"
)

TABLE_DIR = (
    PROJECT_ROOT /
    "16_Tables"
)

ADMIN_DIR = (
    PROJECT_ROOT /
    "00_Project_Admin"
)

INPUT_CSV_FILE = (
    TRAINING_DIR /
    "Enugu_ML_Samples_With_Predictors_2020_2025.csv"
)

MODEL_LEADERBOARD_FILE = (
    TABLE_DIR /
    "Enugu_Candidate_Model_Validation_Leaderboard.csv"
)

MODEL_METRICS_FILE = (
    TABLE_DIR /
    "Enugu_Candidate_Model_Train_Validation_Metrics.csv"
)

CONFUSION_MATRIX_FILE = (
    TABLE_DIR /
    "Enugu_Candidate_Model_Validation_Confusion_Matrices.csv"
)

VALIDATION_PREDICTIONS_FILE = (
    TRAINING_DIR /
    "Enugu_Candidate_Model_Validation_Predictions.csv"
)

TRAINING_PREDICTIONS_FILE = (
    TRAINING_DIR /
    "Enugu_Candidate_Model_Training_Predictions.csv"
)

BEST_MODEL_FILE = (
    MODEL_DIR /
    "Enugu_Selected_Expansion_Model_Validation_Stage.joblib"
)

MODEL_SELECTION_FILE = (
    ADMIN_DIR /
    "ML_Candidate_Model_Selection.json"
)

VALIDATION_FILE = (
    TRAINING_DIR /
    "Stage_10A_Candidate_Model_Validation.csv"
)

for folder in [
    TRAINING_DIR,
    MODEL_DIR,
    TABLE_DIR,
    ADMIN_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

print("=" * 120)
print("STAGE 10A — CANDIDATE MODEL TRAINING + VALIDATION-BASED SELECTION")
print("=" * 120)

# ------------------------------------------------------------------------------------------
# REQUIRED FILE CHECK
# ------------------------------------------------------------------------------------------
print("\nREQUIRED FILE CHECK")
print("-" * 120)

print(
    f"{'✓' if INPUT_CSV_FILE.exists() else '✗'} "
    f"Predictor-enriched sample table: {INPUT_CSV_FILE}"
)

if not INPUT_CSV_FILE.exists():

    raise FileNotFoundError(
        "The Stage 9F predictor-enriched sample table is missing."
    )

# ------------------------------------------------------------------------------------------
# LOAD DATASET
# ------------------------------------------------------------------------------------------
data = pd.read_csv(
    INPUT_CSV_FILE
)

continuous_predictors = [
    "Elevation_m",
    "Slope_deg",
    "Distance_to_Road_m",
    "Distance_to_Surface_Water_m",
    "Distance_to_Drainage_m",
    "Population_Density",
    "Distance_to_Built_2020_m"
]

categorical_predictors = [
    "Baseline_LULC_2020"
]

predictor_columns = (
    continuous_predictors
    +
    categorical_predictors
)

required_columns = {
    "Sample_ID",
    "Target",
    "Target_Name",
    "Dataset_Split",
    "Block_ID",
    "Predictor_Complete",
    *predictor_columns
}

missing_columns = (
    required_columns -
    set(data.columns)
)

if missing_columns:

    raise RuntimeError(
        "Required modelling columns are missing: "
        f"{sorted(missing_columns)}"
    )

# Ensure expected data types.
data["Target"] = (
    data["Target"]
    .astype(np.uint8)
)

data["Baseline_LULC_2020"] = (
    data["Baseline_LULC_2020"]
    .astype(str)
)

# Robustly interpret Predictor_Complete if loaded from CSV as text.
if data["Predictor_Complete"].dtype == object:

    data["Predictor_Complete"] = (
        data["Predictor_Complete"]
        .astype(str)
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False
        })
    )

if data["Predictor_Complete"].isna().any():

    raise RuntimeError(
        "Predictor_Complete contains unrecognised values."
    )

# ------------------------------------------------------------------------------------------
# DATASET QA
# ------------------------------------------------------------------------------------------
print("\nMODELLING DATASET")
print("-" * 120)

print(
    f"Total records: {data.shape[0]:,}"
)

print(
    "Dataset splits:",
    data[
        "Dataset_Split"
    ].value_counts().to_dict()
)

print(
    "Target classes:",
    data[
        "Target"
    ].value_counts().sort_index().to_dict()
)

print(
    f"Predictor-complete records: "
    f"{int(data['Predictor_Complete'].sum()):,}"
)

expected_split_counts = {
    "Train": 16_800,
    "Validation": 3_600,
    "Test": 3_600
}

actual_split_counts = (
    data[
        "Dataset_Split"
    ].value_counts().to_dict()
)

if data.shape[0] != 24_000:

    raise RuntimeError(
        "Expected 24,000 modelling records."
    )

if actual_split_counts != expected_split_counts:

    raise RuntimeError(
        "The train-validation-test sample counts have changed."
    )

if not data["Predictor_Complete"].all():

    raise RuntimeError(
        "Incomplete predictor records remain in the modelling dataset."
    )

if sorted(
    data["Target"].unique().tolist()
) != [0, 1]:

    raise RuntimeError(
        "Target values must contain only classes 0 and 1."
    )

# ------------------------------------------------------------------------------------------
# LEAKAGE COLUMN AUDIT
# ------------------------------------------------------------------------------------------
forbidden_name_fragments = [
    "ndvi_2025",
    "2025_ndvi",
    "ndbi_2025",
    "2025_ndbi",
    "mndwi_2025",
    "2025_mndwi",
    "distance_to_built_2025",
    "transition",
    "label_eligibility",
    "spatial_reliability",
    "dynamic_world_2025",
    "lulc_2025"
]

predictor_names_lower = [
    predictor.lower()
    for predictor in predictor_columns
]

detected_forbidden_predictors = []

for predictor_name in predictor_names_lower:

    for fragment in forbidden_name_fragments:

        if fragment in predictor_name:

            detected_forbidden_predictors.append(
                predictor_name
            )

if detected_forbidden_predictors:

    raise RuntimeError(
        "Forbidden leakage predictors detected: "
        f"{sorted(set(detected_forbidden_predictors))}"
    )

print("\nLEAKAGE AUDIT")
print("-" * 120)

print(
    "✓ No 2025, transition-derived or label-derived "
    "predictors are included."
)

# ------------------------------------------------------------------------------------------
# CREATE TRAIN, VALIDATION AND RESERVED TEST DATASETS
# ------------------------------------------------------------------------------------------
train_df = data.loc[
    data[
        "Dataset_Split"
    ] == "Train"
].copy()

validation_df = data.loc[
    data[
        "Dataset_Split"
    ] == "Validation"
].copy()

test_df = data.loc[
    data[
        "Dataset_Split"
    ] == "Test"
].copy()

X_train = train_df[
    predictor_columns
].copy()

y_train = train_df[
    "Target"
].to_numpy(
    dtype=np.uint8
)

X_validation = validation_df[
    predictor_columns
].copy()

y_validation = validation_df[
    "Target"
].to_numpy(
    dtype=np.uint8
)

# Deliberately do not create X_test or y_test here.
# Test records are counted only for validation of reservation status.

print("\nDATASET PARTITIONS")
print("-" * 120)

print(
    f"Training records: {train_df.shape[0]:,}"
)

print(
    f"Validation records: {validation_df.shape[0]:,}"
)

print(
    f"Reserved test records: {test_df.shape[0]:,}"
)

print(
    "✓ Test predictor matrix was not created."
)

print(
    "✓ Test labels were not evaluated."
)

# ------------------------------------------------------------------------------------------
# PREPROCESSOR
# ------------------------------------------------------------------------------------------
continuous_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

# Compatibility across scikit-learn versions.
try:

    categorical_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )

except TypeError:

    categorical_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False
    )

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            categorical_encoder
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "continuous",
            continuous_transformer,
            continuous_predictors
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_predictors
        )
    ],
    remainder="drop"
)

# ------------------------------------------------------------------------------------------
# CANDIDATE MODELS
# ------------------------------------------------------------------------------------------
candidate_estimators = {
    "Logistic Regression":
        LogisticRegression(
            penalty="l2",
            C=1.0,
            solver="lbfgs",
            max_iter=2_000,
            random_state=RANDOM_SEED
        ),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_split=6,
            min_samples_leaf=3,
            max_features="sqrt",
            bootstrap=True,
            n_jobs=-1,
            random_state=RANDOM_SEED
        ),

    "Extra Trees":
        ExtraTreesClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_split=6,
            min_samples_leaf=3,
            max_features="sqrt",
            bootstrap=False,
            n_jobs=-1,
            random_state=RANDOM_SEED
        ),

    "Histogram Gradient Boosting":
        HistGradientBoostingClassifier(
            learning_rate=0.06,
            max_iter=350,
            max_leaf_nodes=31,
            max_depth=None,
            min_samples_leaf=20,
            l2_regularization=1.0,
            early_stopping=True,
            validation_fraction=0.10,
            n_iter_no_change=30,
            random_state=RANDOM_SEED
        )
}

# ------------------------------------------------------------------------------------------
# METRIC FUNCTION
# ------------------------------------------------------------------------------------------
def calculate_binary_metrics(
    y_true,
    probability,
    threshold=0.50
):

    prediction = (
        probability >= threshold
    ).astype(np.uint8)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        prediction,
        labels=[0, 1]
    ).ravel()

    specificity = (
        tn /
        (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    false_positive_rate = (
        fp /
        (fp + tn)
        if (fp + tn) > 0
        else np.nan
    )

    false_negative_rate = (
        fn /
        (fn + tp)
        if (fn + tp) > 0
        else np.nan
    )

    return {
        "Accuracy":
            accuracy_score(
                y_true,
                prediction
            ),

        "Balanced_Accuracy":
            balanced_accuracy_score(
                y_true,
                prediction
            ),

        "Precision":
            precision_score(
                y_true,
                prediction,
                zero_division=0
            ),

        "Recall_Sensitivity":
            recall_score(
                y_true,
                prediction,
                zero_division=0
            ),

        "Specificity":
            specificity,

        "F1_Score":
            f1_score(
                y_true,
                prediction,
                zero_division=0
            ),

        "ROC_AUC":
            roc_auc_score(
                y_true,
                probability
            ),

        "PR_AUC":
            average_precision_score(
                y_true,
                probability
            ),

        "Matthews_Correlation":
            matthews_corrcoef(
                y_true,
                prediction
            ),

        "Brier_Score":
            brier_score_loss(
                y_true,
                probability
            ),

        "Log_Loss":
            log_loss(
                y_true,
                np.column_stack(
                    [
                        1 - probability,
                        probability
                    ]
                ),
                labels=[0, 1]
            ),

        "True_Negative":
            int(tn),

        "False_Positive":
            int(fp),

        "False_Negative":
            int(fn),

        "True_Positive":
            int(tp),

        "False_Positive_Rate":
            false_positive_rate,

        "False_Negative_Rate":
            false_negative_rate
    }

# ------------------------------------------------------------------------------------------
# TRAIN CANDIDATE MODELS
# ------------------------------------------------------------------------------------------
metric_records = []
confusion_records = []

validation_prediction_output = validation_df[
    [
        "Sample_ID",
        "Target",
        "Dataset_Split",
        "Block_ID",
        "X",
        "Y"
    ]
].copy()

training_prediction_output = train_df[
    [
        "Sample_ID",
        "Target",
        "Dataset_Split",
        "Block_ID",
        "X",
        "Y"
    ]
].copy()

fitted_models = {}

print("\nCANDIDATE MODEL TRAINING")
print("-" * 120)

for model_name, estimator in candidate_estimators.items():

    print(
        f"\nTraining: {model_name}"
    )

    model_pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),
            (
                "classifier",
                estimator
            )
        ]
    )

    model_pipeline.fit(
        X_train,
        y_train
    )

    fitted_models[
        model_name
    ] = model_pipeline

    training_probability = (
        model_pipeline.predict_proba(
            X_train
        )[:, 1]
    )

    validation_probability = (
        model_pipeline.predict_proba(
            X_validation
        )[:, 1]
    )

    training_metrics = calculate_binary_metrics(
        y_train,
        training_probability,
        threshold=CLASSIFICATION_THRESHOLD
    )

    validation_metrics = calculate_binary_metrics(
        y_validation,
        validation_probability,
        threshold=CLASSIFICATION_THRESHOLD
    )

    for dataset_name, metrics in [
        (
            "Train",
            training_metrics
        ),
        (
            "Validation",
            validation_metrics
        )
    ]:

        record = {
            "Model":
                model_name,

            "Dataset":
                dataset_name,

            "Classification_Threshold":
                CLASSIFICATION_THRESHOLD
        }

        record.update(
            metrics
        )

        metric_records.append(
            record
        )

        confusion_records.append({
            "Model":
                model_name,

            "Dataset":
                dataset_name,

            "True_Negative":
                metrics[
                    "True_Negative"
                ],

            "False_Positive":
                metrics[
                    "False_Positive"
                ],

            "False_Negative":
                metrics[
                    "False_Negative"
                ],

            "True_Positive":
                metrics[
                    "True_Positive"
                ]
        })

    safe_model_name = (
        model_name
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    validation_prediction_output[
        f"{safe_model_name}_Probability"
    ] = validation_probability

    validation_prediction_output[
        f"{safe_model_name}_Prediction"
    ] = (
        validation_probability
        >=
        CLASSIFICATION_THRESHOLD
    ).astype(np.uint8)

    training_prediction_output[
        f"{safe_model_name}_Probability"
    ] = training_probability

    training_prediction_output[
        f"{safe_model_name}_Prediction"
    ] = (
        training_probability
        >=
        CLASSIFICATION_THRESHOLD
    ).astype(np.uint8)

    model_output_file = (
        MODEL_DIR /
        f"Enugu_{safe_model_name}_Candidate_Model.joblib"
    )

    joblib.dump(
        model_pipeline,
        model_output_file,
        compress=3
    )

    print(
        f"  Train ROC-AUC: "
        f"{training_metrics['ROC_AUC']:.4f}"
    )

    print(
        f"  Validation ROC-AUC: "
        f"{validation_metrics['ROC_AUC']:.4f}"
    )

    print(
        f"  Validation PR-AUC: "
        f"{validation_metrics['PR_AUC']:.4f}"
    )

    print(
        f"  Validation balanced accuracy: "
        f"{validation_metrics['Balanced_Accuracy']:.4f}"
    )

# ------------------------------------------------------------------------------------------
# CREATE METRICS TABLES
# ------------------------------------------------------------------------------------------
metrics_df = pd.DataFrame(
    metric_records
)

confusion_df = pd.DataFrame(
    confusion_records
)

validation_leaderboard = (
    metrics_df.loc[
        metrics_df[
            "Dataset"
        ] == "Validation"
    ]
    .copy()
)

training_metrics_lookup = (
    metrics_df.loc[
        metrics_df[
            "Dataset"
        ] == "Train",
        [
            "Model",
            "ROC_AUC",
            "PR_AUC",
            "Balanced_Accuracy",
            "F1_Score"
        ]
    ]
    .rename(
        columns={
            "ROC_AUC":
                "Train_ROC_AUC",

            "PR_AUC":
                "Train_PR_AUC",

            "Balanced_Accuracy":
                "Train_Balanced_Accuracy",

            "F1_Score":
                "Train_F1_Score"
        }
    )
)

validation_leaderboard = (
    validation_leaderboard
    .merge(
        training_metrics_lookup,
        on="Model",
        how="left"
    )
)

validation_leaderboard[
    "ROC_AUC_Generalisation_Gap"
] = (
    validation_leaderboard[
        "Train_ROC_AUC"
    ]
    -
    validation_leaderboard[
        "ROC_AUC"
    ]
)

validation_leaderboard[
    "PR_AUC_Generalisation_Gap"
] = (
    validation_leaderboard[
        "Train_PR_AUC"
    ]
    -
    validation_leaderboard[
        "PR_AUC"
    ]
)

# ------------------------------------------------------------------------------------------
# MODEL SELECTION
#
# Identify models within 0.001 of the highest ROC-AUC.
# Among those, select by PR-AUC and then balanced accuracy.
# ------------------------------------------------------------------------------------------
best_validation_roc_auc = float(
    validation_leaderboard[
        "ROC_AUC"
    ].max()
)

validation_leaderboard[
    "Within_ROC_AUC_Tie_Range"
] = (
    validation_leaderboard[
        "ROC_AUC"
    ]
    >=
    (
        best_validation_roc_auc
        -
        ROC_AUC_TIE_TOLERANCE
    )
)

eligible_for_selection = (
    validation_leaderboard.loc[
        validation_leaderboard[
            "Within_ROC_AUC_Tie_Range"
        ]
    ]
    .copy()
)

eligible_for_selection = (
    eligible_for_selection
    .sort_values(
        by=[
            "ROC_AUC",
            "PR_AUC",
            "Balanced_Accuracy",
            "Matthews_Correlation"
        ],
        ascending=False
    )
)

selected_model_name = str(
    eligible_for_selection.iloc[0][
        "Model"
    ]
)

validation_leaderboard[
    "Selected_Model"
] = (
    validation_leaderboard[
        "Model"
    ]
    ==
    selected_model_name
)

validation_leaderboard = (
    validation_leaderboard
    .sort_values(
        by=[
            "Selected_Model",
            "ROC_AUC",
            "PR_AUC",
            "Balanced_Accuracy"
        ],
        ascending=[
            False,
            False,
            False,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)

validation_leaderboard[
    "Validation_Rank"
] = np.arange(
    1,
    validation_leaderboard.shape[0] + 1
)

selected_model = fitted_models[
    selected_model_name
]

joblib.dump(
    selected_model,
    BEST_MODEL_FILE,
    compress=3
)

selected_validation_record = (
    validation_leaderboard.loc[
        validation_leaderboard[
            "Selected_Model"
        ]
    ]
    .iloc[0]
)

print("\nVALIDATION MODEL LEADERBOARD")
print("-" * 120)

leaderboard_display_columns = [
    "Validation_Rank",
    "Model",
    "ROC_AUC",
    "PR_AUC",
    "Balanced_Accuracy",
    "Precision",
    "Recall_Sensitivity",
    "Specificity",
    "F1_Score",
    "Matthews_Correlation",
    "Brier_Score",
    "ROC_AUC_Generalisation_Gap",
    "Selected_Model"
]

print(
    validation_leaderboard[
        leaderboard_display_columns
    ].to_string(
        index=False,
        formatters={
            column:
                lambda value: f"{value:.4f}"
            for column in [
                "ROC_AUC",
                "PR_AUC",
                "Balanced_Accuracy",
                "Precision",
                "Recall_Sensitivity",
                "Specificity",
                "F1_Score",
                "Matthews_Correlation",
                "Brier_Score",
                "ROC_AUC_Generalisation_Gap"
            ]
        }
    )
)

print("\nSELECTED MODEL")
print("-" * 120)

print(
    f"Model: {selected_model_name}"
)

print(
    f"Validation ROC-AUC: "
    f"{selected_validation_record['ROC_AUC']:.4f}"
)

print(
    f"Validation PR-AUC: "
    f"{selected_validation_record['PR_AUC']:.4f}"
)

print(
    f"Validation balanced accuracy: "
    f"{selected_validation_record['Balanced_Accuracy']:.4f}"
)

print(
    f"Validation F1 score: "
    f"{selected_validation_record['F1_Score']:.4f}"
)

print(
    f"ROC-AUC generalisation gap: "
    f"{selected_validation_record['ROC_AUC_Generalisation_Gap']:.4f}"
)

# ------------------------------------------------------------------------------------------
# SAVE TABLE OUTPUTS
# ------------------------------------------------------------------------------------------
metrics_df.to_csv(
    MODEL_METRICS_FILE,
    index=False
)

validation_leaderboard.to_csv(
    MODEL_LEADERBOARD_FILE,
    index=False
)

confusion_df.to_csv(
    CONFUSION_MATRIX_FILE,
    index=False
)

validation_prediction_output.to_csv(
    VALIDATION_PREDICTIONS_FILE,
    index=False
)

training_prediction_output.to_csv(
    TRAINING_PREDICTIONS_FILE,
    index=False
)

# ------------------------------------------------------------------------------------------
# SAVE MODEL-SELECTION REGISTER
# ------------------------------------------------------------------------------------------
model_selection_register = {
    "project":
        "Machine-Learning-Based Land Suitability Analysis "
        "for Sustainable Urban Development — Enugu",

    "prediction_target":
        "Urban expansion from 2020 to 2025",

    "training_sample_count":
        int(train_df.shape[0]),

    "validation_sample_count":
        int(validation_df.shape[0]),

    "reserved_test_sample_count":
        int(test_df.shape[0]),

    "continuous_predictors":
        continuous_predictors,

    "categorical_predictors":
        categorical_predictors,

    "candidate_models":
        list(
            candidate_estimators.keys()
        ),

    "classification_threshold":
        CLASSIFICATION_THRESHOLD,

    "primary_selection_metric":
        "Validation ROC-AUC",

    "secondary_selection_metrics": [
        "Validation PR-AUC",
        "Validation balanced accuracy",
        "Validation Matthews correlation coefficient"
    ],

    "roc_auc_tie_tolerance":
        ROC_AUC_TIE_TOLERANCE,

    "selected_model":
        selected_model_name,

    "selected_validation_metrics": {
        "roc_auc":
            float(
                selected_validation_record[
                    "ROC_AUC"
                ]
            ),

        "pr_auc":
            float(
                selected_validation_record[
                    "PR_AUC"
                ]
            ),

        "balanced_accuracy":
            float(
                selected_validation_record[
                    "Balanced_Accuracy"
                ]
            ),

        "precision":
            float(
                selected_validation_record[
                    "Precision"
                ]
            ),

        "recall":
            float(
                selected_validation_record[
                    "Recall_Sensitivity"
                ]
            ),

        "specificity":
            float(
                selected_validation_record[
                    "Specificity"
                ]
            ),

        "f1_score":
            float(
                selected_validation_record[
                    "F1_Score"
                ]
            ),

        "matthews_correlation":
            float(
                selected_validation_record[
                    "Matthews_Correlation"
                ]
            ),

        "brier_score":
            float(
                selected_validation_record[
                    "Brier_Score"
                ]
            )
    },

    "test_evaluation_status":
        "Not performed",

    "test_dataset_status":
        "Reserved and untouched",

    "software": {
        "python":
            platform.python_version(),

        "scikit_learn":
            sklearn.__version__,

        "numpy":
            np.__version__,

        "pandas":
            pd.__version__
    }
}

with open(
    MODEL_SELECTION_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        model_selection_register,
        file,
        indent=4
    )

# ------------------------------------------------------------------------------------------
# REOPEN SELECTED MODEL
# ------------------------------------------------------------------------------------------
reloaded_selected_model = joblib.load(
    BEST_MODEL_FILE
)

reloaded_validation_probability = (
    reloaded_selected_model.predict_proba(
        X_validation
    )[:, 1]
)

saved_model_probability_match = np.allclose(
    reloaded_validation_probability,
    selected_model.predict_proba(
        X_validation
    )[:, 1],
    atol=1e-10
)

# ------------------------------------------------------------------------------------------
# VALIDATION REGISTER
# ------------------------------------------------------------------------------------------
selected_validation_roc_auc = float(
    selected_validation_record[
        "ROC_AUC"
    ]
)

selected_validation_pr_auc = float(
    selected_validation_record[
        "PR_AUC"
    ]
)

selected_validation_balanced_accuracy = float(
    selected_validation_record[
        "Balanced_Accuracy"
    ]
)

selected_generalisation_gap = float(
    selected_validation_record[
        "ROC_AUC_Generalisation_Gap"
    ]
)

validation_register_df = pd.DataFrame([{
    "Training_Samples":
        int(train_df.shape[0]),

    "Validation_Samples":
        int(validation_df.shape[0]),

    "Reserved_Test_Samples":
        int(test_df.shape[0]),

    "Continuous_Predictors":
        len(continuous_predictors),

    "Categorical_Predictors":
        len(categorical_predictors),

    "Candidate_Model_Count":
        len(candidate_estimators),

    "Selected_Model":
        selected_model_name,

    "Selected_Validation_ROC_AUC":
        selected_validation_roc_auc,

    "Selected_Validation_PR_AUC":
        selected_validation_pr_auc,

    "Selected_Validation_Balanced_Accuracy":
        selected_validation_balanced_accuracy,

    "Selected_ROC_AUC_Generalisation_Gap":
        selected_generalisation_gap,

    "Classification_Threshold":
        CLASSIFICATION_THRESHOLD,

    "Leakage_Predictors_Detected":
        len(
            detected_forbidden_predictors
        ),

    "Saved_Model_Probability_Match":
        saved_model_probability_match,

    "Test_Predictor_Matrix_Created":
        False,

    "Test_Evaluation_Completed":
        False,

    "Model_Selection_Completed":
        True
}])

validation_register_df.to_csv(
    VALIDATION_FILE,
    index=False
)

# ------------------------------------------------------------------------------------------
# FINAL VALIDATION
# ------------------------------------------------------------------------------------------
candidate_model_files = [
    (
        MODEL_DIR /
        (
            "Enugu_"
            +
            model_name
            .lower()
            .replace(" ", "_")
            .replace("-", "_")
            +
            "_Candidate_Model.joblib"
        )
    )
    for model_name in candidate_estimators
]

all_candidate_models_saved = all(
    path.exists()
    for path in candidate_model_files
)

selected_model_name_valid = (
    selected_model_name
    in
    candidate_estimators
)

each_model_has_train_validation_metrics = (
    metrics_df.groupby(
        "Model"
    )[
        "Dataset"
    ].nunique().min()
    ==
    2
)

test_columns_in_prediction_outputs = any(
    column.lower().startswith("test")
    for column in validation_prediction_output.columns
)

checks = {
    "Training sample count is 16,800":
        train_df.shape[0] == 16_800,

    "Validation sample count is 3,600":
        validation_df.shape[0] == 3_600,

    "Reserved test sample count is 3,600":
        test_df.shape[0] == 3_600,

    "Training set contains both classes":
        sorted(
            train_df[
                "Target"
            ].unique().tolist()
        ) == [0, 1],

    "Validation set contains both classes":
        sorted(
            validation_df[
                "Target"
            ].unique().tolist()
        ) == [0, 1],

    "All eight predictors used":
        len(predictor_columns) == 8,

    "No forbidden leakage predictor detected":
        len(
            detected_forbidden_predictors
        ) == 0,

    "Four candidate models trained":
        len(
            fitted_models
        ) == 4,

    "Every model has train and validation metrics":
        each_model_has_train_validation_metrics,

    "Selected model belongs to candidate set":
        selected_model_name_valid,

    "Selected validation ROC-AUC is finite":
        np.isfinite(
            selected_validation_roc_auc
        ),

    "Selected validation PR-AUC is finite":
        np.isfinite(
            selected_validation_pr_auc
        ),

    "Selected validation balanced accuracy is finite":
        np.isfinite(
            selected_validation_balanced_accuracy
        ),

    "All candidate model files saved":
        all_candidate_models_saved,

    "Selected model file saved":
        BEST_MODEL_FILE.exists(),

    "Reloaded model predictions match":
        saved_model_probability_match,

    "Validation leaderboard saved":
        MODEL_LEADERBOARD_FILE.exists(),

    "Train-validation metrics saved":
        MODEL_METRICS_FILE.exists(),

    "Confusion-matrix table saved":
        CONFUSION_MATRIX_FILE.exists(),

    "Validation predictions saved":
        VALIDATION_PREDICTIONS_FILE.exists(),

    "Training predictions saved":
        TRAINING_PREDICTIONS_FILE.exists(),

    "Model-selection register saved":
        MODEL_SELECTION_FILE.exists(),

    "Validation register saved":
        VALIDATION_FILE.exists(),

    "Test predictor matrix remains uncreated":
        validation_register_df[
            "Test_Predictor_Matrix_Created"
        ].iloc[0] == False,

    "Test evaluation remains incomplete":
        validation_register_df[
            "Test_Evaluation_Completed"
        ].iloc[0] == False,

    "No test fields written to prediction outputs":
        not test_columns_in_prediction_outputs,

    "Model selection marked complete":
        validation_register_df[
            "Model_Selection_Completed"
        ].iloc[0] == True
}

print("\nVALIDATION SUMMARY")
print("-" * 120)

for label, passed in checks.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{label}"
    )

print("\nOUTPUT FILES")
print("-" * 120)

for path in [
    *candidate_model_files,
    BEST_MODEL_FILE,
    MODEL_LEADERBOARD_FILE,
    MODEL_METRICS_FILE,
    CONFUSION_MATRIX_FILE,
    VALIDATION_PREDICTIONS_FILE,
    TRAINING_PREDICTIONS_FILE,
    MODEL_SELECTION_FILE,
    VALIDATION_FILE
]:

    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{path.name}"
    )

if all(
    checks.values()
):

    print("\n✓ STAGE 10A FULLY PASSED")

    print(
        f"The selected candidate model is: "
        f"{selected_model_name}."
    )

    print(
        "Candidate comparison used training and validation "
        "data only."
    )

    print(
        "The 3,600-record spatial test dataset remains "
        "reserved for final unbiased evaluation."
    )

else:

    print("\n⚠ STAGE 10A REQUIRES REVIEW")

print("=" * 120)

In [ ]:
# ==========================================================================================
# PROJECT 7 — STAGE 10B
# LABEL-RULE LEAKAGE AUDIT + CORRECTED VALIDATION-BASED MODEL SELECTION
#
# Critical issue:
# - Positive-label eligibility required distance to 2020 built-up <= 90 m.
# - Negative-label eligibility required distance to 2020 built-up >= 90 m.
# - Distance_to_Built_2020_m therefore directly reflects the class-construction rule.
#
# This stage:
# - audits the separation caused by Distance_to_Built_2020_m;
# - excludes that variable from candidate modelling;
# - retains seven leakage-safe predictors;
# - retrains Logistic Regression, Random Forest, Extra Trees and
#   Histogram Gradient Boosting;
# - compares models using training and validation data only;
# - keeps the 3,600-record test set untouched;
# - saves a corrected selected model.
#
# No final test evaluation is performed here.
# ==========================================================================================

from pathlib import Path
import json
import warnings
import platform

import numpy as np
import pandas as pd
import sklearn
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    confusion_matrix,
    brier_score_loss,
    log_loss
)

warnings.filterwarnings(
    "ignore",
    category=FutureWarning
)

# ------------------------------------------------------------------------------------------
# PARAMETERS
# ------------------------------------------------------------------------------------------
RANDOM_SEED = 42
CLASSIFICATION_THRESHOLD = 0.50
ROC_AUC_TIE_TOLERANCE = 0.001

# Distance rule used during Stage 9D label construction.
LABEL_DISTANCE_THRESHOLD_M = 90.0

# ------------------------------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------------------------------
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

TRAINING_DIR = PROJECT_ROOT / "09_Training_Data"
MODEL_DIR = PROJECT_ROOT / "11_ML_Models"
TABLE_DIR = PROJECT_ROOT / "16_Tables"
ADMIN_DIR = PROJECT_ROOT / "00_Project_Admin"

INPUT_CSV_FILE = (
    TRAINING_DIR /
    "Enugu_ML_Samples_With_Predictors_2020_2025.csv"
)

LEAKAGE_AUDIT_FILE = (
    TABLE_DIR /
    "Enugu_Distance_to_Built_Label_Rule_Leakage_Audit.csv"
)

CORRECTED_METRICS_FILE = (
    TABLE_DIR /
    "Enugu_Corrected_Candidate_Model_Train_Validation_Metrics.csv"
)

CORRECTED_LEADERBOARD_FILE = (
    TABLE_DIR /
    "Enugu_Corrected_Candidate_Model_Validation_Leaderboard.csv"
)

CORRECTED_CONFUSION_FILE = (
    TABLE_DIR /
    "Enugu_Corrected_Candidate_Model_Validation_Confusion_Matrices.csv"
)

CORRECTED_VALIDATION_PREDICTIONS_FILE = (
    TRAINING_DIR /
    "Enugu_Corrected_Candidate_Model_Validation_Predictions.csv"
)

CORRECTED_TRAINING_PREDICTIONS_FILE = (
    TRAINING_DIR /
    "Enugu_Corrected_Candidate_Model_Training_Predictions.csv"
)

CORRECTED_BEST_MODEL_FILE = (
    MODEL_DIR /
    "Enugu_Selected_Expansion_Model_Leakage_Corrected.joblib"
)

MODEL_SELECTION_FILE = (
    ADMIN_DIR /
    "ML_Leakage_Corrected_Model_Selection.json"
)

VALIDATION_FILE = (
    TRAINING_DIR /
    "Stage_10B_Leakage_Corrected_Model_Validation.csv"
)

for folder in [
    TRAINING_DIR,
    MODEL_DIR,
    TABLE_DIR,
    ADMIN_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

print("=" * 120)
print("STAGE 10B — LABEL-RULE LEAKAGE AUDIT + CORRECTED MODEL SELECTION")
print("=" * 120)

# ------------------------------------------------------------------------------------------
# REQUIRED FILE CHECK
# ------------------------------------------------------------------------------------------
print("\nREQUIRED FILE CHECK")
print("-" * 120)

print(
    f"{'✓' if INPUT_CSV_FILE.exists() else '✗'} "
    f"Predictor-enriched sample table: {INPUT_CSV_FILE}"
)

if not INPUT_CSV_FILE.exists():

    raise FileNotFoundError(
        "The Stage 9F predictor-enriched sample table is missing."
    )

# ------------------------------------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------------------------------------
data = pd.read_csv(
    INPUT_CSV_FILE
)

all_original_predictors = [
    "Elevation_m",
    "Slope_deg",
    "Distance_to_Road_m",
    "Distance_to_Surface_Water_m",
    "Distance_to_Drainage_m",
    "Population_Density",
    "Distance_to_Built_2020_m",
    "Baseline_LULC_2020"
]

continuous_predictors = [
    "Elevation_m",
    "Slope_deg",
    "Distance_to_Road_m",
    "Distance_to_Surface_Water_m",
    "Distance_to_Drainage_m",
    "Population_Density"
]

categorical_predictors = [
    "Baseline_LULC_2020"
]

corrected_predictor_columns = (
    continuous_predictors
    +
    categorical_predictors
)

excluded_rule_predictor = (
    "Distance_to_Built_2020_m"
)

required_columns = {
    "Sample_ID",
    "Target",
    "Target_Name",
    "Dataset_Split",
    "Block_ID",
    "X",
    "Y",
    "Predictor_Complete",
    *all_original_predictors
}

missing_columns = (
    required_columns -
    set(data.columns)
)

if missing_columns:

    raise RuntimeError(
        "Required columns are missing: "
        f"{sorted(missing_columns)}"
    )

data["Target"] = (
    data["Target"]
    .astype(np.uint8)
)

data["Baseline_LULC_2020"] = (
    data["Baseline_LULC_2020"]
    .astype(str)
)

if data["Predictor_Complete"].dtype == object:

    data["Predictor_Complete"] = (
        data["Predictor_Complete"]
        .astype(str)
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False
        })
    )

if data["Predictor_Complete"].isna().any():

    raise RuntimeError(
        "Predictor_Complete contains unrecognised values."
    )

# ------------------------------------------------------------------------------------------
# DATASET QA
# ------------------------------------------------------------------------------------------
expected_split_counts = {
    "Train": 16_800,
    "Validation": 3_600,
    "Test": 3_600
}

actual_split_counts = (
    data["Dataset_Split"]
    .value_counts()
    .to_dict()
)

print("\nMODELLING DATASET")
print("-" * 120)

print(
    f"Total records: {data.shape[0]:,}"
)

print(
    "Dataset splits:",
    actual_split_counts
)

print(
    "Target classes:",
    data[
        "Target"
    ].value_counts().sort_index().to_dict()
)

if data.shape[0] != 24_000:

    raise RuntimeError(
        "Expected 24,000 modelling records."
    )

if actual_split_counts != expected_split_counts:

    raise RuntimeError(
        "The spatial dataset split has changed."
    )

if not data["Predictor_Complete"].all():

    raise RuntimeError(
        "Incomplete predictor records remain."
    )

# ------------------------------------------------------------------------------------------
# LABEL-RULE LEAKAGE AUDIT
# ------------------------------------------------------------------------------------------
distance_values = data[
    excluded_rule_predictor
].to_numpy(
    dtype=np.float64
)

target_values = data[
    "Target"
].to_numpy(
    dtype=np.uint8
)

positive_distance = distance_values[
    target_values == 1
]

negative_distance = distance_values[
    target_values == 0
]

# Use the Stage 9D rule directly as a simple classifier.
rule_prediction = (
    distance_values
    <=
    LABEL_DISTANCE_THRESHOLD_M
).astype(np.uint8)

rule_probability = (
    1.0
    -
    np.clip(
        distance_values /
        max(
            np.nanmax(distance_values),
            1
        ),
        0,
        1
    )
)

rule_balanced_accuracy = balanced_accuracy_score(
    target_values,
    rule_prediction
)

rule_accuracy = accuracy_score(
    target_values,
    rule_prediction
)

rule_roc_auc = roc_auc_score(
    target_values,
    -distance_values
)

tn_rule, fp_rule, fn_rule, tp_rule = confusion_matrix(
    target_values,
    rule_prediction,
    labels=[0, 1]
).ravel()

positive_at_or_below_90 = int(
    np.count_nonzero(
        positive_distance
        <=
        LABEL_DISTANCE_THRESHOLD_M
    )
)

negative_at_or_below_90 = int(
    np.count_nonzero(
        negative_distance
        <=
        LABEL_DISTANCE_THRESHOLD_M
    )
)

positive_above_90 = int(
    np.count_nonzero(
        positive_distance
        >
        LABEL_DISTANCE_THRESHOLD_M
    )
)

negative_above_90 = int(
    np.count_nonzero(
        negative_distance
        >
        LABEL_DISTANCE_THRESHOLD_M
    )
)

leakage_audit_df = pd.DataFrame([
    {
        "Audit_Item":
            "Positive samples",

        "Record_Count":
            int(positive_distance.size),

        "Minimum_Distance_m":
            float(np.min(positive_distance)),

        "Median_Distance_m":
            float(np.median(positive_distance)),

        "Maximum_Distance_m":
            float(np.max(positive_distance)),

        "At_or_Below_90m":
            positive_at_or_below_90,

        "Above_90m":
            positive_above_90
    },
    {
        "Audit_Item":
            "Negative samples",

        "Record_Count":
            int(negative_distance.size),

        "Minimum_Distance_m":
            float(np.min(negative_distance)),

        "Median_Distance_m":
            float(np.median(negative_distance)),

        "Maximum_Distance_m":
            float(np.max(negative_distance)),

        "At_or_Below_90m":
            negative_at_or_below_90,

        "Above_90m":
            negative_above_90
    }
])

leakage_audit_df.to_csv(
    LEAKAGE_AUDIT_FILE,
    index=False
)

print("\nDISTANCE-TO-BUILT LABEL-RULE AUDIT")
print("-" * 120)

print(
    leakage_audit_df.to_string(
        index=False,
        formatters={
            "Minimum_Distance_m":
                lambda value: f"{value:.3f}",

            "Median_Distance_m":
                lambda value: f"{value:.3f}",

            "Maximum_Distance_m":
                lambda value: f"{value:.3f}"
        }
    )
)

print("\nSIMPLE 90 m RULE PERFORMANCE")
print("-" * 120)

print(
    f"Accuracy: {rule_accuracy:.4f}"
)

print(
    f"Balanced accuracy: "
    f"{rule_balanced_accuracy:.4f}"
)

print(
    f"ROC-AUC using inverse distance only: "
    f"{rule_roc_auc:.4f}"
)

print(
    f"Confusion matrix — TN: {tn_rule:,}, "
    f"FP: {fp_rule:,}, FN: {fn_rule:,}, TP: {tp_rule:,}"
)

# Audit conclusion.
rule_predictor_is_leakage = (
    rule_roc_auc >= 0.95
    or
    rule_balanced_accuracy >= 0.95
)

print("\nLEAKAGE DETERMINATION")
print("-" * 120)

print(
    f"{'✓' if rule_predictor_is_leakage else '✗'} "
    "Distance_to_Built_2020_m directly reproduces "
    "the label eligibility rule."
)

print(
    "✓ Distance_to_Built_2020_m will be excluded "
    "from corrected candidate modelling."
)

# ------------------------------------------------------------------------------------------
# CREATE TRAIN AND VALIDATION DATASETS
# ------------------------------------------------------------------------------------------
train_df = data.loc[
    data["Dataset_Split"] == "Train"
].copy()

validation_df = data.loc[
    data["Dataset_Split"] == "Validation"
].copy()

test_df = data.loc[
    data["Dataset_Split"] == "Test"
].copy()

X_train = train_df[
    corrected_predictor_columns
].copy()

y_train = train_df[
    "Target"
].to_numpy(
    dtype=np.uint8
)

X_validation = validation_df[
    corrected_predictor_columns
].copy()

y_validation = validation_df[
    "Target"
].to_numpy(
    dtype=np.uint8
)

# Test predictors and labels are deliberately not created.

print("\nCORRECTED DATASET PARTITIONS")
print("-" * 120)

print(
    f"Training records: {train_df.shape[0]:,}"
)

print(
    f"Validation records: {validation_df.shape[0]:,}"
)

print(
    f"Reserved test records: {test_df.shape[0]:,}"
)

print(
    f"Corrected predictor count: "
    f"{len(corrected_predictor_columns)}"
)

print(
    "Corrected predictors:"
)

for predictor in corrected_predictor_columns:

    print(
        f"  ✓ {predictor}"
    )

print(
    f"  ✗ Excluded: {excluded_rule_predictor}"
)

# ------------------------------------------------------------------------------------------
# PREPROCESSING
# ------------------------------------------------------------------------------------------
continuous_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

try:

    categorical_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )

except TypeError:

    categorical_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse=False
    )

categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            categorical_encoder
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "continuous",
            continuous_transformer,
            continuous_predictors
        ),
        (
            "categorical",
            categorical_transformer,
            categorical_predictors
        )
    ],
    remainder="drop"
)

# ------------------------------------------------------------------------------------------
# CANDIDATE MODELS
# ------------------------------------------------------------------------------------------
candidate_estimators = {
    "Logistic Regression":
        LogisticRegression(
            penalty="l2",
            C=1.0,
            solver="lbfgs",
            max_iter=2_000,
            random_state=RANDOM_SEED
        ),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_split=6,
            min_samples_leaf=3,
            max_features="sqrt",
            bootstrap=True,
            n_jobs=-1,
            random_state=RANDOM_SEED
        ),

    "Extra Trees":
        ExtraTreesClassifier(
            n_estimators=500,
            max_depth=None,
            min_samples_split=6,
            min_samples_leaf=3,
            max_features="sqrt",
            bootstrap=False,
            n_jobs=-1,
            random_state=RANDOM_SEED
        ),

    "Histogram Gradient Boosting":
        HistGradientBoostingClassifier(
            learning_rate=0.06,
            max_iter=350,
            max_leaf_nodes=31,
            max_depth=None,
            min_samples_leaf=20,
            l2_regularization=1.0,
            early_stopping=True,
            validation_fraction=0.10,
            n_iter_no_change=30,
            random_state=RANDOM_SEED
        )
}

# ------------------------------------------------------------------------------------------
# METRIC FUNCTION
# ------------------------------------------------------------------------------------------
def calculate_binary_metrics(
    y_true,
    probability,
    threshold=0.50
):

    prediction = (
        probability >= threshold
    ).astype(np.uint8)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        prediction,
        labels=[0, 1]
    ).ravel()

    specificity = (
        tn /
        (tn + fp)
        if (tn + fp) > 0
        else np.nan
    )

    return {
        "Accuracy":
            accuracy_score(
                y_true,
                prediction
            ),

        "Balanced_Accuracy":
            balanced_accuracy_score(
                y_true,
                prediction
            ),

        "Precision":
            precision_score(
                y_true,
                prediction,
                zero_division=0
            ),

        "Recall_Sensitivity":
            recall_score(
                y_true,
                prediction,
                zero_division=0
            ),

        "Specificity":
            specificity,

        "F1_Score":
            f1_score(
                y_true,
                prediction,
                zero_division=0
            ),

        "ROC_AUC":
            roc_auc_score(
                y_true,
                probability
            ),

        "PR_AUC":
            average_precision_score(
                y_true,
                probability
            ),

        "Matthews_Correlation":
            matthews_corrcoef(
                y_true,
                prediction
            ),

        "Brier_Score":
            brier_score_loss(
                y_true,
                probability
            ),

        "Log_Loss":
            log_loss(
                y_true,
                np.column_stack(
                    [
                        1 - probability,
                        probability
                    ]
                ),
                labels=[0, 1]
            ),

        "True_Negative":
            int(tn),

        "False_Positive":
            int(fp),

        "False_Negative":
            int(fn),

        "True_Positive":
            int(tp)
    }

# ------------------------------------------------------------------------------------------
# TRAIN CORRECTED CANDIDATE MODELS
# ------------------------------------------------------------------------------------------
metric_records = []
confusion_records = []
fitted_models = {}

validation_prediction_output = validation_df[
    [
        "Sample_ID",
        "Target",
        "Dataset_Split",
        "Block_ID",
        "X",
        "Y"
    ]
].copy()

training_prediction_output = train_df[
    [
        "Sample_ID",
        "Target",
        "Dataset_Split",
        "Block_ID",
        "X",
        "Y"
    ]
].copy()

print("\nCORRECTED CANDIDATE MODEL TRAINING")
print("-" * 120)

for model_name, estimator in candidate_estimators.items():

    print(
        f"\nTraining: {model_name}"
    )

    model_pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor
            ),
            (
                "classifier",
                estimator
            )
        ]
    )

    model_pipeline.fit(
        X_train,
        y_train
    )

    fitted_models[
        model_name
    ] = model_pipeline

    train_probability = (
        model_pipeline.predict_proba(
            X_train
        )[:, 1]
    )

    validation_probability = (
        model_pipeline.predict_proba(
            X_validation
        )[:, 1]
    )

    train_metrics = calculate_binary_metrics(
        y_train,
        train_probability,
        CLASSIFICATION_THRESHOLD
    )

    validation_metrics = calculate_binary_metrics(
        y_validation,
        validation_probability,
        CLASSIFICATION_THRESHOLD
    )

    for dataset_name, metrics in [
        (
            "Train",
            train_metrics
        ),
        (
            "Validation",
            validation_metrics
        )
    ]:

        record = {
            "Model":
                model_name,

            "Dataset":
                dataset_name,

            "Classification_Threshold":
                CLASSIFICATION_THRESHOLD
        }

        record.update(
            metrics
        )

        metric_records.append(
            record
        )

        confusion_records.append({
            "Model":
                model_name,

            "Dataset":
                dataset_name,

            "True_Negative":
                metrics["True_Negative"],

            "False_Positive":
                metrics["False_Positive"],

            "False_Negative":
                metrics["False_Negative"],

            "True_Positive":
                metrics["True_Positive"]
        })

    safe_model_name = (
        model_name
        .lower()
        .replace(" ", "_")
        .replace("-", "_")
    )

    validation_prediction_output[
        f"{safe_model_name}_Probability"
    ] = validation_probability

    validation_prediction_output[
        f"{safe_model_name}_Prediction"
    ] = (
        validation_probability
        >=
        CLASSIFICATION_THRESHOLD
    ).astype(np.uint8)

    training_prediction_output[
        f"{safe_model_name}_Probability"
    ] = train_probability

    training_prediction_output[
        f"{safe_model_name}_Prediction"
    ] = (
        train_probability
        >=
        CLASSIFICATION_THRESHOLD
    ).astype(np.uint8)

    corrected_candidate_file = (
        MODEL_DIR /
        f"Enugu_{safe_model_name}_Leakage_Corrected_Candidate.joblib"
    )

    joblib.dump(
        model_pipeline,
        corrected_candidate_file,
        compress=3
    )

    print(
        f"  Train ROC-AUC: "
        f"{train_metrics['ROC_AUC']:.4f}"
    )

    print(
        f"  Validation ROC-AUC: "
        f"{validation_metrics['ROC_AUC']:.4f}"
    )

    print(
        f"  Validation PR-AUC: "
        f"{validation_metrics['PR_AUC']:.4f}"
    )

    print(
        f"  Validation balanced accuracy: "
        f"{validation_metrics['Balanced_Accuracy']:.4f}"
    )

# ------------------------------------------------------------------------------------------
# BUILD CORRECTED LEADERBOARD
# ------------------------------------------------------------------------------------------
metrics_df = pd.DataFrame(
    metric_records
)

confusion_df = pd.DataFrame(
    confusion_records
)

validation_leaderboard = (
    metrics_df.loc[
        metrics_df["Dataset"] == "Validation"
    ]
    .copy()
)

training_lookup = (
    metrics_df.loc[
        metrics_df["Dataset"] == "Train",
        [
            "Model",
            "ROC_AUC",
            "PR_AUC",
            "Balanced_Accuracy",
            "F1_Score"
        ]
    ]
    .rename(
        columns={
            "ROC_AUC":
                "Train_ROC_AUC",

            "PR_AUC":
                "Train_PR_AUC",

            "Balanced_Accuracy":
                "Train_Balanced_Accuracy",

            "F1_Score":
                "Train_F1_Score"
        }
    )
)

validation_leaderboard = (
    validation_leaderboard
    .merge(
        training_lookup,
        on="Model",
        how="left"
    )
)

validation_leaderboard[
    "ROC_AUC_Generalisation_Gap"
] = (
    validation_leaderboard[
        "Train_ROC_AUC"
    ]
    -
    validation_leaderboard[
        "ROC_AUC"
    ]
)

validation_leaderboard[
    "PR_AUC_Generalisation_Gap"
] = (
    validation_leaderboard[
        "Train_PR_AUC"
    ]
    -
    validation_leaderboard[
        "PR_AUC"
    ]
)

best_validation_roc_auc = float(
    validation_leaderboard[
        "ROC_AUC"
    ].max()
)

validation_leaderboard[
    "Within_ROC_AUC_Tie_Range"
] = (
    validation_leaderboard[
        "ROC_AUC"
    ]
    >=
    (
        best_validation_roc_auc
        -
        ROC_AUC_TIE_TOLERANCE
    )
)

eligible_models = (
    validation_leaderboard.loc[
        validation_leaderboard[
            "Within_ROC_AUC_Tie_Range"
        ]
    ]
    .sort_values(
        by=[
            "ROC_AUC",
            "PR_AUC",
            "Balanced_Accuracy",
            "Matthews_Correlation"
        ],
        ascending=False
    )
)

selected_model_name = str(
    eligible_models.iloc[0][
        "Model"
    ]
)

validation_leaderboard[
    "Selected_Model"
] = (
    validation_leaderboard[
        "Model"
    ]
    ==
    selected_model_name
)

validation_leaderboard = (
    validation_leaderboard
    .sort_values(
        by=[
            "Selected_Model",
            "ROC_AUC",
            "PR_AUC",
            "Balanced_Accuracy"
        ],
        ascending=[
            False,
            False,
            False,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)

validation_leaderboard[
    "Validation_Rank"
] = np.arange(
    1,
    validation_leaderboard.shape[0] + 1
)

selected_model = fitted_models[
    selected_model_name
]

joblib.dump(
    selected_model,
    CORRECTED_BEST_MODEL_FILE,
    compress=3
)

selected_record = (
    validation_leaderboard.loc[
        validation_leaderboard[
            "Selected_Model"
        ]
    ]
    .iloc[0]
)

print("\nCORRECTED VALIDATION MODEL LEADERBOARD")
print("-" * 120)

display_columns = [
    "Validation_Rank",
    "Model",
    "ROC_AUC",
    "PR_AUC",
    "Balanced_Accuracy",
    "Precision",
    "Recall_Sensitivity",
    "Specificity",
    "F1_Score",
    "Matthews_Correlation",
    "Brier_Score",
    "ROC_AUC_Generalisation_Gap",
    "Selected_Model"
]

print(
    validation_leaderboard[
        display_columns
    ].to_string(
        index=False,
        formatters={
            column:
                lambda value: f"{value:.4f}"
            for column in [
                "ROC_AUC",
                "PR_AUC",
                "Balanced_Accuracy",
                "Precision",
                "Recall_Sensitivity",
                "Specificity",
                "F1_Score",
                "Matthews_Correlation",
                "Brier_Score",
                "ROC_AUC_Generalisation_Gap"
            ]
        }
    )
)

print("\nCORRECTED SELECTED MODEL")
print("-" * 120)

print(
    f"Model: {selected_model_name}"
)

print(
    f"Validation ROC-AUC: "
    f"{selected_record['ROC_AUC']:.4f}"
)

print(
    f"Validation PR-AUC: "
    f"{selected_record['PR_AUC']:.4f}"
)

print(
    f"Validation balanced accuracy: "
    f"{selected_record['Balanced_Accuracy']:.4f}"
)

print(
    f"Validation F1 score: "
    f"{selected_record['F1_Score']:.4f}"
)

print(
    f"ROC-AUC generalisation gap: "
    f"{selected_record['ROC_AUC_Generalisation_Gap']:.4f}"
)

# ------------------------------------------------------------------------------------------
# SAVE OUTPUT TABLES
# ------------------------------------------------------------------------------------------
metrics_df.to_csv(
    CORRECTED_METRICS_FILE,
    index=False
)

validation_leaderboard.to_csv(
    CORRECTED_LEADERBOARD_FILE,
    index=False
)

confusion_df.to_csv(
    CORRECTED_CONFUSION_FILE,
    index=False
)

validation_prediction_output.to_csv(
    CORRECTED_VALIDATION_PREDICTIONS_FILE,
    index=False
)

training_prediction_output.to_csv(
    CORRECTED_TRAINING_PREDICTIONS_FILE,
    index=False
)

# ------------------------------------------------------------------------------------------
# SAVE MODEL-SELECTION REGISTER
# ------------------------------------------------------------------------------------------
model_selection_register = {
    "prediction_target":
        "Urban expansion from 2020 to 2025",

    "identified_label_rule_leakage":
        excluded_rule_predictor,

    "label_rule":
        "Positive distance <= 90 m; negative distance >= 90 m",

    "distance_only_rule_metrics": {
        "accuracy":
            float(rule_accuracy),

        "balanced_accuracy":
            float(rule_balanced_accuracy),

        "roc_auc":
            float(rule_roc_auc)
    },

    "corrective_action":
        "Distance_to_Built_2020_m excluded from modelling",

    "retained_continuous_predictors":
        continuous_predictors,

    "retained_categorical_predictors":
        categorical_predictors,

    "retained_predictor_count":
        len(corrected_predictor_columns),

    "training_sample_count":
        int(train_df.shape[0]),

    "validation_sample_count":
        int(validation_df.shape[0]),

    "reserved_test_sample_count":
        int(test_df.shape[0]),

    "candidate_models":
        list(candidate_estimators.keys()),

    "selected_model":
        selected_model_name,

    "selected_validation_metrics": {
        "roc_auc":
            float(selected_record["ROC_AUC"]),

        "pr_auc":
            float(selected_record["PR_AUC"]),

        "balanced_accuracy":
            float(
                selected_record[
                    "Balanced_Accuracy"
                ]
            ),

        "precision":
            float(selected_record["Precision"]),

        "recall":
            float(
                selected_record[
                    "Recall_Sensitivity"
                ]
            ),

        "specificity":
            float(selected_record["Specificity"]),

        "f1_score":
            float(selected_record["F1_Score"]),

        "matthews_correlation":
            float(
                selected_record[
                    "Matthews_Correlation"
                ]
            ),

        "brier_score":
            float(selected_record["Brier_Score"]),

        "roc_auc_generalisation_gap":
            float(
                selected_record[
                    "ROC_AUC_Generalisation_Gap"
                ]
            )
    },

    "test_predictor_matrix_created":
        False,

    "test_evaluation_status":
        "Not performed",

    "software": {
        "python":
            platform.python_version(),

        "scikit_learn":
            sklearn.__version__,

        "numpy":
            np.__version__,

        "pandas":
            pd.__version__
    }
}

with open(
    MODEL_SELECTION_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        model_selection_register,
        file,
        indent=4
    )

# ------------------------------------------------------------------------------------------
# RELOAD MODEL AND VERIFY
# ------------------------------------------------------------------------------------------
reloaded_model = joblib.load(
    CORRECTED_BEST_MODEL_FILE
)

original_probability = (
    selected_model.predict_proba(
        X_validation
    )[:, 1]
)

reloaded_probability = (
    reloaded_model.predict_proba(
        X_validation
    )[:, 1]
)

saved_model_probability_match = np.allclose(
    original_probability,
    reloaded_probability,
    atol=1e-10
)

# ------------------------------------------------------------------------------------------
# VALIDATION REGISTER
# ------------------------------------------------------------------------------------------
selected_validation_roc_auc = float(
    selected_record["ROC_AUC"]
)

selected_validation_pr_auc = float(
    selected_record["PR_AUC"]
)

selected_validation_balanced_accuracy = float(
    selected_record[
        "Balanced_Accuracy"
    ]
)

selected_generalisation_gap = float(
    selected_record[
        "ROC_AUC_Generalisation_Gap"
    ]
)

validation_register_df = pd.DataFrame([{
    "Total_Samples":
        int(data.shape[0]),

    "Training_Samples":
        int(train_df.shape[0]),

    "Validation_Samples":
        int(validation_df.shape[0]),

    "Reserved_Test_Samples":
        int(test_df.shape[0]),

    "Original_Predictor_Count":
        len(all_original_predictors),

    "Corrected_Predictor_Count":
        len(corrected_predictor_columns),

    "Excluded_Rule_Predictor":
        excluded_rule_predictor,

    "Distance_Rule_ROC_AUC":
        rule_roc_auc,

    "Distance_Rule_Balanced_Accuracy":
        rule_balanced_accuracy,

    "Rule_Predictor_Leakage_Confirmed":
        rule_predictor_is_leakage,

    "Selected_Model":
        selected_model_name,

    "Selected_Validation_ROC_AUC":
        selected_validation_roc_auc,

    "Selected_Validation_PR_AUC":
        selected_validation_pr_auc,

    "Selected_Validation_Balanced_Accuracy":
        selected_validation_balanced_accuracy,

    "Selected_ROC_AUC_Generalisation_Gap":
        selected_generalisation_gap,

    "Saved_Model_Probability_Match":
        saved_model_probability_match,

    "Test_Predictor_Matrix_Created":
        False,

    "Test_Evaluation_Completed":
        False,

    "Corrected_Model_Selection_Completed":
        True
}])

validation_register_df.to_csv(
    VALIDATION_FILE,
    index=False
)

# ------------------------------------------------------------------------------------------
# FINAL VALIDATION
# ------------------------------------------------------------------------------------------
corrected_candidate_files = [
    (
        MODEL_DIR /
        (
            "Enugu_"
            +
            model_name
            .lower()
            .replace(" ", "_")
            .replace("-", "_")
            +
            "_Leakage_Corrected_Candidate.joblib"
        )
    )
    for model_name in candidate_estimators
]

all_candidate_files_saved = all(
    path.exists()
    for path in corrected_candidate_files
)

each_model_has_two_datasets = (
    metrics_df
    .groupby("Model")["Dataset"]
    .nunique()
    .min()
    ==
    2
)

corrected_predictor_exclusion_confirmed = (
    excluded_rule_predictor
    not in
    corrected_predictor_columns
)

checks = {
    "All 24,000 samples preserved":
        data.shape[0] == 24_000,

    "Training sample count is 16,800":
        train_df.shape[0] == 16_800,

    "Validation sample count is 3,600":
        validation_df.shape[0] == 3_600,

    "Test sample count remains 3,600":
        test_df.shape[0] == 3_600,

    "Distance rule predictor leakage confirmed":
        rule_predictor_is_leakage,

    "Distance-to-built predictor excluded":
        corrected_predictor_exclusion_confirmed,

    "Seven corrected predictors retained":
        len(corrected_predictor_columns) == 7,

    "Training set contains both classes":
        sorted(
            train_df["Target"]
            .unique()
            .tolist()
        ) == [0, 1],

    "Validation set contains both classes":
        sorted(
            validation_df["Target"]
            .unique()
            .tolist()
        ) == [0, 1],

    "Four corrected candidate models trained":
        len(fitted_models) == 4,

    "Each model has train and validation metrics":
        each_model_has_two_datasets,

    "Corrected selected model is valid":
        selected_model_name
        in
        candidate_estimators,

    "Selected validation ROC-AUC is finite":
        np.isfinite(
            selected_validation_roc_auc
        ),

    "Selected validation PR-AUC is finite":
        np.isfinite(
            selected_validation_pr_auc
        ),

    "Selected balanced accuracy is finite":
        np.isfinite(
            selected_validation_balanced_accuracy
        ),

    "All corrected candidate models saved":
        all_candidate_files_saved,

    "Corrected selected model saved":
        CORRECTED_BEST_MODEL_FILE.exists(),

    "Reloaded model predictions match":
        saved_model_probability_match,

    "Leakage audit table saved":
        LEAKAGE_AUDIT_FILE.exists(),

    "Corrected metrics table saved":
        CORRECTED_METRICS_FILE.exists(),

    "Corrected leaderboard saved":
        CORRECTED_LEADERBOARD_FILE.exists(),

    "Corrected confusion matrices saved":
        CORRECTED_CONFUSION_FILE.exists(),

    "Corrected validation predictions saved":
        CORRECTED_VALIDATION_PREDICTIONS_FILE.exists(),

    "Corrected training predictions saved":
        CORRECTED_TRAINING_PREDICTIONS_FILE.exists(),

    "Corrected selection register saved":
        MODEL_SELECTION_FILE.exists(),

    "Validation register saved":
        VALIDATION_FILE.exists(),

    "Test predictor matrix remains uncreated":
        validation_register_df[
            "Test_Predictor_Matrix_Created"
        ].iloc[0] == False,

    "Test evaluation remains incomplete":
        validation_register_df[
            "Test_Evaluation_Completed"
        ].iloc[0] == False,

    "Corrected model selection marked complete":
        validation_register_df[
            "Corrected_Model_Selection_Completed"
        ].iloc[0] == True
}

print("\nVALIDATION SUMMARY")
print("-" * 120)

for label, passed in checks.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{label}"
    )

print("\nOUTPUT FILES")
print("-" * 120)

for path in [
    *corrected_candidate_files,
    CORRECTED_BEST_MODEL_FILE,
    LEAKAGE_AUDIT_FILE,
    CORRECTED_METRICS_FILE,
    CORRECTED_LEADERBOARD_FILE,
    CORRECTED_CONFUSION_FILE,
    CORRECTED_VALIDATION_PREDICTIONS_FILE,
    CORRECTED_TRAINING_PREDICTIONS_FILE,
    MODEL_SELECTION_FILE,
    VALIDATION_FILE
]:

    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{path.name}"
    )

if all(
    checks.values()
):

    print("\n✓ STAGE 10B FULLY PASSED")

    print(
        f"The leakage-corrected selected model is: "
        f"{selected_model_name}."
    )

    print(
        "Distance_to_Built_2020_m has been removed because "
        "it directly encoded the label eligibility rule."
    )

    print(
        "The 3,600-record spatial test dataset remains "
        "untouched for final evaluation."
    )

else:

    print("\n⚠ STAGE 10B REQUIRES REVIEW")

print("=" * 120)

In [ ]:
# ==========================================================================================
# PROJECT 7 — STAGE 10C
# FINAL INDEPENDENT SPATIAL TEST EVALUATION
#
# This stage:
# - loads the leakage-corrected selected model from Stage 10B;
# - opens the previously untouched 3,600-record spatial test set;
# - evaluates the model once using the fixed probability threshold of 0.50;
# - computes discrimination, classification and calibration metrics;
# - creates ROC, Precision–Recall, calibration and confusion-matrix charts;
# - saves test predictions, metrics, curves and validation registers;
# - does not retrain, tune or alter the selected model.
#
# The resulting test statistics are the official unbiased model-performance results.
# ==========================================================================================

from pathlib import Path
import json
import platform
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import sklearn

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    cohen_kappa_score,
    confusion_matrix,
    brier_score_loss,
    log_loss,
    roc_curve,
    precision_recall_curve
)

from sklearn.calibration import calibration_curve

warnings.filterwarnings(
    "ignore",
    category=FutureWarning
)

# ------------------------------------------------------------------------------------------
# PARAMETERS
# ------------------------------------------------------------------------------------------
CLASSIFICATION_THRESHOLD = 0.50

EXPECTED_TEST_SAMPLES = 3_600
EXPECTED_NEGATIVE_TEST_SAMPLES = 1_800
EXPECTED_POSITIVE_TEST_SAMPLES = 1_800

# ------------------------------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------------------------------
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

TRAINING_DIR = (
    PROJECT_ROOT /
    "09_Training_Data"
)

MODEL_DIR = (
    PROJECT_ROOT /
    "11_ML_Models"
)

CHART_DIR = (
    PROJECT_ROOT /
    "15_Charts"
)

TABLE_DIR = (
    PROJECT_ROOT /
    "16_Tables"
)

ADMIN_DIR = (
    PROJECT_ROOT /
    "00_Project_Admin"
)

INPUT_CSV_FILE = (
    TRAINING_DIR /
    "Enugu_ML_Samples_With_Predictors_2020_2025.csv"
)

SELECTED_MODEL_FILE = (
    MODEL_DIR /
    "Enugu_Selected_Expansion_Model_Leakage_Corrected.joblib"
)

MODEL_SELECTION_FILE = (
    ADMIN_DIR /
    "ML_Leakage_Corrected_Model_Selection.json"
)

TEST_PREDICTIONS_FILE = (
    TRAINING_DIR /
    "Enugu_Final_Independent_Test_Predictions.csv"
)

TEST_METRICS_FILE = (
    TABLE_DIR /
    "Enugu_Final_Independent_Test_Metrics.csv"
)

TEST_CONFUSION_FILE = (
    TABLE_DIR /
    "Enugu_Final_Independent_Test_Confusion_Matrix.csv"
)

TEST_THRESHOLD_FILE = (
    TABLE_DIR /
    "Enugu_Final_Test_Threshold_050_Performance.csv"
)

ROC_CURVE_DATA_FILE = (
    TABLE_DIR /
    "Enugu_Final_Test_ROC_Curve_Data.csv"
)

PR_CURVE_DATA_FILE = (
    TABLE_DIR /
    "Enugu_Final_Test_Precision_Recall_Curve_Data.csv"
)

CALIBRATION_DATA_FILE = (
    TABLE_DIR /
    "Enugu_Final_Test_Calibration_Data.csv"
)

ROC_CHART_FILE = (
    CHART_DIR /
    "10C_Final_Test_ROC_Curve.png"
)

PR_CHART_FILE = (
    CHART_DIR /
    "10C_Final_Test_Precision_Recall_Curve.png"
)

CALIBRATION_CHART_FILE = (
    CHART_DIR /
    "10C_Final_Test_Calibration_Curve.png"
)

CONFUSION_CHART_FILE = (
    CHART_DIR /
    "10C_Final_Test_Confusion_Matrix.png"
)

PROBABILITY_DISTRIBUTION_CHART_FILE = (
    CHART_DIR /
    "10C_Final_Test_Probability_Distribution.png"
)

TEST_EVALUATION_REGISTER_FILE = (
    ADMIN_DIR /
    "ML_Final_Independent_Test_Evaluation.json"
)

VALIDATION_FILE = (
    TRAINING_DIR /
    "Stage_10C_Final_Test_Evaluation_Validation.csv"
)

for folder in [
    TRAINING_DIR,
    MODEL_DIR,
    CHART_DIR,
    TABLE_DIR,
    ADMIN_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

print("=" * 120)
print("STAGE 10C — FINAL INDEPENDENT SPATIAL TEST EVALUATION")
print("=" * 120)

# ------------------------------------------------------------------------------------------
# REQUIRED FILE CHECK
# ------------------------------------------------------------------------------------------
required_files = {
    "Predictor-enriched sample table":
        INPUT_CSV_FILE,

    "Leakage-corrected selected model":
        SELECTED_MODEL_FILE,

    "Stage 10B model-selection register":
        MODEL_SELECTION_FILE
}

print("\nREQUIRED FILE CHECK")
print("-" * 120)

for label, path in required_files.items():

    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{label}: {path}"
    )

if not all(
    path.exists()
    for path in required_files.values()
):

    raise FileNotFoundError(
        "One or more required Stage 10C inputs are missing."
    )

# ------------------------------------------------------------------------------------------
# LOAD MODEL-SELECTION REGISTER
# ------------------------------------------------------------------------------------------
with open(
    MODEL_SELECTION_FILE,
    "r",
    encoding="utf-8"
) as file:

    model_selection_register = json.load(
        file
    )

selected_model_name = (
    model_selection_register.get(
        "selected_model",
        "Unknown"
    )
)

registered_test_status = (
    model_selection_register.get(
        "test_evaluation_status",
        "Unknown"
    )
)

registered_test_matrix_status = (
    model_selection_register.get(
        "test_predictor_matrix_created",
        None
    )
)

print("\nSTAGE 10B MODEL-SELECTION REGISTER")
print("-" * 120)

print(
    f"Selected model: {selected_model_name}"
)

print(
    f"Previous test-evaluation status: "
    f"{registered_test_status}"
)

print(
    f"Previous test predictor matrix created: "
    f"{registered_test_matrix_status}"
)

if selected_model_name != "Extra Trees":

    print(
        "⚠ The selected model name differs from the expected "
        "Stage 10B Extra Trees result."
    )

if registered_test_status != "Not performed":

    print(
        "⚠ The model-selection register does not indicate an "
        "untouched test dataset."
    )

# ------------------------------------------------------------------------------------------
# LOAD COMPLETE SAMPLE TABLE
# ------------------------------------------------------------------------------------------
data = pd.read_csv(
    INPUT_CSV_FILE
)

continuous_predictors = [
    "Elevation_m",
    "Slope_deg",
    "Distance_to_Road_m",
    "Distance_to_Surface_Water_m",
    "Distance_to_Drainage_m",
    "Population_Density"
]

categorical_predictors = [
    "Baseline_LULC_2020"
]

predictor_columns = (
    continuous_predictors
    +
    categorical_predictors
)

excluded_rule_predictor = (
    "Distance_to_Built_2020_m"
)

required_columns = {
    "Sample_ID",
    "Target",
    "Target_Name",
    "Dataset_Split",
    "Block_ID",
    "X",
    "Y",
    "Predictor_Complete",
    *predictor_columns,
    excluded_rule_predictor
}

missing_columns = (
    required_columns -
    set(data.columns)
)

if missing_columns:

    raise RuntimeError(
        "Required test-evaluation columns are missing: "
        f"{sorted(missing_columns)}"
    )

data["Target"] = (
    data["Target"]
    .astype(np.uint8)
)

data["Baseline_LULC_2020"] = (
    data["Baseline_LULC_2020"]
    .astype(str)
)

if data["Predictor_Complete"].dtype == object:

    data["Predictor_Complete"] = (
        data["Predictor_Complete"]
        .astype(str)
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False
        })
    )

if data["Predictor_Complete"].isna().any():

    raise RuntimeError(
        "Predictor_Complete contains unrecognised values."
    )

# ------------------------------------------------------------------------------------------
# OPEN RESERVED TEST DATASET
# ------------------------------------------------------------------------------------------
test_df = data.loc[
    data[
        "Dataset_Split"
    ] == "Test"
].copy()

X_test = test_df[
    predictor_columns
].copy()

y_test = test_df[
    "Target"
].to_numpy(
    dtype=np.uint8
)

test_count = int(
    test_df.shape[0]
)

test_negative_count = int(
    np.count_nonzero(
        y_test == 0
    )
)

test_positive_count = int(
    np.count_nonzero(
        y_test == 1
    )
)

test_block_count = int(
    test_df[
        "Block_ID"
    ].nunique()
)

print("\nFINAL TEST DATASET")
print("-" * 120)

print(
    f"Test records: {test_count:,}"
)

print(
    f"Negative records: {test_negative_count:,}"
)

print(
    f"Positive records: {test_positive_count:,}"
)

print(
    f"Independent spatial blocks: {test_block_count:,}"
)

print(
    f"Predictors used: {len(predictor_columns)}"
)

print(
    "Predictor fields:"
)

for predictor in predictor_columns:

    print(
        f"  ✓ {predictor}"
    )

print(
    f"  ✗ Excluded label-rule predictor: "
    f"{excluded_rule_predictor}"
)

if test_count != EXPECTED_TEST_SAMPLES:

    raise RuntimeError(
        f"Expected {EXPECTED_TEST_SAMPLES:,} test records, "
        f"found {test_count:,}."
    )

if (
    test_negative_count
    !=
    EXPECTED_NEGATIVE_TEST_SAMPLES
    or
    test_positive_count
    !=
    EXPECTED_POSITIVE_TEST_SAMPLES
):

    raise RuntimeError(
        "The final test dataset is not balanced as expected."
    )

if not test_df["Predictor_Complete"].all():

    raise RuntimeError(
        "Incomplete predictor records exist in the test set."
    )

# ------------------------------------------------------------------------------------------
# LOAD SELECTED MODEL
# ------------------------------------------------------------------------------------------
selected_model = joblib.load(
    SELECTED_MODEL_FILE
)

print("\nSELECTED MODEL")
print("-" * 120)

print(
    f"Registered model: {selected_model_name}"
)

print(
    f"Model pipeline type: "
    f"{type(selected_model).__name__}"
)

if not hasattr(
    selected_model,
    "predict_proba"
):

    raise RuntimeError(
        "The selected model does not support probability prediction."
    )

# ------------------------------------------------------------------------------------------
# FINAL TEST PREDICTION
#
# The test set is evaluated exactly once.
# No threshold optimisation is conducted.
# ------------------------------------------------------------------------------------------
test_probability = (
    selected_model.predict_proba(
        X_test
    )[:, 1]
)

test_prediction = (
    test_probability
    >=
    CLASSIFICATION_THRESHOLD
).astype(np.uint8)

if not np.all(
    np.isfinite(
        test_probability
    )
):

    raise RuntimeError(
        "Non-finite test probabilities were produced."
    )

if (
    np.min(test_probability) < 0
    or
    np.max(test_probability) > 1
):

    raise RuntimeError(
        "Predicted probabilities fall outside the range 0–1."
    )

# ------------------------------------------------------------------------------------------
# CONFUSION MATRIX AND CLASSIFICATION METRICS
# ------------------------------------------------------------------------------------------
tn, fp, fn, tp = confusion_matrix(
    y_test,
    test_prediction,
    labels=[0, 1]
).ravel()

accuracy = accuracy_score(
    y_test,
    test_prediction
)

balanced_accuracy = balanced_accuracy_score(
    y_test,
    test_prediction
)

precision = precision_score(
    y_test,
    test_prediction,
    zero_division=0
)

recall = recall_score(
    y_test,
    test_prediction,
    zero_division=0
)

specificity = (
    tn /
    (tn + fp)
    if (tn + fp) > 0
    else np.nan
)

negative_predictive_value = (
    tn /
    (tn + fn)
    if (tn + fn) > 0
    else np.nan
)

f1 = f1_score(
    y_test,
    test_prediction,
    zero_division=0
)

mcc = matthews_corrcoef(
    y_test,
    test_prediction
)

cohen_kappa = cohen_kappa_score(
    y_test,
    test_prediction
)

false_positive_rate_value = (
    fp /
    (fp + tn)
    if (fp + tn) > 0
    else np.nan
)

false_negative_rate_value = (
    fn /
    (fn + tp)
    if (fn + tp) > 0
    else np.nan
)

# ------------------------------------------------------------------------------------------
# PROBABILITY-BASED METRICS
# ------------------------------------------------------------------------------------------
roc_auc = roc_auc_score(
    y_test,
    test_probability
)

pr_auc = average_precision_score(
    y_test,
    test_probability
)

brier_score = brier_score_loss(
    y_test,
    test_probability
)

test_log_loss = log_loss(
    y_test,
    np.column_stack(
        [
            1 - test_probability,
            test_probability
        ]
    ),
    labels=[0, 1]
)

# ------------------------------------------------------------------------------------------
# CURVE DATA
# ------------------------------------------------------------------------------------------
roc_fpr, roc_tpr, roc_thresholds = roc_curve(
    y_test,
    test_probability
)

pr_precision, pr_recall, pr_thresholds = (
    precision_recall_curve(
        y_test,
        test_probability
    )
)

calibration_fraction_positive, calibration_mean_prediction = (
    calibration_curve(
        y_test,
        test_probability,
        n_bins=10,
        strategy="quantile"
    )
)

roc_curve_df = pd.DataFrame({
    "False_Positive_Rate":
        roc_fpr,

    "True_Positive_Rate":
        roc_tpr,

    "Threshold":
        roc_thresholds
})

pr_curve_df = pd.DataFrame({
    "Recall":
        pr_recall,

    "Precision":
        pr_precision,

    "Threshold":
        np.append(
            pr_thresholds,
            np.nan
        )
})

calibration_df = pd.DataFrame({
    "Mean_Predicted_Probability":
        calibration_mean_prediction,

    "Observed_Positive_Fraction":
        calibration_fraction_positive
})

# ------------------------------------------------------------------------------------------
# METRICS TABLE
# ------------------------------------------------------------------------------------------
metrics_df = pd.DataFrame([{
    "Model":
        selected_model_name,

    "Dataset":
        "Independent spatial test",

    "Classification_Threshold":
        CLASSIFICATION_THRESHOLD,

    "Test_Samples":
        test_count,

    "Negative_Samples":
        test_negative_count,

    "Positive_Samples":
        test_positive_count,

    "Independent_Test_Blocks":
        test_block_count,

    "Accuracy":
        accuracy,

    "Balanced_Accuracy":
        balanced_accuracy,

    "Precision":
        precision,

    "Recall_Sensitivity":
        recall,

    "Specificity":
        specificity,

    "Negative_Predictive_Value":
        negative_predictive_value,

    "F1_Score":
        f1,

    "ROC_AUC":
        roc_auc,

    "PR_AUC":
        pr_auc,

    "Matthews_Correlation":
        mcc,

    "Cohen_Kappa":
        cohen_kappa,

    "Brier_Score":
        brier_score,

    "Log_Loss":
        test_log_loss,

    "False_Positive_Rate":
        false_positive_rate_value,

    "False_Negative_Rate":
        false_negative_rate_value,

    "True_Negative":
        int(tn),

    "False_Positive":
        int(fp),

    "False_Negative":
        int(fn),

    "True_Positive":
        int(tp)
}])

confusion_df = pd.DataFrame(
    [
        {
            "Actual_Class":
                "Stable non-built",

            "Predicted_Stable_NonBuilt":
                int(tn),

            "Predicted_Urban_Expansion":
                int(fp),

            "Total_Actual":
                int(tn + fp)
        },
        {
            "Actual_Class":
                "Urban expansion",

            "Predicted_Stable_NonBuilt":
                int(fn),

            "Predicted_Urban_Expansion":
                int(tp),

            "Total_Actual":
                int(fn + tp)
        }
    ]
)

threshold_df = pd.DataFrame([{
    "Threshold":
        CLASSIFICATION_THRESHOLD,

    "Accuracy":
        accuracy,

    "Balanced_Accuracy":
        balanced_accuracy,

    "Precision":
        precision,

    "Recall":
        recall,

    "Specificity":
        specificity,

    "F1_Score":
        f1,

    "Matthews_Correlation":
        mcc,

    "Cohen_Kappa":
        cohen_kappa,

    "True_Negative":
        int(tn),

    "False_Positive":
        int(fp),

    "False_Negative":
        int(fn),

    "True_Positive":
        int(tp)
}])

# ------------------------------------------------------------------------------------------
# TEST PREDICTION TABLE
# ------------------------------------------------------------------------------------------
test_predictions_df = test_df[
    [
        "Sample_ID",
        "Target",
        "Target_Name",
        "Dataset_Split",
        "Block_ID",
        "X",
        "Y",
        *predictor_columns
    ]
].copy()

test_predictions_df[
    "Predicted_Probability"
] = test_probability

test_predictions_df[
    "Predicted_Class"
] = test_prediction

test_predictions_df[
    "Predicted_Class_Name"
] = np.where(
    test_prediction == 1,
    "Urban expansion",
    "Stable non-built"
)

test_predictions_df[
    "Correct_Prediction"
] = (
    test_predictions_df[
        "Target"
    ]
    ==
    test_predictions_df[
        "Predicted_Class"
    ]
)

test_predictions_df[
    "Absolute_Probability_Error"
] = np.abs(
    test_predictions_df[
        "Target"
    ]
    -
    test_predictions_df[
        "Predicted_Probability"
    ]
)

# ------------------------------------------------------------------------------------------
# DISPLAY FINAL TEST RESULTS
# ------------------------------------------------------------------------------------------
print("\nFINAL INDEPENDENT TEST METRICS")
print("-" * 120)

print(
    f"Accuracy:                 {accuracy:.4f}"
)

print(
    f"Balanced accuracy:        {balanced_accuracy:.4f}"
)

print(
    f"Precision:                {precision:.4f}"
)

print(
    f"Recall / sensitivity:     {recall:.4f}"
)

print(
    f"Specificity:              {specificity:.4f}"
)

print(
    f"Negative predictive value:{negative_predictive_value:>10.4f}"
)

print(
    f"F1 score:                 {f1:.4f}"
)

print(
    f"ROC-AUC:                  {roc_auc:.4f}"
)

print(
    f"PR-AUC:                   {pr_auc:.4f}"
)

print(
    f"Matthews correlation:     {mcc:.4f}"
)

print(
    f"Cohen's kappa:            {cohen_kappa:.4f}"
)

print(
    f"Brier score:              {brier_score:.4f}"
)

print(
    f"Log loss:                 {test_log_loss:.4f}"
)

print("\nFINAL TEST CONFUSION MATRIX")
print("-" * 120)

print(
    f"True negatives:  {tn:,}"
)

print(
    f"False positives: {fp:,}"
)

print(
    f"False negatives: {fn:,}"
)

print(
    f"True positives:  {tp:,}"
)

# ------------------------------------------------------------------------------------------
# CREATE ROC CURVE
# ------------------------------------------------------------------------------------------
plt.figure(
    figsize=(8, 7)
)

plt.plot(
    roc_fpr,
    roc_tpr,
    linewidth=2,
    label=(
        f"{selected_model_name} "
        f"(AUC = {roc_auc:.3f})"
    )
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1.5,
    label="Random classifier"
)

plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.title(
    "Independent Test ROC Curve\n"
    "Urban Expansion Model — Enugu State"
)

plt.xlim(
    0,
    1
)

plt.ylim(
    0,
    1.02
)

plt.grid(
    alpha=0.25
)

plt.legend(
    loc="lower right"
)

plt.tight_layout()

plt.savefig(
    ROC_CHART_FILE,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

plt.close()

# ------------------------------------------------------------------------------------------
# CREATE PRECISION–RECALL CURVE
# ------------------------------------------------------------------------------------------
positive_prevalence = float(
    np.mean(
        y_test
    )
)

plt.figure(
    figsize=(8, 7)
)

plt.plot(
    pr_recall,
    pr_precision,
    linewidth=2,
    label=(
        f"{selected_model_name} "
        f"(AP = {pr_auc:.3f})"
    )
)

plt.axhline(
    positive_prevalence,
    linestyle="--",
    linewidth=1.5,
    label=(
        f"Baseline prevalence "
        f"({positive_prevalence:.2f})"
    )
)

plt.xlabel(
    "Recall"
)

plt.ylabel(
    "Precision"
)

plt.title(
    "Independent Test Precision–Recall Curve\n"
    "Urban Expansion Model — Enugu State"
)

plt.xlim(
    0,
    1
)

plt.ylim(
    0,
    1.02
)

plt.grid(
    alpha=0.25
)

plt.legend(
    loc="lower left"
)

plt.tight_layout()

plt.savefig(
    PR_CHART_FILE,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

plt.close()

# ------------------------------------------------------------------------------------------
# CREATE CALIBRATION CURVE
# ------------------------------------------------------------------------------------------
plt.figure(
    figsize=(8, 7)
)

plt.plot(
    calibration_mean_prediction,
    calibration_fraction_positive,
    marker="o",
    linewidth=2,
    label=selected_model_name
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1.5,
    label="Perfect calibration"
)

plt.xlabel(
    "Mean Predicted Probability"
)

plt.ylabel(
    "Observed Positive Fraction"
)

plt.title(
    "Independent Test Calibration Curve\n"
    "Urban Expansion Model — Enugu State"
)

plt.xlim(
    0,
    1
)

plt.ylim(
    0,
    1
)

plt.grid(
    alpha=0.25
)

plt.legend(
    loc="upper left"
)

plt.tight_layout()

plt.savefig(
    CALIBRATION_CHART_FILE,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

plt.close()

# ------------------------------------------------------------------------------------------
# CREATE CONFUSION-MATRIX CHART
# ------------------------------------------------------------------------------------------
confusion_array = np.array(
    [
        [tn, fp],
        [fn, tp]
    ]
)

plt.figure(
    figsize=(7, 6)
)

image = plt.imshow(
    confusion_array,
    interpolation="nearest"
)

plt.colorbar(
    image,
    fraction=0.046,
    pad=0.04
)

plt.xticks(
    [0, 1],
    [
        "Stable non-built",
        "Urban expansion"
    ],
    rotation=15
)

plt.yticks(
    [0, 1],
    [
        "Stable non-built",
        "Urban expansion"
    ]
)

plt.xlabel(
    "Predicted Class"
)

plt.ylabel(
    "Actual Class"
)

plt.title(
    "Independent Test Confusion Matrix\n"
    f"Threshold = {CLASSIFICATION_THRESHOLD:.2f}"
)

threshold_for_text = (
    confusion_array.max() / 2
)

for row_index in range(2):

    for column_index in range(2):

        value = int(
            confusion_array[
                row_index,
                column_index
            ]
        )

        plt.text(
            column_index,
            row_index,
            f"{value:,}",
            horizontalalignment="center",
            verticalalignment="center",
            color=(
                "white"
                if value > threshold_for_text
                else "black"
            ),
            fontsize=13,
            fontweight="bold"
        )

plt.tight_layout()

plt.savefig(
    CONFUSION_CHART_FILE,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

plt.close()

# ------------------------------------------------------------------------------------------
# CREATE PROBABILITY DISTRIBUTION CHART
# ------------------------------------------------------------------------------------------
negative_probabilities = test_probability[
    y_test == 0
]

positive_probabilities = test_probability[
    y_test == 1
]

plt.figure(
    figsize=(9, 7)
)

plt.hist(
    negative_probabilities,
    bins=30,
    alpha=0.60,
    density=True,
    label="Stable non-built"
)

plt.hist(
    positive_probabilities,
    bins=30,
    alpha=0.60,
    density=True,
    label="Urban expansion"
)

plt.axvline(
    CLASSIFICATION_THRESHOLD,
    linestyle="--",
    linewidth=2,
    label=(
        f"Decision threshold "
        f"({CLASSIFICATION_THRESHOLD:.2f})"
    )
)

plt.xlabel(
    "Predicted Expansion Probability"
)

plt.ylabel(
    "Density"
)

plt.title(
    "Independent Test Probability Distribution\n"
    "Urban Expansion Model — Enugu State"
)

plt.xlim(
    0,
    1
)

plt.grid(
    alpha=0.20
)

plt.legend()

plt.tight_layout()

plt.savefig(
    PROBABILITY_DISTRIBUTION_CHART_FILE,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

plt.close()

# ------------------------------------------------------------------------------------------
# SAVE TABLE OUTPUTS
# ------------------------------------------------------------------------------------------
test_predictions_df.to_csv(
    TEST_PREDICTIONS_FILE,
    index=False
)

metrics_df.to_csv(
    TEST_METRICS_FILE,
    index=False
)

confusion_df.to_csv(
    TEST_CONFUSION_FILE,
    index=False
)

threshold_df.to_csv(
    TEST_THRESHOLD_FILE,
    index=False
)

roc_curve_df.to_csv(
    ROC_CURVE_DATA_FILE,
    index=False
)

pr_curve_df.to_csv(
    PR_CURVE_DATA_FILE,
    index=False
)

calibration_df.to_csv(
    CALIBRATION_DATA_FILE,
    index=False
)

# ------------------------------------------------------------------------------------------
# SAVE FINAL TEST-EVALUATION REGISTER
# ------------------------------------------------------------------------------------------
evaluation_register = {
    "project":
        "Machine-Learning-Based Land Suitability Analysis "
        "for Sustainable Urban Development — Enugu",

    "prediction_target":
        "Urban expansion from 2020 to 2025",

    "evaluation_type":
        "Final independent spatial test evaluation",

    "selected_model":
        selected_model_name,

    "model_file":
        str(SELECTED_MODEL_FILE),

    "classification_threshold":
        CLASSIFICATION_THRESHOLD,

    "threshold_selection":
        "Fixed before test evaluation; no test-set optimisation",

    "test_sample_count":
        test_count,

    "test_negative_count":
        test_negative_count,

    "test_positive_count":
        test_positive_count,

    "independent_test_block_count":
        test_block_count,

    "retained_predictors":
        predictor_columns,

    "excluded_label_rule_predictor":
        excluded_rule_predictor,

    "final_test_metrics": {
        "accuracy":
            float(accuracy),

        "balanced_accuracy":
            float(balanced_accuracy),

        "precision":
            float(precision),

        "recall_sensitivity":
            float(recall),

        "specificity":
            float(specificity),

        "negative_predictive_value":
            float(negative_predictive_value),

        "f1_score":
            float(f1),

        "roc_auc":
            float(roc_auc),

        "pr_auc":
            float(pr_auc),

        "matthews_correlation":
            float(mcc),

        "cohen_kappa":
            float(cohen_kappa),

        "brier_score":
            float(brier_score),

        "log_loss":
            float(test_log_loss),

        "false_positive_rate":
            float(false_positive_rate_value),

        "false_negative_rate":
            float(false_negative_rate_value)
    },

    "confusion_matrix": {
        "true_negative":
            int(tn),

        "false_positive":
            int(fp),

        "false_negative":
            int(fn),

        "true_positive":
            int(tp)
    },

    "test_evaluation_status":
        "Completed",

    "model_retraining_performed":
        False,

    "test_threshold_optimisation_performed":
        False,

    "software": {
        "python":
            platform.python_version(),

        "scikit_learn":
            sklearn.__version__,

        "numpy":
            np.__version__,

        "pandas":
            pd.__version__
    }
}

with open(
    TEST_EVALUATION_REGISTER_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        evaluation_register,
        file,
        indent=4
    )

# ------------------------------------------------------------------------------------------
# REOPEN SAVED OUTPUTS
# ------------------------------------------------------------------------------------------
saved_predictions_df = pd.read_csv(
    TEST_PREDICTIONS_FILE
)

saved_metrics_df = pd.read_csv(
    TEST_METRICS_FILE
)

saved_prediction_count = int(
    saved_predictions_df.shape[0]
)

saved_correct_count = int(
    saved_predictions_df[
        "Correct_Prediction"
    ].sum()
)

recalculated_accuracy = (
    saved_correct_count /
    saved_prediction_count
)

saved_probability_valid = (
    saved_predictions_df[
        "Predicted_Probability"
    ]
    .between(
        0,
        1,
        inclusive="both"
    )
    .all()
)

# ------------------------------------------------------------------------------------------
# VALIDATION REGISTER
# ------------------------------------------------------------------------------------------
validation_df = pd.DataFrame([{
    "Selected_Model":
        selected_model_name,

    "Test_Samples":
        test_count,

    "Test_Negative_Samples":
        test_negative_count,

    "Test_Positive_Samples":
        test_positive_count,

    "Independent_Test_Blocks":
        test_block_count,

    "Predictor_Count":
        len(predictor_columns),

    "Classification_Threshold":
        CLASSIFICATION_THRESHOLD,

    "Accuracy":
        accuracy,

    "Balanced_Accuracy":
        balanced_accuracy,

    "Precision":
        precision,

    "Recall":
        recall,

    "Specificity":
        specificity,

    "F1_Score":
        f1,

    "ROC_AUC":
        roc_auc,

    "PR_AUC":
        pr_auc,

    "Matthews_Correlation":
        mcc,

    "Cohen_Kappa":
        cohen_kappa,

    "Brier_Score":
        brier_score,

    "Log_Loss":
        test_log_loss,

    "True_Negative":
        int(tn),

    "False_Positive":
        int(fp),

    "False_Negative":
        int(fn),

    "True_Positive":
        int(tp),

    "Test_Evaluation_Completed":
        True,

    "Model_Retraining_Performed":
        False,

    "Test_Threshold_Optimisation_Performed":
        False,

    "Suitability_Surface_Generation_Completed":
        False
}])

validation_df.to_csv(
    VALIDATION_FILE,
    index=False
)

# ------------------------------------------------------------------------------------------
# FINAL VALIDATION CHECKS
# ------------------------------------------------------------------------------------------
metric_values_to_check = [
    accuracy,
    balanced_accuracy,
    precision,
    recall,
    specificity,
    negative_predictive_value,
    f1,
    roc_auc,
    pr_auc,
    mcc,
    cohen_kappa,
    brier_score,
    test_log_loss
]

all_metrics_finite = all(
    np.isfinite(
        value
    )
    for value in metric_values_to_check
)

confusion_total_correct = (
    tn + fp + fn + tp
    ==
    test_count
)

saved_accuracy_matches = np.isclose(
    recalculated_accuracy,
    accuracy,
    atol=1e-12
)

checks = {
    "Selected leakage-corrected model loaded":
        selected_model is not None,

    "Exactly 3,600 test samples evaluated":
        test_count == 3_600,

    "Exactly 1,800 negative test samples evaluated":
        test_negative_count == 1_800,

    "Exactly 1,800 positive test samples evaluated":
        test_positive_count == 1_800,

    "Test dataset contains both classes":
        sorted(
            np.unique(
                y_test
            ).tolist()
        ) == [0, 1],

    "Seven leakage-corrected predictors used":
        len(
            predictor_columns
        ) == 7,

    "Distance-to-built rule predictor excluded":
        excluded_rule_predictor
        not in
        predictor_columns,

    "All test probabilities are finite":
        np.all(
            np.isfinite(
                test_probability
            )
        ),

    "All test probabilities are within 0–1":
        (
            np.min(
                test_probability
            ) >= 0
            and
            np.max(
                test_probability
            ) <= 1
        ),

    "All final test metrics are finite":
        all_metrics_finite,

    "Confusion-matrix total equals test sample count":
        confusion_total_correct,

    "No model retraining performed":
        validation_df[
            "Model_Retraining_Performed"
        ].iloc[0] == False,

    "No test-threshold optimisation performed":
        validation_df[
            "Test_Threshold_Optimisation_Performed"
        ].iloc[0] == False,

    "Saved prediction count is 3,600":
        saved_prediction_count == 3_600,

    "Saved probabilities are valid":
        saved_probability_valid,

    "Saved prediction accuracy matches reported accuracy":
        saved_accuracy_matches,

    "Final test prediction table saved":
        TEST_PREDICTIONS_FILE.exists(),

    "Final test metrics saved":
        TEST_METRICS_FILE.exists(),

    "Final confusion-matrix table saved":
        TEST_CONFUSION_FILE.exists(),

    "Fixed-threshold performance table saved":
        TEST_THRESHOLD_FILE.exists(),

    "ROC curve data saved":
        ROC_CURVE_DATA_FILE.exists(),

    "Precision–Recall curve data saved":
        PR_CURVE_DATA_FILE.exists(),

    "Calibration data saved":
        CALIBRATION_DATA_FILE.exists(),

    "ROC chart saved":
        ROC_CHART_FILE.exists(),

    "Precision–Recall chart saved":
        PR_CHART_FILE.exists(),

    "Calibration chart saved":
        CALIBRATION_CHART_FILE.exists(),

    "Confusion-matrix chart saved":
        CONFUSION_CHART_FILE.exists(),

    "Probability-distribution chart saved":
        PROBABILITY_DISTRIBUTION_CHART_FILE.exists(),

    "Final test-evaluation register saved":
        TEST_EVALUATION_REGISTER_FILE.exists(),

    "Stage validation register saved":
        VALIDATION_FILE.exists(),

    "Test evaluation marked complete":
        validation_df[
            "Test_Evaluation_Completed"
        ].iloc[0] == True,

    "Suitability-surface generation remains pending":
        validation_df[
            "Suitability_Surface_Generation_Completed"
        ].iloc[0] == False
}

print("\nVALIDATION SUMMARY")
print("-" * 120)

for label, passed in checks.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{label}"
    )

print("\nOUTPUT FILES")
print("-" * 120)

for path in [
    TEST_PREDICTIONS_FILE,
    TEST_METRICS_FILE,
    TEST_CONFUSION_FILE,
    TEST_THRESHOLD_FILE,
    ROC_CURVE_DATA_FILE,
    PR_CURVE_DATA_FILE,
    CALIBRATION_DATA_FILE,
    ROC_CHART_FILE,
    PR_CHART_FILE,
    CALIBRATION_CHART_FILE,
    CONFUSION_CHART_FILE,
    PROBABILITY_DISTRIBUTION_CHART_FILE,
    TEST_EVALUATION_REGISTER_FILE,
    VALIDATION_FILE
]:

    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{path.name}"
    )

if all(
    checks.values()
):

    print("\n✓ STAGE 10C FULLY PASSED")

    print(
        f"The leakage-corrected {selected_model_name} model "
        "has completed its final independent spatial test evaluation."
    )

    print(
        f"Final test ROC-AUC: {roc_auc:.4f}"
    )

    print(
        f"Final test PR-AUC: {pr_auc:.4f}"
    )

    print(
        f"Final test balanced accuracy: "
        f"{balanced_accuracy:.4f}"
    )

    print(
        f"Final test F1 score: {f1:.4f}"
    )

    print(
        "These are now the official unbiased performance "
        "statistics for the project."
    )

    print(
        "The next stage will examine predictor importance "
        "and model interpretation before producing the "
        "statewide suitability probability surface."
    )

else:

    print("\n⚠ STAGE 10C REQUIRES REVIEW")

print("=" * 120)

In [ ]:
# ==========================================================================================
# PROJECT 7 — STAGE 10D
# PREDICTOR IMPORTANCE AND MODEL INTERPRETATION
#
# This stage:
# - loads the fixed leakage-corrected Extra Trees model;
# - does not retrain, tune or alter the model;
# - extracts native Extra Trees impurity importance;
# - calculates permutation importance on validation and test datasets;
# - groups one-hot encoded baseline land-cover classes;
# - calculates partial-dependence response profiles for continuous predictors;
# - compares validation and test importance rankings;
# - saves tables, charts and a model-interpretation register.
#
# Predictor importance is descriptive. It is not interpreted as proof of causality.
# ==========================================================================================

from pathlib import Path
import json
import platform
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import sklearn

from sklearn.inspection import (
    permutation_importance,
    partial_dependence
)

from scipy.stats import spearmanr

warnings.filterwarnings(
    "ignore",
    category=FutureWarning
)

# ------------------------------------------------------------------------------------------
# PARAMETERS
# ------------------------------------------------------------------------------------------
RANDOM_SEED = 42
PERMUTATION_REPEATS = 15
PERMUTATION_SCORING = "roc_auc"

# ------------------------------------------------------------------------------------------
# PATHS
# ------------------------------------------------------------------------------------------
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

TRAINING_DIR = PROJECT_ROOT / "09_Training_Data"
MODEL_DIR = PROJECT_ROOT / "11_ML_Models"
TABLE_DIR = PROJECT_ROOT / "16_Tables"
CHART_DIR = PROJECT_ROOT / "15_Charts"
ADMIN_DIR = PROJECT_ROOT / "00_Project_Admin"

INPUT_CSV_FILE = (
    TRAINING_DIR /
    "Enugu_ML_Samples_With_Predictors_2020_2025.csv"
)

MODEL_FILE = (
    MODEL_DIR /
    "Enugu_Selected_Expansion_Model_Leakage_Corrected.joblib"
)

TEST_REGISTER_FILE = (
    ADMIN_DIR /
    "ML_Final_Independent_Test_Evaluation.json"
)

NATIVE_IMPORTANCE_FILE = (
    TABLE_DIR /
    "Enugu_ExtraTrees_Native_Feature_Importance.csv"
)

GROUPED_NATIVE_IMPORTANCE_FILE = (
    TABLE_DIR /
    "Enugu_ExtraTrees_Grouped_Native_Predictor_Importance.csv"
)

VALIDATION_PERMUTATION_FILE = (
    TABLE_DIR /
    "Enugu_Validation_Permutation_Importance.csv"
)

TEST_PERMUTATION_FILE = (
    TABLE_DIR /
    "Enugu_Test_Permutation_Importance.csv"
)

COMBINED_IMPORTANCE_FILE = (
    TABLE_DIR /
    "Enugu_Combined_Predictor_Importance_Ranking.csv"
)

PARTIAL_DEPENDENCE_FILE = (
    TABLE_DIR /
    "Enugu_Continuous_Predictor_Partial_Dependence.csv"
)

RANK_CONSISTENCY_FILE = (
    TABLE_DIR /
    "Enugu_Predictor_Importance_Rank_Consistency.csv"
)

NATIVE_IMPORTANCE_CHART = (
    CHART_DIR /
    "10D_Grouped_Native_Predictor_Importance.png"
)

PERMUTATION_IMPORTANCE_CHART = (
    CHART_DIR /
    "10D_Validation_Test_Permutation_Importance.png"
)

PARTIAL_DEPENDENCE_CHART = (
    CHART_DIR /
    "10D_Continuous_Predictor_Response_Profiles.png"
)

INTERPRETATION_REGISTER_FILE = (
    ADMIN_DIR /
    "ML_Model_Interpretation_Register.json"
)

VALIDATION_FILE = (
    TRAINING_DIR /
    "Stage_10D_Model_Interpretation_Validation.csv"
)

for folder in [
    TRAINING_DIR,
    MODEL_DIR,
    TABLE_DIR,
    CHART_DIR,
    ADMIN_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

print("=" * 120)
print("STAGE 10D — PREDICTOR IMPORTANCE AND MODEL INTERPRETATION")
print("=" * 120)

# ------------------------------------------------------------------------------------------
# REQUIRED FILE CHECK
# ------------------------------------------------------------------------------------------
required_files = {
    "Predictor-enriched sample table":
        INPUT_CSV_FILE,

    "Leakage-corrected selected model":
        MODEL_FILE,

    "Final independent test register":
        TEST_REGISTER_FILE
}

print("\nREQUIRED FILE CHECK")
print("-" * 120)

for label, path in required_files.items():

    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{label}: {path}"
    )

if not all(
    path.exists()
    for path in required_files.values()
):

    raise FileNotFoundError(
        "One or more required Stage 10D inputs are missing."
    )

# ------------------------------------------------------------------------------------------
# LOAD TEST REGISTER
# ------------------------------------------------------------------------------------------
with open(
    TEST_REGISTER_FILE,
    "r",
    encoding="utf-8"
) as file:

    test_register = json.load(
        file
    )

registered_model_name = test_register.get(
    "selected_model",
    "Unknown"
)

registered_test_status = test_register.get(
    "test_evaluation_status",
    "Unknown"
)

print("\nMODEL STATUS")
print("-" * 120)

print(
    f"Selected model: {registered_model_name}"
)

print(
    f"Final test status: {registered_test_status}"
)

if registered_test_status != "Completed":

    raise RuntimeError(
        "Final independent test evaluation has not been completed."
    )

# ------------------------------------------------------------------------------------------
# LOAD DATA
# ------------------------------------------------------------------------------------------
data = pd.read_csv(
    INPUT_CSV_FILE
)

continuous_predictors = [
    "Elevation_m",
    "Slope_deg",
    "Distance_to_Road_m",
    "Distance_to_Surface_Water_m",
    "Distance_to_Drainage_m",
    "Population_Density"
]

categorical_predictors = [
    "Baseline_LULC_2020"
]

predictor_columns = (
    continuous_predictors
    +
    categorical_predictors
)

excluded_rule_predictor = (
    "Distance_to_Built_2020_m"
)

required_columns = {
    "Sample_ID",
    "Target",
    "Dataset_Split",
    "Block_ID",
    "Predictor_Complete",
    *predictor_columns,
    excluded_rule_predictor
}

missing_columns = (
    required_columns -
    set(data.columns)
)

if missing_columns:

    raise RuntimeError(
        "Required interpretation columns are missing: "
        f"{sorted(missing_columns)}"
    )

data["Target"] = (
    data["Target"]
    .astype(np.uint8)
)

data["Baseline_LULC_2020"] = (
    data["Baseline_LULC_2020"]
    .astype(str)
)

if data["Predictor_Complete"].dtype == object:

    data["Predictor_Complete"] = (
        data["Predictor_Complete"]
        .astype(str)
        .str.lower()
        .map({
            "true": True,
            "false": False,
            "1": True,
            "0": False
        })
    )

if data["Predictor_Complete"].isna().any():

    raise RuntimeError(
        "Predictor_Complete contains unrecognised values."
    )

if not data["Predictor_Complete"].all():

    raise RuntimeError(
        "Incomplete predictor records remain."
    )

validation_df = data.loc[
    data["Dataset_Split"] == "Validation"
].copy()

test_df = data.loc[
    data["Dataset_Split"] == "Test"
].copy()

X_validation = validation_df[
    predictor_columns
].copy()

y_validation = validation_df[
    "Target"
].to_numpy(
    dtype=np.uint8
)

X_test = test_df[
    predictor_columns
].copy()

y_test = test_df[
    "Target"
].to_numpy(
    dtype=np.uint8
)

print("\nINTERPRETATION DATASETS")
print("-" * 120)

print(
    f"Validation records: {validation_df.shape[0]:,}"
)

print(
    f"Independent test records: {test_df.shape[0]:,}"
)

print(
    f"Retained predictors: {len(predictor_columns)}"
)

print(
    f"Excluded label-rule predictor: "
    f"{excluded_rule_predictor}"
)

# ------------------------------------------------------------------------------------------
# LOAD FIXED MODEL
# ------------------------------------------------------------------------------------------
model_pipeline = joblib.load(
    MODEL_FILE
)

if not hasattr(
    model_pipeline,
    "named_steps"
):

    raise RuntimeError(
        "The saved model is not a compatible sklearn Pipeline."
    )

if (
    "preprocessor"
    not in model_pipeline.named_steps
    or
    "classifier"
    not in model_pipeline.named_steps
):

    raise RuntimeError(
        "The model pipeline lacks expected preprocessing or classifier steps."
    )

preprocessor = model_pipeline.named_steps[
    "preprocessor"
]

classifier = model_pipeline.named_steps[
    "classifier"
]

if not hasattr(
    classifier,
    "feature_importances_"
):

    raise RuntimeError(
        "The selected classifier does not expose native feature importance."
    )

print("\nMODEL PIPELINE")
print("-" * 120)

print(
    f"Pipeline type: {type(model_pipeline).__name__}"
)

print(
    f"Classifier type: {type(classifier).__name__}"
)

print(
    "✓ The fixed selected model was loaded without retraining."
)

# ------------------------------------------------------------------------------------------
# EXTRACT TRANSFORMED FEATURE NAMES
# ------------------------------------------------------------------------------------------
try:

    transformed_feature_names = (
        preprocessor.get_feature_names_out()
        .tolist()
    )

except Exception:

    transformed_feature_names = []

    transformed_feature_names.extend(
        [
            f"continuous__{name}"
            for name in continuous_predictors
        ]
    )

    categorical_pipeline = (
        preprocessor
        .named_transformers_[
            "categorical"
        ]
    )

    onehot_encoder = (
        categorical_pipeline
        .named_steps[
            "onehot"
        ]
    )

    category_values = (
        onehot_encoder.categories_[0]
    )

    transformed_feature_names.extend(
        [
            f"categorical__Baseline_LULC_2020_{value}"
            for value in category_values
        ]
    )

native_importance_values = (
    classifier.feature_importances_
)

if (
    len(transformed_feature_names)
    !=
    len(native_importance_values)
):

    raise RuntimeError(
        "Transformed feature-name count does not match "
        "native importance count."
    )

# ------------------------------------------------------------------------------------------
# CLEAN TRANSFORMED FEATURE NAMES
# ------------------------------------------------------------------------------------------
def clean_feature_name(feature_name):

    cleaned = str(feature_name)

    cleaned = cleaned.replace(
        "continuous__",
        ""
    )

    cleaned = cleaned.replace(
        "categorical__",
        ""
    )

    return cleaned


cleaned_feature_names = [
    clean_feature_name(name)
    for name in transformed_feature_names
]

native_importance_df = pd.DataFrame({
    "Transformed_Feature":
        cleaned_feature_names,

    "Native_Importance":
        native_importance_values
})

native_importance_df[
    "Native_Importance_Percent"
] = (
    native_importance_df[
        "Native_Importance"
    ]
    *
    100
)

native_importance_df = (
    native_importance_df
    .sort_values(
        "Native_Importance",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

native_importance_df[
    "Native_Rank"
] = np.arange(
    1,
    native_importance_df.shape[0] + 1
)

# ------------------------------------------------------------------------------------------
# GROUP ONE-HOT LAND-COVER FEATURES
# ------------------------------------------------------------------------------------------
def original_predictor_group(
    transformed_feature_name
):

    name = str(
        transformed_feature_name
    )

    if name.startswith(
        "Baseline_LULC_2020"
    ):

        return "Baseline_LULC_2020"

    return name


native_importance_df[
    "Original_Predictor"
] = native_importance_df[
    "Transformed_Feature"
].apply(
    original_predictor_group
)

grouped_native_df = (
    native_importance_df
    .groupby(
        "Original_Predictor",
        as_index=False
    )
    .agg(
        Native_Importance=(
            "Native_Importance",
            "sum"
        ),

        Encoded_Feature_Count=(
            "Transformed_Feature",
            "count"
        )
    )
)

grouped_native_df[
    "Native_Importance_Percent"
] = (
    grouped_native_df[
        "Native_Importance"
    ]
    *
    100
)

grouped_native_df = (
    grouped_native_df
    .sort_values(
        "Native_Importance",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

grouped_native_df[
    "Native_Rank"
] = np.arange(
    1,
    grouped_native_df.shape[0] + 1
)

# ------------------------------------------------------------------------------------------
# PERMUTATION IMPORTANCE
#
# Permuting original input columns automatically evaluates baseline land cover
# as one grouped categorical predictor rather than separate one-hot categories.
# ------------------------------------------------------------------------------------------
print("\nPERMUTATION IMPORTANCE")
print("-" * 120)

print(
    f"Scoring metric: {PERMUTATION_SCORING}"
)

print(
    f"Repeats per predictor: {PERMUTATION_REPEATS}"
)

print(
    "Calculating validation permutation importance..."
)

validation_permutation = permutation_importance(
    model_pipeline,
    X_validation,
    y_validation,
    scoring=PERMUTATION_SCORING,
    n_repeats=PERMUTATION_REPEATS,
    random_state=RANDOM_SEED,
    n_jobs=-1
)

print(
    "Calculating independent-test permutation importance..."
)

test_permutation = permutation_importance(
    model_pipeline,
    X_test,
    y_test,
    scoring=PERMUTATION_SCORING,
    n_repeats=PERMUTATION_REPEATS,
    random_state=RANDOM_SEED,
    n_jobs=-1
)

validation_permutation_df = pd.DataFrame({
    "Predictor":
        predictor_columns,

    "Validation_Importance_Mean":
        validation_permutation.importances_mean,

    "Validation_Importance_Standard_Deviation":
        validation_permutation.importances_std
})

validation_permutation_df = (
    validation_permutation_df
    .sort_values(
        "Validation_Importance_Mean",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

validation_permutation_df[
    "Validation_Rank"
] = np.arange(
    1,
    validation_permutation_df.shape[0] + 1
)

test_permutation_df = pd.DataFrame({
    "Predictor":
        predictor_columns,

    "Test_Importance_Mean":
        test_permutation.importances_mean,

    "Test_Importance_Standard_Deviation":
        test_permutation.importances_std
})

test_permutation_df = (
    test_permutation_df
    .sort_values(
        "Test_Importance_Mean",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

test_permutation_df[
    "Test_Rank"
] = np.arange(
    1,
    test_permutation_df.shape[0] + 1
)

# ------------------------------------------------------------------------------------------
# COMBINE IMPORTANCE RESULTS
# ------------------------------------------------------------------------------------------
combined_importance_df = (
    grouped_native_df[
        [
            "Original_Predictor",
            "Native_Importance",
            "Native_Importance_Percent",
            "Native_Rank"
        ]
    ]
    .rename(
        columns={
            "Original_Predictor":
                "Predictor"
        }
    )
    .merge(
        validation_permutation_df,
        on="Predictor",
        how="outer"
    )
    .merge(
        test_permutation_df,
        on="Predictor",
        how="outer"
    )
)

combined_importance_df[
    "Mean_Permutation_Importance"
] = combined_importance_df[
    [
        "Validation_Importance_Mean",
        "Test_Importance_Mean"
    ]
].mean(
    axis=1
)

combined_importance_df[
    "Mean_Permutation_Rank"
] = combined_importance_df[
    "Mean_Permutation_Importance"
].rank(
    method="min",
    ascending=False
).astype(int)

combined_importance_df[
    "Validation_Test_Rank_Difference"
] = (
    combined_importance_df[
        "Validation_Rank"
    ]
    -
    combined_importance_df[
        "Test_Rank"
    ]
).abs()

combined_importance_df = (
    combined_importance_df
    .sort_values(
        [
            "Mean_Permutation_Rank",
            "Native_Rank"
        ]
    )
    .reset_index(
        drop=True
    )
)

# ------------------------------------------------------------------------------------------
# RANK CONSISTENCY
# ------------------------------------------------------------------------------------------
rank_consistency_df = combined_importance_df[
    [
        "Predictor",
        "Native_Rank",
        "Validation_Rank",
        "Test_Rank",
        "Validation_Test_Rank_Difference"
    ]
].copy()

validation_test_spearman = spearmanr(
    combined_importance_df[
        "Validation_Rank"
    ],
    combined_importance_df[
        "Test_Rank"
    ]
)

native_test_spearman = spearmanr(
    combined_importance_df[
        "Native_Rank"
    ],
    combined_importance_df[
        "Test_Rank"
    ]
)

rank_consistency_summary = pd.DataFrame([
    {
        "Rank_Comparison":
            "Validation permutation vs test permutation",

        "Spearman_Rho":
            float(validation_test_spearman.statistic),

        "P_Value":
            float(validation_test_spearman.pvalue)
    },
    {
        "Rank_Comparison":
            "Native importance vs test permutation",

        "Spearman_Rho":
            float(native_test_spearman.statistic),

        "P_Value":
            float(native_test_spearman.pvalue)
    }
])

# ------------------------------------------------------------------------------------------
# PARTIAL DEPENDENCE
#
# The profiles are descriptive average model responses.
# They do not establish causal relationships.
# ------------------------------------------------------------------------------------------
print("\nPARTIAL DEPENDENCE")
print("-" * 120)

partial_dependence_records = []

for predictor in continuous_predictors:

    print(
        f"Calculating response profile: {predictor}"
    )

    predictor_index = predictor_columns.index(
        predictor
    )

    pd_result = partial_dependence(
        model_pipeline,
        X_validation,
        features=[
            predictor_index
        ],
        kind="average",
        grid_resolution=25,
        percentiles=(
            0.05,
            0.95
        )
    )

    grid_values = (
        pd_result[
            "grid_values"
        ][0]
        if "grid_values" in pd_result
        else
        pd_result[
            "values"
        ][0]
    )

    average_response = (
        pd_result[
            "average"
        ][0]
    )

    for predictor_value, response_value in zip(
        grid_values,
        average_response
    ):

        partial_dependence_records.append({
            "Predictor":
                predictor,

            "Predictor_Value":
                float(predictor_value),

            "Average_Model_Response":
                float(response_value)
        })

partial_dependence_df = pd.DataFrame(
    partial_dependence_records
)

# ------------------------------------------------------------------------------------------
# DISPLAY IMPORTANCE RESULTS
# ------------------------------------------------------------------------------------------
print("\nGROUPED NATIVE FEATURE IMPORTANCE")
print("-" * 120)

print(
    grouped_native_df[
        [
            "Native_Rank",
            "Original_Predictor",
            "Native_Importance",
            "Native_Importance_Percent",
            "Encoded_Feature_Count"
        ]
    ].to_string(
        index=False,
        formatters={
            "Native_Importance":
                lambda value: f"{value:.5f}",

            "Native_Importance_Percent":
                lambda value: f"{value:.2f}%"
        }
    )
)

print("\nVALIDATION PERMUTATION IMPORTANCE")
print("-" * 120)

print(
    validation_permutation_df.to_string(
        index=False,
        formatters={
            "Validation_Importance_Mean":
                lambda value: f"{value:.5f}",

            "Validation_Importance_Standard_Deviation":
                lambda value: f"{value:.5f}"
        }
    )
)

print("\nINDEPENDENT TEST PERMUTATION IMPORTANCE")
print("-" * 120)

print(
    test_permutation_df.to_string(
        index=False,
        formatters={
            "Test_Importance_Mean":
                lambda value: f"{value:.5f}",

            "Test_Importance_Standard_Deviation":
                lambda value: f"{value:.5f}"
        }
    )
)

print("\nIMPORTANCE-RANK CONSISTENCY")
print("-" * 120)

print(
    rank_consistency_summary.to_string(
        index=False,
        formatters={
            "Spearman_Rho":
                lambda value: f"{value:.4f}",

            "P_Value":
                lambda value: f"{value:.4f}"
        }
    )
)

# ------------------------------------------------------------------------------------------
# SAVE DATA TABLES
# ------------------------------------------------------------------------------------------
native_importance_df.to_csv(
    NATIVE_IMPORTANCE_FILE,
    index=False
)

grouped_native_df.to_csv(
    GROUPED_NATIVE_IMPORTANCE_FILE,
    index=False
)

validation_permutation_df.to_csv(
    VALIDATION_PERMUTATION_FILE,
    index=False
)

test_permutation_df.to_csv(
    TEST_PERMUTATION_FILE,
    index=False
)

combined_importance_df.to_csv(
    COMBINED_IMPORTANCE_FILE,
    index=False
)

partial_dependence_df.to_csv(
    PARTIAL_DEPENDENCE_FILE,
    index=False
)

rank_consistency_df.to_csv(
    RANK_CONSISTENCY_FILE,
    index=False
)

# ------------------------------------------------------------------------------------------
# CHART 1 — GROUPED NATIVE IMPORTANCE
# ------------------------------------------------------------------------------------------
native_chart_df = (
    grouped_native_df
    .sort_values(
        "Native_Importance",
        ascending=True
    )
)

plt.figure(
    figsize=(10, 7)
)

plt.barh(
    native_chart_df[
        "Original_Predictor"
    ],
    native_chart_df[
        "Native_Importance_Percent"
    ]
)

plt.xlabel(
    "Native Extra Trees Importance (%)"
)

plt.ylabel(
    "Predictor"
)

plt.title(
    "Grouped Native Predictor Importance\n"
    "Leakage-Corrected Extra Trees Model"
)

plt.grid(
    axis="x",
    alpha=0.25
)

plt.tight_layout()

plt.savefig(
    NATIVE_IMPORTANCE_CHART,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

plt.close()

# ------------------------------------------------------------------------------------------
# CHART 2 — VALIDATION AND TEST PERMUTATION IMPORTANCE
# ------------------------------------------------------------------------------------------
permutation_chart_df = (
    combined_importance_df
    .sort_values(
        "Mean_Permutation_Importance",
        ascending=True
    )
)

chart_y = np.arange(
    permutation_chart_df.shape[0]
)

bar_height = 0.36

plt.figure(
    figsize=(11, 7)
)

plt.barh(
    chart_y - bar_height / 2,
    permutation_chart_df[
        "Validation_Importance_Mean"
    ],
    height=bar_height,
    xerr=permutation_chart_df[
        "Validation_Importance_Standard_Deviation"
    ],
    label="Validation"
)

plt.barh(
    chart_y + bar_height / 2,
    permutation_chart_df[
        "Test_Importance_Mean"
    ],
    height=bar_height,
    xerr=permutation_chart_df[
        "Test_Importance_Standard_Deviation"
    ],
    label="Independent test"
)

plt.yticks(
    chart_y,
    permutation_chart_df[
        "Predictor"
    ]
)

plt.axvline(
    0,
    linewidth=1
)

plt.xlabel(
    "Decrease in ROC-AUC After Permutation"
)

plt.ylabel(
    "Predictor"
)

plt.title(
    "Validation and Independent-Test Permutation Importance\n"
    "Urban Expansion Model — Enugu State"
)

plt.grid(
    axis="x",
    alpha=0.25
)

plt.legend()

plt.tight_layout()

plt.savefig(
    PERMUTATION_IMPORTANCE_CHART,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

plt.close()

# ------------------------------------------------------------------------------------------
# CHART 3 — PARTIAL-DEPENDENCE RESPONSE PROFILES
# ------------------------------------------------------------------------------------------
figure, axes = plt.subplots(
    nrows=3,
    ncols=2,
    figsize=(14, 14)
)

axes = axes.flatten()

for axis, predictor in zip(
    axes,
    continuous_predictors
):

    predictor_profile = partial_dependence_df.loc[
        partial_dependence_df[
            "Predictor"
        ] == predictor
    ]

    axis.plot(
        predictor_profile[
            "Predictor_Value"
        ],
        predictor_profile[
            "Average_Model_Response"
        ],
        linewidth=2
    )

    axis.set_xlabel(
        predictor
    )

    axis.set_ylabel(
        "Average predicted response"
    )

    axis.grid(
        alpha=0.25
    )

    axis.set_title(
        predictor.replace(
            "_",
            " "
        )
    )

figure.suptitle(
    "Continuous Predictor Response Profiles\n"
    "Leakage-Corrected Extra Trees Model",
    fontsize=16
)

figure.tight_layout(
    rect=[
        0,
        0,
        1,
        0.96
    ]
)

figure.savefig(
    PARTIAL_DEPENDENCE_CHART,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

plt.close(
    figure
)

# ------------------------------------------------------------------------------------------
# IDENTIFY TOP PREDICTORS
# ------------------------------------------------------------------------------------------
top_native_predictor = str(
    grouped_native_df.iloc[0][
        "Original_Predictor"
    ]
)

top_validation_predictor = str(
    validation_permutation_df.iloc[0][
        "Predictor"
    ]
)

top_test_predictor = str(
    test_permutation_df.iloc[0][
        "Predictor"
    ]
)

negative_test_importance_count = int(
    (
        test_permutation_df[
            "Test_Importance_Mean"
        ]
        <
        0
    ).sum()
)

# ------------------------------------------------------------------------------------------
# MODEL-INTERPRETATION REGISTER
# ------------------------------------------------------------------------------------------
interpretation_register = {
    "project":
        "Machine-Learning-Based Land Suitability Analysis "
        "for Sustainable Urban Development — Enugu",

    "selected_model":
        registered_model_name,

    "model_file":
        str(MODEL_FILE),

    "model_retrained":
        False,

    "model_tuned":
        False,

    "retained_predictors":
        predictor_columns,

    "excluded_label_rule_predictor":
        excluded_rule_predictor,

    "importance_methods": [
        "Extra Trees native impurity importance",
        "Validation permutation importance using ROC-AUC",
        "Independent-test permutation importance using ROC-AUC"
    ],

    "permutation_repeats":
        PERMUTATION_REPEATS,

    "partial_dependence_dataset":
        "Validation",

    "top_native_predictor":
        top_native_predictor,

    "top_validation_permutation_predictor":
        top_validation_predictor,

    "top_test_permutation_predictor":
        top_test_predictor,

    "validation_test_rank_spearman_rho":
        float(
            validation_test_spearman.statistic
        ),

    "native_test_rank_spearman_rho":
        float(
            native_test_spearman.statistic
        ),

    "negative_test_permutation_importance_count":
        negative_test_importance_count,

    "interpretation_caution":
        "Importance and partial dependence describe model behaviour "
        "and predictive association, not causal effects.",

    "statewide_probability_surface_status":
        "Pending",

    "software": {
        "python":
            platform.python_version(),

        "scikit_learn":
            sklearn.__version__,

        "numpy":
            np.__version__,

        "pandas":
            pd.__version__
    }
}

with open(
    INTERPRETATION_REGISTER_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        interpretation_register,
        file,
        indent=4
    )

# ------------------------------------------------------------------------------------------
# REOPEN SAVED TABLES
# ------------------------------------------------------------------------------------------
saved_combined_df = pd.read_csv(
    COMBINED_IMPORTANCE_FILE
)

saved_partial_dependence_df = pd.read_csv(
    PARTIAL_DEPENDENCE_FILE
)

saved_predictor_count = int(
    saved_combined_df[
        "Predictor"
    ].nunique()
)

saved_partial_predictor_count = int(
    saved_partial_dependence_df[
        "Predictor"
    ].nunique()
)

native_importance_sum = float(
    native_importance_df[
        "Native_Importance"
    ].sum()
)

grouped_native_importance_sum = float(
    grouped_native_df[
        "Native_Importance"
    ].sum()
)

# ------------------------------------------------------------------------------------------
# VALIDATION REGISTER
# ------------------------------------------------------------------------------------------
validation_register_df = pd.DataFrame([{
    "Selected_Model":
        registered_model_name,

    "Retained_Predictor_Count":
        len(predictor_columns),

    "Continuous_Predictor_Count":
        len(continuous_predictors),

    "Categorical_Predictor_Count":
        len(categorical_predictors),

    "Transformed_Feature_Count":
        len(transformed_feature_names),

    "Permutation_Repeats":
        PERMUTATION_REPEATS,

    "Validation_Samples":
        int(validation_df.shape[0]),

    "Test_Samples":
        int(test_df.shape[0]),

    "Top_Native_Predictor":
        top_native_predictor,

    "Top_Validation_Permutation_Predictor":
        top_validation_predictor,

    "Top_Test_Permutation_Predictor":
        top_test_predictor,

    "Validation_Test_Rank_Spearman":
        float(
            validation_test_spearman.statistic
        ),

    "Native_Test_Rank_Spearman":
        float(
            native_test_spearman.statistic
        ),

    "Native_Importance_Sum":
        native_importance_sum,

    "Grouped_Native_Importance_Sum":
        grouped_native_importance_sum,

    "Model_Retraining_Performed":
        False,

    "Model_Tuning_Performed":
        False,

    "Model_Interpretation_Completed":
        True,

    "Statewide_Probability_Surface_Completed":
        False
}])

validation_register_df.to_csv(
    VALIDATION_FILE,
    index=False
)

# ------------------------------------------------------------------------------------------
# FINAL VALIDATION
# ------------------------------------------------------------------------------------------
all_validation_importance_finite = np.all(
    np.isfinite(
        validation_permutation_df[
            "Validation_Importance_Mean"
        ]
    )
)

all_test_importance_finite = np.all(
    np.isfinite(
        test_permutation_df[
            "Test_Importance_Mean"
        ]
    )
)

all_partial_dependence_finite = np.all(
    np.isfinite(
        partial_dependence_df[
            [
                "Predictor_Value",
                "Average_Model_Response"
            ]
        ].to_numpy()
    )
)

checks = {
    "Fixed selected model loaded":
        model_pipeline is not None,

    "Selected classifier is Extra Trees":
        type(classifier).__name__
        ==
        "ExtraTreesClassifier",

    "Validation dataset contains 3,600 samples":
        validation_df.shape[0] == 3_600,

    "Independent test dataset contains 3,600 samples":
        test_df.shape[0] == 3_600,

    "Seven retained predictors interpreted":
        saved_predictor_count == 7,

    "All six continuous predictors have response profiles":
        saved_partial_predictor_count == 6,

    "Distance-to-built rule predictor remains excluded":
        excluded_rule_predictor
        not in
        predictor_columns,

    "Native feature importance sums to one":
        np.isclose(
            native_importance_sum,
            1.0,
            atol=1e-8
        ),

    "Grouped native importance sums to one":
        np.isclose(
            grouped_native_importance_sum,
            1.0,
            atol=1e-8
        ),

    "Validation permutation importance is finite":
        all_validation_importance_finite,

    "Test permutation importance is finite":
        all_test_importance_finite,

    "Partial-dependence values are finite":
        all_partial_dependence_finite,

    "Validation-test rank correlation is finite":
        np.isfinite(
            validation_test_spearman.statistic
        ),

    "Native-test rank correlation is finite":
        np.isfinite(
            native_test_spearman.statistic
        ),

    "No model retraining performed":
        validation_register_df[
            "Model_Retraining_Performed"
        ].iloc[0] == False,

    "No model tuning performed":
        validation_register_df[
            "Model_Tuning_Performed"
        ].iloc[0] == False,

    "Native feature-importance table saved":
        NATIVE_IMPORTANCE_FILE.exists(),

    "Grouped native importance table saved":
        GROUPED_NATIVE_IMPORTANCE_FILE.exists(),

    "Validation permutation table saved":
        VALIDATION_PERMUTATION_FILE.exists(),

    "Test permutation table saved":
        TEST_PERMUTATION_FILE.exists(),

    "Combined importance ranking saved":
        COMBINED_IMPORTANCE_FILE.exists(),

    "Partial-dependence table saved":
        PARTIAL_DEPENDENCE_FILE.exists(),

    "Rank-consistency table saved":
        RANK_CONSISTENCY_FILE.exists(),

    "Native importance chart saved":
        NATIVE_IMPORTANCE_CHART.exists(),

    "Permutation importance chart saved":
        PERMUTATION_IMPORTANCE_CHART.exists(),

    "Partial-dependence chart saved":
        PARTIAL_DEPENDENCE_CHART.exists(),

    "Interpretation register saved":
        INTERPRETATION_REGISTER_FILE.exists(),

    "Validation register saved":
        VALIDATION_FILE.exists(),

    "Model interpretation marked complete":
        validation_register_df[
            "Model_Interpretation_Completed"
        ].iloc[0] == True,

    "Statewide probability surface remains pending":
        validation_register_df[
            "Statewide_Probability_Surface_Completed"
        ].iloc[0] == False
}

print("\nVALIDATION SUMMARY")
print("-" * 120)

for label, passed in checks.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{label}"
    )

print("\nOUTPUT FILES")
print("-" * 120)

for path in [
    NATIVE_IMPORTANCE_FILE,
    GROUPED_NATIVE_IMPORTANCE_FILE,
    VALIDATION_PERMUTATION_FILE,
    TEST_PERMUTATION_FILE,
    COMBINED_IMPORTANCE_FILE,
    PARTIAL_DEPENDENCE_FILE,
    RANK_CONSISTENCY_FILE,
    NATIVE_IMPORTANCE_CHART,
    PERMUTATION_IMPORTANCE_CHART,
    PARTIAL_DEPENDENCE_CHART,
    INTERPRETATION_REGISTER_FILE,
    VALIDATION_FILE
]:

    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{path.name}"
    )

if all(
    checks.values()
):

    print("\n✓ STAGE 10D FULLY PASSED")

    print(
        f"Top native predictor: "
        f"{top_native_predictor}"
    )

    print(
        f"Top validation permutation predictor: "
        f"{top_validation_predictor}"
    )

    print(
        f"Top independent-test permutation predictor: "
        f"{top_test_predictor}"
    )

    print(
        f"Validation-test rank correlation: "
        f"{validation_test_spearman.statistic:.4f}"
    )

    print(
        "The fixed model has been interpreted without "
        "retraining or test-driven optimisation."
    )

    print(
        "The next stage will apply the validated model "
        "across the Enugu reference grid to generate the "
        "statewide urban-expansion probability surface."
    )

else:

    print("\n⚠ STAGE 10D REQUIRES REVIEW")

print("=" * 120)

In [ ]:
# ==========================================================================================
# PROJECT 7 — STAGE 11A-FIX 1
# LOCATE AND INSPECT POSSIBLE SLOPE RASTERS
#
# This cell:
# - searches the entire project for GeoTIFFs;
# - prioritises terrain-related folders and filenames;
# - inspects raster metadata and sampled value ranges;
# - identifies likely slope rasters without modifying any file.
# ==========================================================================================

from pathlib import Path
import re

import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

print("=" * 120)
print("STAGE 11A-FIX 1 — LOCATE POSSIBLE SLOPE RASTER")
print("=" * 120)

# ------------------------------------------------------------------------------------------
# FIND ALL GEOTIFFS
# ------------------------------------------------------------------------------------------
all_tifs = sorted(
    list(PROJECT_ROOT.rglob("*.tif"))
    +
    list(PROJECT_ROOT.rglob("*.tiff"))
)

print("\nPROJECT GEOTIFF INVENTORY")
print("-" * 120)

print(
    f"Total GeoTIFF files found: {len(all_tifs):,}"
)

if len(all_tifs) == 0:

    raise FileNotFoundError(
        "No GeoTIFF files were found anywhere in the project."
    )

# ------------------------------------------------------------------------------------------
# RANK POSSIBLE TERRAIN / SLOPE FILES
# ------------------------------------------------------------------------------------------
def normalise_text(path):

    return re.sub(
        r"[^a-z0-9]+",
        " ",
        str(path).lower()
    )


terrain_terms = [
    "slope",
    "terrain",
    "dem",
    "elevation",
    "copernicus",
    "glo30",
    "topography",
    "relief",
    "degree",
    "degrees"
]

candidate_records = []

for path in all_tifs:

    text = normalise_text(
        path
    )

    score = 0
    matched_terms = []

    for term in terrain_terms:

        if term in text:

            matched_terms.append(
                term
            )

            if term == "slope":
                score += 100

            elif term in [
                "degree",
                "degrees"
            ]:
                score += 40

            elif term in [
                "terrain",
                "topography"
            ]:
                score += 25

            elif term in [
                "dem",
                "elevation",
                "copernicus",
                "glo30"
            ]:
                score += 10

    if (
        "03 terrain" in text
        or
        "terrain" in text
    ):

        score += 30

    if score > 0:

        candidate_records.append({
            "Path":
                path,

            "Filename":
                path.name,

            "Parent_Folder":
                path.parent.name,

            "Resolver_Score":
                score,

            "Matched_Terms":
                ", ".join(
                    matched_terms
                )
        })

candidate_df = pd.DataFrame(
    candidate_records
)

if candidate_df.empty:

    print(
        "No filename-based terrain candidates were found."
    )

    print(
        "\nALL GEOTIFF FILENAMES"
    )

    print("-" * 120)

    for index, path in enumerate(
        all_tifs,
        start=1
    ):

        print(
            f"{index:03d}. {path}"
        )

    raise RuntimeError(
        "No terrain-related filenames were detected. "
        "Review the complete GeoTIFF list printed above."
    )

candidate_df = (
    candidate_df
    .sort_values(
        [
            "Resolver_Score",
            "Filename"
        ],
        ascending=[
            False,
            True
        ]
    )
    .reset_index(
        drop=True
    )
)

# ------------------------------------------------------------------------------------------
# INSPECT CANDIDATE RASTER VALUES
#
# Slope in degrees is expected to have:
# - minimum near 0;
# - maximum commonly between about 20 and 90;
# - no strongly negative values;
# - mean commonly below about 20 for this project.
# ------------------------------------------------------------------------------------------
inspection_records = []

for _, record in candidate_df.iterrows():

    path = Path(
        record["Path"]
    )

    try:

        with rasterio.open(
            path
        ) as source:

            sampled = source.read(
                1,
                out_shape=(
                    min(
                        source.height,
                        500
                    ),
                    min(
                        source.width,
                        500
                    )
                ),
                resampling=Resampling.nearest,
                masked=True
            )

            values = sampled.compressed().astype(
                np.float64
            )

            values = values[
                np.isfinite(
                    values
                )
            ]

            if values.size > 0:

                minimum = float(
                    np.min(
                        values
                    )
                )

                maximum = float(
                    np.max(
                        values
                    )
                )

                mean = float(
                    np.mean(
                        values
                    )
                )

                median = float(
                    np.median(
                        values
                    )
                )

                percentile_95 = float(
                    np.percentile(
                        values,
                        95
                    )
                )

            else:

                minimum = np.nan
                maximum = np.nan
                mean = np.nan
                median = np.nan
                percentile_95 = np.nan

            likely_slope_by_values = (
                np.isfinite(
                    minimum
                )
                and
                np.isfinite(
                    maximum
                )
                and
                minimum >= -0.01
                and
                maximum > 5
                and
                maximum <= 90.5
                and
                mean <= 20
            )

            filename_contains_slope = (
                "slope"
                in
                normalise_text(
                    path
                )
            )

            likely_slope = (
                filename_contains_slope
                or
                likely_slope_by_values
            )

            inspection_records.append({
                "Likely_Slope":
                    likely_slope,

                "Filename":
                    path.name,

                "Parent_Folder":
                    path.parent.name,

                "Width":
                    source.width,

                "Height":
                    source.height,

                "Bands":
                    source.count,

                "CRS":
                    str(
                        source.crs
                    ),

                "Data_Type":
                    source.dtypes[0],

                "NoData":
                    source.nodata,

                "Sample_Min":
                    minimum,

                "Sample_Median":
                    median,

                "Sample_Mean":
                    mean,

                "Sample_95th_Percentile":
                    percentile_95,

                "Sample_Max":
                    maximum,

                "Resolver_Score":
                    record[
                        "Resolver_Score"
                    ],

                "Matched_Terms":
                    record[
                        "Matched_Terms"
                    ],

                "Full_Path":
                    str(
                        path
                    )
            })

    except Exception as error:

        inspection_records.append({
            "Likely_Slope":
                False,

            "Filename":
                path.name,

            "Parent_Folder":
                path.parent.name,

            "Width":
                np.nan,

            "Height":
                np.nan,

            "Bands":
                np.nan,

            "CRS":
                "READ ERROR",

            "Data_Type":
                "READ ERROR",

            "NoData":
                np.nan,

            "Sample_Min":
                np.nan,

            "Sample_Median":
                np.nan,

            "Sample_Mean":
                np.nan,

            "Sample_95th_Percentile":
                np.nan,

            "Sample_Max":
                np.nan,

            "Resolver_Score":
                record[
                    "Resolver_Score"
                ],

            "Matched_Terms":
                record[
                    "Matched_Terms"
                ],

            "Full_Path":
                str(
                    path
                ),

            "Read_Error":
                str(
                    error
                )
        })

inspection_df = pd.DataFrame(
    inspection_records
)

inspection_df = (
    inspection_df
    .sort_values(
        [
            "Likely_Slope",
            "Resolver_Score",
            "Sample_Max"
        ],
        ascending=[
            False,
            False,
            False
        ]
    )
    .reset_index(
        drop=True
    )
)

# ------------------------------------------------------------------------------------------
# DISPLAY RESULTS
# ------------------------------------------------------------------------------------------
print("\nPOSSIBLE TERRAIN AND SLOPE RASTERS")
print("-" * 120)

display_columns = [
    "Likely_Slope",
    "Filename",
    "Parent_Folder",
    "Width",
    "Height",
    "Bands",
    "CRS",
    "Sample_Min",
    "Sample_Median",
    "Sample_Mean",
    "Sample_95th_Percentile",
    "Sample_Max",
    "Full_Path"
]

print(
    inspection_df[
        display_columns
    ].to_string(
        index=False,
        formatters={
            "Sample_Min":
                lambda value: (
                    f"{value:.4f}"
                    if pd.notna(value)
                    else "NA"
                ),

            "Sample_Median":
                lambda value: (
                    f"{value:.4f}"
                    if pd.notna(value)
                    else "NA"
                ),

            "Sample_Mean":
                lambda value: (
                    f"{value:.4f}"
                    if pd.notna(value)
                    else "NA"
                ),

            "Sample_95th_Percentile":
                lambda value: (
                    f"{value:.4f}"
                    if pd.notna(value)
                    else "NA"
                ),

            "Sample_Max":
                lambda value: (
                    f"{value:.4f}"
                    if pd.notna(value)
                    else "NA"
                )
        }
    )
)

likely_slope_df = inspection_df.loc[
    inspection_df[
        "Likely_Slope"
    ]
].copy()

print("\nLIKELY SLOPE RESULT")
print("-" * 120)

if likely_slope_df.shape[0] == 1:

    selected_slope_path = likely_slope_df.iloc[0][
        "Full_Path"
    ]

    print(
        "✓ One likely slope raster was identified:"
    )

    print(
        selected_slope_path
    )

elif likely_slope_df.shape[0] > 1:

    print(
        f"⚠ {likely_slope_df.shape[0]} possible slope rasters were identified."
    )

    print(
        "Their full paths are listed above."
    )

else:

    print(
        "✗ No raster could yet be confirmed as slope."
    )

    print(
        "The terrain candidates and sampled ranges are listed above."
    )

print("=" * 120)

In [ ]:
# ------------------------------------------------------------------------------------------
# CONFIRMED SLOPE RASTER
# Explicitly assigned after Stage 11A-FIX 1 validation.
# ------------------------------------------------------------------------------------------
CONFIRMED_SLOPE_FILE = (
    PROJECT_ROOT /
    "02_Terrain" /
    "Enugu_Slope_Degrees_30m.tif"
)

if not CONFIRMED_SLOPE_FILE.exists():

    raise FileNotFoundError(
        f"Confirmed slope raster is missing: {CONFIRMED_SLOPE_FILE}"
    )

with rasterio.open(
    CONFIRMED_SLOPE_FILE
) as slope_source:

    if slope_source.count != 1:

        raise RuntimeError(
            "The confirmed slope raster must contain exactly one band."
        )

    if slope_source.crs is None:

        raise RuntimeError(
            "The confirmed slope raster has no CRS."
        )

resolved["Slope_deg"] = {
    "label":
        "Slope",

    "path":
        CONFIRMED_SLOPE_FILE,

    "score":
        999,

    "candidate_count":
        1,

    "top_candidates": [
        str(CONFIRMED_SLOPE_FILE)
    ]
}

In [ ]:
# ==========================================================================================
# PROJECT 7 — STAGE 11A
# STATEWIDE URBAN-EXPANSION PROBABILITY, UNCERTAINTY AND SUITABILITY SURFACES
#
# Study area: Enugu State, Nigeria
#
# Methodological safeguards:
#   1. Loads the fixed leakage-corrected Extra Trees model.
#   2. Does not retrain, tune or modify the model.
#   3. Uses only the seven approved leakage-safe predictors.
#   4. Explicitly reads Band 1 of the 2020 Dynamic World raster.
#   5. Uses nearest-neighbour resampling for categorical land cover.
#   6. Uses bilinear resampling for continuous predictors.
#   7. Excludes existing 2020 built-up land, water and steep terrain from suitability.
#   8. Produces probability, uncertainty, confidence, applicability and suitability outputs.
# ==========================================================================================

# ==========================================================================================
# 1. IMPORTS
# ==========================================================================================

from pathlib import Path
from contextlib import ExitStack
from datetime import datetime, timezone
import gc
import json
import math
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio

from matplotlib.colors import BoundaryNorm, ListedColormap
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT
from rasterio.windows import Window

warnings.filterwarnings("ignore")


# ==========================================================================================
# 2. PROJECT PATHS
# ==========================================================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

MODEL_FILE = (
    PROJECT_ROOT
    / "11_ML_Models"
    / "Enugu_Selected_Expansion_Model_Leakage_Corrected.joblib"
)

INTERPRETATION_REGISTER_FILE = (
    PROJECT_ROOT
    / "00_Project_Admin"
    / "ML_Model_Interpretation_Register.json"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "12_Statewide_Suitability"
)

ADMIN_DIR = (
    PROJECT_ROOT
    / "00_Project_Admin"
)

TABLE_DIR = (
    OUTPUT_DIR
    / "Tables"
)

MAP_DIR = (
    OUTPUT_DIR
    / "Maps"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MAP_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ==========================================================================================
# 3. FIXED INPUT RASTERS
# ==========================================================================================

RASTER_INPUTS = {
    "Elevation_m": {
        "path": (
            PROJECT_ROOT
            / "02_Terrain"
            / "Enugu_Elevation_30m.tif"
        ),
        "band": 1,
        "resampling": Resampling.bilinear,
        "categorical": False
    },

    "Slope_deg": {
        "path": (
            PROJECT_ROOT
            / "02_Terrain"
            / "Enugu_Slope_Degrees_30m.tif"
        ),
        "band": 1,
        "resampling": Resampling.bilinear,
        "categorical": False
    },

    "Distance_to_Road_m": {
        "path": (
            PROJECT_ROOT
            / "05_Roads_Accessibility"
            / "Enugu_Distance_to_Drive_Road_30m.tif"
        ),
        "band": 1,
        "resampling": Resampling.bilinear,
        "categorical": False
    },

    "Distance_to_Surface_Water_m": {
        "path": (
            PROJECT_ROOT
            / "06_Environmental_Constraints"
            / "Enugu_Distance_to_Recurring_SurfaceWater_30m.tif"
        ),
        "band": 1,
        "resampling": Resampling.bilinear,
        "categorical": False
    },

    "Distance_to_Drainage_m": {
        "path": (
            PROJECT_ROOT
            / "06_Environmental_Constraints"
            / "Enugu_Distance_to_Drainage_30m.tif"
        ),
        "band": 1,
        "resampling": Resampling.bilinear,
        "categorical": False
    },

    "Population_Density": {
        "path": (
            PROJECT_ROOT
            / "07_Population_Urban_Pressure"
            / "Enugu_Population_Density_2020_30m.tif"
        ),
        "band": 1,
        "resampling": Resampling.bilinear,
        "categorical": False
    },

    "Baseline_LULC_2020": {
        "path": (
            PROJECT_ROOT
            / "04_Land_Cover"
            / "Enugu_DynamicWorld_2020.tif"
        ),
        "band": 1,
        "resampling": Resampling.nearest,
        "categorical": True
    }
}

PREDICTOR_NAMES = [
    "Elevation_m",
    "Slope_deg",
    "Distance_to_Road_m",
    "Distance_to_Surface_Water_m",
    "Distance_to_Drainage_m",
    "Population_Density",
    "Baseline_LULC_2020"
]


# ==========================================================================================
# 4. OUTPUT FILES
# ==========================================================================================

PROBABILITY_FILE = (
    OUTPUT_DIR
    / "Enugu_Urban_Expansion_Probability_2025_30m.tif"
)

CONSTRAINED_PROBABILITY_FILE = (
    OUTPUT_DIR
    / "Enugu_Planning_Constrained_Expansion_Probability_30m.tif"
)

UNCERTAINTY_FILE = (
    OUTPUT_DIR
    / "Enugu_Model_Uncertainty_30m.tif"
)

CONFIDENCE_FILE = (
    OUTPUT_DIR
    / "Enugu_Model_Confidence_30m.tif"
)

APPLICABILITY_FILE = (
    OUTPUT_DIR
    / "Enugu_Model_Applicability_Mask_30m.tif"
)

SUITABILITY_FILE = (
    OUTPUT_DIR
    / "Enugu_Urban_Development_Suitability_Classes_30m.tif"
)

CONSTRAINT_FILE = (
    OUTPUT_DIR
    / "Enugu_Planning_Constraint_Codes_30m.tif"
)

AREA_TABLE_FILE = (
    TABLE_DIR
    / "Enugu_Suitability_Class_Area_Statistics.csv"
)

CONSTRAINT_TABLE_FILE = (
    TABLE_DIR
    / "Enugu_Planning_Constraint_Area_Statistics.csv"
)

PROBABILITY_SUMMARY_FILE = (
    TABLE_DIR
    / "Enugu_Statewide_Probability_Summary.csv"
)

INPUT_REGISTER_FILE = (
    TABLE_DIR
    / "Enugu_Stage11A_Raster_Input_Register.csv"
)

QUICKLOOK_FILE = (
    MAP_DIR
    / "Enugu_Statewide_Urban_Development_Suitability_Quicklook.png"
)

PROBABILITY_QUICKLOOK_FILE = (
    MAP_DIR
    / "Enugu_Urban_Expansion_Probability_Quicklook.png"
)

REGISTER_FILE = (
    ADMIN_DIR
    / "Stage_11A_Statewide_Surface_Register.json"
)

VALIDATION_FILE = (
    ADMIN_DIR
    / "Stage_11A_Validation_Register.csv"
)


# ==========================================================================================
# 5. CONSTANTS AND PLANNING RULES
# ==========================================================================================

REFERENCE_PREDICTOR = "Elevation_m"

TILE_SIZE = 256

OUTPUT_FLOAT_NODATA = -9999.0
OUTPUT_BYTE_NODATA = 255

RANDOM_SEED = 42

# A representative subset of trees is used for uncertainty estimation.
# The model probability itself still uses all 500 fitted trees.
UNCERTAINTY_TREE_COUNT = 50

# Dynamic World class codes:
# 0 = Water
# 1 = Trees
# 2 = Grass
# 3 = Flooded vegetation
# 4 = Crops
# 5 = Shrub and scrub
# 6 = Built
# 7 = Bare ground
# 8 = Snow and ice

DW_WATER_CLASS = 0
DW_FLOODED_VEGETATION_CLASS = 3
DW_BUILT_CLASS = 6
DW_SNOW_ICE_CLASS = 8

MAX_ACCEPTABLE_SLOPE_DEGREES = 15.0

# Conservative environmental setbacks.
MIN_SURFACE_WATER_DISTANCE_M = 90.0
MIN_DRAINAGE_DISTANCE_M = 60.0

# Suitability classification:
# 1 = Very Low
# 2 = Low
# 3 = Moderate
# 4 = High
# 5 = Very High
SUITABILITY_BREAKS = [
    0.00,
    0.20,
    0.40,
    0.60,
    0.80,
    1.00
]

SUITABILITY_LABELS = {
    1: "Very Low",
    2: "Low",
    3: "Moderate",
    4: "High",
    5: "Very High"
}

CONSTRAINT_LABELS = {
    0: "Applicable land",
    1: "Outside valid study area or missing predictor",
    2: "Existing built-up land in 2020",
    3: "Water or flooded vegetation",
    4: "Slope greater than 15 degrees",
    5: "Within recurring surface-water setback",
    6: "Within drainage setback",
    7: "Snow or ice class anomaly"
}


# ==========================================================================================
# 6. HELPER FUNCTIONS
# ==========================================================================================

def print_heading(title):
    print("\n" + title)
    print("-" * 120)


def validate_file(path, label):
    if not path.exists():
        raise FileNotFoundError(
            f"{label} was not found:\n{path}"
        )

    if path.stat().st_size == 0:
        raise RuntimeError(
            f"{label} exists but is empty:\n{path}"
        )


def get_positive_class_index(estimator):
    classes = np.asarray(
        estimator.classes_
    )

    positive_matches = np.where(
        classes == 1
    )[0]

    if positive_matches.size != 1:
        raise RuntimeError(
            "The fitted model does not contain exactly one positive class coded as 1. "
            f"Observed classes: {classes.tolist()}"
        )

    return int(
        positive_matches[0]
    )


def get_model_feature_names(model, fallback_names):
    if hasattr(
        model,
        "feature_names_in_"
    ):
        names = list(
            model.feature_names_in_
        )

        return names

    if hasattr(
        model,
        "named_steps"
    ):
        for step in model.named_steps.values():
            if hasattr(
                step,
                "feature_names_in_"
            ):
                return list(
                    step.feature_names_in_
                )

    return list(
        fallback_names
    )


def build_output_profile(reference_source, dtype, nodata, count=1):
    profile = reference_source.profile.copy()

    profile.update({
        "driver": "GTiff",
        "height": reference_source.height,
        "width": reference_source.width,
        "count": count,
        "dtype": dtype,
        "crs": reference_source.crs,
        "transform": reference_source.transform,
        "nodata": nodata,
        "compress": "DEFLATE",
        "predictor": 2,
        "tiled": True,
        "blockxsize": 256,
        "blockysize": 256,
        "BIGTIFF": "IF_SAFER"
    })

    return profile


def calculate_window_area_km2(transform):
    pixel_area_square_metres = abs(
        (
            transform.a
            * transform.e
        )
        -
        (
            transform.b
            * transform.d
        )
    )

    return pixel_area_square_metres / 1_000_000.0


def make_windows(width, height, tile_size):
    for row_off in range(
        0,
        height,
        tile_size
    ):
        window_height = min(
            tile_size,
            height - row_off
        )

        for col_off in range(
            0,
            width,
            tile_size
        ):
            window_width = min(
                tile_size,
                width - col_off
            )

            yield Window(
                col_off=col_off,
                row_off=row_off,
                width=window_width,
                height=window_height
            )


def transform_predictors_for_trees(model, predictor_frame):
    if not hasattr(
        model,
        "steps"
    ):
        return predictor_frame.to_numpy()

    if len(
        model.steps
    ) == 1:
        return predictor_frame.to_numpy()

    transformed = model[:-1].transform(
        predictor_frame
    )

    return transformed


def classify_probability(probability_values):
    classes = np.zeros(
        probability_values.shape,
        dtype=np.uint8
    )

    classes[
        (
            probability_values >= 0.00
        )
        &
        (
            probability_values < 0.20
        )
    ] = 1

    classes[
        (
            probability_values >= 0.20
        )
        &
        (
            probability_values < 0.40
        )
    ] = 2

    classes[
        (
            probability_values >= 0.40
        )
        &
        (
            probability_values < 0.60
        )
    ] = 3

    classes[
        (
            probability_values >= 0.60
        )
        &
        (
            probability_values < 0.80
        )
    ] = 4

    classes[
        probability_values >= 0.80
    ] = 5

    return classes


def write_float_array(destination, array, valid_mask, window):
    output = np.full(
        array.shape,
        OUTPUT_FLOAT_NODATA,
        dtype=np.float32
    )

    output[
        valid_mask
    ] = array[
        valid_mask
    ].astype(
        np.float32
    )

    destination.write(
        output,
        1,
        window=window
    )


def read_quicklook(raster_path, maximum_dimension=1600):
    with rasterio.open(
        raster_path
    ) as source:
        scale = max(
            source.width / maximum_dimension,
            source.height / maximum_dimension,
            1
        )

        output_width = max(
            1,
            int(
                source.width / scale
            )
        )

        output_height = max(
            1,
            int(
                source.height / scale
            )
        )

        data = source.read(
            1,
            out_shape=(
                output_height,
                output_width
            ),
            resampling=Resampling.nearest,
            masked=True
        )

        return data


# ==========================================================================================
# 7. START
# ==========================================================================================

print("=" * 120)
print("STAGE 11A — STATEWIDE URBAN-EXPANSION PROBABILITY SURFACE")
print("=" * 120)


# ==========================================================================================
# 8. REQUIRED FILE CHECKS
# ==========================================================================================

print_heading(
    "REQUIRED MODEL FILE CHECK"
)

validate_file(
    MODEL_FILE,
    "Leakage-corrected selected model"
)

print(
    f"✓ Leakage-corrected selected model: {MODEL_FILE}"
)

if INTERPRETATION_REGISTER_FILE.exists():
    print(
        f"✓ Stage 10D interpretation register: "
        f"{INTERPRETATION_REGISTER_FILE}"
    )
else:
    print(
        "⚠ Stage 10D interpretation register was not found. "
        "This does not prevent statewide prediction."
    )

print_heading(
    "FIXED RASTER INPUT CHECK"
)

input_register_records = []

for predictor_name in PREDICTOR_NAMES:
    raster_record = RASTER_INPUTS[
        predictor_name
    ]

    raster_path = raster_record[
        "path"
    ]

    validate_file(
        raster_path,
        predictor_name
    )

    with rasterio.open(
        raster_path
    ) as source:
        requested_band = int(
            raster_record["band"]
        )

        if requested_band < 1 or requested_band > source.count:
            raise RuntimeError(
                f"Invalid band number for {predictor_name}. "
                f"Requested Band {requested_band}; raster has {source.count} bands."
            )

        band_description = source.descriptions[
            requested_band - 1
        ]

        input_register_records.append({
            "Predictor": predictor_name,
            "File": str(raster_path),
            "Band": requested_band,
            "Band_Description": band_description,
            "CRS": str(source.crs),
            "Width": int(source.width),
            "Height": int(source.height),
            "Data_Type": source.dtypes[
                requested_band - 1
            ],
            "Resampling": raster_record[
                "resampling"
            ].name,
            "Categorical": bool(
                raster_record["categorical"]
            )
        })

        print(
            f"✓ {predictor_name}"
        )
        print(
            f"    File: {raster_path}"
        )
        print(
            f"    Band: {requested_band}"
            +
            (
                f" — {band_description}"
                if band_description is not None
                else ""
            )
        )
        print(
            f"    CRS: {source.crs} | "
            f"Dimensions: {source.width:,} × {source.height:,}"
        )

input_register_df = pd.DataFrame(
    input_register_records
)

input_register_df.to_csv(
    INPUT_REGISTER_FILE,
    index=False
)


# ==========================================================================================
# 9. LOAD AND VALIDATE FIXED MODEL
# ==========================================================================================

print_heading(
    "MODEL STATUS"
)

model = joblib.load(
    MODEL_FILE
)

print(
    f"Pipeline type: {type(model).__name__}"
)

if hasattr(
    model,
    "steps"
):
    classifier = model.steps[-1][1]
else:
    classifier = model

print(
    f"Classifier type: {type(classifier).__name__}"
)

if not hasattr(
    classifier,
    "estimators_"
):
    raise RuntimeError(
        "The selected classifier does not contain fitted tree estimators."
    )

tree_count = len(
    classifier.estimators_
)

print(
    f"Fitted trees: {tree_count}"
)

if tree_count < 1:
    raise RuntimeError(
        "The classifier contains no fitted trees."
    )

model_feature_names = get_model_feature_names(
    model,
    PREDICTOR_NAMES
)

print(
    f"Model predictor order: {model_feature_names}"
)

if "Distance_to_Built_2020_m" in model_feature_names:
    raise RuntimeError(
        "Leakage audit failed: Distance_to_Built_2020_m is present in the model."
    )

missing_model_predictors = [
    feature
    for feature in model_feature_names
    if feature not in PREDICTOR_NAMES
]

unexpected_available_predictors = [
    feature
    for feature in PREDICTOR_NAMES
    if feature not in model_feature_names
]

if missing_model_predictors:
    raise RuntimeError(
        "The model expects predictors that are unavailable: "
        f"{missing_model_predictors}"
    )

if unexpected_available_predictors:
    raise RuntimeError(
        "The statewide predictor list does not match the fitted model. "
        f"Unexpected available predictors: {unexpected_available_predictors}"
    )

positive_class_index = get_positive_class_index(
    model
)

classifier_positive_class_index = get_positive_class_index(
    classifier
)

print(
    f"Positive class index: {positive_class_index}"
)

print(
    "✓ The fixed leakage-corrected model was loaded without retraining."
)


# ==========================================================================================
# 10. SELECT TREES FOR UNCERTAINTY
# ==========================================================================================

rng = np.random.default_rng(
    RANDOM_SEED
)

uncertainty_tree_count = min(
    UNCERTAINTY_TREE_COUNT,
    tree_count
)

uncertainty_tree_indices = np.sort(
    rng.choice(
        tree_count,
        size=uncertainty_tree_count,
        replace=False
    )
)

uncertainty_trees = [
    classifier.estimators_[
        int(index)
    ]
    for index in uncertainty_tree_indices
]

print_heading(
    "UNCERTAINTY CONFIGURATION"
)

print(
    f"All fitted trees used for model probability: {tree_count}"
)

print(
    f"Trees sampled for uncertainty estimation: "
    f"{uncertainty_tree_count}"
)

print(
    f"Random seed: {RANDOM_SEED}"
)


# ==========================================================================================
# 11. OPEN RASTERS AND CREATE ALIGNED VIRTUAL RASTERS
# ==========================================================================================

reference_path = RASTER_INPUTS[
    REFERENCE_PREDICTOR
]["path"]

with ExitStack() as stack:
    reference_source = stack.enter_context(
        rasterio.open(
            reference_path
        )
    )

    if reference_source.crs is None:
        raise RuntimeError(
            "The reference raster has no CRS."
        )

    reference_crs = reference_source.crs
    reference_transform = reference_source.transform
    reference_width = reference_source.width
    reference_height = reference_source.height

    if str(
        reference_crs
    ) != "EPSG:32632":
        raise RuntimeError(
            "The reference raster should use EPSG:32632. "
            f"Observed CRS: {reference_crs}"
        )

    print_heading(
        "REFERENCE GRID"
    )

    print(
        f"Reference raster: {reference_path}"
    )
    print(
        f"CRS: {reference_crs}"
    )
    print(
        f"Dimensions: {reference_width:,} × {reference_height:,}"
    )
    print(
        f"Pixel size: "
        f"{abs(reference_transform.a):.4f} m × "
        f"{abs(reference_transform.e):.4f} m"
    )

    pixel_area_km2 = calculate_window_area_km2(
        reference_transform
    )

    print(
        f"Nominal pixel area: {pixel_area_km2:.8f} km²"
    )

    aligned_sources = {}

    for predictor_name in PREDICTOR_NAMES:
        predictor_info = RASTER_INPUTS[
            predictor_name
        ]

        source = stack.enter_context(
            rasterio.open(
                predictor_info["path"]
            )
        )

        vrt_options = {
            "crs": reference_crs,
            "transform": reference_transform,
            "width": reference_width,
            "height": reference_height,
            "resampling": predictor_info[
                "resampling"
            ]
        }

        if source.nodata is not None:
            vrt_options[
                "src_nodata"
            ] = source.nodata

            vrt_options[
                "nodata"
            ] = source.nodata

        vrt = stack.enter_context(
            WarpedVRT(
                source,
                **vrt_options
            )
        )

        aligned_sources[
            predictor_name
        ] = {
            "source": source,
            "vrt": vrt,
            "band": int(
                predictor_info["band"]
            ),
            "categorical": bool(
                predictor_info["categorical"]
            )
        }

    float_profile = build_output_profile(
        reference_source,
        dtype="float32",
        nodata=OUTPUT_FLOAT_NODATA
    )

    byte_profile = build_output_profile(
        reference_source,
        dtype="uint8",
        nodata=OUTPUT_BYTE_NODATA
    )

    probability_destination = stack.enter_context(
        rasterio.open(
            PROBABILITY_FILE,
            "w",
            **float_profile
        )
    )

    constrained_probability_destination = stack.enter_context(
        rasterio.open(
            CONSTRAINED_PROBABILITY_FILE,
            "w",
            **float_profile
        )
    )

    uncertainty_destination = stack.enter_context(
        rasterio.open(
            UNCERTAINTY_FILE,
            "w",
            **float_profile
        )
    )

    confidence_destination = stack.enter_context(
        rasterio.open(
            CONFIDENCE_FILE,
            "w",
            **float_profile
        )
    )

    applicability_destination = stack.enter_context(
        rasterio.open(
            APPLICABILITY_FILE,
            "w",
            **byte_profile
        )
    )

    suitability_destination = stack.enter_context(
        rasterio.open(
            SUITABILITY_FILE,
            "w",
            **byte_profile
        )
    )

    constraint_destination = stack.enter_context(
        rasterio.open(
            CONSTRAINT_FILE,
            "w",
            **byte_profile
        )
    )

    probability_destination.set_band_description(
        1,
        "Urban_Expansion_Probability"
    )

    constrained_probability_destination.set_band_description(
        1,
        "Planning_Constrained_Expansion_Probability"
    )

    uncertainty_destination.set_band_description(
        1,
        "Tree_Probability_Standard_Deviation"
    )

    confidence_destination.set_band_description(
        1,
        "Model_Confidence"
    )

    applicability_destination.set_band_description(
        1,
        "Applicability_Mask"
    )

    suitability_destination.set_band_description(
        1,
        "Suitability_Class"
    )

    constraint_destination.set_band_description(
        1,
        "Planning_Constraint_Code"
    )


    # ======================================================================================
    # 12. STREAMING STATISTICS
    # ======================================================================================

    suitability_counts = {
        class_code: 0
        for class_code in SUITABILITY_LABELS
    }

    constraint_counts = {
        code: 0
        for code in CONSTRAINT_LABELS
    }

    total_reference_pixels = int(
        reference_width
        * reference_height
    )

    valid_study_pixels = 0
    model_prediction_pixels = 0
    applicable_pixels = 0
    constrained_pixels = 0

    probability_sum = 0.0
    probability_square_sum = 0.0
    probability_minimum = np.inf
    probability_maximum = -np.inf

    constrained_probability_sum = 0.0
    constrained_probability_minimum = np.inf
    constrained_probability_maximum = -np.inf

    uncertainty_sum = 0.0
    uncertainty_minimum = np.inf
    uncertainty_maximum = -np.inf

    processed_pixels = 0

    windows = list(
        make_windows(
            reference_width,
            reference_height,
            TILE_SIZE
        )
    )

    total_windows = len(
        windows
    )

    print_heading(
        "STATEWIDE TILED PREDICTION"
    )

    print(
        f"Tile size: {TILE_SIZE} × {TILE_SIZE}"
    )
    print(
        f"Total processing windows: {total_windows:,}"
    )


    # ======================================================================================
    # 13. PROCESS EACH TILE
    # ======================================================================================

    for window_number, window in enumerate(
        windows,
        start=1
    ):
        window_height = int(
            window.height
        )

        window_width = int(
            window.width
        )

        window_shape = (
            window_height,
            window_width
        )

        predictor_arrays = {}
        valid_predictor_mask = np.ones(
            window_shape,
            dtype=bool
        )

        for predictor_name in PREDICTOR_NAMES:
            aligned_record = aligned_sources[
                predictor_name
            ]

            predictor_data = aligned_record[
                "vrt"
            ].read(
                aligned_record["band"],
                window=window,
                masked=True
            )

            predictor_mask = np.ma.getmaskarray(
                predictor_data
            )

            predictor_array = np.asarray(
                predictor_data.filled(
                    np.nan
                ),
                dtype=np.float64
            )

            finite_mask = np.isfinite(
                predictor_array
            )

            predictor_valid = (
                ~predictor_mask
                &
                finite_mask
            )

            # Elevation values in Enugu are above zero.
            # This also removes zero-filled pixels outside the state boundary.
            if predictor_name == "Elevation_m":
                predictor_valid &= (
                    predictor_array > 0
                )

            if predictor_name == "Slope_deg":
                predictor_valid &= (
                    predictor_array >= 0
                )

            if predictor_name in [
                "Distance_to_Road_m",
                "Distance_to_Surface_Water_m",
                "Distance_to_Drainage_m",
                "Population_Density"
            ]:
                predictor_valid &= (
                    predictor_array >= 0
                )

            if predictor_name == "Baseline_LULC_2020":
                rounded_lulc = np.rint(
                    predictor_array
                )

                predictor_valid &= np.isclose(
                    predictor_array,
                    rounded_lulc,
                    atol=1e-5
                )

                predictor_valid &= (
                    rounded_lulc >= 0
                )

                predictor_valid &= (
                    rounded_lulc <= 8
                )

                predictor_array = rounded_lulc

            predictor_arrays[
                predictor_name
            ] = predictor_array

            valid_predictor_mask &= predictor_valid

        window_pixel_count = int(
            window_height
            * window_width
        )

        processed_pixels += window_pixel_count

        raw_probability = np.full(
            window_shape,
            np.nan,
            dtype=np.float32
        )

        tree_uncertainty = np.full(
            window_shape,
            np.nan,
            dtype=np.float32
        )

        model_confidence = np.full(
            window_shape,
            np.nan,
            dtype=np.float32
        )

        applicability = np.full(
            window_shape,
            OUTPUT_BYTE_NODATA,
            dtype=np.uint8
        )

        suitability = np.full(
            window_shape,
            OUTPUT_BYTE_NODATA,
            dtype=np.uint8
        )

        constraint_code = np.full(
            window_shape,
            OUTPUT_BYTE_NODATA,
            dtype=np.uint8
        )

        constrained_probability = np.full(
            window_shape,
            np.nan,
            dtype=np.float32
        )

        valid_indices = np.flatnonzero(
            valid_predictor_mask.ravel()
        )

        valid_count = int(
            valid_indices.size
        )

        valid_study_pixels += valid_count

        # Pixels without complete valid predictors receive constraint code 1.
        invalid_reference_mask = ~valid_predictor_mask

        constraint_code[
            invalid_reference_mask
        ] = 1

        constraint_counts[1] += int(
            np.count_nonzero(
                invalid_reference_mask
            )
        )

        if valid_count > 0:
            predictor_data_dictionary = {}

            for feature_name in model_feature_names:
                feature_values = predictor_arrays[
                    feature_name
                ].ravel()[
                    valid_indices
                ]

                if feature_name == "Baseline_LULC_2020":
                    feature_values = feature_values.astype(
                        np.int16
                    )
                else:
                    feature_values = feature_values.astype(
                        np.float64
                    )

                predictor_data_dictionary[
                    feature_name
                ] = feature_values

            predictor_frame = pd.DataFrame(
                predictor_data_dictionary,
                columns=model_feature_names
            )

            if predictor_frame.isna().any().any():
                raise RuntimeError(
                    "Missing predictor values reached the model."
                )

            probability_values = model.predict_proba(
                predictor_frame
            )[
                :,
                positive_class_index
            ].astype(
                np.float64
            )

            if not np.all(
                np.isfinite(
                    probability_values
                )
            ):
                raise RuntimeError(
                    "The model returned non-finite probabilities."
                )

            if (
                np.min(
                    probability_values
                ) < 0
                or
                np.max(
                    probability_values
                ) > 1
            ):
                raise RuntimeError(
                    "The model returned probabilities outside 0–1."
                )

            raw_probability.ravel()[
                valid_indices
            ] = probability_values.astype(
                np.float32
            )

            model_prediction_pixels += valid_count

            probability_sum += float(
                np.sum(
                    probability_values
                )
            )

            probability_square_sum += float(
                np.sum(
                    probability_values ** 2
                )
            )

            probability_minimum = min(
                probability_minimum,
                float(
                    np.min(
                        probability_values
                    )
                )
            )

            probability_maximum = max(
                probability_maximum,
                float(
                    np.max(
                        probability_values
                    )
                )
            )

            # --------------------------------------------------------------------------
            # Tree-based uncertainty
            # --------------------------------------------------------------------------

            transformed_predictors = transform_predictors_for_trees(
                model,
                predictor_frame
            )

            running_mean = np.zeros(
                valid_count,
                dtype=np.float64
            )

            running_m2 = np.zeros(
                valid_count,
                dtype=np.float64
            )

            for tree_iteration, tree in enumerate(
                uncertainty_trees,
                start=1
            ):
                tree_probabilities = tree.predict_proba(
                    transformed_predictors
                )[
                    :,
                    classifier_positive_class_index
                ].astype(
                    np.float64
                )

                delta = (
                    tree_probabilities
                    -
                    running_mean
                )

                running_mean += (
                    delta
                    /
                    tree_iteration
                )

                delta_second = (
                    tree_probabilities
                    -
                    running_mean
                )

                running_m2 += (
                    delta
                    *
                    delta_second
                )

            if uncertainty_tree_count > 1:
                uncertainty_values = np.sqrt(
                    running_m2
                    /
                    (
                        uncertainty_tree_count
                        -
                        1
                    )
                )
            else:
                uncertainty_values = np.zeros(
                    valid_count,
                    dtype=np.float64
                )

            uncertainty_values = np.clip(
                uncertainty_values,
                0.0,
                0.5
            )

            # Tree disagreement has a theoretical maximum close to 0.5
            # for binary probability estimates.
            confidence_values = np.clip(
                1.0
                -
                (
                    2.0
                    *
                    uncertainty_values
                ),
                0.0,
                1.0
            )

            tree_uncertainty.ravel()[
                valid_indices
            ] = uncertainty_values.astype(
                np.float32
            )

            model_confidence.ravel()[
                valid_indices
            ] = confidence_values.astype(
                np.float32
            )

            uncertainty_sum += float(
                np.sum(
                    uncertainty_values
                )
            )

            uncertainty_minimum = min(
                uncertainty_minimum,
                float(
                    np.min(
                        uncertainty_values
                    )
                )
            )

            uncertainty_maximum = max(
                uncertainty_maximum,
                float(
                    np.max(
                        uncertainty_values
                    )
                )
            )


        # ==================================================================================
        # 14. PLANNING CONSTRAINTS AND APPLICABILITY
        # ==================================================================================

        lulc = predictor_arrays[
            "Baseline_LULC_2020"
        ]

        slope = predictor_arrays[
            "Slope_deg"
        ]

        water_distance = predictor_arrays[
            "Distance_to_Surface_Water_m"
        ]

        drainage_distance = predictor_arrays[
            "Distance_to_Drainage_m"
        ]

        existing_built_mask = (
            valid_predictor_mask
            &
            (
                lulc == DW_BUILT_CLASS
            )
        )

        water_or_flooded_mask = (
            valid_predictor_mask
            &
            (
                (
                    lulc == DW_WATER_CLASS
                )
                |
                (
                    lulc == DW_FLOODED_VEGETATION_CLASS
                )
            )
        )

        steep_slope_mask = (
            valid_predictor_mask
            &
            (
                slope
                >
                MAX_ACCEPTABLE_SLOPE_DEGREES
            )
        )

        water_setback_mask = (
            valid_predictor_mask
            &
            (
                water_distance
                <
                MIN_SURFACE_WATER_DISTANCE_M
            )
        )

        drainage_setback_mask = (
            valid_predictor_mask
            &
            (
                drainage_distance
                <
                MIN_DRAINAGE_DISTANCE_M
            )
        )

        snow_anomaly_mask = (
            valid_predictor_mask
            &
            (
                lulc == DW_SNOW_ICE_CLASS
            )
        )

        # Assign one principal constraint per pixel using a defined hierarchy.
        constraint_code[
            existing_built_mask
        ] = 2

        constraint_code[
            (
                constraint_code == OUTPUT_BYTE_NODATA
            )
            &
            water_or_flooded_mask
        ] = 3

        constraint_code[
            (
                constraint_code == OUTPUT_BYTE_NODATA
            )
            &
            steep_slope_mask
        ] = 4

        constraint_code[
            (
                constraint_code == OUTPUT_BYTE_NODATA
            )
            &
            water_setback_mask
        ] = 5

        constraint_code[
            (
                constraint_code == OUTPUT_BYTE_NODATA
            )
            &
            drainage_setback_mask
        ] = 6

        constraint_code[
            (
                constraint_code == OUTPUT_BYTE_NODATA
            )
            &
            snow_anomaly_mask
        ] = 7

        applicable_mask = (
            valid_predictor_mask
            &
            ~existing_built_mask
            &
            ~water_or_flooded_mask
            &
            ~steep_slope_mask
            &
            ~water_setback_mask
            &
            ~drainage_setback_mask
            &
            ~snow_anomaly_mask
        )

        constraint_code[
            applicable_mask
        ] = 0

        applicability[
            valid_predictor_mask
        ] = 0

        applicability[
            applicable_mask
        ] = 1

        applicable_count = int(
            np.count_nonzero(
                applicable_mask
            )
        )

        applicable_pixels += applicable_count

        constrained_pixels += int(
            valid_count
            -
            applicable_count
        )

        for code in range(
            0,
            8
        ):
            constraint_counts[
                code
            ] += int(
                np.count_nonzero(
                    constraint_code
                    ==
                    code
                )
            )

        constrained_probability[
            applicable_mask
        ] = raw_probability[
            applicable_mask
        ]

        if applicable_count > 0:
            applicable_probabilities = constrained_probability[
                applicable_mask
            ].astype(
                np.float64
            )

            constrained_probability_sum += float(
                np.sum(
                    applicable_probabilities
                )
            )

            constrained_probability_minimum = min(
                constrained_probability_minimum,
                float(
                    np.min(
                        applicable_probabilities
                    )
                )
            )

            constrained_probability_maximum = max(
                constrained_probability_maximum,
                float(
                    np.max(
                        applicable_probabilities
                    )
                )
            )

            suitability_values = classify_probability(
                applicable_probabilities
            )

            suitability[
                applicable_mask
            ] = suitability_values

            for class_code in SUITABILITY_LABELS:
                suitability_counts[
                    class_code
                ] += int(
                    np.count_nonzero(
                        suitability_values
                        ==
                        class_code
                    )
                )


        # ==================================================================================
        # 15. WRITE TILE OUTPUTS
        # ==================================================================================

        write_float_array(
            probability_destination,
            raw_probability,
            valid_predictor_mask,
            window
        )

        write_float_array(
            constrained_probability_destination,
            constrained_probability,
            applicable_mask,
            window
        )

        write_float_array(
            uncertainty_destination,
            tree_uncertainty,
            valid_predictor_mask,
            window
        )

        write_float_array(
            confidence_destination,
            model_confidence,
            valid_predictor_mask,
            window
        )

        applicability_destination.write(
            applicability,
            1,
            window=window
        )

        suitability_destination.write(
            suitability,
            1,
            window=window
        )

        constraint_destination.write(
            constraint_code,
            1,
            window=window
        )

        if (
            window_number == 1
            or
            window_number % 50 == 0
            or
            window_number == total_windows
        ):
            progress_percent = (
                window_number
                /
                total_windows
                *
                100
            )

            print(
                f"Window {window_number:,}/{total_windows:,} "
                f"({progress_percent:.1f}%) completed"
            )

        del predictor_arrays
        del raw_probability
        del tree_uncertainty
        del model_confidence
        del applicability
        del suitability
        del constraint_code
        del constrained_probability

        if valid_count > 0:
            del predictor_frame
            del probability_values
            del transformed_predictors
            del running_mean
            del running_m2
            del uncertainty_values
            del confidence_values

        gc.collect()


# ==========================================================================================
# 16. POST-PROCESSING STATISTICS
# ==========================================================================================

if model_prediction_pixels == 0:
    raise RuntimeError(
        "No statewide prediction pixels were generated."
    )

probability_mean = (
    probability_sum
    /
    model_prediction_pixels
)

probability_variance = max(
    (
        probability_square_sum
        /
        model_prediction_pixels
    )
    -
    (
        probability_mean ** 2
    ),
    0.0
)

probability_standard_deviation = math.sqrt(
    probability_variance
)

mean_uncertainty = (
    uncertainty_sum
    /
    model_prediction_pixels
)

if applicable_pixels > 0:
    constrained_probability_mean = (
        constrained_probability_sum
        /
        applicable_pixels
    )
else:
    constrained_probability_mean = np.nan

valid_area_km2 = (
    valid_study_pixels
    *
    pixel_area_km2
)

applicable_area_km2 = (
    applicable_pixels
    *
    pixel_area_km2
)

constrained_area_km2 = (
    constrained_pixels
    *
    pixel_area_km2
)


# ==========================================================================================
# 17. SUITABILITY AREA TABLE
# ==========================================================================================

suitability_records = []

for class_code, class_label in SUITABILITY_LABELS.items():
    pixel_count = int(
        suitability_counts[
            class_code
        ]
    )

    area_km2 = (
        pixel_count
        *
        pixel_area_km2
    )

    percentage_of_applicable = (
        area_km2
        /
        applicable_area_km2
        *
        100
        if applicable_area_km2 > 0
        else np.nan
    )

    percentage_of_valid_state = (
        area_km2
        /
        valid_area_km2
        *
        100
        if valid_area_km2 > 0
        else np.nan
    )

    suitability_records.append({
        "Suitability_Class_Code": class_code,
        "Suitability_Class": class_label,
        "Probability_Range": (
            f"{SUITABILITY_BREAKS[class_code - 1]:.1f}–"
            f"{SUITABILITY_BREAKS[class_code]:.1f}"
        ),
        "Pixel_Count": pixel_count,
        "Area_km2": area_km2,
        "Percentage_of_Applicable_Land": percentage_of_applicable,
        "Percentage_of_Valid_State_Area": percentage_of_valid_state
    })

suitability_df = pd.DataFrame(
    suitability_records
)

suitability_df.to_csv(
    AREA_TABLE_FILE,
    index=False
)


# ==========================================================================================
# 18. CONSTRAINT AREA TABLE
# ==========================================================================================

constraint_records = []

for constraint_code_value, constraint_label in CONSTRAINT_LABELS.items():
    pixel_count = int(
        constraint_counts[
            constraint_code_value
        ]
    )

    area_km2 = (
        pixel_count
        *
        pixel_area_km2
    )

    percentage_of_valid_state = (
        area_km2
        /
        valid_area_km2
        *
        100
        if valid_area_km2 > 0
        else np.nan
    )

    constraint_records.append({
        "Constraint_Code": constraint_code_value,
        "Constraint": constraint_label,
        "Pixel_Count": pixel_count,
        "Area_km2": area_km2,
        "Percentage_of_Valid_State_Area": percentage_of_valid_state
    })

constraint_df = pd.DataFrame(
    constraint_records
)

constraint_df.to_csv(
    CONSTRAINT_TABLE_FILE,
    index=False
)


# ==========================================================================================
# 19. PROBABILITY SUMMARY TABLE
# ==========================================================================================

probability_summary_df = pd.DataFrame([
    {
        "Metric": "Total reference-grid pixels",
        "Value": total_reference_pixels,
        "Unit": "pixels"
    },
    {
        "Metric": "Valid model prediction pixels",
        "Value": model_prediction_pixels,
        "Unit": "pixels"
    },
    {
        "Metric": "Valid statewide model area",
        "Value": valid_area_km2,
        "Unit": "km²"
    },
    {
        "Metric": "Planning-applicable pixels",
        "Value": applicable_pixels,
        "Unit": "pixels"
    },
    {
        "Metric": "Planning-applicable area",
        "Value": applicable_area_km2,
        "Unit": "km²"
    },
    {
        "Metric": "Planning-constrained area",
        "Value": constrained_area_km2,
        "Unit": "km²"
    },
    {
        "Metric": "Mean raw expansion probability",
        "Value": probability_mean,
        "Unit": "probability"
    },
    {
        "Metric": "Standard deviation of raw probability",
        "Value": probability_standard_deviation,
        "Unit": "probability"
    },
    {
        "Metric": "Minimum raw expansion probability",
        "Value": probability_minimum,
        "Unit": "probability"
    },
    {
        "Metric": "Maximum raw expansion probability",
        "Value": probability_maximum,
        "Unit": "probability"
    },
    {
        "Metric": "Mean constrained expansion probability",
        "Value": constrained_probability_mean,
        "Unit": "probability"
    },
    {
        "Metric": "Minimum constrained probability",
        "Value": (
            constrained_probability_minimum
            if applicable_pixels > 0
            else np.nan
        ),
        "Unit": "probability"
    },
    {
        "Metric": "Maximum constrained probability",
        "Value": (
            constrained_probability_maximum
            if applicable_pixels > 0
            else np.nan
        ),
        "Unit": "probability"
    },
    {
        "Metric": "Mean tree-based uncertainty",
        "Value": mean_uncertainty,
        "Unit": "standard deviation"
    },
    {
        "Metric": "Minimum tree-based uncertainty",
        "Value": uncertainty_minimum,
        "Unit": "standard deviation"
    },
    {
        "Metric": "Maximum tree-based uncertainty",
        "Value": uncertainty_maximum,
        "Unit": "standard deviation"
    }
])

probability_summary_df.to_csv(
    PROBABILITY_SUMMARY_FILE,
    index=False
)


# ==========================================================================================
# 20. QUICKLOOK MAPS
# ==========================================================================================

print_heading(
    "GENERATING QUICKLOOK MAPS"
)

suitability_quicklook = read_quicklook(
    SUITABILITY_FILE
)

suitability_colormap = ListedColormap([
    "#d73027",
    "#fc8d59",
    "#fee08b",
    "#91cf60",
    "#1a9850"
])

suitability_norm = BoundaryNorm(
    [
        0.5,
        1.5,
        2.5,
        3.5,
        4.5,
        5.5
    ],
    suitability_colormap.N
)

figure, axis = plt.subplots(
    figsize=(
        10,
        11
    )
)

image = axis.imshow(
    suitability_quicklook,
    cmap=suitability_colormap,
    norm=suitability_norm
)

axis.set_title(
    "Enugu State Urban Development Suitability\n"
    "Leakage-Corrected Extra Trees Model",
    fontsize=15,
    fontweight="bold"
)

axis.axis(
    "off"
)

colour_bar = figure.colorbar(
    image,
    ax=axis,
    fraction=0.035,
    pad=0.02,
    ticks=[
        1,
        2,
        3,
        4,
        5
    ]
)

colour_bar.ax.set_yticklabels([
    "Very Low",
    "Low",
    "Moderate",
    "High",
    "Very High"
])

colour_bar.set_label(
    "Suitability class",
    fontsize=11
)

figure.text(
    0.5,
    0.025,
    "Applicable land excludes existing 2020 built-up areas, water, "
    "steep slopes and environmental setbacks.",
    ha="center",
    fontsize=9
)

figure.tight_layout(
    rect=[
        0,
        0.04,
        1,
        1
    ]
)

figure.savefig(
    QUICKLOOK_FILE,
    dpi=300,
    bbox_inches="tight"
)

plt.close(
    figure
)

probability_quicklook = read_quicklook(
    CONSTRAINED_PROBABILITY_FILE
)

figure, axis = plt.subplots(
    figsize=(
        10,
        11
    )
)

image = axis.imshow(
    probability_quicklook,
    cmap="viridis",
    vmin=0,
    vmax=1
)

axis.set_title(
    "Enugu State Planning-Constrained\n"
    "Urban Expansion Probability",
    fontsize=15,
    fontweight="bold"
)

axis.axis(
    "off"
)

colour_bar = figure.colorbar(
    image,
    ax=axis,
    fraction=0.035,
    pad=0.02
)

colour_bar.set_label(
    "Predicted expansion probability",
    fontsize=11
)

figure.tight_layout()

figure.savefig(
    PROBABILITY_QUICKLOOK_FILE,
    dpi=300,
    bbox_inches="tight"
)

plt.close(
    figure
)

print(
    f"✓ Suitability quicklook: {QUICKLOOK_FILE}"
)

print(
    f"✓ Probability quicklook: {PROBABILITY_QUICKLOOK_FILE}"
)


# ==========================================================================================
# 21. VALIDATION
# ==========================================================================================

validation_records = []


def add_validation(check, passed, observed, expected):
    validation_records.append({
        "Check": check,
        "Passed": bool(passed),
        "Observed": str(observed),
        "Expected": str(expected)
    })


output_files = [
    PROBABILITY_FILE,
    CONSTRAINED_PROBABILITY_FILE,
    UNCERTAINTY_FILE,
    CONFIDENCE_FILE,
    APPLICABILITY_FILE,
    SUITABILITY_FILE,
    CONSTRAINT_FILE,
    AREA_TABLE_FILE,
    CONSTRAINT_TABLE_FILE,
    PROBABILITY_SUMMARY_FILE,
    INPUT_REGISTER_FILE,
    QUICKLOOK_FILE,
    PROBABILITY_QUICKLOOK_FILE
]

all_outputs_exist = all(
    path.exists()
    and
    path.stat().st_size > 0
    for path in output_files
)

add_validation(
    "All required Stage 11A outputs were created",
    all_outputs_exist,
    all_outputs_exist,
    True
)

add_validation(
    "No leakage predictor used",
    "Distance_to_Built_2020_m" not in model_feature_names,
    model_feature_names,
    "Seven leakage-safe predictors only"
)

add_validation(
    "Dynamic World 2020 predictor uses Band 1",
    RASTER_INPUTS[
        "Baseline_LULC_2020"
    ]["band"] == 1,
    RASTER_INPUTS[
        "Baseline_LULC_2020"
    ]["band"],
    1
)

add_validation(
    "Categorical land cover uses nearest-neighbour resampling",
    RASTER_INPUTS[
        "Baseline_LULC_2020"
    ]["resampling"] == Resampling.nearest,
    RASTER_INPUTS[
        "Baseline_LULC_2020"
    ]["resampling"].name,
    "nearest"
)

add_validation(
    "Model produced statewide valid predictions",
    model_prediction_pixels > 0,
    model_prediction_pixels,
    "> 0"
)

add_validation(
    "Raw probabilities are within 0–1",
    (
        probability_minimum >= 0
        and
        probability_maximum <= 1
    ),
    (
        probability_minimum,
        probability_maximum
    ),
    "0–1"
)

add_validation(
    "Applicability area is positive",
    applicable_pixels > 0,
    applicable_pixels,
    "> 0"
)

add_validation(
    "Suitability class counts equal applicable pixels",
    sum(
        suitability_counts.values()
    ) == applicable_pixels,
    sum(
        suitability_counts.values()
    ),
    applicable_pixels
)

add_validation(
    "Suitability codes are limited to 1–5",
    set(
        suitability_counts.keys()
    ) == {
        1,
        2,
        3,
        4,
        5
    },
    sorted(
        suitability_counts.keys()
    ),
    [
        1,
        2,
        3,
        4,
        5
    ]
)

add_validation(
    "Uncertainty values are finite and within 0–0.5",
    (
        np.isfinite(
            uncertainty_minimum
        )
        and
        np.isfinite(
            uncertainty_maximum
        )
        and
        uncertainty_minimum >= 0
        and
        uncertainty_maximum <= 0.5
    ),
    (
        uncertainty_minimum,
        uncertainty_maximum
    ),
    "0–0.5"
)

add_validation(
    "Model was used without retraining",
    True,
    "Loaded fixed joblib pipeline",
    "No fit or tuning operation"
)

validation_df = pd.DataFrame(
    validation_records
)

validation_df.to_csv(
    VALIDATION_FILE,
    index=False
)

failed_checks = validation_df.loc[
    ~validation_df[
        "Passed"
    ]
]

if not failed_checks.empty:
    print_heading(
        "FAILED VALIDATION CHECKS"
    )

    print(
        failed_checks.to_string(
            index=False
        )
    )

    raise RuntimeError(
        "Stage 11A completed processing but one or more validation checks failed."
    )


# ==========================================================================================
# 22. JSON REGISTER
# ==========================================================================================

register = {
    "stage": "11A",
    "stage_name": "Statewide Urban Expansion Probability Surface",
    "completion_time_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "study_area": "Enugu State, Nigeria",
    "model_file": str(
        MODEL_FILE
    ),
    "model_type": type(
        classifier
    ).__name__,
    "fitted_tree_count": int(
        tree_count
    ),
    "uncertainty_tree_count": int(
        uncertainty_tree_count
    ),
    "model_retrained": False,
    "threshold_optimised": False,
    "leakage_predictor_excluded": True,
    "predictors": model_feature_names,
    "reference_grid": {
        "raster": str(
            reference_path
        ),
        "crs": str(
            reference_crs
        ),
        "width": int(
            reference_width
        ),
        "height": int(
            reference_height
        ),
        "pixel_width_m": float(
            abs(
                reference_transform.a
            )
        ),
        "pixel_height_m": float(
            abs(
                reference_transform.e
            )
        ),
        "pixel_area_km2": float(
            pixel_area_km2
        )
    },
    "dynamic_world_2020": {
        "file": str(
            RASTER_INPUTS[
                "Baseline_LULC_2020"
            ]["path"]
        ),
        "predictor_band": 1,
        "predictor_band_description": "LULC_2020",
        "observation_count_band": 2,
        "observation_count_band_description": "DW_Obs_Count_2020",
        "resampling": "nearest"
    },
    "planning_constraints": {
        "existing_built_up_2020_excluded": True,
        "water_class_excluded": True,
        "flooded_vegetation_excluded": True,
        "maximum_slope_degrees": MAX_ACCEPTABLE_SLOPE_DEGREES,
        "minimum_surface_water_distance_m": MIN_SURFACE_WATER_DISTANCE_M,
        "minimum_drainage_distance_m": MIN_DRAINAGE_DISTANCE_M
    },
    "suitability_breaks": SUITABILITY_BREAKS,
    "suitability_labels": {
        str(
            key
        ): value
        for key, value in SUITABILITY_LABELS.items()
    },
    "statewide_statistics": {
        "valid_prediction_pixels": int(
            model_prediction_pixels
        ),
        "valid_area_km2": float(
            valid_area_km2
        ),
        "applicable_pixels": int(
            applicable_pixels
        ),
        "applicable_area_km2": float(
            applicable_area_km2
        ),
        "constrained_area_km2": float(
            constrained_area_km2
        ),
        "mean_raw_probability": float(
            probability_mean
        ),
        "standard_deviation_raw_probability": float(
            probability_standard_deviation
        ),
        "minimum_raw_probability": float(
            probability_minimum
        ),
        "maximum_raw_probability": float(
            probability_maximum
        ),
        "mean_constrained_probability": (
            float(
                constrained_probability_mean
            )
            if np.isfinite(
                constrained_probability_mean
            )
            else None
        ),
        "mean_uncertainty": float(
            mean_uncertainty
        ),
        "minimum_uncertainty": float(
            uncertainty_minimum
        ),
        "maximum_uncertainty": float(
            uncertainty_maximum
        )
    },
    "outputs": {
        "raw_probability": str(
            PROBABILITY_FILE
        ),
        "constrained_probability": str(
            CONSTRAINED_PROBABILITY_FILE
        ),
        "uncertainty": str(
            UNCERTAINTY_FILE
        ),
        "confidence": str(
            CONFIDENCE_FILE
        ),
        "applicability_mask": str(
            APPLICABILITY_FILE
        ),
        "suitability_classes": str(
            SUITABILITY_FILE
        ),
        "constraint_codes": str(
            CONSTRAINT_FILE
        ),
        "suitability_area_table": str(
            AREA_TABLE_FILE
        ),
        "constraint_area_table": str(
            CONSTRAINT_TABLE_FILE
        ),
        "probability_summary": str(
            PROBABILITY_SUMMARY_FILE
        ),
        "input_register": str(
            INPUT_REGISTER_FILE
        ),
        "suitability_quicklook": str(
            QUICKLOOK_FILE
        ),
        "probability_quicklook": str(
            PROBABILITY_QUICKLOOK_FILE
        ),
        "validation_register": str(
            VALIDATION_FILE
        )
    },
    "validation_passed": True
}

with open(
    REGISTER_FILE,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        register,
        file,
        indent=4
    )


# ==========================================================================================
# 23. FINAL OUTPUT
# ==========================================================================================

print_heading(
    "STATEWIDE PROBABILITY SUMMARY"
)

print(
    f"Valid model area: {valid_area_km2:,.2f} km²"
)

print(
    f"Planning-applicable area: {applicable_area_km2:,.2f} km²"
)

print(
    f"Planning-constrained area: {constrained_area_km2:,.2f} km²"
)

print(
    f"Mean raw expansion probability: {probability_mean:.4f}"
)

print(
    f"Raw probability standard deviation: "
    f"{probability_standard_deviation:.4f}"
)

print(
    f"Raw probability range: "
    f"{probability_minimum:.4f}–{probability_maximum:.4f}"
)

print(
    f"Mean planning-constrained probability: "
    f"{constrained_probability_mean:.4f}"
)

print(
    f"Mean model uncertainty: {mean_uncertainty:.4f}"
)

print_heading(
    "SUITABILITY CLASS DISTRIBUTION"
)

print(
    suitability_df.to_string(
        index=False,
        formatters={
            "Area_km2":
                lambda value: f"{value:,.2f}",

            "Percentage_of_Applicable_Land":
                lambda value: f"{value:.2f}%",

            "Percentage_of_Valid_State_Area":
                lambda value: f"{value:.2f}%"
        }
    )
)

print_heading(
    "OUTPUT FILES"
)

for output_path in output_files:
    print(
        f"✓ {output_path}"
    )

print(
    f"✓ {REGISTER_FILE}"
)

print(
    f"✓ {VALIDATION_FILE}"
)

print_heading(
    "VALIDATION RESULTS"
)

print(
    validation_df.to_string(
        index=False
    )
)

print("=" * 120)
print("STAGE 11A COMPLETED SUCCESSFULLY")
print("=" * 120)
print(
    "✓ The fixed leakage-corrected Extra Trees model was used without retraining."
)
print(
    "✓ Seven leakage-safe predictors were used."
)
print(
    "✓ Dynamic World 2020 Band 1 was used as the baseline land-cover predictor."
)
print(
    "✓ Statewide probability, uncertainty and confidence surfaces were generated."
)
print(
    "✓ Existing built-up land and environmental constraints were excluded."
)
print(
    "✓ Five urban-development suitability classes were generated."
)
print(
    "✓ All validation checks passed."
)
print("=" * 120)

In [ ]:
# ==========================================================================================
# PROJECT 7 — STAGE 11B
# SUITABILITY RELIABILITY, CONFIDENCE AND PLANNING-PRIORITY VALIDATION
#
# Outputs:
#   1. Confidence classes
#   2. High-confidence suitability surface
#   3. Planning-priority zones
#   4. Suitability × confidence cross-tabulation
#   5. Mean probability and uncertainty by suitability class
#   6. Area statistics
#   7. Validation register
#   8. Quicklook maps
# ==========================================================================================

from pathlib import Path
from contextlib import ExitStack
from datetime import datetime, timezone
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio

from matplotlib.colors import BoundaryNorm, ListedColormap
from rasterio.enums import Resampling
from rasterio.windows import Window

warnings.filterwarnings("ignore")


# ==========================================================================================
# 1. PROJECT PATHS
# ==========================================================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

STAGE_11A_DIR = (
    PROJECT_ROOT
    / "12_Statewide_Suitability"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "13_Planning_Priority"
)

TABLE_DIR = (
    OUTPUT_DIR
    / "Tables"
)

MAP_DIR = (
    OUTPUT_DIR
    / "Maps"
)

ADMIN_DIR = (
    PROJECT_ROOT
    / "00_Project_Admin"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

MAP_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ==========================================================================================
# 2. REQUIRED INPUTS
# ==========================================================================================

PROBABILITY_FILE = (
    STAGE_11A_DIR
    / "Enugu_Urban_Expansion_Probability_2025_30m.tif"
)

CONSTRAINED_PROBABILITY_FILE = (
    STAGE_11A_DIR
    / "Enugu_Planning_Constrained_Expansion_Probability_30m.tif"
)

UNCERTAINTY_FILE = (
    STAGE_11A_DIR
    / "Enugu_Model_Uncertainty_30m.tif"
)

CONFIDENCE_FILE = (
    STAGE_11A_DIR
    / "Enugu_Model_Confidence_30m.tif"
)

APPLICABILITY_FILE = (
    STAGE_11A_DIR
    / "Enugu_Model_Applicability_Mask_30m.tif"
)

SUITABILITY_FILE = (
    STAGE_11A_DIR
    / "Enugu_Urban_Development_Suitability_Classes_30m.tif"
)

CONSTRAINT_FILE = (
    STAGE_11A_DIR
    / "Enugu_Planning_Constraint_Codes_30m.tif"
)

STAGE_11A_REGISTER = (
    ADMIN_DIR
    / "Stage_11A_Statewide_Surface_Register.json"
)


# ==========================================================================================
# 3. OUTPUT FILES
# ==========================================================================================

CONFIDENCE_CLASS_FILE = (
    OUTPUT_DIR
    / "Enugu_Model_Confidence_Classes_30m.tif"
)

HIGH_CONFIDENCE_SUITABILITY_FILE = (
    OUTPUT_DIR
    / "Enugu_High_Confidence_Suitability_30m.tif"
)

PLANNING_PRIORITY_FILE = (
    OUTPUT_DIR
    / "Enugu_Urban_Development_Planning_Priority_30m.tif"
)

SUITABILITY_CONFIDENCE_TABLE = (
    TABLE_DIR
    / "Enugu_Suitability_by_Confidence_Crosstab.csv"
)

SUITABILITY_RELIABILITY_TABLE = (
    TABLE_DIR
    / "Enugu_Suitability_Class_Reliability_Statistics.csv"
)

CONFIDENCE_AREA_TABLE = (
    TABLE_DIR
    / "Enugu_Confidence_Class_Area_Statistics.csv"
)

PRIORITY_AREA_TABLE = (
    TABLE_DIR
    / "Enugu_Planning_Priority_Area_Statistics.csv"
)

PRIORITY_QUICKLOOK_FILE = (
    MAP_DIR
    / "Enugu_Planning_Priority_Quicklook.png"
)

CONFIDENCE_QUICKLOOK_FILE = (
    MAP_DIR
    / "Enugu_Model_Confidence_Classes_Quicklook.png"
)

REGISTER_FILE = (
    ADMIN_DIR
    / "Stage_11B_Planning_Priority_Register.json"
)

VALIDATION_FILE = (
    ADMIN_DIR
    / "Stage_11B_Validation_Register.csv"
)


# ==========================================================================================
# 4. CLASS DEFINITIONS
# ==========================================================================================

FLOAT_NODATA = -9999.0
BYTE_NODATA = 255

TILE_SIZE = 512

SUITABILITY_LABELS = {
    1: "Very Low",
    2: "Low",
    3: "Moderate",
    4: "High",
    5: "Very High"
}

CONFIDENCE_LABELS = {
    1: "Low Confidence",
    2: "Moderate Confidence",
    3: "High Confidence"
}

# Confidence thresholds:
# Low:      <0.40
# Moderate: 0.40–<0.70
# High:     ≥0.70

PRIORITY_LABELS = {
    1: "Very Low Priority",
    2: "Low Priority",
    3: "Moderate Priority",
    4: "High Priority",
    5: "Very High Priority"
}

# Priority is intentionally conservative:
#
# 1 — Very Low:
#     Very-low or low suitability
#
# 2 — Low:
#     Moderate suitability with low confidence
#
# 3 — Moderate:
#     Moderate suitability with moderate/high confidence,
#     or high suitability with low confidence
#
# 4 — High:
#     High suitability with moderate/high confidence,
#     or very-high suitability with moderate confidence
#
# 5 — Very High:
#     Very-high suitability with high confidence


# ==========================================================================================
# 5. HELPER FUNCTIONS
# ==========================================================================================

def print_heading(title):
    print("\n" + title)
    print("-" * 120)


def validate_file(path, label):
    if not path.exists():
        raise FileNotFoundError(
            f"{label} was not found:\n{path}"
        )

    if path.stat().st_size == 0:
        raise RuntimeError(
            f"{label} exists but is empty:\n{path}"
        )


def make_windows(width, height, tile_size):
    for row_offset in range(
        0,
        height,
        tile_size
    ):
        window_height = min(
            tile_size,
            height - row_offset
        )

        for column_offset in range(
            0,
            width,
            tile_size
        ):
            window_width = min(
                tile_size,
                width - column_offset
            )

            yield Window(
                column_offset,
                row_offset,
                window_width,
                window_height
            )


def build_byte_profile(reference):
    profile = reference.profile.copy()

    profile.update({
        "driver": "GTiff",
        "count": 1,
        "dtype": "uint8",
        "nodata": BYTE_NODATA,
        "compress": "DEFLATE",
        "predictor": 2,
        "tiled": True,
        "blockxsize": 256,
        "blockysize": 256,
        "BIGTIFF": "IF_SAFER"
    })

    return profile


def calculate_pixel_area_km2(transform):
    pixel_area_m2 = abs(
        (
            transform.a
            * transform.e
        )
        -
        (
            transform.b
            * transform.d
        )
    )

    return pixel_area_m2 / 1_000_000.0


def classify_confidence(confidence):
    classes = np.zeros(
        confidence.shape,
        dtype=np.uint8
    )

    classes[
        confidence < 0.40
    ] = 1

    classes[
        (
            confidence >= 0.40
        )
        &
        (
            confidence < 0.70
        )
    ] = 2

    classes[
        confidence >= 0.70
    ] = 3

    return classes


def classify_priority(suitability, confidence_class):
    priority = np.zeros(
        suitability.shape,
        dtype=np.uint8
    )

    # Very-low and low suitability.
    priority[
        (
            suitability == 1
        )
        |
        (
            suitability == 2
        )
    ] = 1

    # Moderate suitability with low confidence.
    priority[
        (
            suitability == 3
        )
        &
        (
            confidence_class == 1
        )
    ] = 2

    # Moderate suitability with moderate or high confidence.
    priority[
        (
            suitability == 3
        )
        &
        (
            confidence_class >= 2
        )
    ] = 3

    # High suitability with low confidence.
    priority[
        (
            suitability == 4
        )
        &
        (
            confidence_class == 1
        )
    ] = 3

    # High suitability with moderate or high confidence.
    priority[
        (
            suitability == 4
        )
        &
        (
            confidence_class >= 2
        )
    ] = 4

    # Very-high suitability with low confidence.
    priority[
        (
            suitability == 5
        )
        &
        (
            confidence_class == 1
        )
    ] = 3

    # Very-high suitability with moderate confidence.
    priority[
        (
            suitability == 5
        )
        &
        (
            confidence_class == 2
        )
    ] = 4

    # Very-high suitability with high confidence.
    priority[
        (
            suitability == 5
        )
        &
        (
            confidence_class == 3
        )
    ] = 5

    return priority


def read_quicklook(path, maximum_dimension=1600):
    with rasterio.open(
        path
    ) as source:
        scale = max(
            source.width / maximum_dimension,
            source.height / maximum_dimension,
            1
        )

        output_width = max(
            1,
            int(
                source.width / scale
            )
        )

        output_height = max(
            1,
            int(
                source.height / scale
            )
        )

        return source.read(
            1,
            out_shape=(
                output_height,
                output_width
            ),
            resampling=Resampling.nearest,
            masked=True
        )


# ==========================================================================================
# 6. START
# ==========================================================================================

print("=" * 120)
print("STAGE 11B — SUITABILITY RELIABILITY AND PLANNING-PRIORITY VALIDATION")
print("=" * 120)


# ==========================================================================================
# 7. INPUT VALIDATION
# ==========================================================================================

required_inputs = {
    "Raw probability": PROBABILITY_FILE,
    "Constrained probability": CONSTRAINED_PROBABILITY_FILE,
    "Model uncertainty": UNCERTAINTY_FILE,
    "Model confidence": CONFIDENCE_FILE,
    "Applicability mask": APPLICABILITY_FILE,
    "Suitability classes": SUITABILITY_FILE,
    "Planning constraints": CONSTRAINT_FILE
}

print_heading(
    "REQUIRED STAGE 11A INPUT CHECK"
)

for label, path in required_inputs.items():
    validate_file(
        path,
        label
    )

    print(
        f"✓ {label}: {path}"
    )

if STAGE_11A_REGISTER.exists():
    print(
        f"✓ Stage 11A register: {STAGE_11A_REGISTER}"
    )
else:
    print(
        "⚠ Stage 11A register was not found."
    )


# ==========================================================================================
# 8. GRID CONSISTENCY CHECK
# ==========================================================================================

print_heading(
    "RASTER GRID CONSISTENCY"
)

grid_records = []

for label, path in required_inputs.items():
    with rasterio.open(
        path
    ) as source:
        grid_records.append({
            "Layer": label,
            "CRS": str(
                source.crs
            ),
            "Width": int(
                source.width
            ),
            "Height": int(
                source.height
            ),
            "Transform": tuple(
                source.transform
            ),
            "Data_Type": source.dtypes[0],
            "NoData": source.nodata
        })

grid_df = pd.DataFrame(
    grid_records
)

reference_record = grid_records[0]

grids_match = all(
    record["CRS"]
    ==
    reference_record["CRS"]
    and
    record["Width"]
    ==
    reference_record["Width"]
    and
    record["Height"]
    ==
    reference_record["Height"]
    and
    np.allclose(
        np.asarray(
            record["Transform"]
        ),
        np.asarray(
            reference_record["Transform"]
        )
    )
    for record in grid_records
)

print(
    grid_df[
        [
            "Layer",
            "CRS",
            "Width",
            "Height",
            "Data_Type",
            "NoData"
        ]
    ].to_string(
        index=False
    )
)

if not grids_match:
    raise RuntimeError(
        "Stage 11A raster grids are not identical."
    )

print(
    "✓ All Stage 11A rasters use the same grid."
)


# ==========================================================================================
# 9. CREATE OUTPUT RASTERS AND STREAM STATISTICS
# ==========================================================================================

with ExitStack() as stack:
    probability_source = stack.enter_context(
        rasterio.open(
            PROBABILITY_FILE
        )
    )

    constrained_probability_source = stack.enter_context(
        rasterio.open(
            CONSTRAINED_PROBABILITY_FILE
        )
    )

    uncertainty_source = stack.enter_context(
        rasterio.open(
            UNCERTAINTY_FILE
        )
    )

    confidence_source = stack.enter_context(
        rasterio.open(
            CONFIDENCE_FILE
        )
    )

    applicability_source = stack.enter_context(
        rasterio.open(
            APPLICABILITY_FILE
        )
    )

    suitability_source = stack.enter_context(
        rasterio.open(
            SUITABILITY_FILE
        )
    )

    constraint_source = stack.enter_context(
        rasterio.open(
            CONSTRAINT_FILE
        )
    )

    byte_profile = build_byte_profile(
        suitability_source
    )

    confidence_destination = stack.enter_context(
        rasterio.open(
            CONFIDENCE_CLASS_FILE,
            "w",
            **byte_profile
        )
    )

    high_confidence_destination = stack.enter_context(
        rasterio.open(
            HIGH_CONFIDENCE_SUITABILITY_FILE,
            "w",
            **byte_profile
        )
    )

    priority_destination = stack.enter_context(
        rasterio.open(
            PLANNING_PRIORITY_FILE,
            "w",
            **byte_profile
        )
    )

    confidence_destination.set_band_description(
        1,
        "Confidence_Class"
    )

    high_confidence_destination.set_band_description(
        1,
        "High_Confidence_Suitability_Class"
    )

    priority_destination.set_band_description(
        1,
        "Planning_Priority_Class"
    )

    pixel_area_km2 = calculate_pixel_area_km2(
        suitability_source.transform
    )

    width = suitability_source.width
    height = suitability_source.height

    windows = list(
        make_windows(
            width,
            height,
            TILE_SIZE
        )
    )

    suitability_counts = {
        code: 0
        for code in SUITABILITY_LABELS
    }

    confidence_counts = {
        code: 0
        for code in CONFIDENCE_LABELS
    }

    priority_counts = {
        code: 0
        for code in PRIORITY_LABELS
    }

    cross_tab_counts = {
        (
            suitability_code,
            confidence_code
        ): 0
        for suitability_code in SUITABILITY_LABELS
        for confidence_code in CONFIDENCE_LABELS
    }

    suitability_probability_sum = {
        code: 0.0
        for code in SUITABILITY_LABELS
    }

    suitability_uncertainty_sum = {
        code: 0.0
        for code in SUITABILITY_LABELS
    }

    suitability_confidence_sum = {
        code: 0.0
        for code in SUITABILITY_LABELS
    }

    suitability_min_probability = {
        code: np.inf
        for code in SUITABILITY_LABELS
    }

    suitability_max_probability = {
        code: -np.inf
        for code in SUITABILITY_LABELS
    }

    valid_applicable_pixels = 0
    high_confidence_pixels = 0
    high_or_very_high_pixels = 0
    high_confidence_high_suitability_pixels = 0

    print_heading(
        "PROCESSING RELIABILITY AND PRIORITY SURFACES"
    )

    for window_number, window in enumerate(
        windows,
        start=1
    ):
        probability = probability_source.read(
            1,
            window=window,
            masked=True
        )

        constrained_probability = constrained_probability_source.read(
            1,
            window=window,
            masked=True
        )

        uncertainty = uncertainty_source.read(
            1,
            window=window,
            masked=True
        )

        confidence = confidence_source.read(
            1,
            window=window,
            masked=True
        )

        applicability = applicability_source.read(
            1,
            window=window,
            masked=True
        )

        suitability = suitability_source.read(
            1,
            window=window,
            masked=True
        )

        constraints = constraint_source.read(
            1,
            window=window,
            masked=True
        )

        probability_array = probability.filled(
            np.nan
        ).astype(
            np.float64
        )

        constrained_probability_array = constrained_probability.filled(
            np.nan
        ).astype(
            np.float64
        )

        uncertainty_array = uncertainty.filled(
            np.nan
        ).astype(
            np.float64
        )

        confidence_array = confidence.filled(
            np.nan
        ).astype(
            np.float64
        )

        applicability_array = applicability.filled(
            BYTE_NODATA
        ).astype(
            np.uint8
        )

        suitability_array = suitability.filled(
            BYTE_NODATA
        ).astype(
            np.uint8
        )

        constraint_array = constraints.filled(
            BYTE_NODATA
        ).astype(
            np.uint8
        )

        valid_mask = (
            applicability_array == 1
        )

        valid_mask &= np.isfinite(
            constrained_probability_array
        )

        valid_mask &= np.isfinite(
            uncertainty_array
        )

        valid_mask &= np.isfinite(
            confidence_array
        )

        valid_mask &= np.isin(
            suitability_array,
            [
                1,
                2,
                3,
                4,
                5
            ]
        )

        confidence_output = np.full(
            suitability_array.shape,
            BYTE_NODATA,
            dtype=np.uint8
        )

        high_confidence_output = np.full(
            suitability_array.shape,
            BYTE_NODATA,
            dtype=np.uint8
        )

        priority_output = np.full(
            suitability_array.shape,
            BYTE_NODATA,
            dtype=np.uint8
        )

        if np.any(
            valid_mask
        ):
            confidence_classes = classify_confidence(
                confidence_array[
                    valid_mask
                ]
            )

            suitability_values = suitability_array[
                valid_mask
            ]

            priority_values = classify_priority(
                suitability_values,
                confidence_classes
            )

            confidence_output[
                valid_mask
            ] = confidence_classes

            priority_output[
                valid_mask
            ] = priority_values

            # High-confidence suitability retains suitability class only
            # where model confidence is at least 0.70.
            high_confidence_mask = (
                valid_mask
                &
                (
                    confidence_array >= 0.70
                )
            )

            high_confidence_output[
                high_confidence_mask
            ] = suitability_array[
                high_confidence_mask
            ]

            valid_count = int(
                np.count_nonzero(
                    valid_mask
                )
            )

            valid_applicable_pixels += valid_count

            high_confidence_pixels += int(
                np.count_nonzero(
                    high_confidence_mask
                )
            )

            high_suitability_mask = (
                valid_mask
                &
                (
                    suitability_array >= 4
                )
            )

            high_or_very_high_pixels += int(
                np.count_nonzero(
                    high_suitability_mask
                )
            )

            high_confidence_high_suitability_pixels += int(
                np.count_nonzero(
                    high_suitability_mask
                    &
                    high_confidence_mask
                )
            )

            for suitability_code in SUITABILITY_LABELS:
                class_mask = (
                    valid_mask
                    &
                    (
                        suitability_array
                        ==
                        suitability_code
                    )
                )

                class_count = int(
                    np.count_nonzero(
                        class_mask
                    )
                )

                suitability_counts[
                    suitability_code
                ] += class_count

                if class_count > 0:
                    class_probabilities = constrained_probability_array[
                        class_mask
                    ]

                    class_uncertainty = uncertainty_array[
                        class_mask
                    ]

                    class_confidence = confidence_array[
                        class_mask
                    ]

                    suitability_probability_sum[
                        suitability_code
                    ] += float(
                        np.sum(
                            class_probabilities
                        )
                    )

                    suitability_uncertainty_sum[
                        suitability_code
                    ] += float(
                        np.sum(
                            class_uncertainty
                        )
                    )

                    suitability_confidence_sum[
                        suitability_code
                    ] += float(
                        np.sum(
                            class_confidence
                        )
                    )

                    suitability_min_probability[
                        suitability_code
                    ] = min(
                        suitability_min_probability[
                            suitability_code
                        ],
                        float(
                            np.min(
                                class_probabilities
                            )
                        )
                    )

                    suitability_max_probability[
                        suitability_code
                    ] = max(
                        suitability_max_probability[
                            suitability_code
                        ],
                        float(
                            np.max(
                                class_probabilities
                            )
                        )
                    )

                for confidence_code in CONFIDENCE_LABELS:
                    joint_count = int(
                        np.count_nonzero(
                            class_mask
                            &
                            (
                                confidence_output
                                ==
                                confidence_code
                            )
                        )
                    )

                    cross_tab_counts[
                        (
                            suitability_code,
                            confidence_code
                        )
                    ] += joint_count

            for confidence_code in CONFIDENCE_LABELS:
                confidence_counts[
                    confidence_code
                ] += int(
                    np.count_nonzero(
                        valid_mask
                        &
                        (
                            confidence_output
                            ==
                            confidence_code
                        )
                    )
                )

            for priority_code in PRIORITY_LABELS:
                priority_counts[
                    priority_code
                ] += int(
                    np.count_nonzero(
                        valid_mask
                        &
                        (
                            priority_output
                            ==
                            priority_code
                        )
                    )
                )

        confidence_destination.write(
            confidence_output,
            1,
            window=window
        )

        high_confidence_destination.write(
            high_confidence_output,
            1,
            window=window
        )

        priority_destination.write(
            priority_output,
            1,
            window=window
        )

        if (
            window_number == 1
            or
            window_number % 20 == 0
            or
            window_number == len(
                windows
            )
        ):
            progress = (
                window_number
                /
                len(
                    windows
                )
                *
                100
            )

            print(
                f"Window {window_number:,}/{len(windows):,} "
                f"({progress:.1f}%) completed"
            )


# ==========================================================================================
# 10. SUITABILITY × CONFIDENCE CROSS-TABULATION
# ==========================================================================================

cross_tab_records = []

for suitability_code, suitability_label in SUITABILITY_LABELS.items():
    suitability_total = suitability_counts[
        suitability_code
    ]

    for confidence_code, confidence_label in CONFIDENCE_LABELS.items():
        pixel_count = cross_tab_counts[
            (
                suitability_code,
                confidence_code
            )
        ]

        area_km2 = (
            pixel_count
            *
            pixel_area_km2
        )

        percentage_within_suitability = (
            pixel_count
            /
            suitability_total
            *
            100
            if suitability_total > 0
            else np.nan
        )

        cross_tab_records.append({
            "Suitability_Code": suitability_code,
            "Suitability_Class": suitability_label,
            "Confidence_Code": confidence_code,
            "Confidence_Class": confidence_label,
            "Pixel_Count": pixel_count,
            "Area_km2": area_km2,
            "Percentage_Within_Suitability_Class": percentage_within_suitability
        })

cross_tab_df = pd.DataFrame(
    cross_tab_records
)

cross_tab_df.to_csv(
    SUITABILITY_CONFIDENCE_TABLE,
    index=False
)


# ==========================================================================================
# 11. SUITABILITY RELIABILITY TABLE
# ==========================================================================================

reliability_records = []

for suitability_code, suitability_label in SUITABILITY_LABELS.items():
    pixel_count = suitability_counts[
        suitability_code
    ]

    area_km2 = (
        pixel_count
        *
        pixel_area_km2
    )

    if pixel_count > 0:
        mean_probability = (
            suitability_probability_sum[
                suitability_code
            ]
            /
            pixel_count
        )

        mean_uncertainty = (
            suitability_uncertainty_sum[
                suitability_code
            ]
            /
            pixel_count
        )

        mean_confidence = (
            suitability_confidence_sum[
                suitability_code
            ]
            /
            pixel_count
        )

        minimum_probability = suitability_min_probability[
            suitability_code
        ]

        maximum_probability = suitability_max_probability[
            suitability_code
        ]
    else:
        mean_probability = np.nan
        mean_uncertainty = np.nan
        mean_confidence = np.nan
        minimum_probability = np.nan
        maximum_probability = np.nan

    high_confidence_count = cross_tab_counts[
        (
            suitability_code,
            3
        )
    ]

    high_confidence_percentage = (
        high_confidence_count
        /
        pixel_count
        *
        100
        if pixel_count > 0
        else np.nan
    )

    reliability_records.append({
        "Suitability_Code": suitability_code,
        "Suitability_Class": suitability_label,
        "Pixel_Count": pixel_count,
        "Area_km2": area_km2,
        "Minimum_Probability": minimum_probability,
        "Maximum_Probability": maximum_probability,
        "Mean_Probability": mean_probability,
        "Mean_Uncertainty": mean_uncertainty,
        "Mean_Confidence": mean_confidence,
        "High_Confidence_Pixel_Count": high_confidence_count,
        "High_Confidence_Percentage": high_confidence_percentage
    })

reliability_df = pd.DataFrame(
    reliability_records
)

reliability_df.to_csv(
    SUITABILITY_RELIABILITY_TABLE,
    index=False
)


# ==========================================================================================
# 12. CONFIDENCE AREA TABLE
# ==========================================================================================

confidence_records = []

for confidence_code, confidence_label in CONFIDENCE_LABELS.items():
    pixel_count = confidence_counts[
        confidence_code
    ]

    area_km2 = (
        pixel_count
        *
        pixel_area_km2
    )

    percentage = (
        pixel_count
        /
        valid_applicable_pixels
        *
        100
        if valid_applicable_pixels > 0
        else np.nan
    )

    confidence_records.append({
        "Confidence_Code": confidence_code,
        "Confidence_Class": confidence_label,
        "Pixel_Count": pixel_count,
        "Area_km2": area_km2,
        "Percentage_of_Applicable_Land": percentage
    })

confidence_df = pd.DataFrame(
    confidence_records
)

confidence_df.to_csv(
    CONFIDENCE_AREA_TABLE,
    index=False
)


# ==========================================================================================
# 13. PLANNING-PRIORITY AREA TABLE
# ==========================================================================================

priority_records = []

for priority_code, priority_label in PRIORITY_LABELS.items():
    pixel_count = priority_counts[
        priority_code
    ]

    area_km2 = (
        pixel_count
        *
        pixel_area_km2
    )

    percentage = (
        pixel_count
        /
        valid_applicable_pixels
        *
        100
        if valid_applicable_pixels > 0
        else np.nan
    )

    priority_records.append({
        "Priority_Code": priority_code,
        "Planning_Priority": priority_label,
        "Pixel_Count": pixel_count,
        "Area_km2": area_km2,
        "Percentage_of_Applicable_Land": percentage
    })

priority_df = pd.DataFrame(
    priority_records
)

priority_df.to_csv(
    PRIORITY_AREA_TABLE,
    index=False
)


# ==========================================================================================
# 14. QUICKLOOK MAPS
# ==========================================================================================

print_heading(
    "GENERATING QUICKLOOK MAPS"
)

priority_quicklook = read_quicklook(
    PLANNING_PRIORITY_FILE
)

priority_colormap = ListedColormap([
    "#d73027",
    "#fc8d59",
    "#fee08b",
    "#91cf60",
    "#1a9850"
])

priority_norm = BoundaryNorm(
    [
        0.5,
        1.5,
        2.5,
        3.5,
        4.5,
        5.5
    ],
    priority_colormap.N
)

figure, axis = plt.subplots(
    figsize=(
        10,
        11
    )
)

image = axis.imshow(
    priority_quicklook,
    cmap=priority_colormap,
    norm=priority_norm
)

axis.set_title(
    "Enugu State Urban Development Planning Priority\n"
    "Suitability Adjusted for Model Confidence",
    fontsize=15,
    fontweight="bold"
)

axis.axis(
    "off"
)

colour_bar = figure.colorbar(
    image,
    ax=axis,
    fraction=0.035,
    pad=0.02,
    ticks=[
        1,
        2,
        3,
        4,
        5
    ]
)

colour_bar.ax.set_yticklabels([
    "Very Low",
    "Low",
    "Moderate",
    "High",
    "Very High"
])

colour_bar.set_label(
    "Planning-priority class",
    fontsize=11
)

figure.text(
    0.5,
    0.025,
    "Priority classes combine urban-expansion suitability with model confidence.",
    ha="center",
    fontsize=9
)

figure.tight_layout(
    rect=[
        0,
        0.04,
        1,
        1
    ]
)

figure.savefig(
    PRIORITY_QUICKLOOK_FILE,
    dpi=300,
    bbox_inches="tight"
)

plt.close(
    figure
)


confidence_quicklook = read_quicklook(
    CONFIDENCE_CLASS_FILE
)

confidence_colormap = ListedColormap([
    "#d73027",
    "#fee08b",
    "#1a9850"
])

confidence_norm = BoundaryNorm(
    [
        0.5,
        1.5,
        2.5,
        3.5
    ],
    confidence_colormap.N
)

figure, axis = plt.subplots(
    figsize=(
        10,
        11
    )
)

image = axis.imshow(
    confidence_quicklook,
    cmap=confidence_colormap,
    norm=confidence_norm
)

axis.set_title(
    "Enugu State Model Confidence Classes",
    fontsize=15,
    fontweight="bold"
)

axis.axis(
    "off"
)

colour_bar = figure.colorbar(
    image,
    ax=axis,
    fraction=0.035,
    pad=0.02,
    ticks=[
        1,
        2,
        3
    ]
)

colour_bar.ax.set_yticklabels([
    "Low",
    "Moderate",
    "High"
])

colour_bar.set_label(
    "Model-confidence class",
    fontsize=11
)

figure.tight_layout()

figure.savefig(
    CONFIDENCE_QUICKLOOK_FILE,
    dpi=300,
    bbox_inches="tight"
)

plt.close(
    figure
)

print(
    f"✓ Planning-priority quicklook: {PRIORITY_QUICKLOOK_FILE}"
)

print(
    f"✓ Confidence quicklook: {CONFIDENCE_QUICKLOOK_FILE}"
)


# ==========================================================================================
# 15. VALIDATION
# ==========================================================================================

validation_records = []


def add_validation(check, passed, observed, expected):
    validation_records.append({
        "Check": check,
        "Passed": bool(
            passed
        ),
        "Observed": str(
            observed
        ),
        "Expected": str(
            expected
        )
    })


output_files = [
    CONFIDENCE_CLASS_FILE,
    HIGH_CONFIDENCE_SUITABILITY_FILE,
    PLANNING_PRIORITY_FILE,
    SUITABILITY_CONFIDENCE_TABLE,
    SUITABILITY_RELIABILITY_TABLE,
    CONFIDENCE_AREA_TABLE,
    PRIORITY_AREA_TABLE,
    PRIORITY_QUICKLOOK_FILE,
    CONFIDENCE_QUICKLOOK_FILE
]

add_validation(
    "All required Stage 11B outputs were created",
    all(
        path.exists()
        and
        path.stat().st_size > 0
        for path in output_files
    ),
    [
        str(
            path
        )
        for path in output_files
        if path.exists()
    ],
    "All outputs exist and are non-empty"
)

add_validation(
    "All Stage 11A input rasters use the same grid",
    grids_match,
    grids_match,
    True
)

add_validation(
    "Suitability counts equal applicable-pixel count",
    sum(
        suitability_counts.values()
    )
    ==
    valid_applicable_pixels,
    sum(
        suitability_counts.values()
    ),
    valid_applicable_pixels
)

add_validation(
    "Confidence counts equal applicable-pixel count",
    sum(
        confidence_counts.values()
    )
    ==
    valid_applicable_pixels,
    sum(
        confidence_counts.values()
    ),
    valid_applicable_pixels
)

add_validation(
    "Planning-priority counts equal applicable-pixel count",
    sum(
        priority_counts.values()
    )
    ==
    valid_applicable_pixels,
    sum(
        priority_counts.values()
    ),
    valid_applicable_pixels
)

add_validation(
    "Suitability-confidence cross-tab equals applicable-pixel count",
    sum(
        cross_tab_counts.values()
    )
    ==
    valid_applicable_pixels,
    sum(
        cross_tab_counts.values()
    ),
    valid_applicable_pixels
)

add_validation(
    "High-confidence pixels do not exceed applicable pixels",
    high_confidence_pixels
    <=
    valid_applicable_pixels,
    high_confidence_pixels,
    f"≤ {valid_applicable_pixels}"
)

add_validation(
    "High-confidence high-suitability pixels do not exceed all high-suitability pixels",
    high_confidence_high_suitability_pixels
    <=
    high_or_very_high_pixels,
    high_confidence_high_suitability_pixels,
    f"≤ {high_or_very_high_pixels}"
)

mean_probabilities = reliability_df[
    "Mean_Probability"
].to_numpy()

probability_order_valid = np.all(
    np.diff(
        mean_probabilities
    )
    >
    0
)

add_validation(
    "Mean probability increases across suitability classes",
    probability_order_valid,
    mean_probabilities.tolist(),
    "Strictly increasing from Very Low to Very High"
)

probability_ranges_valid = all(
    (
        row.Minimum_Probability >= 0
        and
        row.Maximum_Probability <= 1
    )
    for row in reliability_df.itertuples()
)

add_validation(
    "Suitability-class probability values remain within 0–1",
    probability_ranges_valid,
    reliability_df[
        [
            "Minimum_Probability",
            "Maximum_Probability"
        ]
    ].values.tolist(),
    "0–1"
)

validation_df = pd.DataFrame(
    validation_records
)

validation_df.to_csv(
    VALIDATION_FILE,
    index=False
)

failed_validation = validation_df.loc[
    ~validation_df[
        "Passed"
    ]
]

if not failed_validation.empty:
    print_heading(
        "FAILED VALIDATION CHECKS"
    )

    print(
        failed_validation.to_string(
            index=False
        )
    )

    raise RuntimeError(
        "Stage 11B processing completed, but validation failed."
    )


# ==========================================================================================
# 16. REGISTER
# ==========================================================================================

high_confidence_area_km2 = (
    high_confidence_pixels
    *
    pixel_area_km2
)

high_suitability_area_km2 = (
    high_or_very_high_pixels
    *
    pixel_area_km2
)

high_confidence_high_suitability_area_km2 = (
    high_confidence_high_suitability_pixels
    *
    pixel_area_km2
)

register = {
    "stage": "11B",
    "stage_name": (
        "Suitability Reliability and Planning-Priority Validation"
    ),
    "completion_time_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "study_area": "Enugu State, Nigeria",
    "confidence_thresholds": {
        "low": "< 0.40",
        "moderate": "0.40 to < 0.70",
        "high": ">= 0.70"
    },
    "priority_method": (
        "Suitability classes adjusted by model-confidence class"
    ),
    "pixel_area_km2": float(
        pixel_area_km2
    ),
    "applicable_pixels": int(
        valid_applicable_pixels
    ),
    "applicable_area_km2": float(
        valid_applicable_pixels
        *
        pixel_area_km2
    ),
    "high_confidence_pixels": int(
        high_confidence_pixels
    ),
    "high_confidence_area_km2": float(
        high_confidence_area_km2
    ),
    "high_or_very_high_suitability_pixels": int(
        high_or_very_high_pixels
    ),
    "high_or_very_high_suitability_area_km2": float(
        high_suitability_area_km2
    ),
    "high_confidence_high_suitability_pixels": int(
        high_confidence_high_suitability_pixels
    ),
    "high_confidence_high_suitability_area_km2": float(
        high_confidence_high_suitability_area_km2
    ),
    "outputs": {
        "confidence_classes": str(
            CONFIDENCE_CLASS_FILE
        ),
        "high_confidence_suitability": str(
            HIGH_CONFIDENCE_SUITABILITY_FILE
        ),
        "planning_priority": str(
            PLANNING_PRIORITY_FILE
        ),
        "suitability_confidence_table": str(
            SUITABILITY_CONFIDENCE_TABLE
        ),
        "suitability_reliability_table": str(
            SUITABILITY_RELIABILITY_TABLE
        ),
        "confidence_area_table": str(
            CONFIDENCE_AREA_TABLE
        ),
        "priority_area_table": str(
            PRIORITY_AREA_TABLE
        ),
        "priority_quicklook": str(
            PRIORITY_QUICKLOOK_FILE
        ),
        "confidence_quicklook": str(
            CONFIDENCE_QUICKLOOK_FILE
        ),
        "validation_register": str(
            VALIDATION_FILE
        )
    },
    "validation_passed": True
}

with open(
    REGISTER_FILE,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        register,
        file,
        indent=4
    )


# ==========================================================================================
# 17. FINAL OUTPUT
# ==========================================================================================

print_heading(
    "CONFIDENCE CLASS DISTRIBUTION"
)

print(
    confidence_df.to_string(
        index=False,
        formatters={
            "Area_km2":
                lambda value: f"{value:,.2f}",

            "Percentage_of_Applicable_Land":
                lambda value: f"{value:.2f}%"
        }
    )
)

print_heading(
    "SUITABILITY RELIABILITY STATISTICS"
)

print(
    reliability_df.to_string(
        index=False,
        formatters={
            "Area_km2":
                lambda value: f"{value:,.2f}",

            "Minimum_Probability":
                lambda value: f"{value:.4f}",

            "Maximum_Probability":
                lambda value: f"{value:.4f}",

            "Mean_Probability":
                lambda value: f"{value:.4f}",

            "Mean_Uncertainty":
                lambda value: f"{value:.4f}",

            "Mean_Confidence":
                lambda value: f"{value:.4f}",

            "High_Confidence_Percentage":
                lambda value: f"{value:.2f}%"
        }
    )
)

print_heading(
    "PLANNING-PRIORITY DISTRIBUTION"
)

print(
    priority_df.to_string(
        index=False,
        formatters={
            "Area_km2":
                lambda value: f"{value:,.2f}",

            "Percentage_of_Applicable_Land":
                lambda value: f"{value:.2f}%"
        }
    )
)

print_heading(
    "KEY RELIABILITY RESULTS"
)

print(
    f"Applicable area: "
    f"{valid_applicable_pixels * pixel_area_km2:,.2f} km²"
)

print(
    f"High-confidence area: "
    f"{high_confidence_area_km2:,.2f} km²"
)

print(
    f"High and very-high suitability area: "
    f"{high_suitability_area_km2:,.2f} km²"
)

print(
    f"High-confidence high/very-high suitability area: "
    f"{high_confidence_high_suitability_area_km2:,.2f} km²"
)

print_heading(
    "VALIDATION RESULTS"
)

print(
    validation_df.to_string(
        index=False
    )
)

print_heading(
    "OUTPUT FILES"
)

for output_path in output_files:
    print(
        f"✓ {output_path}"
    )

print(
    f"✓ {REGISTER_FILE}"
)

print(
    f"✓ {VALIDATION_FILE}"
)

print("=" * 120)
print("STAGE 11B COMPLETED SUCCESSFULLY")
print("=" * 120)
print(
    "✓ Confidence classes were generated from the Stage 11A confidence surface."
)
print(
    "✓ Suitability classes were evaluated against probability, uncertainty and confidence."
)
print(
    "✓ High-confidence suitability areas were isolated."
)
print(
    "✓ Planning-priority zones were generated using suitability and confidence."
)
print(
    "✓ Suitability-confidence cross-tabulation and area statistics were saved."
)
print(
    "✓ All Stage 11B validation checks passed."
)
print("=" * 120)

In [ ]:
# ==========================================================================================
# PROJECT 7 — STAGE 12A–12B
# PROFESSIONAL CARTOGRAPHIC ENGINE AND FINAL MAP PRODUCTION
#
# Study area:
# Enugu State, Nigeria
#
# Final maps:
#   01. Urban Expansion Probability
#   02. Planning-Constrained Expansion Probability
#   03. Urban Development Suitability
#   04. Urban Development Planning Priority
#   05. Model Confidence
#   06. Model Uncertainty
#   07. High-Confidence Suitability
#
# Export formats:
#   PNG
#   PDF
#
# Outputs:
#   Publication maps
#   Cartographic register
#   Input register
#   Validation register
# ==========================================================================================


# ==========================================================================================
# 1. INSTALL AND IMPORT REQUIRED LIBRARIES
# ==========================================================================================

import sys
import subprocess
import importlib.util


def install_if_missing(package_name, import_name=None):
    import_name = import_name or package_name

    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                package_name
            ]
        )


install_if_missing(
    "geopandas",
    "geopandas"
)

install_if_missing(
    "rasterio",
    "rasterio"
)

install_if_missing(
    "matplotlib",
    "matplotlib"
)

install_if_missing(
    "shapely",
    "shapely"
)


from pathlib import Path
from datetime import datetime, timezone
from matplotlib.colors import (
    BoundaryNorm,
    ListedColormap,
    Normalize
)
from matplotlib.lines import Line2D
from matplotlib.patches import (
    FancyArrowPatch,
    Rectangle
)
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

import gc
import json
import math
import warnings

import geopandas as gpd
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio

from rasterio.enums import Resampling
from rasterio.plot import plotting_extent
from shapely.geometry import box

warnings.filterwarnings(
    "ignore"
)

matplotlib.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "axes.titlesize": 15,
    "axes.labelsize": 10,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "figure.titlesize": 17,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})


# ==========================================================================================
# 2. PROJECT PATHS
# ==========================================================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

STAGE_11A_DIR = (
    PROJECT_ROOT
    / "12_Statewide_Suitability"
)

STAGE_11B_DIR = (
    PROJECT_ROOT
    / "13_Planning_Priority"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "14_Final_Cartographic_Maps"
)

PNG_DIR = (
    OUTPUT_ROOT
    / "PNG"
)

PDF_DIR = (
    OUTPUT_ROOT
    / "PDF"
)

TABLE_DIR = (
    OUTPUT_ROOT
    / "Tables"
)

ADMIN_DIR = (
    PROJECT_ROOT
    / "00_Project_Admin"
)

for directory in [
    OUTPUT_ROOT,
    PNG_DIR,
    PDF_DIR,
    TABLE_DIR,
    ADMIN_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ==========================================================================================
# 3. FINAL MAP INPUTS
# ==========================================================================================

MAP_INPUTS = {
    "01_Urban_Expansion_Probability": {
        "path": (
            STAGE_11A_DIR
            / "Enugu_Urban_Expansion_Probability_2025_30m.tif"
        ),
        "title": (
            "Machine-Learning Predicted Urban Expansion Probability"
        ),
        "subtitle": (
            "Enugu State, Nigeria — Expansion from the 2020 baseline"
        ),
        "map_type": "continuous",
        "cmap": "magma",
        "minimum": 0.0,
        "maximum": 1.0,
        "colourbar_label": "Predicted probability",
        "method_note": (
            "Leakage-corrected Extra Trees model; unconstrained statewide probability."
        )
    },

    "02_Planning_Constrained_Expansion_Probability": {
        "path": (
            STAGE_11A_DIR
            / "Enugu_Planning_Constrained_Expansion_Probability_30m.tif"
        ),
        "title": (
            "Planning-Constrained Urban Expansion Probability"
        ),
        "subtitle": (
            "Enugu State, Nigeria — Environmentally applicable land"
        ),
        "map_type": "continuous",
        "cmap": "viridis",
        "minimum": 0.0,
        "maximum": 1.0,
        "colourbar_label": "Constrained probability",
        "method_note": (
            "Existing built-up land, water, steep terrain and environmental setbacks excluded."
        )
    },

    "03_Urban_Development_Suitability": {
        "path": (
            STAGE_11A_DIR
            / "Enugu_Urban_Development_Suitability_Classes_30m.tif"
        ),
        "title": (
            "Urban Development Suitability"
        ),
        "subtitle": (
            "Enugu State, Nigeria — Five-class suitability surface"
        ),
        "map_type": "categorical",
        "classes": {
            1: {
                "label": "Very Low",
                "colour": "#a50026"
            },
            2: {
                "label": "Low",
                "colour": "#f46d43"
            },
            3: {
                "label": "Moderate",
                "colour": "#fee08b"
            },
            4: {
                "label": "High",
                "colour": "#66bd63"
            },
            5: {
                "label": "Very High",
                "colour": "#006837"
            }
        },
        "method_note": (
            "Suitability derived from constrained urban-expansion probabilities."
        )
    },

    "04_Urban_Development_Planning_Priority": {
        "path": (
            STAGE_11B_DIR
            / "Enugu_Urban_Development_Planning_Priority_30m.tif"
        ),
        "title": (
            "Urban Development Planning Priority"
        ),
        "subtitle": (
            "Enugu State, Nigeria — Suitability adjusted for model confidence"
        ),
        "map_type": "categorical",
        "classes": {
            1: {
                "label": "Very Low Priority",
                "colour": "#d73027"
            },
            2: {
                "label": "Low Priority",
                "colour": "#fc8d59"
            },
            3: {
                "label": "Moderate Priority",
                "colour": "#fee08b"
            },
            4: {
                "label": "High Priority",
                "colour": "#91cf60"
            },
            5: {
                "label": "Very High Priority",
                "colour": "#1a9850"
            }
        },
        "method_note": (
            "Planning priority combines suitability class and tree-based model confidence."
        )
    },

    "05_Model_Confidence": {
        "path": (
            STAGE_11B_DIR
            / "Enugu_Model_Confidence_Classes_30m.tif"
        ),
        "title": (
            "Urban Expansion Model Confidence"
        ),
        "subtitle": (
            "Enugu State, Nigeria — Confidence classification"
        ),
        "map_type": "categorical",
        "classes": {
            1: {
                "label": "Low Confidence",
                "colour": "#d73027"
            },
            2: {
                "label": "Moderate Confidence",
                "colour": "#fee08b"
            },
            3: {
                "label": "High Confidence",
                "colour": "#1a9850"
            }
        },
        "method_note": (
            "Confidence calculated from disagreement among fitted Extra Trees estimators."
        )
    },

    "06_Model_Uncertainty": {
        "path": (
            STAGE_11A_DIR
            / "Enugu_Model_Uncertainty_30m.tif"
        ),
        "title": (
            "Urban Expansion Model Uncertainty"
        ),
        "subtitle": (
            "Enugu State, Nigeria — Tree-probability disagreement"
        ),
        "map_type": "continuous",
        "cmap": "inferno_r",
        "minimum": 0.0,
        "maximum": 0.5,
        "colourbar_label": "Probability standard deviation",
        "method_note": (
            "Higher values indicate greater disagreement among sampled model trees."
        )
    },

    "07_High_Confidence_Suitability": {
        "path": (
            STAGE_11B_DIR
            / "Enugu_High_Confidence_Suitability_30m.tif"
        ),
        "title": (
            "High-Confidence Urban Development Suitability"
        ),
        "subtitle": (
            "Enugu State, Nigeria — Suitability retained where confidence ≥ 0.70"
        ),
        "map_type": "categorical",
        "classes": {
            1: {
                "label": "Very Low",
                "colour": "#a50026"
            },
            2: {
                "label": "Low",
                "colour": "#f46d43"
            },
            3: {
                "label": "Moderate",
                "colour": "#fee08b"
            },
            4: {
                "label": "High",
                "colour": "#66bd63"
            },
            5: {
                "label": "Very High",
                "colour": "#006837"
            }
        },
        "method_note": (
            "Only locations meeting the high-confidence threshold are displayed."
        )
    }
}


# ==========================================================================================
# 4. CARTOGRAPHIC SETTINGS
# ==========================================================================================

EXPORT_DPI = 450

MAXIMUM_DISPLAY_DIMENSION = 2600

PAGE_SIZE_INCHES = (
    11.69,
    8.27
)

MAP_BACKGROUND_COLOUR = "#f3f1eb"

BOUNDARY_COLOUR = "#202020"

GRID_COLOUR = "#777777"

NODATA_COLOUR = "#ececec"

AUTHOR_NAME = "Abdullah Abdazeez Ayomide"

PROJECT_NAME = (
    "Machine-Learning-Based Land Suitability Analysis "
    "for Sustainable Urban Development — Enugu"
)

DATA_NOTE = (
    "Data: Copernicus DEM, Dynamic World, OpenStreetMap, "
    "JRC Surface Water, MERIT Hydro and WorldPop."
)


# ==========================================================================================
# 5. OUTPUT REGISTERS
# ==========================================================================================

INPUT_REGISTER_FILE = (
    TABLE_DIR
    / "Stage_12_Final_Map_Input_Register.csv"
)

OUTPUT_REGISTER_FILE = (
    TABLE_DIR
    / "Stage_12_Final_Map_Output_Register.csv"
)

VALIDATION_REGISTER_FILE = (
    ADMIN_DIR
    / "Stage_12_Final_Cartography_Validation_Register.csv"
)

JSON_REGISTER_FILE = (
    ADMIN_DIR
    / "Stage_12_Final_Cartography_Register.json"
)


# ==========================================================================================
# 6. HELPER FUNCTIONS
# ==========================================================================================

def print_heading(title):
    print(
        "\n"
        +
        title
    )

    print(
        "-"
        *
        120
    )


def validate_file(path, label):
    if not path.exists():
        raise FileNotFoundError(
            f"{label} was not found:\n{path}"
        )

    if path.stat().st_size == 0:
        raise RuntimeError(
            f"{label} exists but is empty:\n{path}"
        )


def format_coordinate(value, position=None):
    return f"{value:,.0f}"


def calculate_nice_distance(target_distance):
    if target_distance <= 0:
        return 1

    exponent = math.floor(
        math.log10(
            target_distance
        )
    )

    fraction = (
        target_distance
        /
        (
            10 ** exponent
        )
    )

    if fraction < 1.5:
        nice_fraction = 1
    elif fraction < 3.5:
        nice_fraction = 2
    elif fraction < 7.5:
        nice_fraction = 5
    else:
        nice_fraction = 10

    return nice_fraction * (
        10 ** exponent
    )


def add_scale_bar(
    axis,
    xmin,
    xmax,
    ymin,
    ymax
):
    map_width = xmax - xmin
    map_height = ymax - ymin

    target_length = (
        map_width
        *
        0.18
    )

    scale_length_m = calculate_nice_distance(
        target_length
    )

    x_start = (
        xmin
        +
        map_width
        *
        0.055
    )

    y_start = (
        ymin
        +
        map_height
        *
        0.055
    )

    segment_count = 4

    segment_length = (
        scale_length_m
        /
        segment_count
    )

    bar_height = (
        map_height
        *
        0.010
    )

    for segment in range(
        segment_count
    ):
        rectangle = Rectangle(
            (
                x_start
                +
                segment
                *
                segment_length,
                y_start
            ),
            segment_length,
            bar_height,
            facecolor=(
                "black"
                if segment % 2 == 0
                else "white"
            ),
            edgecolor="black",
            linewidth=0.7,
            zorder=20
        )

        axis.add_patch(
            rectangle
        )

    axis.plot(
        [
            x_start,
            x_start
            +
            scale_length_m
        ],
        [
            y_start,
            y_start
        ],
        color="black",
        linewidth=0.8,
        zorder=21
    )

    for segment in range(
        segment_count
        +
        1
    ):
        x_position = (
            x_start
            +
            segment
            *
            segment_length
        )

        axis.plot(
            [
                x_position,
                x_position
            ],
            [
                y_start,
                y_start
                +
                bar_height
                *
                1.35
            ],
            color="black",
            linewidth=0.7,
            zorder=21
        )

    scale_length_km = (
        scale_length_m
        /
        1000
    )

    axis.text(
        x_start,
        y_start
        -
        map_height
        *
        0.012,
        "0",
        ha="center",
        va="top",
        fontsize=7.5,
        zorder=21
    )

    axis.text(
        x_start
        +
        scale_length_m
        /
        2,
        y_start
        -
        map_height
        *
        0.012,
        f"{scale_length_km / 2:g}",
        ha="center",
        va="top",
        fontsize=7.5,
        zorder=21
    )

    axis.text(
        x_start
        +
        scale_length_m,
        y_start
        -
        map_height
        *
        0.012,
        f"{scale_length_km:g} km",
        ha="center",
        va="top",
        fontsize=7.5,
        zorder=21
    )


def add_north_arrow(
    axis,
    xmin,
    xmax,
    ymin,
    ymax
):
    map_width = xmax - xmin
    map_height = ymax - ymin

    arrow_x = (
        xmax
        -
        map_width
        *
        0.065
    )

    arrow_bottom = (
        ymax
        -
        map_height
        *
        0.17
    )

    arrow_top = (
        ymax
        -
        map_height
        *
        0.055
    )

    arrow = FancyArrowPatch(
        (
            arrow_x,
            arrow_bottom
        ),
        (
            arrow_x,
            arrow_top
        ),
        arrowstyle="-|>",
        mutation_scale=22,
        linewidth=1.4,
        facecolor="black",
        edgecolor="black",
        zorder=25
    )

    axis.add_patch(
        arrow
    )

    axis.text(
        arrow_x,
        arrow_top
        +
        map_height
        *
        0.015,
        "N",
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold",
        zorder=25
    )


def read_raster_for_display(
    raster_path,
    maximum_dimension=MAXIMUM_DISPLAY_DIMENSION
):
    with rasterio.open(
        raster_path
    ) as source:
        scale_factor = max(
            source.width
            /
            maximum_dimension,
            source.height
            /
            maximum_dimension,
            1
        )

        output_width = max(
            1,
            int(
                source.width
                /
                scale_factor
            )
        )

        output_height = max(
            1,
            int(
                source.height
                /
                scale_factor
            )
        )

        data = source.read(
            1,
            out_shape=(
                output_height,
                output_width
            ),
            masked=True,
            resampling=Resampling.nearest
        )

        extent = plotting_extent(
            source
        )

        metadata = {
            "crs": source.crs,
            "transform": source.transform,
            "width": source.width,
            "height": source.height,
            "bounds": source.bounds,
            "nodata": source.nodata,
            "dtype": source.dtypes[0],
            "description": source.descriptions[0]
        }

    return data, extent, metadata


def find_enugu_boundary(
    project_root,
    target_crs
):
    vector_extensions = {
        ".gpkg",
        ".shp",
        ".geojson"
    }

    vector_files = [
        path
        for path in project_root.rglob(
            "*"
        )
        if (
            path.is_file()
            and
            path.suffix.lower()
            in
            vector_extensions
        )
    ]

    candidate_records = []

    for vector_path in vector_files:
        try:
            layers = [
                None
            ]

            if vector_path.suffix.lower() == ".gpkg":
                try:
                    layers = list(
                        gpd.list_layers(
                            vector_path
                        )["name"]
                    )
                except Exception:
                    layers = [
                        None
                    ]

            for layer_name in layers:
                try:
                    if layer_name is None:
                        vector_data = gpd.read_file(
                            vector_path
                        )
                    else:
                        vector_data = gpd.read_file(
                            vector_path,
                            layer=layer_name
                        )
                except Exception:
                    continue

                if vector_data.empty:
                    continue

                if vector_data.crs is None:
                    continue

                geometry_types = set(
                    vector_data.geometry.geom_type.dropna()
                )

                polygon_types = {
                    "Polygon",
                    "MultiPolygon"
                }

                if not geometry_types.intersection(
                    polygon_types
                ):
                    continue

                text_columns = [
                    column
                    for column in vector_data.columns
                    if (
                        column
                        !=
                        vector_data.geometry.name
                        and
                        vector_data[
                            column
                        ].dtype
                        ==
                        "object"
                    )
                ]

                matching_mask = pd.Series(
                    False,
                    index=vector_data.index
                )

                for column in text_columns:
                    matching_mask |= (
                        vector_data[
                            column
                        ]
                        .astype(
                            str
                        )
                        .str.contains(
                            "enugu",
                            case=False,
                            na=False
                        )
                    )

                if matching_mask.any():
                    selected = vector_data.loc[
                        matching_mask
                    ].copy()

                    selected = selected.to_crs(
                        target_crs
                    )

                    selected = selected[
                        selected.geometry.notna()
                        &
                        ~selected.geometry.is_empty
                    ]

                    if selected.empty:
                        continue

                    dissolved = gpd.GeoDataFrame(
                        {
                            "Name": [
                                "Enugu State"
                            ]
                        },
                        geometry=[
                            selected.geometry.union_all()
                        ],
                        crs=target_crs
                    )

                    candidate_records.append({
                        "path": vector_path,
                        "layer": layer_name,
                        "boundary": dissolved,
                        "source_feature_count": len(
                            vector_data
                        ),
                        "score": (
                            100
                            +
                            (
                                10
                                if 30
                                <=
                                len(
                                    vector_data
                                )
                                <=
                                50
                                else 0
                            )
                        )
                    })

        except Exception:
            continue

    if candidate_records:
        candidate_records.sort(
            key=lambda record: record[
                "score"
            ],
            reverse=True
        )

        selected_record = candidate_records[
            0
        ]

        return (
            selected_record[
                "boundary"
            ],
            selected_record
        )

    return None, None


def find_nigeria_adm1(
    project_root
):
    vector_extensions = {
        ".gpkg",
        ".shp",
        ".geojson"
    }

    vector_files = [
        path
        for path in project_root.rglob(
            "*"
        )
        if (
            path.is_file()
            and
            path.suffix.lower()
            in
            vector_extensions
        )
    ]

    best_record = None

    for vector_path in vector_files:
        try:
            layers = [
                None
            ]

            if vector_path.suffix.lower() == ".gpkg":
                try:
                    layers = list(
                        gpd.list_layers(
                            vector_path
                        )["name"]
                    )
                except Exception:
                    layers = [
                        None
                    ]

            for layer_name in layers:
                try:
                    if layer_name is None:
                        vector_data = gpd.read_file(
                            vector_path
                        )
                    else:
                        vector_data = gpd.read_file(
                            vector_path,
                            layer=layer_name
                        )
                except Exception:
                    continue

                if vector_data.empty:
                    continue

                if vector_data.crs is None:
                    continue

                feature_count = len(
                    vector_data
                )

                if not (
                    30
                    <=
                    feature_count
                    <=
                    50
                ):
                    continue

                geometry_types = set(
                    vector_data.geometry.geom_type.dropna()
                )

                if not geometry_types.intersection(
                    {
                        "Polygon",
                        "MultiPolygon"
                    }
                ):
                    continue

                text_columns = [
                    column
                    for column in vector_data.columns
                    if (
                        column
                        !=
                        vector_data.geometry.name
                        and
                        vector_data[
                            column
                        ].dtype
                        ==
                        "object"
                    )
                ]

                contains_enugu = False
                name_column = None

                for column in text_columns:
                    column_contains_enugu = (
                        vector_data[
                            column
                        ]
                        .astype(
                            str
                        )
                        .str.fullmatch(
                            "Enugu",
                            case=False,
                            na=False
                        )
                        .any()
                    )

                    if column_contains_enugu:
                        contains_enugu = True
                        name_column = column
                        break

                if not contains_enugu:
                    continue

                score = (
                    100
                    -
                    abs(
                        feature_count
                        -
                        37
                    )
                )

                record = {
                    "path": vector_path,
                    "layer": layer_name,
                    "data": vector_data.to_crs(
                        "EPSG:4326"
                    ),
                    "name_column": name_column,
                    "score": score
                }

                if (
                    best_record is None
                    or
                    record[
                        "score"
                    ]
                    >
                    best_record[
                        "score"
                    ]
                ):
                    best_record = record

        except Exception:
            continue

    return best_record


def add_nigeria_inset(
    figure,
    main_axis,
    nigeria_record
):
    if nigeria_record is None:
        return False

    nigeria_data = nigeria_record[
        "data"
    ]

    name_column = nigeria_record[
        "name_column"
    ]

    enugu_mask = (
        nigeria_data[
            name_column
        ]
        .astype(
            str
        )
        .str.fullmatch(
            "Enugu",
            case=False,
            na=False
        )
    )

    inset_axis = inset_axes(
        main_axis,
        width="22%",
        height="25%",
        loc="lower right",
        borderpad=1.2
    )

    inset_axis.set_facecolor(
        "white"
    )

    nigeria_data.plot(
        ax=inset_axis,
        facecolor="#efefef",
        edgecolor="#666666",
        linewidth=0.35
    )

    nigeria_data.loc[
        enugu_mask
    ].plot(
        ax=inset_axis,
        facecolor="#d73027",
        edgecolor="#111111",
        linewidth=0.8
    )

    inset_axis.set_title(
        "Location in Nigeria",
        fontsize=7.5,
        fontweight="bold",
        pad=2
    )

    inset_axis.set_xticks(
        []
    )

    inset_axis.set_yticks(
        []
    )

    for spine in inset_axis.spines.values():
        spine.set_visible(
            True
        )

        spine.set_linewidth(
            0.7
        )

        spine.set_edgecolor(
            "#333333"
        )

    return True


def add_categorical_legend(
    axis,
    class_definitions
):
    legend_handles = []

    for class_code, class_record in class_definitions.items():
        legend_handles.append(
            Line2D(
                [
                    0
                ],
                [
                    0
                ],
                marker="s",
                linestyle="None",
                markerfacecolor=class_record[
                    "colour"
                ],
                markeredgecolor="#333333",
                markeredgewidth=0.35,
                markersize=9,
                label=class_record[
                    "label"
                ]
            )
        )

    legend = axis.legend(
        handles=legend_handles,
        title="Legend",
        loc="upper left",
        bbox_to_anchor=(
            0.018,
            0.985
        ),
        frameon=True,
        facecolor="white",
        edgecolor="#444444",
        framealpha=0.94,
        fontsize=8,
        title_fontsize=9,
        borderpad=0.7,
        labelspacing=0.55
    )

    legend.get_frame().set_linewidth(
        0.65
    )


def create_categorical_colormap(
    class_definitions
):
    class_codes = sorted(
        class_definitions.keys()
    )

    colours = [
        class_definitions[
            class_code
        ][
            "colour"
        ]
        for class_code in class_codes
    ]

    colormap = ListedColormap(
        colours
    )

    boundaries = [
        class_codes[
            0
        ]
        -
        0.5
    ]

    for class_code in class_codes:
        boundaries.append(
            class_code
            +
            0.5
        )

    norm = BoundaryNorm(
        boundaries,
        colormap.N
    )

    colormap.set_bad(
        NODATA_COLOUR,
        alpha=0.0
    )

    return colormap, norm


# ==========================================================================================
# 7. VALIDATE INPUT RASTERS
# ==========================================================================================

print(
    "="
    *
    120
)

print(
    "STAGE 12A–12B — PROFESSIONAL CARTOGRAPHIC ENGINE AND FINAL MAP PRODUCTION"
)

print(
    "="
    *
    120
)

print_heading(
    "FINAL MAP INPUT CHECK"
)

input_records = []

reference_crs = None
reference_width = None
reference_height = None
reference_transform = None
reference_bounds = None

for map_number, map_configuration in MAP_INPUTS.items():
    raster_path = map_configuration[
        "path"
    ]

    validate_file(
        raster_path,
        map_number
    )

    with rasterio.open(
        raster_path
    ) as source:
        input_record = {
            "Map_ID": map_number,
            "Raster": str(
                raster_path
            ),
            "CRS": str(
                source.crs
            ),
            "Width": int(
                source.width
            ),
            "Height": int(
                source.height
            ),
            "Data_Type": source.dtypes[
                0
            ],
            "NoData": source.nodata,
            "Band_Description": source.descriptions[
                0
            ]
        }

        input_records.append(
            input_record
        )

        if reference_crs is None:
            reference_crs = source.crs
            reference_width = source.width
            reference_height = source.height
            reference_transform = source.transform
            reference_bounds = source.bounds
        else:
            if source.crs != reference_crs:
                raise RuntimeError(
                    f"CRS mismatch detected for {map_number}."
                )

            if (
                source.width
                !=
                reference_width
                or
                source.height
                !=
                reference_height
            ):
                raise RuntimeError(
                    f"Raster dimension mismatch detected for {map_number}."
                )

            if not np.allclose(
                tuple(
                    source.transform
                ),
                tuple(
                    reference_transform
                )
            ):
                raise RuntimeError(
                    f"Raster transform mismatch detected for {map_number}."
                )

        print(
            f"✓ {map_number}"
        )

        print(
            f"    Raster: {raster_path}"
        )

        print(
            f"    CRS: {source.crs} | "
            f"Dimensions: {source.width:,} × {source.height:,}"
        )

input_df = pd.DataFrame(
    input_records
)

input_df.to_csv(
    INPUT_REGISTER_FILE,
    index=False
)

print(
    f"\n✓ Input register: {INPUT_REGISTER_FILE}"
)


# ==========================================================================================
# 8. RESOLVE ENUGU AND NIGERIA BOUNDARIES
# ==========================================================================================

print_heading(
    "ADMINISTRATIVE BOUNDARY RESOLUTION"
)

enugu_boundary, enugu_boundary_record = find_enugu_boundary(
    PROJECT_ROOT,
    reference_crs
)

if enugu_boundary is not None:
    print(
        "✓ Enugu boundary resolved."
    )

    print(
        f"    Source: "
        f"{enugu_boundary_record['path']}"
    )

    print(
        f"    Layer: "
        f"{enugu_boundary_record['layer']}"
    )
else:
    print(
        "⚠ Enugu vector boundary was not resolved."
    )

    print(
        "  Raster valid-data extent will be used as the cartographic boundary."
    )

nigeria_adm1_record = find_nigeria_adm1(
    PROJECT_ROOT
)

if nigeria_adm1_record is not None:
    print(
        "✓ Nigeria state boundary dataset resolved."
    )

    print(
        f"    Source: "
        f"{nigeria_adm1_record['path']}"
    )

    print(
        f"    Enugu name field: "
        f"{nigeria_adm1_record['name_column']}"
    )
else:
    print(
        "⚠ Nigeria state boundary dataset was not resolved."
    )

    print(
        "  Final maps will be generated without the national inset."
    )


# ==========================================================================================
# 9. DETERMINE FINAL MAP EXTENT
# ==========================================================================================

if enugu_boundary is not None:
    boundary_bounds = enugu_boundary.total_bounds

    xmin = float(
        boundary_bounds[
            0
        ]
    )

    ymin = float(
        boundary_bounds[
            1
        ]
    )

    xmax = float(
        boundary_bounds[
            2
        ]
    )

    ymax = float(
        boundary_bounds[
            3
        ]
    )
else:
    xmin = float(
        reference_bounds.left
    )

    ymin = float(
        reference_bounds.bottom
    )

    xmax = float(
        reference_bounds.right
    )

    ymax = float(
        reference_bounds.top
    )

map_width = xmax - xmin
map_height = ymax - ymin

horizontal_padding = (
    map_width
    *
    0.025
)

vertical_padding = (
    map_height
    *
    0.025
)

final_extent = (
    xmin
    -
    horizontal_padding,
    xmax
    +
    horizontal_padding,
    ymin
    -
    vertical_padding,
    ymax
    +
    vertical_padding
)

print_heading(
    "FINAL MAP EXTENT"
)

print(
    f"X extent: "
    f"{final_extent[0]:,.2f} to "
    f"{final_extent[1]:,.2f}"
)

print(
    f"Y extent: "
    f"{final_extent[2]:,.2f} to "
    f"{final_extent[3]:,.2f}"
)

print(
    f"Map CRS: {reference_crs}"
)


# ==========================================================================================
# 10. GENERATE FINAL MAPS
# ==========================================================================================

print_heading(
    "GENERATING PUBLICATION-QUALITY MAPS"
)

output_records = []

for map_index, (
    map_identifier,
    map_configuration
) in enumerate(
    MAP_INPUTS.items(),
    start=1
):
    raster_path = map_configuration[
        "path"
    ]

    print(
        f"\n[{map_index}/{len(MAP_INPUTS)}] "
        f"Generating {map_identifier}"
    )

    raster_data, raster_extent, raster_metadata = read_raster_for_display(
        raster_path
    )

    figure = plt.figure(
        figsize=PAGE_SIZE_INCHES,
        facecolor="white"
    )

    axis = figure.add_axes(
        [
            0.065,
            0.115,
            0.78,
            0.77
        ]
    )

    axis.set_facecolor(
        MAP_BACKGROUND_COLOUR
    )

    if map_configuration[
        "map_type"
    ] == "continuous":
        colourmap = plt.get_cmap(
            map_configuration[
                "cmap"
            ]
        ).copy()

        colourmap.set_bad(
            NODATA_COLOUR,
            alpha=0.0
        )

        normalisation = Normalize(
            vmin=map_configuration[
                "minimum"
            ],
            vmax=map_configuration[
                "maximum"
            ]
        )

        image = axis.imshow(
            raster_data,
            extent=raster_extent,
            origin="upper",
            cmap=colourmap,
            norm=normalisation,
            interpolation="nearest",
            zorder=2
        )

        colourbar_axis = figure.add_axes(
            [
                0.875,
                0.275,
                0.022,
                0.43
            ]
        )

        colourbar = figure.colorbar(
            image,
            cax=colourbar_axis
        )

        colourbar.set_label(
            map_configuration[
                "colourbar_label"
            ],
            fontsize=9,
            labelpad=9
        )

        colourbar.ax.tick_params(
            labelsize=8
        )

        colourbar.outline.set_linewidth(
            0.65
        )

    else:
        colourmap, normalisation = create_categorical_colormap(
            map_configuration[
                "classes"
            ]
        )

        image = axis.imshow(
            raster_data,
            extent=raster_extent,
            origin="upper",
            cmap=colourmap,
            norm=normalisation,
            interpolation="nearest",
            zorder=2
        )

        add_categorical_legend(
            axis,
            map_configuration[
                "classes"
            ]
        )

    if enugu_boundary is not None:
        enugu_boundary.boundary.plot(
            ax=axis,
            color=BOUNDARY_COLOUR,
            linewidth=0.85,
            zorder=12
        )

    axis.set_xlim(
        final_extent[
            0
        ],
        final_extent[
            1
        ]
    )

    axis.set_ylim(
        final_extent[
            2
        ],
        final_extent[
            3
        ]
    )

    axis.set_aspect(
        "equal"
    )

    axis.set_xlabel(
        "Easting (metres)"
    )

    axis.set_ylabel(
        "Northing (metres)"
    )

    axis.xaxis.set_major_formatter(
        matplotlib.ticker.FuncFormatter(
            format_coordinate
        )
    )

    axis.yaxis.set_major_formatter(
        matplotlib.ticker.FuncFormatter(
            format_coordinate
        )
    )

    axis.grid(
        True,
        color=GRID_COLOUR,
        linewidth=0.35,
        linestyle="--",
        alpha=0.45,
        zorder=1
    )

    axis.tick_params(
        direction="out",
        length=3.5,
        width=0.6,
        pad=4
    )

    for spine in axis.spines.values():
        spine.set_linewidth(
            1.0
        )

        spine.set_edgecolor(
            "#222222"
        )

    add_scale_bar(
        axis,
        final_extent[
            0
        ],
        final_extent[
            1
        ],
        final_extent[
            2
        ],
        final_extent[
            3
        ]
    )

    add_north_arrow(
        axis,
        final_extent[
            0
        ],
        final_extent[
            1
        ],
        final_extent[
            2
        ],
        final_extent[
            3
        ]
    )

    inset_created = add_nigeria_inset(
        figure,
        axis,
        nigeria_adm1_record
    )

    figure.suptitle(
        map_configuration[
            "title"
        ],
        fontsize=17,
        fontweight="bold",
        y=0.965
    )

    figure.text(
        0.455,
        0.918,
        map_configuration[
            "subtitle"
        ],
        ha="center",
        va="center",
        fontsize=10.5,
        color="#444444"
    )

    figure.text(
        0.065,
        0.067,
        map_configuration[
            "method_note"
        ],
        ha="left",
        va="center",
        fontsize=8.2,
        color="#333333"
    )

    figure.text(
        0.065,
        0.042,
        DATA_NOTE,
        ha="left",
        va="center",
        fontsize=7.4,
        color="#555555"
    )

    figure.text(
        0.935,
        0.042,
        f"Prepared by {AUTHOR_NAME}",
        ha="right",
        va="center",
        fontsize=7.4,
        color="#555555"
    )

    figure.text(
        0.935,
        0.067,
        "Projection: WGS 84 / UTM Zone 32N",
        ha="right",
        va="center",
        fontsize=7.4,
        color="#555555"
    )

    neatline = Rectangle(
        (
            0.018,
            0.022
        ),
        0.964,
        0.955,
        fill=False,
        transform=figure.transFigure,
        figure=figure,
        linewidth=1.1,
        edgecolor="#222222"
    )

    figure.patches.append(
        neatline
    )

    png_output = (
        PNG_DIR
        /
        f"{map_identifier}.png"
    )

    pdf_output = (
        PDF_DIR
        /
        f"{map_identifier}.pdf"
    )

    figure.savefig(
        png_output,
        dpi=EXPORT_DPI,
        bbox_inches="tight",
        facecolor="white"
    )

    figure.savefig(
        pdf_output,
        dpi=EXPORT_DPI,
        bbox_inches="tight",
        facecolor="white"
    )

    plt.close(
        figure
    )

    validate_file(
        png_output,
        f"{map_identifier} PNG"
    )

    validate_file(
        pdf_output,
        f"{map_identifier} PDF"
    )

    output_records.append({
        "Map_ID": map_identifier,
        "Title": map_configuration[
            "title"
        ],
        "Input_Raster": str(
            raster_path
        ),
        "PNG_Output": str(
            png_output
        ),
        "PDF_Output": str(
            pdf_output
        ),
        "PNG_Size_MB": (
            png_output.stat().st_size
            /
            (
                1024 ** 2
            )
        ),
        "PDF_Size_MB": (
            pdf_output.stat().st_size
            /
            (
                1024 ** 2
            )
        ),
        "Export_DPI": EXPORT_DPI,
        "CRS": str(
            raster_metadata[
                "crs"
            ]
        ),
        "Boundary_Overlay": (
            enugu_boundary is not None
        ),
        "Nigeria_Inset": inset_created,
        "Status": "Completed"
    })

    print(
        f"✓ PNG: {png_output}"
    )

    print(
        f"✓ PDF: {pdf_output}"
    )

    del raster_data
    del figure
    del axis

    gc.collect()


# ==========================================================================================
# 11. SAVE OUTPUT REGISTER
# ==========================================================================================

output_df = pd.DataFrame(
    output_records
)

output_df.to_csv(
    OUTPUT_REGISTER_FILE,
    index=False
)


# ==========================================================================================
# 12. VALIDATION
# ==========================================================================================

print_heading(
    "FINAL CARTOGRAPHIC VALIDATION"
)

validation_records = []


def add_validation(
    check,
    passed,
    observed,
    expected
):
    validation_records.append({
        "Check": check,
        "Passed": bool(
            passed
        ),
        "Observed": str(
            observed
        ),
        "Expected": str(
            expected
        )
    })


expected_map_count = len(
    MAP_INPUTS
)

png_files = sorted(
    PNG_DIR.glob(
        "*.png"
    )
)

pdf_files = sorted(
    PDF_DIR.glob(
        "*.pdf"
    )
)

add_validation(
    "All map inputs exist",
    all(
        configuration[
            "path"
        ].exists()
        for configuration in MAP_INPUTS.values()
    ),
    sum(
        configuration[
            "path"
        ].exists()
        for configuration in MAP_INPUTS.values()
    ),
    expected_map_count
)

add_validation(
    "All raster grids match",
    True,
    (
        reference_width,
        reference_height,
        str(
            reference_crs
        )
    ),
    "Identical dimensions, transform and CRS"
)

add_validation(
    "Expected PNG map count generated",
    len(
        png_files
    )
    ==
    expected_map_count,
    len(
        png_files
    ),
    expected_map_count
)

add_validation(
    "Expected PDF map count generated",
    len(
        pdf_files
    )
    ==
    expected_map_count,
    len(
        pdf_files
    ),
    expected_map_count
)

add_validation(
    "All PNG maps are non-empty",
    all(
        path.stat().st_size
        >
        0
        for path in png_files
    ),
    [
        round(
            path.stat().st_size
            /
            (
                1024 ** 2
            ),
            2
        )
        for path in png_files
    ],
    "All file sizes greater than zero"
)

add_validation(
    "All PDF maps are non-empty",
    all(
        path.stat().st_size
        >
        0
        for path in pdf_files
    ),
    [
        round(
            path.stat().st_size
            /
            (
                1024 ** 2
            ),
            2
        )
        for path in pdf_files
    ],
    "All file sizes greater than zero"
)

add_validation(
    "Seven final map products registered",
    len(
        output_df
    )
    ==
    expected_map_count,
    len(
        output_df
    ),
    expected_map_count
)

add_validation(
    "All outputs use WGS 84 / UTM Zone 32N",
    all(
        value
        ==
        "EPSG:32632"
        for value in output_df[
            "CRS"
        ]
    ),
    sorted(
        output_df[
            "CRS"
        ].unique()
    ),
    [
        "EPSG:32632"
    ]
)

add_validation(
    "Publication export resolution recorded",
    all(
        output_df[
            "Export_DPI"
        ]
        ==
        EXPORT_DPI
    ),
    sorted(
        output_df[
            "Export_DPI"
        ].unique()
    ),
    [
        EXPORT_DPI
    ]
)

validation_df = pd.DataFrame(
    validation_records
)

validation_df.to_csv(
    VALIDATION_REGISTER_FILE,
    index=False
)

failed_validation = validation_df.loc[
    ~validation_df[
        "Passed"
    ]
]

if not failed_validation.empty:
    print(
        failed_validation.to_string(
            index=False
        )
    )

    raise RuntimeError(
        "Stage 12 map production completed, but validation failed."
    )


# ==========================================================================================
# 13. JSON CARTOGRAPHIC REGISTER
# ==========================================================================================

json_register = {
    "stage": "12A-12B",
    "stage_name": (
        "Professional Cartographic Engine and Final Map Production"
    ),
    "completion_time_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "project": PROJECT_NAME,
    "study_area": "Enugu State, Nigeria",
    "author": AUTHOR_NAME,
    "projection": str(
        reference_crs
    ),
    "reference_width": int(
        reference_width
    ),
    "reference_height": int(
        reference_height
    ),
    "export_dpi": int(
        EXPORT_DPI
    ),
    "page_size_inches": [
        float(
            PAGE_SIZE_INCHES[
                0
            ]
        ),
        float(
            PAGE_SIZE_INCHES[
                1
            ]
        )
    ],
    "enugu_boundary_resolved": (
        enugu_boundary is not None
    ),
    "enugu_boundary_source": (
        str(
            enugu_boundary_record[
                "path"
            ]
        )
        if enugu_boundary_record is not None
        else None
    ),
    "nigeria_inset_resolved": (
        nigeria_adm1_record is not None
    ),
    "nigeria_boundary_source": (
        str(
            nigeria_adm1_record[
                "path"
            ]
        )
        if nigeria_adm1_record is not None
        else None
    ),
    "map_count": int(
        len(
            output_df
        )
    ),
    "png_directory": str(
        PNG_DIR
    ),
    "pdf_directory": str(
        PDF_DIR
    ),
    "input_register": str(
        INPUT_REGISTER_FILE
    ),
    "output_register": str(
        OUTPUT_REGISTER_FILE
    ),
    "validation_register": str(
        VALIDATION_REGISTER_FILE
    ),
    "validation_passed": True,
    "maps": output_records
}

with open(
    JSON_REGISTER_FILE,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        json_register,
        file,
        indent=4
    )


# ==========================================================================================
# 14. FINAL RESULTS
# ==========================================================================================

print_heading(
    "FINAL MAP OUTPUT REGISTER"
)

print(
    output_df[
        [
            "Map_ID",
            "PNG_Size_MB",
            "PDF_Size_MB",
            "Boundary_Overlay",
            "Nigeria_Inset",
            "Status"
        ]
    ].to_string(
        index=False,
        formatters={
            "PNG_Size_MB":
                lambda value: f"{value:.2f}",

            "PDF_Size_MB":
                lambda value: f"{value:.2f}"
        }
    )
)

print_heading(
    "VALIDATION RESULTS"
)

print(
    validation_df.to_string(
        index=False
    )
)

print_heading(
    "FINAL OUTPUT LOCATIONS"
)

print(
    f"✓ PNG maps: {PNG_DIR}"
)

print(
    f"✓ PDF maps: {PDF_DIR}"
)

print(
    f"✓ Input register: {INPUT_REGISTER_FILE}"
)

print(
    f"✓ Output register: {OUTPUT_REGISTER_FILE}"
)

print(
    f"✓ Validation register: {VALIDATION_REGISTER_FILE}"
)

print(
    f"✓ Cartographic JSON register: {JSON_REGISTER_FILE}"
)

print(
    "="
    *
    120
)

print(
    "STAGE 12A–12B COMPLETED SUCCESSFULLY"
)

print(
    "="
    *
    120
)

print(
    "✓ The professional cartographic engine was created."
)

print(
    "✓ Seven final publication-quality maps were generated."
)

print(
    "✓ Every map was exported in PNG and PDF formats."
)

print(
    "✓ Coordinate grids, scale bars and north arrows were included."
)

print(
    "✓ Enugu administrative boundaries were added where available."
)

print(
    "✓ A Nigeria location inset was added where the ADM1 dataset was available."
)

print(
    "✓ All final cartographic validation checks passed."
)

print(
    "="
    *
    120
)


In [ ]:
# ==========================================================================================
# PROJECT 7 — STAGE 12C
# MULTI-PANEL FIGURES, PORTFOLIO BOARD AND FINAL MAP ATLAS
#
# Outputs:
#   1. Four-panel journal results figure
#   2. Six-panel model and planning figure
#   3. Portfolio presentation board
#   4. Combined final map atlas PDF
#   5. Figure and atlas registers
#   6. Validation register
# ==========================================================================================

from pathlib import Path
from datetime import datetime, timezone
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.colors import BoundaryNorm, ListedColormap, Normalize
from matplotlib.lines import Line2D
from PIL import Image, ImageOps

import json
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio

from rasterio.enums import Resampling
from rasterio.plot import plotting_extent

warnings.filterwarnings("ignore")


# ==========================================================================================
# 1. PROJECT PATHS
# ==========================================================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

STAGE_11A_DIR = (
    PROJECT_ROOT
    / "12_Statewide_Suitability"
)

STAGE_11B_DIR = (
    PROJECT_ROOT
    / "13_Planning_Priority"
)

FINAL_MAP_DIR = (
    PROJECT_ROOT
    / "14_Final_Cartographic_Maps"
)

FINAL_MAP_PNG_DIR = (
    FINAL_MAP_DIR
    / "PNG"
)

FINAL_MAP_PDF_DIR = (
    FINAL_MAP_DIR
    / "PDF"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "15_Publication_Portfolio"
)

FIGURE_DIR = (
    OUTPUT_ROOT
    / "Figures"
)

PORTFOLIO_DIR = (
    OUTPUT_ROOT
    / "Portfolio"
)

ATLAS_DIR = (
    OUTPUT_ROOT
    / "Atlas"
)

TABLE_DIR = (
    OUTPUT_ROOT
    / "Tables"
)

ADMIN_DIR = (
    PROJECT_ROOT
    / "00_Project_Admin"
)

for directory in [
    OUTPUT_ROOT,
    FIGURE_DIR,
    PORTFOLIO_DIR,
    ATLAS_DIR,
    TABLE_DIR,
    ADMIN_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ==========================================================================================
# 2. REQUIRED RASTERS
# ==========================================================================================

RASTERS = {
    "Urban Expansion Probability": (
        STAGE_11A_DIR
        / "Enugu_Urban_Expansion_Probability_2025_30m.tif"
    ),

    "Constrained Probability": (
        STAGE_11A_DIR
        / "Enugu_Planning_Constrained_Expansion_Probability_30m.tif"
    ),

    "Suitability": (
        STAGE_11A_DIR
        / "Enugu_Urban_Development_Suitability_Classes_30m.tif"
    ),

    "Planning Priority": (
        STAGE_11B_DIR
        / "Enugu_Urban_Development_Planning_Priority_30m.tif"
    ),

    "Confidence": (
        STAGE_11B_DIR
        / "Enugu_Model_Confidence_Classes_30m.tif"
    ),

    "Uncertainty": (
        STAGE_11A_DIR
        / "Enugu_Model_Uncertainty_30m.tif"
    )
}


# ==========================================================================================
# 3. REQUIRED TABLES
# ==========================================================================================

SUITABILITY_TABLE = (
    STAGE_11A_DIR
    / "Tables"
    / "Enugu_Suitability_Class_Area_Statistics.csv"
)

PRIORITY_TABLE = (
    STAGE_11B_DIR
    / "Tables"
    / "Enugu_Planning_Priority_Area_Statistics.csv"
)

CONFIDENCE_TABLE = (
    STAGE_11B_DIR
    / "Tables"
    / "Enugu_Confidence_Class_Area_Statistics.csv"
)

RELIABILITY_TABLE = (
    STAGE_11B_DIR
    / "Tables"
    / "Enugu_Suitability_Class_Reliability_Statistics.csv"
)


# ==========================================================================================
# 4. OUTPUT FILES
# ==========================================================================================

FOUR_PANEL_PNG = (
    FIGURE_DIR
    / "Figure_01_Expansion_Suitability_and_Priority.png"
)

FOUR_PANEL_PDF = (
    FIGURE_DIR
    / "Figure_01_Expansion_Suitability_and_Priority.pdf"
)

SIX_PANEL_PNG = (
    FIGURE_DIR
    / "Figure_02_Model_Reliability_and_Planning_Outputs.png"
)

SIX_PANEL_PDF = (
    FIGURE_DIR
    / "Figure_02_Model_Reliability_and_Planning_Outputs.pdf"
)

PORTFOLIO_BOARD_PNG = (
    PORTFOLIO_DIR
    / "Enugu_ML_Land_Suitability_Portfolio_Board.png"
)

PORTFOLIO_BOARD_PDF = (
    PORTFOLIO_DIR
    / "Enugu_ML_Land_Suitability_Portfolio_Board.pdf"
)

ATLAS_PDF = (
    ATLAS_DIR
    / "Enugu_ML_Land_Suitability_Final_Map_Atlas.pdf"
)

OUTPUT_REGISTER_FILE = (
    TABLE_DIR
    / "Stage_12C_Output_Register.csv"
)

VALIDATION_REGISTER_FILE = (
    ADMIN_DIR
    / "Stage_12C_Validation_Register.csv"
)

JSON_REGISTER_FILE = (
    ADMIN_DIR
    / "Stage_12C_Publication_Portfolio_Register.json"
)


# ==========================================================================================
# 5. SETTINGS
# ==========================================================================================

EXPORT_DPI = 400

MAX_DISPLAY_DIMENSION = 1600

NODATA_COLOUR = "#f2f2f2"

SUITABILITY_COLOURS = [
    "#a50026",
    "#f46d43",
    "#fee08b",
    "#66bd63",
    "#006837"
]

PRIORITY_COLOURS = [
    "#d73027",
    "#fc8d59",
    "#fee08b",
    "#91cf60",
    "#1a9850"
]

CONFIDENCE_COLOURS = [
    "#d73027",
    "#fee08b",
    "#1a9850"
]

SUITABILITY_LABELS = [
    "Very Low",
    "Low",
    "Moderate",
    "High",
    "Very High"
]

PRIORITY_LABELS = [
    "Very Low",
    "Low",
    "Moderate",
    "High",
    "Very High"
]

CONFIDENCE_LABELS = [
    "Low",
    "Moderate",
    "High"
]

AUTHOR_NAME = "Abdullah Abdazeez Ayomide"

PROJECT_TITLE = (
    "Machine-Learning-Based Land Suitability Analysis "
    "for Sustainable Urban Development — Enugu State"
)


# ==========================================================================================
# 6. HELPER FUNCTIONS
# ==========================================================================================

def print_heading(title):
    print("\n" + title)
    print("-" * 120)


def validate_file(path, label):
    if not path.exists():
        raise FileNotFoundError(
            f"{label} was not found:\n{path}"
        )

    if path.stat().st_size == 0:
        raise RuntimeError(
            f"{label} exists but is empty:\n{path}"
        )


def read_raster(path, maximum_dimension=MAX_DISPLAY_DIMENSION):
    with rasterio.open(path) as source:
        scale = max(
            source.width / maximum_dimension,
            source.height / maximum_dimension,
            1
        )

        output_width = max(
            1,
            int(source.width / scale)
        )

        output_height = max(
            1,
            int(source.height / scale)
        )

        data = source.read(
            1,
            out_shape=(
                output_height,
                output_width
            ),
            masked=True,
            resampling=Resampling.nearest
        )

        extent = plotting_extent(source)

        metadata = {
            "crs": str(source.crs),
            "width": source.width,
            "height": source.height,
            "bounds": tuple(source.bounds)
        }

    return data, extent, metadata


def categorical_style(colours, class_count):
    colormap = ListedColormap(colours)

    boundaries = np.arange(
        0.5,
        class_count + 1.5,
        1
    )

    norm = BoundaryNorm(
        boundaries,
        colormap.N
    )

    colormap.set_bad(
        NODATA_COLOUR,
        alpha=0
    )

    return colormap, norm


def add_category_legend(
    axis,
    colours,
    labels,
    title
):
    handles = [
        Line2D(
            [0],
            [0],
            marker="s",
            linestyle="None",
            markerfacecolor=colour,
            markeredgecolor="#333333",
            markeredgewidth=0.4,
            markersize=7,
            label=label
        )
        for colour, label in zip(
            colours,
            labels
        )
    ]

    axis.legend(
        handles=handles,
        title=title,
        loc="lower left",
        fontsize=6.7,
        title_fontsize=7.2,
        frameon=True,
        facecolor="white",
        edgecolor="#555555",
        framealpha=0.93
    )


def style_map_axis(axis, title):
    axis.set_title(
        title,
        fontsize=10,
        fontweight="bold",
        pad=7
    )

    axis.set_xticks([])
    axis.set_yticks([])

    for spine in axis.spines.values():
        spine.set_linewidth(0.8)
        spine.set_edgecolor("#333333")


def plot_continuous(
    axis,
    path,
    title,
    cmap,
    vmin,
    vmax,
    colourbar_label
):
    data, extent, metadata = read_raster(path)

    selected_cmap = plt.get_cmap(
        cmap
    ).copy()

    selected_cmap.set_bad(
        NODATA_COLOUR,
        alpha=0
    )

    image = axis.imshow(
        data,
        extent=extent,
        origin="upper",
        cmap=selected_cmap,
        norm=Normalize(
            vmin=vmin,
            vmax=vmax
        ),
        interpolation="nearest"
    )

    style_map_axis(
        axis,
        title
    )

    colourbar = plt.colorbar(
        image,
        ax=axis,
        fraction=0.035,
        pad=0.02
    )

    colourbar.ax.tick_params(
        labelsize=6
    )

    colourbar.set_label(
        colourbar_label,
        fontsize=6.5
    )

    return metadata


def plot_categorical(
    axis,
    path,
    title,
    colours,
    labels,
    legend_title
):
    data, extent, metadata = read_raster(path)

    colormap, norm = categorical_style(
        colours,
        len(colours)
    )

    axis.imshow(
        data,
        extent=extent,
        origin="upper",
        cmap=colormap,
        norm=norm,
        interpolation="nearest"
    )

    style_map_axis(
        axis,
        title
    )

    add_category_legend(
        axis,
        colours,
        labels,
        legend_title
    )

    return metadata


def load_final_png(path):
    validate_file(
        path,
        path.name
    )

    image = Image.open(
        path
    ).convert(
        "RGB"
    )

    return image


# ==========================================================================================
# 7. INPUT VALIDATION
# ==========================================================================================

print("=" * 120)
print("STAGE 12C — MULTI-PANEL FIGURES, PORTFOLIO BOARD AND FINAL MAP ATLAS")
print("=" * 120)

print_heading(
    "REQUIRED INPUT CHECK"
)

for label, path in RASTERS.items():
    validate_file(
        path,
        label
    )

    print(
        f"✓ {label}: {path}"
    )

for table_path in [
    SUITABILITY_TABLE,
    PRIORITY_TABLE,
    CONFIDENCE_TABLE,
    RELIABILITY_TABLE
]:
    validate_file(
        table_path,
        table_path.name
    )

    print(
        f"✓ Table: {table_path}"
    )

final_png_maps = sorted(
    FINAL_MAP_PNG_DIR.glob(
        "*.png"
    )
)

final_pdf_maps = sorted(
    FINAL_MAP_PDF_DIR.glob(
        "*.pdf"
    )
)

if len(final_png_maps) != 7:
    raise RuntimeError(
        f"Expected 7 Stage 12 PNG maps; found {len(final_png_maps)}."
    )

if len(final_pdf_maps) != 7:
    raise RuntimeError(
        f"Expected 7 Stage 12 PDF maps; found {len(final_pdf_maps)}."
    )

print(
    f"✓ Final map PNG count: {len(final_png_maps)}"
)

print(
    f"✓ Final map PDF count: {len(final_pdf_maps)}"
)


# ==========================================================================================
# 8. LOAD STATISTICAL TABLES
# ==========================================================================================

suitability_df = pd.read_csv(
    SUITABILITY_TABLE
)

priority_df = pd.read_csv(
    PRIORITY_TABLE
)

confidence_df = pd.read_csv(
    CONFIDENCE_TABLE
)

reliability_df = pd.read_csv(
    RELIABILITY_TABLE
)


# ==========================================================================================
# 9. FOUR-PANEL PUBLICATION FIGURE
# ==========================================================================================

print_heading(
    "GENERATING FOUR-PANEL PUBLICATION FIGURE"
)

figure, axes = plt.subplots(
    2,
    2,
    figsize=(
        12,
        12
    )
)

plot_continuous(
    axes[0, 0],
    RASTERS["Urban Expansion Probability"],
    "(a) Urban expansion probability",
    "magma",
    0,
    1,
    "Probability"
)

plot_continuous(
    axes[0, 1],
    RASTERS["Constrained Probability"],
    "(b) Planning-constrained probability",
    "viridis",
    0,
    1,
    "Probability"
)

plot_categorical(
    axes[1, 0],
    RASTERS["Suitability"],
    "(c) Urban development suitability",
    SUITABILITY_COLOURS,
    SUITABILITY_LABELS,
    "Suitability"
)

plot_categorical(
    axes[1, 1],
    RASTERS["Planning Priority"],
    "(d) Confidence-adjusted planning priority",
    PRIORITY_COLOURS,
    PRIORITY_LABELS,
    "Priority"
)

figure.suptitle(
    "Machine-Learning Urban Expansion and Land Suitability Assessment\n"
    "Enugu State, Nigeria",
    fontsize=16,
    fontweight="bold",
    y=0.975
)

figure.text(
    0.5,
    0.025,
    "The planning outputs combine leakage-corrected urban-expansion probabilities, "
    "environmental constraints and model confidence.",
    ha="center",
    fontsize=9
)

figure.tight_layout(
    rect=[
        0,
        0.045,
        1,
        0.95
    ]
)

figure.savefig(
    FOUR_PANEL_PNG,
    dpi=EXPORT_DPI,
    bbox_inches="tight",
    facecolor="white"
)

figure.savefig(
    FOUR_PANEL_PDF,
    dpi=EXPORT_DPI,
    bbox_inches="tight",
    facecolor="white"
)

plt.close(figure)

print(
    f"✓ {FOUR_PANEL_PNG}"
)

print(
    f"✓ {FOUR_PANEL_PDF}"
)


# ==========================================================================================
# 10. SIX-PANEL MODEL AND PLANNING FIGURE
# ==========================================================================================

print_heading(
    "GENERATING SIX-PANEL RELIABILITY FIGURE"
)

figure, axes = plt.subplots(
    2,
    3,
    figsize=(
        16,
        10.5
    )
)

plot_continuous(
    axes[0, 0],
    RASTERS["Urban Expansion Probability"],
    "(a) Expansion probability",
    "magma",
    0,
    1,
    "Probability"
)

plot_continuous(
    axes[0, 1],
    RASTERS["Uncertainty"],
    "(b) Model uncertainty",
    "inferno_r",
    0,
    0.5,
    "Tree probability SD"
)

plot_categorical(
    axes[0, 2],
    RASTERS["Confidence"],
    "(c) Model confidence",
    CONFIDENCE_COLOURS,
    CONFIDENCE_LABELS,
    "Confidence"
)

plot_continuous(
    axes[1, 0],
    RASTERS["Constrained Probability"],
    "(d) Constrained probability",
    "viridis",
    0,
    1,
    "Probability"
)

plot_categorical(
    axes[1, 1],
    RASTERS["Suitability"],
    "(e) Development suitability",
    SUITABILITY_COLOURS,
    SUITABILITY_LABELS,
    "Suitability"
)

plot_categorical(
    axes[1, 2],
    RASTERS["Planning Priority"],
    "(f) Planning priority",
    PRIORITY_COLOURS,
    PRIORITY_LABELS,
    "Priority"
)

figure.suptitle(
    "Model Reliability and Urban Development Planning Outputs — Enugu State",
    fontsize=16,
    fontweight="bold",
    y=0.98
)

figure.tight_layout(
    rect=[
        0,
        0,
        1,
        0.95
    ]
)

figure.savefig(
    SIX_PANEL_PNG,
    dpi=EXPORT_DPI,
    bbox_inches="tight",
    facecolor="white"
)

figure.savefig(
    SIX_PANEL_PDF,
    dpi=EXPORT_DPI,
    bbox_inches="tight",
    facecolor="white"
)

plt.close(figure)

print(
    f"✓ {SIX_PANEL_PNG}"
)

print(
    f"✓ {SIX_PANEL_PDF}"
)


# ==========================================================================================
# 11. PORTFOLIO PRESENTATION BOARD
# ==========================================================================================

print_heading(
    "GENERATING PORTFOLIO PRESENTATION BOARD"
)

probability_image = load_final_png(
    FINAL_MAP_PNG_DIR
    / "01_Urban_Expansion_Probability.png"
)

suitability_image = load_final_png(
    FINAL_MAP_PNG_DIR
    / "03_Urban_Development_Suitability.png"
)

priority_image = load_final_png(
    FINAL_MAP_PNG_DIR
    / "04_Urban_Development_Planning_Priority.png"
)

uncertainty_image = load_final_png(
    FINAL_MAP_PNG_DIR
    / "06_Model_Uncertainty.png"
)

high_suitability_area = float(
    suitability_df.loc[
        suitability_df[
            "Suitability_Class"
        ].isin(
            [
                "High",
                "Very High"
            ]
        ),
        "Area_km2"
    ].sum()
)

very_high_priority_area = float(
    priority_df.loc[
        priority_df[
            "Planning_Priority"
        ]
        ==
        "Very High Priority",
        "Area_km2"
    ].sum()
)

moderate_confidence_percentage = float(
    confidence_df.loc[
        confidence_df[
            "Confidence_Class"
        ]
        ==
        "Moderate Confidence",
        "Percentage_of_Applicable_Land"
    ].iloc[0]
)

board_figure = plt.figure(
    figsize=(
        16,
        10
    ),
    facecolor="#f5f5f3"
)

board_figure.text(
    0.04,
    0.945,
    PROJECT_TITLE,
    fontsize=22,
    fontweight="bold",
    color="#153243"
)

board_figure.text(
    0.04,
    0.905,
    "Spatial prediction, environmental constraints and confidence-adjusted planning priority",
    fontsize=11,
    color="#4c5b66"
)

board_figure.add_artist(
    plt.Line2D(
        [
            0.04,
            0.96
        ],
        [
            0.885,
            0.885
        ],
        transform=board_figure.transFigure,
        linewidth=1.4,
        color="#153243"
    )
)

map_positions = [
    (
        0.04,
        0.48,
        0.43,
        0.38,
        probability_image,
        "Urban expansion probability"
    ),
    (
        0.50,
        0.48,
        0.43,
        0.38,
        suitability_image,
        "Development suitability"
    ),
    (
        0.04,
        0.08,
        0.43,
        0.34,
        priority_image,
        "Confidence-adjusted priority"
    ),
    (
        0.50,
        0.08,
        0.25,
        0.34,
        uncertainty_image,
        "Model uncertainty"
    )
]

for left, bottom, width, height, image, title in map_positions:
    axis = board_figure.add_axes(
        [
            left,
            bottom,
            width,
            height
        ]
    )

    axis.imshow(image)
    axis.axis("off")

    axis.set_title(
        title,
        fontsize=10,
        fontweight="bold",
        pad=5
    )

summary_axis = board_figure.add_axes(
    [
        0.77,
        0.08,
        0.19,
        0.34
    ]
)

summary_axis.axis("off")

summary_axis.text(
    0,
    0.96,
    "KEY RESULTS",
    fontsize=12,
    fontweight="bold",
    color="#153243"
)

summary_lines = [
    (
        "Applicable planning area",
        "6,050.85 km²"
    ),
    (
        "High and very high suitability",
        f"{high_suitability_area:,.2f} km²"
    ),
    (
        "High-priority area",
        "392.83 km²"
    ),
    (
        "Very-high-priority area",
        f"{very_high_priority_area:,.2f} km²"
    ),
    (
        "Moderate-confidence land",
        f"{moderate_confidence_percentage:.2f}%"
    ),
    (
        "Independent test ROC-AUC",
        "0.7517"
    ),
    (
        "Independent test F1 score",
        "0.7324"
    )
]

vertical_position = 0.84

for label, value in summary_lines:
    summary_axis.text(
        0,
        vertical_position,
        label,
        fontsize=8.5,
        color="#52616b"
    )

    summary_axis.text(
        0,
        vertical_position - 0.055,
        value,
        fontsize=11,
        fontweight="bold",
        color="#152f3e"
    )

    vertical_position -= 0.125

board_figure.text(
    0.04,
    0.025,
    "Methods: Dynamic World transition labels • spatial block sampling • "
    "Extra Trees classification • independent spatial test • uncertainty analysis",
    fontsize=8,
    color="#555555"
)

board_figure.text(
    0.96,
    0.025,
    f"Prepared by {AUTHOR_NAME}",
    fontsize=8,
    color="#555555",
    ha="right"
)

board_figure.savefig(
    PORTFOLIO_BOARD_PNG,
    dpi=EXPORT_DPI,
    bbox_inches="tight",
    facecolor=board_figure.get_facecolor()
)

board_figure.savefig(
    PORTFOLIO_BOARD_PDF,
    dpi=EXPORT_DPI,
    bbox_inches="tight",
    facecolor=board_figure.get_facecolor()
)

plt.close(board_figure)

print(
    f"✓ {PORTFOLIO_BOARD_PNG}"
)

print(
    f"✓ {PORTFOLIO_BOARD_PDF}"
)


# ==========================================================================================
# 12. COMBINED MAP ATLAS
# ==========================================================================================

print_heading(
    "GENERATING FINAL MAP ATLAS"
)

with PdfPages(
    ATLAS_PDF
) as pdf:
    title_figure = plt.figure(
        figsize=(
            11.69,
            8.27
        ),
        facecolor="white"
    )

    title_figure.text(
        0.5,
        0.67,
        "FINAL MAP ATLAS",
        ha="center",
        fontsize=28,
        fontweight="bold",
        color="#153243"
    )

    title_figure.text(
        0.5,
        0.55,
        PROJECT_TITLE,
        ha="center",
        fontsize=17,
        fontweight="bold"
    )

    title_figure.text(
        0.5,
        0.43,
        "Enugu State, Nigeria",
        ha="center",
        fontsize=14,
        color="#555555"
    )

    title_figure.text(
        0.5,
        0.28,
        "Probability • Suitability • Planning Priority • Confidence • Uncertainty",
        ha="center",
        fontsize=11
    )

    title_figure.text(
        0.5,
        0.12,
        f"Prepared by {AUTHOR_NAME}",
        ha="center",
        fontsize=10
    )

    plt.axis("off")

    pdf.savefig(
        title_figure,
        bbox_inches="tight"
    )

    plt.close(title_figure)

    for map_path in final_png_maps:
        image = Image.open(
            map_path
        ).convert(
            "RGB"
        )

        page = plt.figure(
            figsize=(
                11.69,
                8.27
            ),
            facecolor="white"
        )

        axis = page.add_axes(
            [
                0.015,
                0.015,
                0.97,
                0.97
            ]
        )

        axis.imshow(image)
        axis.axis("off")

        pdf.savefig(
            page,
            bbox_inches="tight"
        )

        plt.close(page)

print(
    f"✓ {ATLAS_PDF}"
)


# ==========================================================================================
# 13. OUTPUT REGISTER
# ==========================================================================================

output_files = [
    FOUR_PANEL_PNG,
    FOUR_PANEL_PDF,
    SIX_PANEL_PNG,
    SIX_PANEL_PDF,
    PORTFOLIO_BOARD_PNG,
    PORTFOLIO_BOARD_PDF,
    ATLAS_PDF
]

output_records = []

for path in output_files:
    validate_file(
        path,
        path.name
    )

    output_records.append({
        "Output_Name": path.stem,
        "Output_Type": path.suffix.lower().replace(
            ".",
            ""
        ),
        "Output_Path": str(path),
        "File_Size_MB": (
            path.stat().st_size
            /
            (
                1024 ** 2
            )
        ),
        "Status": "Completed"
    })

output_df = pd.DataFrame(
    output_records
)

output_df.to_csv(
    OUTPUT_REGISTER_FILE,
    index=False
)


# ==========================================================================================
# 14. VALIDATION
# ==========================================================================================

print_heading(
    "STAGE 12C VALIDATION"
)

validation_records = []


def add_validation(
    check,
    passed,
    observed,
    expected
):
    validation_records.append({
        "Check": check,
        "Passed": bool(passed),
        "Observed": str(observed),
        "Expected": str(expected)
    })


add_validation(
    "All six required rasters exist",
    all(
        path.exists()
        for path in RASTERS.values()
    ),
    sum(
        path.exists()
        for path in RASTERS.values()
    ),
    6
)

add_validation(
    "All four analytical tables exist",
    all(
        path.exists()
        for path in [
            SUITABILITY_TABLE,
            PRIORITY_TABLE,
            CONFIDENCE_TABLE,
            RELIABILITY_TABLE
        ]
    ),
    sum(
        path.exists()
        for path in [
            SUITABILITY_TABLE,
            PRIORITY_TABLE,
            CONFIDENCE_TABLE,
            RELIABILITY_TABLE
        ]
    ),
    4
)

add_validation(
    "Four-panel publication figure created",
    FOUR_PANEL_PNG.exists()
    and
    FOUR_PANEL_PDF.exists(),
    (
        FOUR_PANEL_PNG.exists(),
        FOUR_PANEL_PDF.exists()
    ),
    (
        True,
        True
    )
)

add_validation(
    "Six-panel reliability figure created",
    SIX_PANEL_PNG.exists()
    and
    SIX_PANEL_PDF.exists(),
    (
        SIX_PANEL_PNG.exists(),
        SIX_PANEL_PDF.exists()
    ),
    (
        True,
        True
    )
)

add_validation(
    "Portfolio board created",
    PORTFOLIO_BOARD_PNG.exists()
    and
    PORTFOLIO_BOARD_PDF.exists(),
    (
        PORTFOLIO_BOARD_PNG.exists(),
        PORTFOLIO_BOARD_PDF.exists()
    ),
    (
        True,
        True
    )
)

add_validation(
    "Final map atlas created",
    ATLAS_PDF.exists()
    and
    ATLAS_PDF.stat().st_size > 0,
    ATLAS_PDF.stat().st_size
    if ATLAS_PDF.exists()
    else 0,
    "> 0 bytes"
)

add_validation(
    "All registered outputs are non-empty",
    all(
        path.exists()
        and
        path.stat().st_size > 0
        for path in output_files
    ),
    [
        round(
            path.stat().st_size
            /
            (
                1024 ** 2
            ),
            2
        )
        for path in output_files
    ],
    "Every file size greater than zero"
)

validation_df = pd.DataFrame(
    validation_records
)

validation_df.to_csv(
    VALIDATION_REGISTER_FILE,
    index=False
)

failed_checks = validation_df.loc[
    ~validation_df[
        "Passed"
    ]
]

if not failed_checks.empty:
    print(
        failed_checks.to_string(
            index=False
        )
    )

    raise RuntimeError(
        "Stage 12C completed, but one or more validation checks failed."
    )


# ==========================================================================================
# 15. JSON REGISTER
# ==========================================================================================

register = {
    "stage": "12C",
    "stage_name": (
        "Multi-panel Figures, Portfolio Board and Final Map Atlas"
    ),
    "completion_time_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "project": PROJECT_TITLE,
    "study_area": "Enugu State, Nigeria",
    "author": AUTHOR_NAME,
    "export_dpi": EXPORT_DPI,
    "publication_figures": {
        "four_panel_png": str(
            FOUR_PANEL_PNG
        ),
        "four_panel_pdf": str(
            FOUR_PANEL_PDF
        ),
        "six_panel_png": str(
            SIX_PANEL_PNG
        ),
        "six_panel_pdf": str(
            SIX_PANEL_PDF
        )
    },
    "portfolio_outputs": {
        "portfolio_board_png": str(
            PORTFOLIO_BOARD_PNG
        ),
        "portfolio_board_pdf": str(
            PORTFOLIO_BOARD_PDF
        )
    },
    "atlas_output": str(
        ATLAS_PDF
    ),
    "output_register": str(
        OUTPUT_REGISTER_FILE
    ),
    "validation_register": str(
        VALIDATION_REGISTER_FILE
    ),
    "output_count": len(
        output_files
    ),
    "validation_passed": True
}

with open(
    JSON_REGISTER_FILE,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        register,
        file,
        indent=4
    )


# ==========================================================================================
# 16. FINAL RESULTS
# ==========================================================================================

print_heading(
    "STAGE 12C OUTPUT REGISTER"
)

print(
    output_df.to_string(
        index=False,
        formatters={
            "File_Size_MB":
                lambda value: f"{value:.2f}"
        }
    )
)

print_heading(
    "VALIDATION RESULTS"
)

print(
    validation_df.to_string(
        index=False
    )
)

print_heading(
    "OUTPUT LOCATIONS"
)

print(
    f"✓ Publication figures: {FIGURE_DIR}"
)

print(
    f"✓ Portfolio board: {PORTFOLIO_DIR}"
)

print(
    f"✓ Final map atlas: {ATLAS_PDF}"
)

print(
    f"✓ Output register: {OUTPUT_REGISTER_FILE}"
)

print(
    f"✓ Validation register: {VALIDATION_REGISTER_FILE}"
)

print(
    f"✓ JSON register: {JSON_REGISTER_FILE}"
)

print("=" * 120)
print("STAGE 12C COMPLETED SUCCESSFULLY")
print("=" * 120)
print(
    "✓ Publication-ready four-panel and six-panel figures were generated."
)
print(
    "✓ A scholarship and portfolio-ready project board was generated."
)
print(
    "✓ All seven final cartographic maps were compiled into one PDF atlas."
)
print(
    "✓ Output and validation registers were saved."
)
print(
    "✓ All Stage 12C validation checks passed."
)
print("=" * 120)

In [ ]:
# ==================================================================================================
# PROJECT 7 — STAGE 13A
# FINAL DELIVERABLES PACKAGE, GITHUB DOCUMENTATION AND PROJECT ARCHIVE
#
# Project:
# Machine-Learning-Based Land Suitability Analysis for Sustainable Urban Development — Enugu State
#
# Outputs:
#   01_Maps
#   02_Figures
#   03_GIS_Data
#   04_Tables
#   05_Reports
#   06_Portfolio
#   07_GitHub
#   08_Methodology
#   09_Validation
#   10_Project_Archive
#
# Major products:
#   - Final maps
#   - Publication figures
#   - GIS rasters
#   - Statistical tables
#   - README.md
#   - Portfolio project description
#   - CV-ready entry
#   - Methodology summary
#   - Findings and limitations
#   - File inventory
#   - SHA-256 checksums
#   - Final ZIP archive
# ==================================================================================================


# ==================================================================================================
# 1. IMPORT LIBRARIES
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from textwrap import dedent

import csv
import hashlib
import json
import os
import shutil
import zipfile

import pandas as pd


# ==================================================================================================
# 2. PROJECT PATHS
# ==================================================================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

PACKAGE_ROOT = (
    PROJECT_ROOT
    / "16_Final_Deliverables_Package"
)

MAPS_DIR = (
    PACKAGE_ROOT
    / "01_Maps"
)

FIGURES_DIR = (
    PACKAGE_ROOT
    / "02_Figures"
)

GIS_DIR = (
    PACKAGE_ROOT
    / "03_GIS_Data"
)

TABLES_DIR = (
    PACKAGE_ROOT
    / "04_Tables"
)

REPORTS_DIR = (
    PACKAGE_ROOT
    / "05_Reports"
)

PORTFOLIO_DIR = (
    PACKAGE_ROOT
    / "06_Portfolio"
)

GITHUB_DIR = (
    PACKAGE_ROOT
    / "07_GitHub"
)

METHODOLOGY_DIR = (
    PACKAGE_ROOT
    / "08_Methodology"
)

VALIDATION_DIR = (
    PACKAGE_ROOT
    / "09_Validation"
)

ARCHIVE_DIR = (
    PACKAGE_ROOT
    / "10_Project_Archive"
)

ADMIN_DIR = (
    PROJECT_ROOT
    / "00_Project_Admin"
)

for directory in [
    PACKAGE_ROOT,
    MAPS_DIR,
    FIGURES_DIR,
    GIS_DIR,
    TABLES_DIR,
    REPORTS_DIR,
    PORTFOLIO_DIR,
    GITHUB_DIR,
    METHODOLOGY_DIR,
    VALIDATION_DIR,
    ARCHIVE_DIR,
    ADMIN_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ==================================================================================================
# 3. SOURCE DIRECTORIES
# ==================================================================================================

FINAL_MAP_ROOT = (
    PROJECT_ROOT
    / "14_Final_Cartographic_Maps"
)

PUBLICATION_ROOT = (
    PROJECT_ROOT
    / "15_Publication_Portfolio"
)

STATEWIDE_DIR = (
    PROJECT_ROOT
    / "12_Statewide_Suitability"
)

PRIORITY_DIR = (
    PROJECT_ROOT
    / "13_Planning_Priority"
)

MODEL_DIR = (
    PROJECT_ROOT
    / "11_Model_Development"
)

TRAINING_DIR = (
    PROJECT_ROOT
    / "10_ML_Training_Data"
)


# ==================================================================================================
# 4. PROJECT METADATA AND VERIFIED RESULTS
# ==================================================================================================

PROJECT_TITLE = (
    "Machine-Learning-Based Land Suitability Analysis "
    "for Sustainable Urban Development — Enugu State, Nigeria"
)

AUTHOR = "Abdullah Abdazeez Ayomide"

STUDY_AREA = "Enugu State, Nigeria"

CRS = "EPSG:32632 — WGS 84 / UTM Zone 32N"

MODEL_NAME = "Leakage-corrected Extra Trees classifier"

MODEL_TREE_COUNT = 500

PIXEL_RESOLUTION = "30 metres"

INDEPENDENT_TEST_RESULTS = {
    "Accuracy": 0.7111,
    "Balanced Accuracy": 0.7111,
    "Precision": 0.6822,
    "Recall": 0.7906,
    "Specificity": 0.6317,
    "F1 Score": 0.7324,
    "ROC-AUC": 0.7517,
    "PR-AUC": 0.6718,
    "Matthews Correlation Coefficient": 0.4277,
    "Cohen Kappa": 0.4222,
    "Brier Score": 0.2015,
    "Log Loss": 0.5956
}

CONFUSION_MATRIX = {
    "True Negative": 1137,
    "False Positive": 663,
    "False Negative": 377,
    "True Positive": 1423
}

SUITABILITY_RESULTS = {
    "Very Low": 2177.69,
    "Low": 1988.35,
    "Moderate": 1353.05,
    "High": 515.09,
    "Very High": 16.67
}

PRIORITY_RESULTS = {
    "Very Low Priority": 4166.04,
    "Low Priority": 923.80,
    "Moderate Priority": 567.79,
    "High Priority": 392.83,
    "Very High Priority": 0.37
}

CONFIDENCE_RESULTS = {
    "Low Confidence": {
        "Area_km2": 2146.92,
        "Percentage": 35.48
    },
    "Moderate Confidence": {
        "Area_km2": 3873.85,
        "Percentage": 64.02
    },
    "High Confidence": {
        "Area_km2": 30.08,
        "Percentage": 0.50
    }
}

STATEWIDE_RESULTS = {
    "Valid model area": 7619.47,
    "Planning-applicable area": 6050.85,
    "Planning-constrained area": 1568.63,
    "Mean raw expansion probability": 0.3500,
    "Mean constrained probability": 0.3137,
    "Mean model uncertainty": 0.2844
}

FEATURE_IMPORTANCE = {
    "Distance to roads": 41.67,
    "Population density": 16.97,
    "Distance to recurring surface water": 12.02,
    "Elevation": 11.54,
    "Distance to drainage": 9.06,
    "Slope": 5.62,
    "Baseline land cover 2020": 3.12
}

LEAKAGE_SAFE_PREDICTORS = [
    "Elevation",
    "Slope",
    "Distance to roads",
    "Distance to recurring surface water",
    "Distance to drainage",
    "Population density",
    "Baseline Dynamic World land cover 2020"
]


# ==================================================================================================
# 5. HELPER FUNCTIONS
# ==================================================================================================

def print_heading(title):
    print("\n" + title)
    print("-" * 120)


def validate_file(path, label=None):
    label = label or path.name

    if not path.exists():
        raise FileNotFoundError(
            f"{label} was not found:\n{path}"
        )

    if path.stat().st_size == 0:
        raise RuntimeError(
            f"{label} exists but is empty:\n{path}"
        )


def safe_copy(source, destination_directory, output_name=None):
    source = Path(source)

    validate_file(
        source,
        source.name
    )

    destination_directory.mkdir(
        parents=True,
        exist_ok=True
    )

    destination = (
        destination_directory
        /
        (
            output_name
            if output_name is not None
            else source.name
        )
    )

    shutil.copy2(
        source,
        destination
    )

    return destination


def copy_matching_files(
    source_directory,
    destination_directory,
    patterns,
    recursive=False
):
    copied = []

    if not source_directory.exists():
        return copied

    for pattern in patterns:
        if recursive:
            matches = source_directory.rglob(
                pattern
            )
        else:
            matches = source_directory.glob(
                pattern
            )

        for source in matches:
            if not source.is_file():
                continue

            destination = (
                destination_directory
                /
                source.name
            )

            destination_directory.mkdir(
                parents=True,
                exist_ok=True
            )

            shutil.copy2(
                source,
                destination
            )

            copied.append(
                destination
            )

    return copied


def write_text_file(path, content):
    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    path.write_text(
        dedent(content).strip() + "\n",
        encoding="utf-8"
    )

    return path


def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as file:
        while True:
            block = file.read(
                block_size
            )

            if not block:
                break

            digest.update(
                block
            )

    return digest.hexdigest()


def human_file_size(size_bytes):
    units = [
        "B",
        "KB",
        "MB",
        "GB",
        "TB"
    ]

    size = float(
        size_bytes
    )

    for unit in units:
        if size < 1024 or unit == units[-1]:
            return f"{size:.2f} {unit}"

        size /= 1024


def create_zip_archive(
    source_directory,
    zip_path
):
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(
        zip_path,
        "w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=6
    ) as archive:
        for file_path in sorted(
            source_directory.rglob(
                "*"
            )
        ):
            if not file_path.is_file():
                continue

            if file_path == zip_path:
                continue

            archive_name = file_path.relative_to(
                source_directory.parent
            )

            archive.write(
                file_path,
                arcname=archive_name
            )

    return zip_path


# ==================================================================================================
# 6. COPY FINAL MAPS
# ==================================================================================================

print("=" * 120)
print("STAGE 13A — FINAL DELIVERABLES PACKAGE, GITHUB DOCUMENTATION AND PROJECT ARCHIVE")
print("=" * 120)

print_heading(
    "COPYING FINAL MAPS"
)

map_files = []

map_files.extend(
    copy_matching_files(
        FINAL_MAP_ROOT / "PNG",
        MAPS_DIR,
        [
            "*.png"
        ]
    )
)

map_files.extend(
    copy_matching_files(
        FINAL_MAP_ROOT / "PDF",
        MAPS_DIR,
        [
            "*.pdf"
        ]
    )
)

for file_path in map_files:
    print(
        f"✓ {file_path.name}"
    )


# ==================================================================================================
# 7. COPY PUBLICATION FIGURES AND PORTFOLIO PRODUCTS
# ==================================================================================================

print_heading(
    "COPYING FIGURES AND PORTFOLIO OUTPUTS"
)

figure_files = []

figure_files.extend(
    copy_matching_files(
        PUBLICATION_ROOT / "Figures",
        FIGURES_DIR,
        [
            "*.png",
            "*.pdf"
        ]
    )
)

atlas_source = (
    PUBLICATION_ROOT
    / "Atlas"
    / "Enugu_ML_Land_Suitability_Final_Map_Atlas.pdf"
)

if atlas_source.exists():
    figure_files.append(
        safe_copy(
            atlas_source,
            FIGURES_DIR
        )
    )

portfolio_files = copy_matching_files(
    PUBLICATION_ROOT / "Portfolio",
    PORTFOLIO_DIR,
    [
        "*.png",
        "*.pdf"
    ]
)

for file_path in figure_files:
    print(
        f"✓ Figure: {file_path.name}"
    )

for file_path in portfolio_files:
    print(
        f"✓ Portfolio: {file_path.name}"
    )


# ==================================================================================================
# 8. COPY KEY GIS DATA
# ==================================================================================================

print_heading(
    "COPYING KEY GIS DATA"
)

gis_sources = [
    (
        STATEWIDE_DIR
        / "Enugu_Urban_Expansion_Probability_2025_30m.tif"
    ),
    (
        STATEWIDE_DIR
        / "Enugu_Planning_Constrained_Expansion_Probability_30m.tif"
    ),
    (
        STATEWIDE_DIR
        / "Enugu_Model_Uncertainty_30m.tif"
    ),
    (
        STATEWIDE_DIR
        / "Enugu_Model_Confidence_30m.tif"
    ),
    (
        STATEWIDE_DIR
        / "Enugu_Model_Applicability_Mask_30m.tif"
    ),
    (
        STATEWIDE_DIR
        / "Enugu_Urban_Development_Suitability_Classes_30m.tif"
    ),
    (
        STATEWIDE_DIR
        / "Enugu_Planning_Constraint_Codes_30m.tif"
    ),
    (
        PRIORITY_DIR
        / "Enugu_Model_Confidence_Classes_30m.tif"
    ),
    (
        PRIORITY_DIR
        / "Enugu_High_Confidence_Suitability_30m.tif"
    ),
    (
        PRIORITY_DIR
        / "Enugu_Urban_Development_Planning_Priority_30m.tif"
    )
]

gis_files = []

for source in gis_sources:
    if source.exists():
        copied_file = safe_copy(
            source,
            GIS_DIR
        )

        gis_files.append(
            copied_file
        )

        print(
            f"✓ {copied_file.name}"
        )
    else:
        print(
            f"⚠ Optional GIS file not found: {source.name}"
        )


# ==================================================================================================
# 9. COPY IMPORTANT TABLES
# ==================================================================================================

print_heading(
    "COPYING TABLES"
)

table_files = []

for source_directory in [
    STATEWIDE_DIR / "Tables",
    PRIORITY_DIR / "Tables",
    FINAL_MAP_ROOT / "Tables",
    PUBLICATION_ROOT / "Tables"
]:
    table_files.extend(
        copy_matching_files(
            source_directory,
            TABLES_DIR,
            [
                "*.csv",
                "*.xlsx",
                "*.json"
            ]
        )
    )

additional_table_patterns = [
    "*Test*Metrics*.csv",
    "*Confusion*Matrix*.csv",
    "*Feature*Importance*.csv",
    "*Permutation*Importance*.csv",
    "*Model*Comparison*.csv",
    "*Sample*Summary*.csv",
    "*Predictor*Statistics*.csv"
]

for pattern in additional_table_patterns:
    for source in PROJECT_ROOT.rglob(
        pattern
    ):
        if not source.is_file():
            continue

        destination = (
            TABLES_DIR
            / source.name
        )

        if destination.exists():
            continue

        shutil.copy2(
            source,
            destination
        )

        table_files.append(
            destination
        )

for file_path in sorted(
    set(
        table_files
    )
):
    print(
        f"✓ {file_path.name}"
    )


# ==================================================================================================
# 10. COPY VALIDATION AND REGISTER FILES
# ==================================================================================================

print_heading(
    "COPYING VALIDATION AND REPRODUCIBILITY REGISTERS"
)

validation_patterns = [
    "*Validation*.csv",
    "*Register*.json",
    "*Register*.csv",
    "*Evaluation*.json",
    "*Evaluation*.csv",
    "*QA*.csv"
]

validation_files = []

for pattern in validation_patterns:
    for source in ADMIN_DIR.glob(
        pattern
    ):
        if not source.is_file():
            continue

        destination = (
            VALIDATION_DIR
            / source.name
        )

        if destination.exists():
            continue

        shutil.copy2(
            source,
            destination
        )

        validation_files.append(
            destination
        )

for file_path in validation_files:
    print(
        f"✓ {file_path.name}"
    )


# ==================================================================================================
# 11. GENERATE PROJECT ABSTRACT
# ==================================================================================================

print_heading(
    "GENERATING PROJECT DOCUMENTATION"
)

abstract_text = f"""
# Project Abstract

## {PROJECT_TITLE}

Rapid urban expansion can place pressure on infrastructure, environmentally sensitive land and
existing settlements when development decisions are not supported by reliable spatial evidence.
This project developed a machine-learning-based framework for identifying land with differing
levels of urban development suitability across Enugu State, Nigeria.

A transition-based target was constructed from Dynamic World land-cover observations for 2020
and 2025. Positive samples represented quality-controlled non-built-to-built transitions, while
negative samples represented stable non-built areas located within a defined urban influence
zone. Spatial patch size, neighbourhood support, observation count and distance-to-existing-built
criteria were used to reduce label noise. The data were divided using spatial blocks to prevent
geographical overlap among training, validation and independent test samples.

Seven leakage-safe predictors were used: elevation, slope, distance to roads, distance to recurring
surface water, distance to drainage, population density and baseline 2020 land cover. A predictor
that was also used in target construction—distance to 2020 built-up land—was removed following a
formal leakage audit. The final Extra Trees model achieved an independent spatial test ROC-AUC of
{INDEPENDENT_TEST_RESULTS["ROC-AUC"]:.4f}, an F1 score of
{INDEPENDENT_TEST_RESULTS["F1 Score"]:.4f} and balanced accuracy of
{INDEPENDENT_TEST_RESULTS["Balanced Accuracy"]:.4f}.

Statewide prediction identified {SUITABILITY_RESULTS["High"] + SUITABILITY_RESULTS["Very High"]:,.2f}
km² of high or very-high suitability. After accounting for model confidence, approximately
{PRIORITY_RESULTS["High Priority"]:,.2f} km² was classified as high planning priority, while only
{PRIORITY_RESULTS["Very High Priority"]:,.2f} km² met the strict very-high-priority criteria.
Road accessibility was the dominant model predictor, followed by population density, proximity
to recurring surface water and elevation.

The outputs provide a screening-level decision-support framework for strategic urban planning.
They should support, rather than replace, detailed site investigation, statutory planning review,
environmental impact assessment and local stakeholder consultation.
"""

abstract_file = write_text_file(
    REPORTS_DIR
    / "01_Project_Abstract.md",
    abstract_text
)

print(
    f"✓ {abstract_file.name}"
)


# ==================================================================================================
# 12. GENERATE TECHNICAL SUMMARY
# ==================================================================================================

technical_summary_text = f"""
# Technical Project Summary

## Project title

{PROJECT_TITLE}

## Study area

{STUDY_AREA}

## Coordinate reference system

{CRS}

## Spatial resolution

{PIXEL_RESOLUTION}

## Problem statement

Urban growth in Enugu State requires spatially explicit evidence that distinguishes locations with
high development potential from areas affected by physical limitations, environmental constraints
or model uncertainty. Conventional weighted-overlay suitability models often depend heavily on
subjective criterion weights. This project applied a supervised machine-learning framework based
on observed urban transitions while explicitly addressing target leakage, spatial dependence,
classification uncertainty and planning constraints.

## Objective

To develop and validate a machine-learning-based spatial model that estimates urban expansion
probability and converts the results into environmentally constrained suitability and
confidence-adjusted planning-priority classes.

## Data sources

- Copernicus DEM GLO-30 for elevation and slope
- Dynamic World V1 for 2020 and 2025 land cover
- OpenStreetMap for road-network proximity
- JRC Global Surface Water for recurring surface water
- MERIT Hydro for drainage proximity
- WorldPop for population density
- Administrative boundaries for Enugu State

## Leakage-safe predictors

{chr(10).join(f"- {predictor}" for predictor in LEAKAGE_SAFE_PREDICTORS)}

## Target definition

Positive samples represented reliable urban gain between 2020 and 2025:

- non-built in 2020;
- built in 2025;
- at least five valid observations;
- transition patch of at least five pixels;
- within 90 metres of stable 2020 built-up land;
- neighbourhood support of at least 0.67.

Negative samples represented stable non-built land:

- non-built in both 2020 and 2025;
- at least five valid observations;
- slope not exceeding 15 degrees;
- located 90–1,500 metres from stable 2020 built-up land;
- at least 90 metres from accepted positive samples.

## Sampling and validation

A balanced sample containing 24,000 records was generated.

- Training: 16,800 samples
- Validation: 3,600 samples
- Independent test: 3,600 samples
- Training spatial blocks: 145
- Validation spatial blocks: 40
- Test spatial blocks: 32
- Spatial block overlap: zero

## Model

{MODEL_NAME}

Number of trees: {MODEL_TREE_COUNT}

## Independent spatial test results

{chr(10).join(f"- {metric}: {value:.4f}" for metric, value in INDEPENDENT_TEST_RESULTS.items())}

## Confusion matrix

{chr(10).join(f"- {label}: {value:,}" for label, value in CONFUSION_MATRIX.items())}

## Statewide results

{chr(10).join(f"- {label}: {value:,.4f}" if "probability" in label.lower() or "uncertainty" in label.lower() else f"- {label}: {value:,.2f} km²" for label, value in STATEWIDE_RESULTS.items())}

## Suitability distribution

{chr(10).join(f"- {class_name}: {area:,.2f} km²" for class_name, area in SUITABILITY_RESULTS.items())}

## Planning-priority distribution

{chr(10).join(f"- {class_name}: {area:,.2f} km²" for class_name, area in PRIORITY_RESULTS.items())}

## Model-confidence distribution

{chr(10).join(f'- {class_name}: {record["Area_km2"]:,.2f} km² ({record["Percentage"]:.2f}%)' for class_name, record in CONFIDENCE_RESULTS.items())}

## Grouped feature importance

{chr(10).join(f"- {name}: {importance:.2f}%" for name, importance in FEATURE_IMPORTANCE.items())}

## Principal interpretation

The model indicates that proximity to road infrastructure is the strongest spatial factor
associated with recent urban expansion. Population density provides an additional indicator of
development pressure. Terrain, surface-water proximity, drainage proximity and baseline land
cover influence where expansion is physically or environmentally more likely.

The confidence results show that most applicable land falls within the moderate-confidence class.
Only a small area achieved high confidence. Planning-priority results were therefore intentionally
conservative.

## Intended use

The outputs are appropriate for:

- state-level strategic planning;
- regional urban-growth screening;
- identification of candidate development corridors;
- comparison of alternative planning zones;
- prioritisation of areas for detailed site investigation;
- environmental and infrastructure planning.

## Appropriate caution

The suitability and priority outputs are screening products. They do not constitute building
approval, cadastral verification, geotechnical certification or environmental-impact clearance.
"""

technical_summary_file = write_text_file(
    REPORTS_DIR
    / "02_Technical_Project_Summary.md",
    technical_summary_text
)

print(
    f"✓ {technical_summary_file.name}"
)


# ==================================================================================================
# 13. GENERATE METHODOLOGY DOCUMENT
# ==================================================================================================

methodology_text = """
# Methodology

## 1. Study-area preparation

The Enugu State administrative boundary was prepared in WGS 84 geographic coordinates and
reprojected to WGS 84 / UTM Zone 32N for area, distance and raster analysis.

## 2. Predictor preparation

The modelling predictors were prepared from terrain, land-cover, hydrological, population and
transport datasets. All continuous predictor rasters were aligned to a common 30-metre reference
grid using bilinear resampling where required. Categorical land cover was aligned using
nearest-neighbour resampling.

The final predictors were:

1. Elevation
2. Slope
3. Distance to roads
4. Distance to recurring surface water
5. Distance to drainage
6. Population density
7. Baseline Dynamic World 2020 land cover

## 3. Urban-transition target

Dynamic World observations for 2020 and 2025 were compared to derive stable non-built, stable
built, urban-gain and apparent urban-loss transitions. Observation-count and spatial-persistence
tests showed that raw transitions contained substantial noise.

A conservative spatial reliability procedure was therefore applied. Accepted gain pixels had to
meet minimum observation, patch-size, proximity and neighbourhood-support thresholds.

## 4. Negative-sample construction

Negative samples were selected from stable non-built locations that were physically plausible
candidates for expansion but did not transition during the observation period. Restricting
negative samples to the urban influence zone prevented the model from learning an overly simple
contrast between remote rural areas and urban-edge expansion.

## 5. Spatial sampling

Positive and negative samples were balanced. Spatial block allocation was solved using mixed
integer linear programming to preserve class balance while maintaining complete separation
between training, validation and test blocks.

## 6. Leakage audit

The initial feature set included distance to 2020 built-up land. Because this variable also formed
part of the target-label definition, it created circular prediction and unrealistically high model
performance.

A rule-based leakage test achieved approximately 97.9% accuracy using the 90-metre label boundary
alone. The distance-to-built predictor was therefore removed from the official model.

## 7. Candidate models

The following algorithms were evaluated:

- Logistic Regression
- Random Forest
- Extra Trees
- Histogram Gradient Boosting

Training data were used for fitting, validation data were used for model comparison and the test
set remained untouched until final evaluation.

## 8. Independent evaluation

The final Extra Trees classifier was evaluated against the spatially independent test set using:

- accuracy;
- balanced accuracy;
- precision;
- recall;
- specificity;
- F1 score;
- ROC-AUC;
- precision-recall AUC;
- Matthews correlation coefficient;
- Cohen's kappa;
- Brier score;
- log loss;
- confusion matrix.

## 9. Model interpretation

Model interpretation used grouped native feature importance, validation permutation importance,
independent-test permutation importance and partial-dependence analysis.

## 10. Statewide prediction

The fixed final model was applied across Enugu State using tiled raster processing. Predictor order,
band identity, categorical resampling and output ranges were validated during prediction.

## 11. Planning constraints

The raw probability surface was converted to a planning-applicable surface by excluding or
constraining unsuitable land such as existing built-up areas, water-related zones and steep terrain.

## 12. Suitability classes

The constrained probability surface was reclassified into:

- Very Low
- Low
- Moderate
- High
- Very High

## 13. Confidence and uncertainty

Tree-level prediction variation was used to estimate uncertainty and confidence. Confidence was
classified into low, moderate and high categories.

## 14. Planning priority

Planning priority combined development suitability with model confidence. This ensured that high
predicted suitability did not automatically translate into high planning priority where predictive
confidence was weak.

## 15. Cartography and reporting

All final maps were produced using a consistent professional layout, projected coordinate grid,
scale bar, north arrow, legend, boundary overlay and 450-dpi PNG/PDF export.
"""

methodology_file = write_text_file(
    METHODOLOGY_DIR
    / "01_Detailed_Methodology.md",
    methodology_text
)

print(
    f"✓ {methodology_file.name}"
)


# ==================================================================================================
# 14. GENERATE LIMITATIONS AND RECOMMENDATIONS
# ==================================================================================================

limitations_text = """
# Limitations and Planning Recommendations

## Limitations

### Dynamic World transition noise

The target variable depends on satellite-derived land-cover classifications. Apparent built-up gain
or loss can reflect spectral confusion, seasonal variation, mixed pixels or incomplete observation
coverage rather than actual land development.

### Conservative positive labels

The quality-control thresholds improved target reliability but excluded some genuine expansion
that did not satisfy the patch, proximity or support requirements.

### Temporal scope

The model explains transitions observed between 2020 and 2025. Development relationships may
change under future infrastructure investment, policy changes, migration or economic shocks.

### Road-network completeness

OpenStreetMap road data may be more complete in urban centres than in remote locations. Distance
to roads therefore reflects both accessibility and mapping completeness.

### Population uncertainty

WorldPop provides modelled population estimates rather than a complete building- or
household-level census.

### Missing planning variables

The model does not include every factor required for statutory land-allocation decisions. Examples
include:

- land ownership and cadastral status;
- property prices;
- soil-bearing capacity;
- detailed flood-depth modelling;
- protected-area regulations;
- utility-service capacity;
- detailed transport capacity;
- cultural-heritage constraints;
- community preferences.

### Moderate generalisation

The independent spatial test performance was useful but not perfect. The model should therefore be
interpreted as a planning-screening tool rather than a deterministic prediction of future
development.

### Limited high-confidence area

Only a small proportion of applicable land achieved high model confidence. This reinforces the need
for ground verification before planning decisions.

## Planning recommendations

1. Prioritise high-priority zones for detailed local planning and field investigation.
2. Treat very-high-priority zones as focused candidate areas rather than automatic development sites.
3. Require environmental and drainage assessment before development near surface-water systems.
4. Coordinate future road investment with compact-growth and infrastructure-capacity objectives.
5. Protect steep terrain and environmentally sensitive land from poorly controlled expansion.
6. Use moderate-priority zones for longer-term monitoring rather than immediate allocation.
7. Update the model when improved census, road, cadastral and development-permit data become available.
8. Conduct local-scale modelling before neighbourhood or parcel-level planning.
9. Integrate the outputs with flood-risk, agricultural-value and ecosystem-service assessments.
10. Maintain a transparent audit trail of model assumptions, exclusions and confidence levels.
"""

limitations_file = write_text_file(
    METHODOLOGY_DIR
    / "02_Limitations_and_Planning_Recommendations.md",
    limitations_text
)

print(
    f"✓ {limitations_file.name}"
)


# ==================================================================================================
# 15. GENERATE DATA DICTIONARY
# ==================================================================================================

data_dictionary = pd.DataFrame([
    {
        "Variable": "Elevation_m",
        "Role": "Predictor",
        "Type": "Continuous",
        "Source": "Copernicus DEM GLO-30",
        "Unit": "metres",
        "Interpretation": "Terrain elevation"
    },
    {
        "Variable": "Slope_deg",
        "Role": "Predictor and planning constraint",
        "Type": "Continuous",
        "Source": "Derived from Copernicus DEM",
        "Unit": "degrees",
        "Interpretation": "Terrain steepness"
    },
    {
        "Variable": "Distance_to_Road_m",
        "Role": "Predictor",
        "Type": "Continuous",
        "Source": "OpenStreetMap",
        "Unit": "metres",
        "Interpretation": "Proximity to mapped road infrastructure"
    },
    {
        "Variable": "Distance_to_Surface_Water_m",
        "Role": "Predictor",
        "Type": "Continuous",
        "Source": "JRC Global Surface Water",
        "Unit": "metres",
        "Interpretation": "Proximity to recurring surface water"
    },
    {
        "Variable": "Distance_to_Drainage_m",
        "Role": "Predictor",
        "Type": "Continuous",
        "Source": "MERIT Hydro",
        "Unit": "metres",
        "Interpretation": "Proximity to drainage network"
    },
    {
        "Variable": "Population_Density",
        "Role": "Predictor",
        "Type": "Continuous",
        "Source": "WorldPop",
        "Unit": "persons per raster cell or density unit",
        "Interpretation": "Population concentration and development pressure"
    },
    {
        "Variable": "Baseline_LULC_2020",
        "Role": "Predictor",
        "Type": "Categorical",
        "Source": "Dynamic World V1",
        "Unit": "class code",
        "Interpretation": "Baseline land-cover context"
    },
    {
        "Variable": "Urban_Expansion_Label",
        "Role": "Target",
        "Type": "Binary",
        "Source": "Dynamic World 2020–2025 transition",
        "Unit": "0 or 1",
        "Interpretation": "Reliable urban gain versus stable non-built land"
    },
    {
        "Variable": "Expansion_Probability",
        "Role": "Model output",
        "Type": "Continuous",
        "Source": "Extra Trees classifier",
        "Unit": "0–1",
        "Interpretation": "Predicted urban-expansion probability"
    },
    {
        "Variable": "Suitability_Class",
        "Role": "Planning output",
        "Type": "Categorical",
        "Source": "Constrained expansion probability",
        "Unit": "1–5",
        "Interpretation": "Very Low to Very High suitability"
    },
    {
        "Variable": "Planning_Priority",
        "Role": "Planning output",
        "Type": "Categorical",
        "Source": "Suitability and model confidence",
        "Unit": "1–5",
        "Interpretation": "Very Low to Very High planning priority"
    }
])

data_dictionary_file = (
    METHODOLOGY_DIR
    / "03_Project_Data_Dictionary.csv"
)

data_dictionary.to_csv(
    data_dictionary_file,
    index=False
)

print(
    f"✓ {data_dictionary_file.name}"
)


# ==================================================================================================
# 16. GENERATE PORTFOLIO DESCRIPTION
# ==================================================================================================

portfolio_text = f"""
# Portfolio Project Description

## {PROJECT_TITLE}

### Project overview

This project developed a spatial machine-learning framework for identifying land suitable for
sustainable urban development across Enugu State. Rather than relying exclusively on subjective
weighted-overlay scores, the model learned from quality-controlled urban land transitions observed
between 2020 and 2025.

### Problem

Urban growth can expand into environmentally sensitive, inaccessible or physically constrained
locations when development decisions lack reliable spatial evidence. A defensible planning model
must distinguish predicted expansion pressure from genuine suitability and must communicate model
uncertainty.

### Approach

I prepared terrain, land-cover, transport, hydrological and population predictors and aligned them
to a common 30-metre raster grid. Urban-gain labels were derived from Dynamic World observations
using observation-count, patch-size, proximity and neighbourhood-support rules.

Balanced samples were separated using non-overlapping spatial blocks. Four candidate algorithms
were assessed. A formal leakage audit identified that distance to existing built-up land had also
been used to construct the labels. I removed this circular predictor and rebuilt the final model
using seven leakage-safe variables.

The selected Extra Trees classifier was assessed against a completely independent spatial test set.
I then generated statewide expansion probability, environmental applicability, suitability,
uncertainty, confidence and confidence-adjusted planning-priority surfaces.

### Key results

- Independent spatial test ROC-AUC: {INDEPENDENT_TEST_RESULTS["ROC-AUC"]:.4f}
- Independent spatial test F1 score: {INDEPENDENT_TEST_RESULTS["F1 Score"]:.4f}
- Independent spatial test balanced accuracy: {INDEPENDENT_TEST_RESULTS["Balanced Accuracy"]:.4f}
- Planning-applicable land: {STATEWIDE_RESULTS["Planning-applicable area"]:,.2f} km²
- High and very-high suitability: {SUITABILITY_RESULTS["High"] + SUITABILITY_RESULTS["Very High"]:,.2f} km²
- High planning priority: {PRIORITY_RESULTS["High Priority"]:,.2f} km²
- Very-high planning priority: {PRIORITY_RESULTS["Very High Priority"]:,.2f} km²
- Strongest predictor: distance to roads

### Skills demonstrated

- GIS and spatial analysis
- Remote sensing
- Raster processing
- Machine learning
- Spatial sampling
- Target engineering
- Data-leakage detection
- Independent spatial validation
- Model uncertainty analysis
- Explainable machine learning
- Urban and regional planning
- Scientific cartography
- Reproducible project documentation

### Software and tools

- Python
- Google Colab
- Google Earth Engine
- GeoPandas
- Rasterio
- NumPy
- pandas
- scikit-learn
- Matplotlib
- OpenStreetMap
- Git and GitHub

### Planning significance

The final priority surface separates areas that are merely predicted to experience urban expansion
from locations that are simultaneously suitable, environmentally applicable and supported by
stronger model confidence.
"""

portfolio_description_file = write_text_file(
    PORTFOLIO_DIR
    / "01_Portfolio_Project_Description.md",
    portfolio_text
)

print(
    f"✓ {portfolio_description_file.name}"
)


# ==================================================================================================
# 17. GENERATE CV-READY ENTRY
# ==================================================================================================

cv_text = f"""
# CV-Ready Project Entry

## Full version

**Machine-Learning-Based Land Suitability Analysis for Sustainable Urban Development — Enugu State**

Developed a leakage-corrected Extra Trees spatial model using remote-sensing, terrain,
infrastructure, hydrological and population predictors to estimate urban expansion and development
suitability across Enugu State. Constructed quality-controlled 2020–2025 urban-transition labels,
implemented non-overlapping spatial train-validation-test blocks and evaluated the model using an
independent spatial test set. Achieved ROC-AUC of {INDEPENDENT_TEST_RESULTS["ROC-AUC"]:.4f} and
F1 score of {INDEPENDENT_TEST_RESULTS["F1 Score"]:.4f}. Produced 30-metre statewide probability,
suitability, uncertainty, confidence and planning-priority maps, identifying
{PRIORITY_RESULTS["High Priority"]:,.2f} km² of high-priority land.

**Tools:** Python, Google Earth Engine, GeoPandas, Rasterio, scikit-learn, OpenStreetMap,
Dynamic World, Copernicus DEM, WorldPop and Matplotlib.

## Concise version

Built and spatially validated a leakage-safe Extra Trees model for urban expansion and land
suitability in Enugu State, producing 30-metre probability, uncertainty, confidence and
planning-priority maps. The independent test achieved ROC-AUC {INDEPENDENT_TEST_RESULTS["ROC-AUC"]:.4f}
and F1 {INDEPENDENT_TEST_RESULTS["F1 Score"]:.4f}.

## One-line version

Machine-learning land-suitability modelling and spatial validation for sustainable urban
development in Enugu State using Python, Earth Engine and remote-sensing data.
"""

cv_file = write_text_file(
    PORTFOLIO_DIR
    / "02_CV_Ready_Project_Entry.md",
    cv_text
)

print(
    f"✓ {cv_file.name}"
)
# ==================================================================================================
# 18. GENERATE GITHUB README — CORRECTED VERSION
# ==================================================================================================

predictor_markdown = "\n".join(
    [
        f"- {predictor}"
        for predictor in LEAKAGE_SAFE_PREDICTORS
    ]
)

test_metric_markdown = "\n".join(
    [
        f"| {metric} | {value:.4f} |"
        for metric, value in INDEPENDENT_TEST_RESULTS.items()
    ]
)

suitability_markdown = "\n".join(
    [
        f"| {class_name} | {area:,.2f} |"
        for class_name, area in SUITABILITY_RESULTS.items()
    ]
)

priority_markdown = "\n".join(
    [
        f"| {class_name} | {area:,.2f} |"
        for class_name, area in PRIORITY_RESULTS.items()
    ]
)

current_year = datetime.now().year

readme_text = f"""
# Machine-Learning-Based Land Suitability Analysis for Sustainable Urban Development — Enugu

## Overview

This repository documents a spatial machine-learning project that predicts urban expansion and
identifies suitable and priority areas for sustainable urban development across Enugu State,
Nigeria.

The workflow combines remote sensing, terrain, population, road accessibility, surface-water and
drainage data. It includes explicit target-quality assessment, spatial sampling, leakage detection,
independent spatial validation, uncertainty analysis and professional cartographic production.

## Study area

Enugu State, Nigeria

## Coordinate system

{CRS}

## Spatial resolution

{PIXEL_RESOLUTION}

## Project objectives

1. Construct quality-controlled urban-expansion labels from 2020–2025 land-cover transitions.
2. Prepare terrain, accessibility, environmental and demographic predictors.
3. Train and compare machine-learning models.
4. Identify and remove target-definition leakage.
5. Evaluate the final model using an independent spatial test set.
6. Create statewide expansion-probability and uncertainty surfaces.
7. Apply planning and environmental constraints.
8. Classify development suitability and confidence-adjusted planning priority.
9. Produce publication-quality maps, figures and documentation.

## Data sources

| Dataset | Application |
|---|---|
| Dynamic World V1 | Baseline land cover and urban-transition labels |
| Copernicus DEM GLO-30 | Elevation and slope |
| OpenStreetMap | Roads and distance to roads |
| JRC Global Surface Water | Recurring surface water and proximity |
| MERIT Hydro | Drainage network and proximity |
| WorldPop | Population density |
| Administrative boundaries | Study-area extraction and cartography |

## Final predictors

{predictor_markdown}

## Model development

Four candidate models were evaluated:

- Logistic Regression
- Random Forest
- Extra Trees
- Histogram Gradient Boosting

The official model is a leakage-corrected Extra Trees classifier containing
{MODEL_TREE_COUNT} trees.

## Leakage correction

Distance to 2020 built-up land was initially included as a predictor. The same distance threshold
was also used during positive and negative label construction. A rule based on this threshold alone
achieved approximately 97.9% accuracy, confirming circularity.

The variable was therefore removed before the official model was selected and evaluated.

## Independent spatial test performance

| Metric | Value |
|---|---:|
{test_metric_markdown}

## Suitability results

| Suitability class | Area (km²) |
|---|---:|
{suitability_markdown}

## Planning-priority results

| Priority class | Area (km²) |
|---|---:|
{priority_markdown}

## Major interpretation

Distance to roads was the strongest predictor of observed urban expansion. Population density,
proximity to recurring surface water and elevation were also influential.

The confidence-adjusted priority result is intentionally conservative. Only
{PRIORITY_RESULTS["Very High Priority"]:,.2f} km² was assigned to the very-high-priority class.

## Repository structure

16_Final_Deliverables_Package/

- 01_Maps
- 02_Figures
- 03_GIS_Data
- 04_Tables
- 05_Reports
- 06_Portfolio
- 07_GitHub
- 08_Methodology
- 09_Validation
- 10_Project_Archive

## Main outputs

- Urban-expansion probability map
- Planning-constrained expansion-probability map
- Urban-development suitability map
- Planning-priority map
- Model-confidence map
- Model-uncertainty map
- High-confidence suitability map
- Publication multi-panel figures
- Portfolio presentation board
- Final map atlas
- Statistical and validation tables

## Software

- Python
- Google Colab
- Google Earth Engine
- GeoPandas
- Rasterio
- pandas
- NumPy
- scikit-learn
- Matplotlib

## Reproducibility

The package includes:

- model and stage registers;
- input and output inventories;
- validation reports;
- methodology documentation;
- file checksums;
- raster and table outputs.

## Limitations

This is a strategic planning and screening model. It does not replace parcel-level cadastral review,
geotechnical investigation, environmental impact assessment, infrastructure-capacity analysis or
community consultation.

## Author

{AUTHOR}

Urban and Regional Planning graduate and geospatial analyst.

## Suggested citation

{AUTHOR}. ({current_year}). *{PROJECT_TITLE}*. Geospatial portfolio project.
"""

readme_file = write_text_file(
    GITHUB_DIR
    / "README.md",
    readme_text
)

print(
    f"✓ {readme_file.name}"
)


In [ ]:
# ==================================================================================================
# PROJECT 7 — STAGE 13A CONTINUATION
# SECTIONS 19–28 ONLY
# Complete GitHub metadata, inventory, checksums, validation, ZIP archive and final register
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone
from textwrap import dedent

import hashlib
import json
import zipfile

import pandas as pd


# ==================================================================================================
# 19. RECONNECT TO EXISTING PROJECT FOLDERS
# ==================================================================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

PACKAGE_ROOT = PROJECT_ROOT / "16_Final_Deliverables_Package"

MAPS_DIR = PACKAGE_ROOT / "01_Maps"
FIGURES_DIR = PACKAGE_ROOT / "02_Figures"
GIS_DIR = PACKAGE_ROOT / "03_GIS_Data"
TABLES_DIR = PACKAGE_ROOT / "04_Tables"
REPORTS_DIR = PACKAGE_ROOT / "05_Reports"
PORTFOLIO_DIR = PACKAGE_ROOT / "06_Portfolio"
GITHUB_DIR = PACKAGE_ROOT / "07_GitHub"
METHODOLOGY_DIR = PACKAGE_ROOT / "08_Methodology"
VALIDATION_DIR = PACKAGE_ROOT / "09_Validation"
ARCHIVE_DIR = PACKAGE_ROOT / "10_Project_Archive"

ADMIN_DIR = PROJECT_ROOT / "00_Project_Admin"

PROJECT_TITLE = (
    "Machine-Learning-Based Land Suitability Analysis "
    "for Sustainable Urban Development — Enugu State, Nigeria"
)

AUTHOR = "Abdullah Abdazeez Ayomide"
STUDY_AREA = "Enugu State, Nigeria"

for directory in [
    PACKAGE_ROOT,
    MAPS_DIR,
    FIGURES_DIR,
    GIS_DIR,
    TABLES_DIR,
    REPORTS_DIR,
    PORTFOLIO_DIR,
    GITHUB_DIR,
    METHODOLOGY_DIR,
    VALIDATION_DIR,
    ARCHIVE_DIR,
    ADMIN_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ==================================================================================================
# 20. HELPERS
# ==================================================================================================

def print_heading(title):
    print("\n" + title)
    print("-" * 120)


def write_text_file(path, content):
    path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    path.write_text(
        dedent(content).strip() + "\n",
        encoding="utf-8"
    )

    return path


def validate_file(path, label=None):
    label = label or path.name

    if not path.exists():
        raise FileNotFoundError(
            f"{label} was not found:\n{path}"
        )

    if path.stat().st_size == 0:
        raise RuntimeError(
            f"{label} exists but is empty:\n{path}"
        )


def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as file:
        while True:
            block = file.read(
                block_size
            )

            if not block:
                break

            digest.update(
                block
            )

    return digest.hexdigest()


def human_file_size(size_bytes):
    units = [
        "B",
        "KB",
        "MB",
        "GB",
        "TB"
    ]

    size = float(size_bytes)

    for unit in units:
        if size < 1024 or unit == units[-1]:
            return f"{size:.2f} {unit}"

        size /= 1024


def create_zip_archive(source_directory, zip_path):
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(
        zip_path,
        "w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=6
    ) as archive:

        for file_path in sorted(
            source_directory.rglob("*")
        ):
            if not file_path.is_file():
                continue

            if file_path == zip_path:
                continue

            archive_name = file_path.relative_to(
                source_directory.parent
            )

            archive.write(
                file_path,
                arcname=archive_name
            )

    return zip_path


# ==================================================================================================
# 21. VERIFY EXISTING STAGE 13A CONTENT
# ==================================================================================================

print("=" * 120)
print("STAGE 13A CONTINUATION — FINAL DOCUMENTATION, VALIDATION AND PROJECT ARCHIVE")
print("=" * 120)

print_heading(
    "VERIFYING EXISTING STAGE 13A CONTENT"
)

required_existing_files = [
    REPORTS_DIR / "01_Project_Abstract.md",
    REPORTS_DIR / "02_Technical_Project_Summary.md",
    PORTFOLIO_DIR / "01_Portfolio_Project_Description.md",
    PORTFOLIO_DIR / "02_CV_Ready_Project_Entry.md",
    METHODOLOGY_DIR / "01_Detailed_Methodology.md",
    METHODOLOGY_DIR / "02_Limitations_and_Planning_Recommendations.md",
    METHODOLOGY_DIR / "03_Project_Data_Dictionary.csv",
    GITHUB_DIR / "README.md"
]

for path in required_existing_files:
    validate_file(
        path,
        path.name
    )

    print(
        f"✓ {path.name}"
    )


# ==================================================================================================
# 22. GENERATE GITHUB REPOSITORY METADATA
# ==================================================================================================

print_heading(
    "GENERATING GITHUB REPOSITORY METADATA"
)

github_metadata_text = """
# GitHub Repository Metadata

## Recommended repository name

enugu-ml-urban-land-suitability

## Repository description

Leakage-safe spatial machine-learning analysis of urban expansion, land suitability,
uncertainty and planning priority in Enugu State, Nigeria.

## Recommended topics

- geospatial-analysis
- machine-learning
- urban-planning
- land-suitability
- remote-sensing
- spatial-data-science
- google-earth-engine
- python
- rasterio
- geopandas
- scikit-learn
- urban-expansion
- sustainable-development
- nigeria
- enugu

## Recommended website project title

Machine-Learning Urban Land Suitability — Enugu State

## Recommended project subtitle

Spatial prediction, environmental constraints and confidence-adjusted planning priority.

## Recommended repository licence

MIT Licence for original code.

Third-party datasets remain subject to their original provider licences and attribution requirements.

## Suggested Version 1.0 release title

Final Urban Land Suitability and Planning-Priority Outputs

## Suggested Version 1.0 release notes

- Added leakage-corrected Extra Trees model results.
- Added independent spatial test evaluation.
- Added statewide urban-expansion probability surface.
- Added planning constraints and suitability classification.
- Added model-confidence and uncertainty surfaces.
- Added confidence-adjusted planning-priority output.
- Added seven final maps in PNG and PDF.
- Added publication multi-panel figures.
- Added portfolio presentation board.
- Added final map atlas.
- Added methodology and validation documentation.
"""

github_metadata_file = write_text_file(
    GITHUB_DIR
    / "GitHub_Repository_Metadata.md",
    github_metadata_text
)

print(
    f"✓ {github_metadata_file.name}"
)


# ==================================================================================================
# 23. GENERATE CITATION FILE
# ==================================================================================================

current_year = datetime.now().year

citation_text = f"""
cff-version: 1.2.0
message: "Please cite this project if you use its workflow or derived outputs."
title: "{PROJECT_TITLE}"
type: software
authors:
  - family-names: "Ayomide"
    given-names: "Abdullah Abdazeez"
year: {current_year}
license: MIT
abstract: >
  A leakage-corrected spatial machine-learning framework for urban expansion,
  land suitability, uncertainty and planning-priority assessment in Enugu State, Nigeria.
keywords:
  - geospatial analysis
  - machine learning
  - urban planning
  - land suitability
  - remote sensing
  - Enugu
  - Nigeria
"""

citation_file = write_text_file(
    GITHUB_DIR
    / "CITATION.cff",
    citation_text
)

print(
    f"✓ {citation_file.name}"
)


# ==================================================================================================
# 24. GENERATE REQUIREMENTS FILE
# ==================================================================================================

requirements_text = """
earthengine-api
geemap
geopandas
rasterio
numpy
pandas
scikit-learn
scipy
matplotlib
shapely
pyproj
joblib
openpyxl
highspy
"""

requirements_file = write_text_file(
    GITHUB_DIR
    / "requirements.txt",
    requirements_text
)

print(
    f"✓ {requirements_file.name}"
)


# ==================================================================================================
# 25. GENERATE FINAL FILE INVENTORY
# ==================================================================================================

print_heading(
    "GENERATING FINAL FILE INVENTORY"
)

inventory_records = []

for file_path in sorted(
    PACKAGE_ROOT.rglob("*")
):
    if not file_path.is_file():
        continue

    if file_path.parent == ARCHIVE_DIR:
        continue

    relative_path = file_path.relative_to(
        PACKAGE_ROOT
    )

    inventory_records.append({
        "Relative_Path": str(relative_path),
        "Folder": relative_path.parts[0],
        "File_Name": file_path.name,
        "Extension": file_path.suffix.lower(),
        "Size_Bytes": file_path.stat().st_size,
        "Size_Human": human_file_size(
            file_path.stat().st_size
        ),
        "SHA256": sha256_file(
            file_path
        )
    })

inventory_df = pd.DataFrame(
    inventory_records
)

if inventory_df.empty:
    raise RuntimeError(
        "No package files were found."
    )

inventory_file = (
    TABLES_DIR
    / "Final_Deliverables_File_Inventory.csv"
)

inventory_df.to_csv(
    inventory_file,
    index=False
)

print(
    f"✓ Inventory entries: {len(inventory_df):,}"
)

print(
    f"✓ {inventory_file}"
)


# ==================================================================================================
# 26. GENERATE CHECKSUM MANIFEST
# ==================================================================================================

print_heading(
    "GENERATING SHA-256 CHECKSUM MANIFEST"
)

checksum_lines = []

for _, record in inventory_df.iterrows():
    checksum_lines.append(
        f"{record['SHA256']}  {record['Relative_Path']}"
    )

checksum_file = (
    VALIDATION_DIR
    / "SHA256_CHECKSUMS.txt"
)

write_text_file(
    checksum_file,
    "\n".join(
        checksum_lines
    )
)

print(
    f"✓ {checksum_file.name}"
)


# ==================================================================================================
# 27. GENERATE FOLDER SUMMARY
# ==================================================================================================

folder_counts = (
    inventory_df
    .groupby(
        "Folder"
    )
    .agg(
        File_Count=(
            "File_Name",
            "count"
        ),
        Total_Size_Bytes=(
            "Size_Bytes",
            "sum"
        )
    )
    .reset_index()
)

folder_counts[
    "Total_Size"
] = folder_counts[
    "Total_Size_Bytes"
].apply(
    human_file_size
)

delivery_manifest_file = (
    TABLES_DIR
    / "Final_Delivery_Folder_Summary.csv"
)

folder_counts.to_csv(
    delivery_manifest_file,
    index=False
)

print(
    f"✓ {delivery_manifest_file.name}"
)


# ==================================================================================================
# 28. FINAL PACKAGE VALIDATION
# ==================================================================================================

print_heading(
    "FINAL PACKAGE VALIDATION"
)

validation_records = []


def add_validation(
    check,
    passed,
    observed,
    expected
):
    validation_records.append({
        "Check": check,
        "Passed": bool(passed),
        "Observed": str(observed),
        "Expected": str(expected)
    })


required_directories = [
    MAPS_DIR,
    FIGURES_DIR,
    GIS_DIR,
    TABLES_DIR,
    REPORTS_DIR,
    PORTFOLIO_DIR,
    GITHUB_DIR,
    METHODOLOGY_DIR,
    VALIDATION_DIR,
    ARCHIVE_DIR
]

add_validation(
    "All final package directories exist",
    all(
        directory.exists()
        for directory in required_directories
    ),
    sum(
        directory.exists()
        for directory in required_directories
    ),
    10
)

png_map_count = len(
    list(
        MAPS_DIR.glob("*.png")
    )
)

pdf_map_count = len(
    list(
        MAPS_DIR.glob("*.pdf")
    )
)

add_validation(
    "Seven PNG final maps available",
    png_map_count == 7,
    png_map_count,
    7
)

add_validation(
    "Seven PDF final maps available",
    pdf_map_count == 7,
    pdf_map_count,
    7
)

figure_count = len(
    [
        path
        for path in FIGURES_DIR.iterdir()
        if path.is_file()
    ]
)

add_validation(
    "Publication figures and atlas available",
    figure_count >= 5,
    figure_count,
    "At least 5 files"
)

gis_raster_count = len(
    list(
        GIS_DIR.glob("*.tif")
    )
)

add_validation(
    "Key final GIS rasters available",
    gis_raster_count >= 7,
    gis_raster_count,
    "At least 7 rasters"
)

table_count = len(
    [
        path
        for path in TABLES_DIR.iterdir()
        if path.is_file()
    ]
)

add_validation(
    "Analytical tables available",
    table_count >= 10,
    table_count,
    "At least 10 files"
)

add_validation(
    "Project abstract available",
    (
        REPORTS_DIR
        / "01_Project_Abstract.md"
    ).exists(),
    (
        REPORTS_DIR
        / "01_Project_Abstract.md"
    ).exists(),
    True
)

add_validation(
    "Technical summary available",
    (
        REPORTS_DIR
        / "02_Technical_Project_Summary.md"
    ).exists(),
    (
        REPORTS_DIR
        / "02_Technical_Project_Summary.md"
    ).exists(),
    True
)

add_validation(
    "Portfolio description available",
    (
        PORTFOLIO_DIR
        / "01_Portfolio_Project_Description.md"
    ).exists(),
    (
        PORTFOLIO_DIR
        / "01_Portfolio_Project_Description.md"
    ).exists(),
    True
)

add_validation(
    "Complete README available",
    (
        GITHUB_DIR
        / "README.md"
    ).exists(),
    (
        GITHUB_DIR
        / "README.md"
    ).exists(),
    True
)

add_validation(
    "GitHub metadata available",
    github_metadata_file.exists(),
    github_metadata_file.exists(),
    True
)

add_validation(
    "CITATION.cff available",
    citation_file.exists(),
    citation_file.exists(),
    True
)

add_validation(
    "requirements.txt available",
    requirements_file.exists(),
    requirements_file.exists(),
    True
)

add_validation(
    "Inventory contains package files",
    len(
        inventory_df
    ) > 0,
    len(
        inventory_df
    ),
    "> 0"
)

add_validation(
    "All inventoried files have valid SHA-256 hashes",
    inventory_df[
        "SHA256"
    ].str.len().eq(
        64
    ).all(),
    int(
        inventory_df[
            "SHA256"
        ].str.len().eq(
            64
        ).sum()
    ),
    len(
        inventory_df
    )
)

validation_df = pd.DataFrame(
    validation_records
)

validation_file = (
    VALIDATION_DIR
    / "Stage_13A_Final_Package_Validation.csv"
)

validation_df.to_csv(
    validation_file,
    index=False
)

failed_checks = validation_df.loc[
    ~validation_df[
        "Passed"
    ]
]

if not failed_checks.empty:
    print(
        failed_checks.to_string(
            index=False
        )
    )

    raise RuntimeError(
        "Final package validation failed."
    )


# ==================================================================================================
# 29. CREATE FINAL ZIP ARCHIVE
# ==================================================================================================

print_heading(
    "CREATING FINAL ZIP ARCHIVE"
)

ZIP_FILE = (
    ARCHIVE_DIR
    / "Project_7_Enugu_ML_Land_Suitability_Final_Deliverables.zip"
)

create_zip_archive(
    PACKAGE_ROOT,
    ZIP_FILE
)

validate_file(
    ZIP_FILE,
    "Final ZIP archive"
)

zip_hash = sha256_file(
    ZIP_FILE
)

print(
    f"✓ ZIP archive: {ZIP_FILE}"
)

print(
    f"✓ ZIP size: {human_file_size(ZIP_FILE.stat().st_size)}"
)

print(
    f"✓ ZIP SHA-256: {zip_hash}"
)


# ==================================================================================================
# 30. VALIDATE ZIP INTEGRITY
# ==================================================================================================

print_heading(
    "VALIDATING ZIP ARCHIVE"
)

with zipfile.ZipFile(
    ZIP_FILE,
    "r"
) as archive:

    bad_file = archive.testzip()

    archive_names = archive.namelist()

if bad_file is not None:
    raise RuntimeError(
        f"Corrupted ZIP member detected: {bad_file}"
    )

print(
    "✓ ZIP integrity test passed."
)

print(
    f"✓ Archived file count: {len(archive_names):,}"
)


# ==================================================================================================
# 31. FINAL STAGE 13A JSON REGISTER
# ==================================================================================================

json_register_file = (
    ADMIN_DIR
    / "Stage_13A_Final_Deliverables_Register.json"
)

final_register = {
    "stage": "13A",
    "stage_name": (
        "Final Deliverables Package, GitHub Documentation and Project Archive"
    ),
    "project": PROJECT_TITLE,
    "author": AUTHOR,
    "study_area": STUDY_AREA,
    "completion_time_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "package_root": str(
        PACKAGE_ROOT
    ),
    "map_png_count": png_map_count,
    "map_pdf_count": pdf_map_count,
    "figure_and_atlas_count": figure_count,
    "gis_raster_count": gis_raster_count,
    "table_count": table_count,
    "inventory_file_count": len(
        inventory_df
    ),
    "zip_archive": str(
        ZIP_FILE
    ),
    "zip_size_bytes": ZIP_FILE.stat().st_size,
    "zip_sha256": zip_hash,
    "zip_integrity_passed": True,
    "validation_passed": True
}

with open(
    json_register_file,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        final_register,
        file,
        indent=4
    )


# ==================================================================================================
# 32. FINAL OUTPUT
# ==================================================================================================

print_heading(
    "FINAL FOLDER SUMMARY"
)

print(
    folder_counts[
        [
            "Folder",
            "File_Count",
            "Total_Size"
        ]
    ].to_string(
        index=False
    )
)

print_heading(
    "VALIDATION RESULTS"
)

print(
    validation_df.to_string(
        index=False
    )
)

print_heading(
    "FINAL OUTPUT LOCATIONS"
)

print(
    f"✓ Final package root: {PACKAGE_ROOT}"
)

print(
    f"✓ Final maps: {MAPS_DIR}"
)

print(
    f"✓ Publication figures: {FIGURES_DIR}"
)

print(
    f"✓ GIS data: {GIS_DIR}"
)

print(
    f"✓ Tables: {TABLES_DIR}"
)

print(
    f"✓ Reports: {REPORTS_DIR}"
)

print(
    f"✓ Portfolio documentation: {PORTFOLIO_DIR}"
)

print(
    f"✓ GitHub documentation: {GITHUB_DIR}"
)

print(
    f"✓ Methodology documentation: {METHODOLOGY_DIR}"
)

print(
    f"✓ Validation package: {VALIDATION_DIR}"
)

print(
    f"✓ Final ZIP archive: {ZIP_FILE}"
)

print(
    f"✓ Final register: {json_register_file}"
)

print("=" * 120)
print("STAGE 13A COMPLETED SUCCESSFULLY")
print("=" * 120)
print("✓ Existing Stage 13A outputs were preserved.")
print("✓ GitHub metadata, CITATION and requirements files were generated.")
print("✓ Final file inventory was created.")
print("✓ SHA-256 checksums were generated.")
print("✓ All final package validation checks passed.")
print("✓ The complete deliverables package was compressed into one ZIP archive.")
print("✓ ZIP archive integrity was verified.")
print("=" * 120)

In [ ]:
# ==================================================================================================
# PROJECT 7 — STAGE 13B
# FINAL TECHNICAL REPORT + INFOGRAPHIC CONTENT PACKAGE
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone

import json
import subprocess
import sys
import textwrap

import pandas as pd


# ==================================================================================================
# 1. INSTALL REQUIRED PACKAGES
# ==================================================================================================

required_packages = {
    "python-docx": "docx",
    "reportlab": "reportlab",
    "Pillow": "PIL"
}

for package_name, import_name in required_packages.items():
    try:
        __import__(import_name)
    except ImportError:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                package_name
            ]
        )


# ==================================================================================================
# 2. IMPORT DOCUMENT LIBRARIES
# ==================================================================================================

from docx import Document
from docx.enum.section import WD_ORIENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.shared import Inches, Pt

from PIL import Image

from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER, TA_LEFT
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    Table,
    TableStyle,
    PageBreak,
    Image as RLImage,
    KeepTogether
)


# ==================================================================================================
# 3. PROJECT PATHS
# ==================================================================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

PACKAGE_ROOT = (
    PROJECT_ROOT
    / "16_Final_Deliverables_Package"
)

MAPS_DIR = PACKAGE_ROOT / "01_Maps"
FIGURES_DIR = PACKAGE_ROOT / "02_Figures"
TABLES_DIR = PACKAGE_ROOT / "04_Tables"
REPORTS_DIR = PACKAGE_ROOT / "05_Reports"
PORTFOLIO_DIR = PACKAGE_ROOT / "06_Portfolio"
VALIDATION_DIR = PACKAGE_ROOT / "09_Validation"
ADMIN_DIR = PROJECT_ROOT / "00_Project_Admin"

FINAL_REPORT_DIR = (
    PACKAGE_ROOT
    / "05_Reports"
    / "Final_Report"
)

INFOGRAPHIC_DIR = (
    PACKAGE_ROOT
    / "06_Portfolio"
    / "Infographic_Content"
)

for directory in [
    FINAL_REPORT_DIR,
    INFOGRAPHIC_DIR,
    VALIDATION_DIR,
    ADMIN_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ==================================================================================================
# 4. VERIFIED PROJECT RESULTS
# ==================================================================================================

PROJECT_TITLE = (
    "Machine-Learning-Based Land Suitability Analysis "
    "for Sustainable Urban Development — Enugu State, Nigeria"
)

AUTHOR = "Abdullah Abdazeez Ayomide"

MODEL = "Leakage-corrected Extra Trees classifier"

CRS = "EPSG:32632 — WGS 84 / UTM Zone 32N"

RESOLUTION = "30 m"

TEST_RESULTS = {
    "Accuracy": 0.7111,
    "Balanced Accuracy": 0.7111,
    "Precision": 0.6822,
    "Recall": 0.7906,
    "Specificity": 0.6317,
    "F1 Score": 0.7324,
    "ROC-AUC": 0.7517,
    "PR-AUC": 0.6718,
    "MCC": 0.4277,
    "Cohen Kappa": 0.4222,
    "Brier Score": 0.2015,
    "Log Loss": 0.5956
}

CONFUSION_MATRIX = {
    "True Negative": 1137,
    "False Positive": 663,
    "False Negative": 377,
    "True Positive": 1423
}

SUITABILITY = {
    "Very Low": 2177.69,
    "Low": 1988.35,
    "Moderate": 1353.05,
    "High": 515.09,
    "Very High": 16.67
}

PRIORITY = {
    "Very Low Priority": 4166.04,
    "Low Priority": 923.80,
    "Moderate Priority": 567.79,
    "High Priority": 392.83,
    "Very High Priority": 0.37
}

CONFIDENCE = {
    "Low Confidence": (2146.92, 35.48),
    "Moderate Confidence": (3873.85, 64.02),
    "High Confidence": (30.08, 0.50)
}

STATEWIDE = {
    "Valid model area": 7619.47,
    "Planning applicable area": 6050.85,
    "Planning constrained area": 1568.63,
    "Mean raw probability": 0.3500,
    "Mean constrained probability": 0.3137,
    "Mean uncertainty": 0.2844
}

FEATURE_IMPORTANCE = [
    ("Distance to roads", 41.67),
    ("Population density", 16.97),
    ("Distance to recurring surface water", 12.02),
    ("Elevation", 11.54),
    ("Distance to drainage", 9.06),
    ("Slope", 5.62),
    ("Baseline land cover 2020", 3.12)
]

PREDICTORS = [
    "Elevation",
    "Slope",
    "Distance to roads",
    "Distance to recurring surface water",
    "Distance to drainage",
    "Population density",
    "Baseline Dynamic World 2020 land cover"
]


# ==================================================================================================
# 5. REQUIRED FIGURES
# ==================================================================================================

figure_paths = {
    "Expansion Probability": (
        MAPS_DIR
        / "01_Urban_Expansion_Probability.png"
    ),
    "Suitability": (
        MAPS_DIR
        / "03_Urban_Development_Suitability.png"
    ),
    "Planning Priority": (
        MAPS_DIR
        / "04_Urban_Development_Planning_Priority.png"
    ),
    "Confidence": (
        MAPS_DIR
        / "05_Model_Confidence.png"
    ),
    "Uncertainty": (
        MAPS_DIR
        / "06_Model_Uncertainty.png"
    ),
    "Publication Figure": (
        FIGURES_DIR
        / "Figure_01_Expansion_Suitability_and_Priority.png"
    ),
    "Reliability Figure": (
        FIGURES_DIR
        / "Figure_02_Model_Reliability_and_Planning_Outputs.png"
    )
}


# ==================================================================================================
# 6. INPUT CHECK
# ==================================================================================================

print("=" * 120)
print("STAGE 13B — FINAL TECHNICAL REPORT + INFOGRAPHIC CONTENT PACKAGE")
print("=" * 120)

print("\nINPUT CHECK")
print("-" * 120)

for label, path in figure_paths.items():
    if not path.exists():
        raise FileNotFoundError(
            f"Required figure missing: {label}\n{path}"
        )

    print(
        f"✓ {label}: {path.name}"
    )


# ==================================================================================================
# 7. CREATE KEY RESULTS TABLE
# ==================================================================================================

key_results = pd.DataFrame([
    ["Independent test ROC-AUC", 0.7517, "Model discrimination"],
    ["Independent test F1 score", 0.7324, "Positive-class performance"],
    ["Balanced accuracy", 0.7111, "Equal weighting of both classes"],
    ["Planning-applicable area", 6050.85, "km²"],
    ["Planning-constrained area", 1568.63, "km²"],
    ["High suitability", 515.09, "km²"],
    ["Very high suitability", 16.67, "km²"],
    ["High planning priority", 392.83, "km²"],
    ["Very high planning priority", 0.37, "km²"],
    ["High-confidence land", 30.08, "km²"],
    ["Mean statewide probability", 0.3500, "Probability"],
    ["Mean constrained probability", 0.3137, "Probability"]
], columns=[
    "Indicator",
    "Value",
    "Interpretation"
])

key_results_file = (
    TABLES_DIR
    / "Final_Project_Key_Results.csv"
)

key_results.to_csv(
    key_results_file,
    index=False
)


# ==================================================================================================
# 8. REPORT TEXT
# ==================================================================================================

abstract = """
Rapid urbanisation requires planning approaches that distinguish development pressure from
environmentally and physically suitable land. This project developed a machine-learning-based
spatial framework for sustainable urban development planning across Enugu State, Nigeria.
Quality-controlled urban expansion between 2020 and 2025 was derived from Dynamic World land-cover
observations and used to construct training labels. Terrain, accessibility, hydrological,
population and baseline land-cover predictors were prepared and harmonised on a common spatial grid.

A formal leakage audit identified distance to 2020 built-up land as circular because the same
distance relationship had been used during target construction. The variable was removed before
official modelling. Four candidate algorithms were assessed using spatially separated training and
validation data, and the final Extra Trees classifier was evaluated on a completely independent
spatial test set. The final model achieved ROC-AUC of 0.7517, F1 score of 0.7324 and balanced
accuracy of 0.7111.

Statewide prediction identified 531.76 km² of high or very-high development suitability.
Confidence-adjusted planning analysis identified 392.83 km² of high-priority land and only
0.37 km² of very-high-priority land. Road accessibility was the strongest predictor of observed
urban expansion, followed by population density. The results demonstrate the importance of
combining probability, physical constraints and predictive confidence when translating
machine-learning outputs into planning decisions.
"""

problem_statement = """
Urban development decisions are commonly influenced by infrastructure accessibility, topography,
environmental sensitivity, population pressure and existing land-cover patterns. Conventional GIS
suitability analysis often relies on weighted overlays in which criterion weights are partly
subjective. Although such methods remain useful, they do not directly learn from observed spatial
development patterns and may provide limited information about predictive uncertainty.

Enugu State continues to experience spatial development pressure around established settlements and
transport corridors. Planning authorities therefore require evidence that distinguishes locations
likely to experience development from locations that are genuinely appropriate for development.
This distinction is important because areas with high expansion pressure may simultaneously be
environmentally constrained or supported by weak model confidence.

The project addressed this problem using a spatial machine-learning approach based on observed
2020–2025 urban transitions. It also incorporated explicit leakage assessment, independent spatial
testing, environmental constraints and uncertainty analysis.
"""

objectives = [
    "Prepare spatial predictors representing terrain, accessibility, hydrology, population and baseline land cover.",
    "Construct quality-controlled urban-expansion labels from 2020–2025 Dynamic World observations.",
    "Create spatially independent training, validation and test datasets.",
    "Compare multiple machine-learning algorithms.",
    "Identify and correct target-definition leakage.",
    "Evaluate the final model on a spatially independent test dataset.",
    "Generate statewide urban-expansion probability and uncertainty surfaces.",
    "Apply environmental and planning constraints.",
    "Classify land into development-suitability and confidence-adjusted planning-priority categories.",
    "Produce reproducible cartographic, analytical and portfolio-ready outputs."
]

study_area = """
Enugu State is located in southeastern Nigeria and contains a mixture of urban centres, peri-urban
development, rural settlements, agricultural land, forested areas and topographically variable
terrain. The project used WGS 84 / UTM Zone 32N (EPSG:32632) for metric spatial analysis and final
30-metre statewide modelling.
"""

data_methods = """
The analysis combined Copernicus DEM GLO-30 elevation data, derived slope, Dynamic World V1 land
cover, OpenStreetMap roads, JRC Global Surface Water, MERIT Hydro drainage and WorldPop population
data.

Seven predictors were retained in the official leakage-safe model: elevation, slope, distance to
roads, distance to recurring surface water, distance to drainage, population density and baseline
Dynamic World 2020 land cover.

Urban expansion was defined as a reliable transition from non-built land in 2020 to built land in
2025. Transition quality controls included minimum observation frequency, patch size, spatial
proximity and neighbourhood support. Negative samples represented stable non-built locations within
a realistic urban influence zone.

A balanced sample containing 24,000 observations was created. The training set contained 16,800
samples, while validation and independent test sets contained 3,600 samples each. Spatial block
allocation ensured zero overlap between training, validation and test blocks.
"""

leakage_text = """
Initial modelling produced near-perfect validation performance. Diagnostic review showed that
distance to 2020 built-up land had been used both as a predictor and as part of the sample-definition
rules. A simple 90-metre rule reproduced approximately 97.9% accuracy, confirming target-definition
leakage.

The variable was removed from the official predictor set. This reduced model performance to a more
realistic level but produced a substantially more defensible scientific workflow.
"""

model_results_text = """
Following leakage correction, Extra Trees was selected as the final classifier. Independent spatial
testing produced an accuracy of 0.7111, balanced accuracy of 0.7111, precision of 0.6822, recall of
0.7906 and F1 score of 0.7324. ROC-AUC was 0.7517 and PR-AUC was 0.6718.

The confusion matrix contained 1,137 true negatives, 663 false positives, 377 false negatives and
1,423 true positives. The relatively strong recall indicates that the model identified a large
proportion of observed expansion locations, while lower precision reflects some overprediction.
"""

interpretation_text = """
Distance to roads accounted for approximately 41.67% of grouped native feature importance and was
the dominant predictor. Population density contributed 16.97%, while distance to recurring surface
water contributed 12.02% and elevation 11.54%.

The dominance of road proximity is consistent with the spatial logic of peri-urban expansion:
accessible locations generally face stronger development pressure. Population density acts as an
indicator of settlement concentration and demand. Terrain and hydrological variables further
condition where expansion is physically or environmentally plausible.
"""

statewide_text = """
The final model was applied statewide using tiled raster prediction without retraining. The valid
model area was approximately 7,619.47 km². After environmental and planning constraints were
applied, approximately 6,050.85 km² remained planning-applicable and 1,568.63 km² was constrained.

Mean raw expansion probability was 0.3500, while the mean constrained probability was 0.3137.
Mean uncertainty was 0.2844.
"""

suitability_text = """
The largest suitability class was Very Low, covering 2,177.69 km². Low suitability covered
1,988.35 km² and Moderate suitability covered 1,353.05 km². High suitability occupied
515.09 km², while Very High suitability occupied only 16.67 km².

Combined High and Very High suitability therefore covered approximately 531.76 km².
"""

confidence_text = """
Model-confidence analysis showed that 2,146.92 km² of applicable land fell within the Low
Confidence class, equivalent to 35.48%. Moderate Confidence represented 3,873.85 km² or 64.02%.
Only 30.08 km², approximately 0.50%, achieved High Confidence.

This distribution demonstrates why probability alone should not be interpreted as planning
priority.
"""

priority_text = """
Confidence-adjusted planning priority was intentionally conservative. Very Low Priority accounted
for 4,166.04 km², Low Priority for 923.80 km² and Moderate Priority for 567.79 km². High Priority
covered 392.83 km², while only 0.37 km² was classified as Very High Priority.

The limited Very High Priority area is not a weakness. It reflects the requirement for both high
suitability and strong predictive confidence.
"""

limitations = [
    "Dynamic World transition labels may contain residual classification noise.",
    "Strict positive-label rules may omit some genuine urban expansion.",
    "The model explains the 2020–2025 period and may not capture future structural changes.",
    "OpenStreetMap completeness varies spatially.",
    "WorldPop is modelled population data rather than parcel-level census information.",
    "Land ownership, property price, geotechnical conditions and detailed infrastructure capacity were unavailable.",
    "Independent test performance indicates useful but imperfect spatial generalisation.",
    "The small high-confidence area requires cautious interpretation and local verification."
]

recommendations = [
    "Prioritise high-priority zones for detailed field investigation and local planning.",
    "Treat very-high-priority areas as candidate zones rather than automatic development sites.",
    "Integrate the results with flood-risk, ecosystem-service and agricultural-value assessments.",
    "Coordinate road investment with compact-development objectives.",
    "Protect steep terrain and hydrologically sensitive areas from uncontrolled expansion.",
    "Update the model when improved census, cadastral and planning-permit data become available.",
    "Undertake parcel-level assessment before development approval.",
    "Maintain explicit model-confidence and uncertainty information in future planning applications."
]

conclusion = """
The project demonstrates a defensible approach for combining machine learning, remote sensing and
spatial planning in urban land-suitability assessment. A key methodological contribution was the
formal identification and removal of label-definition leakage before final evaluation.

The independent spatial test confirms that the final model provides useful, though not perfect,
generalisation beyond the training areas. Statewide modelling shows that large areas may exhibit
some development potential, but the amount of land qualifying as both highly suitable and
high-confidence is much smaller.

The final confidence-adjusted priority map therefore provides a more cautious planning product than
a conventional suitability surface alone. It is best interpreted as a strategic screening tool to
guide more detailed planning, environmental assessment and field investigation.
"""


# ==================================================================================================
# 9. WORD REPORT
# ==================================================================================================

print("\nGENERATING WORD REPORT")
print("-" * 120)

doc = Document()

section = doc.sections[0]

section.top_margin = Inches(0.65)
section.bottom_margin = Inches(0.65)
section.left_margin = Inches(0.7)
section.right_margin = Inches(0.7)

styles = doc.styles

styles["Normal"].font.name = "Arial"
styles["Normal"].font.size = Pt(10)

for style_name in [
    "Title",
    "Heading 1",
    "Heading 2"
]:
    styles[style_name].font.name = "Arial"


# Cover
title = doc.add_paragraph()

title.alignment = WD_ALIGN_PARAGRAPH.CENTER

run = title.add_run(
    PROJECT_TITLE
)

run.bold = True
run.font.size = Pt(20)

doc.add_paragraph("")

subtitle = doc.add_paragraph()
subtitle.alignment = WD_ALIGN_PARAGRAPH.CENTER

run = subtitle.add_run(
    "Final Technical Report"
)

run.bold = True
run.font.size = Pt(16)

doc.add_paragraph("")

author_para = doc.add_paragraph()
author_para.alignment = WD_ALIGN_PARAGRAPH.CENTER

author_para.add_run(
    f"Prepared by\n{AUTHOR}\n\n"
    f"Spatial Resolution: {RESOLUTION}\n"
    f"Projection: {CRS}"
)

doc.add_page_break()


def add_heading(text, level=1):
    doc.add_heading(
        text,
        level=level
    )


def add_body(text):
    for block in text.strip().split("\n\n"):
        paragraph = doc.add_paragraph(
            block.strip()
        )

        paragraph.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY


def add_bullets(items):
    for item in items:
        doc.add_paragraph(
            item,
            style="List Bullet"
        )


def add_image(path, caption, width=6.6):
    if path.exists():
        paragraph = doc.add_paragraph()
        paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER

        run = paragraph.add_run()

        run.add_picture(
            str(path),
            width=Inches(width)
        )

        caption_para = doc.add_paragraph()
        caption_para.alignment = WD_ALIGN_PARAGRAPH.CENTER

        caption_run = caption_para.add_run(
            caption
        )

        caption_run.italic = True
        caption_run.font.size = Pt(9)


# Abstract
add_heading(
    "Abstract"
)

add_body(
    abstract
)

add_heading(
    "1. Introduction"
)

add_body(
    problem_statement
)

add_heading(
    "2. Project Objectives"
)

add_bullets(
    objectives
)

add_heading(
    "3. Study Area"
)

add_body(
    study_area
)

add_heading(
    "4. Data and Methods"
)

add_body(
    data_methods
)

add_heading(
    "4.1 Leakage Audit"
)

add_body(
    leakage_text
)

add_heading(
    "5. Machine-Learning Model Performance"
)

add_body(
    model_results_text
)

metrics_table = doc.add_table(
    rows=1,
    cols=2
)

metrics_table.alignment = WD_TABLE_ALIGNMENT.CENTER

metrics_table.rows[0].cells[0].text = "Metric"
metrics_table.rows[0].cells[1].text = "Value"

for metric, value in TEST_RESULTS.items():
    row = metrics_table.add_row().cells
    row[0].text = metric
    row[1].text = f"{value:.4f}"

add_heading(
    "6. Model Interpretation"
)

add_body(
    interpretation_text
)

importance_table = doc.add_table(
    rows=1,
    cols=2
)

importance_table.alignment = WD_TABLE_ALIGNMENT.CENTER

importance_table.rows[0].cells[0].text = "Predictor"
importance_table.rows[0].cells[1].text = "Importance (%)"

for predictor, importance in FEATURE_IMPORTANCE:
    row = importance_table.add_row().cells
    row[0].text = predictor
    row[1].text = f"{importance:.2f}"

add_heading(
    "7. Statewide Urban Expansion Probability"
)

add_body(
    statewide_text
)

add_image(
    figure_paths["Expansion Probability"],
    "Figure 1. Machine-learning predicted urban expansion probability across Enugu State."
)

add_heading(
    "8. Urban Development Suitability"
)

add_body(
    suitability_text
)

suitability_table = doc.add_table(
    rows=1,
    cols=2
)

suitability_table.alignment = WD_TABLE_ALIGNMENT.CENTER

suitability_table.rows[0].cells[0].text = "Suitability Class"
suitability_table.rows[0].cells[1].text = "Area (km²)"

for label, area in SUITABILITY.items():
    row = suitability_table.add_row().cells
    row[0].text = label
    row[1].text = f"{area:,.2f}"

add_image(
    figure_paths["Suitability"],
    "Figure 2. Urban development suitability classes."
)

add_heading(
    "9. Model Confidence and Uncertainty"
)

add_body(
    confidence_text
)

add_image(
    figure_paths["Confidence"],
    "Figure 3. Spatial distribution of model confidence."
)

add_image(
    figure_paths["Uncertainty"],
    "Figure 4. Spatial distribution of model uncertainty."
)

add_heading(
    "10. Planning Priority"
)

add_body(
    priority_text
)

priority_table = doc.add_table(
    rows=1,
    cols=2
)

priority_table.alignment = WD_TABLE_ALIGNMENT.CENTER

priority_table.rows[0].cells[0].text = "Planning Priority"
priority_table.rows[0].cells[1].text = "Area (km²)"

for label, area in PRIORITY.items():
    row = priority_table.add_row().cells
    row[0].text = label
    row[1].text = f"{area:,.2f}"

add_image(
    figure_paths["Planning Priority"],
    "Figure 5. Confidence-adjusted urban-development planning priority."
)

add_heading(
    "11. Integrated Results"
)

add_image(
    figure_paths["Publication Figure"],
    "Figure 6. Integrated urban-expansion, suitability and planning-priority outputs."
)

add_image(
    figure_paths["Reliability Figure"],
    "Figure 7. Model reliability and planning outputs."
)

add_heading(
    "12. Limitations"
)

add_bullets(
    limitations
)

add_heading(
    "13. Planning Recommendations"
)

add_bullets(
    recommendations
)

add_heading(
    "14. Conclusion"
)

add_body(
    conclusion
)

add_heading(
    "15. Data Sources"
)

add_bullets([
    "Dynamic World V1 — land-cover and urban-transition analysis.",
    "Copernicus DEM GLO-30 — elevation and terrain derivatives.",
    "OpenStreetMap — road-network accessibility.",
    "JRC Global Surface Water — recurring surface-water information.",
    "MERIT Hydro — drainage-network information.",
    "WorldPop — population-density estimates."
])

add_heading(
    "16. Software and Tools"
)

add_bullets([
    "Google Earth Engine",
    "Google Colab",
    "Python",
    "GeoPandas",
    "Rasterio",
    "NumPy",
    "pandas",
    "scikit-learn",
    "Matplotlib",
    "Git/GitHub"
])


WORD_REPORT = (
    FINAL_REPORT_DIR
    / "Enugu_ML_Land_Suitability_Final_Technical_Report.docx"
)

doc.save(
    WORD_REPORT
)

print(
    f"✓ Word report: {WORD_REPORT}"
)


# ==================================================================================================
# 10. PDF REPORT
# ==================================================================================================

print("\nGENERATING PDF REPORT")
print("-" * 120)

PDF_REPORT = (
    FINAL_REPORT_DIR
    / "Enugu_ML_Land_Suitability_Final_Technical_Report.pdf"
)

styles_pdf = getSampleStyleSheet()

styles_pdf.add(
    ParagraphStyle(
        name="ReportTitle",
        parent=styles_pdf["Title"],
        alignment=TA_CENTER,
        fontSize=19,
        leading=23,
        spaceAfter=16
    )
)

styles_pdf.add(
    ParagraphStyle(
        name="Section",
        parent=styles_pdf["Heading1"],
        fontSize=13,
        leading=16,
        spaceBefore=12,
        spaceAfter=7
    )
)

styles_pdf.add(
    ParagraphStyle(
        name="Body2",
        parent=styles_pdf["BodyText"],
        fontSize=9.5,
        leading=14,
        alignment=TA_LEFT,
        spaceAfter=8
    )
)

pdf = SimpleDocTemplate(
    str(PDF_REPORT),
    pagesize=A4,
    rightMargin=1.5 * cm,
    leftMargin=1.5 * cm,
    topMargin=1.5 * cm,
    bottomMargin=1.5 * cm
)

story = []

story.append(
    Paragraph(
        PROJECT_TITLE,
        styles_pdf["ReportTitle"]
    )
)

story.append(
    Spacer(
        1,
        0.3 * cm
    )
)

story.append(
    Paragraph(
        f"<b>Final Technical Report</b><br/><br/>"
        f"Prepared by {AUTHOR}<br/>"
        f"Spatial Resolution: {RESOLUTION}<br/>"
        f"Projection: {CRS}",
        styles_pdf["Body2"]
    )
)

story.append(
    PageBreak()
)


def pdf_section(title, text):
    story.append(
        Paragraph(
            title,
            styles_pdf["Section"]
        )
    )

    for block in text.strip().split("\n\n"):
        story.append(
            Paragraph(
                block.strip(),
                styles_pdf["Body2"]
            )
        )


def pdf_bullets(title, items):
    story.append(
        Paragraph(
            title,
            styles_pdf["Section"]
        )
    )

    for item in items:
        story.append(
            Paragraph(
                f"• {item}",
                styles_pdf["Body2"]
            )
        )


def pdf_image(path, caption):
    if not path.exists():
        return

    img = Image.open(path)

    width_px, height_px = img.size

    target_width = 17.0 * cm

    target_height = (
        target_width
        *
        height_px
        /
        width_px
    )

    if target_height > 20 * cm:
        target_height = 20 * cm

        target_width = (
            target_height
            *
            width_px
            /
            height_px
        )

    story.append(
        RLImage(
            str(path),
            width=target_width,
            height=target_height
        )
    )

    story.append(
        Paragraph(
            f"<i>{caption}</i>",
            styles_pdf["Body2"]
        )
    )


pdf_section(
    "Abstract",
    abstract
)

pdf_section(
    "1. Introduction",
    problem_statement
)

pdf_bullets(
    "2. Project Objectives",
    objectives
)

pdf_section(
    "3. Study Area",
    study_area
)

pdf_section(
    "4. Data and Methods",
    data_methods
)

pdf_section(
    "4.1 Leakage Audit",
    leakage_text
)

pdf_section(
    "5. Machine-Learning Model Performance",
    model_results_text
)

metric_data = [
    [
        "Metric",
        "Value"
    ]
]

for metric, value in TEST_RESULTS.items():
    metric_data.append(
        [
            metric,
            f"{value:.4f}"
        ]
    )

metric_table = Table(
    metric_data,
    colWidths=[
        10 * cm,
        4 * cm
    ]
)

metric_table.setStyle(
    TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),
        ("GRID", (0, 0), (-1, -1), 0.4, colors.grey),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("FONTSIZE", (0, 0), (-1, -1), 8.5),
        ("ALIGN", (1, 1), (-1, -1), "RIGHT")
    ])
)

story.append(
    metric_table
)

story.append(
    Spacer(
        1,
        0.4 * cm
    )
)

pdf_section(
    "6. Model Interpretation",
    interpretation_text
)

pdf_section(
    "7. Statewide Urban Expansion Probability",
    statewide_text
)

pdf_image(
    figure_paths["Expansion Probability"],
    "Figure 1. Machine-learning predicted urban expansion probability."
)

pdf_section(
    "8. Urban Development Suitability",
    suitability_text
)

pdf_image(
    figure_paths["Suitability"],
    "Figure 2. Urban-development suitability classes."
)

pdf_section(
    "9. Model Confidence and Uncertainty",
    confidence_text
)

pdf_image(
    figure_paths["Confidence"],
    "Figure 3. Model-confidence classes."
)

pdf_image(
    figure_paths["Uncertainty"],
    "Figure 4. Model uncertainty."
)

pdf_section(
    "10. Planning Priority",
    priority_text
)

pdf_image(
    figure_paths["Planning Priority"],
    "Figure 5. Confidence-adjusted planning priority."
)

pdf_section(
    "11. Integrated Results",
    """
The integrated figures combine probability, environmental applicability, suitability,
model uncertainty, confidence and final planning-priority outputs.
"""
)

pdf_image(
    figure_paths["Publication Figure"],
    "Figure 6. Expansion, suitability and planning-priority outputs."
)

pdf_image(
    figure_paths["Reliability Figure"],
    "Figure 7. Model-reliability and planning outputs."
)

pdf_bullets(
    "12. Limitations",
    limitations
)

pdf_bullets(
    "13. Planning Recommendations",
    recommendations
)

pdf_section(
    "14. Conclusion",
    conclusion
)

pdf.build(
    story
)

print(
    f"✓ PDF report: {PDF_REPORT}"
)


# ==================================================================================================
# 11. INFOGRAPHIC CONTENT PACKAGE
# ==================================================================================================

print("\nGENERATING INFOGRAPHIC CONTENT PACKAGE")
print("-" * 120)

infographic_text = f"""
PROJECT TITLE
{PROJECT_TITLE}

SHORT TITLE
Machine-Learning Urban Land Suitability — Enugu State

PROBLEM
Urban expansion does not necessarily occur in locations that are physically, environmentally or
strategically appropriate for development. A planning framework is required that separates urban
growth pressure from genuine development suitability.

OBJECTIVE
Develop a leakage-safe spatial machine-learning model that predicts urban expansion and converts
the results into environmentally constrained, confidence-adjusted development priority.

DATA
• Dynamic World V1
• Copernicus DEM GLO-30
• OpenStreetMap roads
• JRC Global Surface Water
• MERIT Hydro
• WorldPop
• Enugu State administrative boundary

WORKFLOW
1. Prepare statewide predictors
2. Construct 2020–2025 urban-transition labels
3. Apply transition quality control
4. Build balanced spatial samples
5. Separate train, validation and independent test blocks
6. Train candidate ML models
7. Detect and remove target leakage
8. Evaluate Extra Trees model
9. Predict statewide expansion probability
10. Apply environmental constraints
11. Derive suitability classes
12. Map uncertainty and confidence
13. Create confidence-adjusted planning priority

FINAL MODEL
{MODEL}

MODEL PERFORMANCE
• ROC-AUC: {TEST_RESULTS["ROC-AUC"]:.4f}
• F1 Score: {TEST_RESULTS["F1 Score"]:.4f}
• Balanced Accuracy: {TEST_RESULTS["Balanced Accuracy"]:.4f}
• Recall: {TEST_RESULTS["Recall"]:.4f}
• Precision: {TEST_RESULTS["Precision"]:.4f}

KEY STATEWIDE RESULTS
• Valid model area: {STATEWIDE["Valid model area"]:,.2f} km²
• Planning-applicable land: {STATEWIDE["Planning applicable area"]:,.2f} km²
• Planning-constrained land: {STATEWIDE["Planning constrained area"]:,.2f} km²
• High + Very High suitability: {SUITABILITY["High"] + SUITABILITY["Very High"]:,.2f} km²
• High planning priority: {PRIORITY["High Priority"]:,.2f} km²
• Very High planning priority: {PRIORITY["Very High Priority"]:,.2f} km²
• High-confidence land: {CONFIDENCE["High Confidence"][0]:,.2f} km²

TOP DRIVERS OF URBAN EXPANSION
1. Distance to roads — 41.67%
2. Population density — 16.97%
3. Distance to recurring surface water — 12.02%
4. Elevation — 11.54%
5. Distance to drainage — 9.06%

KEY INSIGHT
Road accessibility was the strongest predictor of observed urban expansion. However, most
planning-applicable land had only moderate model confidence. The confidence-adjusted planning
priority surface therefore provides a more conservative and defensible decision-support product
than probability or suitability alone.

PLANNING IMPLICATION
Prioritise high-priority zones for detailed planning and field assessment while protecting steep,
hydrologically sensitive and environmentally constrained land.

TOOLS
Google Earth Engine • Python • Google Colab • GeoPandas • Rasterio • scikit-learn • Matplotlib

SKILLS DEMONSTRATED
GIS • Remote Sensing • Machine Learning • Spatial Sampling • Data Leakage Detection • Model
Validation • Explainable ML • Raster Modelling • Urban Planning • Scientific Cartography

AUTHOR
{AUTHOR}
"""

INFOGRAPHIC_TEXT = (
    INFOGRAPHIC_DIR
    / "Enugu_ML_Land_Suitability_Infographic_Content.txt"
)

INFOGRAPHIC_TEXT.write_text(
    infographic_text.strip() + "\n",
    encoding="utf-8"
)

print(
    f"✓ Infographic content: {INFOGRAPHIC_TEXT}"
)


# ==================================================================================================
# 12. PORTFOLIO SHORT SUMMARY
# ==================================================================================================

portfolio_summary = f"""
# Machine-Learning Urban Land Suitability — Enugu State

A spatial machine-learning project that predicts urban expansion and translates model outputs into
environmentally constrained and confidence-adjusted planning priorities.

The workflow combined Dynamic World, Copernicus DEM, OpenStreetMap, JRC Surface Water, MERIT Hydro
and WorldPop data. A formal leakage audit identified and removed a circular predictor before final
model evaluation.

The final Extra Trees model achieved an independent spatial ROC-AUC of
{TEST_RESULTS["ROC-AUC"]:.4f} and F1 score of {TEST_RESULTS["F1 Score"]:.4f}.

Key outputs include statewide urban-expansion probability, development suitability, model
uncertainty, confidence and planning-priority maps.

High and very-high suitability covered approximately
{SUITABILITY["High"] + SUITABILITY["Very High"]:,.2f} km², while confidence-adjusted high planning
priority covered {PRIORITY["High Priority"]:,.2f} km².

The project demonstrates practical integration of urban planning, remote sensing, spatial data
science and machine learning.
"""

PORTFOLIO_SUMMARY = (
    PORTFOLIO_DIR
    / "03_Final_Portfolio_Project_Summary.md"
)

PORTFOLIO_SUMMARY.write_text(
    portfolio_summary.strip() + "\n",
    encoding="utf-8"
)


# ==================================================================================================
# 13. VALIDATION
# ==================================================================================================

print("\nSTAGE 13B VALIDATION")
print("-" * 120)

outputs = [
    WORD_REPORT,
    PDF_REPORT,
    INFOGRAPHIC_TEXT,
    PORTFOLIO_SUMMARY,
    key_results_file
]

validation_records = []

for output in outputs:
    passed = (
        output.exists()
        and
        output.stat().st_size > 0
    )

    validation_records.append({
        "Output": output.name,
        "Passed": passed,
        "Size_Bytes": (
            output.stat().st_size
            if output.exists()
            else 0
        ),
        "Path": str(output)
    })

validation_df = pd.DataFrame(
    validation_records
)

VALIDATION_FILE = (
    VALIDATION_DIR
    / "Stage_13B_Final_Report_Validation.csv"
)

validation_df.to_csv(
    VALIDATION_FILE,
    index=False
)

if not validation_df[
    "Passed"
].all():

    print(
        validation_df.to_string(
            index=False
        )
    )

    raise RuntimeError(
        "Stage 13B validation failed."
    )


# ==================================================================================================
# 14. JSON REGISTER
# ==================================================================================================

REGISTER_FILE = (
    ADMIN_DIR
    / "Stage_13B_Final_Report_Register.json"
)

register = {
    "stage": "13B",
    "stage_name": "Final Technical Report and Infographic Content Package",
    "project": PROJECT_TITLE,
    "author": AUTHOR,
    "completion_time_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "outputs": {
        "word_report": str(
            WORD_REPORT
        ),
        "pdf_report": str(
            PDF_REPORT
        ),
        "infographic_content": str(
            INFOGRAPHIC_TEXT
        ),
        "portfolio_summary": str(
            PORTFOLIO_SUMMARY
        ),
        "key_results_table": str(
            key_results_file
        )
    },
    "validation_file": str(
        VALIDATION_FILE
    ),
    "validation_passed": True
}

with open(
    REGISTER_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        register,
        file,
        indent=4
    )


# ==================================================================================================
# 15. FINAL OUTPUT
# ==================================================================================================

print(
    validation_df.to_string(
        index=False
    )
)

print("\nFINAL OUTPUT LOCATIONS")
print("-" * 120)

print(
    f"✓ Word technical report: {WORD_REPORT}"
)

print(
    f"✓ PDF technical report: {PDF_REPORT}"
)

print(
    f"✓ Infographic content: {INFOGRAPHIC_TEXT}"
)

print(
    f"✓ Portfolio summary: {PORTFOLIO_SUMMARY}"
)

print(
    f"✓ Key-results table: {key_results_file}"
)

print(
    f"✓ Validation register: {VALIDATION_FILE}"
)

print(
    f"✓ Stage register: {REGISTER_FILE}"
)

print("=" * 120)
print("STAGE 13B COMPLETED SUCCESSFULLY")
print("=" * 120)
print("✓ Final Word technical report generated.")
print("✓ Final PDF technical report generated.")
print("✓ Verified project maps and figures embedded.")
print("✓ Infographic-ready content package generated.")
print("✓ Final portfolio project summary generated.")
print("✓ Key-results table generated.")
print("✓ All Stage 13B validation checks passed.")
print("=" * 120)

In [ ]:
# ==================================================================================================
# PROJECT 7 — STAGE 13C
# DEFINITIVE FINAL ARCHIVE REFRESH, INVENTORY, CHECKSUMS AND PROJECT CLOSURE
#
# Purpose:
#   - incorporate all Stage 13B outputs into the definitive package
#   - refresh final validation
#   - rebuild file inventory
#   - rebuild SHA-256 checksum manifest
#   - recreate the final ZIP archive
#   - verify ZIP integrity
#   - generate final project closure register
# ==================================================================================================

from pathlib import Path
from datetime import datetime, timezone

import hashlib
import json
import shutil
import zipfile

import pandas as pd


# ==================================================================================================
# 1. PROJECT PATHS
# ==================================================================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Project_7_Enugu_ML_Land_Suitability"
)

PACKAGE_ROOT = (
    PROJECT_ROOT
    / "16_Final_Deliverables_Package"
)

MAPS_DIR = PACKAGE_ROOT / "01_Maps"
FIGURES_DIR = PACKAGE_ROOT / "02_Figures"
GIS_DIR = PACKAGE_ROOT / "03_GIS_Data"
TABLES_DIR = PACKAGE_ROOT / "04_Tables"
REPORTS_DIR = PACKAGE_ROOT / "05_Reports"
PORTFOLIO_DIR = PACKAGE_ROOT / "06_Portfolio"
GITHUB_DIR = PACKAGE_ROOT / "07_GitHub"
METHODOLOGY_DIR = PACKAGE_ROOT / "08_Methodology"
VALIDATION_DIR = PACKAGE_ROOT / "09_Validation"
ARCHIVE_DIR = PACKAGE_ROOT / "10_Project_Archive"

FINAL_REPORT_DIR = (
    REPORTS_DIR
    / "Final_Report"
)

INFOGRAPHIC_DIR = (
    PORTFOLIO_DIR
    / "Infographic_Content"
)

ADMIN_DIR = (
    PROJECT_ROOT
    / "00_Project_Admin"
)

for directory in [
    MAPS_DIR,
    FIGURES_DIR,
    GIS_DIR,
    TABLES_DIR,
    REPORTS_DIR,
    PORTFOLIO_DIR,
    GITHUB_DIR,
    METHODOLOGY_DIR,
    VALIDATION_DIR,
    ARCHIVE_DIR,
    FINAL_REPORT_DIR,
    INFOGRAPHIC_DIR,
    ADMIN_DIR
]:
    directory.mkdir(
        parents=True,
        exist_ok=True
    )


# ==================================================================================================
# 2. METADATA
# ==================================================================================================

PROJECT_TITLE = (
    "Machine-Learning-Based Land Suitability Analysis "
    "for Sustainable Urban Development — Enugu State, Nigeria"
)

AUTHOR = "Abdullah Abdazeez Ayomide"

STUDY_AREA = "Enugu State, Nigeria"


# ==================================================================================================
# 3. FILE PATHS
# ==================================================================================================

WORD_REPORT = (
    FINAL_REPORT_DIR
    / "Enugu_ML_Land_Suitability_Final_Technical_Report.docx"
)

PDF_REPORT = (
    FINAL_REPORT_DIR
    / "Enugu_ML_Land_Suitability_Final_Technical_Report.pdf"
)

INFOGRAPHIC_CONTENT = (
    INFOGRAPHIC_DIR
    / "Enugu_ML_Land_Suitability_Infographic_Content.txt"
)

PORTFOLIO_SUMMARY = (
    PORTFOLIO_DIR
    / "03_Final_Portfolio_Project_Summary.md"
)

KEY_RESULTS = (
    TABLES_DIR
    / "Final_Project_Key_Results.csv"
)

STAGE_13B_VALIDATION = (
    VALIDATION_DIR
    / "Stage_13B_Final_Report_Validation.csv"
)

STAGE_13A_REGISTER_SOURCE = (
    ADMIN_DIR
    / "Stage_13A_Final_Deliverables_Register.json"
)

STAGE_13B_REGISTER_SOURCE = (
    ADMIN_DIR
    / "Stage_13B_Final_Report_Register.json"
)

FINAL_VALIDATION_FILE = (
    VALIDATION_DIR
    / "Stage_13C_Definitive_Project_Validation.csv"
)

INVENTORY_FILE = (
    TABLES_DIR
    / "Final_Deliverables_File_Inventory.csv"
)

FOLDER_SUMMARY_FILE = (
    TABLES_DIR
    / "Final_Delivery_Folder_Summary.csv"
)

CHECKSUM_FILE = (
    VALIDATION_DIR
    / "SHA256_CHECKSUMS.txt"
)

ZIP_FILE = (
    ARCHIVE_DIR
    / "Project_7_Enugu_ML_Land_Suitability_Final_Deliverables.zip"
)

FINAL_REGISTER_FILE = (
    ADMIN_DIR
    / "Stage_13C_Project_Closure_Register.json"
)


# ==================================================================================================
# 4. HELPERS
# ==================================================================================================

def print_heading(title):
    print("\n" + title)
    print("-" * 120)


def validate_file(path, label=None):
    label = label or path.name

    if not path.exists():
        raise FileNotFoundError(
            f"{label} was not found:\n{path}"
        )

    if path.stat().st_size == 0:
        raise RuntimeError(
            f"{label} exists but is empty:\n{path}"
        )


def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as file:

        while True:
            block = file.read(
                block_size
            )

            if not block:
                break

            digest.update(
                block
            )

    return digest.hexdigest()


def human_file_size(size_bytes):
    units = [
        "B",
        "KB",
        "MB",
        "GB",
        "TB"
    ]

    size = float(
        size_bytes
    )

    for unit in units:
        if size < 1024 or unit == units[-1]:
            return f"{size:.2f} {unit}"

        size /= 1024


def create_zip_archive(
    source_directory,
    zip_path
):
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(
        zip_path,
        "w",
        compression=zipfile.ZIP_DEFLATED,
        compresslevel=6
    ) as archive:

        for file_path in sorted(
            source_directory.rglob("*")
        ):

            if not file_path.is_file():
                continue

            # Do not recursively place archives inside the archive.
            if file_path.parent == ARCHIVE_DIR:
                continue

            archive_name = file_path.relative_to(
                source_directory.parent
            )

            archive.write(
                file_path,
                arcname=archive_name
            )

    return zip_path


# ==================================================================================================
# 5. START
# ==================================================================================================

print("=" * 120)
print("STAGE 13C — DEFINITIVE FINAL ARCHIVE REFRESH AND PROJECT CLOSURE")
print("=" * 120)


# ==================================================================================================
# 6. VERIFY STAGE 13B OUTPUTS
# ==================================================================================================

print_heading(
    "VERIFYING STAGE 13B OUTPUTS"
)

required_13b_outputs = [
    WORD_REPORT,
    PDF_REPORT,
    INFOGRAPHIC_CONTENT,
    PORTFOLIO_SUMMARY,
    KEY_RESULTS,
    STAGE_13B_VALIDATION
]

for path in required_13b_outputs:

    validate_file(
        path,
        path.name
    )

    print(
        f"✓ {path.name}"
    )


# ==================================================================================================
# 7. COPY FINAL STAGE REGISTERS INTO PACKAGE
# ==================================================================================================

print_heading(
    "ADDING FINAL STAGE REGISTERS TO DELIVERABLES PACKAGE"
)

for source in [
    STAGE_13A_REGISTER_SOURCE,
    STAGE_13B_REGISTER_SOURCE
]:

    if source.exists():

        destination = (
            VALIDATION_DIR
            / source.name
        )

        shutil.copy2(
            source,
            destination
        )

        print(
            f"✓ {destination.name}"
        )


# ==================================================================================================
# 8. REMOVE STALE FINALIZATION FILES
# ==================================================================================================

print_heading(
    "REMOVING STALE FINALIZATION FILES"
)

for stale_file in [
    INVENTORY_FILE,
    FOLDER_SUMMARY_FILE,
    CHECKSUM_FILE,
    FINAL_VALIDATION_FILE,
    ZIP_FILE
]:

    if stale_file.exists():

        stale_file.unlink()

        print(
            f"✓ Removed old {stale_file.name}"
        )


# ==================================================================================================
# 9. DEFINITIVE PACKAGE VALIDATION
# ==================================================================================================

print_heading(
    "DEFINITIVE PACKAGE VALIDATION"
)

validation_records = []


def add_validation(
    check,
    passed,
    observed,
    expected
):

    validation_records.append({
        "Check": check,
        "Passed": bool(
            passed
        ),
        "Observed": str(
            observed
        ),
        "Expected": str(
            expected
        )
    })


required_directories = [
    MAPS_DIR,
    FIGURES_DIR,
    GIS_DIR,
    TABLES_DIR,
    REPORTS_DIR,
    PORTFOLIO_DIR,
    GITHUB_DIR,
    METHODOLOGY_DIR,
    VALIDATION_DIR,
    ARCHIVE_DIR
]

add_validation(
    "All ten final package directories exist",
    all(
        directory.exists()
        for directory in required_directories
    ),
    sum(
        directory.exists()
        for directory in required_directories
    ),
    10
)


png_maps = list(
    MAPS_DIR.glob(
        "*.png"
    )
)

pdf_maps = list(
    MAPS_DIR.glob(
        "*.pdf"
    )
)

add_validation(
    "Seven final PNG maps available",
    len(
        png_maps
    ) == 7,
    len(
        png_maps
    ),
    7
)

add_validation(
    "Seven final PDF maps available",
    len(
        pdf_maps
    ) == 7,
    len(
        pdf_maps
    ),
    7
)


figure_files = [
    path
    for path in FIGURES_DIR.iterdir()
    if path.is_file()
]

add_validation(
    "Publication figures and map atlas available",
    len(
        figure_files
    ) >= 5,
    len(
        figure_files
    ),
    "At least 5"
)


gis_files = list(
    GIS_DIR.glob(
        "*.tif"
    )
)

add_validation(
    "Final GIS rasters available",
    len(
        gis_files
    ) >= 10,
    len(
        gis_files
    ),
    "At least 10"
)


table_files = [
    path
    for path in TABLES_DIR.iterdir()
    if path.is_file()
]

add_validation(
    "Analytical tables available",
    len(
        table_files
    ) >= 17,
    len(
        table_files
    ),
    "At least 17"
)


add_validation(
    "Final Word report exists",
    WORD_REPORT.exists()
    and
    WORD_REPORT.stat().st_size > 0,
    human_file_size(
        WORD_REPORT.stat().st_size
    )
    if WORD_REPORT.exists()
    else "Missing",
    "Non-empty"
)


add_validation(
    "Final PDF report exists",
    PDF_REPORT.exists()
    and
    PDF_REPORT.stat().st_size > 0,
    human_file_size(
        PDF_REPORT.stat().st_size
    )
    if PDF_REPORT.exists()
    else "Missing",
    "Non-empty"
)


add_validation(
    "Infographic content package exists",
    INFOGRAPHIC_CONTENT.exists()
    and
    INFOGRAPHIC_CONTENT.stat().st_size > 0,
    INFOGRAPHIC_CONTENT.exists(),
    True
)


add_validation(
    "Final portfolio summary exists",
    PORTFOLIO_SUMMARY.exists()
    and
    PORTFOLIO_SUMMARY.stat().st_size > 0,
    PORTFOLIO_SUMMARY.exists(),
    True
)


add_validation(
    "Final key-results table exists",
    KEY_RESULTS.exists()
    and
    KEY_RESULTS.stat().st_size > 0,
    KEY_RESULTS.exists(),
    True
)


readme_file = (
    GITHUB_DIR
    / "README.md"
)

citation_file = (
    GITHUB_DIR
    / "CITATION.cff"
)

requirements_file = (
    GITHUB_DIR
    / "requirements.txt"
)

add_validation(
    "Complete GitHub README exists",
    readme_file.exists()
    and
    readme_file.stat().st_size > 0,
    readme_file.exists(),
    True
)

add_validation(
    "CITATION.cff exists",
    citation_file.exists()
    and
    citation_file.stat().st_size > 0,
    citation_file.exists(),
    True
)

add_validation(
    "requirements.txt exists",
    requirements_file.exists()
    and
    requirements_file.stat().st_size > 0,
    requirements_file.exists(),
    True
)


methodology_files = [
    path
    for path in METHODOLOGY_DIR.iterdir()
    if path.is_file()
]

add_validation(
    "Methodology package available",
    len(
        methodology_files
    ) >= 3,
    len(
        methodology_files
    ),
    "At least 3"
)


add_validation(
    "Stage 13B validation passed",
    STAGE_13B_VALIDATION.exists(),
    STAGE_13B_VALIDATION.exists(),
    True
)


validation_df = pd.DataFrame(
    validation_records
)

failed_checks = validation_df.loc[
    ~validation_df[
        "Passed"
    ]
]

if not failed_checks.empty:

    print(
        failed_checks.to_string(
            index=False
        )
    )

    raise RuntimeError(
        "Stage 13C package validation failed."
    )


validation_df.to_csv(
    FINAL_VALIDATION_FILE,
    index=False
)

print(
    validation_df.to_string(
        index=False
    )
)

print(
    f"\n✓ Validation register: {FINAL_VALIDATION_FILE}"
)


# ==================================================================================================
# 10. CREATE FINAL FOLDER SUMMARY
# ==================================================================================================

print_heading(
    "GENERATING DEFINITIVE FOLDER SUMMARY"
)

summary_records = []

for directory in required_directories[:-1]:

    folder_files = [
        path
        for path in directory.rglob("*")
        if path.is_file()
    ]

    total_size = sum(
        path.stat().st_size
        for path in folder_files
    )

    summary_records.append({
        "Folder": directory.name,
        "File_Count": len(
            folder_files
        ),
        "Total_Size_Bytes": total_size,
        "Total_Size": human_file_size(
            total_size
        )
    })


folder_summary_df = pd.DataFrame(
    summary_records
)

folder_summary_df.to_csv(
    FOLDER_SUMMARY_FILE,
    index=False
)

print(
    folder_summary_df[
        [
            "Folder",
            "File_Count",
            "Total_Size"
        ]
    ].to_string(
        index=False
    )
)

print(
    f"\n✓ {FOLDER_SUMMARY_FILE}"
)


# ==================================================================================================
# 11. CREATE DEFINITIVE FILE INVENTORY
#
# Inventory deliberately excludes:
#   - itself
#   - checksum manifest
#   - ZIP archive
#
# This avoids circular/self-referential hashes.
# ==================================================================================================

print_heading(
    "GENERATING DEFINITIVE FILE INVENTORY"
)

excluded_inventory_paths = {
    INVENTORY_FILE.resolve(),
    CHECKSUM_FILE.resolve()
}

inventory_records = []

for file_path in sorted(
    PACKAGE_ROOT.rglob("*")
):

    if not file_path.is_file():
        continue

    if file_path.parent == ARCHIVE_DIR:
        continue

    if file_path.resolve() in excluded_inventory_paths:
        continue

    relative_path = file_path.relative_to(
        PACKAGE_ROOT
    )

    inventory_records.append({
        "Relative_Path": str(
            relative_path
        ),
        "Folder": relative_path.parts[0],
        "File_Name": file_path.name,
        "Extension": file_path.suffix.lower(),
        "Size_Bytes": file_path.stat().st_size,
        "Size_Human": human_file_size(
            file_path.stat().st_size
        ),
        "SHA256": sha256_file(
            file_path
        )
    })


inventory_df = pd.DataFrame(
    inventory_records
)

if inventory_df.empty:
    raise RuntimeError(
        "Final inventory is empty."
    )


inventory_df.to_csv(
    INVENTORY_FILE,
    index=False
)

print(
    f"✓ Definitive inventory entries: {len(inventory_df):,}"
)

print(
    f"✓ {INVENTORY_FILE}"
)


# ==================================================================================================
# 12. CREATE DEFINITIVE SHA-256 MANIFEST
#
# Includes the final inventory itself.
# Excludes only:
#   - checksum manifest itself
#   - archive
# ==================================================================================================

print_heading(
    "GENERATING DEFINITIVE SHA-256 MANIFEST"
)

checksum_records = []

for file_path in sorted(
    PACKAGE_ROOT.rglob("*")
):

    if not file_path.is_file():
        continue

    if file_path.parent == ARCHIVE_DIR:
        continue

    if file_path.resolve() == CHECKSUM_FILE.resolve():
        continue

    relative_path = file_path.relative_to(
        PACKAGE_ROOT
    )

    checksum_records.append(
        (
            sha256_file(
                file_path
            ),
            str(
                relative_path
            )
        )
    )


with open(
    CHECKSUM_FILE,
    "w",
    encoding="utf-8"
) as file:

    for file_hash, relative_path in checksum_records:

        file.write(
            f"{file_hash}  {relative_path}\n"
        )


print(
    f"✓ Files hashed: {len(checksum_records):,}"
)

print(
    f"✓ {CHECKSUM_FILE}"
)


# ==================================================================================================
# 13. VERIFY SHA-256 MANIFEST
# ==================================================================================================

print_heading(
    "VERIFYING SHA-256 MANIFEST"
)

checksum_failures = []

for expected_hash, relative_path in checksum_records:

    path = (
        PACKAGE_ROOT
        / relative_path
    )

    observed_hash = sha256_file(
        path
    )

    if observed_hash != expected_hash:

        checksum_failures.append(
            relative_path
        )


if checksum_failures:

    raise RuntimeError(
        "Checksum verification failed for:\n"
        +
        "\n".join(
            checksum_failures
        )
    )


print(
    f"✓ All {len(checksum_records):,} checksums verified."
)


# ==================================================================================================
# 14. CREATE DEFINITIVE FINAL ZIP
# ==================================================================================================

print_heading(
    "CREATING DEFINITIVE FINAL ZIP ARCHIVE"
)

create_zip_archive(
    PACKAGE_ROOT,
    ZIP_FILE
)

validate_file(
    ZIP_FILE,
    "Definitive final ZIP archive"
)

zip_size = ZIP_FILE.stat().st_size

zip_hash = sha256_file(
    ZIP_FILE
)

print(
    f"✓ ZIP archive: {ZIP_FILE}"
)

print(
    f"✓ ZIP size: {human_file_size(zip_size)}"
)

print(
    f"✓ ZIP SHA-256: {zip_hash}"
)


# ==================================================================================================
# 15. ZIP INTEGRITY TEST
# ==================================================================================================

print_heading(
    "VALIDATING DEFINITIVE ZIP ARCHIVE"
)

with zipfile.ZipFile(
    ZIP_FILE,
    "r"
) as archive:

    bad_member = archive.testzip()

    archive_names = archive.namelist()


if bad_member is not None:

    raise RuntimeError(
        f"Corrupted ZIP member detected: {bad_member}"
    )


print(
    "✓ ZIP integrity test passed."
)

print(
    f"✓ Archived file count: {len(archive_names):,}"
)


# ==================================================================================================
# 16. CONFIRM CRITICAL FILES ARE INSIDE ZIP
# ==================================================================================================

print_heading(
    "VERIFYING CRITICAL FILES INSIDE ZIP"
)

critical_archive_files = [
    (
        "16_Final_Deliverables_Package/"
        "05_Reports/Final_Report/"
        "Enugu_ML_Land_Suitability_Final_Technical_Report.docx"
    ),
    (
        "16_Final_Deliverables_Package/"
        "05_Reports/Final_Report/"
        "Enugu_ML_Land_Suitability_Final_Technical_Report.pdf"
    ),
    (
        "16_Final_Deliverables_Package/"
        "06_Portfolio/Infographic_Content/"
        "Enugu_ML_Land_Suitability_Infographic_Content.txt"
    ),
    (
        "16_Final_Deliverables_Package/"
        "04_Tables/"
        "Final_Project_Key_Results.csv"
    ),
    (
        "16_Final_Deliverables_Package/"
        "07_GitHub/README.md"
    ),
    (
        "16_Final_Deliverables_Package/"
        "09_Validation/SHA256_CHECKSUMS.txt"
    )
]


missing_archive_files = [
    path
    for path in critical_archive_files
    if path not in archive_names
]


if missing_archive_files:

    raise RuntimeError(
        "Critical files missing from final ZIP:\n"
        +
        "\n".join(
            missing_archive_files
        )
    )


for path in critical_archive_files:

    print(
        f"✓ {Path(path).name}"
    )


# ==================================================================================================
# 17. FINAL PROJECT CLOSURE REGISTER
# ==================================================================================================

final_register = {
    "project": PROJECT_TITLE,
    "author": AUTHOR,
    "study_area": STUDY_AREA,
    "stage": "13C",
    "stage_name": (
        "Definitive Final Archive Refresh and Project Closure"
    ),
    "completion_time_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "project_status": "COMPLETED",
    "package_root": str(
        PACKAGE_ROOT
    ),
    "final_word_report": str(
        WORD_REPORT
    ),
    "final_pdf_report": str(
        PDF_REPORT
    ),
    "infographic_content": str(
        INFOGRAPHIC_CONTENT
    ),
    "final_inventory": str(
        INVENTORY_FILE
    ),
    "checksum_manifest": str(
        CHECKSUM_FILE
    ),
    "final_validation": str(
        FINAL_VALIDATION_FILE
    ),
    "inventory_entry_count": int(
        len(
            inventory_df
        )
    ),
    "checksum_entry_count": int(
        len(
            checksum_records
        )
    ),
    "archive_file_count": int(
        len(
            archive_names
        )
    ),
    "zip_archive": str(
        ZIP_FILE
    ),
    "zip_size_bytes": int(
        zip_size
    ),
    "zip_sha256": zip_hash,
    "zip_integrity_passed": True,
    "critical_archive_files_verified": True,
    "final_validation_passed": True
}


with open(
    FINAL_REGISTER_FILE,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        final_register,
        file,
        indent=4
    )


# ==================================================================================================
# 18. FINAL PROJECT SUMMARY
# ==================================================================================================

print_heading(
    "FINAL PROJECT DELIVERY SUMMARY"
)

print(
    f"Final PNG maps              : {len(png_maps)}"
)

print(
    f"Final PDF maps              : {len(pdf_maps)}"
)

print(
    f"Publication figures/atlas   : {len(figure_files)}"
)

print(
    f"Final GIS rasters           : {len(gis_files)}"
)

print(
    f"Inventory entries           : {len(inventory_df):,}"
)

print(
    f"Checksum entries            : {len(checksum_records):,}"
)

print(
    f"Files inside final ZIP      : {len(archive_names):,}"
)

print(
    f"Final ZIP size              : {human_file_size(zip_size)}"
)

print(
    f"Final ZIP SHA-256           : {zip_hash}"
)


print_heading(
    "FINAL OUTPUT LOCATIONS"
)

print(
    f"✓ Final deliverables package:\n  {PACKAGE_ROOT}"
)

print(
    f"\n✓ Final Word report:\n  {WORD_REPORT}"
)

print(
    f"\n✓ Final PDF report:\n  {PDF_REPORT}"
)

print(
    f"\n✓ Final inventory:\n  {INVENTORY_FILE}"
)

print(
    f"\n✓ Final checksum manifest:\n  {CHECKSUM_FILE}"
)

print(
    f"\n✓ Definitive ZIP archive:\n  {ZIP_FILE}"
)

print(
    f"\n✓ Project closure register:\n  {FINAL_REGISTER_FILE}"
)


print("\n" + "=" * 120)
print("PROJECT 7 COMPLETED SUCCESSFULLY")
print("=" * 120)

print(
    "✓ All GIS and remote-sensing inputs were processed."
)

print(
    "✓ Urban-transition labels were quality controlled."
)

print(
    "✓ Spatial train-validation-test separation was implemented."
)

print(
    "✓ Target-definition leakage was identified and corrected."
)

print(
    "✓ The final Extra Trees model was independently evaluated."
)

print(
    "✓ Statewide probability, suitability, uncertainty and confidence surfaces were generated."
)

print(
    "✓ Confidence-adjusted planning priorities were produced."
)

print(
    "✓ Publication-quality maps and multi-panel figures were generated."
)

print(
    "✓ Final Word and PDF technical reports were generated."
)

print(
    "✓ Portfolio, infographic and GitHub documentation were completed."
)

print(
    "✓ Final inventory and SHA-256 checksums were verified."
)

print(
    "✓ The definitive ZIP archive was created and integrity-tested."
)

print(
    "✓ PROJECT 7 IS NOW FULLY COMPLETED AND ARCHIVED."
)

print("=" * 120)